# SINCON 2026 v10 최종 노트북
입력: 대회 `sincon-2026` 데이터만 연결. 달력·주최 측 날씨(날짜 단위)·h9−v8d 차이는 노트북 안에 포함. GPU·인터넷 필요.
v10 = 0.8·[LightGBM lt1·lt2 0.5 + Chronos-2 (chr_e3 + wx_chr_st) 0.5] + 0.2·가벼운 LightGBM(lgbL) → v8d 후보정.
`submission.csv` = h9 + (v10 − v8d) (로그 공간).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting==2.3.2"], check=True)
import os, json, time, shutil
from pathlib import Path
import pandas as pd
WORK = Path("/kaggle/working") if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else Path.cwd()
os.chdir(WORK)
for n in ("train.csv", "test.csv", "metaData.csv"):  # 대회 데이터를 작업 폴더로
    if not (WORK / n).exists():
        src = sorted(Path("/kaggle/input").glob(f"**/{n}")) if Path("/kaggle/input").exists() else []
        assert src, f"대회 데이터 {n} 를 찾을 수 없음"
        shutil.copy(src[0], WORK / n)
print("work dir:", WORK)

In [ ]:
# 보조 데이터 내장
AUX = {"station_geo_graph.csv": "station_id,lat,lon,g1_eig1,g1_eig2,g1_eig3,g1_eig4,g1_eig5,g1_eig6,g2_eig1,g2_eig2,g2_eig3,g2_eig4,g2_eig5,g2_eig6,g1_betweenness,g1_closeness,g1_degree,n_external_lines,is_transfer,gdist_airport,gdist_seoul,hops_seoul,gdist_songdo,hops_new_ext,hops_end_of_line,n_within_1500m,nearest_other_m,dist_munhak,dist_asiad,dist_songdo_park,dist_bucheon_stadium,dist_incheon_grand_park,dist_bupyeong_market,g2_cluster_sim_mean\n3107,37.60244,126.688566,-0.069868,-0.175686,0.167031,-0.092317,0.067137,-0.020717,-0.102756,-0.004573,-0.019325,-0.145215,0.287237,0.242968,0.0,6.1e-05,1,0,0,5966.379677,5966.379677,4,24613.3807,0,0,1,877.83591,18400.571462,5849.021262,22422.506385,14512.015273,17909.163436,12779.205414,0.700687\n3108,37.602676,126.698526,-0.098378,-0.246723,0.233502,-0.128695,0.091878,-0.028116,-0.083371,-0.068129,0.024488,0.19129,-0.007631,-0.057241,0.026667,6.4e-05,2,0,0,5088.543768,5088.543768,3,23735.544791,0,1,1,877.83591,18428.363523,6093.975446,22671.56147,13965.404874,17675.500079,12619.291133,0.89976\n3109,37.592259,126.713301,-0.097088,-0.241541,0.225415,-0.123165,0.082872,-0.024663,-0.07094,-0.085486,0.059295,0.274336,-0.019051,0.014547,0.052613,7.2e-05,2,0,0,3346.140257,3346.140257,2,21993.14128,0,2,0,1742.40351,17354.825261,5640.6814,21968.447492,12250.301971,16233.90432,11298.957552,0.898321\n3110,37.571501,126.735858,-0.09495,-0.232986,0.212144,-0.114122,0.06851,-0.019219,-0.123093,-0.029367,-0.041522,-0.092658,-0.016177,-0.017711,0.245405,9e-05,2,1,1,300.0,300.0,1,18947.001023,1,3,1,828.501291,15421.975218,5712.503114,20648.139247,9210.733689,13627.810929,9012.192532,0.656679\n3111,37.566354,126.742655,-0.091984,-0.221176,0.193995,-0.101826,0.049721,-0.012223,-0.095665,-0.053931,-0.029878,0.031292,0.078749,-0.065921,0.236036,9.3e-05,2,0,0,1128.501291,1128.501291,2,18118.499732,2,4,2,828.501291,15030.626968,6077.254157,20430.389858,8395.84858,13003.431617,8538.99343,0.64963\n3112,37.5532,126.7449,-0.088214,-0.206277,0.171383,-0.086627,0.027718,-0.004241,-0.14989,-0.029005,-0.046682,-0.142189,0.147295,0.112944,0.247207,9.9e-05,2,0,0,2604.524502,2604.524502,3,16642.476521,3,5,2,1063.076278,13702.265974,6027.431275,19276.634793,7122.15201,11530.835576,7157.714714,0.865389\n3113,37.545,126.7387,-0.083675,-0.188496,0.14483,-0.068958,0.003924,0.004085,-0.19006,-0.053525,-0.054028,-0.077006,-0.067358,-0.099604,0.259459,0.000104,2,0,0,3667.60078,3667.60078,4,15579.400243,4,6,2,973.702274,12662.240466,5511.320747,18213.843514,6866.213497,10671.705063,6142.896769,0.945952\n3114,37.543336,126.727857,-0.078405,-0.168082,0.114946,-0.049323,-0.020123,0.01208,0.0117,0.243224,0.018472,0.043264,0.118717,0.089776,0.272793,0.00011,2,0,0,4641.303054,4641.303054,5,14605.697969,5,7,2,717.593254,12212.998134,4592.481029,17560.943278,7510.486696,10645.500704,5829.432024,0.831983\n3115,37.538408,126.722603,-0.072451,-0.145321,0.082419,-0.028281,-0.042871,0.0191,-0.092177,0.045832,-0.043902,-0.13244,-0.269102,0.117812,0.286126,0.000114,2,0,0,5358.896308,5358.896308,6,13888.104714,6,6,2,717.593254,11570.587634,4279.92159,16855.236775,7612.176849,10221.894071,5272.925024,0.734952\n3116,37.530313,126.722531,-0.065864,-0.120529,0.047996,-0.006433,-0.062847,0.024578,-0.159897,0.000496,-0.065289,-0.178362,-0.22683,0.067256,0.299099,0.00012,2,0,0,6259.008271,4809.963978,5,12987.992752,7,5,1,900.111962,10693.812527,4640.991171,16075.04571,7219.22298,9358.178426,4373.31571,0.937342\n3117,37.516813,126.721518,-0.058703,-0.094054,0.012469,0.015598,-0.078762,0.028071,0.165543,-0.073011,-0.185108,0.020125,-0.00359,0.063365,0.311712,0.000131,2,0,0,7762.828477,3306.143771,4,11484.172546,8,4,3,958.309313,9219.853363,5466.799419,14760.18045,6859.708522,7962.835234,2878.569935,0.910794\n3118,37.508228,126.720554,-0.062497,-0.081159,-0.028591,0.045542,-0.109721,0.035883,-0.06757,0.089516,-0.031114,-0.075101,-0.170732,0.067162,0.481441,0.000139,3,0,1,8721.13779,2347.834458,3,10525.863233,9,3,5,48.873498,8279.926006,6109.521368,13927.49133,6824.469516,7109.99659,1939.671172,0.747977\n3119,37.498368,126.722289,-0.034913,-0.058209,-0.041088,0.016183,-0.08897,0.063056,0.008352,0.259751,-0.001324,0.045477,-0.059521,0.022352,0.388829,0.000138,2,0,0,9828.206986,1240.765262,2,9418.794037,10,4,5,940.765262,7290.737746,7077.867724,13148.894957,6699.490081,6052.319362,833.052241,0.891276\n3120,37.489985,126.723735,-0.018493,-0.049339,-0.057886,-0.005281,-0.082603,0.091723,0.132638,-0.062775,0.276447,-0.142904,-0.001832,-0.076941,0.405045,0.000139,2,1,1,10768.972249,300.0,1,8478.028774,11,5,2,683.677162,6478.823352,7928.315896,12524.237304,6738.49942,5170.761521,115.222749,0.860785\n3121,37.485422,126.718542,-0.001911,-0.039779,-0.073353,-0.026595,-0.070898,0.112984,-0.183312,-0.053781,-0.055435,-0.080616,0.00066,-0.113598,0.034234,0.00013,2,0,0,11452.649411,983.677162,2,8527.781274,12,6,3,683.677162,5830.374062,8146.814611,11840.561587,7316.834035,5002.575921,785.266759,0.938531\n3122,37.477909,126.710357,0.014687,-0.029665,-0.087133,-0.04715,-0.054611,0.125122,0.013032,0.266697,0.00793,0.062249,-0.002417,0.052996,0.025946,0.000124,2,0,0,12556.916947,2087.944698,3,7423.513739,13,7,2,1104.267536,4794.369761,8624.203792,10736.631792,8274.01806,4899.398959,1888.895124,0.886282\n3123,37.467155,126.707949,0.031157,-0.019135,-0.098909,-0.066361,-0.034794,0.127159,-0.089659,-0.007939,-0.010114,-0.114985,0.351101,0.234196,0.017658,0.000119,2,0,0,13771.437898,3302.465649,4,6208.992788,14,8,4,1185.422899,3597.018545,9692.564488,9731.143142,8965.053282,4424.848125,3005.988127,0.672234\n3124,37.457292,126.702085,0.057998,-0.010213,-0.132775,-0.102486,-0.01559,0.145658,0.103739,0.060858,-0.055652,0.005474,0.017379,-0.155231,0.357477,0.000123,3,1,1,14250.814764,3181.842515,5,4996.186259,15,7,5,61.626041,2388.75533,10627.055014,8608.870306,9961.876086,4572.952589,4217.733323,0.568908\n3125,37.449506,126.700976,0.073078,-0.003502,-0.087943,-0.072735,-0.014924,0.185975,0.129314,-0.030017,0.250387,-0.136578,0.017384,-0.13228,0.047568,0.000112,2,0,0,15122.038159,4053.06591,6,5867.409654,16,8,5,822.533945,1550.827769,11455.892518,7989.060962,10526.387474,4592.53181,5041.51932,0.811561\n3126,37.441946,126.699716,0.098163,0.001383,-0.065454,-0.059717,-0.016155,0.238005,0.108731,-0.041856,0.187225,-0.127935,0.014749,-0.156043,0.022703,0.000103,2,0,0,15970.036903,4901.064655,7,6715.408398,17,9,2,779.36917,795.835899,12263.669514,7421.82204,11125.179952,4782.353584,5860.550196,0.760189\n3127,37.435104,126.697796,0.122391,0.00625,-0.041459,-0.044996,-0.016341,0.27082,0.087824,-0.054618,0.222555,-0.060276,-0.049252,0.049267,0.001081,9.6e-05,2,0,0,16749.406073,5680.433824,8,7181.913437,18,10,2,779.36917,457.225391,12988.945322,6896.744996,11745.251747,5136.83933,6631.670245,0.57421\n3128,37.42724,126.698989,0.145551,0.011029,-0.01651,-0.028992,-0.015472,0.281768,-0.109711,0.005949,-0.049728,-0.151588,-0.248798,0.101206,0.004324,9.7e-05,2,0,0,16755.773477,5686.801228,9,6301.144972,19,11,2,880.768465,1199.577333,13869.468813,6638.729067,12264.919736,5382.951919,7425.473111,0.63129\n3129,37.418065,126.693931,0.16744,0.015654,0.008818,-0.012162,-0.013602,0.269967,0.158846,-0.053909,-0.169269,-0.001629,0.006078,-0.018043,0.029189,0.000108,2,0,0,15641.967379,4572.995131,8,5187.338874,18,10,2,791.152615,2106.498239,14827.43848,5920.229567,13306.548098,6290.910928,8533.319043,0.712496\n3130,37.412626,126.688156,0.187868,0.02006,0.033944,0.005016,-0.010854,0.236369,0.198187,-0.090349,-0.200491,0.015341,-0.011082,0.081327,0.258378,0.000117,2,1,1,14850.814764,3781.842515,7,4396.186259,17,9,2,791.152615,2746.668379,15386.995591,5312.858491,14096.40676,7055.325787,9271.456236,0.901245\n3131,37.40482,126.680271,0.206656,0.024186,0.058289,0.02205,-0.007404,0.183687,0.1613,-0.05352,-0.081776,-0.043895,0.015249,-0.094603,0.193153,0.000106,2,0,0,15963.60428,4894.632031,8,3283.396743,18,8,2,910.062523,3755.965569,16224.304993,4576.616322,15205.908732,8133.18123,10331.077842,0.854794\n3132,37.398623,126.673541,0.223639,0.027974,0.081293,0.038455,-0.003475,0.116175,0.161047,-0.063193,-0.168079,-0.004958,0.004436,-0.00543,0.171532,9.9e-05,2,0,0,16873.666803,5804.694554,9,2373.33422,19,7,1,910.062523,4606.971817,16912.333977,4073.842369,16115.337538,9027.23505,11196.171018,0.891264\n3133,37.388174,126.661893,0.238671,0.031372,0.102428,0.053764,0.000678,0.039282,0.075923,-0.023795,0.176143,-0.135807,0.121065,-0.044026,0.149189,8.8e-05,2,0,0,18425.763396,7356.791147,10,821.237627,20,6,1,821.237627,6092.793265,18118.65006,3585.27974,17667.056676,10561.323506,12680.608987,0.437621\n3134,37.382302,126.656254,0.251619,0.034331,0.121207,0.06754,0.004787,-0.040782,0.159716,-0.061942,0.112718,-0.111631,0.005565,-0.133068,0.126126,8.3e-05,2,0,0,19247.001023,8178.028774,11,0.0,21,5,2,821.237627,6902.206627,18810.880533,3640.235645,18484.829061,11365.195131,13485.690046,0.849329\n3135,37.377471,126.645923,0.26237,0.036811,0.137198,0.079391,0.008587,-0.117553,0.186678,-0.061912,-0.169915,-0.0049,0.004402,-0.025027,0.102342,7.7e-05,2,0,0,20306.204783,9237.232534,12,0.0,22,4,2,1059.20376,7826.424551,19451.553631,3576.097421,19500.025953,12407.191845,14383.52738,0.900254\n3136,37.386116,126.639447,0.270831,0.038776,0.150034,0.088978,0.011832,-0.184834,0.132638,-0.062775,0.276447,-0.142904,-0.001832,-0.076941,0.077838,7.1e-05,2,0,0,21424.886262,10355.914013,13,0.0,23,3,2,928.273205,7385.943938,18592.313876,2464.288294,19224.734919,12270.040331,13847.135497,0.824604\n3137,37.393507,126.634561,0.276928,0.0402,0.15942,0.096027,0.014312,-0.237191,0.139286,-0.04214,0.17066,-0.143526,0.044034,-0.200378,0.052613,6.7e-05,2,0,0,22353.159467,11284.187218,14,0.0,24,2,2,788.705198,7085.838562,17868.48136,1540.820082,18994.760589,12183.540493,13411.587164,0.812649\n3138,37.399787,126.630412,0.280608,0.041062,0.165139,0.100339,0.015868,-0.270397,0.149899,-0.04143,-0.118474,-0.012489,0.006597,-0.054419,0.026667,6.4e-05,2,0,0,23141.864664,12072.892416,15,0.0,25,1,2,788.705198,6918.799717,17268.688163,765.322241,18832.892101,12165.094612,13081.719041,0.818337\n3139,37.406805,126.626106,0.19929,0.02924,0.118129,0.071977,0.011595,-0.199243,-0.149164,-0.040258,-0.04314,-0.085525,0.042494,-0.071908,0.0,6e-05,1,0,0,24010.007914,12941.035665,16,0.0,26,0,1,868.143249,6810.139933,16609.154415,214.129518,18665.867364,12177.827914,12740.779799,0.815064\n3201,37.594986,126.62793,-0.103727,0.198964,0.106437,-0.019733,0.136476,0.078342,0.205395,-0.083034,-0.22024,0.003153,0.002222,0.0192,0.0,5.6e-05,1,0,0,7520.998065,7520.998065,7,26767.999088,9,0,1,1273.021613,18486.464135,6530.575399,20958.80288,18022.999636,19561.401743,14333.198122,0.928555\n3202,37.595178,126.642377,-0.146052,0.279413,0.148794,-0.027509,0.18677,0.10632,0.012037,-0.072918,-0.041567,0.13932,0.003072,0.081085,0.026667,6e-05,2,0,0,6247.976453,6247.976453,6,25494.977475,8,1,2,1273.021613,18150.736843,5792.801707,21015.664734,16992.079635,18890.522911,13637.341146,0.563675\n3203,37.601821,126.656734,-0.144137,0.273545,0.143641,-0.026327,0.168463,0.093263,-0.107757,-0.03875,-0.022371,-0.007132,-0.006834,-0.111089,0.052613,6.5e-05,2,0,0,4783.204141,4783.204141,5,24030.205164,7,2,2,1040.043173,18607.337847,5947.932055,21861.069084,16462.975709,18925.928942,13675.506567,0.662743\n3204,37.597566,126.667247,-0.140963,0.263855,0.135184,-0.024394,0.139268,0.072677,-0.133112,-0.053919,-0.021801,0.005599,0.129686,-0.114034,0.077838,7e-05,2,0,0,3743.160968,3743.160968,4,22990.161991,6,3,2,719.901372,18000.289334,5276.083966,21518.7937,15455.145583,18097.789503,12862.738619,0.889128\n3205,37.592872,126.672874,-0.136559,0.250481,0.123619,-0.021766,0.101073,0.046222,-0.115103,-0.055254,-0.014199,0.048963,0.068569,-0.175457,0.102342,7.3e-05,2,0,0,3023.259596,3023.259596,3,22270.260619,5,4,2,719.901372,17424.694599,4700.848538,21090.158577,14737.71667,17417.869458,12191.608647,0.803417\n3206,37.584863,126.675958,-0.130963,0.233607,0.10921,-0.018517,0.056346,0.016036,-0.090084,-0.007943,-0.009705,-0.114639,0.353445,0.226847,0.126126,7.8e-05,2,0,0,2092.164061,2092.164061,2,21339.165084,4,5,1,931.095535,16511.844872,3799.218015,20270.217294,13951.6566,16494.088034,11263.970947,0.771341\n3207,37.568749,126.675571,-0.124224,0.213471,0.09229,-0.01474,0.007977,-0.015446,-0.080024,-0.090871,0.059448,0.284999,-0.010761,-0.003047,0.187027,8.8e-05,2,1,1,300.0,300.0,1,19547.001023,3,5,1,851.382365,14732.127836,2009.005394,18513.128167,12911.796972,14895.632605,9642.540302,0.893144\n3208,37.561248,126.67751,-0.116401,0.190352,0.073247,-0.010543,-0.040907,-0.04568,-0.038845,0.03816,0.035616,-0.114678,0.358695,0.238848,0.056577,8.8e-05,2,0,0,1151.382365,1151.382365,2,18961.214862,4,6,2,851.382365,13884.929814,1175.647424,17742.298619,12324.925934,14078.039173,8821.760679,0.598223\n3209,37.551588,126.6771,-0.107561,0.164575,0.052519,-0.006045,-0.087147,-0.072226,0.105973,-0.067712,0.043487,0.009585,-0.057079,0.137286,0.064865,9.1e-05,2,0,0,2226.143434,2226.143434,3,17886.453792,5,5,2,846.609224,12820.5371,108.160878,16693.339826,11847.175555,13160.040737,7905.4461,0.435996\n3210,37.543978,126.676816,-0.097782,0.136499,0.030584,-0.001375,-0.127756,-0.09294,0.0627,0.180199,0.034118,0.027615,0.023012,-0.128895,0.077477,9.5e-05,2,0,0,3072.752658,3072.752658,4,17039.844568,6,4,1,846.609224,11982.885025,747.728405,15870.290843,11524.432148,12452.678078,7212.469475,0.710937\n3211,37.524719,126.675321,-0.08715,0.106516,0.007946,0.003334,-0.160108,-0.106151,-0.037109,-0.101023,0.100901,0.281769,-0.055106,0.149096,0.09045,0.000105,2,0,0,5218.230983,5218.230983,5,14894.366243,7,3,1,813.336946,9880.711265,2891.110493,13779.605824,11027.294412,10796.256768,5700.460472,0.736169\n3212,37.517498,126.67679,-0.075757,0.075046,-0.014876,0.007949,-0.182112,-0.110791,-0.012365,0.235813,-0.007853,0.013159,-0.121612,0.059812,0.103784,0.00011,2,0,0,6031.567929,6031.567929,6,14081.029297,8,2,3,813.336946,9067.374408,3691.89877,13057.653205,10766.482914,10102.0841,5101.404997,0.918662\n3213,37.507284,126.676213,-0.07802,0.052084,-0.04575,0.015109,-0.235576,-0.130419,-0.144939,-0.008097,-0.055041,-0.137055,-0.152413,0.014574,0.137297,0.000117,3,0,1,7168.544615,5366.023837,6,12944.052611,9,1,3,88.904325,7957.654365,4827.8154,11971.833048,10728.745345,9322.609817,4588.090979,0.941623\n3214,37.499981,126.675823,-0.051906,0.038291,-0.056547,-0.006976,-0.207069,-0.117086,-0.12015,0.003915,-0.052873,-0.154874,-0.266365,0.089162,0.103063,0.000116,2,0,0,7981.294385,4553.274066,5,12131.302841,10,2,3,757.39144,7169.094787,5640.133859,11202.562697,10773.410038,8811.536403,4366.010849,0.798418\n3215,37.490459,126.675239,-0.039657,0.033521,-0.074439,-0.026089,-0.208408,-0.118233,0.148092,-0.059537,-0.168297,0.003848,0.000995,0.020547,0.107748,0.000116,2,0,0,9041.347001,3493.221451,4,11071.250225,11,3,2,1057.613647,6152.09808,6699.591606,10206.89797,10929.279518,8224.10695,4302.523925,0.794595\n3216,37.48395,126.683979,-0.027061,0.028282,-0.090619,-0.044459,-0.196278,-0.109834,0.115849,-0.069729,-0.142346,0.052891,-0.005767,0.103453,0.118198,0.000122,2,0,0,10098.960648,2435.607804,3,10013.636578,12,4,2,1057.613647,5283.649329,7450.750446,9907.340951,10305.843385,7184.047324,3617.083893,0.621346\n3217,37.47399,126.681159,-0.014229,0.022648,-0.104715,-0.061561,-0.171463,-0.092567,0.212356,-0.089047,-0.231874,0.011046,0.001118,0.038633,0.129009,0.000129,2,0,0,11234.026439,1300.542012,2,8878.570787,13,5,2,1000.542012,4248.231036,8539.203814,8827.985378,10846.389995,6892.089371,4226.932676,0.922975\n3218,37.465153,126.679028,-0.001273,0.016698,-0.116402,-0.076907,-0.135567,-0.067826,0.085809,-0.064858,0.239893,-0.026406,-0.072318,0.115246,0.473153,0.000139,2,1,1,11368.972249,300.0,1,7878.028774,13,6,3,766.40012,3371.171606,9514.835591,7901.732974,11369.367474,6751.636819,4899.92729,0.684604\n3219,37.45841,126.68083,0.011694,0.010515,-0.125412,-0.090061,-0.09091,-0.037609,0.032961,0.272548,0.012553,0.068146,0.000102,-0.046951,0.423784,0.000134,2,0,0,12135.372369,1066.40012,2,7111.628654,14,7,2,766.40012,2619.229358,10268.916384,7393.746235,11538.486459,6443.742203,5257.862839,0.904489\n3220,37.457681,126.692211,0.02456,0.004184,-0.131538,-0.100647,-0.040377,-0.004355,0.032634,0.331187,0.014303,0.080915,0.010391,-0.011679,0.415495,0.000129,2,0,0,13143.233685,2074.261436,3,6103.767338,15,7,5,872.590896,2301.311388,10434.479023,8002.192519,10689.161885,5438.808604,4647.102742,0.921653\n3221,37.456812,126.702435,0.045573,-0.0027,-0.164899,-0.132718,0.015634,0.035823,0.061848,0.212403,-0.006105,0.045801,0.005888,-0.130244,0.45045,0.000124,3,0,1,14050.814764,2981.842515,4,5196.186259,16,6,5,61.626041,2349.006956,10685.717701,8597.451294,9964.172125,4532.992209,4251.269088,0.716285\n3222,37.456734,126.710779,0.039229,-0.002412,-0.159324,-0.13613,0.090164,-0.030367,-0.158085,-0.037745,-0.051359,-0.100009,-0.034196,-0.078207,0.126126,0.000115,2,0,0,14787.364719,3718.39247,5,5932.736213,17,5,5,736.549955,2687.602974,10874.868101,9164.123787,9355.667434,3808.127982,3984.858589,0.940881\n3223,37.455858,126.71948,0.040905,-0.002586,-0.180344,-0.160015,0.161736,-0.087531,0.026096,0.321923,0.010191,0.080125,0.006772,0.012424,0.102342,0.000107,2,0,0,15561.538214,4492.565965,6,6706.909708,18,4,2,774.173495,3127.205448,11203.78635,9729.174421,8796.477702,3037.103624,3927.918834,0.940112\n3224,37.454968,126.731859,0.042225,-0.002724,-0.197217,-0.179337,0.222855,-0.137629,-0.149846,0.006855,-0.060932,-0.18019,-0.211778,0.108932,0.077838,9.7e-05,2,0,0,16658.733066,5589.760818,7,7804.104561,19,3,2,873.749635,3954.682941,11707.937799,10598.981767,8034.920654,1955.935611,4066.153366,0.888737\n3225,37.448165,126.736812,0.043175,-0.002825,-0.209554,-0.193547,0.269572,-0.176614,0.022345,0.294795,0.013157,0.078856,0.027792,0.026969,0.052613,9e-05,2,0,0,17532.482701,6463.510453,8,8677.854196,20,2,2,873.749635,4045.446421,12578.029188,10627.226725,8283.961809,1439.210755,4895.442129,0.893802\n3226,37.448549,126.752815,0.043749,-0.002885,-0.217071,-0.202238,0.298867,-0.20134,0.029816,0.262008,0.018278,0.06984,0.045262,-0.015608,0.026667,8e-05,2,0,0,18945.849676,7876.877428,9,10091.221171,21,1,2,1157.605338,5417.484373,13199.375002,11931.605536,7408.149302,128.980103,5361.710694,0.890747\n3227,37.439656,126.759632,0.031071,-0.002054,-0.155278,-0.145072,0.218387,-0.148358,-0.109949,-0.042281,-0.026662,-0.004903,0.007502,-0.14284,0.0,7.3e-05,1,0,0,20103.455014,9034.482765,10,11248.826509,22,0,1,1157.605338,5872.719684,14356.894057,12152.728956,7988.459192,1260.967121,6517.989618,0.880929\n3753,37.506241,126.811384,-0.056706,-0.054579,-0.0823,0.255216,0.144614,0.007355,-0.129985,-0.053886,-0.02458,0.003413,0.116837,-0.076685,0.026667,7.1e-05,1,1,1,16971.580338,300.0,1,18776.30578,18,0,1,1237.554065,12963.741912,12872.823876,19585.59353,1208.928148,8128.336661,7892.980762,0.832085\n3754,37.505646,126.797375,-0.079844,-0.076647,-0.115052,0.355785,0.197906,0.009981,0.113757,0.024118,-0.034238,-0.02698,0.022671,-0.172365,0.078559,7.7e-05,2,1,1,15734.026273,300.0,1,17538.751715,17,1,2,922.30742,11949.264171,11769.74178,18540.237128,101.332339,7350.752022,6674.754972,0.532978\n3755,37.503731,126.787201,-0.078797,-0.075038,-0.111067,0.340495,0.178507,0.008756,0.154657,-0.0848,-0.158818,0.046601,-0.015418,0.118899,0.102342,8.3e-05,2,0,0,14811.718853,1222.30742,2,16616.444295,16,2,2,922.30742,11130.117638,11063.639732,17695.504052,942.442452,6723.472728,5752.540207,0.842769\n3756,37.503039,126.776232,-0.077062,-0.07238,-0.104528,0.315498,0.147572,0.006823,0.029714,-0.076721,0.175715,0.125341,-0.099942,0.232299,0.126126,8.9e-05,2,0,0,13841.06072,2192.965553,3,15645.786162,15,3,2,970.658133,10373.958779,10260.713263,16890.345073,1912.570092,6275.341334,4798.478179,0.679124\n3757,37.5047,126.763,-0.074655,-0.068711,-0.095585,0.281505,0.1071,0.004339,0.056047,-0.087487,0.201104,0.103264,-0.11262,0.253606,0.149189,9.8e-05,2,0,0,12659.266433,3374.75984,4,14463.991875,14,4,2,875.879682,9726.976762,9176.801776,16130.143214,3069.771773,6179.056523,3762.723588,0.685675\n3758,37.505836,126.753174,-0.071595,-0.064082,-0.084444,0.239485,0.059706,0.001505,0.096051,-0.077067,0.240509,-0.015898,-0.093821,0.174676,0.171532,0.000105,2,0,0,11783.386751,4250.639522,5,13588.112193,13,5,2,875.879682,9305.887264,8395.510296,15592.033924,3938.14261,6242.001754,3057.011713,0.763228\n3759,37.506496,126.742348,-0.067911,-0.058558,-0.071361,0.190638,0.008453,-0.00145,-0.15412,-0.093143,-0.000173,0.164469,0.064573,-0.186203,0.193153,0.000115,2,0,0,10825.539385,4452.236053,6,12630.264827,12,4,2,957.847366,8861.592353,7600.950876,14987.258417,4895.109719,6385.00035,2364.067218,0.938299\n3760,37.507023,126.731337,-0.063634,-0.052217,-0.056637,0.136355,-0.043346,-0.004288,-0.119591,-0.089873,0.027766,0.193956,0.050111,-0.105814,0.214054,0.000125,2,0,0,9852.551958,3479.248626,5,11657.2774,11,3,5,931.414168,8482.418773,6847.346646,14399.113214,5867.900098,6654.460719,1895.561115,0.895181\n3761,37.507847,126.720829,-0.072017,-0.055292,-0.049736,0.095755,-0.113097,-0.008304,-0.152012,-0.074049,-0.02328,0.061927,0.07067,-0.200049,0.263784,0.000137,3,0,1,8921.13779,2547.834458,4,10725.863233,10,2,4,48.873498,8246.497942,6157.71586,13907.280393,6798.01656,7061.393298,1894.045392,0.934557\n3762,37.508697,126.703693,-0.060972,-0.016008,-0.040447,0.057668,-0.135146,-0.04453,-0.08522,-0.1001,0.045449,0.289745,-0.00016,-0.014026,0.040721,0.000121,2,0,0,9871.039912,4062.338687,5,12240.367462,11,1,1,1488.196864,8025.980291,5246.047756,13160.041518,8312.413878,7871.923967,2661.16552,0.924695\n3763,37.506785,126.675425,-0.062611,0.013352,-0.039353,0.035509,-0.169215,-0.078685,-0.040871,-0.094759,0.112885,0.27053,-0.058254,0.137214,0.04036,0.000116,2,0,0,7368.544615,5566.023837,6,13144.052611,10,0,3,88.904325,7916.690558,4884.241524,11895.558402,10796.97881,9335.489226,4630.747156,0.807931\n", "weather_incheon_hourly.csv": "date,hour,temperature_2m,precipitation,snowfall,wind_speed_10m\n2024-12-01,0,0.1,0.0,0.0,5.6\n2024-12-01,1,0.3,0.0,0.0,7.6\n2024-12-01,2,0.4,0.0,0.0,7.5\n2024-12-01,3,0.6,0.0,0.0,7.6\n2024-12-01,4,0.3,0.0,0.0,7.6\n2024-12-01,5,-0.3,0.0,0.0,7.2\n2024-12-01,6,-0.1,0.0,0.0,9.0\n2024-12-01,7,0.2,0.0,0.0,8.6\n2024-12-01,8,0.4,0.0,0.0,8.5\n2024-12-01,9,1.1,0.0,0.0,7.9\n2024-12-01,10,1.1,0.0,0.0,6.7\n2024-12-01,11,2.1,0.0,0.0,5.0\n2024-12-01,12,3.0,0.0,0.0,4.1\n2024-12-01,13,4.0,0.0,0.0,4.5\n2024-12-01,14,4.6,0.2,0.0,5.1\n2024-12-01,15,4.8,0.0,0.0,6.4\n2024-12-01,16,4.9,0.0,0.0,7.2\n2024-12-01,17,4.3,0.3,0.0,7.4\n2024-12-01,18,5.1,0.2,0.0,12.0\n2024-12-01,19,4.3,0.0,0.0,12.9\n2024-12-01,20,4.9,0.0,0.0,13.1\n2024-12-01,21,5.1,0.0,0.0,13.2\n2024-12-01,22,4.8,0.0,0.0,12.6\n2024-12-01,23,5.4,0.0,0.0,14.7\n2024-12-02,0,5.4,0.0,0.0,15.4\n2024-12-02,1,5.2,0.0,0.0,15.2\n2024-12-02,2,4.8,0.0,0.0,15.1\n2024-12-02,3,4.7,0.0,0.0,15.1\n2024-12-02,4,4.7,0.0,0.0,15.5\n2024-12-02,5,5.0,0.0,0.0,15.6\n2024-12-02,6,5.2,0.0,0.0,15.6\n2024-12-02,7,5.6,0.0,0.0,16.2\n2024-12-02,8,6.3,0.0,0.0,14.6\n2024-12-02,9,8.4,0.0,0.0,14.3\n2024-12-02,10,8.7,1.1,0.0,14.2\n2024-12-02,11,10.5,0.5,0.0,10.7\n2024-12-02,12,12.8,0.0,0.0,12.2\n2024-12-02,13,12.6,0.0,0.0,16.1\n2024-12-02,14,11.6,0.0,0.0,17.2\n2024-12-02,15,10.7,0.0,0.0,17.1\n2024-12-02,16,10.1,0.0,0.0,15.5\n2024-12-02,17,8.1,0.0,0.0,15.9\n2024-12-02,18,5.7,0.0,0.0,16.7\n2024-12-02,19,3.8,0.0,0.0,16.2\n2024-12-02,20,2.0,0.0,0.0,16.0\n2024-12-02,21,0.8,0.0,0.0,14.6\n2024-12-02,22,1.2,0.0,0.0,15.0\n2024-12-02,23,0.2,0.0,0.0,14.4\n2024-12-03,0,-0.6,0.0,0.0,14.4\n2024-12-03,1,-1.5,0.0,0.0,12.7\n2024-12-03,2,-2.4,0.0,0.0,11.4\n2024-12-03,3,-3.1,0.0,0.0,11.2\n2024-12-03,4,-3.5,0.0,0.0,12.3\n2024-12-03,5,-3.7,0.0,0.0,12.4\n2024-12-03,6,-3.8,0.0,0.0,11.2\n2024-12-03,7,-4.0,0.0,0.0,10.8\n2024-12-03,8,-3.7,0.0,0.0,11.0\n2024-12-03,9,-3.0,0.0,0.0,10.9\n2024-12-03,10,-1.8,0.0,0.0,10.4\n2024-12-03,11,-0.3,0.0,0.0,9.3\n2024-12-03,12,1.3,0.0,0.0,9.7\n2024-12-03,13,2.3,0.0,0.0,11.1\n2024-12-03,14,2.8,0.0,0.0,11.6\n2024-12-03,15,2.7,0.0,0.0,11.9\n2024-12-03,16,2.3,0.0,0.0,10.1\n2024-12-03,17,1.2,0.0,0.0,7.6\n2024-12-03,18,0.4,0.0,0.0,1.6\n2024-12-03,19,1.3,0.0,0.0,4.5\n2024-12-03,20,1.4,0.0,0.0,5.4\n2024-12-03,21,1.5,0.0,0.0,5.3\n2024-12-03,22,0.7,0.0,0.0,2.2\n2024-12-03,23,0.5,0.0,0.0,5.4\n2024-12-04,0,0.8,0.0,0.0,6.4\n2024-12-04,1,-0.1,0.0,0.0,8.6\n2024-12-04,2,-1.0,0.0,0.0,7.7\n2024-12-04,3,-1.5,0.0,0.0,5.0\n2024-12-04,4,-2.0,0.0,0.0,2.6\n2024-12-04,5,-1.8,0.0,0.0,4.4\n2024-12-04,6,-1.3,0.0,0.0,5.4\n2024-12-04,7,-1.6,0.0,0.0,5.4\n2024-12-04,8,-1.6,0.0,0.0,5.8\n2024-12-04,9,-0.3,0.0,0.0,6.1\n2024-12-04,10,1.4,0.0,0.0,1.6\n2024-12-04,11,2.8,0.0,0.0,5.6\n2024-12-04,12,3.7,0.0,0.0,8.9\n2024-12-04,13,4.3,0.0,0.0,8.3\n2024-12-04,14,4.9,0.0,0.0,7.8\n2024-12-04,15,5.0,0.0,0.0,8.1\n2024-12-04,16,4.6,0.0,0.0,6.8\n2024-12-04,17,2.7,0.0,0.0,3.4\n2024-12-04,18,1.7,0.0,0.0,3.6\n2024-12-04,19,1.2,0.0,0.0,2.4\n2024-12-04,20,0.5,0.0,0.0,2.2\n2024-12-04,21,-0.5,0.0,0.0,4.7\n2024-12-04,22,-0.4,0.0,0.0,10.4\n2024-12-04,23,-0.5,0.0,0.0,10.1\n2024-12-05,0,-1.0,0.0,0.0,10.4\n2024-12-05,1,-1.3,0.0,0.0,7.4\n2024-12-05,2,-1.7,0.0,0.0,4.1\n2024-12-05,3,-1.6,0.0,0.0,0.9\n2024-12-05,4,-2.0,0.0,0.0,2.6\n2024-12-05,5,-2.0,0.0,0.0,2.3\n2024-12-05,6,-1.5,0.0,0.0,3.0\n2024-12-05,7,-1.3,0.0,0.0,7.2\n2024-12-05,8,-1.0,0.0,0.0,2.9\n2024-12-05,9,0.9,0.0,0.0,4.4\n2024-12-05,10,2.5,0.0,0.0,0.8\n2024-12-05,11,4.0,0.0,0.0,1.9\n2024-12-05,12,5.3,0.0,0.0,12.1\n2024-12-05,13,6.8,0.0,0.0,17.9\n2024-12-05,14,7.6,0.0,0.0,15.6\n2024-12-05,15,7.0,0.0,0.0,22.5\n2024-12-05,16,6.5,0.0,0.0,16.4\n2024-12-05,17,5.7,0.0,0.0,16.3\n2024-12-05,18,5.7,0.0,0.0,16.5\n2024-12-05,19,5.2,0.0,0.0,14.6\n2024-12-05,20,3.5,0.0,0.0,13.8\n2024-12-05,21,2.1,0.0,0.0,9.5\n2024-12-05,22,2.7,0.0,0.0,3.5\n2024-12-05,23,2.2,0.0,0.0,3.6\n2024-12-06,0,2.2,0.0,0.0,9.0\n2024-12-06,1,2.3,0.0,0.0,18.9\n2024-12-06,2,0.8,0.0,0.0,11.7\n2024-12-06,3,-0.2,0.0,0.0,9.4\n2024-12-06,4,-0.3,0.0,0.0,12.0\n2024-12-06,5,-1.1,0.0,0.0,12.4\n2024-12-06,6,-1.8,0.0,0.0,9.6\n2024-12-06,7,-2.2,0.0,0.0,7.8\n2024-12-06,8,-2.2,0.0,0.0,7.6\n2024-12-06,9,-1.0,0.0,0.0,8.2\n2024-12-06,10,1.8,0.0,0.0,8.6\n2024-12-06,11,2.9,0.0,0.0,10.3\n2024-12-06,12,3.7,0.0,0.0,12.9\n2024-12-06,13,4.1,0.0,0.0,15.9\n2024-12-06,14,4.0,0.0,0.0,18.0\n2024-12-06,15,3.8,0.0,0.0,20.8\n2024-12-06,16,3.6,0.0,0.0,16.8\n2024-12-06,17,2.7,0.0,0.0,15.1\n2024-12-06,18,1.4,0.0,0.0,14.3\n2024-12-06,19,0.5,0.0,0.0,13.4\n2024-12-06,20,-0.2,0.0,0.0,10.8\n2024-12-06,21,-1.0,0.0,0.0,10.3\n2024-12-06,22,-1.0,0.0,0.0,9.0\n2024-12-06,23,-1.6,0.0,0.0,10.0\n2024-12-07,0,-1.9,0.0,0.0,8.5\n2024-12-07,1,-2.2,0.0,0.0,8.2\n2024-12-07,2,-2.5,0.0,0.0,6.3\n2024-12-07,3,-2.6,0.0,0.0,4.6\n2024-12-07,4,-2.6,0.0,0.0,6.5\n2024-12-07,5,-2.8,0.0,0.0,6.2\n2024-12-07,6,-2.8,0.0,0.0,5.0\n2024-12-07,7,-2.8,0.0,0.0,4.5\n2024-12-07,8,-2.6,0.0,0.0,6.1\n2024-12-07,9,-1.7,0.0,0.0,8.1\n2024-12-07,10,0.1,0.0,0.0,8.8\n2024-12-07,11,1.3,0.0,0.0,12.0\n2024-12-07,12,1.9,0.0,0.0,13.4\n2024-12-07,13,2.2,0.0,0.0,15.5\n2024-12-07,14,2.1,0.0,0.0,15.1\n2024-12-07,15,2.0,0.0,0.0,16.5\n2024-12-07,16,1.4,0.0,0.0,13.9\n2024-12-07,17,0.1,0.0,0.0,11.3\n2024-12-07,18,-1.1,0.0,0.0,10.3\n2024-12-07,19,-1.7,0.0,0.0,11.5\n2024-12-07,20,-2.2,0.0,0.0,10.2\n2024-12-07,21,-2.5,0.0,0.0,11.4\n2024-12-07,22,-2.4,0.0,0.0,13.6\n2024-12-07,23,-2.7,0.0,0.0,12.9\n2024-12-08,0,-2.8,0.0,0.0,13.9\n2024-12-08,1,-3.0,0.0,0.0,14.4\n2024-12-08,2,-3.1,0.0,0.0,14.9\n2024-12-08,3,-3.3,0.0,0.0,15.1\n2024-12-08,4,-3.5,0.0,0.0,14.8\n2024-12-08,5,-3.7,0.0,0.0,14.0\n2024-12-08,6,-3.9,0.0,0.0,14.1\n2024-12-08,7,-3.9,0.0,0.0,13.4\n2024-12-08,8,-3.9,0.0,0.0,12.9\n2024-12-08,9,-2.8,0.0,0.0,12.9\n2024-12-08,10,-1.8,0.0,0.0,12.7\n2024-12-08,11,-0.3,0.0,0.0,13.4\n2024-12-08,12,1.1,0.0,0.0,12.0\n2024-12-08,13,2.0,0.0,0.0,10.3\n2024-12-08,14,2.7,0.0,0.0,10.1\n2024-12-08,15,2.8,0.0,0.0,10.4\n2024-12-08,16,2.4,0.0,0.0,9.0\n2024-12-08,17,0.4,0.0,0.0,5.1\n2024-12-08,18,-0.7,0.0,0.0,4.6\n2024-12-08,19,-0.6,0.0,0.0,6.1\n2024-12-08,20,-0.8,0.0,0.0,8.1\n2024-12-08,21,-1.4,0.0,0.0,8.7\n2024-12-08,22,-2.0,0.0,0.0,6.3\n2024-12-08,23,-2.4,0.0,0.0,8.2\n2024-12-09,0,-2.6,0.0,0.0,7.9\n2024-12-09,1,-2.7,0.0,0.0,6.4\n2024-12-09,2,-2.9,0.0,0.0,4.7\n2024-12-09,3,-3.1,0.0,0.0,3.8\n2024-12-09,4,-2.9,0.0,0.0,5.0\n2024-12-09,5,-3.4,0.0,0.0,3.4\n2024-12-09,6,-3.3,0.0,0.0,2.6\n2024-12-09,7,-3.3,0.0,0.0,1.8\n2024-12-09,8,-2.7,0.0,0.0,4.1\n2024-12-09,9,-0.9,0.0,0.0,5.2\n2024-12-09,10,-0.2,0.0,0.0,4.1\n2024-12-09,11,1.9,0.0,0.0,3.3\n2024-12-09,12,3.1,0.0,0.0,4.1\n2024-12-09,13,4.0,0.0,0.0,2.3\n2024-12-09,14,4.4,0.0,0.0,2.8\n2024-12-09,15,4.3,0.0,0.0,4.2\n2024-12-09,16,3.7,0.0,0.0,3.8\n2024-12-09,17,1.3,0.0,0.0,3.4\n2024-12-09,18,0.4,0.0,0.0,4.0\n2024-12-09,19,0.6,0.0,0.0,2.0\n2024-12-09,20,0.3,0.0,0.0,2.2\n2024-12-09,21,-0.1,0.0,0.0,3.4\n2024-12-09,22,-0.3,0.0,0.0,7.2\n2024-12-09,23,-1.0,0.0,0.0,7.9\n2024-12-10,0,-1.4,0.0,0.0,9.0\n2024-12-10,1,-1.9,0.0,0.0,6.8\n2024-12-10,2,-2.5,0.0,0.0,4.5\n2024-12-10,3,-2.7,0.0,0.0,4.2\n2024-12-10,4,-2.7,0.0,0.0,4.1\n2024-12-10,5,-2.4,0.0,0.0,4.3\n2024-12-10,6,-2.3,0.0,0.0,4.1\n2024-12-10,7,-2.1,0.0,0.0,4.7\n2024-12-10,8,-1.7,0.0,0.0,6.6\n2024-12-10,9,-0.4,0.0,0.0,6.2\n2024-12-10,10,1.0,0.0,0.0,3.7\n2024-12-10,11,2.9,0.0,0.0,3.1\n2024-12-10,12,4.3,0.0,0.0,5.3\n2024-12-10,13,5.1,0.0,0.0,8.2\n2024-12-10,14,5.5,0.0,0.0,9.2\n2024-12-10,15,5.4,0.0,0.0,10.3\n2024-12-10,16,4.6,0.0,0.0,6.9\n2024-12-10,17,2.3,0.0,0.0,1.3\n2024-12-10,18,0.9,0.0,0.0,5.3\n2024-12-10,19,1.5,0.0,0.0,7.7\n2024-12-10,20,1.3,0.0,0.0,7.4\n2024-12-10,21,1.3,0.0,0.0,9.8\n2024-12-10,22,1.3,0.0,0.0,10.8\n2024-12-10,23,0.7,0.0,0.0,12.1\n2024-12-11,0,0.3,0.0,0.0,11.4\n2024-12-11,1,-0.3,0.0,0.0,9.4\n2024-12-11,2,-0.5,0.0,0.0,6.9\n2024-12-11,3,-0.7,0.0,0.0,4.9\n2024-12-11,4,-0.6,0.0,0.0,5.5\n2024-12-11,5,-0.5,0.0,0.0,7.3\n2024-12-11,6,-0.7,0.0,0.0,7.0\n2024-12-11,7,-1.0,0.0,0.0,7.0\n2024-12-11,8,-1.0,0.0,0.0,7.2\n2024-12-11,9,-0.4,0.0,0.0,7.1\n2024-12-11,10,1.0,0.0,0.0,4.9\n2024-12-11,11,2.5,0.0,0.0,6.0\n2024-12-11,12,3.6,0.0,0.0,7.5\n2024-12-11,13,4.2,0.0,0.0,8.6\n2024-12-11,14,5.0,0.0,0.0,9.0\n2024-12-11,15,5.2,0.0,0.0,8.6\n2024-12-11,16,4.5,0.0,0.0,7.0\n2024-12-11,17,2.6,0.0,0.0,9.0\n2024-12-11,18,1.0,0.0,0.0,13.4\n2024-12-11,19,-0.4,0.0,0.0,13.2\n2024-12-11,20,-1.4,0.0,0.0,11.8\n2024-12-11,21,-2.1,0.0,0.0,11.2\n2024-12-11,22,-1.9,0.0,0.0,11.1\n2024-12-11,23,-2.4,0.0,0.0,11.2\n2024-12-12,0,-2.8,0.0,0.0,10.2\n2024-12-12,1,-3.1,0.0,0.0,9.0\n2024-12-12,2,-3.4,0.0,0.0,8.8\n2024-12-12,3,-3.7,0.0,0.0,8.8\n2024-12-12,4,-3.9,0.0,0.0,9.0\n2024-12-12,5,-4.0,0.0,0.0,8.6\n2024-12-12,6,-4.1,0.0,0.0,8.7\n2024-12-12,7,-4.1,0.0,0.0,8.2\n2024-12-12,8,-4.0,0.0,0.0,8.1\n2024-12-12,9,-3.0,0.0,0.0,7.7\n2024-12-12,10,-1.7,0.0,0.0,8.1\n2024-12-12,11,0.2,0.0,0.0,7.3\n2024-12-12,12,1.7,0.0,0.0,5.9\n2024-12-12,13,2.7,0.0,0.0,8.6\n2024-12-12,14,3.2,0.0,0.0,9.4\n2024-12-12,15,2.9,0.0,0.0,8.9\n2024-12-12,16,2.2,0.0,0.0,6.7\n2024-12-12,17,0.2,0.0,0.0,3.4\n2024-12-12,18,0.2,0.0,0.0,5.7\n2024-12-12,19,0.4,0.0,0.0,10.2\n2024-12-12,20,-0.2,0.0,0.0,8.7\n2024-12-12,21,-0.5,0.0,0.0,9.3\n2024-12-12,22,-0.5,0.0,0.0,9.3\n2024-12-12,23,-1.2,0.0,0.0,8.9\n2024-12-13,0,-1.1,0.0,0.0,9.8\n2024-12-13,1,-1.3,0.0,0.0,8.4\n2024-12-13,2,-1.5,0.0,0.0,5.1\n2024-12-13,3,-1.5,0.0,0.0,5.2\n2024-12-13,4,-1.5,0.0,0.0,5.0\n2024-12-13,5,-1.4,0.0,0.0,4.6\n2024-12-13,6,-1.4,0.0,0.0,3.7\n2024-12-13,7,-1.6,0.0,0.0,3.2\n2024-12-13,8,-1.2,0.0,0.0,2.5\n2024-12-13,9,-0.4,0.0,0.0,1.5\n2024-12-13,10,0.4,0.0,0.0,4.2\n2024-12-13,11,1.0,0.0,0.0,6.8\n2024-12-13,12,1.7,0.0,0.0,9.5\n2024-12-13,13,2.6,0.0,0.0,10.4\n2024-12-13,14,3.1,0.0,0.0,12.1\n2024-12-13,15,3.2,0.0,0.0,13.1\n2024-12-13,16,2.7,0.0,0.0,11.9\n2024-12-13,17,1.5,0.0,0.0,11.5\n2024-12-13,18,0.1,0.0,0.0,12.0\n2024-12-13,19,-0.5,0.0,0.0,13.9\n2024-12-13,20,-1.1,0.0,0.0,15.0\n2024-12-13,21,-1.8,0.0,0.0,14.1\n2024-12-13,22,-1.2,0.0,0.0,15.3\n2024-12-13,23,-1.7,0.0,0.0,14.3\n2024-12-14,0,-2.1,0.0,0.0,13.4\n2024-12-14,1,-2.5,0.0,0.0,13.0\n2024-12-14,2,-3.0,0.0,0.0,12.5\n2024-12-14,3,-3.3,0.0,0.0,11.9\n2024-12-14,4,-3.6,0.0,0.0,11.3\n2024-12-14,5,-3.5,0.0,0.0,12.2\n2024-12-14,6,-3.7,0.0,0.0,12.5\n2024-12-14,7,-4.0,0.0,0.0,12.1\n2024-12-14,8,-4.0,0.0,0.0,11.2\n2024-12-14,9,-2.9,0.0,0.0,12.1\n2024-12-14,10,-1.1,0.0,0.0,13.2\n2024-12-14,11,0.1,0.0,0.0,15.5\n2024-12-14,12,0.9,0.0,0.0,15.6\n2024-12-14,13,1.2,0.0,0.0,15.5\n2024-12-14,14,2.0,0.0,0.0,14.3\n2024-12-14,15,2.2,0.0,0.0,15.8\n2024-12-14,16,1.7,0.0,0.0,14.6\n2024-12-14,17,0.5,0.0,0.0,11.2\n2024-12-14,18,-0.8,0.0,0.0,10.2\n2024-12-14,19,-1.3,0.0,0.0,9.4\n2024-12-14,20,-1.4,0.0,0.0,9.6\n2024-12-14,21,-1.9,0.0,0.0,4.3\n2024-12-14,22,-1.9,0.0,0.0,4.5\n2024-12-14,23,-1.8,0.0,0.0,3.7\n2024-12-15,0,-0.9,0.0,0.0,4.5\n2024-12-15,1,-1.1,0.0,0.0,2.5\n2024-12-15,2,-1.7,0.0,0.0,3.0\n2024-12-15,3,-1.9,0.0,0.0,5.8\n2024-12-15,4,-2.2,0.0,0.0,8.5\n2024-12-15,5,-1.9,0.0,0.0,9.8\n2024-12-15,6,-0.8,0.0,0.0,4.0\n2024-12-15,7,0.5,0.0,0.0,11.9\n2024-12-15,8,-0.2,0.0,0.0,8.6\n2024-12-15,9,0.4,0.0,0.0,1.0\n2024-12-15,10,1.2,0.0,0.0,2.8\n2024-12-15,11,2.8,0.0,0.0,6.3\n2024-12-15,12,3.8,0.0,0.0,9.8\n2024-12-15,13,5.3,0.0,0.0,17.5\n2024-12-15,14,5.0,0.0,0.0,18.2\n2024-12-15,15,4.3,0.0,0.0,19.1\n2024-12-15,16,4.5,0.0,0.0,16.2\n2024-12-15,17,2.7,0.0,0.0,16.2\n2024-12-15,18,0.3,0.0,0.0,12.6\n2024-12-15,19,-0.7,0.0,0.0,14.6\n2024-12-15,20,-1.2,0.0,0.0,13.0\n2024-12-15,21,-1.7,0.0,0.0,11.8\n2024-12-15,22,-1.6,0.0,0.0,12.4\n2024-12-15,23,-2.3,0.0,0.0,8.3\n2024-12-16,0,-2.5,0.0,0.0,7.2\n2024-12-16,1,-2.4,0.0,0.0,5.5\n2024-12-16,2,-3.1,0.0,0.0,3.0\n2024-12-16,3,-2.8,0.0,0.0,1.8\n2024-12-16,4,-2.6,0.0,0.0,1.0\n2024-12-16,5,-2.4,0.0,0.0,0.7\n2024-12-16,6,-2.3,0.0,0.0,0.7\n2024-12-16,7,-2.3,0.0,0.0,1.0\n2024-12-16,8,-2.1,0.0,0.0,5.6\n2024-12-16,9,-0.5,0.0,0.0,7.6\n2024-12-16,10,0.7,0.0,0.0,3.4\n2024-12-16,11,1.0,0.0,0.0,4.2\n2024-12-16,12,1.8,0.0,0.0,3.2\n2024-12-16,13,2.8,0.0,0.0,5.7\n2024-12-16,14,4.1,0.0,0.0,4.6\n2024-12-16,15,4.3,0.0,0.0,2.9\n2024-12-16,16,3.9,0.0,0.0,1.8\n2024-12-16,17,1.6,0.0,0.0,1.8\n2024-12-16,18,1.6,0.0,0.0,4.4\n2024-12-16,19,1.5,0.0,0.0,7.0\n2024-12-16,20,-0.3,0.0,0.0,9.9\n2024-12-16,21,-0.9,0.0,0.0,15.1\n2024-12-16,22,-1.2,0.0,0.0,12.7\n2024-12-16,23,-2.0,0.0,0.0,11.6\n2024-12-17,0,-2.3,0.0,0.0,7.9\n2024-12-17,1,-2.5,0.0,0.0,6.0\n2024-12-17,2,-2.8,0.0,0.0,4.6\n2024-12-17,3,-2.6,0.0,0.0,6.8\n2024-12-17,4,-2.8,0.0,0.0,6.5\n2024-12-17,5,-2.9,0.0,0.0,5.9\n2024-12-17,6,-2.9,0.0,0.0,6.8\n2024-12-17,7,-2.8,0.0,0.0,7.3\n2024-12-17,8,-2.7,0.0,0.0,8.5\n2024-12-17,9,-2.0,0.0,0.0,7.8\n2024-12-17,10,-0.7,0.0,0.0,6.9\n2024-12-17,11,0.5,0.0,0.0,7.8\n2024-12-17,12,1.2,0.0,0.0,8.9\n2024-12-17,13,1.9,0.0,0.0,9.2\n2024-12-17,14,2.0,0.0,0.0,9.7\n2024-12-17,15,1.7,0.0,0.0,10.8\n2024-12-17,16,1.3,0.0,0.0,12.7\n2024-12-17,17,0.1,0.0,0.0,6.6\n2024-12-17,18,-0.2,0.0,0.0,14.5\n2024-12-17,19,-1.2,0.0,0.0,13.6\n2024-12-17,20,-2.6,0.0,0.0,15.8\n2024-12-17,21,-3.5,0.0,0.0,15.1\n2024-12-17,22,-3.0,0.0,0.0,14.2\n2024-12-17,23,-3.3,0.0,0.0,11.9\n2024-12-18,0,-3.6,0.0,0.0,11.6\n2024-12-18,1,-3.7,0.0,0.0,12.2\n2024-12-18,2,-4.1,0.0,0.0,12.1\n2024-12-18,3,-4.7,0.0,0.0,11.6\n2024-12-18,4,-5.1,0.0,0.0,11.0\n2024-12-18,5,-5.4,0.0,0.0,11.5\n2024-12-18,6,-5.9,0.0,0.0,11.5\n2024-12-18,7,-6.4,0.0,0.0,10.9\n2024-12-18,8,-6.7,0.0,0.0,10.3\n2024-12-18,9,-6.0,0.0,0.0,10.0\n2024-12-18,10,-4.6,0.0,0.0,10.1\n2024-12-18,11,-3.5,0.0,0.0,9.5\n2024-12-18,12,-2.4,0.0,0.0,9.2\n2024-12-18,13,-1.6,0.0,0.0,9.7\n2024-12-18,14,-1.1,0.0,0.0,10.2\n2024-12-18,15,-0.9,0.0,0.0,9.4\n2024-12-18,16,-1.2,0.0,0.0,10.0\n2024-12-18,17,-2.1,0.0,0.0,9.2\n2024-12-18,18,-3.0,0.0,0.0,10.6\n2024-12-18,19,-3.4,0.0,0.0,13.0\n2024-12-18,20,-3.9,0.0,0.0,13.8\n2024-12-18,21,-4.5,0.0,0.0,12.2\n2024-12-18,22,-4.8,0.0,0.0,11.0\n2024-12-18,23,-5.0,0.0,0.0,10.6\n2024-12-19,0,-5.3,0.0,0.0,10.1\n2024-12-19,1,-5.6,0.0,0.0,9.1\n2024-12-19,2,-5.9,0.0,0.0,7.7\n2024-12-19,3,-6.1,0.0,0.0,7.5\n2024-12-19,4,-6.3,0.0,0.0,7.5\n2024-12-19,5,-6.5,0.0,0.0,7.2\n2024-12-19,6,-6.7,0.0,0.0,7.1\n2024-12-19,7,-7.0,0.0,0.0,6.4\n2024-12-19,8,-7.2,0.0,0.0,5.4\n2024-12-19,9,-5.9,0.0,0.0,3.9\n2024-12-19,10,-4.7,0.0,0.0,2.3\n2024-12-19,11,-2.2,0.0,0.0,1.0\n2024-12-19,12,-0.5,0.0,0.0,2.6\n2024-12-19,13,0.7,0.0,0.0,2.1\n2024-12-19,14,1.3,0.0,0.0,1.9\n2024-12-19,15,1.5,0.0,0.0,1.9\n2024-12-19,16,1.0,0.0,0.0,5.4\n2024-12-19,17,-0.9,0.0,0.0,5.0\n2024-12-19,18,-2.2,0.0,0.0,2.3\n2024-12-19,19,-2.2,0.0,0.0,2.9\n2024-12-19,20,-2.6,0.0,0.0,1.8\n2024-12-19,21,-2.8,0.0,0.0,2.1\n2024-12-19,22,-2.5,0.0,0.0,2.6\n2024-12-19,23,-2.6,0.0,0.0,3.1\n2024-12-20,0,-2.6,0.0,0.0,2.5\n2024-12-20,1,-2.3,0.0,0.0,3.9\n2024-12-20,2,-2.5,0.0,0.0,3.7\n2024-12-20,3,-2.6,0.0,0.0,3.3\n2024-12-20,4,-2.5,0.0,0.0,4.9\n2024-12-20,5,-2.5,0.0,0.0,5.5\n2024-12-20,6,-2.2,0.0,0.0,6.5\n2024-12-20,7,-1.5,0.0,0.0,8.0\n2024-12-20,8,-1.0,0.0,0.0,6.6\n2024-12-20,9,-0.4,0.0,0.0,8.0\n2024-12-20,10,-0.1,0.0,0.0,8.2\n2024-12-20,11,2.0,0.0,0.0,6.9\n2024-12-20,12,2.5,0.0,0.0,5.7\n2024-12-20,13,3.3,0.0,0.0,4.6\n2024-12-20,14,3.4,0.0,0.0,6.3\n2024-12-20,15,4.0,0.0,0.0,5.6\n2024-12-20,16,3.5,0.0,0.0,4.8\n2024-12-20,17,2.1,0.0,0.0,5.6\n2024-12-20,18,0.5,0.0,0.0,8.5\n2024-12-20,19,-0.5,0.0,0.0,10.6\n2024-12-20,20,0.1,0.0,0.0,7.9\n2024-12-20,21,0.3,0.3,0.21,5.6\n2024-12-20,22,0.2,0.5,0.35,0.5\n2024-12-20,23,0.2,0.5,0.28,7.2\n2024-12-21,0,0.4,0.1,0.07,7.6\n2024-12-21,1,0.1,0.0,0.0,8.1\n2024-12-21,2,0.4,0.0,0.0,11.1\n2024-12-21,3,0.6,0.3,0.07,5.3\n2024-12-21,4,1.4,0.6,0.35,12.8\n2024-12-21,5,1.0,0.1,0.07,12.8\n2024-12-21,6,0.2,0.1,0.07,14.7\n2024-12-21,7,-0.4,0.1,0.07,13.3\n2024-12-21,8,-1.0,0.0,0.0,10.6\n2024-12-21,9,-0.8,0.0,0.0,13.9\n2024-12-21,10,-0.6,0.0,0.0,14.5\n2024-12-21,11,-0.3,0.0,0.0,15.9\n2024-12-21,12,-0.2,0.0,0.0,17.0\n2024-12-21,13,-0.4,0.0,0.0,19.0\n2024-12-21,14,-0.5,0.0,0.0,20.0\n2024-12-21,15,-0.8,0.0,0.0,20.2\n2024-12-21,16,-1.2,0.0,0.0,17.3\n2024-12-21,17,-2.4,0.0,0.0,16.8\n2024-12-21,18,-3.6,0.0,0.0,18.7\n2024-12-21,19,-4.2,0.0,0.0,18.1\n2024-12-21,20,-4.9,0.0,0.0,17.4\n2024-12-21,21,-5.1,0.0,0.0,17.7\n2024-12-21,22,-5.3,0.0,0.0,15.8\n2024-12-21,23,-5.8,0.0,0.0,17.1\n2024-12-22,0,-6.2,0.0,0.0,15.8\n2024-12-22,1,-6.7,0.0,0.0,15.8\n2024-12-22,2,-7.1,0.0,0.0,16.1\n2024-12-22,3,-7.4,0.0,0.0,16.0\n2024-12-22,4,-7.6,0.0,0.0,16.3\n2024-12-22,5,-7.8,0.0,0.0,17.2\n2024-12-22,6,-7.9,0.0,0.0,17.6\n2024-12-22,7,-7.9,0.0,0.0,17.0\n2024-12-22,8,-7.8,0.0,0.0,17.3\n2024-12-22,9,-6.9,0.0,0.0,16.8\n2024-12-22,10,-5.1,0.0,0.0,14.4\n2024-12-22,11,-4.2,0.0,0.0,13.6\n2024-12-22,12,-3.1,0.0,0.0,11.8\n2024-12-22,13,-2.2,0.0,0.0,10.4\n2024-12-22,14,-1.7,0.0,0.0,9.3\n2024-12-22,15,-1.6,0.0,0.0,9.1\n2024-12-22,16,-1.9,0.0,0.0,5.9\n2024-12-22,17,-4.2,0.0,0.0,4.6\n2024-12-22,18,-4.9,0.0,0.0,6.9\n2024-12-22,19,-3.9,0.0,0.0,8.0\n2024-12-22,20,-4.7,0.0,0.0,7.0\n2024-12-22,21,-5.4,0.0,0.0,6.8\n2024-12-22,22,-5.0,0.0,0.0,6.6\n2024-12-22,23,-4.6,0.0,0.0,7.2\n2024-12-23,0,-5.0,0.0,0.0,7.0\n2024-12-23,1,-6.2,0.0,0.0,5.6\n2024-12-23,2,-6.3,0.0,0.0,1.8\n2024-12-23,3,-6.3,0.0,0.0,1.5\n2024-12-23,4,-6.6,0.0,0.0,2.4\n2024-12-23,5,-6.0,0.0,0.0,2.2\n2024-12-23,6,-7.5,0.0,0.0,4.6\n2024-12-23,7,-4.8,0.0,0.0,6.4\n2024-12-23,8,-5.1,0.0,0.0,5.6\n2024-12-23,9,-3.8,0.0,0.0,5.2\n2024-12-23,10,-2.6,0.0,0.0,4.5\n2024-12-23,11,-0.1,0.0,0.0,2.7\n2024-12-23,12,1.5,0.0,0.0,0.7\n2024-12-23,13,2.2,0.0,0.0,1.1\n2024-12-23,14,2.7,0.0,0.0,3.8\n2024-12-23,15,3.0,0.0,0.0,7.0\n2024-12-23,16,2.3,0.0,0.0,7.2\n2024-12-23,17,-0.3,0.0,0.0,6.2\n2024-12-23,18,-1.5,0.0,0.0,6.9\n2024-12-23,19,0.7,0.0,0.0,12.3\n2024-12-23,20,-0.5,0.0,0.0,13.4\n2024-12-23,21,-2.4,0.0,0.0,12.3\n2024-12-23,22,-2.0,0.0,0.0,13.0\n2024-12-23,23,-2.6,0.0,0.0,13.0\n2024-12-24,0,-3.1,0.0,0.0,11.6\n2024-12-24,1,-3.5,0.0,0.0,10.9\n2024-12-24,2,-3.7,0.0,0.0,9.8\n2024-12-24,3,-3.7,0.0,0.0,9.1\n2024-12-24,4,-3.7,0.0,0.0,10.3\n2024-12-24,5,-4.0,0.0,0.0,10.9\n2024-12-24,6,-4.4,0.0,0.0,10.6\n2024-12-24,7,-4.8,0.0,0.0,9.9\n2024-12-24,8,-5.2,0.0,0.0,8.6\n2024-12-24,9,-4.4,0.0,0.0,7.9\n2024-12-24,10,-2.2,0.0,0.0,6.3\n2024-12-24,11,-0.9,0.0,0.0,6.2\n2024-12-24,12,0.3,0.0,0.0,5.9\n2024-12-24,13,1.1,0.0,0.0,5.4\n2024-12-24,14,1.4,0.0,0.0,6.1\n2024-12-24,15,1.3,0.0,0.0,4.7\n2024-12-24,16,0.9,0.0,0.0,4.9\n2024-12-24,17,-1.1,0.0,0.0,3.5\n2024-12-24,18,-2.2,0.0,0.0,1.8\n2024-12-24,19,-2.4,0.0,0.0,3.1\n2024-12-24,20,-2.4,0.0,0.0,1.3\n2024-12-24,21,-2.4,0.0,0.0,3.1\n2024-12-24,22,-1.7,0.0,0.0,4.5\n2024-12-24,23,-2.0,0.0,0.0,3.4\n2024-12-25,0,-2.1,0.0,0.0,2.7\n2024-12-25,1,-1.7,0.0,0.0,3.6\n2024-12-25,2,-1.7,0.0,0.0,3.8\n2024-12-25,3,-1.6,0.0,0.0,4.3\n2024-12-25,4,-1.6,0.0,0.0,4.5\n2024-12-25,5,-2.3,0.0,0.0,4.1\n2024-12-25,6,-3.0,0.0,0.0,3.8\n2024-12-25,7,-2.7,0.0,0.0,3.6\n2024-12-25,8,-3.4,0.0,0.0,2.8\n2024-12-25,9,-1.7,0.0,0.0,4.0\n2024-12-25,10,-0.6,0.0,0.0,1.7\n2024-12-25,11,1.3,0.0,0.0,0.5\n2024-12-25,12,3.1,0.0,0.0,3.9\n2024-12-25,13,3.9,0.0,0.0,6.5\n2024-12-25,14,4.6,0.0,0.0,8.0\n2024-12-25,15,4.7,0.0,0.0,9.7\n2024-12-25,16,4.2,0.0,0.0,10.1\n2024-12-25,17,2.8,0.0,0.0,9.6\n2024-12-25,18,3.7,0.0,0.0,14.1\n2024-12-25,19,4.5,0.0,0.0,15.8\n2024-12-25,20,4.5,0.0,0.0,13.2\n2024-12-25,21,4.9,0.0,0.0,15.9\n2024-12-25,22,4.5,0.0,0.0,12.4\n2024-12-25,23,4.6,0.0,0.0,19.2\n2024-12-26,0,3.6,0.0,0.0,14.8\n2024-12-26,1,3.7,0.0,0.0,16.5\n2024-12-26,2,4.0,0.0,0.0,18.6\n2024-12-26,3,3.6,0.0,0.0,17.4\n2024-12-26,4,2.8,0.0,0.0,15.2\n2024-12-26,5,1.6,0.0,0.0,17.4\n2024-12-26,6,-0.5,0.0,0.0,16.5\n2024-12-26,7,-2.0,0.0,0.0,14.8\n2024-12-26,8,-2.9,0.0,0.0,15.9\n2024-12-26,9,-2.3,0.0,0.0,14.8\n2024-12-26,10,0.2,0.0,0.0,18.0\n2024-12-26,11,-0.1,0.0,0.0,18.8\n2024-12-26,12,0.0,0.0,0.0,18.8\n2024-12-26,13,0.3,0.0,0.0,17.0\n2024-12-26,14,0.4,0.0,0.0,16.9\n2024-12-26,15,0.2,0.0,0.0,17.2\n2024-12-26,16,-0.5,0.0,0.0,17.1\n2024-12-26,17,-1.5,0.0,0.0,16.0\n2024-12-26,18,-2.6,0.0,0.0,17.7\n2024-12-26,19,-3.2,0.0,0.0,19.0\n2024-12-26,20,-4.0,0.0,0.0,19.4\n2024-12-26,21,-4.4,0.0,0.0,16.8\n2024-12-26,22,-4.0,0.0,0.0,17.7\n2024-12-26,23,-4.2,0.0,0.0,17.1\n2024-12-27,0,-4.5,0.0,0.0,16.3\n2024-12-27,1,-4.7,0.0,0.0,14.9\n2024-12-27,2,-4.8,0.0,0.0,13.6\n2024-12-27,3,-4.8,0.0,0.0,13.0\n2024-12-27,4,-4.8,0.0,0.0,12.1\n2024-12-27,5,-4.9,0.0,0.0,11.0\n2024-12-27,6,-5.0,0.0,0.0,10.1\n2024-12-27,7,-5.2,0.0,0.0,9.6\n2024-12-27,8,-5.3,0.0,0.0,9.5\n2024-12-27,9,-4.2,0.0,0.0,7.9\n2024-12-27,10,-3.2,0.0,0.0,7.7\n2024-12-27,11,-1.8,0.0,0.0,7.3\n2024-12-27,12,-0.6,0.0,0.0,8.9\n2024-12-27,13,0.2,0.0,0.0,10.9\n2024-12-27,14,0.6,0.0,0.0,12.2\n2024-12-27,15,0.3,0.0,0.0,11.4\n2024-12-27,16,-0.1,0.0,0.0,11.9\n2024-12-27,17,-0.7,0.0,0.0,9.7\n2024-12-27,18,-1.1,0.0,0.0,8.9\n2024-12-27,19,-1.2,0.0,0.0,9.8\n2024-12-27,20,-1.6,0.0,0.0,17.2\n2024-12-27,21,-2.8,0.0,0.0,14.7\n2024-12-27,22,-2.6,0.0,0.0,15.5\n2024-12-27,23,-2.8,0.0,0.0,15.4\n2024-12-28,0,-4.0,0.0,0.0,13.4\n2024-12-28,1,-4.5,0.0,0.0,13.1\n2024-12-28,2,-4.5,0.0,0.0,14.4\n2024-12-28,3,-4.9,0.0,0.0,15.2\n2024-12-28,4,-5.5,0.0,0.0,14.8\n2024-12-28,5,-5.8,0.0,0.0,13.2\n2024-12-28,6,-6.0,0.0,0.0,11.5\n2024-12-28,7,-6.2,0.0,0.0,10.5\n2024-12-28,8,-6.3,0.0,0.0,9.9\n2024-12-28,9,-5.2,0.0,0.0,8.2\n2024-12-28,10,-3.9,0.0,0.0,8.1\n2024-12-28,11,-2.5,0.0,0.0,8.6\n2024-12-28,12,-1.3,0.0,0.0,13.1\n2024-12-28,13,-0.5,0.0,0.0,15.8\n2024-12-28,14,-0.2,0.0,0.0,20.7\n2024-12-28,15,-0.1,0.0,0.0,21.3\n2024-12-28,16,-0.3,0.0,0.0,19.7\n2024-12-28,17,-0.6,0.0,0.0,16.6\n2024-12-28,18,-1.4,0.0,0.0,15.0\n2024-12-28,19,-2.0,0.0,0.0,11.7\n2024-12-28,20,-2.2,0.0,0.0,8.9\n2024-12-28,21,-2.6,0.0,0.0,5.1\n2024-12-28,22,-2.6,0.0,0.0,5.8\n2024-12-28,23,-2.4,0.0,0.0,2.8\n2024-12-29,0,-2.1,0.0,0.0,0.7\n2024-12-29,1,-2.1,0.0,0.0,2.4\n2024-12-29,2,-2.3,0.0,0.0,3.1\n2024-12-29,3,-3.0,0.0,0.0,2.3\n2024-12-29,4,-3.1,0.0,0.0,2.3\n2024-12-29,5,-2.7,0.0,0.0,1.5\n2024-12-29,6,-3.0,0.0,0.0,1.0\n2024-12-29,7,-3.2,0.0,0.0,1.0\n2024-12-29,8,-3.7,0.0,0.0,1.9\n2024-12-29,9,-2.2,0.0,0.0,3.1\n2024-12-29,10,-1.0,0.0,0.0,3.1\n2024-12-29,11,1.0,0.0,0.0,1.7\n2024-12-29,12,2.4,0.0,0.0,1.7\n2024-12-29,13,3.2,0.0,0.0,4.1\n2024-12-29,14,3.6,0.0,0.0,5.2\n2024-12-29,15,3.7,0.0,0.0,5.0\n2024-12-29,16,3.6,0.0,0.0,5.1\n2024-12-29,17,1.6,0.0,0.0,2.7\n2024-12-29,18,1.4,0.0,0.0,8.5\n2024-12-29,19,0.9,0.0,0.0,10.5\n2024-12-29,20,0.6,0.0,0.0,7.6\n2024-12-29,21,-0.2,0.0,0.0,4.4\n2024-12-29,22,0.2,0.0,0.0,6.5\n2024-12-29,23,0.5,0.0,0.0,10.2\n2024-12-30,0,0.9,0.0,0.0,11.2\n2024-12-30,1,0.6,0.0,0.0,12.6\n2024-12-30,2,1.0,0.0,0.0,12.0\n2024-12-30,3,2.5,0.0,0.0,9.0\n2024-12-30,4,4.1,0.0,0.0,11.5\n2024-12-30,5,4.8,0.1,0.0,13.7\n2024-12-30,6,5.2,0.0,0.0,14.8\n2024-12-30,7,5.2,0.0,0.0,13.7\n2024-12-30,8,5.7,0.0,0.0,13.8\n2024-12-30,9,6.4,0.0,0.0,14.8\n2024-12-30,10,6.4,0.0,0.0,10.9\n2024-12-30,11,8.4,0.0,0.0,20.9\n2024-12-30,12,8.6,0.0,0.0,23.4\n2024-12-30,13,9.0,0.0,0.0,24.1\n2024-12-30,14,8.6,0.0,0.0,20.5\n2024-12-30,15,8.5,0.0,0.0,20.7\n2024-12-30,16,7.5,0.0,0.0,15.6\n2024-12-30,17,5.5,0.0,0.0,13.0\n2024-12-30,18,4.5,0.0,0.0,13.9\n2024-12-30,19,3.9,0.0,0.0,13.3\n2024-12-30,20,3.3,0.0,0.0,12.1\n2024-12-30,21,3.2,0.0,0.0,12.4\n2024-12-30,22,4.0,0.0,0.0,12.6\n2024-12-30,23,4.0,0.0,0.0,15.8\n2024-12-31,0,3.5,0.2,0.0,17.4\n2024-12-31,1,1.8,0.0,0.0,18.8\n2024-12-31,2,-0.1,0.0,0.0,18.6\n2024-12-31,3,-1.2,0.0,0.0,17.2\n2024-12-31,4,-1.9,0.0,0.0,16.8\n2024-12-31,5,-2.4,0.0,0.0,16.7\n2024-12-31,6,-2.7,0.0,0.0,16.6\n2024-12-31,7,-2.9,0.0,0.0,15.8\n2024-12-31,8,-2.9,0.0,0.0,14.7\n2024-12-31,9,-2.0,0.0,0.0,12.1\n2024-12-31,10,0.2,0.0,0.0,16.8\n2024-12-31,11,1.0,0.0,0.0,16.4\n2024-12-31,12,1.5,0.0,0.0,14.3\n2024-12-31,13,2.0,0.0,0.0,12.5\n2024-12-31,14,2.3,0.0,0.0,12.5\n2024-12-31,15,2.2,0.0,0.0,13.3\n2024-12-31,16,1.7,0.0,0.0,13.6\n2024-12-31,17,0.7,0.0,0.0,11.8\n2024-12-31,18,-0.4,0.0,0.0,9.9\n2024-12-31,19,-0.6,0.0,0.0,9.2\n2024-12-31,20,-0.8,0.0,0.0,7.8\n2024-12-31,21,-1.2,0.0,0.0,8.1\n2024-12-31,22,-1.5,0.0,0.0,6.3\n2024-12-31,23,-2.4,0.0,0.0,4.1\n2025-01-01,0,-2.6,0.0,0.0,2.7\n2025-01-01,1,-2.9,0.0,0.0,1.4\n2025-01-01,2,-2.5,0.0,0.0,5.2\n2025-01-01,3,-2.2,0.0,0.0,7.4\n2025-01-01,4,-2.1,0.0,0.0,10.2\n2025-01-01,5,-2.3,0.0,0.0,11.5\n2025-01-01,6,-2.5,0.0,0.0,12.2\n2025-01-01,7,-1.7,0.0,0.0,12.8\n2025-01-01,8,-0.3,0.0,0.0,11.1\n2025-01-01,9,2.1,0.0,0.0,9.3\n2025-01-01,10,1.9,0.0,0.0,9.3\n2025-01-01,11,4.6,0.0,0.0,4.7\n2025-01-01,12,7.1,0.0,0.0,12.2\n2025-01-01,13,7.4,0.0,0.0,14.9\n2025-01-01,14,7.3,0.0,0.0,14.5\n2025-01-01,15,6.9,0.0,0.0,13.4\n2025-01-01,16,5.8,0.0,0.0,11.2\n2025-01-01,17,3.6,0.0,0.0,8.9\n2025-01-01,18,2.3,0.0,0.0,10.6\n2025-01-01,19,2.2,0.0,0.0,11.8\n2025-01-01,20,1.5,0.0,0.0,10.8\n2025-01-01,21,-0.3,0.0,0.0,9.8\n2025-01-01,22,-0.4,0.0,0.0,9.6\n2025-01-01,23,-1.3,0.0,0.0,10.2\n2025-01-02,0,-2.0,0.0,0.0,10.0\n2025-01-02,1,-2.5,0.0,0.0,8.3\n2025-01-02,2,-2.9,0.0,0.0,7.6\n2025-01-02,3,-3.1,0.0,0.0,6.7\n2025-01-02,4,-3.3,0.0,0.0,7.0\n2025-01-02,5,-3.4,0.0,0.0,7.1\n2025-01-02,6,-3.3,0.0,0.0,6.7\n2025-01-02,7,-3.3,0.0,0.0,6.9\n2025-01-02,8,-3.3,0.0,0.0,7.0\n2025-01-02,9,-2.0,0.0,0.0,6.6\n2025-01-02,10,-1.0,0.0,0.0,6.1\n2025-01-02,11,1.0,0.0,0.0,6.9\n2025-01-02,12,2.7,0.0,0.0,8.2\n2025-01-02,13,3.9,0.0,0.0,9.1\n2025-01-02,14,4.5,0.0,0.0,10.5\n2025-01-02,15,4.4,0.0,0.0,11.6\n2025-01-02,16,3.8,0.0,0.0,11.2\n2025-01-02,17,1.8,0.0,0.0,9.0\n2025-01-02,18,0.3,0.0,0.0,10.9\n2025-01-02,19,-0.4,0.0,0.0,8.9\n2025-01-02,20,-0.6,0.0,0.0,11.8\n2025-01-02,21,-1.6,0.0,0.0,13.0\n2025-01-02,22,-1.8,0.0,0.0,13.5\n2025-01-02,23,-2.5,0.0,0.0,13.7\n2025-01-03,0,-3.1,0.0,0.0,13.0\n2025-01-03,1,-3.6,0.0,0.0,11.2\n2025-01-03,2,-3.8,0.0,0.0,9.3\n2025-01-03,3,-3.9,0.0,0.0,9.1\n2025-01-03,4,-3.9,0.0,0.0,11.2\n2025-01-03,5,-4.2,0.0,0.0,10.5\n2025-01-03,6,-4.5,0.0,0.0,9.9\n2025-01-03,7,-4.7,0.0,0.0,9.7\n2025-01-03,8,-5.0,0.0,0.0,9.2\n2025-01-03,9,-4.0,0.0,0.0,5.5\n2025-01-03,10,-2.6,0.0,0.0,4.6\n2025-01-03,11,-0.9,0.0,0.0,4.2\n2025-01-03,12,0.4,0.0,0.0,2.4\n2025-01-03,13,1.0,0.0,0.0,4.6\n2025-01-03,14,1.3,0.0,0.0,9.7\n2025-01-03,15,1.1,0.0,0.0,11.1\n2025-01-03,16,1.2,0.0,0.0,10.1\n2025-01-03,17,0.2,0.0,0.0,5.0\n2025-01-03,18,-1.1,0.0,0.0,4.8\n2025-01-03,19,-0.2,0.0,0.0,10.0\n2025-01-03,20,-0.1,0.0,0.0,16.4\n2025-01-03,21,-1.1,0.0,0.0,15.9\n2025-01-03,22,-1.5,0.0,0.0,13.3\n2025-01-03,23,-2.7,0.0,0.0,11.6\n2025-01-04,0,-3.3,0.0,0.0,10.6\n2025-01-04,1,-3.6,0.0,0.0,9.8\n2025-01-04,2,-3.8,0.0,0.0,8.8\n2025-01-04,3,-3.9,0.0,0.0,5.5\n2025-01-04,4,-4.1,0.0,0.0,3.7\n2025-01-04,5,-4.3,0.0,0.0,3.1\n2025-01-04,6,-4.5,0.0,0.0,2.3\n2025-01-04,7,-4.5,0.0,0.0,1.5\n2025-01-04,8,-4.4,0.0,0.0,0.8\n2025-01-04,9,-2.9,0.0,0.0,1.2\n2025-01-04,10,-1.9,0.0,0.0,1.5\n2025-01-04,11,0.1,0.0,0.0,0.2\n2025-01-04,12,1.5,0.0,0.0,1.1\n2025-01-04,13,2.6,0.0,0.0,1.5\n2025-01-04,14,3.3,0.0,0.0,1.7\n2025-01-04,15,3.4,0.0,0.0,5.4\n2025-01-04,16,2.8,0.0,0.0,6.8\n2025-01-04,17,1.7,0.0,0.0,5.3\n2025-01-04,18,0.9,0.0,0.0,5.7\n2025-01-04,19,0.1,0.0,0.0,5.5\n2025-01-04,20,-0.6,0.0,0.0,6.5\n2025-01-04,21,0.2,0.0,0.0,8.9\n2025-01-04,22,-0.9,0.0,0.0,11.8\n2025-01-04,23,-0.9,0.0,0.0,12.2\n2025-01-05,0,-1.2,0.0,0.0,12.0\n2025-01-05,1,-1.1,0.0,0.0,13.2\n2025-01-05,2,-0.8,0.0,0.0,11.4\n2025-01-05,3,-0.3,0.1,0.07,11.3\n2025-01-05,4,-0.3,0.4,0.28,15.0\n2025-01-05,5,0.1,0.1,0.07,13.8\n2025-01-05,6,0.6,0.2,0.14,15.9\n2025-01-05,7,1.2,0.2,0.14,15.6\n2025-01-05,8,1.5,0.6,0.21,16.0\n2025-01-05,9,1.2,1.1,0.56,15.6\n2025-01-05,10,0.8,0.9,0.63,18.3\n2025-01-05,11,1.1,0.9,0.63,16.6\n2025-01-05,12,1.6,0.5,0.21,14.2\n2025-01-05,13,2.2,0.0,0.0,13.5\n2025-01-05,14,2.8,0.2,0.0,11.4\n2025-01-05,15,3.3,0.2,0.0,11.7\n2025-01-05,16,2.9,0.0,0.0,13.3\n2025-01-05,17,2.1,0.0,0.0,13.4\n2025-01-05,18,1.9,0.0,0.0,11.8\n2025-01-05,19,1.5,0.1,0.0,11.8\n2025-01-05,20,1.3,0.0,0.0,11.5\n2025-01-05,21,1.0,0.0,0.0,11.6\n2025-01-05,22,0.2,1.3,0.35,6.2\n2025-01-05,23,0.0,0.0,0.0,9.4\n2025-01-06,0,-0.3,0.0,0.0,9.6\n2025-01-06,1,-0.9,0.0,0.0,9.4\n2025-01-06,2,-1.4,0.0,0.0,8.6\n2025-01-06,3,-1.2,0.0,0.0,8.1\n2025-01-06,4,-0.7,0.6,0.07,5.2\n2025-01-06,5,-1.1,0.0,0.0,4.9\n2025-01-06,6,-2.7,0.2,0.07,4.0\n2025-01-06,7,-2.0,0.0,0.0,8.9\n2025-01-06,8,-2.0,0.0,0.0,6.6\n2025-01-06,9,-0.3,0.4,0.28,12.4\n2025-01-06,10,1.1,0.3,0.21,15.8\n2025-01-06,11,1.0,0.0,0.0,15.8\n2025-01-06,12,1.1,0.0,0.0,15.0\n2025-01-06,13,1.3,0.0,0.0,15.4\n2025-01-06,14,1.4,0.0,0.0,16.9\n2025-01-06,15,1.2,0.0,0.0,17.2\n2025-01-06,16,0.5,0.0,0.0,16.5\n2025-01-06,17,-0.8,0.0,0.0,16.5\n2025-01-06,18,-2.2,0.0,0.0,16.1\n2025-01-06,19,-3.0,0.0,0.0,16.0\n2025-01-06,20,-3.8,0.0,0.0,16.0\n2025-01-06,21,-4.4,0.0,0.0,14.5\n2025-01-06,22,-4.0,0.0,0.0,14.3\n2025-01-06,23,-4.4,0.0,0.0,13.2\n2025-01-07,0,-4.7,0.0,0.0,13.4\n2025-01-07,1,-5.6,0.0,0.0,12.4\n2025-01-07,2,-6.2,0.0,0.0,11.2\n2025-01-07,3,-6.7,0.0,0.0,10.1\n2025-01-07,4,-7.1,0.0,0.0,9.1\n2025-01-07,5,-8.6,0.0,0.0,6.8\n2025-01-07,6,-9.3,0.0,0.0,5.2\n2025-01-07,7,-9.2,0.0,0.0,3.7\n2025-01-07,8,-9.3,0.0,0.0,3.6\n2025-01-07,9,-8.3,0.0,0.0,4.8\n2025-01-07,10,-4.8,0.0,0.0,2.3\n2025-01-07,11,-3.0,0.0,0.0,2.6\n2025-01-07,12,-2.3,0.0,0.0,5.3\n2025-01-07,13,-1.2,0.0,0.0,16.4\n2025-01-07,14,-0.8,0.0,0.0,19.5\n2025-01-07,15,-0.8,0.0,0.0,16.2\n2025-01-07,16,-0.9,0.0,0.0,13.7\n2025-01-07,17,-1.5,0.0,0.0,10.3\n2025-01-07,18,-2.8,0.0,0.0,9.4\n2025-01-07,19,-4.1,0.0,0.0,7.7\n2025-01-07,20,-4.3,0.0,0.0,8.3\n2025-01-07,21,-5.5,0.0,0.0,6.5\n2025-01-07,22,-5.6,0.0,0.0,6.1\n2025-01-07,23,-5.3,0.0,0.0,7.1\n2025-01-08,0,-6.7,0.0,0.0,4.8\n2025-01-08,1,-6.8,0.0,0.0,0.6\n2025-01-08,2,-6.7,0.0,0.0,1.0\n2025-01-08,3,-6.5,0.0,0.0,2.2\n2025-01-08,4,-6.5,0.0,0.0,3.6\n2025-01-08,5,-6.6,0.0,0.0,7.8\n2025-01-08,6,-6.1,0.0,0.0,7.3\n2025-01-08,7,-7.2,0.0,0.0,4.2\n2025-01-08,8,-7.0,0.0,0.0,1.8\n2025-01-08,9,-5.6,0.0,0.0,0.8\n2025-01-08,10,-4.3,0.0,0.0,2.7\n2025-01-08,11,-3.1,0.0,0.0,4.0\n2025-01-08,12,-1.8,0.0,0.0,4.3\n2025-01-08,13,-1.2,0.0,0.0,7.6\n2025-01-08,14,-1.0,0.0,0.0,14.3\n2025-01-08,15,-1.4,0.0,0.0,14.3\n2025-01-08,16,-1.7,0.0,0.0,15.9\n2025-01-08,17,-2.5,0.0,0.0,13.9\n2025-01-08,18,-3.3,0.0,0.0,14.8\n2025-01-08,19,-5.1,0.0,0.0,16.7\n2025-01-08,20,-6.3,0.0,0.0,15.4\n2025-01-08,21,-6.8,0.0,0.0,13.9\n2025-01-08,22,-6.2,0.0,0.0,12.8\n2025-01-08,23,-6.6,0.0,0.0,12.8\n2025-01-09,0,-7.2,0.0,0.0,17.6\n2025-01-09,1,-8.6,0.0,0.0,19.9\n2025-01-09,2,-9.6,0.0,0.0,17.8\n2025-01-09,3,-10.2,0.0,0.0,12.9\n2025-01-09,4,-10.7,0.0,0.0,11.1\n2025-01-09,5,-10.9,0.0,0.0,12.9\n2025-01-09,6,-11.3,0.0,0.0,13.5\n2025-01-09,7,-11.8,0.0,0.0,13.2\n2025-01-09,8,-12.1,0.0,0.0,12.9\n2025-01-09,9,-11.7,0.0,0.0,14.3\n2025-01-09,10,-10.1,0.0,0.0,16.6\n2025-01-09,11,-9.7,0.0,0.0,20.3\n2025-01-09,12,-9.3,0.0,0.0,21.4\n2025-01-09,13,-9.2,0.0,0.0,22.8\n2025-01-09,14,-9.3,0.0,0.0,22.7\n2025-01-09,15,-9.3,0.0,0.0,23.3\n2025-01-09,16,-9.8,0.0,0.0,23.8\n2025-01-09,17,-10.5,0.0,0.0,22.7\n2025-01-09,18,-11.7,0.0,0.0,19.9\n2025-01-09,19,-12.0,0.0,0.0,18.4\n2025-01-09,20,-12.1,0.0,0.0,16.1\n2025-01-09,21,-12.2,0.0,0.0,15.0\n2025-01-09,22,-12.4,0.0,0.0,12.5\n2025-01-09,23,-12.3,0.0,0.0,11.7\n2025-01-10,0,-12.1,0.0,0.0,11.0\n2025-01-10,1,-12.0,0.0,0.0,10.2\n2025-01-10,2,-12.0,0.0,0.0,8.8\n2025-01-10,3,-12.1,0.0,0.0,8.3\n2025-01-10,4,-12.0,0.0,0.0,8.9\n2025-01-10,5,-12.1,0.0,0.0,8.1\n2025-01-10,6,-12.2,0.0,0.0,7.1\n2025-01-10,7,-12.8,0.0,0.0,5.4\n2025-01-10,8,-13.5,0.0,0.0,4.9\n2025-01-10,9,-11.5,0.0,0.0,3.3\n2025-01-10,10,-9.9,0.0,0.0,3.0\n2025-01-10,11,-8.1,0.0,0.0,3.1\n2025-01-10,12,-6.5,0.0,0.0,2.8\n2025-01-10,13,-5.0,0.0,0.0,3.1\n2025-01-10,14,-3.9,0.0,0.0,7.0\n2025-01-10,15,-2.8,0.0,0.0,7.3\n2025-01-10,16,-2.3,0.0,0.0,10.4\n2025-01-10,17,-2.7,0.0,0.0,12.3\n2025-01-10,18,-3.8,0.0,0.0,11.1\n2025-01-10,19,-4.0,0.0,0.0,10.5\n2025-01-10,20,-4.8,0.0,0.0,12.5\n2025-01-10,21,-6.1,0.0,0.0,12.3\n2025-01-10,22,-6.7,0.0,0.0,9.9\n2025-01-10,23,-7.2,0.0,0.0,7.6\n2025-01-11,0,-7.6,0.0,0.0,7.6\n2025-01-11,1,-8.0,0.0,0.0,7.1\n2025-01-11,2,-8.2,0.0,0.0,6.5\n2025-01-11,3,-8.7,0.0,0.0,4.2\n2025-01-11,4,-9.0,0.0,0.0,3.6\n2025-01-11,5,-8.9,0.0,0.0,5.9\n2025-01-11,6,-8.9,0.0,0.0,6.6\n2025-01-11,7,-9.1,0.0,0.0,4.6\n2025-01-11,8,-9.2,0.0,0.0,4.0\n2025-01-11,9,-7.6,0.0,0.0,4.5\n2025-01-11,10,-6.4,0.0,0.0,3.6\n2025-01-11,11,-4.7,0.0,0.0,1.4\n2025-01-11,12,-3.6,0.0,0.0,3.1\n2025-01-11,13,-2.7,0.0,0.0,5.7\n2025-01-11,14,-2.3,0.0,0.0,7.6\n2025-01-11,15,-2.0,0.0,0.0,11.2\n2025-01-11,16,-2.4,0.0,0.0,13.0\n2025-01-11,17,-3.1,0.0,0.0,10.1\n2025-01-11,18,-4.1,0.0,0.0,5.9\n2025-01-11,19,-4.3,0.0,0.0,3.2\n2025-01-11,20,-3.7,0.0,0.0,0.2\n2025-01-11,21,-3.3,0.0,0.0,2.4\n2025-01-11,22,-2.8,0.0,0.0,3.6\n2025-01-11,23,-3.6,0.0,0.0,2.8\n2025-01-12,0,-4.4,0.0,0.0,7.6\n2025-01-12,1,-5.0,0.0,0.0,6.8\n2025-01-12,2,-5.3,0.0,0.0,5.1\n2025-01-12,3,-5.6,0.0,0.0,3.8\n2025-01-12,4,-6.3,0.0,0.0,2.8\n2025-01-12,5,-6.0,0.0,0.0,2.2\n2025-01-12,6,-5.2,0.0,0.0,2.9\n2025-01-12,7,-5.2,0.0,0.0,3.5\n2025-01-12,8,-6.3,0.0,0.0,1.2\n2025-01-12,9,-4.8,0.0,0.0,2.1\n2025-01-12,10,-4.9,0.0,0.0,4.8\n2025-01-12,11,-2.9,0.0,0.0,4.9\n2025-01-12,12,-1.3,0.0,0.0,4.7\n2025-01-12,13,-0.1,0.0,0.0,4.1\n2025-01-12,14,0.7,0.0,0.0,4.2\n2025-01-12,15,1.1,0.0,0.0,3.7\n2025-01-12,16,1.0,0.0,0.0,3.8\n2025-01-12,17,-0.4,0.0,0.0,3.4\n2025-01-12,18,-2.3,0.0,0.0,4.5\n2025-01-12,19,-2.6,0.0,0.0,4.7\n2025-01-12,20,-2.5,0.0,0.0,6.3\n2025-01-12,21,-2.7,0.0,0.0,8.1\n2025-01-12,22,-2.8,0.0,0.0,8.1\n2025-01-12,23,-3.5,0.0,0.0,6.3\n2025-01-13,0,-4.0,0.0,0.0,4.7\n2025-01-13,1,-4.6,0.0,0.0,3.6\n2025-01-13,2,-4.5,0.0,0.0,3.6\n2025-01-13,3,-4.4,0.0,0.0,4.2\n2025-01-13,4,-5.0,0.0,0.0,4.0\n2025-01-13,5,-4.9,0.0,0.0,4.6\n2025-01-13,6,-4.8,0.0,0.0,7.1\n2025-01-13,7,-3.5,0.0,0.0,7.9\n2025-01-13,8,-2.8,0.0,0.0,8.0\n2025-01-13,9,-1.9,0.0,0.0,8.8\n2025-01-13,10,-2.3,0.0,0.0,10.6\n2025-01-13,11,0.7,0.0,0.0,11.2\n2025-01-13,12,2.6,0.0,0.0,12.0\n2025-01-13,13,3.5,0.0,0.0,12.3\n2025-01-13,14,3.5,0.0,0.0,11.4\n2025-01-13,15,3.9,0.0,0.0,13.2\n2025-01-13,16,2.3,0.2,0.0,14.1\n2025-01-13,17,2.3,0.2,0.0,10.9\n2025-01-13,18,2.3,0.0,0.0,10.3\n2025-01-13,19,2.4,0.0,0.0,11.0\n2025-01-13,20,2.2,0.0,0.0,11.1\n2025-01-13,21,2.0,0.0,0.0,10.1\n2025-01-13,22,0.9,0.0,0.0,8.5\n2025-01-13,23,0.9,0.0,0.0,8.6\n2025-01-14,0,0.9,0.0,0.0,8.6\n2025-01-14,1,0.4,0.0,0.0,8.8\n2025-01-14,2,-0.3,0.0,0.0,9.6\n2025-01-14,3,0.5,0.0,0.0,10.2\n2025-01-14,4,0.6,0.0,0.0,7.6\n2025-01-14,5,0.3,0.0,0.0,6.5\n2025-01-14,6,0.9,0.0,0.0,7.6\n2025-01-14,7,1.2,0.0,0.0,7.1\n2025-01-14,8,0.6,0.0,0.0,7.2\n2025-01-14,9,2.3,0.0,0.0,10.8\n2025-01-14,10,3.8,0.0,0.0,4.8\n2025-01-14,11,5.0,0.0,0.0,9.3\n2025-01-14,12,5.5,0.0,0.0,10.7\n2025-01-14,13,5.9,0.0,0.0,12.9\n2025-01-14,14,5.6,0.0,0.0,13.9\n2025-01-14,15,4.8,0.0,0.0,16.0\n2025-01-14,16,3.9,0.0,0.0,14.2\n2025-01-14,17,2.9,0.0,0.0,14.5\n2025-01-14,18,2.2,0.0,0.0,16.0\n2025-01-14,19,1.1,0.0,0.0,15.0\n2025-01-14,20,0.1,0.0,0.0,11.9\n2025-01-14,21,-0.8,0.0,0.0,15.0\n2025-01-14,22,-0.9,0.0,0.0,17.7\n2025-01-14,23,-1.9,0.0,0.0,15.5\n2025-01-15,0,-2.5,0.0,0.0,14.9\n2025-01-15,1,-2.8,0.0,0.0,16.8\n2025-01-15,2,-3.5,0.0,0.0,17.6\n2025-01-15,3,-4.1,0.0,0.0,17.8\n2025-01-15,4,-4.3,0.0,0.0,16.0\n2025-01-15,5,-4.5,0.0,0.0,15.3\n2025-01-15,6,-4.8,0.0,0.0,14.7\n2025-01-15,7,-5.1,0.0,0.0,14.7\n2025-01-15,8,-5.5,0.0,0.0,14.0\n2025-01-15,9,-5.0,0.0,0.0,16.1\n2025-01-15,10,-3.8,0.0,0.0,16.7\n2025-01-15,11,-3.3,0.0,0.0,17.2\n2025-01-15,12,-2.9,0.0,0.0,16.5\n2025-01-15,13,-2.5,0.0,0.0,15.7\n2025-01-15,14,-2.2,0.0,0.0,15.3\n2025-01-15,15,-2.0,0.0,0.0,14.9\n2025-01-15,16,-2.3,0.0,0.0,14.8\n2025-01-15,17,-2.7,0.0,0.0,13.8\n2025-01-15,18,-3.6,0.0,0.0,11.3\n2025-01-15,19,-3.9,0.0,0.0,10.8\n2025-01-15,20,-4.2,0.0,0.0,10.9\n2025-01-15,21,-4.6,0.0,0.0,9.4\n2025-01-15,22,-4.8,0.0,0.0,8.6\n2025-01-15,23,-5.0,0.0,0.0,4.0\n2025-01-16,0,-4.9,0.0,0.0,5.1\n2025-01-16,1,-4.1,0.0,0.0,4.7\n2025-01-16,2,-3.7,0.0,0.0,5.1\n2025-01-16,3,-3.5,0.0,0.0,5.5\n2025-01-16,4,-2.9,0.0,0.0,6.5\n2025-01-16,5,-2.7,0.0,0.0,7.4\n2025-01-16,6,-2.7,0.0,0.0,7.9\n2025-01-16,7,-2.4,0.0,0.0,10.4\n2025-01-16,8,-2.0,0.0,0.0,11.0\n2025-01-16,9,-1.2,0.0,0.0,10.5\n2025-01-16,10,-2.3,0.0,0.0,7.1\n2025-01-16,11,-0.7,0.0,0.0,7.3\n2025-01-16,12,1.3,0.0,0.0,7.0\n2025-01-16,13,3.1,0.0,0.0,16.5\n2025-01-16,14,2.8,0.0,0.0,18.3\n2025-01-16,15,2.9,0.0,0.0,16.9\n2025-01-16,16,3.0,0.0,0.0,12.9\n2025-01-16,17,2.7,0.0,0.0,11.0\n2025-01-16,18,1.6,0.0,0.0,10.4\n2025-01-16,19,1.3,0.0,0.0,10.7\n2025-01-16,20,0.2,0.0,0.0,11.3\n2025-01-16,21,-0.8,0.0,0.0,15.4\n2025-01-16,22,-1.9,0.0,0.0,13.8\n2025-01-16,23,-3.0,0.0,0.0,13.0\n2025-01-17,0,-3.7,0.0,0.0,11.5\n2025-01-17,1,-4.2,0.0,0.0,10.9\n2025-01-17,2,-4.6,0.0,0.0,10.4\n2025-01-17,3,-5.0,0.0,0.0,9.6\n2025-01-17,4,-5.3,0.0,0.0,8.6\n2025-01-17,5,-5.6,0.0,0.0,7.5\n2025-01-17,6,-6.0,0.0,0.0,7.0\n2025-01-17,7,-6.4,0.0,0.0,5.5\n2025-01-17,8,-7.0,0.0,0.0,3.9\n2025-01-17,9,-5.3,0.0,0.0,4.4\n2025-01-17,10,-2.8,0.0,0.0,3.9\n2025-01-17,11,-0.1,0.0,0.0,1.8\n2025-01-17,12,1.7,0.0,0.0,1.6\n2025-01-17,13,2.6,0.0,0.0,4.9\n2025-01-17,14,3.0,0.0,0.0,6.0\n2025-01-17,15,2.9,0.0,0.0,6.7\n2025-01-17,16,2.2,0.0,0.0,7.4\n2025-01-17,17,0.8,0.0,0.0,8.6\n2025-01-17,18,-1.1,0.0,0.0,6.4\n2025-01-17,19,-2.1,0.0,0.0,4.3\n2025-01-17,20,-2.2,0.0,0.0,3.7\n2025-01-17,21,-2.4,0.0,0.0,2.1\n2025-01-17,22,-2.3,0.0,0.0,3.2\n2025-01-17,23,-2.6,0.0,0.0,2.7\n2025-01-18,0,-2.8,0.0,0.0,2.6\n2025-01-18,1,-2.9,0.0,0.0,2.1\n2025-01-18,2,-2.8,0.0,0.0,1.7\n2025-01-18,3,-2.7,0.0,0.0,2.0\n2025-01-18,4,-2.7,0.0,0.0,1.8\n2025-01-18,5,-2.7,0.0,0.0,3.2\n2025-01-18,6,-2.9,0.0,0.0,1.4\n2025-01-18,7,-3.2,0.0,0.0,2.8\n2025-01-18,8,-3.1,0.0,0.0,3.6\n2025-01-18,9,-1.4,0.0,0.0,4.4\n2025-01-18,10,-0.5,0.0,0.0,2.0\n2025-01-18,11,2.2,0.0,0.0,1.7\n2025-01-18,12,3.8,0.0,0.0,0.3\n2025-01-18,13,4.6,0.0,0.0,1.7\n2025-01-18,14,4.9,0.0,0.0,3.6\n2025-01-18,15,4.7,0.0,0.0,4.9\n2025-01-18,16,4.0,0.0,0.0,5.9\n2025-01-18,17,2.4,0.0,0.0,5.5\n2025-01-18,18,-0.2,0.0,0.0,4.2\n2025-01-18,19,-0.2,0.0,0.0,1.5\n2025-01-18,20,-0.4,0.0,0.0,1.2\n2025-01-18,21,-0.8,0.0,0.0,2.7\n2025-01-18,22,-1.1,0.0,0.0,1.5\n2025-01-18,23,-1.6,0.0,0.0,2.4\n2025-01-19,0,-1.6,0.0,0.0,3.6\n2025-01-19,1,-1.7,0.0,0.0,5.7\n2025-01-19,2,-1.6,0.0,0.0,8.2\n2025-01-19,3,-1.5,0.0,0.0,9.4\n2025-01-19,4,-1.4,0.0,0.0,9.7\n2025-01-19,5,-1.5,0.0,0.0,10.5\n2025-01-19,6,-1.3,0.0,0.0,11.2\n2025-01-19,7,-0.9,0.0,0.0,12.0\n2025-01-19,8,-0.6,0.0,0.0,11.9\n2025-01-19,9,0.8,0.0,0.0,12.1\n2025-01-19,10,1.9,0.0,0.0,10.0\n2025-01-19,11,4.0,0.0,0.0,9.4\n2025-01-19,12,4.9,0.0,0.0,8.4\n2025-01-19,13,5.4,0.0,0.0,6.9\n2025-01-19,14,6.6,0.0,0.0,6.5\n2025-01-19,15,7.0,0.0,0.0,8.9\n2025-01-19,16,6.4,0.0,0.0,7.2\n2025-01-19,17,3.9,0.0,0.0,3.3\n2025-01-19,18,2.1,0.0,0.0,4.4\n2025-01-19,19,2.6,0.0,0.0,7.0\n2025-01-19,20,2.6,0.0,0.0,8.6\n2025-01-19,21,2.2,0.0,0.0,7.4\n2025-01-19,22,1.6,0.0,0.0,5.6\n2025-01-19,23,1.3,0.0,0.0,7.7\n2025-01-20,0,1.0,0.0,0.0,7.8\n2025-01-20,1,0.7,0.0,0.0,7.4\n2025-01-20,2,1.0,0.0,0.0,6.9\n2025-01-20,3,1.0,0.0,0.0,6.8\n2025-01-20,4,1.1,0.0,0.0,6.8\n2025-01-20,5,1.0,0.0,0.0,6.0\n2025-01-20,6,1.0,0.0,0.0,6.0\n2025-01-20,7,1.2,0.0,0.0,6.5\n2025-01-20,8,1.4,0.0,0.0,6.0\n2025-01-20,9,2.0,0.0,0.0,5.1\n2025-01-20,10,2.5,0.0,0.0,4.9\n2025-01-20,11,4.2,0.0,0.0,5.6\n2025-01-20,12,5.0,0.0,0.0,3.9\n2025-01-20,13,5.2,0.0,0.0,3.2\n2025-01-20,14,5.5,0.0,0.0,3.0\n2025-01-20,15,5.5,0.0,0.0,3.3\n2025-01-20,16,4.7,0.0,0.0,6.1\n2025-01-20,17,3.2,0.0,0.0,9.9\n2025-01-20,18,1.3,0.0,0.0,8.6\n2025-01-20,19,0.5,0.0,0.0,7.1\n2025-01-20,20,0.4,0.0,0.0,6.7\n2025-01-20,21,-0.6,0.0,0.0,4.3\n2025-01-20,22,0.2,0.0,0.0,4.7\n2025-01-20,23,0.4,0.0,0.0,5.7\n2025-01-21,0,-0.9,0.0,0.0,4.5\n2025-01-21,1,-1.3,0.0,0.0,2.9\n2025-01-21,2,-1.6,0.0,0.0,2.6\n2025-01-21,3,-1.9,0.0,0.0,2.7\n2025-01-21,4,-2.2,0.0,0.0,1.3\n2025-01-21,5,-2.6,0.0,0.0,1.2\n2025-01-21,6,-2.7,0.0,0.0,2.2\n2025-01-21,7,-2.6,0.0,0.0,3.2\n2025-01-21,8,-2.8,0.0,0.0,3.5\n2025-01-21,9,-1.6,0.0,0.0,4.8\n2025-01-21,10,0.4,0.0,0.0,6.4\n2025-01-21,11,1.3,0.0,0.0,4.5\n2025-01-21,12,3.0,0.0,0.0,2.8\n2025-01-21,13,4.4,0.0,0.0,2.6\n2025-01-21,14,5.2,0.0,0.0,3.6\n2025-01-21,15,5.2,0.0,0.0,5.2\n2025-01-21,16,4.4,0.0,0.0,5.7\n2025-01-21,17,2.5,0.0,0.0,3.6\n2025-01-21,18,0.3,0.0,0.0,2.9\n2025-01-21,19,-0.2,0.0,0.0,3.6\n2025-01-21,20,-0.4,0.0,0.0,5.1\n2025-01-21,21,0.7,0.0,0.0,7.5\n2025-01-21,22,0.6,0.0,0.0,5.7\n2025-01-21,23,0.8,0.0,0.0,8.1\n2025-01-22,0,0.7,0.0,0.0,10.4\n2025-01-22,1,0.5,0.0,0.0,11.2\n2025-01-22,2,0.3,0.0,0.0,10.4\n2025-01-22,3,0.6,0.0,0.0,10.7\n2025-01-22,4,0.5,0.0,0.0,11.4\n2025-01-22,5,0.8,0.0,0.0,9.3\n2025-01-22,6,0.3,0.0,0.0,4.2\n2025-01-22,7,0.0,0.0,0.0,2.9\n2025-01-22,8,0.1,0.0,0.0,6.0\n2025-01-22,9,0.5,0.0,0.0,6.3\n2025-01-22,10,0.8,0.0,0.0,3.5\n2025-01-22,11,1.4,0.0,0.0,4.0\n2025-01-22,12,2.4,0.0,0.0,4.8\n2025-01-22,13,3.5,0.0,0.0,4.8\n2025-01-22,14,4.6,0.0,0.0,6.2\n2025-01-22,15,4.4,0.0,0.0,7.6\n2025-01-22,16,3.3,0.0,0.0,7.6\n2025-01-22,17,1.9,0.0,0.0,7.0\n2025-01-22,18,0.2,0.0,0.0,5.8\n2025-01-22,19,-1.1,0.0,0.0,4.5\n2025-01-22,20,-1.2,0.0,0.0,3.4\n2025-01-22,21,-0.8,0.0,0.0,2.6\n2025-01-22,22,-0.3,0.0,0.0,2.8\n2025-01-22,23,-0.1,0.0,0.0,1.7\n2025-01-23,0,-0.7,0.0,0.0,2.3\n2025-01-23,1,-1.2,0.0,0.0,6.5\n2025-01-23,2,-1.9,0.0,0.0,7.7\n2025-01-23,3,-2.5,0.0,0.0,7.1\n2025-01-23,4,-2.9,0.0,0.0,7.5\n2025-01-23,5,-3.2,0.0,0.0,6.3\n2025-01-23,6,-3.3,0.0,0.0,5.4\n2025-01-23,7,-3.5,0.0,0.0,5.0\n2025-01-23,8,-3.6,0.0,0.0,4.8\n2025-01-23,9,-2.2,0.0,0.0,3.9\n2025-01-23,10,-0.8,0.0,0.0,2.7\n2025-01-23,11,1.7,0.0,0.0,3.7\n2025-01-23,12,3.3,0.0,0.0,4.7\n2025-01-23,13,4.3,0.0,0.0,5.2\n2025-01-23,14,4.9,0.0,0.0,6.5\n2025-01-23,15,4.8,0.0,0.0,7.7\n2025-01-23,16,3.7,0.0,0.0,8.8\n2025-01-23,17,2.0,0.0,0.0,8.1\n2025-01-23,18,-0.7,0.0,0.0,5.4\n2025-01-23,19,-1.3,0.0,0.0,5.3\n2025-01-23,20,-1.6,0.0,0.0,5.0\n2025-01-23,21,-1.9,0.0,0.0,4.0\n2025-01-23,22,-1.7,0.0,0.0,4.0\n2025-01-23,23,-1.7,0.0,0.0,7.1\n2025-01-24,0,-2.0,0.0,0.0,7.6\n2025-01-24,1,-2.5,0.0,0.0,4.3\n2025-01-24,2,-2.8,0.0,0.0,3.5\n2025-01-24,3,-3.0,0.0,0.0,3.3\n2025-01-24,4,-3.1,0.0,0.0,3.9\n2025-01-24,5,-3.3,0.0,0.0,6.7\n2025-01-24,6,-3.4,0.0,0.0,8.2\n2025-01-24,7,-3.2,0.0,0.0,9.0\n2025-01-24,8,-3.0,0.0,0.0,8.1\n2025-01-24,9,-1.6,0.0,0.0,6.1\n2025-01-24,10,-0.5,0.0,0.0,2.7\n2025-01-24,11,2.7,0.0,0.0,3.6\n2025-01-24,12,4.7,0.0,0.0,5.1\n2025-01-24,13,6.1,0.0,0.0,6.1\n2025-01-24,14,6.9,0.0,0.0,6.8\n2025-01-24,15,7.1,0.0,0.0,7.6\n2025-01-24,16,6.9,0.0,0.0,7.7\n2025-01-24,17,4.7,0.0,0.0,6.0\n2025-01-24,18,2.7,0.0,0.0,8.0\n2025-01-24,19,2.1,0.0,0.0,9.3\n2025-01-24,20,1.3,0.0,0.0,9.2\n2025-01-24,21,0.5,0.0,0.0,9.6\n2025-01-24,22,0.3,0.0,0.0,11.0\n2025-01-24,23,0.0,0.0,0.0,10.0\n2025-01-25,0,-0.4,0.0,0.0,8.6\n2025-01-25,1,-0.4,0.0,0.0,8.1\n2025-01-25,2,0.1,0.0,0.0,9.6\n2025-01-25,3,0.2,0.0,0.0,9.3\n2025-01-25,4,0.0,0.0,0.0,8.1\n2025-01-25,5,-0.3,0.0,0.0,7.8\n2025-01-25,6,-0.5,0.0,0.0,8.5\n2025-01-25,7,-0.4,0.0,0.0,9.1\n2025-01-25,8,-0.3,0.0,0.0,8.3\n2025-01-25,9,1.4,0.0,0.0,9.3\n2025-01-25,10,3.5,0.0,0.0,6.9\n2025-01-25,11,5.5,0.0,0.0,8.3\n2025-01-25,12,7.0,0.0,0.0,10.1\n2025-01-25,13,8.1,0.0,0.0,9.5\n2025-01-25,14,8.5,0.0,0.0,9.7\n2025-01-25,15,8.3,0.0,0.0,10.5\n2025-01-25,16,7.7,0.0,0.0,7.5\n2025-01-25,17,5.4,0.0,0.0,3.6\n2025-01-25,18,3.3,0.0,0.0,6.3\n2025-01-25,19,3.5,0.0,0.0,12.5\n2025-01-25,20,3.3,0.0,0.0,17.3\n2025-01-25,21,2.0,0.0,0.0,16.8\n2025-01-25,22,2.7,0.0,0.0,16.8\n2025-01-25,23,2.0,0.0,0.0,16.8\n2025-01-26,0,1.1,0.0,0.0,13.3\n2025-01-26,1,0.5,0.0,0.0,10.3\n2025-01-26,2,1.0,0.0,0.0,9.8\n2025-01-26,3,1.1,0.0,0.0,9.5\n2025-01-26,4,1.0,0.0,0.0,11.0\n2025-01-26,5,0.8,0.0,0.0,11.6\n2025-01-26,6,0.8,0.0,0.0,12.9\n2025-01-26,7,0.8,0.0,0.0,16.6\n2025-01-26,8,0.5,0.0,0.0,13.5\n2025-01-26,9,1.5,0.0,0.0,8.5\n2025-01-26,10,2.7,0.0,0.0,9.9\n2025-01-26,11,3.9,0.0,0.0,8.6\n2025-01-26,12,5.8,0.0,0.0,8.0\n2025-01-26,13,6.2,0.0,0.0,6.2\n2025-01-26,14,6.3,0.0,0.0,6.2\n2025-01-26,15,6.1,0.0,0.0,6.5\n2025-01-26,16,5.7,0.0,0.0,5.9\n2025-01-26,17,5.1,0.0,0.0,6.1\n2025-01-26,18,3.8,0.0,0.0,8.5\n2025-01-26,19,3.1,0.0,0.0,8.1\n2025-01-26,20,2.4,0.0,0.0,9.3\n2025-01-26,21,2.6,0.0,0.0,11.0\n2025-01-26,22,3.1,0.0,0.0,8.8\n2025-01-26,23,2.7,0.1,0.0,7.9\n2025-01-27,0,2.1,0.1,0.0,8.6\n2025-01-27,1,1.7,0.2,0.07,10.5\n2025-01-27,2,1.4,0.4,0.14,12.8\n2025-01-27,3,1.0,0.3,0.07,12.3\n2025-01-27,4,0.8,0.5,0.28,13.2\n2025-01-27,5,0.5,0.9,0.56,9.6\n2025-01-27,6,0.5,0.6,0.35,8.5\n2025-01-27,7,0.4,0.7,0.49,12.0\n2025-01-27,8,0.8,0.8,0.56,19.8\n2025-01-27,9,-0.3,0.3,0.21,20.5\n2025-01-27,10,-0.7,0.4,0.28,21.1\n2025-01-27,11,-0.7,0.0,0.0,21.0\n2025-01-27,12,-0.3,0.0,0.0,21.9\n2025-01-27,13,0.2,0.3,0.21,25.0\n2025-01-27,14,0.4,0.2,0.14,27.5\n2025-01-27,15,-0.1,0.2,0.14,23.0\n2025-01-27,16,-0.2,0.0,0.0,22.8\n2025-01-27,17,-0.7,0.4,0.28,20.9\n2025-01-27,18,-1.5,0.0,0.0,19.3\n2025-01-27,19,-2.1,0.0,0.0,17.1\n2025-01-27,20,-2.4,0.0,0.0,14.8\n2025-01-27,21,-3.0,0.0,0.0,13.3\n2025-01-27,22,-2.6,0.2,0.14,15.2\n2025-01-27,23,-3.0,0.0,0.0,14.3\n2025-01-28,0,-3.5,0.0,0.0,15.1\n2025-01-28,1,-3.8,0.0,0.0,15.6\n2025-01-28,2,-4.2,0.0,0.0,14.9\n2025-01-28,3,-4.6,0.0,0.0,15.5\n2025-01-28,4,-4.3,0.0,0.0,16.0\n2025-01-28,5,-4.0,0.0,0.0,16.1\n2025-01-28,6,-4.3,0.0,0.0,15.4\n2025-01-28,7,-3.9,0.0,0.0,17.1\n2025-01-28,8,-3.7,0.1,0.07,19.8\n2025-01-28,9,-3.3,0.0,0.0,24.6\n2025-01-28,10,-3.2,0.2,0.14,19.6\n2025-01-28,11,-3.1,0.0,0.0,25.7\n2025-01-28,12,-3.3,0.1,0.07,24.4\n2025-01-28,13,-3.8,0.0,0.07,22.9\n2025-01-28,14,-3.8,0.0,0.0,24.0\n2025-01-28,15,-3.8,0.0,0.0,24.3\n2025-01-28,16,-3.9,0.0,0.0,22.5\n2025-01-28,17,-4.2,0.0,0.0,20.2\n2025-01-28,18,-4.5,0.0,0.0,18.4\n2025-01-28,19,-4.6,0.0,0.0,17.8\n2025-01-28,20,-4.7,0.0,0.0,19.0\n2025-01-28,21,-4.6,0.0,0.0,19.3\n2025-01-28,22,-4.9,0.0,0.0,16.7\n2025-01-28,23,-5.1,0.0,0.0,17.7\n2025-01-29,0,-5.0,0.0,0.0,17.3\n2025-01-29,1,-5.2,0.0,0.0,17.8\n2025-01-29,2,-5.6,0.0,0.0,16.7\n2025-01-29,3,-6.0,0.0,0.0,16.0\n2025-01-29,4,-6.3,0.0,0.0,15.3\n2025-01-29,5,-6.7,0.0,0.0,15.8\n2025-01-29,6,-7.2,0.0,0.0,15.2\n2025-01-29,7,-7.8,0.0,0.0,13.7\n2025-01-29,8,-8.1,0.0,0.0,12.3\n2025-01-29,9,-7.6,0.0,0.0,13.3\n2025-01-29,10,-5.5,0.0,0.0,19.9\n2025-01-29,11,-5.1,0.0,0.0,20.5\n2025-01-29,12,-4.6,0.0,0.0,20.8\n2025-01-29,13,-4.2,0.0,0.0,21.1\n2025-01-29,14,-3.9,0.0,0.0,20.1\n2025-01-29,15,-3.6,0.0,0.0,20.8\n2025-01-29,16,-3.6,0.0,0.0,16.8\n2025-01-29,17,-4.2,0.0,0.0,13.6\n2025-01-29,18,-5.1,0.0,0.0,13.6\n2025-01-29,19,-5.8,0.0,0.0,13.2\n2025-01-29,20,-6.3,0.0,0.0,11.8\n2025-01-29,21,-7.0,0.0,0.0,10.6\n2025-01-29,22,-5.7,0.0,0.0,11.5\n2025-01-29,23,-5.9,0.0,0.0,11.4\n2025-01-30,0,-6.3,0.0,0.0,10.3\n2025-01-30,1,-6.9,0.0,0.0,9.5\n2025-01-30,2,-7.7,0.0,0.0,8.5\n2025-01-30,3,-7.6,0.0,0.0,7.0\n2025-01-30,4,-7.5,0.0,0.0,8.7\n2025-01-30,5,-7.7,0.0,0.0,9.4\n2025-01-30,6,-7.9,0.0,0.0,9.0\n2025-01-30,7,-8.2,0.0,0.0,9.3\n2025-01-30,8,-8.3,0.0,0.0,5.8\n2025-01-30,9,-8.1,0.0,0.0,6.0\n2025-01-30,10,-5.8,0.0,0.0,3.4\n2025-01-30,11,-4.3,0.0,0.0,3.1\n2025-01-30,12,-3.2,0.0,0.0,3.9\n2025-01-30,13,-2.3,0.0,0.0,3.8\n2025-01-30,14,-1.6,0.0,0.0,4.7\n2025-01-30,15,-1.1,0.0,0.0,5.4\n2025-01-30,16,-1.4,0.0,0.0,5.2\n2025-01-30,17,-1.7,0.0,0.0,4.7\n2025-01-30,18,-2.4,0.0,0.0,4.5\n2025-01-30,19,-3.4,0.0,0.0,3.0\n2025-01-30,20,-3.6,0.0,0.0,3.2\n2025-01-30,21,-3.0,0.0,0.0,3.3\n2025-01-30,22,-2.1,0.0,0.0,4.8\n2025-01-30,23,-2.3,0.0,0.0,5.0\n2025-01-31,0,-3.0,0.0,0.0,6.5\n2025-01-31,1,-2.9,0.0,0.0,7.6\n2025-01-31,2,-3.0,0.0,0.0,6.4\n2025-01-31,3,-2.9,0.0,0.0,4.2\n2025-01-31,4,-2.8,0.1,0.07,3.9\n2025-01-31,5,-3.0,0.1,0.07,8.1\n2025-01-31,6,-2.8,0.1,0.07,7.8\n2025-01-31,7,-2.7,0.1,0.07,7.8\n2025-01-31,8,-2.7,0.1,0.07,9.3\n2025-01-31,9,-2.9,0.6,0.42,8.7\n2025-01-31,10,-2.3,0.0,0.0,6.6\n2025-01-31,11,-1.7,0.0,0.0,8.5\n2025-01-31,12,-1.2,0.0,0.0,7.5\n2025-01-31,13,-0.8,0.0,0.0,6.9\n2025-01-31,14,-0.6,0.1,0.07,7.0\n2025-01-31,15,-0.7,0.3,0.21,9.2\n2025-01-31,16,-0.5,0.3,0.21,9.6\n2025-01-31,17,-0.5,0.4,0.21,7.3\n2025-01-31,18,-0.6,0.4,0.28,9.1\n2025-01-31,19,-0.6,0.4,0.28,8.8\n2025-01-31,20,-0.5,0.3,0.21,6.9\n2025-01-31,21,-0.6,0.3,0.21,6.2\n2025-01-31,22,-0.4,0.0,0.0,6.5\n2025-01-31,23,-0.3,0.0,0.0,5.4\n2025-02-01,0,-0.4,0.0,0.0,5.9\n2025-02-01,1,-0.6,0.0,0.0,6.1\n2025-02-01,2,-1.0,0.0,0.0,7.0\n2025-02-01,3,-1.6,0.0,0.0,7.9\n2025-02-01,4,-2.0,0.0,0.0,6.5\n2025-02-01,5,-2.6,0.0,0.0,8.2\n2025-02-01,6,-3.2,0.0,0.0,7.9\n2025-02-01,7,-3.5,0.0,0.0,8.3\n2025-02-01,8,-3.9,0.0,0.0,7.3\n2025-02-01,9,-3.5,0.0,0.0,6.1\n2025-02-01,10,-1.3,0.0,0.0,6.6\n2025-02-01,11,0.9,0.0,0.0,5.5\n2025-02-01,12,2.5,0.0,0.0,5.9\n2025-02-01,13,3.5,0.0,0.0,5.8\n2025-02-01,14,4.2,0.0,0.0,6.1\n2025-02-01,15,4.6,0.0,0.0,6.9\n2025-02-01,16,4.4,0.0,0.0,7.3\n2025-02-01,17,2.6,0.0,0.0,6.8\n2025-02-01,18,0.7,0.0,0.0,8.1\n2025-02-01,19,-0.4,0.0,0.0,9.0\n2025-02-01,20,-1.5,0.0,0.0,8.2\n2025-02-01,21,-2.0,0.0,0.0,8.8\n2025-02-01,22,-1.3,0.0,0.0,9.6\n2025-02-01,23,-1.7,0.0,0.0,10.3\n2025-02-02,0,-2.0,0.0,0.0,9.7\n2025-02-02,1,-2.3,0.0,0.0,9.0\n2025-02-02,2,-2.7,0.0,0.0,9.1\n2025-02-02,3,-3.1,0.0,0.0,9.2\n2025-02-02,4,-3.4,0.0,0.0,7.6\n2025-02-02,5,-3.5,0.0,0.0,5.8\n2025-02-02,6,-3.7,0.0,0.0,7.1\n2025-02-02,7,-3.9,0.0,0.0,6.0\n2025-02-02,8,-3.9,0.0,0.0,6.8\n2025-02-02,9,-2.5,0.0,0.0,6.0\n2025-02-02,10,-0.2,0.0,0.0,2.9\n2025-02-02,11,1.8,0.0,0.0,3.7\n2025-02-02,12,3.3,0.0,0.0,4.8\n2025-02-02,13,4.8,0.0,0.0,5.8\n2025-02-02,14,5.4,0.0,0.0,7.7\n2025-02-02,15,4.6,0.0,0.0,10.3\n2025-02-02,16,3.2,0.0,0.0,12.3\n2025-02-02,17,1.6,0.0,0.0,11.9\n2025-02-02,18,0.2,0.0,0.0,10.2\n2025-02-02,19,-0.4,0.0,0.0,9.9\n2025-02-02,20,0.3,0.0,0.0,10.0\n2025-02-02,21,0.4,0.0,0.0,11.0\n2025-02-02,22,1.1,0.0,0.0,11.5\n2025-02-02,23,0.6,0.0,0.0,11.2\n2025-02-03,0,-0.2,0.0,0.0,12.2\n2025-02-03,1,-0.8,0.0,0.0,13.9\n2025-02-03,2,-1.4,0.0,0.0,12.9\n2025-02-03,3,-2.1,0.0,0.0,15.2\n2025-02-03,4,-3.2,0.0,0.0,16.3\n2025-02-03,5,-4.3,0.0,0.0,17.8\n2025-02-03,6,-5.3,0.0,0.0,18.5\n2025-02-03,7,-6.3,0.0,0.0,17.6\n2025-02-03,8,-6.8,0.0,0.0,18.3\n2025-02-03,9,-6.4,0.0,0.0,19.0\n2025-02-03,10,-4.4,0.0,0.0,22.0\n2025-02-03,11,-4.1,0.0,0.0,22.8\n2025-02-03,12,-3.9,0.0,0.0,22.9\n2025-02-03,13,-3.7,0.0,0.0,22.0\n2025-02-03,14,-3.4,0.0,0.0,21.6\n2025-02-03,15,-3.6,0.0,0.0,21.4\n2025-02-03,16,-4.2,0.0,0.0,20.8\n2025-02-03,17,-4.9,0.0,0.0,19.4\n2025-02-03,18,-6.0,0.0,0.0,17.8\n2025-02-03,19,-6.9,0.0,0.0,18.9\n2025-02-03,20,-7.8,0.0,0.0,20.1\n2025-02-03,21,-8.5,0.0,0.0,20.3\n2025-02-03,22,-7.5,0.0,0.0,19.9\n2025-02-03,23,-7.9,0.0,0.0,21.6\n2025-02-04,0,-8.4,0.0,0.0,23.9\n2025-02-04,1,-9.3,0.0,0.0,21.8\n2025-02-04,2,-10.1,0.0,0.0,21.0\n2025-02-04,3,-10.5,0.0,0.0,20.5\n2025-02-04,4,-11.1,0.0,0.0,20.0\n2025-02-04,5,-11.5,0.0,0.0,19.0\n2025-02-04,6,-12.0,0.0,0.0,18.2\n2025-02-04,7,-12.5,0.0,0.0,16.3\n2025-02-04,8,-12.6,0.0,0.0,15.1\n2025-02-04,9,-11.9,0.0,0.0,15.9\n2025-02-04,10,-9.8,0.0,0.0,18.4\n2025-02-04,11,-8.7,0.0,0.0,17.6\n2025-02-04,12,-7.6,0.0,0.0,17.7\n2025-02-04,13,-6.5,0.0,0.0,20.1\n2025-02-04,14,-6.1,0.0,0.0,20.4\n2025-02-04,15,-6.1,0.0,0.0,19.5\n2025-02-04,16,-6.5,0.0,0.0,19.2\n2025-02-04,17,-7.3,0.0,0.0,17.9\n2025-02-04,18,-8.3,0.0,0.0,15.0\n2025-02-04,19,-9.0,0.0,0.0,14.6\n2025-02-04,20,-9.7,0.0,0.0,15.3\n2025-02-04,21,-10.3,0.0,0.0,15.0\n2025-02-04,22,-9.1,0.0,0.0,13.7\n2025-02-04,23,-9.4,0.0,0.0,12.9\n2025-02-05,0,-9.6,0.0,0.0,11.5\n2025-02-05,1,-9.5,0.0,0.0,11.3\n2025-02-05,2,-9.8,0.0,0.0,11.5\n2025-02-05,3,-10.0,0.0,0.0,11.3\n2025-02-05,4,-10.2,0.0,0.0,12.7\n2025-02-05,5,-10.4,0.0,0.0,12.5\n2025-02-05,6,-10.8,0.0,0.0,10.6\n2025-02-05,7,-11.2,0.0,0.0,10.1\n2025-02-05,8,-11.2,0.0,0.0,9.3\n2025-02-05,9,-10.2,0.0,0.0,10.1\n2025-02-05,10,-8.9,0.0,0.0,11.0\n2025-02-05,11,-7.3,0.0,0.0,11.4\n2025-02-05,12,-6.0,0.0,0.0,13.1\n2025-02-05,13,-5.3,0.0,0.0,14.5\n2025-02-05,14,-4.6,0.0,0.0,16.1\n2025-02-05,15,-4.0,0.0,0.0,23.6\n2025-02-05,16,-4.5,0.0,0.0,20.6\n2025-02-05,17,-5.3,0.0,0.0,17.4\n2025-02-05,18,-6.5,0.0,0.0,15.0\n2025-02-05,19,-7.6,0.0,0.0,16.9\n2025-02-05,20,-8.7,0.0,0.0,16.3\n2025-02-05,21,-9.5,0.0,0.0,14.8\n2025-02-05,22,-8.9,0.0,0.0,15.4\n2025-02-05,23,-9.5,0.0,0.0,13.0\n2025-02-06,0,-9.6,0.0,0.0,9.9\n2025-02-06,1,-9.7,0.0,0.0,7.7\n2025-02-06,2,-9.8,0.0,0.0,8.1\n2025-02-06,3,-9.9,0.0,0.0,8.5\n2025-02-06,4,-10.1,0.0,0.0,7.5\n2025-02-06,5,-10.3,0.0,0.0,6.4\n2025-02-06,6,-10.4,0.0,0.0,3.4\n2025-02-06,7,-10.5,0.0,0.0,0.2\n2025-02-06,8,-10.0,0.0,0.0,3.9\n2025-02-06,9,-8.4,0.0,0.0,4.4\n2025-02-06,10,-6.9,0.0,0.0,1.7\n2025-02-06,11,-4.6,0.0,0.0,2.2\n2025-02-06,12,-2.7,0.0,0.0,7.3\n2025-02-06,13,-1.8,0.0,0.0,12.0\n2025-02-06,14,-0.9,0.0,0.0,18.2\n2025-02-06,15,-0.5,0.1,0.07,21.0\n2025-02-06,16,-1.2,0.2,0.14,23.7\n2025-02-06,17,-1.2,0.5,0.35,18.2\n2025-02-06,18,-1.2,0.1,0.07,12.9\n2025-02-06,19,-1.2,0.0,0.0,13.9\n2025-02-06,20,-1.6,0.0,0.0,14.3\n2025-02-06,21,-3.0,0.0,0.0,12.7\n2025-02-06,22,-4.3,0.0,0.0,9.2\n2025-02-06,23,-5.1,0.0,0.0,5.6\n2025-02-07,0,-6.0,0.0,0.0,5.4\n2025-02-07,1,-8.0,0.0,0.0,5.6\n2025-02-07,2,-8.8,0.0,0.0,6.0\n2025-02-07,3,-9.0,0.0,0.0,5.9\n2025-02-07,4,-7.0,0.0,0.0,5.3\n2025-02-07,5,-7.3,0.0,0.0,11.6\n2025-02-07,6,-8.1,0.4,0.28,20.7\n2025-02-07,7,-8.2,0.2,0.14,24.2\n2025-02-07,8,-8.9,0.2,0.14,24.7\n2025-02-07,9,-9.5,0.0,0.0,21.0\n2025-02-07,10,-8.6,0.1,0.07,24.5\n2025-02-07,11,-8.3,0.0,0.0,23.6\n2025-02-07,12,-7.0,0.0,0.0,24.2\n2025-02-07,13,-6.7,0.0,0.0,28.8\n2025-02-07,14,-6.9,0.0,0.0,28.9\n2025-02-07,15,-7.6,0.0,0.0,28.4\n2025-02-07,16,-7.8,0.0,0.0,32.4\n2025-02-07,17,-8.7,0.0,0.0,26.2\n2025-02-07,18,-9.3,0.0,0.0,21.9\n2025-02-07,19,-10.0,0.0,0.0,21.0\n2025-02-07,20,-10.6,0.0,0.0,20.5\n2025-02-07,21,-10.8,0.0,0.0,19.0\n2025-02-07,22,-10.7,0.0,0.0,18.8\n2025-02-07,23,-10.3,0.0,0.0,18.0\n2025-02-08,0,-10.5,0.0,0.0,18.2\n2025-02-08,1,-10.7,0.0,0.0,17.1\n2025-02-08,2,-10.9,0.0,0.0,16.5\n2025-02-08,3,-11.0,0.0,0.0,16.7\n2025-02-08,4,-11.1,0.0,0.0,16.6\n2025-02-08,5,-11.2,0.0,0.0,16.1\n2025-02-08,6,-11.1,0.0,0.0,15.3\n2025-02-08,7,-11.0,0.0,0.0,15.5\n2025-02-08,8,-10.7,0.0,0.0,15.3\n2025-02-08,9,-9.4,0.0,0.0,14.7\n2025-02-08,10,-8.2,0.0,0.0,18.1\n2025-02-08,11,-6.8,0.0,0.0,17.7\n2025-02-08,12,-5.5,0.0,0.0,16.4\n2025-02-08,13,-4.6,0.0,0.0,13.3\n2025-02-08,14,-3.8,0.0,0.0,12.6\n2025-02-08,15,-3.2,0.0,0.0,12.8\n2025-02-08,16,-3.3,0.0,0.0,12.6\n2025-02-08,17,-3.9,0.0,0.0,12.2\n2025-02-08,18,-4.9,0.0,0.0,10.7\n2025-02-08,19,-5.9,0.0,0.0,12.4\n2025-02-08,20,-6.6,0.0,0.0,13.7\n2025-02-08,21,-7.3,0.0,0.0,13.3\n2025-02-08,22,-7.2,0.0,0.0,10.6\n2025-02-08,23,-7.6,0.0,0.0,9.9\n2025-02-09,0,-8.1,0.0,0.0,6.7\n2025-02-09,1,-8.7,0.0,0.0,3.8\n2025-02-09,2,-9.1,0.0,0.0,2.9\n2025-02-09,3,-8.9,0.0,0.0,2.7\n2025-02-09,4,-9.1,0.0,0.0,4.6\n2025-02-09,5,-8.7,0.0,0.0,5.4\n2025-02-09,6,-8.5,0.0,0.0,7.3\n2025-02-09,7,-8.7,0.0,0.0,7.7\n2025-02-09,8,-8.5,0.0,0.0,8.2\n2025-02-09,9,-6.6,0.0,0.0,7.6\n2025-02-09,10,-5.7,0.0,0.0,2.3\n2025-02-09,11,-3.7,0.0,0.0,4.0\n2025-02-09,12,-2.1,0.0,0.0,11.4\n2025-02-09,13,-1.0,0.0,0.0,14.1\n2025-02-09,14,-0.9,0.0,0.0,14.3\n2025-02-09,15,-1.0,0.0,0.0,14.8\n2025-02-09,16,-1.0,0.0,0.0,15.9\n2025-02-09,17,-1.6,0.0,0.0,15.1\n2025-02-09,18,-2.8,0.0,0.0,11.4\n2025-02-09,19,-3.5,0.0,0.0,11.2\n2025-02-09,20,-4.2,0.0,0.0,11.1\n2025-02-09,21,-5.2,0.0,0.0,10.0\n2025-02-09,22,-4.1,0.0,0.0,8.7\n2025-02-09,23,-4.4,0.0,0.0,8.7\n2025-02-10,0,-4.7,0.0,0.0,8.1\n2025-02-10,1,-4.8,0.0,0.0,7.1\n2025-02-10,2,-5.0,0.0,0.0,6.4\n2025-02-10,3,-5.2,0.0,0.0,3.8\n2025-02-10,4,-5.4,0.0,0.0,3.8\n2025-02-10,5,-4.7,0.0,0.0,1.2\n2025-02-10,6,-4.4,0.0,0.0,1.1\n2025-02-10,7,-4.0,0.0,0.0,1.1\n2025-02-10,8,-3.8,0.1,0.07,1.6\n2025-02-10,9,-3.5,0.1,0.07,1.8\n2025-02-10,10,-5.5,0.1,0.07,2.3\n2025-02-10,11,-4.3,0.0,0.0,3.8\n2025-02-10,12,-2.6,0.0,0.0,5.1\n2025-02-10,13,-0.3,0.0,0.0,5.4\n2025-02-10,14,0.9,0.0,0.0,14.6\n2025-02-10,15,1.0,0.0,0.0,15.1\n2025-02-10,16,0.5,0.0,0.0,14.0\n2025-02-10,17,0.0,0.0,0.0,11.1\n2025-02-10,18,-1.6,0.0,0.0,7.6\n2025-02-10,19,-2.3,0.0,0.0,8.1\n2025-02-10,20,-2.5,0.0,0.0,7.2\n2025-02-10,21,-2.7,0.0,0.0,6.2\n2025-02-10,22,-2.8,0.0,0.0,4.9\n2025-02-10,23,-3.5,0.0,0.0,2.7\n2025-02-11,0,-4.3,0.0,0.0,3.5\n2025-02-11,1,-4.8,0.0,0.0,3.1\n2025-02-11,2,-5.0,0.0,0.0,1.8\n2025-02-11,3,-5.4,0.0,0.0,1.6\n2025-02-11,4,-6.0,0.0,0.0,2.6\n2025-02-11,5,-6.1,0.0,0.0,4.8\n2025-02-11,6,-6.1,0.0,0.0,7.9\n2025-02-11,7,-6.1,0.0,0.0,9.2\n2025-02-11,8,-5.7,0.0,0.0,10.3\n2025-02-11,9,-3.3,0.0,0.0,11.5\n2025-02-11,10,-0.9,0.0,0.0,9.3\n2025-02-11,11,1.8,0.0,0.0,9.5\n2025-02-11,12,3.5,0.0,0.0,8.6\n2025-02-11,13,4.5,0.0,0.0,8.5\n2025-02-11,14,5.2,0.0,0.0,8.7\n2025-02-11,15,5.5,0.0,0.0,9.1\n2025-02-11,16,5.6,0.0,0.0,7.5\n2025-02-11,17,4.8,0.0,0.0,3.6\n2025-02-11,18,1.6,0.0,0.0,3.2\n2025-02-11,19,1.4,0.0,0.0,8.8\n2025-02-11,20,0.9,0.0,0.0,9.4\n2025-02-11,21,0.4,0.0,0.0,9.5\n2025-02-11,22,0.3,0.0,0.0,9.6\n2025-02-11,23,0.4,0.0,0.0,7.0\n2025-02-12,0,-0.2,0.0,0.0,6.8\n2025-02-12,1,0.8,0.0,0.0,6.7\n2025-02-12,2,0.9,0.0,0.0,7.9\n2025-02-12,3,0.9,0.1,0.07,10.2\n2025-02-12,4,0.5,0.3,0.21,13.1\n2025-02-12,5,0.4,0.3,0.21,11.7\n2025-02-12,6,0.2,0.1,0.07,8.8\n2025-02-12,7,-0.1,0.4,0.21,10.3\n2025-02-12,8,-0.1,0.5,0.35,11.6\n2025-02-12,9,0.3,0.3,0.21,8.7\n2025-02-12,10,0.6,0.3,0.21,4.2\n2025-02-12,11,0.9,0.4,0.28,3.5\n2025-02-12,12,2.6,0.1,0.0,1.6\n2025-02-12,13,2.6,0.1,0.0,3.0\n2025-02-12,14,3.0,0.1,0.0,3.9\n2025-02-12,15,3.1,0.0,0.0,8.8\n2025-02-12,16,2.8,0.0,0.0,12.8\n2025-02-12,17,2.5,0.0,0.0,21.1\n2025-02-12,18,1.3,0.0,0.0,24.3\n2025-02-12,19,-1.0,0.0,0.0,24.0\n2025-02-12,20,-2.3,0.0,0.0,23.7\n2025-02-12,21,-3.1,0.0,0.0,22.9\n2025-02-12,22,-2.6,0.0,0.0,23.2\n2025-02-12,23,-3.2,0.0,0.0,22.0\n2025-02-13,0,-3.6,0.0,0.0,20.1\n2025-02-13,1,-4.1,0.0,0.0,16.4\n2025-02-13,2,-4.3,0.0,0.0,14.1\n2025-02-13,3,-4.3,0.0,0.0,13.2\n2025-02-13,4,-4.3,0.0,0.0,12.4\n2025-02-13,5,-4.4,0.0,0.0,9.0\n2025-02-13,6,-4.1,0.0,0.0,6.0\n2025-02-13,7,-4.4,0.0,0.0,3.8\n2025-02-13,8,-4.4,0.0,0.0,4.6\n2025-02-13,9,-2.3,0.0,0.0,3.6\n2025-02-13,10,0.2,0.0,0.0,2.2\n2025-02-13,11,2.1,0.0,0.0,6.6\n2025-02-13,12,3.3,0.0,0.0,12.3\n2025-02-13,13,3.8,0.0,0.0,14.9\n2025-02-13,14,4.4,0.0,0.0,16.5\n2025-02-13,15,4.6,0.0,0.0,18.4\n2025-02-13,16,4.2,0.0,0.0,19.1\n2025-02-13,17,3.4,0.0,0.0,16.0\n2025-02-13,18,1.8,0.0,0.0,15.3\n2025-02-13,19,1.3,0.0,0.0,15.0\n2025-02-13,20,1.5,0.0,0.0,16.0\n2025-02-13,21,1.8,0.0,0.0,16.6\n2025-02-13,22,1.4,0.0,0.0,13.9\n2025-02-13,23,1.4,0.0,0.0,12.5\n2025-02-14,0,1.3,0.0,0.0,11.2\n2025-02-14,1,1.1,0.0,0.0,10.9\n2025-02-14,2,1.0,0.0,0.0,10.4\n2025-02-14,3,0.9,0.0,0.0,9.6\n2025-02-14,4,0.8,0.0,0.0,8.0\n2025-02-14,5,0.2,0.0,0.0,5.8\n2025-02-14,6,-0.2,0.0,0.0,2.4\n2025-02-14,7,-0.4,0.0,0.0,0.6\n2025-02-14,8,-0.1,0.0,0.0,3.6\n2025-02-14,9,2.0,0.0,0.0,5.8\n2025-02-14,10,3.1,0.0,0.0,6.3\n2025-02-14,11,5.2,0.0,0.0,6.6\n2025-02-14,12,6.5,0.0,0.0,7.3\n2025-02-14,13,7.4,0.0,0.0,9.2\n2025-02-14,14,7.6,0.0,0.0,10.9\n2025-02-14,15,7.0,0.0,0.0,11.6\n2025-02-14,16,6.3,0.0,0.0,10.9\n2025-02-14,17,5.0,0.0,0.0,8.3\n2025-02-14,18,1.7,0.0,0.0,5.2\n2025-02-14,19,0.5,0.0,0.0,5.6\n2025-02-14,20,1.1,0.0,0.0,7.9\n2025-02-14,21,-0.3,0.0,0.0,6.3\n2025-02-14,22,0.2,0.0,0.0,5.5\n2025-02-14,23,-0.6,0.0,0.0,5.1\n2025-02-15,0,-0.7,0.0,0.0,4.9\n2025-02-15,1,-1.2,0.0,0.0,4.5\n2025-02-15,2,-1.6,0.0,0.0,4.1\n2025-02-15,3,-2.1,0.0,0.0,4.6\n2025-02-15,4,-2.3,0.0,0.0,5.1\n2025-02-15,5,-1.6,0.0,0.0,6.1\n2025-02-15,6,-1.6,0.0,0.0,6.7\n2025-02-15,7,-1.6,0.0,0.0,7.7\n2025-02-15,8,-1.4,0.0,0.0,8.2\n2025-02-15,9,0.3,0.0,0.0,8.2\n2025-02-15,10,1.1,0.0,0.0,7.5\n2025-02-15,11,3.7,0.0,0.0,5.8\n2025-02-15,12,5.2,0.0,0.0,5.3\n2025-02-15,13,5.8,0.0,0.0,8.4\n2025-02-15,14,6.3,0.0,0.0,10.7\n2025-02-15,15,6.9,0.0,0.0,12.0\n2025-02-15,16,6.4,0.0,0.0,10.1\n2025-02-15,17,5.7,0.0,0.0,6.6\n2025-02-15,18,3.4,0.0,0.0,5.5\n2025-02-15,19,2.9,0.0,0.0,6.6\n2025-02-15,20,2.6,0.0,0.0,8.9\n2025-02-15,21,1.7,0.0,0.0,8.8\n2025-02-15,22,2.6,0.0,0.0,7.3\n2025-02-15,23,1.3,0.0,0.0,8.1\n2025-02-16,0,0.7,0.0,0.0,8.5\n2025-02-16,1,0.3,0.0,0.0,8.3\n2025-02-16,2,-0.1,0.0,0.0,8.0\n2025-02-16,3,-0.3,0.0,0.0,8.2\n2025-02-16,4,-0.4,0.0,0.0,9.0\n2025-02-16,5,-0.2,0.0,0.0,9.9\n2025-02-16,6,-0.2,0.0,0.0,10.7\n2025-02-16,7,-0.1,0.0,0.0,10.3\n2025-02-16,8,0.4,0.0,0.0,11.4\n2025-02-16,9,2.6,0.0,0.0,11.9\n2025-02-16,10,4.7,0.0,0.0,8.2\n2025-02-16,11,6.9,0.0,0.0,10.1\n2025-02-16,12,8.0,0.0,0.0,13.0\n2025-02-16,13,8.5,0.0,0.0,14.1\n2025-02-16,14,8.8,0.0,0.0,18.3\n2025-02-16,15,7.9,0.1,0.0,19.2\n2025-02-16,16,6.5,0.0,0.0,15.9\n2025-02-16,17,5.1,0.0,0.0,12.1\n2025-02-16,18,3.2,0.0,0.0,11.3\n2025-02-16,19,2.7,0.0,0.0,20.3\n2025-02-16,20,1.6,0.0,0.0,18.6\n2025-02-16,21,0.5,0.0,0.0,18.0\n2025-02-16,22,0.7,0.0,0.0,18.9\n2025-02-16,23,-0.4,0.0,0.0,21.3\n2025-02-17,0,-0.9,0.0,0.0,23.0\n2025-02-17,1,-1.3,0.0,0.0,22.9\n2025-02-17,2,-1.7,0.0,0.0,20.0\n2025-02-17,3,-2.1,0.0,0.0,17.5\n2025-02-17,4,-2.5,0.0,0.0,16.7\n2025-02-17,5,-2.6,0.0,0.0,15.7\n2025-02-17,6,-2.5,0.0,0.0,16.6\n2025-02-17,7,-2.6,0.0,0.0,16.8\n2025-02-17,8,-2.6,0.0,0.0,17.6\n2025-02-17,9,-1.9,0.0,0.0,18.6\n2025-02-17,10,0.3,0.0,0.0,19.1\n2025-02-17,11,0.9,0.0,0.0,18.2\n2025-02-17,12,1.4,0.0,0.0,17.4\n2025-02-17,13,1.9,0.0,0.0,18.0\n2025-02-17,14,2.1,0.0,0.0,19.1\n2025-02-17,15,2.0,0.0,0.0,19.8\n2025-02-17,16,1.7,0.0,0.0,19.7\n2025-02-17,17,0.9,0.0,0.0,18.9\n2025-02-17,18,-0.5,0.0,0.0,16.4\n2025-02-17,19,-1.9,0.0,0.0,16.4\n2025-02-17,20,-2.8,0.0,0.0,17.5\n2025-02-17,21,-3.6,0.0,0.0,16.9\n2025-02-17,22,-3.2,0.0,0.0,16.0\n2025-02-17,23,-3.4,0.0,0.0,15.2\n2025-02-18,0,-3.4,0.0,0.0,15.3\n2025-02-18,1,-3.5,0.0,0.0,17.2\n2025-02-18,2,-3.9,0.0,0.0,17.2\n2025-02-18,3,-4.2,0.0,0.0,16.1\n2025-02-18,4,-4.4,0.0,0.0,14.8\n2025-02-18,5,-4.4,0.0,0.0,14.3\n2025-02-18,6,-4.3,0.0,0.0,14.4\n2025-02-18,7,-4.3,0.0,0.0,14.9\n2025-02-18,8,-3.9,0.0,0.0,15.0\n2025-02-18,9,-2.5,0.0,0.0,16.2\n2025-02-18,10,-1.4,0.0,0.0,15.9\n2025-02-18,11,-0.3,0.0,0.0,18.2\n2025-02-18,12,0.5,0.0,0.0,19.6\n2025-02-18,13,1.0,0.0,0.0,20.3\n2025-02-18,14,1.3,0.0,0.0,21.4\n2025-02-18,15,1.2,0.0,0.0,21.4\n2025-02-18,16,0.8,0.0,0.0,20.5\n2025-02-18,17,-0.1,0.0,0.0,20.3\n2025-02-18,18,-1.5,0.0,0.0,17.5\n2025-02-18,19,-2.7,0.0,0.0,17.3\n2025-02-18,20,-3.8,0.0,0.0,18.5\n2025-02-18,21,-4.5,0.0,0.0,17.8\n2025-02-18,22,-4.1,0.0,0.0,16.7\n2025-02-18,23,-4.2,0.0,0.0,14.5\n2025-02-19,0,-4.2,0.0,0.0,14.1\n2025-02-19,1,-4.2,0.0,0.0,14.6\n2025-02-19,2,-4.4,0.0,0.0,13.9\n2025-02-19,3,-4.7,0.0,0.0,13.7\n2025-02-19,4,-5.0,0.0,0.0,13.7\n2025-02-19,5,-5.3,0.0,0.0,14.2\n2025-02-19,6,-5.6,0.0,0.0,14.2\n2025-02-19,7,-5.9,0.0,0.0,14.2\n2025-02-19,8,-5.9,0.0,0.0,15.3\n2025-02-19,9,-5.0,0.0,0.0,16.8\n2025-02-19,10,-2.8,0.0,0.0,16.2\n2025-02-19,11,-1.9,0.0,0.0,17.3\n2025-02-19,12,-1.1,0.0,0.0,17.1\n2025-02-19,13,-0.5,0.0,0.0,17.5\n2025-02-19,14,-0.2,0.0,0.0,18.2\n2025-02-19,15,0.3,0.0,0.0,18.5\n2025-02-19,16,0.3,0.0,0.0,18.9\n2025-02-19,17,-0.2,0.0,0.0,17.9\n2025-02-19,18,-1.4,0.0,0.0,15.0\n2025-02-19,19,-2.7,0.0,0.0,16.5\n2025-02-19,20,-3.6,0.0,0.0,17.6\n2025-02-19,21,-4.2,0.0,0.0,17.0\n2025-02-19,22,-4.1,0.0,0.0,14.4\n2025-02-19,23,-4.3,0.0,0.0,12.4\n2025-02-20,0,-4.4,0.0,0.0,10.5\n2025-02-20,1,-4.4,0.0,0.0,9.6\n2025-02-20,2,-4.4,0.0,0.0,9.7\n2025-02-20,3,-4.3,0.0,0.0,10.8\n2025-02-20,4,-4.5,0.0,0.0,11.1\n2025-02-20,5,-4.8,0.0,0.0,11.6\n2025-02-20,6,-5.0,0.0,0.0,11.8\n2025-02-20,7,-5.3,0.0,0.0,11.2\n2025-02-20,8,-5.1,0.0,0.0,11.2\n2025-02-20,9,-4.0,0.0,0.0,12.1\n2025-02-20,10,-2.6,0.0,0.0,10.4\n2025-02-20,11,-1.0,0.0,0.0,12.7\n2025-02-20,12,0.0,0.0,0.0,14.3\n2025-02-20,13,0.7,0.0,0.0,14.8\n2025-02-20,14,1.0,0.0,0.0,16.5\n2025-02-20,15,1.1,0.0,0.0,17.5\n2025-02-20,16,0.9,0.0,0.0,19.0\n2025-02-20,17,0.0,0.0,0.0,19.4\n2025-02-20,18,-1.4,0.0,0.0,15.2\n2025-02-20,19,-2.5,0.0,0.0,14.7\n2025-02-20,20,-3.3,0.0,0.0,16.7\n2025-02-20,21,-4.2,0.0,0.0,16.6\n2025-02-20,22,-4.1,0.0,0.0,15.2\n2025-02-20,23,-4.5,0.0,0.0,12.7\n2025-02-21,0,-4.7,0.0,0.0,10.4\n2025-02-21,1,-4.8,0.0,0.0,8.1\n2025-02-21,2,-4.8,0.0,0.0,7.4\n2025-02-21,3,-4.7,0.0,0.0,8.9\n2025-02-21,4,-4.7,0.0,0.0,9.4\n2025-02-21,5,-4.8,0.0,0.0,10.0\n2025-02-21,6,-4.9,0.0,0.0,10.3\n2025-02-21,7,-5.2,0.0,0.0,10.3\n2025-02-21,8,-4.9,0.0,0.0,10.0\n2025-02-21,9,-3.5,0.0,0.0,11.1\n2025-02-21,10,-2.3,0.0,0.0,8.9\n2025-02-21,11,-0.9,0.0,0.0,11.6\n2025-02-21,12,0.0,0.0,0.0,12.7\n2025-02-21,13,0.5,0.0,0.0,13.8\n2025-02-21,14,0.4,0.0,0.0,16.2\n2025-02-21,15,0.9,0.0,0.0,18.4\n2025-02-21,16,1.1,0.0,0.0,18.5\n2025-02-21,17,0.9,0.0,0.0,19.2\n2025-02-21,18,-0.6,0.0,0.0,16.9\n2025-02-21,19,-1.9,0.0,0.0,16.1\n2025-02-21,20,-2.6,0.0,0.0,16.1\n2025-02-21,21,-3.2,0.0,0.0,15.5\n2025-02-21,22,-3.2,0.0,0.0,13.0\n2025-02-21,23,-3.5,0.0,0.0,11.6\n2025-02-22,0,-3.8,0.0,0.0,8.4\n2025-02-22,1,-3.9,0.0,0.0,8.5\n2025-02-22,2,-4.0,0.0,0.0,9.0\n2025-02-22,3,-3.9,0.0,0.0,13.3\n2025-02-22,4,-4.3,0.0,0.0,14.3\n2025-02-22,5,-4.8,0.0,0.0,13.8\n2025-02-22,6,-5.2,0.0,0.0,12.1\n2025-02-22,7,-5.4,0.0,0.0,10.9\n2025-02-22,8,-5.0,0.0,0.0,10.0\n2025-02-22,9,-3.8,0.0,0.0,12.4\n2025-02-22,10,-1.8,0.0,0.0,10.6\n2025-02-22,11,-0.7,0.0,0.0,12.4\n2025-02-22,12,0.2,0.0,0.0,12.7\n2025-02-22,13,0.6,0.0,0.0,14.3\n2025-02-22,14,0.4,0.0,0.0,17.1\n2025-02-22,15,0.1,0.0,0.0,18.0\n2025-02-22,16,0.1,0.0,0.0,17.2\n2025-02-22,17,0.1,0.0,0.0,15.4\n2025-02-22,18,-0.6,0.0,0.0,12.6\n2025-02-22,19,-1.8,0.0,0.0,15.3\n2025-02-22,20,-2.9,0.0,0.0,15.7\n2025-02-22,21,-3.8,0.0,0.0,15.7\n2025-02-22,22,-4.0,0.0,0.0,15.9\n2025-02-22,23,-4.5,0.0,0.0,14.7\n2025-02-23,0,-4.6,0.0,0.0,13.1\n2025-02-23,1,-4.3,0.0,0.0,11.5\n2025-02-23,2,-4.0,0.0,0.0,14.4\n2025-02-23,3,-4.3,0.0,0.0,15.2\n2025-02-23,4,-4.7,0.0,0.0,15.0\n2025-02-23,5,-4.9,0.0,0.0,15.0\n2025-02-23,6,-5.2,0.0,0.0,15.0\n2025-02-23,7,-5.4,0.0,0.0,14.6\n2025-02-23,8,-5.1,0.0,0.0,14.7\n2025-02-23,9,-4.3,0.0,0.0,15.8\n2025-02-23,10,-2.9,0.0,0.0,19.7\n2025-02-23,11,-2.1,0.0,0.0,19.0\n2025-02-23,12,-1.1,0.0,0.0,19.2\n2025-02-23,13,-0.3,0.0,0.0,20.6\n2025-02-23,14,-0.5,0.0,0.0,20.8\n2025-02-23,15,-0.3,0.0,0.0,22.0\n2025-02-23,16,0.0,0.0,0.0,23.0\n2025-02-23,17,-1.0,0.0,0.0,22.0\n2025-02-23,18,-2.2,0.0,0.0,18.4\n2025-02-23,19,-3.4,0.0,0.0,17.4\n2025-02-23,20,-4.0,0.0,0.0,16.8\n2025-02-23,21,-4.4,0.0,0.0,15.9\n2025-02-23,22,-4.6,0.0,0.0,15.0\n2025-02-23,23,-4.6,0.0,0.0,14.1\n2025-02-24,0,-4.4,0.0,0.0,13.6\n2025-02-24,1,-4.1,0.0,0.0,13.8\n2025-02-24,2,-3.8,0.0,0.0,14.4\n2025-02-24,3,-4.0,0.0,0.0,13.9\n2025-02-24,4,-4.2,0.0,0.0,13.5\n2025-02-24,5,-4.5,0.0,0.0,13.2\n2025-02-24,6,-4.7,0.0,0.0,12.6\n2025-02-24,7,-4.8,0.0,0.0,11.8\n2025-02-24,8,-4.3,0.0,0.0,11.0\n2025-02-24,9,-3.0,0.0,0.0,13.2\n2025-02-24,10,-1.8,0.0,0.0,12.2\n2025-02-24,11,-0.4,0.0,0.0,12.0\n2025-02-24,12,1.1,0.0,0.0,11.8\n2025-02-24,13,1.9,0.0,0.0,13.6\n2025-02-24,14,2.1,0.0,0.0,16.2\n2025-02-24,15,2.4,0.0,0.0,17.4\n2025-02-24,16,2.6,0.0,0.0,16.8\n2025-02-24,17,2.3,0.0,0.0,16.6\n2025-02-24,18,1.0,0.0,0.0,16.9\n2025-02-24,19,0.2,0.0,0.0,16.8\n2025-02-24,20,0.2,0.0,0.0,15.3\n2025-02-24,21,0.0,0.0,0.0,13.2\n2025-02-24,22,-0.4,0.0,0.0,9.2\n2025-02-24,23,-1.3,0.0,0.0,4.9\n2025-02-25,0,-1.4,0.0,0.0,4.3\n2025-02-25,1,-0.8,0.0,0.0,6.4\n2025-02-25,2,-0.7,0.0,0.0,6.9\n2025-02-25,3,-0.7,0.0,0.0,7.5\n2025-02-25,4,-0.5,0.0,0.0,8.7\n2025-02-25,5,-0.8,0.0,0.0,5.4\n2025-02-25,6,-1.2,0.0,0.0,5.0\n2025-02-25,7,-0.5,0.0,0.0,7.9\n2025-02-25,8,-0.1,0.0,0.0,10.6\n2025-02-25,9,2.0,0.0,0.0,12.4\n2025-02-25,10,2.9,0.0,0.0,15.7\n2025-02-25,11,3.8,0.0,0.0,19.9\n2025-02-25,12,5.0,0.0,0.0,19.5\n2025-02-25,13,5.5,0.0,0.0,21.6\n2025-02-25,14,5.4,0.0,0.0,23.7\n2025-02-25,15,5.5,0.0,0.0,23.6\n2025-02-25,16,5.0,0.0,0.0,20.5\n2025-02-25,17,4.6,0.0,0.0,17.1\n2025-02-25,18,3.6,0.0,0.0,16.6\n2025-02-25,19,3.1,0.0,0.0,16.5\n2025-02-25,20,2.7,0.0,0.0,15.4\n2025-02-25,21,2.6,0.0,0.0,15.0\n2025-02-25,22,2.6,0.0,0.0,13.9\n2025-02-25,23,2.4,0.0,0.0,14.7\n2025-02-26,0,2.3,0.0,0.0,14.8\n2025-02-26,1,2.1,0.0,0.0,14.2\n2025-02-26,2,1.9,0.0,0.0,13.3\n2025-02-26,3,1.5,0.0,0.0,11.6\n2025-02-26,4,0.4,0.0,0.0,10.8\n2025-02-26,5,-0.9,0.0,0.0,8.1\n2025-02-26,6,-2.3,0.0,0.0,5.9\n2025-02-26,7,-3.0,0.0,0.0,6.1\n2025-02-26,8,-1.8,0.0,0.0,8.6\n2025-02-26,9,-0.1,0.0,0.0,6.9\n2025-02-26,10,2.8,0.0,0.0,9.5\n2025-02-26,11,4.3,0.0,0.0,11.0\n2025-02-26,12,5.5,0.0,0.0,11.6\n2025-02-26,13,6.2,0.0,0.0,13.0\n2025-02-26,14,6.4,0.0,0.0,14.5\n2025-02-26,15,6.1,0.0,0.0,14.9\n2025-02-26,16,5.2,0.0,0.0,14.2\n2025-02-26,17,3.9,0.0,0.0,12.7\n2025-02-26,18,2.0,0.0,0.0,10.9\n2025-02-26,19,0.5,0.0,0.0,10.6\n2025-02-26,20,-0.1,0.0,0.0,9.6\n2025-02-26,21,-0.4,0.0,0.0,8.4\n2025-02-26,22,0.1,0.0,0.0,8.5\n2025-02-26,23,-1.0,0.0,0.0,5.4\n2025-02-27,0,-0.8,0.0,0.0,4.9\n2025-02-27,1,-0.9,0.0,0.0,5.7\n2025-02-27,2,-1.5,0.0,0.0,7.9\n2025-02-27,3,-1.8,0.0,0.0,8.1\n2025-02-27,4,-2.0,0.0,0.0,7.7\n2025-02-27,5,-2.0,0.0,0.0,8.1\n2025-02-27,6,-1.8,0.0,0.0,9.1\n2025-02-27,7,-1.6,0.0,0.0,9.8\n2025-02-27,8,-0.7,0.0,0.0,10.7\n2025-02-27,9,2.1,0.0,0.0,8.5\n2025-02-27,10,4.2,0.0,0.0,9.0\n2025-02-27,11,6.6,0.0,0.0,10.0\n2025-02-27,12,8.6,0.0,0.0,9.4\n2025-02-27,13,10.1,0.0,0.0,10.5\n2025-02-27,14,10.8,0.0,0.0,9.1\n2025-02-27,15,11.1,0.0,0.0,8.1\n2025-02-27,16,11.1,0.0,0.0,7.5\n2025-02-27,17,9.5,0.0,0.0,4.0\n2025-02-27,18,5.9,0.0,0.0,6.9\n2025-02-27,19,3.5,0.0,0.0,4.0\n2025-02-27,20,2.6,0.0,0.0,2.7\n2025-02-27,21,1.7,0.0,0.0,4.8\n2025-02-27,22,2.5,0.0,0.0,2.2\n2025-02-27,23,1.9,0.0,0.0,5.2\n2025-02-28,0,1.7,0.0,0.0,5.8\n2025-02-28,1,1.4,0.0,0.0,4.5\n2025-02-28,2,2.5,0.0,0.0,3.1\n2025-02-28,3,2.7,0.0,0.0,2.9\n2025-02-28,4,2.8,0.0,0.0,3.1\n2025-02-28,5,3.0,0.0,0.0,4.1\n2025-02-28,6,3.2,0.0,0.0,6.2\n2025-02-28,7,2.3,0.0,0.0,6.7\n2025-02-28,8,3.3,0.0,0.0,6.2\n2025-02-28,9,4.8,0.0,0.0,7.8\n2025-02-28,10,6.2,0.0,0.0,3.7\n2025-02-28,11,6.9,0.0,0.0,3.6\n2025-02-28,12,7.2,0.0,0.0,4.2\n2025-02-28,13,7.5,0.0,0.0,4.8\n2025-02-28,14,7.8,0.0,0.0,3.1\n2025-02-28,15,8.6,0.0,0.0,3.5\n2025-02-28,16,9.6,0.0,0.0,5.4\n2025-02-28,17,9.3,0.0,0.0,3.8\n2025-02-28,18,7.4,0.0,0.0,1.2\n2025-02-28,19,5.7,0.0,0.0,0.4\n2025-02-28,20,4.4,0.0,0.0,1.0\n2025-02-28,21,3.5,0.0,0.0,1.3\n2025-02-28,22,3.6,0.0,0.0,4.6\n2025-02-28,23,2.6,0.0,0.0,4.7\n2025-03-01,0,2.0,0.0,0.0,5.1\n2025-03-01,1,2.3,0.0,0.0,7.5\n2025-03-01,2,0.6,0.0,0.0,5.7\n2025-03-01,3,0.1,0.0,0.0,5.6\n2025-03-01,4,0.3,0.0,0.0,6.6\n2025-03-01,5,-0.5,0.0,0.0,5.4\n2025-03-01,6,-0.6,0.0,0.0,5.0\n2025-03-01,7,-0.2,0.0,0.0,6.0\n2025-03-01,8,1.2,0.0,0.0,8.4\n2025-03-01,9,4.0,0.0,0.0,7.2\n2025-03-01,10,5.0,0.0,0.0,8.7\n2025-03-01,11,8.0,0.0,0.0,3.9\n2025-03-01,12,10.5,0.0,0.0,4.1\n2025-03-01,13,10.7,0.0,0.0,2.2\n2025-03-01,14,10.5,0.0,0.0,1.8\n2025-03-01,15,8.3,0.9,0.0,2.0\n2025-03-01,16,6.9,1.2,0.0,9.6\n2025-03-01,17,7.2,0.0,0.0,7.4\n2025-03-01,18,5.5,0.1,0.0,3.7\n2025-03-01,19,5.0,0.0,0.0,9.2\n2025-03-01,20,3.8,0.0,0.0,10.2\n2025-03-01,21,2.8,0.0,0.0,7.6\n2025-03-01,22,8.1,0.0,0.0,8.2\n2025-03-01,23,7.7,0.0,0.0,7.3\n2025-03-02,0,7.4,0.0,0.0,8.4\n2025-03-02,1,6.0,0.0,0.0,8.1\n2025-03-02,2,5.2,0.0,0.0,6.5\n2025-03-02,3,4.7,0.0,0.0,7.3\n2025-03-02,4,4.6,0.0,0.0,8.1\n2025-03-02,5,4.8,0.0,0.0,9.2\n2025-03-02,6,5.3,0.0,0.0,8.8\n2025-03-02,7,5.5,0.0,0.0,8.9\n2025-03-02,8,6.1,0.0,0.0,8.2\n2025-03-02,9,7.2,0.0,0.0,8.1\n2025-03-02,10,9.7,0.0,0.0,8.4\n2025-03-02,11,11.5,0.0,0.0,7.8\n2025-03-02,12,12.8,0.0,0.0,11.1\n2025-03-02,13,12.5,0.0,0.0,15.7\n2025-03-02,14,12.5,0.0,0.0,20.2\n2025-03-02,15,10.3,0.1,0.0,22.9\n2025-03-02,16,6.7,1.9,0.0,24.7\n2025-03-02,17,5.3,2.5,0.0,27.9\n2025-03-02,18,4.2,2.5,0.0,29.1\n2025-03-02,19,3.4,2.8,0.0,27.5\n2025-03-02,20,2.7,3.2,0.0,26.2\n2025-03-02,21,2.2,1.5,0.0,25.5\n2025-03-02,22,2.1,1.1,0.0,27.4\n2025-03-02,23,1.8,1.0,0.0,23.4\n2025-03-03,0,1.4,2.7,0.0,22.2\n2025-03-03,1,1.1,3.7,0.28,22.9\n2025-03-03,2,0.9,3.0,0.56,21.1\n2025-03-03,3,0.8,1.9,0.7,21.2\n2025-03-03,4,0.6,1.9,1.26,21.7\n2025-03-03,5,0.7,1.0,0.7,20.3\n2025-03-03,6,0.6,0.6,0.42,19.9\n2025-03-03,7,0.5,0.3,0.21,19.4\n2025-03-03,8,0.4,0.1,0.07,21.0\n2025-03-03,9,0.7,0.0,0.0,21.5\n2025-03-03,10,2.7,0.0,0.0,22.3\n2025-03-03,11,4.1,0.0,0.0,20.7\n2025-03-03,12,5.4,0.0,0.0,23.4\n2025-03-03,13,5.7,0.0,0.0,23.7\n2025-03-03,14,5.8,0.0,0.0,24.1\n2025-03-03,15,5.4,0.0,0.0,24.5\n2025-03-03,16,4.8,0.0,0.0,21.9\n2025-03-03,17,4.0,0.0,0.0,20.3\n2025-03-03,18,2.6,0.0,0.0,20.4\n2025-03-03,19,1.2,0.0,0.0,19.4\n2025-03-03,20,0.4,0.0,0.0,19.6\n2025-03-03,21,-0.1,0.0,0.0,17.8\n2025-03-03,22,0.9,0.0,0.0,17.2\n2025-03-03,23,0.5,0.0,0.0,14.9\n2025-03-04,0,0.4,0.0,0.0,14.4\n2025-03-04,1,0.4,0.0,0.0,13.5\n2025-03-04,2,0.4,0.0,0.0,13.7\n2025-03-04,3,0.3,0.0,0.0,12.8\n2025-03-04,4,0.4,0.0,0.0,13.7\n2025-03-04,5,0.6,0.0,0.0,14.9\n2025-03-04,6,0.8,0.0,0.0,16.9\n2025-03-04,7,0.9,0.0,0.0,14.5\n2025-03-04,8,1.2,0.0,0.0,14.4\n2025-03-04,9,1.2,0.5,0.35,14.0\n2025-03-04,10,1.3,0.8,0.56,14.4\n2025-03-04,11,1.3,1.3,0.91,12.9\n2025-03-04,12,1.1,0.7,0.49,13.7\n2025-03-04,13,1.7,0.2,0.14,14.4\n2025-03-04,14,2.3,0.1,0.0,15.8\n2025-03-04,15,3.0,0.0,0.0,17.4\n2025-03-04,16,3.1,0.0,0.0,17.0\n2025-03-04,17,2.9,0.0,0.0,19.3\n2025-03-04,18,2.4,0.0,0.0,17.7\n2025-03-04,19,2.0,0.0,0.0,12.1\n2025-03-04,20,2.1,0.0,0.0,12.1\n2025-03-04,21,2.0,0.0,0.0,14.5\n2025-03-04,22,1.9,0.0,0.0,16.2\n2025-03-04,23,1.7,0.0,0.0,13.8\n2025-03-05,0,1.7,0.0,0.0,13.0\n2025-03-05,1,1.6,0.0,0.0,10.2\n2025-03-05,2,1.5,0.0,0.0,11.7\n2025-03-05,3,1.4,0.0,0.0,10.8\n2025-03-05,4,1.0,0.0,0.0,10.8\n2025-03-05,5,1.2,0.0,0.0,9.6\n2025-03-05,6,1.1,0.0,0.0,7.5\n2025-03-05,7,1.0,0.0,0.0,7.5\n2025-03-05,8,1.1,0.0,0.0,8.3\n2025-03-05,9,1.3,0.0,0.0,8.3\n2025-03-05,10,2.8,0.0,0.0,9.3\n2025-03-05,11,3.3,0.0,0.0,10.6\n2025-03-05,12,4.2,0.0,0.0,10.7\n2025-03-05,13,5.5,0.0,0.0,11.3\n2025-03-05,14,5.3,0.0,0.0,11.0\n2025-03-05,15,5.6,0.0,0.0,10.1\n2025-03-05,16,6.6,0.0,0.0,10.6\n2025-03-05,17,6.4,0.0,0.0,10.8\n2025-03-05,18,5.5,0.0,0.0,6.9\n2025-03-05,19,4.3,0.0,0.0,7.8\n2025-03-05,20,3.3,0.0,0.0,8.1\n2025-03-05,21,2.2,0.0,0.0,9.8\n2025-03-05,22,3.8,0.0,0.0,11.1\n2025-03-05,23,3.0,0.0,0.0,13.2\n2025-03-06,0,1.5,0.0,0.0,11.0\n2025-03-06,1,0.5,0.0,0.0,8.8\n2025-03-06,2,-0.1,0.0,0.0,8.1\n2025-03-06,3,-0.7,0.0,0.0,9.4\n2025-03-06,4,-0.9,0.0,0.0,10.1\n2025-03-06,5,-1.1,0.0,0.0,8.0\n2025-03-06,6,-1.4,0.0,0.0,8.3\n2025-03-06,7,-1.6,0.0,0.0,5.4\n2025-03-06,8,-0.8,0.0,0.0,5.9\n2025-03-06,9,1.2,0.0,0.0,5.2\n2025-03-06,10,3.3,0.0,0.0,4.6\n2025-03-06,11,5.4,0.0,0.0,5.1\n2025-03-06,12,7.0,0.0,0.0,7.5\n2025-03-06,13,8.2,0.0,0.0,10.8\n2025-03-06,14,8.6,0.0,0.0,13.2\n2025-03-06,15,8.1,0.0,0.0,13.8\n2025-03-06,16,7.5,0.0,0.0,13.9\n2025-03-06,17,6.9,0.0,0.0,12.4\n2025-03-06,18,5.9,0.0,0.0,11.5\n2025-03-06,19,4.5,0.0,0.0,11.9\n2025-03-06,20,3.9,0.0,0.0,15.0\n2025-03-06,21,2.9,0.0,0.0,14.6\n2025-03-06,22,2.6,0.0,0.0,12.2\n2025-03-06,23,1.7,0.0,0.0,9.9\n2025-03-07,0,1.0,0.0,0.0,7.6\n2025-03-07,1,0.3,0.0,0.0,7.0\n2025-03-07,2,-0.7,0.0,0.0,4.7\n2025-03-07,3,-1.6,0.0,0.0,3.6\n2025-03-07,4,-1.7,0.0,0.0,4.0\n2025-03-07,5,-2.1,0.0,0.0,1.5\n2025-03-07,6,-2.3,0.0,0.0,0.9\n2025-03-07,7,-2.5,0.0,0.0,0.4\n2025-03-07,8,-1.4,0.0,0.0,2.4\n2025-03-07,9,1.5,0.0,0.0,2.7\n2025-03-07,10,4.2,0.0,0.0,3.6\n2025-03-07,11,6.3,0.0,0.0,3.2\n2025-03-07,12,7.4,0.0,0.0,4.0\n2025-03-07,13,8.0,0.0,0.0,6.0\n2025-03-07,14,8.1,0.0,0.0,8.7\n2025-03-07,15,7.9,0.0,0.0,9.2\n2025-03-07,16,7.6,0.0,0.0,8.8\n2025-03-07,17,6.3,0.0,0.0,8.2\n2025-03-07,18,4.2,0.0,0.0,4.7\n2025-03-07,19,2.2,0.0,0.0,4.2\n2025-03-07,20,2.2,0.0,0.0,0.7\n2025-03-07,21,1.8,0.0,0.0,0.8\n2025-03-07,22,3.0,0.0,0.0,3.1\n2025-03-07,23,2.7,0.0,0.0,3.1\n2025-03-08,0,2.8,0.0,0.0,5.5\n2025-03-08,1,2.2,0.0,0.0,8.8\n2025-03-08,2,1.9,0.0,0.0,9.1\n2025-03-08,3,1.8,0.0,0.0,7.9\n2025-03-08,4,1.5,0.0,0.0,7.2\n2025-03-08,5,1.4,0.0,0.0,6.5\n2025-03-08,6,0.8,0.0,0.0,4.2\n2025-03-08,7,0.8,0.0,0.0,3.4\n2025-03-08,8,1.9,0.0,0.0,4.6\n2025-03-08,9,3.8,0.0,0.0,4.3\n2025-03-08,10,5.1,0.0,0.0,0.0\n2025-03-08,11,6.6,0.0,0.0,1.8\n2025-03-08,12,7.5,0.0,0.0,5.4\n2025-03-08,13,8.0,0.0,0.0,8.9\n2025-03-08,14,8.0,0.0,0.0,11.0\n2025-03-08,15,8.0,0.0,0.0,11.4\n2025-03-08,16,7.7,0.0,0.0,12.6\n2025-03-08,17,6.9,0.0,0.0,11.9\n2025-03-08,18,5.7,0.0,0.0,7.8\n2025-03-08,19,4.0,0.0,0.0,6.7\n2025-03-08,20,3.3,0.0,0.0,6.1\n2025-03-08,21,2.8,0.0,0.0,6.2\n2025-03-08,22,3.0,0.0,0.0,4.2\n2025-03-08,23,2.2,0.0,0.0,4.2\n2025-03-09,0,1.8,0.0,0.0,3.1\n2025-03-09,1,1.5,0.0,0.0,1.3\n2025-03-09,2,1.0,0.0,0.0,0.3\n2025-03-09,3,0.3,0.0,0.0,0.3\n2025-03-09,4,-0.1,0.0,0.0,0.0\n2025-03-09,5,-0.7,0.0,0.0,0.3\n2025-03-09,6,-1.0,0.0,0.0,1.5\n2025-03-09,7,-0.9,0.0,0.0,3.3\n2025-03-09,8,-0.2,0.0,0.0,3.3\n2025-03-09,9,1.5,0.0,0.0,3.5\n2025-03-09,10,3.2,0.0,0.0,4.3\n2025-03-09,11,6.0,0.0,0.0,2.3\n2025-03-09,12,8.0,0.0,0.0,0.0\n2025-03-09,13,9.2,0.0,0.0,2.6\n2025-03-09,14,9.9,0.0,0.0,4.9\n2025-03-09,15,10.1,0.0,0.0,6.7\n2025-03-09,16,9.8,0.0,0.0,7.8\n2025-03-09,17,8.8,0.0,0.0,7.5\n2025-03-09,18,7.0,0.0,0.0,5.8\n2025-03-09,19,4.8,0.0,0.0,5.4\n2025-03-09,20,3.1,0.0,0.0,4.1\n2025-03-09,21,2.0,0.0,0.0,2.8\n2025-03-09,22,3.2,0.0,0.0,3.2\n2025-03-09,23,2.6,0.0,0.0,5.2\n2025-03-10,0,2.0,0.0,0.0,5.2\n2025-03-10,1,1.4,0.0,0.0,3.8\n2025-03-10,2,0.4,0.0,0.0,3.1\n2025-03-10,3,-0.6,0.0,0.0,4.6\n2025-03-10,4,-0.8,0.0,0.0,4.4\n2025-03-10,5,-0.9,0.0,0.0,4.4\n2025-03-10,6,-0.8,0.0,0.0,3.2\n2025-03-10,7,-0.6,0.0,0.0,2.5\n2025-03-10,8,0.5,0.0,0.0,3.1\n2025-03-10,9,3.2,0.0,0.0,3.0\n2025-03-10,10,6.8,0.0,0.0,3.1\n2025-03-10,11,9.7,0.0,0.0,4.5\n2025-03-10,12,11.5,0.0,0.0,6.7\n2025-03-10,13,12.1,0.0,0.0,11.8\n2025-03-10,14,11.9,0.0,0.0,12.6\n2025-03-10,15,11.4,0.0,0.0,12.7\n2025-03-10,16,10.3,0.0,0.0,12.0\n2025-03-10,17,8.8,0.0,0.0,9.6\n2025-03-10,18,7.2,0.0,0.0,7.6\n2025-03-10,19,5.4,0.0,0.0,6.0\n2025-03-10,20,4.2,0.0,0.0,4.0\n2025-03-10,21,3.8,0.0,0.0,2.2\n2025-03-10,22,4.2,0.0,0.0,1.4\n2025-03-10,23,3.7,0.0,0.0,0.6\n2025-03-11,0,3.3,0.0,0.0,0.2\n2025-03-11,1,2.6,0.0,0.0,0.9\n2025-03-11,2,1.8,0.0,0.0,2.5\n2025-03-11,3,1.2,0.0,0.0,2.4\n2025-03-11,4,0.9,0.0,0.0,1.0\n2025-03-11,5,0.5,0.0,0.0,1.8\n2025-03-11,6,0.2,0.0,0.0,3.3\n2025-03-11,7,0.5,0.0,0.0,3.5\n2025-03-11,8,1.8,0.0,0.0,4.5\n2025-03-11,9,5.3,0.0,0.0,3.0\n2025-03-11,10,7.5,0.0,0.0,3.1\n2025-03-11,11,9.8,0.0,0.0,0.4\n2025-03-11,12,11.2,0.0,0.0,1.7\n2025-03-11,13,12.3,0.0,0.0,0.6\n2025-03-11,14,13.0,0.0,0.0,3.3\n2025-03-11,15,13.2,0.0,0.0,4.5\n2025-03-11,16,12.8,0.0,0.0,10.1\n2025-03-11,17,11.5,0.0,0.0,7.9\n2025-03-11,18,9.8,0.0,0.0,10.0\n2025-03-11,19,7.6,0.0,0.0,9.4\n2025-03-11,20,6.1,0.0,0.0,5.6\n2025-03-11,21,5.4,0.0,0.0,4.2\n2025-03-11,22,6.5,0.0,0.0,2.7\n2025-03-11,23,6.0,0.0,0.0,1.4\n2025-03-12,0,5.9,0.0,0.0,1.4\n2025-03-12,1,5.0,0.0,0.0,0.3\n2025-03-12,2,4.2,0.0,0.0,3.3\n2025-03-12,3,3.9,0.0,0.0,4.1\n2025-03-12,4,3.5,0.0,0.0,2.6\n2025-03-12,5,3.5,0.0,0.0,5.1\n2025-03-12,6,3.2,0.0,0.0,1.3\n2025-03-12,7,3.6,0.0,0.0,3.2\n2025-03-12,8,4.8,0.0,0.0,10.2\n2025-03-12,9,6.9,0.0,0.0,8.7\n2025-03-12,10,7.1,0.0,0.0,7.8\n2025-03-12,11,10.0,0.0,0.0,11.6\n2025-03-12,12,10.1,0.0,0.0,12.9\n2025-03-12,13,9.7,0.0,0.0,13.1\n2025-03-12,14,9.7,0.0,0.0,13.3\n2025-03-12,15,9.7,0.0,0.0,14.7\n2025-03-12,16,9.0,0.0,0.0,17.3\n2025-03-12,17,7.5,0.0,0.0,15.4\n2025-03-12,18,6.9,0.0,0.0,12.6\n2025-03-12,19,6.2,0.0,0.0,12.1\n2025-03-12,20,5.3,0.0,0.0,9.9\n2025-03-12,21,4.9,0.0,0.0,8.1\n2025-03-12,22,5.8,0.0,0.0,6.7\n2025-03-12,23,4.9,0.0,0.0,4.2\n2025-03-13,0,4.1,0.0,0.0,2.3\n2025-03-13,1,3.0,0.0,0.0,6.4\n2025-03-13,2,2.4,0.0,0.0,7.9\n2025-03-13,3,2.2,0.0,0.0,8.0\n2025-03-13,4,2.3,0.0,0.0,7.9\n2025-03-13,5,1.7,0.0,0.0,3.5\n2025-03-13,6,1.0,0.0,0.0,4.8\n2025-03-13,7,1.1,0.0,0.0,5.2\n2025-03-13,8,2.3,0.0,0.0,5.0\n2025-03-13,9,5.1,0.0,0.0,4.2\n2025-03-13,10,7.0,0.0,0.0,3.6\n2025-03-13,11,9.4,0.0,0.0,5.6\n2025-03-13,12,11.1,0.0,0.0,7.2\n2025-03-13,13,12.2,0.0,0.0,8.5\n2025-03-13,14,13.1,0.0,0.0,9.9\n2025-03-13,15,13.4,0.0,0.0,10.2\n2025-03-13,16,13.3,0.0,0.0,9.2\n2025-03-13,17,12.3,0.0,0.0,7.0\n2025-03-13,18,9.9,0.0,0.0,7.5\n2025-03-13,19,7.4,0.0,0.0,6.7\n2025-03-13,20,7.3,0.0,0.0,8.8\n2025-03-13,21,7.1,0.0,0.0,9.7\n2025-03-13,22,7.0,0.0,0.0,9.5\n2025-03-13,23,6.6,0.0,0.0,11.7\n2025-03-14,0,6.0,0.0,0.0,11.7\n2025-03-14,1,5.5,0.0,0.0,11.1\n2025-03-14,2,4.9,0.0,0.0,11.2\n2025-03-14,3,4.5,0.0,0.0,11.7\n2025-03-14,4,3.9,0.0,0.0,10.7\n2025-03-14,5,3.7,0.0,0.0,11.6\n2025-03-14,6,3.7,0.0,0.0,11.3\n2025-03-14,7,3.4,0.0,0.0,10.0\n2025-03-14,8,4.6,0.0,0.0,10.2\n2025-03-14,9,7.3,0.0,0.0,8.1\n2025-03-14,10,10.5,0.0,0.0,8.0\n2025-03-14,11,13.2,0.0,0.0,9.5\n2025-03-14,12,14.7,0.0,0.0,9.9\n2025-03-14,13,15.6,0.0,0.0,7.9\n2025-03-14,14,16.3,0.0,0.0,7.0\n2025-03-14,15,16.7,0.0,0.0,7.8\n2025-03-14,16,16.9,0.0,0.0,7.3\n2025-03-14,17,16.2,0.0,0.0,7.4\n2025-03-14,18,13.9,0.0,0.0,6.0\n2025-03-14,19,11.1,0.0,0.0,5.8\n2025-03-14,20,10.1,0.0,0.0,5.9\n2025-03-14,21,9.8,0.0,0.0,10.7\n2025-03-14,22,9.9,0.0,0.0,11.6\n2025-03-14,23,9.3,0.0,0.0,8.9\n2025-03-15,0,8.8,0.0,0.0,10.5\n2025-03-15,1,8.6,0.0,0.0,9.2\n2025-03-15,2,8.0,0.0,0.0,9.4\n2025-03-15,3,7.5,0.0,0.0,10.3\n2025-03-15,4,7.2,0.0,0.0,11.5\n2025-03-15,5,6.9,0.0,0.0,11.2\n2025-03-15,6,6.6,0.0,0.0,12.2\n2025-03-15,7,6.5,0.0,0.0,12.8\n2025-03-15,8,6.9,0.0,0.0,12.6\n2025-03-15,9,8.1,0.0,0.0,14.0\n2025-03-15,10,10.7,0.0,0.0,20.2\n2025-03-15,11,12.1,0.0,0.0,19.6\n2025-03-15,12,13.0,0.0,0.0,18.0\n2025-03-15,13,13.8,0.0,0.0,15.0\n2025-03-15,14,15.0,0.0,0.0,9.9\n2025-03-15,15,15.3,0.0,0.0,6.5\n2025-03-15,16,14.8,0.0,0.0,3.1\n2025-03-15,17,13.8,0.0,0.0,4.0\n2025-03-15,18,11.6,0.0,0.0,4.3\n2025-03-15,19,10.1,0.0,0.0,6.8\n2025-03-15,20,9.1,0.0,0.0,6.2\n2025-03-15,21,8.3,0.0,0.0,3.1\n2025-03-15,22,8.6,0.2,0.0,3.5\n2025-03-15,23,8.2,0.1,0.0,11.4\n2025-03-16,0,7.8,0.0,0.0,12.5\n2025-03-16,1,7.2,0.0,0.0,10.2\n2025-03-16,2,6.7,0.0,0.0,10.8\n2025-03-16,3,6.5,0.0,0.0,10.4\n2025-03-16,4,6.1,0.0,0.0,8.8\n2025-03-16,5,5.7,0.1,0.0,8.9\n2025-03-16,6,5.0,0.2,0.0,7.3\n2025-03-16,7,4.5,0.1,0.0,6.8\n2025-03-16,8,4.5,0.0,0.0,12.1\n2025-03-16,9,3.2,0.1,0.0,18.0\n2025-03-16,10,4.1,0.1,0.0,19.3\n2025-03-16,11,4.5,0.0,0.0,19.4\n2025-03-16,12,5.5,0.0,0.0,20.8\n2025-03-16,13,6.0,0.0,0.0,22.9\n2025-03-16,14,6.3,0.0,0.0,25.9\n2025-03-16,15,5.7,0.0,0.0,28.7\n2025-03-16,16,4.6,0.0,0.0,30.0\n2025-03-16,17,3.6,0.0,0.0,28.6\n2025-03-16,18,2.8,0.0,0.0,24.1\n2025-03-16,19,1.9,0.0,0.0,23.2\n2025-03-16,20,1.7,0.0,0.0,25.8\n2025-03-16,21,1.2,0.0,0.0,25.0\n2025-03-16,22,1.4,0.0,0.0,18.6\n2025-03-16,23,0.9,0.0,0.0,18.5\n2025-03-17,0,0.4,0.0,0.0,18.2\n2025-03-17,1,-0.1,0.0,0.0,18.5\n2025-03-17,2,-0.6,0.0,0.0,17.5\n2025-03-17,3,-1.2,0.0,0.0,16.8\n2025-03-17,4,-1.7,0.0,0.0,15.2\n2025-03-17,5,-2.3,0.0,0.0,13.1\n2025-03-17,6,-2.7,0.0,0.0,11.5\n2025-03-17,7,-2.8,0.0,0.0,11.1\n2025-03-17,8,-1.5,0.0,0.0,9.3\n2025-03-17,9,0.4,0.0,0.0,11.9\n2025-03-17,10,2.2,0.0,0.0,13.0\n2025-03-17,11,3.3,0.0,0.0,11.5\n2025-03-17,12,3.8,0.0,0.0,14.3\n2025-03-17,13,3.7,0.0,0.0,14.9\n2025-03-17,14,4.1,0.0,0.0,12.8\n2025-03-17,15,3.9,0.0,0.0,14.7\n2025-03-17,16,3.8,0.0,0.0,13.3\n2025-03-17,17,3.6,0.0,0.0,10.5\n2025-03-17,18,3.0,0.0,0.0,7.6\n2025-03-17,19,2.5,0.0,0.0,3.6\n2025-03-17,20,2.4,0.0,0.0,7.1\n2025-03-17,21,2.2,0.0,0.0,12.3\n2025-03-17,22,1.3,0.1,0.07,11.6\n2025-03-17,23,0.9,0.3,0.21,11.4\n2025-03-18,0,0.7,1.9,1.33,18.9\n2025-03-18,1,0.1,2.6,1.82,20.0\n2025-03-18,2,-0.3,0.0,0.0,17.8\n2025-03-18,3,-0.1,1.2,0.84,12.2\n2025-03-18,4,0.2,0.0,0.0,13.9\n2025-03-18,5,0.8,0.1,0.07,15.8\n2025-03-18,6,0.7,0.1,0.07,7.1\n2025-03-18,7,-1.9,0.0,0.0,3.1\n2025-03-18,8,-0.2,0.0,0.0,14.4\n2025-03-18,9,0.2,0.1,0.07,18.2\n2025-03-18,10,1.1,0.1,0.07,18.1\n2025-03-18,11,1.4,0.3,0.21,21.1\n2025-03-18,12,1.8,0.0,0.0,21.7\n2025-03-18,13,2.2,0.0,0.0,23.1\n2025-03-18,14,2.6,0.0,0.0,24.6\n2025-03-18,15,2.9,0.0,0.0,23.7\n2025-03-18,16,2.8,0.0,0.0,22.1\n2025-03-18,17,2.4,0.0,0.0,18.4\n2025-03-18,18,1.5,0.0,0.0,17.1\n2025-03-18,19,-0.1,0.0,0.0,16.7\n2025-03-18,20,-0.9,0.0,0.0,13.8\n2025-03-18,21,-1.5,0.0,0.0,11.5\n2025-03-18,22,-0.6,0.0,0.0,12.0\n2025-03-18,23,-1.1,0.0,0.0,9.5\n2025-03-19,0,-1.4,0.0,0.0,7.2\n2025-03-19,1,-1.6,0.0,0.0,6.1\n2025-03-19,2,-2.4,0.0,0.0,6.9\n2025-03-19,3,-3.0,0.0,0.0,8.4\n2025-03-19,4,-3.4,0.0,0.0,7.7\n2025-03-19,5,-3.5,0.0,0.0,7.5\n2025-03-19,6,-3.5,0.0,0.0,7.3\n2025-03-19,7,-3.4,0.0,0.0,6.8\n2025-03-19,8,-2.2,0.0,0.0,5.2\n2025-03-19,9,-0.3,0.0,0.0,5.1\n2025-03-19,10,2.8,0.0,0.0,6.3\n2025-03-19,11,3.9,0.0,0.0,11.6\n2025-03-19,12,4.4,0.0,0.0,13.4\n2025-03-19,13,4.5,0.0,0.0,14.8\n2025-03-19,14,4.7,0.0,0.0,15.0\n2025-03-19,15,4.6,0.0,0.0,14.9\n2025-03-19,16,4.3,0.0,0.0,15.0\n2025-03-19,17,3.7,0.0,0.0,14.3\n2025-03-19,18,3.0,0.0,0.0,12.6\n2025-03-19,19,1.9,0.0,0.0,9.6\n2025-03-19,20,1.6,0.0,0.0,6.6\n2025-03-19,21,1.4,0.0,0.0,6.6\n2025-03-19,22,2.0,0.0,0.0,6.5\n2025-03-19,23,1.3,0.0,0.0,5.0\n2025-03-20,0,1.4,0.0,0.0,7.4\n2025-03-20,1,0.8,0.0,0.0,8.4\n2025-03-20,2,0.2,0.0,0.0,9.4\n2025-03-20,3,-0.2,0.0,0.0,9.5\n2025-03-20,4,-0.4,0.0,0.0,9.9\n2025-03-20,5,-0.6,0.0,0.0,10.8\n2025-03-20,6,-0.7,0.0,0.0,11.5\n2025-03-20,7,-0.4,0.0,0.0,12.6\n2025-03-20,8,1.5,0.0,0.0,13.7\n2025-03-20,9,4.4,0.0,0.0,11.1\n2025-03-20,10,6.2,0.0,0.0,11.2\n2025-03-20,11,8.5,0.0,0.0,11.1\n2025-03-20,12,10.3,0.0,0.0,12.9\n2025-03-20,13,11.7,0.0,0.0,14.9\n2025-03-20,14,12.3,0.0,0.0,18.1\n2025-03-20,15,11.8,0.0,0.0,20.5\n2025-03-20,16,10.8,0.0,0.0,20.0\n2025-03-20,17,10.0,0.0,0.0,17.6\n2025-03-20,18,8.8,0.0,0.0,14.5\n2025-03-20,19,7.4,0.0,0.0,14.9\n2025-03-20,20,6.7,0.0,0.0,14.8\n2025-03-20,21,6.1,0.0,0.0,14.8\n2025-03-20,22,6.5,0.0,0.0,14.1\n2025-03-20,23,6.0,0.0,0.0,13.1\n2025-03-21,0,5.5,0.0,0.0,11.6\n2025-03-21,1,5.1,0.0,0.0,10.7\n2025-03-21,2,4.8,0.0,0.0,11.4\n2025-03-21,3,4.5,0.0,0.0,11.4\n2025-03-21,4,4.3,0.0,0.0,11.5\n2025-03-21,5,4.0,0.0,0.0,12.7\n2025-03-21,6,3.9,0.0,0.0,13.1\n2025-03-21,7,3.8,0.0,0.0,11.5\n2025-03-21,8,5.5,0.0,0.0,13.0\n2025-03-21,9,8.4,0.0,0.0,11.2\n2025-03-21,10,10.9,0.0,0.0,10.2\n2025-03-21,11,12.4,0.0,0.0,12.4\n2025-03-21,12,12.9,0.0,0.0,14.3\n2025-03-21,13,13.5,0.0,0.0,15.1\n2025-03-21,14,13.6,0.0,0.0,15.7\n2025-03-21,15,13.3,0.0,0.0,15.3\n2025-03-21,16,12.2,0.0,0.0,14.9\n2025-03-21,17,10.3,0.0,0.0,13.4\n2025-03-21,18,8.5,0.0,0.0,11.6\n2025-03-21,19,6.9,0.0,0.0,12.1\n2025-03-21,20,6.4,0.0,0.0,11.9\n2025-03-21,21,6.3,0.0,0.0,12.1\n2025-03-21,22,7.3,0.0,0.0,11.9\n2025-03-21,23,7.3,0.0,0.0,10.4\n2025-03-22,0,6.7,0.0,0.0,10.2\n2025-03-22,1,6.1,0.0,0.0,9.8\n2025-03-22,2,5.9,0.0,0.0,10.3\n2025-03-22,3,5.6,0.0,0.0,10.7\n2025-03-22,4,5.6,0.0,0.0,11.4\n2025-03-22,5,5.6,0.0,0.0,11.7\n2025-03-22,6,5.6,0.0,0.0,11.9\n2025-03-22,7,5.7,0.0,0.0,11.9\n2025-03-22,8,7.5,0.0,0.0,12.3\n2025-03-22,9,10.3,0.0,0.0,10.3\n2025-03-22,10,12.6,0.0,0.0,12.1\n2025-03-22,11,14.0,0.0,0.0,14.3\n2025-03-22,12,14.7,0.0,0.0,15.4\n2025-03-22,13,15.1,0.0,0.0,15.5\n2025-03-22,14,15.0,0.0,0.0,15.3\n2025-03-22,15,14.5,0.0,0.0,15.3\n2025-03-22,16,13.5,0.0,0.0,16.0\n2025-03-22,17,12.3,0.0,0.0,14.6\n2025-03-22,18,10.7,0.0,0.0,13.6\n2025-03-22,19,9.0,0.0,0.0,13.9\n2025-03-22,20,8.3,0.0,0.0,13.6\n2025-03-22,21,7.9,0.0,0.0,10.3\n2025-03-22,22,8.5,0.0,0.0,10.2\n2025-03-22,23,7.8,0.0,0.0,10.5\n2025-03-23,0,7.2,0.0,0.0,9.7\n2025-03-23,1,6.8,0.0,0.0,9.1\n2025-03-23,2,6.5,0.0,0.0,8.8\n2025-03-23,3,6.3,0.0,0.0,9.3\n2025-03-23,4,5.9,0.0,0.0,8.5\n2025-03-23,5,5.4,0.0,0.0,9.7\n2025-03-23,6,5.2,0.0,0.0,10.6\n2025-03-23,7,5.4,0.0,0.0,11.7\n2025-03-23,8,7.3,0.0,0.0,11.6\n2025-03-23,9,10.3,0.0,0.0,9.0\n2025-03-23,10,13.1,0.0,0.0,8.7\n2025-03-23,11,15.7,0.0,0.0,9.0\n2025-03-23,12,17.7,0.0,0.0,9.4\n2025-03-23,13,19.1,0.0,0.0,9.9\n2025-03-23,14,20.2,0.0,0.0,10.7\n2025-03-23,15,19.5,0.0,0.0,11.3\n2025-03-23,16,17.2,0.0,0.0,10.0\n2025-03-23,17,15.7,0.0,0.0,9.1\n2025-03-23,18,14.4,0.0,0.0,8.6\n2025-03-23,19,12.8,0.0,0.0,9.4\n2025-03-23,20,11.8,0.0,0.0,10.5\n2025-03-23,21,11.1,0.0,0.0,11.3\n2025-03-23,22,11.3,0.0,0.0,9.4\n2025-03-23,23,10.7,0.0,0.0,11.2\n2025-03-24,0,10.4,0.0,0.0,12.1\n2025-03-24,1,10.3,0.0,0.0,11.1\n2025-03-24,2,10.0,0.0,0.0,11.6\n2025-03-24,3,9.7,0.0,0.0,10.1\n2025-03-24,4,9.4,0.0,0.0,9.0\n2025-03-24,5,9.2,0.0,0.0,8.0\n2025-03-24,6,8.9,0.8,0.0,9.8\n2025-03-24,7,8.8,0.0,0.0,15.2\n2025-03-24,8,9.7,0.0,0.0,15.7\n2025-03-24,9,12.2,0.0,0.0,12.0\n2025-03-24,10,15.0,0.0,0.0,13.3\n2025-03-24,11,15.2,0.0,0.0,12.9\n2025-03-24,12,15.6,0.0,0.0,11.8\n2025-03-24,13,16.1,0.0,0.0,13.2\n2025-03-24,14,16.0,0.0,0.0,15.3\n2025-03-24,15,15.4,0.0,0.0,16.3\n2025-03-24,16,14.1,0.0,0.0,16.7\n2025-03-24,17,12.2,0.0,0.0,14.4\n2025-03-24,18,11.0,0.0,0.0,11.6\n2025-03-24,19,10.4,0.0,0.0,12.6\n2025-03-24,20,9.8,0.0,0.0,14.0\n2025-03-24,21,8.6,0.0,0.0,12.7\n2025-03-24,22,9.5,0.0,0.0,12.5\n2025-03-24,23,8.5,0.0,0.0,10.1\n2025-03-25,0,7.4,0.0,0.0,7.5\n2025-03-25,1,6.8,0.0,0.0,6.9\n2025-03-25,2,6.5,0.0,0.0,8.4\n2025-03-25,3,6.2,0.0,0.0,9.4\n2025-03-25,4,5.9,0.0,0.0,8.1\n2025-03-25,5,5.7,0.0,0.0,8.3\n2025-03-25,6,5.8,0.0,0.0,11.1\n2025-03-25,7,6.2,0.0,0.0,12.6\n2025-03-25,8,6.9,0.0,0.0,14.5\n2025-03-25,9,8.2,0.0,0.0,14.4\n2025-03-25,10,11.3,0.0,0.0,12.7\n2025-03-25,11,14.3,0.0,0.0,13.9\n2025-03-25,12,16.3,0.0,0.0,14.9\n2025-03-25,13,18.9,0.1,0.0,24.0\n2025-03-25,14,17.4,0.0,0.0,28.8\n2025-03-25,15,13.8,0.0,0.0,29.4\n2025-03-25,16,12.1,0.0,0.0,22.1\n2025-03-25,17,11.0,0.0,0.0,19.3\n2025-03-25,18,9.6,0.0,0.0,17.4\n2025-03-25,19,8.2,0.0,0.0,17.8\n2025-03-25,20,7.3,0.0,0.0,14.7\n2025-03-25,21,6.9,0.0,0.0,13.2\n2025-03-25,22,7.3,0.0,0.0,11.5\n2025-03-25,23,6.7,0.0,0.0,11.9\n2025-03-26,0,6.2,0.0,0.0,9.2\n2025-03-26,1,5.7,0.0,0.0,6.0\n2025-03-26,2,5.3,0.0,0.0,8.7\n2025-03-26,3,4.9,0.0,0.0,10.0\n2025-03-26,4,4.7,0.0,0.0,6.8\n2025-03-26,5,3.7,0.0,0.0,3.6\n2025-03-26,6,3.8,0.0,0.0,8.0\n2025-03-26,7,4.0,0.0,0.0,10.2\n2025-03-26,8,5.8,0.0,0.0,10.0\n2025-03-26,9,9.0,0.0,0.0,7.7\n2025-03-26,10,12.0,0.0,0.0,9.6\n2025-03-26,11,14.4,0.0,0.0,10.3\n2025-03-26,12,16.6,0.0,0.0,12.3\n2025-03-26,13,18.5,0.0,0.0,12.6\n2025-03-26,14,20.0,0.0,0.0,11.9\n2025-03-26,15,21.1,0.0,0.0,11.3\n2025-03-26,16,21.8,0.0,0.0,12.1\n2025-03-26,17,21.3,0.0,0.0,5.9\n2025-03-26,18,18.6,0.0,0.0,3.6\n2025-03-26,19,15.3,0.0,0.0,5.0\n2025-03-26,20,15.7,0.0,0.0,9.5\n2025-03-26,21,15.2,0.0,0.0,10.8\n2025-03-26,22,15.1,0.0,0.0,9.4\n2025-03-26,23,14.2,0.0,0.0,9.9\n2025-03-27,0,14.2,0.0,0.0,13.9\n2025-03-27,1,13.8,0.0,0.0,11.5\n2025-03-27,2,12.7,0.0,0.0,11.2\n2025-03-27,3,11.4,0.1,0.0,15.1\n2025-03-27,4,10.3,1.0,0.0,12.2\n2025-03-27,5,10.2,0.0,0.0,11.0\n2025-03-27,6,10.7,0.0,0.0,11.0\n2025-03-27,7,10.3,0.0,0.0,11.9\n2025-03-27,8,9.4,0.0,0.0,7.6\n2025-03-27,9,9.5,5.1,0.0,7.0\n2025-03-27,10,11.1,1.4,0.0,10.4\n2025-03-27,11,13.6,0.0,0.0,11.6\n2025-03-27,12,14.8,0.0,0.0,15.1\n2025-03-27,13,17.1,0.0,0.0,13.8\n2025-03-27,14,18.9,0.0,0.0,14.9\n2025-03-27,15,18.6,0.0,0.0,18.7\n2025-03-27,16,17.3,0.0,0.0,17.2\n2025-03-27,17,15.1,0.0,0.0,18.4\n2025-03-27,18,12.7,0.0,0.0,21.9\n2025-03-27,19,10.4,0.0,0.0,19.3\n2025-03-27,20,8.8,0.0,0.0,18.0\n2025-03-27,21,8.3,0.0,0.0,18.2\n2025-03-27,22,6.9,0.2,0.0,15.5\n2025-03-27,23,6.6,0.0,0.0,14.9\n2025-03-28,0,6.3,0.0,0.0,16.1\n2025-03-28,1,5.7,0.0,0.0,16.8\n2025-03-28,2,4.9,0.0,0.0,17.2\n2025-03-28,3,4.1,0.0,0.0,16.2\n2025-03-28,4,3.6,0.0,0.0,14.9\n2025-03-28,5,2.7,0.0,0.0,15.7\n2025-03-28,6,2.2,0.0,0.0,16.6\n2025-03-28,7,2.2,0.0,0.0,16.5\n2025-03-28,8,3.1,0.0,0.0,14.6\n2025-03-28,9,4.1,0.0,0.0,19.7\n2025-03-28,10,6.4,0.0,0.0,20.9\n2025-03-28,11,8.0,0.0,0.0,22.8\n2025-03-28,12,9.1,0.0,0.0,23.1\n2025-03-28,13,10.1,0.0,0.0,23.1\n2025-03-28,14,10.3,0.0,0.0,24.5\n2025-03-28,15,10.1,0.0,0.0,23.7\n2025-03-28,16,9.6,0.0,0.0,22.3\n2025-03-28,17,8.8,0.0,0.0,21.9\n2025-03-28,18,7.5,0.0,0.0,18.1\n2025-03-28,19,5.5,0.0,0.0,17.3\n2025-03-28,20,4.2,0.0,0.0,18.4\n2025-03-28,21,3.1,0.0,0.0,17.2\n2025-03-28,22,2.8,0.0,0.0,15.7\n2025-03-28,23,2.3,0.0,0.0,14.0\n2025-03-29,0,1.4,0.0,0.0,11.5\n2025-03-29,1,0.7,0.0,0.0,8.5\n2025-03-29,2,-0.2,0.0,0.0,5.3\n2025-03-29,3,-0.9,0.0,0.0,4.5\n2025-03-29,4,-1.0,0.0,0.0,1.2\n2025-03-29,5,-1.0,0.0,0.0,5.1\n2025-03-29,6,-1.7,0.0,0.0,1.8\n2025-03-29,7,-1.0,0.0,0.0,1.3\n2025-03-29,8,1.9,0.0,0.0,3.6\n2025-03-29,9,4.2,0.5,0.35,7.4\n2025-03-29,10,4.5,0.4,0.28,19.5\n2025-03-29,11,4.3,0.9,0.63,26.2\n2025-03-29,12,4.9,0.1,0.07,21.9\n2025-03-29,13,5.6,0.1,0.0,23.7\n2025-03-29,14,5.7,0.1,0.0,26.7\n2025-03-29,15,5.6,0.0,0.0,27.0\n2025-03-29,16,5.1,0.0,0.0,26.7\n2025-03-29,17,4.1,0.0,0.0,25.0\n2025-03-29,18,3.2,0.0,0.0,20.1\n2025-03-29,19,1.5,0.0,0.0,15.8\n2025-03-29,20,1.2,0.0,0.0,17.4\n2025-03-29,21,0.5,0.0,0.0,15.0\n2025-03-29,22,-0.1,0.0,0.0,14.8\n2025-03-29,23,-0.4,0.0,0.0,14.0\n2025-03-30,0,-0.6,0.0,0.0,13.5\n2025-03-30,1,-0.9,0.0,0.0,11.4\n2025-03-30,2,-1.3,0.0,0.0,9.1\n2025-03-30,3,-1.3,0.0,0.0,8.8\n2025-03-30,4,-1.4,0.0,0.0,6.6\n2025-03-30,5,-2.1,0.0,0.0,4.6\n2025-03-30,6,-2.1,0.0,0.0,3.7\n2025-03-30,7,-1.3,0.0,0.0,3.1\n2025-03-30,8,1.0,0.0,0.0,0.9\n2025-03-30,9,3.1,0.2,0.14,9.9\n2025-03-30,10,4.1,0.1,0.07,17.0\n2025-03-30,11,4.6,0.1,0.07,18.9\n2025-03-30,12,5.3,0.1,0.0,22.3\n2025-03-30,13,5.3,0.2,0.07,23.6\n2025-03-30,14,6.0,0.1,0.0,21.6\n2025-03-30,15,6.3,0.0,0.0,22.0\n2025-03-30,16,6.2,0.0,0.0,20.7\n2025-03-30,17,5.5,0.0,0.0,17.6\n2025-03-30,18,5.0,0.0,0.0,12.0\n2025-03-30,19,3.5,0.0,0.0,7.5\n2025-03-30,20,3.0,0.0,0.0,8.2\n2025-03-30,21,2.8,0.0,0.0,5.4\n2025-03-30,22,2.8,0.0,0.0,7.3\n2025-03-30,23,2.6,0.0,0.0,6.8\n2025-03-31,0,2.2,0.0,0.0,5.1\n2025-03-31,1,2.1,0.0,0.0,4.4\n2025-03-31,2,1.8,0.0,0.0,5.2\n2025-03-31,3,1.3,0.0,0.0,6.7\n2025-03-31,4,0.7,0.0,0.0,5.2\n2025-03-31,5,0.1,0.0,0.0,5.6\n2025-03-31,6,-0.5,0.0,0.0,6.9\n2025-03-31,7,-0.1,0.0,0.0,8.5\n2025-03-31,8,2.3,0.0,0.0,8.2\n2025-03-31,9,5.0,0.0,0.0,7.1\n2025-03-31,10,7.1,0.0,0.0,6.1\n2025-03-31,11,8.7,0.0,0.0,5.8\n2025-03-31,12,9.9,0.0,0.0,8.2\n2025-03-31,13,10.5,0.0,0.0,11.3\n2025-03-31,14,10.9,0.0,0.0,12.2\n2025-03-31,15,10.3,0.0,0.0,13.1\n2025-03-31,16,9.9,0.0,0.0,11.4\n2025-03-31,17,9.2,0.0,0.0,10.7\n2025-03-31,18,7.9,0.0,0.0,8.0\n2025-03-31,19,6.4,0.0,0.0,8.6\n2025-03-31,20,5.9,0.0,0.0,8.6\n2025-03-31,21,5.7,0.0,0.0,8.8\n2025-03-31,22,5.6,0.0,0.0,9.2\n2025-03-31,23,5.2,0.0,0.0,9.3\n2025-04-01,0,4.7,0.0,0.0,7.7\n2025-04-01,1,4.2,0.0,0.0,7.6\n2025-04-01,2,3.8,0.0,0.0,7.1\n2025-04-01,3,3.2,0.0,0.0,6.7\n2025-04-01,4,2.6,0.0,0.0,8.9\n2025-04-01,5,2.1,0.0,0.0,9.8\n2025-04-01,6,1.9,0.0,0.0,9.7\n2025-04-01,7,2.2,0.0,0.0,8.9\n2025-04-01,8,4.4,0.0,0.0,8.1\n2025-04-01,9,7.2,0.0,0.0,8.5\n2025-04-01,10,9.3,0.0,0.0,8.0\n2025-04-01,11,11.1,0.0,0.0,6.9\n2025-04-01,12,12.5,0.0,0.0,6.5\n2025-04-01,13,13.3,0.0,0.0,7.0\n2025-04-01,14,13.6,0.0,0.0,9.3\n2025-04-01,15,13.3,0.0,0.0,9.5\n2025-04-01,16,12.9,0.0,0.0,9.5\n2025-04-01,17,11.9,0.0,0.0,8.4\n2025-04-01,18,10.1,0.0,0.0,6.5\n2025-04-01,19,7.8,0.0,0.0,5.1\n2025-04-01,20,6.3,0.0,0.0,2.5\n2025-04-01,21,6.0,0.0,0.0,0.4\n2025-04-01,22,6.8,0.0,0.0,5.6\n2025-04-01,23,6.1,0.0,0.0,5.7\n2025-04-02,0,5.4,0.0,0.0,0.6\n2025-04-02,1,4.1,0.0,0.0,3.4\n2025-04-02,2,3.4,0.0,0.0,4.0\n2025-04-02,3,3.2,0.0,0.0,5.3\n2025-04-02,4,2.4,0.0,0.0,3.9\n2025-04-02,5,2.0,0.0,0.0,2.4\n2025-04-02,6,1.9,0.0,0.0,5.1\n2025-04-02,7,2.5,0.0,0.0,4.8\n2025-04-02,8,5.0,0.0,0.0,4.0\n2025-04-02,9,8.0,0.0,0.0,2.9\n2025-04-02,10,10.8,0.0,0.0,1.7\n2025-04-02,11,11.8,0.0,0.0,4.4\n2025-04-02,12,12.1,0.0,0.0,9.0\n2025-04-02,13,10.8,0.0,0.0,15.7\n2025-04-02,14,9.0,0.0,0.0,11.3\n2025-04-02,15,8.7,0.0,0.0,9.5\n2025-04-02,16,6.6,0.1,0.0,13.2\n2025-04-02,17,5.5,0.3,0.0,6.8\n2025-04-02,18,5.7,0.1,0.0,2.6\n2025-04-02,19,5.1,0.0,0.0,2.0\n2025-04-02,20,4.8,0.0,0.0,6.0\n2025-04-02,21,5.3,0.0,0.0,8.3\n2025-04-02,22,6.5,0.0,0.0,5.2\n2025-04-02,23,5.0,0.0,0.0,5.2\n2025-04-03,0,3.7,0.0,0.0,5.6\n2025-04-03,1,3.2,0.0,0.0,2.9\n2025-04-03,2,2.8,0.0,0.0,3.4\n2025-04-03,3,2.3,0.0,0.0,4.2\n2025-04-03,4,2.0,0.0,0.0,2.7\n2025-04-03,5,1.2,0.0,0.0,4.6\n2025-04-03,6,1.0,0.0,0.0,4.8\n2025-04-03,7,1.7,0.0,0.0,5.8\n2025-04-03,8,3.6,0.0,0.0,4.0\n2025-04-03,9,6.4,0.0,0.0,2.3\n2025-04-03,10,9.3,0.0,0.0,2.1\n2025-04-03,11,11.1,0.0,0.0,1.0\n2025-04-03,12,12.4,0.0,0.0,1.0\n2025-04-03,13,13.6,0.0,0.0,4.7\n2025-04-03,14,12.9,0.0,0.0,13.7\n2025-04-03,15,12.4,0.0,0.0,14.9\n2025-04-03,16,11.6,0.0,0.0,16.0\n2025-04-03,17,11.2,0.0,0.0,13.5\n2025-04-03,18,9.7,0.0,0.0,10.7\n2025-04-03,19,8.0,0.0,0.0,8.1\n2025-04-03,20,6.8,0.0,0.0,4.8\n2025-04-03,21,6.5,0.0,0.0,6.7\n2025-04-03,22,6.6,0.0,0.0,6.7\n2025-04-03,23,6.0,0.0,0.0,6.3\n2025-04-04,0,5.4,0.0,0.0,1.4\n2025-04-04,1,4.8,0.0,0.0,4.5\n2025-04-04,2,4.6,0.0,0.0,7.4\n2025-04-04,3,4.9,0.0,0.0,7.8\n2025-04-04,4,4.7,0.0,0.0,7.6\n2025-04-04,5,5.2,0.0,0.0,4.7\n2025-04-04,6,4.0,0.0,0.0,3.6\n2025-04-04,7,4.7,0.0,0.0,4.0\n2025-04-04,8,6.1,0.0,0.0,5.2\n2025-04-04,9,8.0,0.0,0.0,5.9\n2025-04-04,10,9.8,0.0,0.0,6.5\n2025-04-04,11,11.3,0.0,0.0,8.3\n2025-04-04,12,12.1,0.0,0.0,11.3\n2025-04-04,13,12.2,0.0,0.0,12.2\n2025-04-04,14,12.1,0.0,0.0,11.4\n2025-04-04,15,11.9,0.0,0.0,11.7\n2025-04-04,16,11.6,0.0,0.0,10.5\n2025-04-04,17,10.5,0.0,0.0,10.1\n2025-04-04,18,9.2,0.0,0.0,7.7\n2025-04-04,19,7.4,0.0,0.0,1.5\n2025-04-04,20,6.8,0.0,0.0,0.5\n2025-04-04,21,6.0,0.0,0.0,3.6\n2025-04-04,22,6.5,0.0,0.0,3.2\n2025-04-04,23,6.7,0.0,0.0,3.9\n2025-04-05,0,6.6,0.2,0.0,3.6\n2025-04-05,1,6.3,0.1,0.0,1.2\n2025-04-05,2,6.0,1.0,0.0,6.0\n2025-04-05,3,5.7,0.0,0.0,9.3\n2025-04-05,4,6.0,0.0,0.0,10.5\n2025-04-05,5,6.1,0.0,0.0,6.5\n2025-04-05,6,6.2,0.0,0.0,5.7\n2025-04-05,7,6.4,0.0,0.0,8.0\n2025-04-05,8,6.8,0.3,0.0,8.3\n2025-04-05,9,6.6,0.8,0.0,9.1\n2025-04-05,10,7.3,2.1,0.0,4.6\n2025-04-05,11,7.6,2.7,0.0,3.2\n2025-04-05,12,7.7,2.0,0.0,3.0\n2025-04-05,13,8.4,0.5,0.0,6.4\n2025-04-05,14,8.9,0.5,0.0,8.0\n2025-04-05,15,9.7,0.1,0.0,8.9\n2025-04-05,16,8.6,0.5,0.0,11.4\n2025-04-05,17,8.2,0.4,0.0,8.9\n2025-04-05,18,8.6,0.0,0.0,6.9\n2025-04-05,19,7.7,1.7,0.0,6.4\n2025-04-05,20,7.4,0.1,0.0,13.2\n2025-04-05,21,6.8,0.0,0.0,9.9\n2025-04-05,22,6.4,0.1,0.0,3.8\n2025-04-05,23,6.2,0.0,0.0,8.1\n2025-04-06,0,6.5,0.0,0.0,7.2\n2025-04-06,1,5.8,0.0,0.0,9.3\n2025-04-06,2,6.0,0.0,0.0,9.7\n2025-04-06,3,6.2,0.0,0.0,11.5\n2025-04-06,4,6.0,0.0,0.0,9.9\n2025-04-06,5,5.8,0.0,0.0,8.5\n2025-04-06,6,4.6,0.0,0.0,3.4\n2025-04-06,7,4.8,0.0,0.0,5.4\n2025-04-06,8,6.5,0.0,0.0,11.3\n2025-04-06,9,8.2,0.0,0.0,7.1\n2025-04-06,10,9.6,0.0,0.0,7.4\n2025-04-06,11,11.2,0.0,0.0,10.0\n2025-04-06,12,11.9,0.0,0.0,13.9\n2025-04-06,13,12.4,0.0,0.0,16.1\n2025-04-06,14,12.4,0.0,0.0,18.0\n2025-04-06,15,12.1,0.0,0.0,18.5\n2025-04-06,16,11.4,0.0,0.0,18.3\n2025-04-06,17,10.6,0.0,0.0,16.0\n2025-04-06,18,9.4,0.0,0.0,13.6\n2025-04-06,19,7.9,0.0,0.0,13.5\n2025-04-06,20,7.3,0.0,0.0,13.4\n2025-04-06,21,6.9,0.0,0.0,11.2\n2025-04-06,22,7.5,0.0,0.0,10.5\n2025-04-06,23,7.1,0.0,0.0,9.1\n2025-04-07,0,6.8,0.0,0.0,8.1\n2025-04-07,1,6.2,0.0,0.0,6.3\n2025-04-07,2,5.7,0.0,0.0,5.4\n2025-04-07,3,5.2,0.0,0.0,6.8\n2025-04-07,4,4.7,0.0,0.0,7.6\n2025-04-07,5,4.2,0.0,0.0,8.3\n2025-04-07,6,3.9,0.0,0.0,8.9\n2025-04-07,7,4.3,0.0,0.0,10.3\n2025-04-07,8,6.8,0.0,0.0,10.8\n2025-04-07,9,9.7,0.0,0.0,9.5\n2025-04-07,10,11.7,0.0,0.0,8.9\n2025-04-07,11,14.0,0.0,0.0,9.9\n2025-04-07,12,15.6,0.0,0.0,12.1\n2025-04-07,13,16.9,0.0,0.0,14.8\n2025-04-07,14,17.4,0.0,0.0,17.1\n2025-04-07,15,17.6,0.0,0.0,19.2\n2025-04-07,16,17.1,0.0,0.0,19.3\n2025-04-07,17,15.9,0.0,0.0,18.7\n2025-04-07,18,14.4,0.0,0.0,18.7\n2025-04-07,19,12.4,0.0,0.0,19.7\n2025-04-07,20,11.3,0.0,0.0,19.4\n2025-04-07,21,10.7,0.0,0.0,16.5\n2025-04-07,22,11.8,0.0,0.0,15.5\n2025-04-07,23,10.7,0.0,0.0,16.2\n2025-04-08,0,10.2,0.0,0.0,17.6\n2025-04-08,1,9.7,0.0,0.0,18.1\n2025-04-08,2,9.0,3.0,0.0,15.3\n2025-04-08,3,8.5,0.0,0.0,11.9\n2025-04-08,4,7.9,0.0,0.0,11.5\n2025-04-08,5,7.1,0.0,0.0,8.0\n2025-04-08,6,5.5,0.0,0.0,4.4\n2025-04-08,7,6.6,0.0,0.0,4.2\n2025-04-08,8,8.8,0.0,0.0,5.1\n2025-04-08,9,11.2,0.0,0.0,7.2\n2025-04-08,10,12.6,0.0,0.0,10.8\n2025-04-08,11,13.9,0.0,0.0,13.8\n2025-04-08,12,14.8,0.0,0.0,15.3\n2025-04-08,13,15.4,0.0,0.0,16.9\n2025-04-08,14,15.2,0.0,0.0,17.9\n2025-04-08,15,14.7,0.0,0.0,17.1\n2025-04-08,16,14.4,0.0,0.0,15.1\n2025-04-08,17,13.5,0.0,0.0,12.7\n2025-04-08,18,11.9,0.0,0.0,9.3\n2025-04-08,19,10.4,0.0,0.0,9.4\n2025-04-08,20,9.5,0.0,0.0,7.0\n2025-04-08,21,8.7,0.0,0.0,4.8\n2025-04-08,22,8.9,0.0,0.0,2.3\n2025-04-08,23,8.3,0.0,0.0,2.7\n2025-04-09,0,7.8,0.0,0.0,4.0\n2025-04-09,1,6.8,0.0,0.0,5.2\n2025-04-09,2,6.2,0.0,0.0,6.6\n2025-04-09,3,5.8,0.0,0.0,8.4\n2025-04-09,4,5.7,0.0,0.0,11.0\n2025-04-09,5,5.7,0.0,0.0,10.3\n2025-04-09,6,5.9,0.0,0.0,7.5\n2025-04-09,7,6.4,0.0,0.0,7.7\n2025-04-09,8,8.7,0.0,0.0,8.8\n2025-04-09,9,11.6,0.0,0.0,7.3\n2025-04-09,10,12.8,0.0,0.0,7.9\n2025-04-09,11,14.1,0.0,0.0,7.8\n2025-04-09,12,14.3,0.0,0.0,6.9\n2025-04-09,13,15.7,0.0,0.0,8.0\n2025-04-09,14,14.4,0.0,0.0,12.1\n2025-04-09,15,12.6,0.0,0.0,12.4\n2025-04-09,16,10.5,0.4,0.0,6.9\n2025-04-09,17,11.0,0.0,0.0,4.0\n2025-04-09,18,11.1,0.0,0.0,4.0\n2025-04-09,19,10.5,0.0,0.0,4.0\n2025-04-09,20,9.8,0.0,0.0,4.4\n2025-04-09,21,9.1,0.0,0.0,2.2\n2025-04-09,22,9.2,0.0,0.0,6.4\n2025-04-09,23,9.0,0.1,0.0,2.3\n2025-04-10,0,8.3,0.1,0.0,1.0\n2025-04-10,1,7.7,0.0,0.0,3.9\n2025-04-10,2,6.9,0.0,0.0,2.1\n2025-04-10,3,6.8,0.0,0.0,0.9\n2025-04-10,4,6.2,0.0,0.0,1.9\n2025-04-10,5,5.1,0.0,0.0,2.1\n2025-04-10,6,4.6,0.0,0.0,2.6\n2025-04-10,7,5.6,0.0,0.0,1.4\n2025-04-10,8,7.7,0.0,0.0,0.2\n2025-04-10,9,10.0,0.0,0.0,1.8\n2025-04-10,10,12.4,0.0,0.0,4.7\n2025-04-10,11,13.8,0.0,0.0,6.4\n2025-04-10,12,15.3,0.0,0.0,8.1\n2025-04-10,13,15.9,0.0,0.0,11.2\n2025-04-10,14,15.0,0.0,0.0,13.5\n2025-04-10,15,15.6,0.0,0.0,12.9\n2025-04-10,16,14.7,0.0,0.0,14.5\n2025-04-10,17,13.4,0.0,0.0,13.5\n2025-04-10,18,12.3,0.0,0.0,9.5\n2025-04-10,19,10.6,0.0,0.0,6.8\n2025-04-10,20,9.5,0.0,0.0,4.9\n2025-04-10,21,8.6,0.0,0.0,3.6\n2025-04-10,22,8.9,0.0,0.0,4.2\n2025-04-10,23,8.3,0.0,0.0,3.6\n2025-04-11,0,7.8,0.0,0.0,3.0\n2025-04-11,1,7.4,0.0,0.0,1.9\n2025-04-11,2,7.1,0.0,0.0,4.3\n2025-04-11,3,6.5,0.0,0.0,5.9\n2025-04-11,4,5.9,0.0,0.0,2.8\n2025-04-11,5,6.0,0.0,0.0,2.8\n2025-04-11,6,5.7,0.0,0.0,4.1\n2025-04-11,7,7.0,0.0,0.0,5.5\n2025-04-11,8,9.6,0.0,0.0,4.2\n2025-04-11,9,12.6,0.0,0.0,2.3\n2025-04-11,10,15.2,0.0,0.0,2.1\n2025-04-11,11,17.4,0.0,0.0,1.2\n2025-04-11,12,18.9,0.0,0.0,4.0\n2025-04-11,13,19.1,0.0,0.0,9.7\n2025-04-11,14,18.8,0.0,0.0,9.9\n2025-04-11,15,18.9,0.0,0.0,8.7\n2025-04-11,16,18.5,0.0,0.0,7.1\n2025-04-11,17,17.9,0.0,0.0,6.8\n2025-04-11,18,16.0,0.0,0.0,6.2\n2025-04-11,19,13.6,0.0,0.0,4.6\n2025-04-11,20,11.7,0.0,0.0,3.9\n2025-04-11,21,10.7,0.0,0.0,0.7\n2025-04-11,22,12.9,0.0,0.0,3.5\n2025-04-11,23,12.0,0.0,0.0,2.0\n2025-04-12,0,10.9,0.0,0.0,9.7\n2025-04-12,1,10.6,0.0,0.0,11.5\n2025-04-12,2,10.9,0.0,0.0,13.2\n2025-04-12,3,11.4,0.0,0.0,12.1\n2025-04-12,4,11.3,0.0,0.0,12.1\n2025-04-12,5,11.0,0.0,0.0,12.0\n2025-04-12,6,11.3,0.0,0.0,12.5\n2025-04-12,7,11.7,0.0,0.0,12.1\n2025-04-12,8,13.7,0.0,0.0,12.7\n2025-04-12,9,16.2,0.0,0.0,16.6\n2025-04-12,10,18.4,0.0,0.0,18.3\n2025-04-12,11,20.6,0.0,0.0,18.5\n2025-04-12,12,21.7,0.0,0.0,18.3\n2025-04-12,13,18.6,0.1,0.0,18.6\n2025-04-12,14,16.4,0.1,0.0,11.2\n2025-04-12,15,16.1,0.1,0.0,11.4\n2025-04-12,16,17.7,0.1,0.0,22.7\n2025-04-12,17,15.0,0.8,0.0,23.1\n2025-04-12,18,11.7,1.6,0.0,18.6\n2025-04-12,19,7.7,2.7,0.0,14.5\n2025-04-12,20,6.6,2.8,0.0,23.9\n2025-04-12,21,4.8,2.1,0.0,19.9\n2025-04-12,22,2.0,2.5,0.91,19.4\n2025-04-12,23,1.6,3.0,1.54,15.8\n2025-04-13,0,1.4,1.9,0.91,7.7\n2025-04-13,1,1.2,1.2,0.42,0.2\n2025-04-13,2,1.7,0.1,0.0,5.8\n2025-04-13,3,2.2,0.0,0.0,9.5\n2025-04-13,4,2.3,0.0,0.0,12.1\n2025-04-13,5,2.5,0.0,0.0,19.7\n2025-04-13,6,4.5,0.0,0.0,19.9\n2025-04-13,7,5.6,0.0,0.0,22.3\n2025-04-13,8,6.7,0.0,0.0,26.5\n2025-04-13,9,7.1,0.0,0.0,34.4\n2025-04-13,10,6.4,0.1,0.0,33.6\n2025-04-13,11,6.2,0.1,0.0,31.6\n2025-04-13,12,6.7,0.3,0.07,32.8\n2025-04-13,13,6.8,0.0,0.0,31.3\n2025-04-13,14,6.5,0.4,0.07,28.7\n2025-04-13,15,6.5,0.1,0.0,28.4\n2025-04-13,16,6.5,0.0,0.0,25.2\n2025-04-13,17,6.6,0.0,0.0,25.3\n2025-04-13,18,6.5,0.0,0.0,21.7\n2025-04-13,19,5.8,0.0,0.0,19.7\n2025-04-13,20,5.3,0.1,0.0,21.0\n2025-04-13,21,6.0,0.0,0.0,18.1\n2025-04-13,22,5.0,0.2,0.0,17.7\n2025-04-13,23,4.6,0.3,0.0,18.6\n2025-04-14,0,4.6,0.2,0.0,20.9\n2025-04-14,1,4.2,0.1,0.0,13.1\n2025-04-14,2,4.2,0.0,0.0,14.6\n2025-04-14,3,3.8,0.6,0.21,12.8\n2025-04-14,4,3.0,1.3,0.42,4.9\n2025-04-14,5,3.3,1.9,0.7,12.9\n2025-04-14,6,2.8,0.0,0.0,17.2\n2025-04-14,7,3.3,0.0,0.0,19.2\n2025-04-14,8,4.2,0.3,0.0,15.4\n2025-04-14,9,3.8,1.4,0.42,15.7\n2025-04-14,10,5.1,0.7,0.0,13.2\n2025-04-14,11,5.9,0.5,0.0,10.8\n2025-04-14,12,7.5,0.3,0.0,8.8\n2025-04-14,13,7.2,0.4,0.0,10.4\n2025-04-14,14,6.2,0.5,0.0,13.9\n2025-04-14,15,6.1,0.2,0.0,14.4\n2025-04-14,16,7.0,0.9,0.0,11.4\n2025-04-14,17,5.4,1.1,0.0,15.5\n2025-04-14,18,4.9,1.0,0.0,9.3\n2025-04-14,19,5.3,0.1,0.0,11.0\n2025-04-14,20,5.2,0.1,0.0,8.3\n2025-04-14,21,4.0,0.1,0.0,10.4\n2025-04-14,22,3.9,0.0,0.0,12.9\n2025-04-14,23,3.4,0.0,0.0,11.9\n2025-04-15,0,3.0,0.0,0.0,10.3\n2025-04-15,1,2.5,0.0,0.0,9.4\n2025-04-15,2,1.8,0.0,0.0,6.9\n2025-04-15,3,0.8,0.0,0.0,5.5\n2025-04-15,4,0.7,0.0,0.0,3.1\n2025-04-15,5,0.9,0.0,0.0,2.9\n2025-04-15,6,1.3,0.0,0.0,4.0\n2025-04-15,7,3.4,0.0,0.0,6.8\n2025-04-15,8,5.6,0.0,0.0,7.6\n2025-04-15,9,7.5,0.0,0.0,8.6\n2025-04-15,10,9.4,0.0,0.0,8.9\n2025-04-15,11,10.8,0.0,0.0,11.5\n2025-04-15,12,11.3,0.0,0.0,14.7\n2025-04-15,13,11.2,0.0,0.0,16.4\n2025-04-15,14,11.1,0.0,0.0,17.5\n2025-04-15,15,10.9,0.0,0.0,17.2\n2025-04-15,16,10.4,0.0,0.0,19.1\n2025-04-15,17,9.3,0.0,0.0,18.4\n2025-04-15,18,8.4,0.0,0.0,15.8\n2025-04-15,19,7.6,0.0,0.0,14.5\n2025-04-15,20,7.1,0.0,0.0,12.5\n2025-04-15,21,7.0,0.0,0.0,11.2\n2025-04-15,22,6.9,0.0,0.0,9.8\n2025-04-15,23,6.6,0.0,0.0,9.2\n2025-04-16,0,6.3,0.0,0.0,7.2\n2025-04-16,1,5.9,0.0,0.0,7.2\n2025-04-16,2,5.5,0.0,0.0,6.7\n2025-04-16,3,4.9,0.0,0.0,7.2\n2025-04-16,4,4.5,0.0,0.0,7.0\n2025-04-16,5,5.2,0.0,0.0,8.7\n2025-04-16,6,5.5,0.0,0.0,10.1\n2025-04-16,7,5.7,0.0,0.0,10.6\n2025-04-16,8,7.8,0.0,0.0,10.0\n2025-04-16,9,10.8,0.0,0.0,11.8\n2025-04-16,10,12.8,0.0,0.0,11.1\n2025-04-16,11,15.2,0.0,0.0,11.5\n2025-04-16,12,17.3,0.0,0.0,11.5\n2025-04-16,13,18.5,0.0,0.0,13.2\n2025-04-16,14,19.9,0.0,0.0,15.5\n2025-04-16,15,20.3,0.0,0.0,17.0\n2025-04-16,16,19.4,0.0,0.0,16.2\n2025-04-16,17,18.5,0.0,0.0,14.8\n2025-04-16,18,17.0,0.0,0.0,12.3\n2025-04-16,19,15.0,0.0,0.0,12.6\n2025-04-16,20,13.8,0.0,0.0,13.4\n2025-04-16,21,13.2,0.0,0.0,15.6\n2025-04-16,22,13.9,0.0,0.0,16.7\n2025-04-16,23,13.7,0.0,0.0,19.8\n2025-04-17,0,13.8,0.1,0.0,21.2\n2025-04-17,1,13.4,0.0,0.0,21.6\n2025-04-17,2,13.5,0.0,0.0,22.5\n2025-04-17,3,13.7,0.0,0.0,21.5\n2025-04-17,4,13.5,0.0,0.0,22.2\n2025-04-17,5,13.5,0.0,0.0,19.3\n2025-04-17,6,13.1,0.0,0.0,15.3\n2025-04-17,7,13.4,0.0,0.0,14.5\n2025-04-17,8,15.3,0.0,0.0,15.8\n2025-04-17,9,17.2,0.0,0.0,21.5\n2025-04-17,10,19.9,0.0,0.0,22.8\n2025-04-17,11,20.8,0.0,0.0,21.0\n2025-04-17,12,21.5,0.0,0.0,20.4\n2025-04-17,13,21.8,0.0,0.0,21.8\n2025-04-17,14,21.7,0.0,0.0,23.1\n2025-04-17,15,21.2,0.0,0.0,21.9\n2025-04-17,16,20.6,0.0,0.0,20.2\n2025-04-17,17,19.9,0.0,0.0,19.2\n2025-04-17,18,18.6,0.0,0.0,18.1\n2025-04-17,19,17.0,0.0,0.0,13.5\n2025-04-17,20,15.5,0.0,0.0,14.6\n2025-04-17,21,14.8,0.0,0.0,12.5\n2025-04-17,22,16.2,0.0,0.0,7.8\n2025-04-17,23,14.6,0.0,0.0,10.5\n2025-04-18,0,14.2,0.0,0.0,12.6\n2025-04-18,1,14.9,0.0,0.0,16.1\n2025-04-18,2,14.0,0.0,0.0,15.1\n2025-04-18,3,13.3,0.0,0.0,14.3\n2025-04-18,4,12.8,0.0,0.0,13.0\n2025-04-18,5,12.1,0.0,0.0,11.3\n2025-04-18,6,11.7,0.0,0.0,9.8\n2025-04-18,7,11.9,0.0,0.0,9.3\n2025-04-18,8,12.5,0.0,0.0,9.6\n2025-04-18,9,12.9,0.0,0.0,8.4\n2025-04-18,10,13.7,0.0,0.0,8.0\n2025-04-18,11,14.9,0.0,0.0,8.1\n2025-04-18,12,16.5,0.0,0.0,7.6\n2025-04-18,13,17.5,0.0,0.0,6.9\n2025-04-18,14,18.6,0.0,0.0,6.8\n2025-04-18,15,18.8,0.0,0.0,7.5\n2025-04-18,16,18.6,0.0,0.0,7.5\n2025-04-18,17,17.4,0.0,0.0,8.7\n2025-04-18,18,16.2,0.0,0.0,5.6\n2025-04-18,19,14.7,0.0,0.0,5.8\n2025-04-18,20,12.9,0.0,0.0,1.9\n2025-04-18,21,12.1,0.0,0.0,2.2\n2025-04-18,22,11.8,0.0,0.0,5.4\n2025-04-18,23,11.5,0.0,0.0,5.0\n2025-04-19,0,11.4,0.0,0.0,7.0\n2025-04-19,1,11.4,0.0,0.0,7.6\n2025-04-19,2,12.4,0.0,0.0,9.7\n2025-04-19,3,13.0,0.0,0.0,10.8\n2025-04-19,4,13.2,0.0,0.0,11.6\n2025-04-19,5,13.0,0.1,0.0,8.8\n2025-04-19,6,12.8,0.2,0.0,7.3\n2025-04-19,7,13.2,0.3,0.0,9.3\n2025-04-19,8,13.9,0.4,0.0,10.6\n2025-04-19,9,15.0,1.3,0.0,17.7\n2025-04-19,10,16.6,5.7,0.0,19.6\n2025-04-19,11,16.2,4.4,0.0,24.5\n2025-04-19,12,16.6,3.1,0.0,28.2\n2025-04-19,13,15.6,4.0,0.0,20.8\n2025-04-19,14,13.3,1.5,0.0,21.3\n2025-04-19,15,12.0,0.5,0.0,20.6\n2025-04-19,16,12.0,0.3,0.0,19.8\n2025-04-19,17,10.9,0.8,0.0,16.1\n2025-04-19,18,10.3,4.8,0.0,8.5\n2025-04-19,19,10.1,6.0,0.0,5.1\n2025-04-19,20,10.2,0.0,0.0,7.1\n2025-04-19,21,10.1,0.0,0.0,6.3\n2025-04-19,22,11.1,0.0,0.0,0.9\n2025-04-19,23,10.5,0.0,0.0,3.4\n2025-04-20,0,10.1,0.0,0.0,4.1\n2025-04-20,1,10.6,0.0,0.0,5.4\n2025-04-20,2,10.4,0.0,0.0,6.6\n2025-04-20,3,10.1,0.0,0.0,8.5\n2025-04-20,4,9.8,0.0,0.0,7.6\n2025-04-20,5,9.7,0.0,0.0,6.8\n2025-04-20,6,9.9,0.0,0.0,7.9\n2025-04-20,7,10.1,0.0,0.0,7.1\n2025-04-20,8,10.4,0.0,0.0,7.9\n2025-04-20,9,11.6,0.0,0.0,9.6\n2025-04-20,10,14.0,0.0,0.0,8.7\n2025-04-20,11,16.1,0.0,0.0,6.7\n2025-04-20,12,17.8,0.0,0.0,5.9\n2025-04-20,13,19.2,0.0,0.0,6.0\n2025-04-20,14,20.4,0.0,0.0,5.2\n2025-04-20,15,21.3,0.0,0.0,3.6\n2025-04-20,16,21.1,0.0,0.0,5.2\n2025-04-20,17,20.1,0.0,0.0,8.9\n2025-04-20,18,18.3,0.0,0.0,9.4\n2025-04-20,19,16.0,0.0,0.0,7.3\n2025-04-20,20,14.7,0.0,0.0,6.1\n2025-04-20,21,13.7,0.0,0.0,4.7\n2025-04-20,22,13.9,0.0,0.0,4.4\n2025-04-20,23,13.4,0.0,0.0,4.9\n2025-04-21,0,13.2,0.0,0.0,5.1\n2025-04-21,1,13.1,0.0,0.0,13.0\n2025-04-21,2,13.0,0.0,0.0,10.9\n2025-04-21,3,13.0,0.0,0.0,12.0\n2025-04-21,4,12.9,0.0,0.0,12.3\n2025-04-21,5,12.7,0.0,0.0,12.0\n2025-04-21,6,12.4,0.0,0.0,11.1\n2025-04-21,7,13.1,0.0,0.0,11.1\n2025-04-21,8,15.4,0.0,0.0,10.0\n2025-04-21,9,17.2,0.0,0.0,7.8\n2025-04-21,10,19.3,0.0,0.0,7.7\n2025-04-21,11,21.1,0.0,0.0,10.1\n2025-04-21,12,22.2,0.0,0.0,9.1\n2025-04-21,13,23.3,0.0,0.0,11.7\n2025-04-21,14,23.4,0.0,0.0,10.6\n2025-04-21,15,24.0,0.0,0.0,10.4\n2025-04-21,16,24.3,0.0,0.0,9.0\n2025-04-21,17,20.6,0.0,0.0,12.2\n2025-04-21,18,19.8,0.0,0.0,5.8\n2025-04-21,19,17.6,0.0,0.0,6.7\n2025-04-21,20,15.8,0.0,0.0,2.3\n2025-04-21,21,15.3,0.0,0.0,3.5\n2025-04-21,22,17.6,0.0,0.0,2.3\n2025-04-21,23,17.7,0.0,0.0,0.4\n2025-04-22,0,16.4,0.1,0.0,2.2\n2025-04-22,1,16.2,0.5,0.0,2.7\n2025-04-22,2,16.2,0.6,0.0,2.5\n2025-04-22,3,16.1,0.4,0.0,4.2\n2025-04-22,4,15.8,0.3,0.0,1.5\n2025-04-22,5,15.1,5.2,0.0,8.5\n2025-04-22,6,14.5,1.5,0.0,6.9\n2025-04-22,7,14.3,3.6,0.0,7.8\n2025-04-22,8,14.6,1.4,0.0,7.5\n2025-04-22,9,15.2,2.6,0.0,6.5\n2025-04-22,10,17.5,2.5,0.0,6.3\n2025-04-22,11,17.4,2.4,0.0,7.1\n2025-04-22,12,16.6,2.1,0.0,10.9\n2025-04-22,13,16.4,1.4,0.0,7.9\n2025-04-22,14,16.9,0.9,0.0,13.0\n2025-04-22,15,16.6,2.1,0.0,10.7\n2025-04-22,16,17.3,0.6,0.0,11.5\n2025-04-22,17,18.0,0.1,0.0,10.6\n2025-04-22,18,17.7,0.0,0.0,10.8\n2025-04-22,19,17.0,0.0,0.0,8.8\n2025-04-22,20,16.1,0.0,0.0,8.9\n2025-04-22,21,15.4,0.0,0.0,8.6\n2025-04-22,22,15.1,0.0,0.0,10.8\n2025-04-22,23,14.5,0.0,0.0,12.8\n2025-04-23,0,13.8,0.0,0.0,11.1\n2025-04-23,1,12.9,0.0,0.0,10.1\n2025-04-23,2,12.0,0.0,0.0,11.1\n2025-04-23,3,11.2,0.0,0.0,11.3\n2025-04-23,4,10.6,0.0,0.0,11.3\n2025-04-23,5,10.1,0.0,0.0,11.8\n2025-04-23,6,9.8,0.0,0.0,11.5\n2025-04-23,7,10.5,0.0,0.0,11.2\n2025-04-23,8,12.1,0.0,0.0,8.9\n2025-04-23,9,14.0,0.0,0.0,8.6\n2025-04-23,10,16.5,0.0,0.0,8.5\n2025-04-23,11,18.1,0.0,0.0,6.1\n2025-04-23,12,19.6,0.0,0.0,1.8\n2025-04-23,13,20.3,0.0,0.0,4.5\n2025-04-23,14,20.6,0.0,0.0,8.6\n2025-04-23,15,20.7,0.0,0.0,9.0\n2025-04-23,16,20.2,0.0,0.0,8.8\n2025-04-23,17,19.4,0.0,0.0,9.0\n2025-04-23,18,17.5,0.0,0.0,8.8\n2025-04-23,19,14.8,0.0,0.0,8.5\n2025-04-23,20,13.0,0.0,0.0,5.8\n2025-04-23,21,12.0,0.0,0.0,5.3\n2025-04-23,22,11.9,0.0,0.0,5.6\n2025-04-23,23,11.2,0.0,0.0,6.7\n2025-04-24,0,10.6,0.0,0.0,4.9\n2025-04-24,1,10.1,0.0,0.0,4.1\n2025-04-24,2,9.6,0.0,0.0,5.9\n2025-04-24,3,9.3,0.0,0.0,6.4\n2025-04-24,4,9.1,0.0,0.0,6.7\n2025-04-24,5,8.8,0.0,0.0,5.8\n2025-04-24,6,8.4,0.0,0.0,8.6\n2025-04-24,7,9.1,0.0,0.0,9.7\n2025-04-24,8,11.0,0.0,0.0,7.7\n2025-04-24,9,12.6,0.0,0.0,9.7\n2025-04-24,10,14.5,0.0,0.0,10.4\n2025-04-24,11,15.1,0.0,0.0,11.5\n2025-04-24,12,15.0,0.0,0.0,14.3\n2025-04-24,13,15.1,0.0,0.0,13.1\n2025-04-24,14,15.7,0.0,0.0,12.5\n2025-04-24,15,15.9,0.0,0.0,14.9\n2025-04-24,16,15.6,0.0,0.0,16.3\n2025-04-24,17,15.1,0.0,0.0,20.3\n2025-04-24,18,13.5,0.0,0.0,20.1\n2025-04-24,19,12.1,0.0,0.0,21.6\n2025-04-24,20,11.2,0.0,0.0,18.4\n2025-04-24,21,10.3,0.0,0.0,16.1\n2025-04-24,22,10.4,0.0,0.0,18.0\n2025-04-24,23,9.8,0.0,0.0,16.5\n2025-04-25,0,9.2,0.0,0.0,13.7\n2025-04-25,1,8.6,0.0,0.0,11.9\n2025-04-25,2,8.1,0.0,0.0,12.1\n2025-04-25,3,7.6,0.0,0.0,11.5\n2025-04-25,4,7.2,0.0,0.0,10.7\n2025-04-25,5,7.0,0.0,0.0,10.1\n2025-04-25,6,6.8,0.0,0.0,9.5\n2025-04-25,7,8.0,0.0,0.0,10.5\n2025-04-25,8,10.1,0.0,0.0,9.5\n2025-04-25,9,11.9,0.0,0.0,10.2\n2025-04-25,10,13.8,0.0,0.0,13.8\n2025-04-25,11,14.6,0.0,0.0,17.4\n2025-04-25,12,14.9,0.0,0.0,19.9\n2025-04-25,13,15.2,0.0,0.0,20.2\n2025-04-25,14,15.0,0.0,0.0,21.3\n2025-04-25,15,14.8,0.0,0.0,23.5\n2025-04-25,16,14.3,0.0,0.0,24.3\n2025-04-25,17,13.5,0.0,0.0,21.2\n2025-04-25,18,12.2,0.0,0.0,16.6\n2025-04-25,19,10.8,0.0,0.0,18.1\n2025-04-25,20,9.9,0.0,0.0,15.5\n2025-04-25,21,9.5,0.0,0.0,13.2\n2025-04-25,22,9.4,0.0,0.0,11.6\n2025-04-25,23,8.9,0.0,0.0,9.5\n2025-04-26,0,8.5,0.0,0.0,7.7\n2025-04-26,1,8.3,0.0,0.0,6.6\n2025-04-26,2,8.3,0.0,0.0,7.1\n2025-04-26,3,8.1,0.0,0.0,8.0\n2025-04-26,4,7.7,0.0,0.0,6.9\n2025-04-26,5,7.0,0.0,0.0,6.7\n2025-04-26,6,6.2,0.0,0.0,8.5\n2025-04-26,7,7.3,0.0,0.0,10.3\n2025-04-26,8,9.9,0.0,0.0,10.0\n2025-04-26,9,12.3,0.0,0.0,12.0\n2025-04-26,10,14.1,0.0,0.0,12.7\n2025-04-26,11,15.0,0.0,0.0,15.3\n2025-04-26,12,15.6,0.0,0.0,17.3\n2025-04-26,13,15.7,0.0,0.0,18.2\n2025-04-26,14,15.6,0.0,0.0,17.7\n2025-04-26,15,15.5,0.0,0.0,17.2\n2025-04-26,16,15.0,0.0,0.0,17.6\n2025-04-26,17,14.5,0.0,0.0,15.8\n2025-04-26,18,13.9,0.0,0.0,13.6\n2025-04-26,19,13.0,0.0,0.0,12.7\n2025-04-26,20,12.2,0.0,0.0,14.2\n2025-04-26,21,11.3,0.0,0.0,13.2\n2025-04-26,22,11.3,0.0,0.0,11.4\n2025-04-26,23,10.6,0.0,0.0,10.3\n2025-04-27,0,10.0,0.0,0.0,10.1\n2025-04-27,1,9.6,0.0,0.0,11.6\n2025-04-27,2,9.3,0.0,0.0,12.6\n2025-04-27,3,9.2,0.0,0.0,11.6\n2025-04-27,4,9.0,0.0,0.0,10.3\n2025-04-27,5,8.8,0.0,0.0,8.8\n2025-04-27,6,9.4,0.0,0.0,11.9\n2025-04-27,7,10.9,0.0,0.0,15.9\n2025-04-27,8,12.9,0.0,0.0,13.0\n2025-04-27,9,14.3,0.0,0.0,15.5\n2025-04-27,10,15.6,0.0,0.0,16.3\n2025-04-27,11,15.8,0.0,0.0,17.9\n2025-04-27,12,15.8,0.0,0.0,18.4\n2025-04-27,13,16.1,0.0,0.0,18.0\n2025-04-27,14,16.0,0.0,0.0,15.3\n2025-04-27,15,15.7,0.0,0.0,14.3\n2025-04-27,16,15.4,0.0,0.0,14.3\n2025-04-27,17,14.3,0.0,0.0,14.4\n2025-04-27,18,13.3,0.0,0.0,13.4\n2025-04-27,19,12.4,0.0,0.0,14.7\n2025-04-27,20,11.7,0.0,0.0,11.2\n2025-04-27,21,10.6,0.0,0.0,10.6\n2025-04-27,22,11.1,0.0,0.0,10.3\n2025-04-27,23,10.2,0.0,0.0,7.0\n2025-04-28,0,10.0,0.0,0.0,6.8\n2025-04-28,1,9.6,0.0,0.0,6.0\n2025-04-28,2,9.9,0.0,0.0,5.0\n2025-04-28,3,9.3,0.0,0.0,4.2\n2025-04-28,4,9.2,0.0,0.0,2.2\n2025-04-28,5,8.0,0.0,0.0,2.0\n2025-04-28,6,7.7,0.0,0.0,1.3\n2025-04-28,7,9.2,0.0,0.0,1.9\n2025-04-28,8,11.5,0.0,0.0,0.9\n2025-04-28,9,13.7,0.0,0.0,3.6\n2025-04-28,10,15.0,0.0,0.0,11.0\n2025-04-28,11,15.8,0.0,0.0,13.1\n2025-04-28,12,16.8,0.0,0.0,14.0\n2025-04-28,13,17.5,0.0,0.0,16.7\n2025-04-28,14,17.9,0.0,0.0,19.7\n2025-04-28,15,18.3,0.0,0.0,21.7\n2025-04-28,16,17.7,0.0,0.0,21.1\n2025-04-28,17,17.1,0.0,0.0,19.5\n2025-04-28,18,15.8,0.0,0.0,16.6\n2025-04-28,19,14.0,0.0,0.0,15.0\n2025-04-28,20,12.4,0.0,0.0,14.3\n2025-04-28,21,11.2,0.0,0.0,11.0\n2025-04-28,22,11.0,0.0,0.0,11.5\n2025-04-28,23,10.2,0.0,0.0,11.4\n2025-04-29,0,9.4,0.0,0.0,9.7\n2025-04-29,1,8.5,0.0,0.0,7.5\n2025-04-29,2,7.8,0.0,0.0,5.3\n2025-04-29,3,7.3,0.0,0.0,4.9\n2025-04-29,4,6.8,0.0,0.0,2.8\n2025-04-29,5,6.5,0.0,0.0,0.6\n2025-04-29,6,5.7,0.0,0.0,2.3\n2025-04-29,7,7.2,0.0,0.0,5.1\n2025-04-29,8,9.9,0.0,0.0,3.1\n2025-04-29,9,12.5,0.0,0.0,3.9\n2025-04-29,10,14.7,0.0,0.0,6.7\n2025-04-29,11,16.3,0.0,0.0,8.3\n2025-04-29,12,17.2,0.0,0.0,9.9\n2025-04-29,13,17.5,0.0,0.0,12.2\n2025-04-29,14,17.4,0.0,0.0,13.2\n2025-04-29,15,17.5,0.0,0.0,12.0\n2025-04-29,16,16.6,0.0,0.0,13.6\n2025-04-29,17,15.7,0.0,0.0,11.5\n2025-04-29,18,14.3,0.0,0.0,8.6\n2025-04-29,19,13.0,0.0,0.0,6.2\n2025-04-29,20,12.4,0.0,0.0,7.4\n2025-04-29,21,12.1,0.0,0.0,9.9\n2025-04-29,22,11.6,0.0,0.0,9.1\n2025-04-29,23,10.9,0.0,0.0,9.3\n2025-04-30,0,10.3,0.0,0.0,10.6\n2025-04-30,1,10.0,0.0,0.0,11.6\n2025-04-30,2,9.8,0.0,0.0,13.1\n2025-04-30,3,9.7,0.0,0.0,13.4\n2025-04-30,4,9.4,0.0,0.0,12.5\n2025-04-30,5,9.0,0.0,0.0,11.2\n2025-04-30,6,9.0,0.0,0.0,11.9\n2025-04-30,7,10.5,0.0,0.0,12.7\n2025-04-30,8,13.1,0.0,0.0,12.1\n2025-04-30,9,15.7,0.0,0.0,14.0\n2025-04-30,10,17.8,0.0,0.0,14.1\n2025-04-30,11,20.1,0.0,0.0,16.3\n2025-04-30,12,21.8,0.0,0.0,18.9\n2025-04-30,13,22.5,0.0,0.0,21.2\n2025-04-30,14,23.0,0.0,0.0,22.0\n2025-04-30,15,23.1,0.0,0.0,22.5\n2025-04-30,16,22.7,0.0,0.0,21.3\n2025-04-30,17,21.9,0.0,0.0,18.8\n2025-04-30,18,20.6,0.0,0.0,16.0\n2025-04-30,19,18.4,0.0,0.0,13.8\n2025-04-30,20,16.4,0.0,0.0,13.0\n2025-04-30,21,15.3,0.0,0.0,13.3\n2025-04-30,22,15.2,0.0,0.0,11.1\n2025-04-30,23,14.6,0.0,0.0,14.4\n2025-05-01,0,14.4,0.0,0.0,15.3\n2025-05-01,1,13.9,0.0,0.0,14.8\n2025-05-01,2,13.5,0.0,0.0,16.2\n2025-05-01,3,13.4,0.0,0.0,18.5\n2025-05-01,4,13.1,0.0,0.0,19.0\n2025-05-01,5,13.0,0.0,0.0,19.7\n2025-05-01,6,13.0,0.0,0.0,19.5\n2025-05-01,7,14.0,0.0,0.0,14.1\n2025-05-01,8,14.6,1.2,0.0,13.4\n2025-05-01,9,15.0,0.7,0.0,16.8\n2025-05-01,10,13.9,1.9,0.0,8.9\n2025-05-01,11,13.8,3.0,0.0,17.0\n2025-05-01,12,13.8,4.9,0.0,18.9\n2025-05-01,13,13.4,3.0,0.0,19.6\n2025-05-01,14,14.0,1.4,0.0,18.3\n2025-05-01,15,14.5,0.2,0.0,17.8\n2025-05-01,16,14.7,0.0,0.0,13.6\n2025-05-01,17,14.3,0.0,0.0,13.5\n2025-05-01,18,13.6,0.0,0.0,13.8\n2025-05-01,19,12.2,0.0,0.0,11.2\n2025-05-01,20,10.7,0.0,0.0,10.9\n2025-05-01,21,10.8,0.0,0.0,12.0\n2025-05-01,22,11.8,0.0,0.0,20.0\n2025-05-01,23,11.3,0.0,0.0,18.1\n2025-05-02,0,10.6,0.0,0.0,16.5\n2025-05-02,1,10.1,0.0,0.0,16.1\n2025-05-02,2,9.9,0.0,0.0,16.3\n2025-05-02,3,9.7,0.0,0.0,17.5\n2025-05-02,4,9.5,0.0,0.0,17.5\n2025-05-02,5,8.7,0.0,0.0,14.7\n2025-05-02,6,8.1,0.0,0.0,12.0\n2025-05-02,7,8.8,0.0,0.0,12.1\n2025-05-02,8,10.3,0.0,0.0,12.7\n2025-05-02,9,12.0,0.0,0.0,12.9\n2025-05-02,10,13.6,0.0,0.0,14.3\n2025-05-02,11,15.0,0.0,0.0,17.5\n2025-05-02,12,16.0,0.0,0.0,17.5\n2025-05-02,13,16.3,0.0,0.0,16.7\n2025-05-02,14,16.7,0.0,0.0,17.2\n2025-05-02,15,16.0,0.0,0.0,18.4\n2025-05-02,16,15.0,0.0,0.0,17.2\n2025-05-02,17,14.4,0.0,0.0,13.5\n2025-05-02,18,12.8,0.0,0.0,11.9\n2025-05-02,19,12.0,0.0,0.0,12.5\n2025-05-02,20,11.8,0.0,0.0,10.5\n2025-05-02,21,11.8,0.0,0.0,6.9\n2025-05-02,22,12.3,0.0,0.0,5.2\n2025-05-02,23,11.6,0.0,0.0,9.4\n2025-05-03,0,11.1,0.0,0.0,7.0\n2025-05-03,1,10.9,0.0,0.0,4.5\n2025-05-03,2,10.6,0.2,0.0,5.3\n2025-05-03,3,10.1,1.1,0.0,4.2\n2025-05-03,4,9.9,0.6,0.0,3.3\n2025-05-03,5,9.5,0.6,0.0,5.7\n2025-05-03,6,9.5,0.9,0.0,2.8\n2025-05-03,7,9.5,1.4,0.0,2.5\n2025-05-03,8,9.4,1.4,0.0,4.9\n2025-05-03,9,9.6,0.8,0.0,3.5\n2025-05-03,10,9.8,0.3,0.0,2.4\n2025-05-03,11,10.8,0.2,0.0,0.2\n2025-05-03,12,12.2,0.2,0.0,8.9\n2025-05-03,13,11.7,0.1,0.0,13.1\n2025-05-03,14,12.9,0.3,0.0,16.3\n2025-05-03,15,12.6,0.0,0.0,18.1\n2025-05-03,16,12.8,0.1,0.0,20.1\n2025-05-03,17,12.2,0.0,0.0,17.5\n2025-05-03,18,11.3,0.0,0.0,17.1\n2025-05-03,19,10.5,0.0,0.0,16.5\n2025-05-03,20,10.5,0.0,0.0,17.1\n2025-05-03,21,10.5,0.0,0.0,15.7\n2025-05-03,22,10.1,0.0,0.0,15.3\n2025-05-03,23,10.1,0.0,0.0,15.3\n2025-05-04,0,9.9,0.0,0.0,13.2\n2025-05-04,1,9.8,0.0,0.0,11.9\n2025-05-04,2,9.7,0.0,0.0,12.1\n2025-05-04,3,9.7,0.0,0.0,11.9\n2025-05-04,4,9.6,0.0,0.0,11.1\n2025-05-04,5,9.5,0.0,0.0,11.7\n2025-05-04,6,9.3,0.0,0.0,10.4\n2025-05-04,7,10.4,0.0,0.0,10.9\n2025-05-04,8,12.3,0.0,0.0,11.0\n2025-05-04,9,13.8,0.0,0.0,13.0\n2025-05-04,10,14.8,0.0,0.0,12.5\n2025-05-04,11,15.4,0.0,0.0,14.6\n2025-05-04,12,16.0,0.0,0.0,16.9\n2025-05-04,13,16.5,0.0,0.0,19.9\n2025-05-04,14,16.6,0.0,0.0,23.6\n2025-05-04,15,16.2,0.0,0.0,24.7\n2025-05-04,16,15.7,0.0,0.0,23.3\n2025-05-04,17,15.0,0.0,0.0,21.4\n2025-05-04,18,13.9,0.0,0.0,17.1\n2025-05-04,19,12.6,0.0,0.0,14.8\n2025-05-04,20,11.3,0.0,0.0,12.4\n2025-05-04,21,10.8,0.0,0.0,9.5\n2025-05-04,22,10.9,0.0,0.0,8.4\n2025-05-04,23,10.4,0.0,0.0,5.5\n2025-05-05,0,9.5,0.0,0.0,4.4\n2025-05-05,1,9.5,0.0,0.0,6.3\n2025-05-05,2,8.8,0.0,0.0,5.2\n2025-05-05,3,8.1,0.0,0.0,6.2\n2025-05-05,4,7.5,0.0,0.0,8.5\n2025-05-05,5,7.2,0.0,0.0,8.0\n2025-05-05,6,7.9,0.0,0.0,7.4\n2025-05-05,7,8.8,0.0,0.0,8.7\n2025-05-05,8,10.7,0.0,0.0,5.9\n2025-05-05,9,11.8,0.0,0.0,4.5\n2025-05-05,10,12.5,0.0,0.0,6.7\n2025-05-05,11,12.6,0.0,0.0,6.8\n2025-05-05,12,13.4,0.0,0.0,5.9\n2025-05-05,13,13.6,0.0,0.0,7.8\n2025-05-05,14,13.8,0.2,0.0,8.6\n2025-05-05,15,14.1,0.2,0.0,12.3\n2025-05-05,16,13.9,0.0,0.0,13.3\n2025-05-05,17,13.6,0.0,0.0,10.8\n2025-05-05,18,13.1,0.0,0.0,11.1\n2025-05-05,19,12.5,0.0,0.0,11.2\n2025-05-05,20,11.8,0.0,0.0,11.4\n2025-05-05,21,11.6,0.0,0.0,11.0\n2025-05-05,22,11.2,0.0,0.0,11.2\n2025-05-05,23,10.6,0.0,0.0,10.4\n2025-05-06,0,10.0,0.0,0.0,9.3\n2025-05-06,1,9.5,0.0,0.0,8.4\n2025-05-06,2,9.0,0.0,0.0,6.0\n2025-05-06,3,8.5,0.0,0.0,5.6\n2025-05-06,4,8.3,0.0,0.0,7.6\n2025-05-06,5,7.9,0.0,0.0,6.7\n2025-05-06,6,7.8,0.0,0.0,6.9\n2025-05-06,7,8.9,0.0,0.0,6.9\n2025-05-06,8,11.3,0.0,0.0,5.2\n2025-05-06,9,13.9,0.0,0.0,6.5\n2025-05-06,10,15.6,0.0,0.0,8.9\n2025-05-06,11,16.4,0.0,0.0,9.0\n2025-05-06,12,17.2,0.0,0.0,8.8\n2025-05-06,13,17.8,0.0,0.0,11.6\n2025-05-06,14,18.2,0.0,0.0,11.5\n2025-05-06,15,17.1,0.0,0.0,11.6\n2025-05-06,16,16.3,0.0,0.0,14.5\n2025-05-06,17,15.9,0.0,0.0,12.5\n2025-05-06,18,15.2,0.0,0.0,11.1\n2025-05-06,19,13.7,0.0,0.0,11.6\n2025-05-06,20,12.5,0.0,0.0,10.7\n2025-05-06,21,11.9,0.0,0.0,8.1\n2025-05-06,22,11.9,0.0,0.0,9.4\n2025-05-06,23,11.5,0.0,0.0,9.1\n2025-05-07,0,10.8,0.0,0.0,7.7\n2025-05-07,1,10.1,0.0,0.0,7.6\n2025-05-07,2,9.7,0.0,0.0,6.7\n2025-05-07,3,9.3,0.0,0.0,4.9\n2025-05-07,4,8.6,0.0,0.0,4.5\n2025-05-07,5,8.2,0.0,0.0,4.3\n2025-05-07,6,8.1,0.0,0.0,3.6\n2025-05-07,7,9.7,0.0,0.0,3.7\n2025-05-07,8,11.3,0.0,0.0,2.2\n2025-05-07,9,13.4,0.0,0.0,2.9\n2025-05-07,10,15.0,0.0,0.0,2.8\n2025-05-07,11,16.6,0.0,0.0,5.6\n2025-05-07,12,17.3,0.0,0.0,9.3\n2025-05-07,13,17.3,0.0,0.0,12.4\n2025-05-07,14,17.1,0.0,0.0,13.2\n2025-05-07,15,16.7,0.0,0.0,12.7\n2025-05-07,16,16.2,0.0,0.0,12.8\n2025-05-07,17,15.4,0.0,0.0,11.9\n2025-05-07,18,14.2,0.0,0.0,10.1\n2025-05-07,19,12.9,0.0,0.0,5.3\n2025-05-07,20,11.6,0.0,0.0,5.5\n2025-05-07,21,10.9,0.0,0.0,6.0\n2025-05-07,22,11.0,0.0,0.0,3.6\n2025-05-07,23,10.5,0.0,0.0,1.9\n2025-05-08,0,10.1,0.0,0.0,1.4\n2025-05-08,1,9.7,0.0,0.0,0.7\n2025-05-08,2,9.2,0.0,0.0,0.6\n2025-05-08,3,9.0,0.0,0.0,4.3\n2025-05-08,4,8.0,0.0,0.0,3.9\n2025-05-08,5,7.7,0.0,0.0,4.4\n2025-05-08,6,7.7,0.0,0.0,6.5\n2025-05-08,7,9.2,0.0,0.0,6.2\n2025-05-08,8,11.9,0.0,0.0,2.6\n2025-05-08,9,14.7,0.0,0.0,3.8\n2025-05-08,10,16.7,0.0,0.0,4.0\n2025-05-08,11,18.7,0.0,0.0,4.8\n2025-05-08,12,20.3,0.0,0.0,4.1\n2025-05-08,13,21.7,0.0,0.0,5.8\n2025-05-08,14,22.3,0.0,0.0,8.7\n2025-05-08,15,22.6,0.0,0.0,6.8\n2025-05-08,16,22.5,0.0,0.0,6.4\n2025-05-08,17,20.0,0.0,0.0,15.5\n2025-05-08,18,18.4,0.0,0.0,10.4\n2025-05-08,19,16.8,0.0,0.0,8.0\n2025-05-08,20,15.4,0.0,0.0,4.4\n2025-05-08,21,14.8,0.0,0.0,4.0\n2025-05-08,22,15.0,0.0,0.0,3.6\n2025-05-08,23,15.3,0.0,0.0,0.4\n2025-05-09,0,14.8,0.0,0.0,1.5\n2025-05-09,1,14.4,0.0,0.0,1.5\n2025-05-09,2,14.0,0.0,0.0,2.4\n2025-05-09,3,14.1,0.1,0.0,8.9\n2025-05-09,4,14.1,0.0,0.0,8.9\n2025-05-09,5,13.8,0.0,0.0,9.8\n2025-05-09,6,13.9,0.0,0.0,8.1\n2025-05-09,7,14.5,0.0,0.0,6.7\n2025-05-09,8,14.5,0.6,0.0,9.8\n2025-05-09,9,13.4,0.9,0.0,14.3\n2025-05-09,10,13.9,0.5,0.0,14.5\n2025-05-09,11,14.4,0.8,0.0,15.2\n2025-05-09,12,14.4,0.9,0.0,19.0\n2025-05-09,13,15.1,1.0,0.0,21.5\n2025-05-09,14,14.2,1.9,0.0,21.2\n2025-05-09,15,14.5,2.2,0.0,19.2\n2025-05-09,16,14.9,4.6,0.0,18.2\n2025-05-09,17,15.0,3.6,0.0,18.8\n2025-05-09,18,14.9,1.8,0.0,17.4\n2025-05-09,19,14.8,1.6,0.0,17.5\n2025-05-09,20,14.9,1.2,0.0,16.8\n2025-05-09,21,15.4,2.2,0.0,14.8\n2025-05-09,22,15.0,2.2,0.0,13.0\n2025-05-09,23,15.0,2.2,0.0,14.0\n2025-05-10,0,15.0,0.9,0.0,14.4\n2025-05-10,1,14.5,0.5,0.0,17.8\n2025-05-10,2,13.8,0.5,0.0,15.8\n2025-05-10,3,12.7,0.6,0.0,17.9\n2025-05-10,4,12.0,0.0,0.0,18.5\n2025-05-10,5,12.2,0.0,0.0,22.1\n2025-05-10,6,12.0,0.0,0.0,18.1\n2025-05-10,7,12.1,0.0,0.0,19.7\n2025-05-10,8,12.2,0.0,0.0,21.4\n2025-05-10,9,11.8,0.1,0.0,20.6\n2025-05-10,10,11.7,0.1,0.0,25.8\n2025-05-10,11,11.7,0.8,0.0,25.7\n2025-05-10,12,11.4,1.0,0.0,24.5\n2025-05-10,13,11.8,0.2,0.0,25.0\n2025-05-10,14,12.8,0.3,0.0,26.0\n2025-05-10,15,13.2,0.1,0.0,21.6\n2025-05-10,16,12.9,0.0,0.0,19.4\n2025-05-10,17,12.8,0.0,0.0,16.5\n2025-05-10,18,12.7,0.0,0.0,14.3\n2025-05-10,19,12.3,0.0,0.0,13.3\n2025-05-10,20,12.1,0.0,0.0,11.8\n2025-05-10,21,12.1,0.0,0.0,10.0\n2025-05-10,22,11.5,0.0,0.0,9.1\n2025-05-10,23,11.2,0.0,0.0,8.7\n2025-05-11,0,10.9,0.0,0.0,9.7\n2025-05-11,1,10.7,0.0,0.0,12.3\n2025-05-11,2,10.8,0.0,0.0,9.8\n2025-05-11,3,10.5,0.0,0.0,8.5\n2025-05-11,4,10.4,0.0,0.0,7.7\n2025-05-11,5,9.9,0.0,0.0,7.3\n2025-05-11,6,9.9,0.0,0.0,7.7\n2025-05-11,7,10.9,0.0,0.0,9.2\n2025-05-11,8,12.7,0.0,0.0,8.5\n2025-05-11,9,14.6,0.0,0.0,8.2\n2025-05-11,10,15.9,0.0,0.0,7.5\n2025-05-11,11,16.0,0.0,0.0,6.9\n2025-05-11,12,17.3,0.0,0.0,6.1\n2025-05-11,13,17.9,0.0,0.0,8.2\n2025-05-11,14,18.0,0.0,0.0,11.6\n2025-05-11,15,17.6,0.0,0.0,11.6\n2025-05-11,16,16.9,0.0,0.0,11.4\n2025-05-11,17,15.6,0.0,0.0,10.6\n2025-05-11,18,14.3,0.0,0.0,9.9\n2025-05-11,19,13.0,0.0,0.0,6.0\n2025-05-11,20,11.8,0.0,0.0,6.8\n2025-05-11,21,11.3,0.0,0.0,4.9\n2025-05-11,22,11.4,0.0,0.0,6.6\n2025-05-11,23,10.9,0.0,0.0,6.3\n2025-05-12,0,10.5,0.0,0.0,7.0\n2025-05-12,1,10.3,0.0,0.0,8.1\n2025-05-12,2,10.2,0.0,0.0,8.8\n2025-05-12,3,10.0,0.0,0.0,8.0\n2025-05-12,4,9.8,0.0,0.0,7.2\n2025-05-12,5,9.5,0.0,0.0,8.0\n2025-05-12,6,9.4,0.0,0.0,8.4\n2025-05-12,7,10.7,0.0,0.0,9.0\n2025-05-12,8,13.1,0.0,0.0,8.3\n2025-05-12,9,15.5,0.0,0.0,9.0\n2025-05-12,10,17.0,0.0,0.0,6.7\n2025-05-12,11,18.7,0.0,0.0,8.0\n2025-05-12,12,19.8,0.0,0.0,10.7\n2025-05-12,13,20.4,0.0,0.0,13.1\n2025-05-12,14,20.6,0.0,0.0,13.9\n2025-05-12,15,20.9,0.0,0.0,13.5\n2025-05-12,16,21.0,0.0,0.0,14.1\n2025-05-12,17,20.6,0.0,0.0,13.9\n2025-05-12,18,19.7,0.0,0.0,12.3\n2025-05-12,19,18.1,0.0,0.0,12.4\n2025-05-12,20,16.3,0.0,0.0,13.4\n2025-05-12,21,15.4,0.0,0.0,13.2\n2025-05-12,22,15.4,0.0,0.0,12.0\n2025-05-12,23,14.7,0.0,0.0,11.5\n2025-05-13,0,14.4,0.0,0.0,12.0\n2025-05-13,1,14.3,0.0,0.0,12.4\n2025-05-13,2,14.2,0.0,0.0,13.3\n2025-05-13,3,14.1,0.0,0.0,13.9\n2025-05-13,4,13.8,0.0,0.0,14.2\n2025-05-13,5,13.6,0.0,0.0,14.5\n2025-05-13,6,13.7,0.0,0.0,14.3\n2025-05-13,7,14.9,0.0,0.0,13.6\n2025-05-13,8,16.7,0.0,0.0,10.5\n2025-05-13,9,19.0,0.0,0.0,11.7\n2025-05-13,10,20.6,0.0,0.0,10.1\n2025-05-13,11,22.1,0.0,0.0,12.4\n2025-05-13,12,22.9,0.0,0.0,13.9\n2025-05-13,13,23.6,0.0,0.0,15.6\n2025-05-13,14,23.6,0.0,0.0,16.9\n2025-05-13,15,21.9,0.0,0.0,13.7\n2025-05-13,16,22.1,0.0,0.0,13.5\n2025-05-13,17,22.3,0.0,0.0,13.7\n2025-05-13,18,21.3,0.0,0.0,12.3\n2025-05-13,19,19.6,0.0,0.0,11.7\n2025-05-13,20,17.8,0.0,0.0,11.2\n2025-05-13,21,16.8,0.0,0.0,11.2\n2025-05-13,22,16.9,0.0,0.0,10.5\n2025-05-13,23,16.3,0.0,0.0,10.9\n2025-05-14,0,15.6,0.0,0.0,11.0\n2025-05-14,1,15.0,0.0,0.0,10.5\n2025-05-14,2,14.5,0.0,0.0,10.0\n2025-05-14,3,14.1,0.0,0.0,10.1\n2025-05-14,4,13.7,0.0,0.0,10.7\n2025-05-14,5,13.4,0.0,0.0,10.7\n2025-05-14,6,13.5,0.0,0.0,11.4\n2025-05-14,7,15.1,0.0,0.0,10.7\n2025-05-14,8,17.6,0.0,0.0,9.5\n2025-05-14,9,19.9,0.0,0.0,12.5\n2025-05-14,10,21.9,0.0,0.0,10.8\n2025-05-14,11,23.2,0.0,0.0,14.2\n2025-05-14,12,24.3,0.0,0.0,16.2\n2025-05-14,13,25.0,0.0,0.0,16.5\n2025-05-14,14,24.8,0.0,0.0,14.0\n2025-05-14,15,24.7,0.0,0.0,14.3\n2025-05-14,16,23.9,0.0,0.0,16.6\n2025-05-14,17,22.7,0.0,0.0,15.2\n2025-05-14,18,21.0,0.0,0.0,12.1\n2025-05-14,19,19.7,0.0,0.0,13.0\n2025-05-14,20,19.0,0.0,0.0,15.3\n2025-05-14,21,19.8,0.0,0.0,17.2\n2025-05-14,22,19.6,0.0,0.0,10.6\n2025-05-14,23,18.9,0.2,0.0,11.1\n2025-05-15,0,18.5,0.2,0.0,11.1\n2025-05-15,1,18.5,0.2,0.0,11.6\n2025-05-15,2,18.1,0.2,0.0,12.3\n2025-05-15,3,16.8,1.3,0.0,16.3\n2025-05-15,4,16.8,0.3,0.0,18.7\n2025-05-15,5,16.2,0.4,0.0,17.6\n2025-05-15,6,15.7,0.3,0.0,15.8\n2025-05-15,7,15.4,0.1,0.0,11.6\n2025-05-15,8,15.2,1.6,0.0,12.4\n2025-05-15,9,14.9,0.1,0.0,13.4\n2025-05-15,10,15.9,0.0,0.0,9.6\n2025-05-15,11,17.1,0.0,0.0,7.8\n2025-05-15,12,18.1,0.0,0.0,6.2\n2025-05-15,13,20.2,0.0,0.0,7.1\n2025-05-15,14,21.0,0.0,0.0,8.9\n2025-05-15,15,22.0,0.0,0.0,11.3\n2025-05-15,16,21.9,0.0,0.0,12.7\n2025-05-15,17,20.2,0.0,0.0,12.6\n2025-05-15,18,18.3,0.0,0.0,10.5\n2025-05-15,19,16.9,0.0,0.0,8.1\n2025-05-15,20,16.1,0.0,0.0,6.1\n2025-05-15,21,16.0,0.0,0.0,5.1\n2025-05-15,22,15.2,0.0,0.0,3.4\n2025-05-15,23,15.0,0.0,0.0,3.1\n2025-05-16,0,15.3,0.0,0.0,1.3\n2025-05-16,1,15.9,0.0,0.0,2.4\n2025-05-16,2,16.3,0.0,0.0,2.5\n2025-05-16,3,16.5,0.0,0.0,3.1\n2025-05-16,4,16.9,0.0,0.0,3.6\n2025-05-16,5,17.2,0.1,0.0,6.9\n2025-05-16,6,17.6,0.1,0.0,5.1\n2025-05-16,7,18.5,0.0,0.0,8.6\n2025-05-16,8,19.6,0.0,0.0,12.5\n2025-05-16,9,20.3,0.0,0.0,16.2\n2025-05-16,10,19.6,1.0,0.0,11.7\n2025-05-16,11,19.5,6.2,0.0,5.9\n2025-05-16,12,20.7,0.1,0.0,10.4\n2025-05-16,13,22.9,0.0,0.0,14.5\n2025-05-16,14,22.9,0.0,0.0,13.5\n2025-05-16,15,22.6,0.0,0.0,10.0\n2025-05-16,16,19.8,1.3,0.0,16.8\n2025-05-16,17,18.7,1.3,0.0,4.8\n2025-05-16,18,18.5,0.6,0.0,8.5\n2025-05-16,19,18.4,0.6,0.0,8.6\n2025-05-16,20,18.1,0.4,0.0,9.6\n2025-05-16,21,17.9,0.2,0.0,11.9\n2025-05-16,22,18.3,0.5,0.0,10.8\n2025-05-16,23,17.9,0.3,0.0,7.7\n2025-05-17,0,17.4,0.1,0.0,5.6\n2025-05-17,1,16.7,0.0,0.0,7.6\n2025-05-17,2,16.3,0.0,0.0,7.1\n2025-05-17,3,15.6,0.0,0.0,6.5\n2025-05-17,4,15.1,0.0,0.0,5.8\n2025-05-17,5,14.8,0.0,0.0,6.4\n2025-05-17,6,14.4,0.0,0.0,4.1\n2025-05-17,7,15.4,0.0,0.0,6.0\n2025-05-17,8,16.7,0.0,0.0,6.6\n2025-05-17,9,18.1,0.0,0.0,4.7\n2025-05-17,10,21.1,0.0,0.0,9.3\n2025-05-17,11,21.9,0.0,0.0,12.2\n2025-05-17,12,22.4,0.0,0.0,15.4\n2025-05-17,13,21.6,0.0,0.0,16.0\n2025-05-17,14,21.4,0.0,0.0,17.2\n2025-05-17,15,19.8,0.0,0.0,17.9\n2025-05-17,16,19.1,0.0,0.0,17.9\n2025-05-17,17,16.9,0.4,0.0,23.9\n2025-05-17,18,15.2,1.5,0.0,20.4\n2025-05-17,19,14.9,0.3,0.0,17.4\n2025-05-17,20,14.8,0.0,0.0,13.3\n2025-05-17,21,14.3,0.0,0.0,11.6\n2025-05-17,22,14.3,0.0,0.0,13.4\n2025-05-17,23,14.3,0.0,0.0,15.4\n2025-05-18,0,13.2,0.0,0.0,15.7\n2025-05-18,1,12.2,0.0,0.0,13.0\n2025-05-18,2,11.8,0.0,0.0,12.6\n2025-05-18,3,11.5,0.0,0.0,11.5\n2025-05-18,4,11.1,0.0,0.0,11.0\n2025-05-18,5,10.4,0.0,0.0,9.7\n2025-05-18,6,10.1,0.0,0.0,7.2\n2025-05-18,7,10.9,0.0,0.0,5.5\n2025-05-18,8,12.2,0.0,0.0,7.0\n2025-05-18,9,13.4,0.0,0.0,7.3\n2025-05-18,10,15.2,0.0,0.0,4.0\n2025-05-18,11,17.0,0.0,0.0,4.2\n2025-05-18,12,18.3,0.0,0.0,8.9\n2025-05-18,13,19.1,0.0,0.0,10.3\n2025-05-18,14,19.4,0.0,0.0,12.1\n2025-05-18,15,18.9,0.0,0.0,13.9\n2025-05-18,16,18.3,0.0,0.0,12.6\n2025-05-18,17,17.5,0.0,0.0,11.9\n2025-05-18,18,16.3,0.0,0.0,9.6\n2025-05-18,19,14.9,0.0,0.0,5.9\n2025-05-18,20,13.4,0.0,0.0,5.0\n2025-05-18,21,12.5,0.0,0.0,3.9\n2025-05-18,22,12.6,0.0,0.0,3.5\n2025-05-18,23,12.2,0.0,0.0,1.3\n2025-05-19,0,11.7,0.0,0.0,1.3\n2025-05-19,1,10.9,0.0,0.0,2.9\n2025-05-19,2,10.3,0.0,0.0,3.9\n2025-05-19,3,10.1,0.0,0.0,7.7\n2025-05-19,4,9.8,0.0,0.0,10.9\n2025-05-19,5,9.8,0.0,0.0,10.6\n2025-05-19,6,10.4,0.0,0.0,9.7\n2025-05-19,7,12.0,0.0,0.0,10.3\n2025-05-19,8,14.6,0.0,0.0,10.5\n2025-05-19,9,17.1,0.0,0.0,13.3\n2025-05-19,10,15.7,0.0,0.0,11.2\n2025-05-19,11,18.8,0.0,0.0,13.6\n2025-05-19,12,19.3,0.0,0.0,16.4\n2025-05-19,13,19.5,0.0,0.0,17.5\n2025-05-19,14,19.5,0.0,0.0,17.4\n2025-05-19,15,18.7,0.0,0.0,22.6\n2025-05-19,16,18.2,0.0,0.0,22.8\n2025-05-19,17,18.0,0.1,0.0,21.4\n2025-05-19,18,17.7,0.0,0.0,19.1\n2025-05-19,19,17.5,0.0,0.0,20.7\n2025-05-19,20,16.9,0.0,0.0,22.6\n2025-05-19,21,16.7,0.0,0.0,18.8\n2025-05-19,22,16.6,0.1,0.0,12.6\n2025-05-19,23,16.3,0.0,0.0,11.8\n2025-05-20,0,16.2,0.0,0.0,12.7\n2025-05-20,1,16.5,0.0,0.0,14.5\n2025-05-20,2,16.6,0.0,0.0,16.6\n2025-05-20,3,16.4,0.0,0.0,16.8\n2025-05-20,4,16.2,0.0,0.0,16.2\n2025-05-20,5,16.2,0.0,0.0,15.7\n2025-05-20,6,16.4,0.0,0.0,15.6\n2025-05-20,7,17.0,0.0,0.0,17.8\n2025-05-20,8,17.5,0.0,0.0,16.9\n2025-05-20,9,18.4,0.0,0.0,17.5\n2025-05-20,10,20.6,0.0,0.0,19.5\n2025-05-20,11,21.7,0.1,0.0,20.0\n2025-05-20,12,22.5,0.0,0.0,21.6\n2025-05-20,13,23.6,0.1,0.0,20.4\n2025-05-20,14,24.3,0.0,0.0,18.9\n2025-05-20,15,25.0,0.0,0.0,20.4\n2025-05-20,16,24.3,0.0,0.0,22.4\n2025-05-20,17,23.3,0.0,0.0,22.8\n2025-05-20,18,22.6,0.0,0.0,21.8\n2025-05-20,19,22.4,0.0,0.0,17.4\n2025-05-20,20,22.0,0.0,0.0,17.0\n2025-05-20,21,21.8,0.0,0.0,14.1\n2025-05-20,22,22.6,0.0,0.0,9.9\n2025-05-20,23,22.0,0.0,0.0,8.7\n2025-05-21,0,21.6,0.0,0.0,9.2\n2025-05-21,1,21.7,0.0,0.0,7.2\n2025-05-21,2,21.4,0.0,0.0,7.0\n2025-05-21,3,20.5,0.0,0.0,6.8\n2025-05-21,4,19.8,0.0,0.0,5.0\n2025-05-21,5,19.2,0.0,0.0,7.3\n2025-05-21,6,19.0,0.0,0.0,7.2\n2025-05-21,7,19.0,0.0,0.0,5.2\n2025-05-21,8,19.3,0.1,0.0,3.7\n2025-05-21,9,20.4,0.0,0.0,3.8\n2025-05-21,10,23.7,0.0,0.0,3.3\n2025-05-21,11,26.4,0.0,0.0,1.4\n2025-05-21,12,27.9,0.0,0.0,2.3\n2025-05-21,13,28.7,0.0,0.0,6.7\n2025-05-21,14,28.7,0.0,0.0,9.4\n2025-05-21,15,29.0,0.0,0.0,10.5\n2025-05-21,16,28.0,0.0,0.0,12.9\n2025-05-21,17,25.9,0.0,0.0,13.0\n2025-05-21,18,24.0,0.0,0.0,13.9\n2025-05-21,19,22.8,0.0,0.0,11.1\n2025-05-21,20,21.4,0.0,0.0,9.3\n2025-05-21,21,20.5,0.0,0.0,7.9\n2025-05-21,22,21.6,0.0,0.0,9.4\n2025-05-21,23,20.5,0.0,0.0,7.8\n2025-05-22,0,19.8,0.0,0.0,6.3\n2025-05-22,1,19.4,0.0,0.0,6.2\n2025-05-22,2,19.1,0.0,0.0,4.6\n2025-05-22,3,19.0,0.0,0.0,4.7\n2025-05-22,4,18.6,5.7,0.0,7.1\n2025-05-22,5,17.8,0.8,0.0,9.8\n2025-05-22,6,16.5,0.0,0.0,10.3\n2025-05-22,7,16.7,0.0,0.0,11.1\n2025-05-22,8,18.5,0.0,0.0,9.0\n2025-05-22,9,21.0,0.0,0.0,5.9\n2025-05-22,10,24.0,0.0,0.0,9.4\n2025-05-22,11,25.0,0.0,0.0,11.9\n2025-05-22,12,25.3,0.0,0.0,12.7\n2025-05-22,13,25.6,0.0,0.0,12.6\n2025-05-22,14,26.9,0.0,0.0,11.8\n2025-05-22,15,27.9,0.0,0.0,10.3\n2025-05-22,16,28.4,0.0,0.0,9.2\n2025-05-22,17,28.3,0.0,0.0,8.7\n2025-05-22,18,27.4,0.0,0.0,9.6\n2025-05-22,19,25.8,0.0,0.0,10.8\n2025-05-22,20,23.4,0.0,0.0,10.7\n2025-05-22,21,21.7,0.0,0.0,10.2\n2025-05-22,22,20.8,0.0,0.0,10.9\n2025-05-22,23,20.6,0.0,0.0,9.5\n2025-05-23,0,20.4,0.0,0.0,11.3\n2025-05-23,1,19.6,0.0,0.0,15.2\n2025-05-23,2,18.8,0.0,0.0,15.0\n2025-05-23,3,18.4,0.0,0.0,15.4\n2025-05-23,4,17.5,0.3,0.0,15.0\n2025-05-23,5,17.3,0.1,0.0,14.6\n2025-05-23,6,17.2,0.0,0.0,14.2\n2025-05-23,7,17.5,0.0,0.0,11.3\n2025-05-23,8,18.4,0.0,0.0,10.0\n2025-05-23,9,19.6,0.0,0.0,10.6\n2025-05-23,10,21.3,0.0,0.0,7.8\n2025-05-23,11,21.6,0.0,0.0,13.4\n2025-05-23,12,22.3,0.0,0.0,14.9\n2025-05-23,13,22.4,0.0,0.0,13.3\n2025-05-23,14,23.2,0.0,0.0,12.8\n2025-05-23,15,24.4,0.0,0.0,12.3\n2025-05-23,16,24.7,0.0,0.0,13.5\n2025-05-23,17,24.3,0.0,0.0,13.2\n2025-05-23,18,23.6,0.0,0.0,12.8\n2025-05-23,19,22.3,0.0,0.0,5.6\n2025-05-23,20,21.0,0.0,0.0,7.5\n2025-05-23,21,20.5,0.0,0.0,7.4\n2025-05-23,22,16.9,0.5,0.0,9.7\n2025-05-23,23,16.7,0.1,0.0,11.6\n2025-05-24,0,16.4,0.1,0.0,10.9\n2025-05-24,1,15.9,0.0,0.0,12.6\n2025-05-24,2,15.4,0.0,0.0,11.8\n2025-05-24,3,14.9,0.0,0.0,9.2\n2025-05-24,4,14.6,0.0,0.0,9.3\n2025-05-24,5,14.4,0.1,0.0,9.0\n2025-05-24,6,14.4,0.0,0.0,9.7\n2025-05-24,7,14.8,0.0,0.0,7.0\n2025-05-24,8,15.0,0.2,0.0,9.9\n2025-05-24,9,15.8,0.0,0.0,11.3\n2025-05-24,10,16.0,0.0,0.0,11.4\n2025-05-24,11,16.5,0.0,0.0,13.0\n2025-05-24,12,17.2,0.0,0.0,13.6\n2025-05-24,13,18.3,0.0,0.0,13.8\n2025-05-24,14,18.6,0.0,0.0,13.6\n2025-05-24,15,18.6,0.0,0.0,14.8\n2025-05-24,16,18.9,0.0,0.0,14.6\n2025-05-24,17,18.8,0.0,0.0,14.5\n2025-05-24,18,18.2,0.0,0.0,13.0\n2025-05-24,19,17.4,0.0,0.0,12.6\n2025-05-24,20,16.1,0.0,0.0,11.2\n2025-05-24,21,15.2,0.0,0.0,10.4\n2025-05-24,22,15.7,0.0,0.0,10.1\n2025-05-24,23,14.4,0.0,0.0,9.2\n2025-05-25,0,14.3,0.0,0.0,7.2\n2025-05-25,1,13.2,0.0,0.0,3.9\n2025-05-25,2,12.7,0.0,0.0,2.2\n2025-05-25,3,12.0,0.0,0.0,3.6\n2025-05-25,4,11.5,0.0,0.0,2.2\n2025-05-25,5,10.9,0.0,0.0,3.1\n2025-05-25,6,11.2,0.0,0.0,1.1\n2025-05-25,7,12.9,0.0,0.0,1.1\n2025-05-25,8,15.5,0.0,0.0,3.6\n2025-05-25,9,17.5,0.0,0.0,6.9\n2025-05-25,10,18.4,0.0,0.0,5.2\n2025-05-25,11,19.7,0.0,0.0,8.2\n2025-05-25,12,20.6,0.0,0.0,11.4\n2025-05-25,13,20.6,0.0,0.0,14.9\n2025-05-25,14,20.6,0.0,0.0,14.3\n2025-05-25,15,20.4,0.0,0.0,14.4\n2025-05-25,16,19.9,0.0,0.0,13.9\n2025-05-25,17,19.3,0.0,0.0,12.6\n2025-05-25,18,18.4,0.0,0.0,10.2\n2025-05-25,19,17.3,0.0,0.0,7.7\n2025-05-25,20,15.8,0.0,0.0,7.7\n2025-05-25,21,15.0,0.0,0.0,6.7\n2025-05-25,22,14.8,0.0,0.0,6.6\n2025-05-25,23,14.3,0.0,0.0,1.3\n2025-05-26,0,14.0,0.0,0.0,0.4\n2025-05-26,1,13.3,0.0,0.0,3.1\n2025-05-26,2,12.9,0.0,0.0,4.5\n2025-05-26,3,12.4,0.0,0.0,4.0\n2025-05-26,4,12.2,0.0,0.0,7.6\n2025-05-26,5,12.0,0.0,0.0,6.2\n2025-05-26,6,12.1,0.0,0.0,6.9\n2025-05-26,7,13.5,0.0,0.0,5.6\n2025-05-26,8,16.0,0.0,0.0,4.6\n2025-05-26,9,18.4,0.0,0.0,5.5\n2025-05-26,10,20.2,0.0,0.0,3.1\n2025-05-26,11,22.2,0.0,0.0,3.1\n2025-05-26,12,24.1,0.0,0.0,2.3\n2025-05-26,13,25.4,0.0,0.0,2.9\n2025-05-26,14,25.9,0.0,0.0,6.9\n2025-05-26,15,25.9,0.0,0.0,10.7\n2025-05-26,16,25.7,0.0,0.0,11.6\n2025-05-26,17,24.1,0.0,0.0,13.8\n2025-05-26,18,21.0,0.0,0.0,12.4\n2025-05-26,19,19.0,0.0,0.0,8.6\n2025-05-26,20,17.6,0.0,0.0,5.2\n2025-05-26,21,17.0,0.0,0.0,2.4\n2025-05-26,22,16.9,0.0,0.0,3.5\n2025-05-26,23,16.1,0.0,0.0,4.2\n2025-05-27,0,15.4,0.0,0.0,4.0\n2025-05-27,1,14.8,0.0,0.0,6.0\n2025-05-27,2,14.2,0.0,0.0,7.4\n2025-05-27,3,13.8,0.0,0.0,7.5\n2025-05-27,4,13.5,0.0,0.0,7.5\n2025-05-27,5,13.1,0.0,0.0,7.9\n2025-05-27,6,13.3,0.0,0.0,8.7\n2025-05-27,7,14.9,0.0,0.0,7.1\n2025-05-27,8,17.2,0.0,0.0,7.5\n2025-05-27,9,19.3,0.0,0.0,6.6\n2025-05-27,10,21.5,0.0,0.0,5.8\n2025-05-27,11,23.4,0.0,0.0,5.4\n2025-05-27,12,24.7,0.0,0.0,6.4\n2025-05-27,13,25.2,0.0,0.0,9.8\n2025-05-27,14,24.9,0.0,0.0,12.7\n2025-05-27,15,24.7,0.0,0.0,10.5\n2025-05-27,16,24.1,0.0,0.0,11.6\n2025-05-27,17,22.9,0.0,0.0,11.2\n2025-05-27,18,21.0,0.0,0.0,9.8\n2025-05-27,19,18.5,0.0,0.0,8.6\n2025-05-27,20,16.7,0.0,0.0,6.7\n2025-05-27,21,15.9,0.0,0.0,4.6\n2025-05-27,22,17.2,0.0,0.0,5.4\n2025-05-27,23,16.3,0.0,0.0,5.8\n2025-05-28,0,15.8,0.0,0.0,2.8\n2025-05-28,1,15.3,0.0,0.0,0.4\n2025-05-28,2,14.9,0.0,0.0,3.2\n2025-05-28,3,14.4,0.0,0.0,4.6\n2025-05-28,4,14.0,0.0,0.0,5.8\n2025-05-28,5,13.7,0.0,0.0,4.6\n2025-05-28,6,13.8,0.0,0.0,4.7\n2025-05-28,7,15.2,0.0,0.0,1.9\n2025-05-28,8,17.4,0.1,0.0,2.1\n2025-05-28,9,19.5,0.0,0.0,1.0\n2025-05-28,10,22.2,0.0,0.0,4.1\n2025-05-28,11,23.8,0.0,0.0,1.6\n2025-05-28,12,24.6,0.0,0.0,7.2\n2025-05-28,13,25.1,0.0,0.0,8.0\n2025-05-28,14,25.3,0.0,0.0,11.9\n2025-05-28,15,24.9,0.0,0.0,9.9\n2025-05-28,16,24.4,0.0,0.0,10.5\n2025-05-28,17,24.2,0.0,0.0,7.2\n2025-05-28,18,23.2,0.3,0.0,3.0\n2025-05-28,19,22.2,0.0,0.0,4.4\n2025-05-28,20,20.6,0.0,0.0,2.8\n2025-05-28,21,19.7,0.0,0.0,1.1\n2025-05-28,22,18.3,0.0,0.0,7.6\n2025-05-28,23,18.2,0.0,0.0,3.9\n2025-05-29,0,17.4,0.0,0.0,0.2\n2025-05-29,1,17.0,0.0,0.0,4.9\n2025-05-29,2,16.7,0.0,0.0,4.3\n2025-05-29,3,16.1,0.0,0.0,4.4\n2025-05-29,4,15.6,0.0,0.0,4.0\n2025-05-29,5,15.2,0.0,0.0,4.7\n2025-05-29,6,15.2,0.0,0.0,2.3\n2025-05-29,7,16.7,0.0,0.0,1.5\n2025-05-29,8,19.0,0.0,0.0,0.8\n2025-05-29,9,21.3,0.0,0.0,1.4\n2025-05-29,10,23.3,0.0,0.0,3.1\n2025-05-29,11,24.6,0.0,0.0,5.5\n2025-05-29,12,25.5,0.0,0.0,6.8\n2025-05-29,13,26.0,0.0,0.0,9.0\n2025-05-29,14,26.1,0.0,0.0,13.1\n2025-05-29,15,25.9,0.0,0.0,13.7\n2025-05-29,16,25.4,0.0,0.0,13.6\n2025-05-29,17,24.6,0.0,0.0,12.3\n2025-05-29,18,23.4,0.0,0.0,12.3\n2025-05-29,19,22.0,0.0,0.0,10.2\n2025-05-29,20,20.8,0.0,0.0,6.9\n2025-05-29,21,19.8,0.0,0.0,5.3\n2025-05-29,22,19.7,0.0,0.0,5.9\n2025-05-29,23,18.8,0.0,0.0,6.0\n2025-05-30,0,18.2,0.0,0.0,5.2\n2025-05-30,1,17.8,0.0,0.0,3.5\n2025-05-30,2,17.3,0.0,0.0,3.4\n2025-05-30,3,17.0,0.0,0.0,4.0\n2025-05-30,4,16.2,0.0,0.0,3.0\n2025-05-30,5,15.9,0.0,0.0,3.2\n2025-05-30,6,16.5,0.0,0.0,4.4\n2025-05-30,7,18.1,0.0,0.0,2.7\n2025-05-30,8,20.4,0.0,0.0,1.7\n2025-05-30,9,22.7,0.0,0.0,2.1\n2025-05-30,10,24.1,0.0,0.0,3.2\n2025-05-30,11,25.7,0.0,0.0,1.9\n2025-05-30,12,26.6,0.0,0.0,6.4\n2025-05-30,13,27.3,0.0,0.0,7.8\n2025-05-30,14,28.1,0.0,0.0,8.4\n2025-05-30,15,28.6,0.0,0.0,8.4\n2025-05-30,16,28.7,0.0,0.0,7.2\n2025-05-30,17,27.6,0.0,0.0,7.7\n2025-05-30,18,26.7,0.0,0.0,3.3\n2025-05-30,19,25.7,0.0,0.0,1.5\n2025-05-30,20,23.5,0.0,0.0,2.1\n2025-05-30,21,21.7,0.0,0.0,1.8\n2025-05-30,22,18.6,0.0,0.0,7.9\n2025-05-30,23,18.5,0.0,0.0,3.8\n2025-05-31,0,18.2,0.0,0.0,0.5\n2025-05-31,1,17.7,0.0,0.0,2.5\n2025-05-31,2,16.9,0.0,0.0,2.5\n2025-05-31,3,16.6,0.0,0.0,7.9\n2025-05-31,4,15.8,0.0,0.0,4.5\n2025-05-31,5,15.1,0.0,0.0,7.4\n2025-05-31,6,15.4,0.0,0.0,6.8\n2025-05-31,7,17.0,0.0,0.0,4.9\n2025-05-31,8,19.1,0.0,0.0,4.8\n2025-05-31,9,21.7,0.0,0.0,4.5\n2025-05-31,10,23.1,0.0,0.0,4.2\n2025-05-31,11,24.4,0.0,0.0,6.2\n2025-05-31,12,24.8,0.0,0.0,8.0\n2025-05-31,13,24.9,0.0,0.0,9.2\n2025-05-31,14,24.8,0.0,0.0,11.1\n2025-05-31,15,23.5,0.0,0.0,13.8\n2025-05-31,16,23.4,0.0,0.0,11.0\n2025-05-31,17,23.6,0.0,0.0,10.1\n2025-05-31,18,22.0,0.0,0.0,10.8\n2025-05-31,19,20.4,0.0,0.0,9.0\n2025-05-31,20,18.7,0.0,0.0,7.2\n2025-05-31,21,18.4,0.0,0.0,3.7\n2025-05-31,22,18.6,0.0,0.0,3.0\n2025-05-31,23,17.9,0.0,0.0,3.4\n2025-06-01,0,16.9,0.0,0.0,5.6\n2025-06-01,1,16.2,0.0,0.0,3.1\n2025-06-01,2,16.1,0.0,0.0,2.8\n2025-06-01,3,15.8,0.0,0.0,3.8\n2025-06-01,4,15.4,0.0,0.0,2.7\n2025-06-01,5,14.7,0.0,0.0,2.3\n2025-06-01,6,15.0,0.0,0.0,2.4\n2025-06-01,7,16.8,0.0,0.0,3.3\n2025-06-01,8,19.5,0.0,0.0,2.2\n2025-06-01,9,22.4,0.0,0.0,3.1\n2025-06-01,10,23.3,0.0,0.0,6.3\n2025-06-01,11,24.4,0.0,0.0,7.8\n2025-06-01,12,25.7,0.0,0.0,8.3\n2025-06-01,13,26.2,0.0,0.0,11.5\n2025-06-01,14,26.2,0.0,0.0,12.0\n2025-06-01,15,26.1,0.0,0.0,12.4\n2025-06-01,16,25.2,0.0,0.0,12.8\n2025-06-01,17,24.0,0.0,0.0,11.0\n2025-06-01,18,22.7,0.0,0.0,9.7\n2025-06-01,19,21.0,0.0,0.0,8.4\n2025-06-01,20,19.5,0.0,0.0,6.3\n2025-06-01,21,18.9,0.0,0.0,5.6\n2025-06-01,22,19.1,0.0,0.0,6.2\n2025-06-01,23,18.7,0.0,0.0,3.5\n2025-06-02,0,18.8,0.0,0.0,3.4\n2025-06-02,1,18.5,0.0,0.0,0.4\n2025-06-02,2,18.5,0.0,0.0,3.1\n2025-06-02,3,18.0,0.0,0.0,6.1\n2025-06-02,4,17.2,0.0,0.0,4.0\n2025-06-02,5,17.0,0.0,0.0,4.3\n2025-06-02,6,16.8,0.0,0.0,4.5\n2025-06-02,7,17.0,0.0,0.0,0.4\n2025-06-02,8,17.3,0.1,0.0,0.9\n2025-06-02,9,17.9,0.1,0.0,2.9\n2025-06-02,10,18.7,0.3,0.0,4.8\n2025-06-02,11,18.4,0.4,0.0,7.8\n2025-06-02,12,20.1,0.1,0.0,9.7\n2025-06-02,13,20.6,0.0,0.0,1.0\n2025-06-02,14,21.4,0.0,0.0,4.1\n2025-06-02,15,23.2,0.0,0.0,3.3\n2025-06-02,16,24.3,0.0,0.0,4.1\n2025-06-02,17,23.8,0.0,0.0,7.5\n2025-06-02,18,22.8,0.0,0.0,9.4\n2025-06-02,19,21.4,0.0,0.0,9.2\n2025-06-02,20,19.7,0.0,0.0,6.9\n2025-06-02,21,18.7,0.0,0.0,5.4\n2025-06-02,22,19.2,0.0,0.0,5.0\n2025-06-02,23,18.8,0.0,0.0,4.7\n2025-06-03,0,18.4,0.0,0.0,4.8\n2025-06-03,1,18.4,0.0,0.0,4.3\n2025-06-03,2,18.4,0.0,0.0,4.1\n2025-06-03,3,18.2,0.0,0.0,0.9\n2025-06-03,4,17.8,0.0,0.0,7.6\n2025-06-03,5,17.1,0.0,0.0,8.1\n2025-06-03,6,16.8,0.0,0.0,11.1\n2025-06-03,7,17.9,0.0,0.0,8.9\n2025-06-03,8,19.6,0.0,0.0,9.0\n2025-06-03,9,21.7,0.0,0.0,6.5\n2025-06-03,10,23.5,0.0,0.0,9.0\n2025-06-03,11,23.7,0.0,0.0,16.2\n2025-06-03,12,23.1,0.0,0.0,19.6\n2025-06-03,13,22.4,0.0,0.0,18.5\n2025-06-03,14,21.8,0.0,0.0,21.3\n2025-06-03,15,21.4,0.0,0.0,21.2\n2025-06-03,16,21.5,0.0,0.0,19.1\n2025-06-03,17,21.7,0.0,0.0,19.5\n2025-06-03,18,21.5,0.0,0.0,22.0\n2025-06-03,19,19.7,0.0,0.0,18.0\n2025-06-03,20,18.0,0.0,0.0,16.4\n2025-06-03,21,17.3,0.0,0.0,13.4\n2025-06-03,22,17.2,0.0,0.0,13.6\n2025-06-03,23,16.7,0.0,0.0,10.2\n2025-06-04,0,16.3,0.0,0.0,11.3\n2025-06-04,1,16.1,0.0,0.0,11.6\n2025-06-04,2,15.6,0.0,0.0,11.3\n2025-06-04,3,15.3,0.0,0.0,10.7\n2025-06-04,4,15.4,0.0,0.0,9.7\n2025-06-04,5,15.7,0.0,0.0,9.6\n2025-06-04,6,16.0,0.0,0.0,10.4\n2025-06-04,7,17.7,0.0,0.0,10.2\n2025-06-04,8,19.4,0.0,0.0,13.6\n2025-06-04,9,21.0,0.0,0.0,15.4\n2025-06-04,10,21.5,0.0,0.0,17.4\n2025-06-04,11,22.9,0.0,0.0,17.1\n2025-06-04,12,23.8,0.0,0.0,21.4\n2025-06-04,13,24.0,0.0,0.0,24.5\n2025-06-04,14,24.0,0.0,0.0,27.6\n2025-06-04,15,23.9,0.0,0.0,27.0\n2025-06-04,16,23.5,0.0,0.0,26.1\n2025-06-04,17,23.1,0.0,0.0,26.5\n2025-06-04,18,22.9,0.0,0.0,23.0\n2025-06-04,19,21.0,0.0,0.0,18.4\n2025-06-04,20,19.1,0.0,0.0,16.4\n2025-06-04,21,18.8,0.0,0.0,16.0\n2025-06-04,22,18.4,0.0,0.0,13.6\n2025-06-04,23,17.9,0.0,0.0,10.4\n2025-06-05,0,17.3,0.0,0.0,8.1\n2025-06-05,1,16.7,0.0,0.0,9.6\n2025-06-05,2,16.2,0.0,0.0,7.2\n2025-06-05,3,15.8,0.0,0.0,5.9\n2025-06-05,4,15.0,0.0,0.0,2.3\n2025-06-05,5,14.8,0.0,0.0,0.6\n2025-06-05,6,15.2,0.0,0.0,4.2\n2025-06-05,7,16.7,0.0,0.0,8.2\n2025-06-05,8,18.9,0.0,0.0,6.2\n2025-06-05,9,21.3,0.0,0.0,6.9\n2025-06-05,10,23.2,0.0,0.0,9.8\n2025-06-05,11,24.5,0.0,0.0,13.2\n2025-06-05,12,25.5,0.0,0.0,15.0\n2025-06-05,13,25.9,0.0,0.0,15.9\n2025-06-05,14,26.0,0.0,0.0,17.1\n2025-06-05,15,26.2,0.0,0.0,16.2\n2025-06-05,16,25.7,0.0,0.0,13.9\n2025-06-05,17,24.8,0.0,0.0,13.7\n2025-06-05,18,23.8,0.0,0.0,15.5\n2025-06-05,19,22.1,0.0,0.0,12.9\n2025-06-05,20,21.4,0.0,0.0,10.8\n2025-06-05,21,21.1,0.0,0.0,11.0\n2025-06-05,22,20.0,0.0,0.0,8.8\n2025-06-05,23,19.2,0.0,0.0,10.3\n2025-06-06,0,18.6,0.0,0.0,10.7\n2025-06-06,1,18.3,0.0,0.0,10.6\n2025-06-06,2,18.5,0.0,0.0,11.0\n2025-06-06,3,18.5,0.0,0.0,11.1\n2025-06-06,4,18.3,0.0,0.0,10.6\n2025-06-06,5,18.3,0.0,0.0,10.2\n2025-06-06,6,18.3,0.0,0.0,10.5\n2025-06-06,7,19.7,0.0,0.0,10.3\n2025-06-06,8,21.3,0.0,0.0,11.4\n2025-06-06,9,23.1,0.0,0.0,13.7\n2025-06-06,10,25.2,0.0,0.0,14.4\n2025-06-06,11,26.6,0.0,0.0,16.0\n2025-06-06,12,28.0,0.0,0.0,16.7\n2025-06-06,13,28.7,0.0,0.0,18.2\n2025-06-06,14,28.4,0.0,0.0,19.2\n2025-06-06,15,27.6,0.0,0.0,21.0\n2025-06-06,16,26.6,0.0,0.0,21.3\n2025-06-06,17,25.8,0.0,0.0,19.2\n2025-06-06,18,24.9,0.0,0.0,16.9\n2025-06-06,19,23.7,0.0,0.0,14.4\n2025-06-06,20,22.5,0.0,0.0,12.8\n2025-06-06,21,21.4,0.0,0.0,12.4\n2025-06-06,22,21.3,0.0,0.0,11.0\n2025-06-06,23,20.6,0.0,0.0,10.8\n2025-06-07,0,20.1,0.0,0.0,10.8\n2025-06-07,1,19.8,0.0,0.0,9.8\n2025-06-07,2,19.3,0.0,0.0,8.5\n2025-06-07,3,19.0,0.0,0.0,7.8\n2025-06-07,4,18.6,0.0,0.0,9.3\n2025-06-07,5,18.2,0.0,0.0,9.6\n2025-06-07,6,18.3,0.0,0.0,8.6\n2025-06-07,7,19.7,0.0,0.0,3.9\n2025-06-07,8,21.3,0.0,0.0,6.4\n2025-06-07,9,23.0,0.0,0.0,8.0\n2025-06-07,10,24.5,0.0,0.0,4.0\n2025-06-07,11,25.6,0.0,0.0,10.6\n2025-06-07,12,26.6,0.0,0.0,12.7\n2025-06-07,13,27.2,0.0,0.0,13.6\n2025-06-07,14,27.2,0.1,0.0,15.6\n2025-06-07,15,25.6,0.0,0.0,17.3\n2025-06-07,16,25.0,0.0,0.0,16.7\n2025-06-07,17,23.8,0.0,0.0,15.8\n2025-06-07,18,22.5,0.0,0.0,13.9\n2025-06-07,19,21.2,0.0,0.0,10.1\n2025-06-07,20,20.1,0.0,0.0,7.5\n2025-06-07,21,20.0,0.0,0.0,7.5\n2025-06-07,22,20.5,0.1,0.0,7.0\n2025-06-07,23,20.5,0.0,0.0,5.0\n2025-06-08,0,20.4,0.0,0.0,5.5\n2025-06-08,1,20.1,0.1,0.0,7.1\n2025-06-08,2,19.9,0.1,0.0,5.6\n2025-06-08,3,19.7,0.1,0.0,5.4\n2025-06-08,4,19.7,0.1,0.0,4.2\n2025-06-08,5,19.8,0.0,0.0,3.5\n2025-06-08,6,19.9,0.0,0.0,2.7\n2025-06-08,7,20.1,0.0,0.0,1.9\n2025-06-08,8,20.8,0.1,0.0,1.4\n2025-06-08,9,22.0,0.1,0.0,1.1\n2025-06-08,10,24.4,0.1,0.0,0.7\n2025-06-08,11,26.4,0.2,0.0,3.7\n2025-06-08,12,27.7,0.0,0.0,5.8\n2025-06-08,13,28.9,0.0,0.0,8.2\n2025-06-08,14,28.6,0.0,0.0,11.5\n2025-06-08,15,27.9,0.0,0.0,13.2\n2025-06-08,16,27.2,0.0,0.0,14.8\n2025-06-08,17,26.2,0.0,0.0,13.8\n2025-06-08,18,25.2,0.0,0.0,11.4\n2025-06-08,19,24.4,0.0,0.0,7.2\n2025-06-08,20,23.4,0.0,0.0,4.6\n2025-06-08,21,21.9,0.0,0.0,7.0\n2025-06-08,22,20.5,0.0,0.0,5.7\n2025-06-08,23,19.4,0.0,0.0,7.1\n2025-06-09,0,19.1,0.0,0.0,8.9\n2025-06-09,1,19.2,0.0,0.0,9.4\n2025-06-09,2,19.2,0.0,0.0,10.5\n2025-06-09,3,19.2,0.0,0.0,11.5\n2025-06-09,4,18.8,0.0,0.0,8.9\n2025-06-09,5,18.4,0.0,0.0,6.1\n2025-06-09,6,18.5,0.0,0.0,2.9\n2025-06-09,7,18.8,0.0,0.0,1.9\n2025-06-09,8,19.9,0.0,0.0,3.7\n2025-06-09,9,21.4,0.1,0.0,6.8\n2025-06-09,10,22.7,0.0,0.0,5.3\n2025-06-09,11,25.5,0.0,0.0,7.8\n2025-06-09,12,26.4,0.0,0.0,9.4\n2025-06-09,13,27.0,0.0,0.0,11.0\n2025-06-09,14,27.1,0.0,0.0,12.3\n2025-06-09,15,26.4,0.0,0.0,14.3\n2025-06-09,16,25.0,0.0,0.0,15.2\n2025-06-09,17,24.2,0.0,0.0,12.8\n2025-06-09,18,22.8,0.0,0.0,11.9\n2025-06-09,19,21.3,0.0,0.0,10.2\n2025-06-09,20,19.8,0.0,0.0,9.4\n2025-06-09,21,19.2,0.0,0.0,7.6\n2025-06-09,22,19.0,0.0,0.0,6.3\n2025-06-09,23,19.0,0.0,0.0,4.9\n2025-06-10,0,18.9,0.0,0.0,1.9\n2025-06-10,1,18.8,0.0,0.0,6.4\n2025-06-10,2,18.7,0.0,0.0,5.4\n2025-06-10,3,18.7,0.5,0.0,1.8\n2025-06-10,4,18.4,0.0,0.0,3.8\n2025-06-10,5,18.0,0.0,0.0,7.5\n2025-06-10,6,17.8,0.0,0.0,9.3\n2025-06-10,7,18.1,0.0,0.0,10.3\n2025-06-10,8,19.1,0.0,0.0,11.7\n2025-06-10,9,19.8,0.0,0.0,13.1\n2025-06-10,10,21.5,0.0,0.0,10.3\n2025-06-10,11,20.6,0.1,0.0,12.0\n2025-06-10,12,20.0,0.2,0.0,11.9\n2025-06-10,13,20.8,0.0,0.0,11.6\n2025-06-10,14,23.3,0.0,0.0,13.5\n2025-06-10,15,23.6,0.0,0.0,14.4\n2025-06-10,16,23.4,0.0,0.0,15.4\n2025-06-10,17,22.5,0.0,0.0,16.1\n2025-06-10,18,21.6,0.0,0.0,14.5\n2025-06-10,19,20.6,0.0,0.0,12.8\n2025-06-10,20,20.2,0.0,0.0,12.7\n2025-06-10,21,20.5,0.0,0.0,14.0\n2025-06-10,22,20.1,0.0,0.0,13.1\n2025-06-10,23,19.4,0.0,0.0,10.6\n2025-06-11,0,18.5,0.0,0.0,8.7\n2025-06-11,1,17.9,0.0,0.0,5.7\n2025-06-11,2,17.2,0.0,0.0,3.6\n2025-06-11,3,16.5,0.0,0.0,3.7\n2025-06-11,4,16.1,0.0,0.0,2.4\n2025-06-11,5,15.8,0.0,0.0,4.2\n2025-06-11,6,16.4,0.0,0.0,4.1\n2025-06-11,7,18.0,0.0,0.0,4.4\n2025-06-11,8,20.1,0.0,0.0,5.0\n2025-06-11,9,22.6,0.0,0.0,5.7\n2025-06-11,10,24.0,0.0,0.0,6.6\n2025-06-11,11,25.4,0.0,0.0,10.6\n2025-06-11,12,25.9,0.0,0.0,13.8\n2025-06-11,13,25.8,0.0,0.0,16.4\n2025-06-11,14,25.9,0.0,0.0,16.8\n2025-06-11,15,25.6,0.0,0.0,18.0\n2025-06-11,16,25.2,0.0,0.0,19.0\n2025-06-11,17,24.2,0.0,0.0,17.7\n2025-06-11,18,23.2,0.0,0.0,13.9\n2025-06-11,19,22.2,0.0,0.0,9.0\n2025-06-11,20,21.1,0.0,0.0,6.2\n2025-06-11,21,20.4,0.0,0.0,4.6\n2025-06-11,22,19.7,0.0,0.0,5.6\n2025-06-11,23,19.3,0.0,0.0,6.8\n2025-06-12,0,18.8,0.0,0.0,8.6\n2025-06-12,1,18.4,0.0,0.0,6.6\n2025-06-12,2,18.0,0.0,0.0,4.3\n2025-06-12,3,17.5,0.0,0.0,3.1\n2025-06-12,4,16.8,0.0,0.0,2.1\n2025-06-12,5,16.6,0.0,0.0,3.7\n2025-06-12,6,16.6,0.0,0.0,4.8\n2025-06-12,7,18.7,0.0,0.0,4.2\n2025-06-12,8,21.5,0.0,0.0,3.6\n2025-06-12,9,24.3,0.0,0.0,2.4\n2025-06-12,10,26.7,0.0,0.0,4.3\n2025-06-12,11,28.7,0.0,0.0,6.1\n2025-06-12,12,30.1,0.0,0.0,6.2\n2025-06-12,13,31.4,0.0,0.0,2.0\n2025-06-12,14,31.4,0.0,0.0,8.5\n2025-06-12,15,30.1,0.0,0.0,15.9\n2025-06-12,16,28.9,0.0,0.0,17.2\n2025-06-12,17,29.3,0.0,0.0,11.6\n2025-06-12,18,28.9,0.0,0.0,11.7\n2025-06-12,19,27.5,0.0,0.0,10.6\n2025-06-12,20,25.8,0.0,0.0,7.1\n2025-06-12,21,24.5,0.0,0.0,4.1\n2025-06-12,22,23.1,0.0,0.0,4.5\n2025-06-12,23,22.5,0.0,0.0,1.9\n2025-06-13,0,21.3,0.0,0.0,5.0\n2025-06-13,1,20.2,0.0,0.0,6.4\n2025-06-13,2,19.5,0.0,0.0,3.2\n2025-06-13,3,18.8,0.0,0.0,2.1\n2025-06-13,4,18.6,0.0,0.0,1.2\n2025-06-13,5,18.6,0.0,0.0,3.6\n2025-06-13,6,18.9,0.0,0.0,6.2\n2025-06-13,7,20.0,0.0,0.0,6.6\n2025-06-13,8,21.6,0.0,0.0,6.3\n2025-06-13,9,23.4,0.0,0.0,6.7\n2025-06-13,10,25.2,0.0,0.0,6.5\n2025-06-13,11,27.1,0.0,0.0,8.0\n2025-06-13,12,26.3,0.0,0.0,8.7\n2025-06-13,13,26.8,0.0,0.0,7.3\n2025-06-13,14,27.2,0.0,0.0,7.2\n2025-06-13,15,27.2,0.0,0.0,11.1\n2025-06-13,16,25.4,0.0,0.0,12.2\n2025-06-13,17,24.0,0.5,0.0,11.0\n2025-06-13,18,22.9,0.3,0.0,9.7\n2025-06-13,19,22.3,0.2,0.0,9.6\n2025-06-13,20,21.7,0.2,0.0,11.7\n2025-06-13,21,21.2,0.1,0.0,11.8\n2025-06-13,22,20.4,0.7,0.0,7.5\n2025-06-13,23,20.2,0.2,0.0,9.7\n2025-06-14,0,19.9,0.0,0.0,13.1\n2025-06-14,1,19.8,0.0,0.0,11.0\n2025-06-14,2,19.5,0.0,0.0,8.0\n2025-06-14,3,19.3,0.0,0.0,7.2\n2025-06-14,4,19.2,0.1,0.0,6.3\n2025-06-14,5,19.2,0.0,0.0,4.3\n2025-06-14,6,19.3,0.0,0.0,5.7\n2025-06-14,7,20.4,0.0,0.0,5.8\n2025-06-14,8,22.1,0.0,0.0,4.9\n2025-06-14,9,23.8,0.0,0.0,4.3\n2025-06-14,10,24.8,0.0,0.0,7.1\n2025-06-14,11,26.3,0.0,0.0,5.2\n2025-06-14,12,28.4,0.0,0.0,3.9\n2025-06-14,13,29.3,0.0,0.0,6.1\n2025-06-14,14,29.2,0.0,0.0,8.4\n2025-06-14,15,30.0,0.0,0.0,10.6\n2025-06-14,16,29.3,0.0,0.0,12.0\n2025-06-14,17,27.5,0.0,0.0,13.7\n2025-06-14,18,24.9,0.0,0.0,12.5\n2025-06-14,19,22.8,0.0,0.0,9.9\n2025-06-14,20,21.3,0.0,0.0,10.7\n2025-06-14,21,20.6,0.0,0.0,10.5\n2025-06-14,22,20.5,0.0,0.0,7.6\n2025-06-14,23,20.7,0.0,0.0,5.4\n2025-06-15,0,20.7,0.0,0.0,2.3\n2025-06-15,1,20.0,0.0,0.0,7.7\n2025-06-15,2,19.3,0.0,0.0,7.4\n2025-06-15,3,18.7,0.0,0.0,7.0\n2025-06-15,4,18.4,0.0,0.0,9.1\n2025-06-15,5,18.1,0.0,0.0,9.3\n2025-06-15,6,18.2,0.0,0.0,8.6\n2025-06-15,7,19.6,0.0,0.0,7.7\n2025-06-15,8,21.4,0.0,0.0,8.0\n2025-06-15,9,22.9,0.0,0.0,7.0\n2025-06-15,10,25.7,0.0,0.0,9.9\n2025-06-15,11,27.7,0.0,0.0,9.4\n2025-06-15,12,29.9,0.0,0.0,10.3\n2025-06-15,13,30.7,0.0,0.0,14.3\n2025-06-15,14,31.0,0.0,0.0,19.0\n2025-06-15,15,30.3,0.0,0.0,18.2\n2025-06-15,16,28.3,0.0,0.0,13.6\n2025-06-15,17,28.3,0.0,0.0,16.1\n2025-06-15,18,27.1,0.0,0.0,18.1\n2025-06-15,19,25.8,0.1,0.0,20.7\n2025-06-15,20,23.2,1.8,0.0,19.0\n2025-06-15,21,22.7,1.5,0.0,18.2\n2025-06-15,22,22.0,3.2,0.0,11.8\n2025-06-15,23,21.2,1.9,0.0,12.3\n2025-06-16,0,20.9,2.6,0.0,12.1\n2025-06-16,1,20.8,1.6,0.0,9.5\n2025-06-16,2,20.9,0.5,0.0,10.9\n2025-06-16,3,21.0,0.2,0.0,8.2\n2025-06-16,4,20.8,0.6,0.0,9.8\n2025-06-16,5,20.4,0.1,0.0,14.0\n2025-06-16,6,20.3,0.0,0.0,11.8\n2025-06-16,7,20.3,0.0,0.0,10.1\n2025-06-16,8,20.7,0.0,0.0,11.5\n2025-06-16,9,21.4,0.0,0.0,11.6\n2025-06-16,10,22.5,0.1,0.0,9.1\n2025-06-16,11,23.6,0.1,0.0,10.5\n2025-06-16,12,23.8,0.4,0.0,14.9\n2025-06-16,13,24.3,0.0,0.0,15.1\n2025-06-16,14,24.6,0.1,0.0,22.6\n2025-06-16,15,24.2,0.1,0.0,17.1\n2025-06-16,16,25.7,0.0,0.0,23.5\n2025-06-16,17,25.8,0.0,0.0,20.4\n2025-06-16,18,24.6,0.0,0.0,21.7\n2025-06-16,19,23.3,0.0,0.0,20.0\n2025-06-16,20,22.1,0.0,0.0,19.1\n2025-06-16,21,21.4,0.0,0.0,17.9\n2025-06-16,22,21.3,0.0,0.0,15.3\n2025-06-16,23,20.9,0.0,0.0,16.2\n2025-06-17,0,20.7,0.0,0.0,14.6\n2025-06-17,1,19.8,0.0,0.0,16.3\n2025-06-17,2,19.9,0.0,0.0,15.3\n2025-06-17,3,20.0,0.0,0.0,15.5\n2025-06-17,4,20.0,0.0,0.0,16.1\n2025-06-17,5,20.1,0.0,0.0,14.4\n2025-06-17,6,20.3,0.0,0.0,13.0\n2025-06-17,7,21.4,0.0,0.0,14.3\n2025-06-17,8,22.5,0.0,0.0,17.5\n2025-06-17,9,23.5,0.0,0.0,16.8\n2025-06-17,10,24.8,0.0,0.0,10.4\n2025-06-17,11,25.6,0.0,0.0,9.4\n2025-06-17,12,25.6,0.0,0.0,9.4\n2025-06-17,13,26.4,0.0,0.0,10.1\n2025-06-17,14,26.6,0.0,0.0,11.1\n2025-06-17,15,26.3,0.0,0.0,10.5\n2025-06-17,16,26.2,0.0,0.0,9.7\n2025-06-17,17,25.4,0.0,0.0,9.8\n2025-06-17,18,24.2,0.0,0.0,9.0\n2025-06-17,19,22.8,0.0,0.0,6.7\n2025-06-17,20,21.6,0.0,0.0,4.0\n2025-06-17,21,21.0,0.0,0.0,3.5\n2025-06-17,22,21.6,0.0,0.0,4.3\n2025-06-17,23,20.9,0.0,0.0,4.5\n2025-06-18,0,20.1,0.0,0.0,5.3\n2025-06-18,1,19.8,0.0,0.0,5.8\n2025-06-18,2,19.7,0.0,0.0,6.7\n2025-06-18,3,19.8,0.0,0.0,6.3\n2025-06-18,4,19.8,0.0,0.0,7.9\n2025-06-18,5,19.7,0.0,0.0,6.2\n2025-06-18,6,19.6,0.0,0.0,4.3\n2025-06-18,7,20.2,0.0,0.0,2.9\n2025-06-18,8,21.7,0.0,0.0,4.8\n2025-06-18,9,23.0,0.0,0.0,6.3\n2025-06-18,10,24.1,0.0,0.0,6.5\n2025-06-18,11,25.2,0.0,0.0,8.0\n2025-06-18,12,26.3,0.0,0.0,7.8\n2025-06-18,13,26.9,0.0,0.0,9.0\n2025-06-18,14,26.9,0.0,0.0,10.8\n2025-06-18,15,26.7,0.0,0.0,10.3\n2025-06-18,16,26.4,0.0,0.0,9.8\n2025-06-18,17,25.8,0.0,0.0,7.8\n2025-06-18,18,25.4,0.0,0.0,5.6\n2025-06-18,19,24.3,0.0,0.0,3.8\n2025-06-18,20,22.7,0.0,0.0,3.9\n2025-06-18,21,21.7,0.0,0.0,3.5\n2025-06-18,22,22.1,0.0,0.0,5.0\n2025-06-18,23,21.3,0.0,0.0,3.1\n2025-06-19,0,20.7,0.0,0.0,2.0\n2025-06-19,1,20.5,0.0,0.0,1.4\n2025-06-19,2,20.3,0.0,0.0,4.0\n2025-06-19,3,20.0,0.0,0.0,6.0\n2025-06-19,4,19.7,0.0,0.0,7.5\n2025-06-19,5,19.5,0.0,0.0,8.5\n2025-06-19,6,20.0,0.0,0.0,9.5\n2025-06-19,7,21.6,0.0,0.0,9.1\n2025-06-19,8,24.0,0.0,0.0,9.3\n2025-06-19,9,26.2,0.0,0.0,9.8\n2025-06-19,10,27.4,0.0,0.0,7.3\n2025-06-19,11,29.2,0.0,0.0,9.6\n2025-06-19,12,30.5,0.0,0.0,9.0\n2025-06-19,13,31.6,0.0,0.0,10.5\n2025-06-19,14,32.0,0.0,0.0,12.0\n2025-06-19,15,32.1,0.1,0.0,17.1\n2025-06-19,16,29.8,0.0,0.0,15.0\n2025-06-19,17,29.7,0.2,0.0,12.0\n2025-06-19,18,27.8,0.1,0.0,10.3\n2025-06-19,19,27.7,0.0,0.0,6.8\n2025-06-19,20,27.0,0.0,0.0,5.2\n2025-06-19,21,26.3,0.0,0.0,4.9\n2025-06-19,22,24.0,0.0,0.0,6.2\n2025-06-19,23,24.1,0.0,0.0,8.8\n2025-06-20,0,24.2,0.1,0.0,11.7\n2025-06-20,1,23.3,1.3,0.0,13.0\n2025-06-20,2,22.6,1.7,0.0,12.5\n2025-06-20,3,22.7,2.2,0.0,13.9\n2025-06-20,4,22.8,1.6,0.0,14.8\n2025-06-20,5,23.1,2.3,0.0,16.2\n2025-06-20,6,23.2,1.8,0.0,20.1\n2025-06-20,7,23.1,3.7,0.0,18.5\n2025-06-20,8,23.2,3.9,0.0,21.4\n2025-06-20,9,23.1,2.8,0.0,25.5\n2025-06-20,10,24.1,0.7,0.0,25.9\n2025-06-20,11,23.7,2.1,0.0,28.1\n2025-06-20,12,23.4,2.6,0.0,28.3\n2025-06-20,13,23.9,2.1,0.0,31.9\n2025-06-20,14,24.0,1.8,0.0,33.7\n2025-06-20,15,23.3,2.5,0.0,35.5\n2025-06-20,16,22.9,5.9,0.0,35.8\n2025-06-20,17,22.3,5.1,0.0,33.4\n2025-06-20,18,21.8,2.4,0.0,30.3\n2025-06-20,19,22.0,0.6,0.0,26.6\n2025-06-20,20,21.9,0.1,0.0,24.1\n2025-06-20,21,22.0,0.0,0.0,22.0\n2025-06-20,22,21.8,0.0,0.0,16.5\n2025-06-20,23,21.4,0.0,0.0,17.5\n2025-06-21,0,21.0,0.0,0.0,17.1\n2025-06-21,1,20.9,0.0,0.0,16.5\n2025-06-21,2,20.4,0.2,0.0,14.8\n2025-06-21,3,19.9,0.5,0.0,13.0\n2025-06-21,4,19.7,0.1,0.0,8.4\n2025-06-21,5,19.8,0.0,0.0,5.9\n2025-06-21,6,20.0,0.0,0.0,4.8\n2025-06-21,7,20.3,0.0,0.0,7.1\n2025-06-21,8,20.5,0.0,0.0,8.1\n2025-06-21,9,20.8,0.0,0.0,9.5\n2025-06-21,10,22.0,0.0,0.0,5.2\n2025-06-21,11,22.5,0.0,0.0,5.9\n2025-06-21,12,22.4,0.0,0.0,9.4\n2025-06-21,13,22.5,0.0,0.0,9.3\n2025-06-21,14,21.1,0.5,0.0,6.9\n2025-06-21,15,21.6,0.1,0.0,6.4\n2025-06-21,16,21.2,0.0,0.0,7.8\n2025-06-21,17,20.8,0.0,0.0,6.0\n2025-06-21,18,20.5,0.0,0.0,6.0\n2025-06-21,19,20.3,0.0,0.0,5.8\n2025-06-21,20,19.6,0.0,0.0,5.8\n2025-06-21,21,19.3,0.0,0.0,6.3\n2025-06-21,22,19.5,0.0,0.0,7.4\n2025-06-21,23,19.1,0.0,0.0,7.1\n2025-06-22,0,18.8,0.0,0.0,6.3\n2025-06-22,1,18.5,0.0,0.0,6.8\n2025-06-22,2,18.4,0.0,0.0,7.4\n2025-06-22,3,18.1,0.0,0.0,6.8\n2025-06-22,4,17.3,0.0,0.0,3.3\n2025-06-22,5,16.9,0.0,0.0,2.3\n2025-06-22,6,17.2,0.0,0.0,5.4\n2025-06-22,7,18.7,0.0,0.0,5.5\n2025-06-22,8,20.5,0.0,0.0,5.8\n2025-06-22,9,21.9,0.0,0.0,7.6\n2025-06-22,10,23.0,0.0,0.0,9.8\n2025-06-22,11,23.5,0.0,0.0,11.5\n2025-06-22,12,23.9,0.0,0.0,12.3\n2025-06-22,13,24.2,0.0,0.0,13.0\n2025-06-22,14,24.4,0.0,0.0,13.0\n2025-06-22,15,24.3,0.0,0.0,13.3\n2025-06-22,16,24.0,0.0,0.0,12.4\n2025-06-22,17,23.5,0.0,0.0,10.2\n2025-06-22,18,22.6,0.0,0.0,10.0\n2025-06-22,19,21.4,0.0,0.0,8.4\n2025-06-22,20,20.2,0.0,0.0,7.2\n2025-06-22,21,19.6,0.0,0.0,6.4\n2025-06-22,22,19.6,0.0,0.0,7.7\n2025-06-22,23,19.3,0.0,0.0,6.0\n2025-06-23,0,18.9,0.0,0.0,5.4\n2025-06-23,1,18.5,0.0,0.0,5.0\n2025-06-23,2,18.2,0.0,0.0,5.4\n2025-06-23,3,18.1,0.0,0.0,6.8\n2025-06-23,4,17.9,0.0,0.0,6.1\n2025-06-23,5,17.7,0.0,0.0,5.6\n2025-06-23,6,18.1,0.0,0.0,6.9\n2025-06-23,7,18.5,0.0,0.0,5.2\n2025-06-23,8,19.5,0.0,0.0,3.2\n2025-06-23,9,21.5,0.0,0.0,2.8\n2025-06-23,10,23.6,0.0,0.0,0.4\n2025-06-23,11,25.4,0.0,0.0,1.9\n2025-06-23,12,26.7,0.0,0.0,2.6\n2025-06-23,13,27.4,0.0,0.0,4.2\n2025-06-23,14,27.7,0.0,0.0,6.5\n2025-06-23,15,27.9,0.0,0.0,6.6\n2025-06-23,16,28.0,0.0,0.0,7.1\n2025-06-23,17,27.6,0.0,0.0,8.0\n2025-06-23,18,26.9,0.0,0.0,7.8\n2025-06-23,19,25.5,0.0,0.0,7.6\n2025-06-23,20,24.2,0.0,0.0,7.4\n2025-06-23,21,23.7,0.0,0.0,5.0\n2025-06-23,22,24.0,0.0,0.0,5.2\n2025-06-23,23,23.7,0.0,0.0,6.6\n2025-06-24,0,23.4,0.0,0.0,5.4\n2025-06-24,1,23.2,0.0,0.0,7.0\n2025-06-24,2,22.8,0.0,0.0,5.5\n2025-06-24,3,22.3,0.0,0.0,4.7\n2025-06-24,4,22.4,0.0,0.0,5.4\n2025-06-24,5,22.4,0.0,0.0,6.3\n2025-06-24,6,22.4,0.0,0.0,7.5\n2025-06-24,7,23.1,0.0,0.0,7.9\n2025-06-24,8,24.0,0.0,0.0,7.2\n2025-06-24,9,25.2,0.0,0.0,5.7\n2025-06-24,10,25.5,0.0,0.0,5.9\n2025-06-24,11,25.9,0.0,0.0,7.8\n2025-06-24,12,26.4,0.0,0.0,7.4\n2025-06-24,13,27.1,0.0,0.0,7.8\n2025-06-24,14,27.0,0.0,0.0,8.7\n2025-06-24,15,27.1,0.0,0.0,8.7\n2025-06-24,16,26.5,0.0,0.0,6.9\n2025-06-24,17,25.6,0.0,0.0,7.9\n2025-06-24,18,26.2,0.2,0.0,10.1\n2025-06-24,19,24.4,0.0,0.0,8.8\n2025-06-24,20,23.1,0.3,0.0,8.1\n2025-06-24,21,21.5,0.0,0.0,4.8\n2025-06-24,22,21.8,1.3,0.0,9.4\n2025-06-24,23,21.6,0.1,0.0,12.2\n2025-06-25,0,20.9,0.4,0.0,11.4\n2025-06-25,1,20.6,0.1,0.0,9.7\n2025-06-25,2,20.5,0.0,0.0,11.3\n2025-06-25,3,20.3,0.0,0.0,11.4\n2025-06-25,4,20.1,0.0,0.0,11.0\n2025-06-25,5,20.2,0.0,0.0,11.7\n2025-06-25,6,20.3,2.4,0.0,7.9\n2025-06-25,7,19.6,1.4,0.0,8.0\n2025-06-25,8,19.8,1.4,0.0,9.5\n2025-06-25,9,20.3,0.2,0.0,9.9\n2025-06-25,10,19.8,0.0,0.0,9.9\n2025-06-25,11,21.1,0.0,0.0,8.5\n2025-06-25,12,22.1,0.2,0.0,7.1\n2025-06-25,13,22.5,0.6,0.0,7.0\n2025-06-25,14,22.7,1.3,0.0,6.9\n2025-06-25,15,22.5,0.9,0.0,5.3\n2025-06-25,16,22.5,1.2,0.0,8.0\n2025-06-25,17,22.2,0.7,0.0,5.1\n2025-06-25,18,22.0,0.7,0.0,4.7\n2025-06-25,19,21.4,0.6,0.0,8.1\n2025-06-25,20,21.0,0.1,0.0,3.4\n2025-06-25,21,20.9,0.0,0.0,0.8\n2025-06-25,22,20.1,0.0,0.0,2.4\n2025-06-25,23,20.3,0.0,0.0,0.9\n2025-06-26,0,20.3,0.0,0.0,1.2\n2025-06-26,1,20.0,0.0,0.0,3.7\n2025-06-26,2,19.7,0.0,0.0,8.6\n2025-06-26,3,19.7,0.0,0.0,8.8\n2025-06-26,4,19.6,0.0,0.0,8.5\n2025-06-26,5,19.4,0.0,0.0,8.4\n2025-06-26,6,19.3,0.0,0.0,6.8\n2025-06-26,7,19.9,0.0,0.0,6.9\n2025-06-26,8,20.6,0.0,0.0,6.0\n2025-06-26,9,22.1,0.0,0.0,4.9\n2025-06-26,10,21.9,0.0,0.0,9.7\n2025-06-26,11,22.1,0.0,0.0,10.3\n2025-06-26,12,22.5,0.0,0.0,10.6\n2025-06-26,13,23.5,0.0,0.0,10.1\n2025-06-26,14,23.3,0.0,0.0,9.2\n2025-06-26,15,23.7,0.0,0.0,9.6\n2025-06-26,16,23.8,0.0,0.0,7.6\n2025-06-26,17,24.0,0.0,0.0,4.5\n2025-06-26,18,23.8,0.0,0.0,4.9\n2025-06-26,19,22.7,0.0,0.0,6.4\n2025-06-26,20,21.2,0.0,0.0,7.6\n2025-06-26,21,20.5,0.0,0.0,4.5\n2025-06-26,22,20.2,0.0,0.0,4.5\n2025-06-26,23,19.9,0.0,0.0,2.5\n2025-06-27,0,19.7,0.0,0.0,2.9\n2025-06-27,1,19.5,0.0,0.0,0.8\n2025-06-27,2,19.8,0.0,0.0,2.3\n2025-06-27,3,19.7,0.0,0.0,2.6\n2025-06-27,4,19.2,0.0,0.0,2.6\n2025-06-27,5,19.4,0.0,0.0,2.2\n2025-06-27,6,19.6,0.0,0.0,5.1\n2025-06-27,7,20.1,0.0,0.0,4.0\n2025-06-27,8,20.8,0.0,0.0,3.4\n2025-06-27,9,21.1,0.1,0.0,2.5\n2025-06-27,10,21.9,0.1,0.0,1.9\n2025-06-27,11,23.8,0.0,0.0,2.8\n2025-06-27,12,24.7,0.0,0.0,5.0\n2025-06-27,13,25.9,0.0,0.0,3.8\n2025-06-27,14,27.0,0.0,0.0,2.1\n2025-06-27,15,28.1,0.0,0.0,5.8\n2025-06-27,16,28.5,0.0,0.0,4.9\n2025-06-27,17,28.1,0.0,0.0,6.8\n2025-06-27,18,27.7,0.0,0.0,6.5\n2025-06-27,19,26.8,0.0,0.0,4.2\n2025-06-27,20,25.9,0.0,0.0,3.2\n2025-06-27,21,25.1,0.0,0.0,2.9\n2025-06-27,22,24.6,0.2,0.0,4.7\n2025-06-27,23,23.5,0.0,0.0,3.9\n2025-06-28,0,22.8,0.0,0.0,8.4\n2025-06-28,1,22.2,0.0,0.0,3.6\n2025-06-28,2,22.2,0.6,0.0,9.2\n2025-06-28,3,22.1,0.1,0.0,7.0\n2025-06-28,4,22.0,0.2,0.0,8.7\n2025-06-28,5,22.2,0.3,0.0,9.2\n2025-06-28,6,22.2,0.4,0.0,9.5\n2025-06-28,7,22.5,0.7,0.0,8.9\n2025-06-28,8,23.4,0.2,0.0,11.0\n2025-06-28,9,24.1,0.2,0.0,12.4\n2025-06-28,10,25.5,0.1,0.0,13.2\n2025-06-28,11,25.6,0.2,0.0,12.2\n2025-06-28,12,26.2,0.1,0.0,8.3\n2025-06-28,13,26.9,0.1,0.0,6.7\n2025-06-28,14,27.5,0.1,0.0,6.1\n2025-06-28,15,27.6,0.0,0.0,9.1\n2025-06-28,16,27.1,0.0,0.0,9.2\n2025-06-28,17,26.5,0.0,0.0,8.4\n2025-06-28,18,25.6,0.0,0.0,6.0\n2025-06-28,19,24.7,0.0,0.0,4.4\n2025-06-28,20,23.8,0.0,0.0,3.6\n2025-06-28,21,23.4,0.0,0.0,4.4\n2025-06-28,22,24.2,0.2,0.0,5.5\n2025-06-28,23,24.3,0.0,0.0,5.8\n2025-06-29,0,24.3,0.0,0.0,5.8\n2025-06-29,1,24.1,0.0,0.0,6.2\n2025-06-29,2,23.9,0.0,0.0,9.4\n2025-06-29,3,23.6,0.0,0.0,10.7\n2025-06-29,4,23.2,0.1,0.0,10.8\n2025-06-29,5,23.1,0.1,0.0,6.0\n2025-06-29,6,23.4,0.0,0.0,8.5\n2025-06-29,7,23.9,0.1,0.0,12.7\n2025-06-29,8,24.6,0.1,0.0,12.4\n2025-06-29,9,25.0,0.1,0.0,11.7\n2025-06-29,10,27.1,0.2,0.0,15.4\n2025-06-29,11,27.5,0.1,0.0,15.6\n2025-06-29,12,27.8,0.2,0.0,15.2\n2025-06-29,13,28.1,0.2,0.0,14.8\n2025-06-29,14,28.4,0.0,0.0,14.0\n2025-06-29,15,28.6,0.0,0.0,12.5\n2025-06-29,16,28.1,0.0,0.0,11.1\n2025-06-29,17,27.2,0.0,0.0,9.7\n2025-06-29,18,26.4,0.0,0.0,7.6\n2025-06-29,19,25.2,0.0,0.0,6.7\n2025-06-29,20,24.2,0.0,0.0,4.5\n2025-06-29,21,23.8,0.0,0.0,7.3\n2025-06-29,22,24.1,0.0,0.0,7.3\n2025-06-29,23,24.7,0.0,0.0,8.4\n2025-06-30,0,24.7,0.0,0.0,10.1\n2025-06-30,1,24.2,0.0,0.0,10.7\n2025-06-30,2,23.9,0.0,0.0,10.4\n2025-06-30,3,23.8,0.1,0.0,9.5\n2025-06-30,4,23.9,0.1,0.0,9.2\n2025-06-30,5,24.0,0.1,0.0,9.4\n2025-06-30,6,24.1,0.1,0.0,11.5\n2025-06-30,7,24.3,0.1,0.0,13.1\n2025-06-30,8,24.5,0.1,0.0,13.4\n2025-06-30,9,25.3,0.1,0.0,13.9\n2025-06-30,10,28.0,0.0,0.0,14.8\n2025-06-30,11,28.4,0.1,0.0,13.6\n2025-06-30,12,27.4,0.1,0.0,9.1\n2025-06-30,13,27.9,0.2,0.0,9.2\n2025-06-30,14,28.9,0.3,0.0,12.6\n2025-06-30,15,28.6,0.2,0.0,15.5\n2025-06-30,16,28.0,0.3,0.0,14.6\n2025-06-30,17,27.2,0.0,0.0,11.5\n2025-06-30,18,27.0,0.0,0.0,9.0\n2025-06-30,19,26.7,0.0,0.0,8.8\n2025-06-30,20,26.0,0.0,0.0,6.8\n2025-06-30,21,25.6,0.0,0.0,7.3\n2025-06-30,22,26.1,0.0,0.0,5.7\n2025-06-30,23,25.6,0.0,0.0,6.8\n2025-07-01,0,25.1,0.0,0.0,9.3\n2025-07-01,1,24.7,0.0,0.0,11.0\n2025-07-01,2,24.5,0.0,0.0,10.5\n2025-07-01,3,24.3,0.0,0.0,10.6\n2025-07-01,4,24.5,0.0,0.0,11.4\n2025-07-01,5,24.6,0.0,0.0,10.9\n2025-07-01,6,24.7,0.0,0.0,11.0\n2025-07-01,7,25.0,0.1,0.0,8.2\n2025-07-01,8,25.4,0.1,0.0,9.3\n2025-07-01,9,26.1,0.1,0.0,13.5\n2025-07-01,10,28.0,0.0,0.0,15.0\n2025-07-01,11,28.8,0.1,0.0,15.5\n2025-07-01,12,28.7,0.4,0.0,16.5\n2025-07-01,13,28.6,0.1,0.0,17.2\n2025-07-01,14,28.9,0.1,0.0,17.2\n2025-07-01,15,28.7,0.1,0.0,12.7\n2025-07-01,16,27.7,0.9,0.0,12.5\n2025-07-01,17,27.5,0.0,0.0,8.3\n2025-07-01,18,27.2,0.0,0.0,9.2\n2025-07-01,19,26.4,0.0,0.0,9.6\n2025-07-01,20,25.9,0.0,0.0,9.6\n2025-07-01,21,25.5,0.0,0.0,8.9\n2025-07-01,22,25.0,0.0,0.0,5.0\n2025-07-01,23,24.9,0.0,0.0,6.0\n2025-07-02,0,25.2,0.0,0.0,8.5\n2025-07-02,1,25.3,0.0,0.0,10.1\n2025-07-02,2,25.2,0.0,0.0,11.2\n2025-07-02,3,24.9,0.0,0.0,11.2\n2025-07-02,4,24.7,0.0,0.0,11.3\n2025-07-02,5,24.7,0.0,0.0,12.7\n2025-07-02,6,24.6,0.0,0.0,15.4\n2025-07-02,7,24.7,0.1,0.0,13.9\n2025-07-02,8,24.9,0.1,0.0,14.1\n2025-07-02,9,25.6,0.1,0.0,16.2\n2025-07-02,10,26.5,0.0,0.0,13.8\n2025-07-02,11,27.2,0.0,0.0,15.0\n2025-07-02,12,28.1,0.0,0.0,16.4\n2025-07-02,13,28.4,0.0,0.0,17.9\n2025-07-02,14,28.5,0.3,0.0,19.7\n2025-07-02,15,26.9,0.1,0.0,14.9\n2025-07-02,16,26.4,0.2,0.0,15.0\n2025-07-02,17,26.5,0.2,0.0,13.9\n2025-07-02,18,25.9,0.0,0.0,10.4\n2025-07-02,19,25.4,0.0,0.0,8.4\n2025-07-02,20,24.8,0.0,0.0,8.2\n2025-07-02,21,24.4,0.0,0.0,7.1\n2025-07-02,22,24.6,0.0,0.0,4.1\n2025-07-02,23,24.5,0.0,0.0,5.1\n2025-07-03,0,24.7,0.0,0.0,6.1\n2025-07-03,1,24.7,0.0,0.0,7.3\n2025-07-03,2,24.6,0.0,0.0,7.7\n2025-07-03,3,24.4,0.0,0.0,9.4\n2025-07-03,4,24.2,0.0,0.0,11.6\n2025-07-03,5,24.2,0.0,0.0,10.4\n2025-07-03,6,24.3,0.0,0.0,10.6\n2025-07-03,7,24.5,0.0,0.0,13.7\n2025-07-03,8,24.9,0.0,0.0,12.5\n2025-07-03,9,25.6,0.1,0.0,13.6\n2025-07-03,10,26.4,0.1,0.0,14.0\n2025-07-03,11,28.0,0.1,0.0,14.7\n2025-07-03,12,28.7,0.1,0.0,14.0\n2025-07-03,13,28.9,0.0,0.0,14.1\n2025-07-03,14,29.5,0.0,0.0,14.9\n2025-07-03,15,29.7,0.0,0.0,14.7\n2025-07-03,16,29.2,0.0,0.0,14.3\n2025-07-03,17,28.4,0.0,0.0,15.0\n2025-07-03,18,27.5,0.0,0.0,13.6\n2025-07-03,19,26.5,0.0,0.0,9.9\n2025-07-03,20,25.7,0.0,0.0,10.7\n2025-07-03,21,25.5,0.0,0.0,10.8\n2025-07-03,22,25.3,0.0,0.0,6.7\n2025-07-03,23,25.0,0.0,0.0,9.2\n2025-07-04,0,24.7,0.0,0.0,10.6\n2025-07-04,1,24.4,0.0,0.0,11.0\n2025-07-04,2,24.3,0.0,0.0,13.3\n2025-07-04,3,24.2,0.0,0.0,13.9\n2025-07-04,4,24.3,0.0,0.0,13.6\n2025-07-04,5,24.3,0.0,0.0,13.9\n2025-07-04,6,24.3,0.0,0.0,13.2\n2025-07-04,7,24.6,0.1,0.0,14.7\n2025-07-04,8,25.0,0.0,0.0,17.4\n2025-07-04,9,25.6,0.0,0.0,19.1\n2025-07-04,10,27.1,0.0,0.0,16.9\n2025-07-04,11,27.4,0.0,0.0,15.8\n2025-07-04,12,27.5,0.0,0.0,14.6\n2025-07-04,13,28.7,0.0,0.0,18.6\n2025-07-04,14,28.8,0.0,0.0,16.9\n2025-07-04,15,29.3,0.0,0.0,15.1\n2025-07-04,16,28.7,0.1,0.0,15.8\n2025-07-04,17,28.2,0.0,0.0,15.0\n2025-07-04,18,27.7,0.1,0.0,14.6\n2025-07-04,19,26.8,0.0,0.0,12.7\n2025-07-04,20,26.0,0.0,0.0,11.6\n2025-07-04,21,25.4,0.0,0.0,11.6\n2025-07-04,22,25.3,0.0,0.0,10.7\n2025-07-04,23,25.0,0.0,0.0,11.2\n2025-07-05,0,24.8,0.0,0.0,11.3\n2025-07-05,1,24.7,0.0,0.0,11.8\n2025-07-05,2,24.6,0.0,0.0,11.3\n2025-07-05,3,24.5,0.0,0.0,10.9\n2025-07-05,4,24.5,0.0,0.0,11.6\n2025-07-05,5,24.5,0.0,0.0,11.7\n2025-07-05,6,24.6,0.0,0.0,12.2\n2025-07-05,7,24.9,0.1,0.0,14.2\n2025-07-05,8,25.5,0.1,0.0,15.6\n2025-07-05,9,26.8,0.2,0.0,17.6\n2025-07-05,10,27.8,0.2,0.0,17.6\n2025-07-05,11,28.5,0.1,0.0,16.2\n2025-07-05,12,29.1,0.0,0.0,14.8\n2025-07-05,13,29.7,0.0,0.0,13.3\n2025-07-05,14,29.8,0.0,0.0,13.6\n2025-07-05,15,29.5,0.0,0.0,14.8\n2025-07-05,16,29.2,0.0,0.0,14.0\n2025-07-05,17,28.4,0.0,0.0,13.1\n2025-07-05,18,27.4,0.0,0.0,11.8\n2025-07-05,19,26.5,0.0,0.0,9.7\n2025-07-05,20,25.9,0.0,0.0,8.7\n2025-07-05,21,25.6,0.0,0.0,9.7\n2025-07-05,22,25.8,0.0,0.0,9.1\n2025-07-05,23,25.3,0.0,0.0,9.9\n2025-07-06,0,25.1,0.0,0.0,9.7\n2025-07-06,1,25.0,0.0,0.0,10.9\n2025-07-06,2,24.9,0.0,0.0,10.0\n2025-07-06,3,24.8,0.1,0.0,11.3\n2025-07-06,4,24.8,0.0,0.0,11.6\n2025-07-06,5,24.8,0.0,0.0,11.0\n2025-07-06,6,24.9,0.0,0.0,11.2\n2025-07-06,7,25.2,0.1,0.0,12.9\n2025-07-06,8,25.8,0.1,0.0,13.8\n2025-07-06,9,26.9,0.1,0.0,13.8\n2025-07-06,10,27.6,0.1,0.0,11.6\n2025-07-06,11,29.0,0.0,0.0,11.8\n2025-07-06,12,29.4,0.0,0.0,9.6\n2025-07-06,13,30.7,0.0,0.0,9.8\n2025-07-06,14,31.0,0.0,0.0,9.3\n2025-07-06,15,30.6,0.0,0.0,12.3\n2025-07-06,16,29.5,0.0,0.0,12.9\n2025-07-06,17,29.0,0.0,0.0,10.9\n2025-07-06,18,28.1,0.0,0.0,9.6\n2025-07-06,19,27.1,0.0,0.0,7.3\n2025-07-06,20,26.1,0.0,0.0,5.3\n2025-07-06,21,25.7,0.0,0.0,5.5\n2025-07-06,22,25.1,0.0,0.0,3.5\n2025-07-06,23,24.9,0.0,0.0,3.0\n2025-07-07,0,24.8,0.0,0.0,4.3\n2025-07-07,1,24.7,0.0,0.0,6.5\n2025-07-07,2,24.6,0.0,0.0,7.3\n2025-07-07,3,24.5,0.0,0.0,6.0\n2025-07-07,4,24.5,0.0,0.0,7.0\n2025-07-07,5,24.5,0.0,0.0,8.2\n2025-07-07,6,24.5,0.0,0.0,7.9\n2025-07-07,7,24.8,0.0,0.0,8.1\n2025-07-07,8,25.5,0.0,0.0,8.4\n2025-07-07,9,26.4,0.1,0.0,7.4\n2025-07-07,10,27.6,0.1,0.0,6.2\n2025-07-07,11,29.3,0.1,0.0,6.8\n2025-07-07,12,30.0,0.0,0.0,6.7\n2025-07-07,13,31.6,0.2,0.0,7.0\n2025-07-07,14,31.7,0.2,0.0,10.7\n2025-07-07,15,30.3,0.0,0.0,14.3\n2025-07-07,16,29.2,0.0,0.0,12.8\n2025-07-07,17,28.9,0.0,0.0,9.8\n2025-07-07,18,28.2,0.0,0.0,8.5\n2025-07-07,19,27.5,0.0,0.0,6.6\n2025-07-07,20,26.7,0.0,0.0,4.5\n2025-07-07,21,26.2,0.0,0.0,4.5\n2025-07-07,22,26.0,0.0,0.0,4.9\n2025-07-07,23,26.3,0.0,0.0,5.4\n2025-07-08,0,26.0,0.0,0.0,3.1\n2025-07-08,1,25.6,0.0,0.0,3.1\n2025-07-08,2,25.4,0.0,0.0,3.4\n2025-07-08,3,25.1,0.0,0.0,3.2\n2025-07-08,4,24.8,0.0,0.0,3.0\n2025-07-08,5,24.7,0.0,0.0,2.1\n2025-07-08,6,24.8,0.0,0.0,4.2\n2025-07-08,7,25.8,0.0,0.0,3.4\n2025-07-08,8,28.0,0.0,0.0,4.1\n2025-07-08,9,30.0,0.0,0.0,5.9\n2025-07-08,10,31.5,0.0,0.0,6.4\n2025-07-08,11,32.6,0.0,0.0,6.4\n2025-07-08,12,33.5,0.0,0.0,5.1\n2025-07-08,13,34.4,0.0,0.0,4.6\n2025-07-08,14,34.9,0.0,0.0,4.5\n2025-07-08,15,35.6,0.0,0.0,2.3\n2025-07-08,16,35.6,0.0,0.0,1.4\n2025-07-08,17,35.2,0.0,0.0,1.2\n2025-07-08,18,34.6,0.0,0.0,1.5\n2025-07-08,19,31.7,0.0,0.0,5.3\n2025-07-08,20,30.6,0.0,0.0,1.3\n2025-07-08,21,29.4,0.0,0.0,2.3\n2025-07-08,22,26.9,0.1,0.0,6.0\n2025-07-08,23,26.8,0.0,0.0,3.1\n2025-07-09,0,27.0,0.0,0.0,0.2\n2025-07-09,1,26.5,0.0,0.0,1.8\n2025-07-09,2,26.4,0.0,0.0,5.3\n2025-07-09,3,26.5,0.0,0.0,8.5\n2025-07-09,4,26.4,0.0,0.0,8.1\n2025-07-09,5,26.1,0.0,0.0,7.6\n2025-07-09,6,26.2,0.0,0.0,7.4\n2025-07-09,7,27.2,0.0,0.0,7.8\n2025-07-09,8,28.5,0.0,0.0,8.7\n2025-07-09,9,30.0,0.0,0.0,8.0\n2025-07-09,10,32.0,0.0,0.0,8.8\n2025-07-09,11,32.8,0.2,0.0,8.8\n2025-07-09,12,33.7,0.1,0.0,9.5\n2025-07-09,13,34.7,0.0,0.0,8.8\n2025-07-09,14,35.4,0.0,0.0,7.9\n2025-07-09,15,35.6,0.0,0.0,6.5\n2025-07-09,16,35.6,0.0,0.0,4.9\n2025-07-09,17,35.6,0.0,0.0,4.5\n2025-07-09,18,34.7,0.0,0.0,5.4\n2025-07-09,19,32.9,0.0,0.0,3.2\n2025-07-09,20,31.1,0.0,0.0,3.2\n2025-07-09,21,31.0,0.0,0.0,7.5\n2025-07-09,22,29.6,0.0,0.0,4.8\n2025-07-09,23,29.6,0.0,0.0,5.8\n2025-07-10,0,29.4,0.0,0.0,11.9\n2025-07-10,1,28.5,0.0,0.0,13.4\n2025-07-10,2,27.3,0.0,0.0,9.4\n2025-07-10,3,26.4,0.0,0.0,9.3\n2025-07-10,4,25.7,0.0,0.0,8.2\n2025-07-10,5,25.0,0.0,0.0,4.8\n2025-07-10,6,25.1,0.0,0.0,3.6\n2025-07-10,7,26.4,0.0,0.0,3.8\n2025-07-10,8,28.1,0.0,0.0,5.4\n2025-07-10,9,30.0,0.0,0.0,6.7\n2025-07-10,10,31.9,0.0,0.0,9.4\n2025-07-10,11,33.2,0.0,0.0,9.6\n2025-07-10,12,34.4,0.0,0.0,9.9\n2025-07-10,13,35.4,0.0,0.0,11.7\n2025-07-10,14,36.1,0.0,0.0,13.1\n2025-07-10,15,36.7,0.0,0.0,13.8\n2025-07-10,16,36.9,0.0,0.0,14.0\n2025-07-10,17,36.4,0.0,0.0,13.0\n2025-07-10,18,35.3,0.0,0.0,10.5\n2025-07-10,19,34.1,0.0,0.0,11.7\n2025-07-10,20,31.7,0.0,0.0,17.9\n2025-07-10,21,29.7,0.0,0.0,17.0\n2025-07-10,22,29.7,0.0,0.0,17.1\n2025-07-10,23,28.6,0.0,0.0,15.7\n2025-07-11,0,27.6,0.0,0.0,14.6\n2025-07-11,1,26.6,0.0,0.0,13.1\n2025-07-11,2,25.5,0.0,0.0,12.5\n2025-07-11,3,24.7,0.0,0.0,12.3\n2025-07-11,4,24.0,0.0,0.0,11.5\n2025-07-11,5,23.5,0.0,0.0,10.2\n2025-07-11,6,23.3,0.0,0.0,10.2\n2025-07-11,7,24.1,0.0,0.0,9.5\n2025-07-11,8,25.8,0.0,0.0,10.6\n2025-07-11,9,28.0,0.0,0.0,12.4\n2025-07-11,10,30.3,0.0,0.0,12.6\n2025-07-11,11,32.1,0.0,0.0,12.0\n2025-07-11,12,33.8,0.0,0.0,11.8\n2025-07-11,13,35.1,0.0,0.0,11.8\n2025-07-11,14,36.0,0.0,0.0,11.9\n2025-07-11,15,36.4,0.0,0.0,13.2\n2025-07-11,16,36.5,0.0,0.0,14.4\n2025-07-11,17,36.2,0.0,0.0,15.1\n2025-07-11,18,35.4,0.0,0.0,16.9\n2025-07-11,19,33.7,0.0,0.0,18.7\n2025-07-11,20,31.6,0.0,0.0,18.3\n2025-07-11,21,30.0,0.0,0.0,19.1\n2025-07-11,22,30.6,0.0,0.0,14.0\n2025-07-11,23,29.1,0.0,0.0,13.1\n2025-07-12,0,27.9,0.0,0.0,12.0\n2025-07-12,1,26.9,0.0,0.0,11.9\n2025-07-12,2,26.0,0.0,0.0,9.5\n2025-07-12,3,25.3,0.0,0.0,7.5\n2025-07-12,4,24.7,0.0,0.0,5.9\n2025-07-12,5,24.1,0.0,0.0,4.1\n2025-07-12,6,24.0,0.0,0.0,4.8\n2025-07-12,7,24.9,0.0,0.0,4.7\n2025-07-12,8,26.3,0.0,0.0,4.9\n2025-07-12,9,28.3,0.0,0.0,5.6\n2025-07-12,10,31.0,0.0,0.0,9.2\n2025-07-12,11,32.9,0.0,0.0,9.4\n2025-07-12,12,34.6,0.0,0.0,8.2\n2025-07-12,13,36.0,0.0,0.0,7.2\n2025-07-12,14,36.9,0.0,0.0,3.1\n2025-07-12,15,36.9,0.0,0.0,4.7\n2025-07-12,16,36.4,0.0,0.0,8.1\n2025-07-12,17,35.5,0.0,0.0,11.7\n2025-07-12,18,34.4,0.0,0.0,13.1\n2025-07-12,19,33.4,0.0,0.0,13.2\n2025-07-12,20,32.1,0.0,0.0,12.3\n2025-07-12,21,31.4,0.0,0.0,9.8\n2025-07-12,22,28.6,0.0,0.0,9.2\n2025-07-12,23,26.7,0.0,0.0,8.0\n2025-07-13,0,26.2,0.0,0.0,5.1\n2025-07-13,1,26.3,0.0,0.0,0.8\n2025-07-13,2,26.1,0.0,0.0,0.4\n2025-07-13,3,25.9,0.0,0.0,2.3\n2025-07-13,4,25.1,0.0,0.0,5.8\n2025-07-13,5,24.5,0.0,0.0,5.7\n2025-07-13,6,24.1,0.0,0.0,5.2\n2025-07-13,7,25.1,0.0,0.0,3.1\n2025-07-13,8,27.1,0.0,0.0,5.0\n2025-07-13,9,28.4,0.0,0.0,4.9\n2025-07-13,10,29.9,0.0,0.0,4.4\n2025-07-13,11,29.7,0.0,0.0,4.1\n2025-07-13,12,29.3,0.0,0.0,9.8\n2025-07-13,13,26.6,0.8,0.0,10.5\n2025-07-13,14,26.1,0.6,0.0,10.1\n2025-07-13,15,25.2,0.4,0.0,7.6\n2025-07-13,16,26.4,0.0,0.0,4.1\n2025-07-13,17,25.9,0.0,0.0,3.6\n2025-07-13,18,26.0,0.0,0.0,2.9\n2025-07-13,19,25.9,0.0,0.0,3.3\n2025-07-13,20,25.0,0.0,0.0,3.2\n2025-07-13,21,24.6,0.0,0.0,3.1\n2025-07-13,22,24.1,0.0,0.0,6.1\n2025-07-13,23,24.0,0.0,0.0,4.5\n2025-07-14,0,23.5,0.0,0.0,7.2\n2025-07-14,1,23.2,0.0,0.0,4.7\n2025-07-14,2,22.9,0.0,0.0,5.1\n2025-07-14,3,22.5,0.0,0.0,3.3\n2025-07-14,4,22.3,0.0,0.0,5.6\n2025-07-14,5,21.8,0.0,0.0,4.8\n2025-07-14,6,21.8,0.0,0.0,4.2\n2025-07-14,7,22.8,0.0,0.0,3.9\n2025-07-14,8,24.6,0.0,0.0,7.4\n2025-07-14,9,26.2,0.0,0.0,7.3\n2025-07-14,10,27.7,0.1,0.0,6.3\n2025-07-14,11,28.8,0.1,0.0,3.9\n2025-07-14,12,23.6,8.6,0.0,9.0\n2025-07-14,13,24.6,2.8,0.0,7.3\n2025-07-14,14,25.0,0.2,0.0,9.9\n2025-07-14,15,23.4,1.1,0.0,14.4\n2025-07-14,16,23.6,3.3,0.0,11.0\n2025-07-14,17,23.5,1.4,0.0,9.4\n2025-07-14,18,23.6,0.8,0.0,8.9\n2025-07-14,19,22.9,0.6,0.0,11.6\n2025-07-14,20,22.3,3.8,0.0,9.6\n2025-07-14,21,22.5,1.9,0.0,12.5\n2025-07-14,22,21.6,1.2,0.0,11.8\n2025-07-14,23,21.7,4.2,0.0,10.4\n2025-07-15,0,21.7,3.5,0.0,12.0\n2025-07-15,1,21.5,1.6,0.0,12.2\n2025-07-15,2,21.3,1.3,0.0,13.9\n2025-07-15,3,21.4,0.1,0.0,12.9\n2025-07-15,4,21.1,0.0,0.0,12.5\n2025-07-15,5,21.3,0.0,0.0,11.9\n2025-07-15,6,21.4,0.2,0.0,9.6\n2025-07-15,7,21.8,0.2,0.0,5.8\n2025-07-15,8,22.7,0.6,0.0,5.8\n2025-07-15,9,22.4,2.2,0.0,6.1\n2025-07-15,10,23.4,0.0,0.0,10.3\n2025-07-15,11,25.1,0.0,0.0,8.8\n2025-07-15,12,24.7,0.3,0.0,9.0\n2025-07-15,13,24.4,0.0,0.0,9.5\n2025-07-15,14,22.9,0.9,0.0,10.9\n2025-07-15,15,22.8,0.9,0.0,8.7\n2025-07-15,16,23.1,1.0,0.0,8.8\n2025-07-15,17,23.1,0.6,0.0,7.3\n2025-07-15,18,22.8,0.3,0.0,5.8\n2025-07-15,19,22.4,0.0,0.0,3.3\n2025-07-15,20,21.6,0.0,0.0,5.6\n2025-07-15,21,21.1,0.0,0.0,6.9\n2025-07-15,22,22.8,0.0,0.0,8.7\n2025-07-15,23,22.9,0.0,0.0,9.7\n2025-07-16,0,22.8,0.0,0.0,10.3\n2025-07-16,1,22.5,0.0,0.0,9.8\n2025-07-16,2,22.2,0.0,0.0,9.6\n2025-07-16,3,22.6,0.0,0.0,8.3\n2025-07-16,4,22.4,1.3,0.0,7.8\n2025-07-16,5,22.3,0.2,0.0,7.1\n2025-07-16,6,21.7,0.0,0.0,14.8\n2025-07-16,7,22.4,0.0,0.0,11.4\n2025-07-16,8,23.3,0.1,0.0,11.6\n2025-07-16,9,22.2,1.3,0.0,9.4\n2025-07-16,10,23.0,0.3,0.0,11.7\n2025-07-16,11,24.2,0.1,0.0,12.0\n2025-07-16,12,24.2,0.5,0.0,12.2\n2025-07-16,13,25.0,0.6,0.0,19.9\n2025-07-16,14,21.4,2.4,0.0,17.5\n2025-07-16,15,20.9,4.8,0.0,17.0\n2025-07-16,16,21.4,4.3,0.0,22.2\n2025-07-16,17,22.2,4.5,0.0,23.3\n2025-07-16,18,23.2,4.2,0.0,25.9\n2025-07-16,19,23.1,2.2,0.0,29.3\n2025-07-16,20,22.9,5.2,0.0,28.1\n2025-07-16,21,23.0,16.0,0.0,26.6\n2025-07-16,22,22.2,3.0,0.0,10.9\n2025-07-16,23,22.1,6.9,0.0,8.9\n2025-07-17,0,22.2,2.3,0.0,13.8\n2025-07-17,1,22.8,0.3,0.0,17.6\n2025-07-17,2,22.5,0.0,0.0,18.7\n2025-07-17,3,22.7,0.3,0.0,19.3\n2025-07-17,4,23.4,0.0,0.0,17.2\n2025-07-17,5,23.5,0.0,0.0,12.8\n2025-07-17,6,23.7,0.0,0.0,17.9\n2025-07-17,7,23.7,0.0,0.0,18.1\n2025-07-17,8,24.2,0.0,0.0,20.2\n2025-07-17,9,23.9,0.0,0.0,16.9\n2025-07-17,10,22.3,4.5,0.0,17.6\n2025-07-17,11,23.1,0.4,0.0,17.7\n2025-07-17,12,24.2,0.2,0.0,17.8\n2025-07-17,13,24.2,0.2,0.0,19.4\n2025-07-17,14,25.8,0.0,0.0,25.6\n2025-07-17,15,26.2,0.2,0.0,25.4\n2025-07-17,16,25.8,0.2,0.0,25.9\n2025-07-17,17,25.0,0.8,0.0,15.9\n2025-07-17,18,25.0,0.4,0.0,19.4\n2025-07-17,19,23.9,1.6,0.0,14.5\n2025-07-17,20,23.6,1.2,0.0,17.3\n2025-07-17,21,23.6,0.1,0.0,17.8\n2025-07-17,22,22.1,1.7,0.0,6.2\n2025-07-17,23,22.3,4.4,0.0,5.8\n2025-07-18,0,22.9,0.6,0.0,12.6\n2025-07-18,1,23.1,0.6,0.0,15.3\n2025-07-18,2,22.9,0.4,0.0,12.1\n2025-07-18,3,22.8,0.6,0.0,12.9\n2025-07-18,4,22.5,4.2,0.0,10.3\n2025-07-18,5,22.3,7.5,0.0,20.0\n2025-07-18,6,22.5,8.3,0.0,17.3\n2025-07-18,7,22.7,1.0,0.0,6.0\n2025-07-18,8,22.9,3.9,0.0,8.9\n2025-07-18,9,23.1,3.2,0.0,4.8\n2025-07-18,10,24.0,5.2,0.0,10.1\n2025-07-18,11,24.5,0.8,0.0,10.2\n2025-07-18,12,24.4,2.2,0.0,5.0\n2025-07-18,13,25.2,2.8,0.0,10.1\n2025-07-18,14,26.2,0.8,0.0,12.1\n2025-07-18,15,26.7,0.5,0.0,9.1\n2025-07-18,16,26.7,0.9,0.0,6.4\n2025-07-18,17,26.4,0.8,0.0,4.7\n2025-07-18,18,25.3,1.7,0.0,7.2\n2025-07-18,19,24.8,1.7,0.0,10.2\n2025-07-18,20,24.5,0.9,0.0,8.6\n2025-07-18,21,24.0,0.9,0.0,6.3\n2025-07-18,22,23.0,0.5,0.0,11.3\n2025-07-18,23,22.9,0.0,0.0,9.7\n2025-07-19,0,21.9,3.2,0.0,9.1\n2025-07-19,1,22.1,0.2,0.0,9.0\n2025-07-19,2,21.9,0.1,0.0,6.1\n2025-07-19,3,22.0,0.3,0.0,9.4\n2025-07-19,4,21.9,6.9,0.0,10.6\n2025-07-19,5,22.0,2.1,0.0,10.6\n2025-07-19,6,22.2,0.7,0.0,13.1\n2025-07-19,7,22.3,1.4,0.0,12.6\n2025-07-19,8,22.7,1.6,0.0,16.0\n2025-07-19,9,22.6,0.8,0.0,13.7\n2025-07-19,10,24.1,1.1,0.0,16.5\n2025-07-19,11,25.2,0.1,0.0,13.0\n2025-07-19,12,25.0,0.0,0.0,13.0\n2025-07-19,13,25.4,0.5,0.0,14.9\n2025-07-19,14,26.5,0.5,0.0,19.1\n2025-07-19,15,28.3,0.6,0.0,20.1\n2025-07-19,16,27.4,0.6,0.0,19.8\n2025-07-19,17,26.9,0.0,0.0,16.1\n2025-07-19,18,25.9,0.2,0.0,15.2\n2025-07-19,19,24.8,0.6,0.0,15.7\n2025-07-19,20,24.0,0.3,0.0,16.4\n2025-07-19,21,23.5,0.5,0.0,16.1\n2025-07-19,22,23.0,10.5,0.0,6.1\n2025-07-19,23,24.0,0.3,0.0,12.3\n2025-07-20,0,23.6,0.8,0.0,14.4\n2025-07-20,1,23.7,0.6,0.0,14.0\n2025-07-20,2,24.1,0.2,0.0,17.6\n2025-07-20,3,24.0,0.8,0.0,17.1\n2025-07-20,4,23.6,1.1,0.0,15.8\n2025-07-20,5,23.3,16.5,0.0,11.1\n2025-07-20,6,23.3,0.5,0.0,12.7\n2025-07-20,7,24.4,0.1,0.0,13.0\n2025-07-20,8,25.6,0.0,0.0,14.1\n2025-07-20,9,26.8,0.1,0.0,16.4\n2025-07-20,10,27.9,0.5,0.0,20.0\n2025-07-20,11,28.6,0.1,0.0,17.8\n2025-07-20,12,29.1,0.0,0.0,15.9\n2025-07-20,13,30.0,0.0,0.0,19.5\n2025-07-20,14,30.0,0.0,0.0,19.9\n2025-07-20,15,29.9,0.1,0.0,17.3\n2025-07-20,16,29.6,0.0,0.0,15.2\n2025-07-20,17,29.0,0.0,0.0,12.0\n2025-07-20,18,28.1,0.0,0.0,9.7\n2025-07-20,19,26.9,0.0,0.0,6.9\n2025-07-20,20,25.8,0.0,0.0,6.0\n2025-07-20,21,25.4,0.0,0.0,3.3\n2025-07-20,22,25.8,0.0,0.0,2.6\n2025-07-20,23,25.5,0.0,0.0,3.0\n2025-07-21,0,25.2,0.0,0.0,2.5\n2025-07-21,1,24.8,0.0,0.0,2.1\n2025-07-21,2,24.9,0.0,0.0,5.6\n2025-07-21,3,24.8,0.0,0.0,7.3\n2025-07-21,4,24.8,0.0,0.0,10.0\n2025-07-21,5,24.6,0.0,0.0,8.9\n2025-07-21,6,24.5,0.0,0.0,8.2\n2025-07-21,7,25.3,0.0,0.0,7.7\n2025-07-21,8,26.2,0.1,0.0,11.5\n2025-07-21,9,27.1,0.1,0.0,13.7\n2025-07-21,10,27.3,0.1,0.0,12.7\n2025-07-21,11,28.4,0.2,0.0,13.1\n2025-07-21,12,28.9,0.0,0.0,14.4\n2025-07-21,13,29.3,0.1,0.0,16.1\n2025-07-21,14,29.5,0.3,0.0,16.6\n2025-07-21,15,29.5,0.1,0.0,16.3\n2025-07-21,16,29.4,0.0,0.0,15.6\n2025-07-21,17,28.8,0.0,0.0,14.5\n2025-07-21,18,28.2,0.0,0.0,12.7\n2025-07-21,19,27.2,0.0,0.0,10.0\n2025-07-21,20,26.2,0.0,0.0,9.3\n2025-07-21,21,25.7,0.0,0.0,8.7\n2025-07-21,22,25.6,0.0,0.0,9.9\n2025-07-21,23,25.3,0.0,0.0,8.5\n2025-07-22,0,25.0,0.0,0.0,7.3\n2025-07-22,1,25.0,0.1,0.0,8.9\n2025-07-22,2,24.7,0.6,0.0,8.0\n2025-07-22,3,24.7,0.2,0.0,7.9\n2025-07-22,4,24.0,0.8,0.0,7.0\n2025-07-22,5,24.2,0.1,0.0,4.6\n2025-07-22,6,24.5,0.1,0.0,6.2\n2025-07-22,7,25.4,0.0,0.0,6.9\n2025-07-22,8,25.9,0.3,0.0,9.3\n2025-07-22,9,27.6,0.1,0.0,9.5\n2025-07-22,10,28.3,0.0,0.0,8.4\n2025-07-22,11,29.1,0.0,0.0,10.3\n2025-07-22,12,29.5,0.1,0.0,11.1\n2025-07-22,13,29.7,0.1,0.0,10.5\n2025-07-22,14,29.9,0.0,0.0,10.2\n2025-07-22,15,29.5,0.1,0.0,10.2\n2025-07-22,16,29.1,0.0,0.0,9.4\n2025-07-22,17,29.0,0.0,0.0,9.0\n2025-07-22,18,28.2,0.0,0.0,7.7\n2025-07-22,19,27.2,0.0,0.0,6.1\n2025-07-22,20,26.1,0.0,0.0,4.3\n2025-07-22,21,25.6,0.0,0.0,3.1\n2025-07-22,22,26.0,0.0,0.0,2.6\n2025-07-22,23,25.6,0.0,0.0,2.7\n2025-07-23,0,25.0,0.0,0.0,3.1\n2025-07-23,1,24.6,0.0,0.0,1.6\n2025-07-23,2,24.5,0.0,0.0,2.7\n2025-07-23,3,24.3,0.0,0.0,2.1\n2025-07-23,4,24.1,0.0,0.0,1.9\n2025-07-23,5,24.2,0.0,0.0,4.3\n2025-07-23,6,24.1,0.0,0.0,4.9\n2025-07-23,7,24.3,0.0,0.0,3.8\n2025-07-23,8,25.4,0.0,0.0,4.2\n2025-07-23,9,26.6,0.0,0.0,3.4\n2025-07-23,10,28.1,0.0,0.0,0.9\n2025-07-23,11,29.4,0.0,0.0,0.8\n2025-07-23,12,30.0,0.0,0.0,2.2\n2025-07-23,13,30.6,0.0,0.0,4.5\n2025-07-23,14,30.6,0.0,0.0,5.4\n2025-07-23,15,30.8,0.0,0.0,7.2\n2025-07-23,16,30.5,0.0,0.0,8.3\n2025-07-23,17,29.9,0.0,0.0,8.9\n2025-07-23,18,29.0,0.0,0.0,8.5\n2025-07-23,19,27.8,0.0,0.0,6.5\n2025-07-23,20,26.6,0.0,0.0,5.2\n2025-07-23,21,25.9,0.0,0.0,5.4\n2025-07-23,22,26.0,0.0,0.0,5.7\n2025-07-23,23,25.5,0.0,0.0,5.2\n2025-07-24,0,25.0,0.0,0.0,5.0\n2025-07-24,1,24.7,0.0,0.0,4.3\n2025-07-24,2,24.6,0.0,0.0,4.5\n2025-07-24,3,24.4,0.0,0.0,3.8\n2025-07-24,4,24.2,0.0,0.0,4.3\n2025-07-24,5,23.7,0.0,0.0,2.8\n2025-07-24,6,23.9,0.0,0.0,4.5\n2025-07-24,7,24.6,0.0,0.0,3.8\n2025-07-24,8,26.2,0.0,0.0,2.2\n2025-07-24,9,27.3,0.0,0.0,3.1\n2025-07-24,10,29.0,0.0,0.0,1.9\n2025-07-24,11,30.3,0.0,0.0,1.1\n2025-07-24,12,30.6,0.0,0.0,2.9\n2025-07-24,13,30.7,0.0,0.0,4.6\n2025-07-24,14,31.3,0.0,0.0,5.1\n2025-07-24,15,31.6,0.0,0.0,5.8\n2025-07-24,16,31.5,0.0,0.0,6.6\n2025-07-24,17,30.9,0.0,0.0,8.4\n2025-07-24,18,29.8,0.0,0.0,9.2\n2025-07-24,19,28.4,0.0,0.0,7.3\n2025-07-24,20,27.0,0.0,0.0,5.0\n2025-07-24,21,26.2,0.0,0.0,4.4\n2025-07-24,22,26.4,0.0,0.0,5.8\n2025-07-24,23,25.9,0.0,0.0,4.9\n2025-07-25,0,25.4,0.0,0.0,5.8\n2025-07-25,1,25.0,0.0,0.0,6.7\n2025-07-25,2,24.7,0.0,0.0,7.8\n2025-07-25,3,24.5,0.0,0.0,6.8\n2025-07-25,4,24.4,0.0,0.0,5.3\n2025-07-25,5,24.4,0.0,0.0,4.9\n2025-07-25,6,24.5,0.0,0.0,4.4\n2025-07-25,7,25.7,0.0,0.0,5.0\n2025-07-25,8,27.4,0.0,0.0,3.6\n2025-07-25,9,29.4,0.0,0.0,3.3\n2025-07-25,10,31.2,0.0,0.0,1.7\n2025-07-25,11,32.3,0.0,0.0,1.5\n2025-07-25,12,33.0,0.0,0.0,3.3\n2025-07-25,13,33.6,0.0,0.0,4.4\n2025-07-25,14,33.9,0.0,0.0,4.2\n2025-07-25,15,34.2,0.0,0.0,4.4\n2025-07-25,16,34.3,0.0,0.0,5.1\n2025-07-25,17,33.9,0.0,0.0,6.0\n2025-07-25,18,32.3,0.0,0.0,8.3\n2025-07-25,19,30.5,0.0,0.0,6.8\n2025-07-25,20,29.3,0.0,0.0,6.4\n2025-07-25,21,28.8,0.0,0.0,8.0\n2025-07-25,22,28.3,0.0,0.0,5.9\n2025-07-25,23,27.8,0.0,0.0,6.0\n2025-07-26,0,27.5,0.0,0.0,6.2\n2025-07-26,1,27.2,0.0,0.0,5.2\n2025-07-26,2,26.7,0.0,0.0,3.4\n2025-07-26,3,26.4,0.0,0.0,3.0\n2025-07-26,4,26.1,0.0,0.0,3.0\n2025-07-26,5,25.9,0.0,0.0,4.2\n2025-07-26,6,26.2,0.0,0.0,6.0\n2025-07-26,7,26.8,0.0,0.0,6.5\n2025-07-26,8,28.1,0.0,0.0,5.9\n2025-07-26,9,29.6,0.0,0.0,7.0\n2025-07-26,10,31.3,0.0,0.0,4.8\n2025-07-26,11,32.6,0.0,0.0,3.7\n2025-07-26,12,33.7,0.0,0.0,2.9\n2025-07-26,13,34.5,0.0,0.0,1.7\n2025-07-26,14,35.2,0.0,0.0,1.5\n2025-07-26,15,35.2,0.0,0.0,6.1\n2025-07-26,16,34.5,0.0,0.0,7.8\n2025-07-26,17,33.0,0.0,0.0,11.2\n2025-07-26,18,31.7,0.0,0.0,9.6\n2025-07-26,19,30.6,0.0,0.0,6.5\n2025-07-26,20,29.6,0.0,0.0,5.1\n2025-07-26,21,29.0,0.0,0.0,6.2\n2025-07-26,22,29.3,0.0,0.0,7.6\n2025-07-26,23,28.9,0.0,0.0,7.4\n2025-07-27,0,28.3,0.0,0.0,6.0\n2025-07-27,1,27.7,0.0,0.0,3.7\n2025-07-27,2,27.5,0.0,0.0,3.4\n2025-07-27,3,27.9,0.0,0.0,7.7\n2025-07-27,4,27.4,0.0,0.0,9.4\n2025-07-27,5,26.7,0.0,0.0,10.0\n2025-07-27,6,26.4,0.0,0.0,9.7\n2025-07-27,7,27.2,0.0,0.0,8.9\n2025-07-27,8,28.8,0.0,0.0,8.2\n2025-07-27,9,30.5,0.0,0.0,8.7\n2025-07-27,10,32.6,0.0,0.0,5.9\n2025-07-27,11,33.7,0.0,0.0,4.5\n2025-07-27,12,34.8,0.0,0.0,3.7\n2025-07-27,13,35.5,0.0,0.0,3.8\n2025-07-27,14,35.7,0.0,0.0,7.1\n2025-07-27,15,35.3,0.0,0.0,9.5\n2025-07-27,16,34.7,0.0,0.0,9.7\n2025-07-27,17,33.9,0.0,0.0,10.5\n2025-07-27,18,32.9,0.0,0.0,9.9\n2025-07-27,19,31.8,0.0,0.0,8.2\n2025-07-27,20,30.4,0.0,0.0,7.8\n2025-07-27,21,29.4,0.0,0.0,7.0\n2025-07-27,22,29.6,0.0,0.0,7.1\n2025-07-27,23,29.1,0.0,0.0,7.7\n2025-07-28,0,28.4,0.0,0.0,6.1\n2025-07-28,1,27.9,0.0,0.0,3.6\n2025-07-28,2,27.4,0.0,0.0,1.4\n2025-07-28,3,27.0,0.0,0.0,0.9\n2025-07-28,4,26.6,0.0,0.0,0.9\n2025-07-28,5,26.2,0.0,0.0,6.7\n2025-07-28,6,26.0,0.0,0.0,3.9\n2025-07-28,7,26.9,0.0,0.0,5.2\n2025-07-28,8,28.8,0.0,0.0,3.9\n2025-07-28,9,30.7,0.0,0.0,5.7\n2025-07-28,10,32.4,0.0,0.0,7.1\n2025-07-28,11,33.5,0.0,0.0,8.0\n2025-07-28,12,34.4,0.0,0.0,8.1\n2025-07-28,13,35.2,0.0,0.0,8.2\n2025-07-28,14,35.7,0.0,0.0,8.5\n2025-07-28,15,35.6,0.0,0.0,9.1\n2025-07-28,16,34.9,0.0,0.0,9.9\n2025-07-28,17,34.1,0.0,0.0,11.4\n2025-07-28,18,32.8,0.0,0.0,11.5\n2025-07-28,19,31.4,0.0,0.0,10.1\n2025-07-28,20,30.0,0.0,0.0,7.3\n2025-07-28,21,29.3,0.0,0.0,5.9\n2025-07-28,22,29.0,0.0,0.0,6.0\n2025-07-28,23,28.6,0.0,0.0,5.2\n2025-07-29,0,28.4,0.0,0.0,3.0\n2025-07-29,1,27.9,0.0,0.0,3.6\n2025-07-29,2,27.5,0.0,0.0,2.1\n2025-07-29,3,27.1,0.0,0.0,2.5\n2025-07-29,4,26.8,0.0,0.0,2.6\n2025-07-29,5,26.5,0.0,0.0,0.6\n2025-07-29,6,26.1,0.0,0.0,1.3\n2025-07-29,7,27.1,0.0,0.0,2.5\n2025-07-29,8,29.0,0.0,0.0,3.8\n2025-07-29,9,31.1,0.0,0.0,2.7\n2025-07-29,10,33.0,0.0,0.0,2.7\n2025-07-29,11,34.7,0.0,0.0,5.3\n2025-07-29,12,35.7,0.0,0.0,7.4\n2025-07-29,13,36.3,0.0,0.0,8.2\n2025-07-29,14,36.5,0.0,0.0,8.6\n2025-07-29,15,36.2,0.0,0.0,10.1\n2025-07-29,16,35.6,0.0,0.0,11.7\n2025-07-29,17,34.7,0.0,0.0,11.0\n2025-07-29,18,33.7,0.0,0.0,8.8\n2025-07-29,19,32.5,0.0,0.0,7.0\n2025-07-29,20,31.4,0.0,0.0,7.0\n2025-07-29,21,30.5,0.0,0.0,6.7\n2025-07-29,22,29.5,0.0,0.0,7.3\n2025-07-29,23,29.1,0.0,0.0,4.8\n2025-07-30,0,28.8,0.0,0.0,4.1\n2025-07-30,1,28.4,0.0,0.0,2.7\n2025-07-30,2,27.9,0.0,0.0,1.1\n2025-07-30,3,27.5,0.0,0.0,1.1\n2025-07-30,4,27.0,0.0,0.0,1.0\n2025-07-30,5,26.7,0.0,0.0,9.8\n2025-07-30,6,26.0,0.0,0.0,6.8\n2025-07-30,7,26.6,0.0,0.0,4.7\n2025-07-30,8,28.0,0.0,0.0,3.0\n2025-07-30,9,29.7,0.0,0.0,1.8\n2025-07-30,10,32.0,0.0,0.0,4.4\n2025-07-30,11,34.1,0.0,0.0,5.1\n2025-07-30,12,35.8,0.0,0.0,6.8\n2025-07-30,13,35.7,0.0,0.0,8.5\n2025-07-30,14,35.0,0.2,0.0,9.9\n2025-07-30,15,34.3,0.3,0.0,9.5\n2025-07-30,16,34.6,0.0,0.0,8.2\n2025-07-30,17,34.4,0.0,0.0,6.8\n2025-07-30,18,33.3,0.0,0.0,8.7\n2025-07-30,19,32.2,0.0,0.0,12.2\n2025-07-30,20,31.6,0.0,0.0,11.1\n2025-07-30,21,31.5,0.0,0.0,8.7\n2025-07-30,22,30.6,0.0,0.0,5.9\n2025-07-30,23,30.2,0.0,0.0,4.4\n2025-07-31,0,29.8,0.0,0.0,2.7\n2025-07-31,1,29.9,0.0,0.0,2.8\n2025-07-31,2,30.1,0.0,0.0,9.6\n2025-07-31,3,28.9,0.0,0.0,14.5\n2025-07-31,4,28.3,0.0,0.0,13.7\n2025-07-31,5,27.3,0.0,0.0,10.5\n2025-07-31,6,26.9,0.0,0.0,7.9\n2025-07-31,7,27.5,0.0,0.0,6.4\n2025-07-31,8,28.5,0.0,0.0,6.3\n2025-07-31,9,29.7,0.0,0.0,6.2\n2025-07-31,10,31.4,0.0,0.0,3.8\n2025-07-31,11,32.3,0.0,0.0,3.1\n2025-07-31,12,32.9,0.0,0.0,2.4\n2025-07-31,13,32.8,0.0,0.0,4.0\n2025-07-31,14,34.4,0.0,0.0,5.9\n2025-07-31,15,34.1,0.0,0.0,9.1\n2025-07-31,16,34.0,0.0,0.0,10.5\n2025-07-31,17,33.9,0.0,0.0,11.4\n2025-07-31,18,32.8,0.0,0.0,12.6\n2025-07-31,19,31.4,0.0,0.0,10.8\n2025-07-31,20,30.4,0.0,0.0,11.3\n2025-07-31,21,29.4,0.0,0.0,11.3\n2025-07-31,22,28.9,0.0,0.0,6.8\n2025-07-31,23,28.5,0.0,0.0,6.3\n2025-08-01,0,28.1,0.0,0.0,8.3\n2025-08-01,1,27.8,0.0,0.0,7.2\n2025-08-01,2,27.9,0.0,0.0,6.1\n2025-08-01,3,28.0,0.0,0.0,5.9\n2025-08-01,4,27.7,0.0,0.0,7.8\n2025-08-01,5,27.8,0.0,0.0,7.2\n2025-08-01,6,27.5,0.0,0.0,4.3\n2025-08-01,7,27.9,0.0,0.0,3.9\n2025-08-01,8,29.2,0.0,0.0,4.6\n2025-08-01,9,30.6,0.0,0.0,4.4\n2025-08-01,10,32.5,0.0,0.0,4.1\n2025-08-01,11,34.3,0.0,0.0,4.5\n2025-08-01,12,34.3,0.0,0.0,7.3\n2025-08-01,13,34.4,0.1,0.0,8.3\n2025-08-01,14,34.3,0.0,0.0,7.2\n2025-08-01,15,32.3,0.0,0.0,10.7\n2025-08-01,16,33.7,0.0,0.0,9.5\n2025-08-01,17,33.7,0.0,0.0,7.9\n2025-08-01,18,32.6,0.0,0.0,9.3\n2025-08-01,19,30.9,0.0,0.0,8.8\n2025-08-01,20,29.5,0.0,0.0,7.9\n2025-08-01,21,28.6,0.0,0.0,6.3\n2025-08-01,22,29.1,0.0,0.0,3.8\n2025-08-01,23,28.6,0.0,0.0,4.0\n2025-08-02,0,28.2,0.0,0.0,4.7\n2025-08-02,1,27.4,0.0,0.0,6.0\n2025-08-02,2,26.8,0.0,0.0,4.9\n2025-08-02,3,26.6,0.0,0.0,3.8\n2025-08-02,4,26.4,0.0,0.0,2.2\n2025-08-02,5,26.3,0.0,0.0,1.0\n2025-08-02,6,26.1,0.0,0.0,3.6\n2025-08-02,7,26.9,0.0,0.0,3.3\n2025-08-02,8,28.4,0.0,0.0,2.6\n2025-08-02,9,30.0,0.0,0.0,3.5\n2025-08-02,10,31.5,0.0,0.0,5.5\n2025-08-02,11,32.6,0.0,0.0,6.0\n2025-08-02,12,33.3,0.0,0.0,8.8\n2025-08-02,13,34.0,0.0,0.0,10.5\n2025-08-02,14,34.4,0.0,0.0,11.8\n2025-08-02,15,34.5,0.0,0.0,11.0\n2025-08-02,16,34.2,0.0,0.0,12.0\n2025-08-02,17,33.1,0.0,0.0,11.1\n2025-08-02,18,31.9,0.0,0.0,10.5\n2025-08-02,19,30.5,0.0,0.0,8.1\n2025-08-02,20,29.4,0.0,0.0,5.4\n2025-08-02,21,28.7,0.0,0.0,3.6\n2025-08-02,22,28.6,0.0,0.0,4.0\n2025-08-02,23,28.2,0.0,0.0,2.6\n2025-08-03,0,27.7,0.0,0.0,2.3\n2025-08-03,1,27.4,0.0,0.0,1.0\n2025-08-03,2,26.9,0.0,0.0,2.2\n2025-08-03,3,27.2,0.0,0.0,0.5\n2025-08-03,4,26.8,0.0,0.0,0.3\n2025-08-03,5,26.6,0.0,0.0,0.4\n2025-08-03,6,26.5,0.0,0.0,2.4\n2025-08-03,7,27.2,0.0,0.0,2.0\n2025-08-03,8,29.1,0.0,0.0,3.1\n2025-08-03,9,30.5,0.0,0.0,5.2\n2025-08-03,10,30.9,0.0,0.0,4.3\n2025-08-03,11,33.1,0.0,0.0,6.4\n2025-08-03,12,32.8,0.0,0.0,8.1\n2025-08-03,13,33.3,0.0,0.0,9.0\n2025-08-03,14,32.9,0.0,0.0,10.3\n2025-08-03,15,32.8,0.0,0.0,11.1\n2025-08-03,16,32.7,0.0,0.0,15.2\n2025-08-03,17,31.5,0.0,0.0,15.3\n2025-08-03,18,31.0,0.0,0.0,15.8\n2025-08-03,19,29.4,0.0,0.0,16.0\n2025-08-03,20,28.0,0.0,0.0,11.5\n2025-08-03,21,26.0,2.4,0.0,5.3\n2025-08-03,22,25.9,0.0,0.0,15.7\n2025-08-03,23,25.3,0.4,0.0,15.0\n2025-08-04,0,23.6,3.3,0.0,12.3\n2025-08-04,1,23.6,0.6,0.0,14.4\n2025-08-04,2,23.4,0.0,0.0,12.6\n2025-08-04,3,23.2,0.0,0.0,8.2\n2025-08-04,4,23.1,0.0,0.0,9.7\n2025-08-04,5,23.2,0.0,0.0,8.4\n2025-08-04,6,23.2,0.0,0.0,8.4\n2025-08-04,7,24.2,0.0,0.0,4.8\n2025-08-04,8,25.8,0.0,0.0,1.8\n2025-08-04,9,27.4,0.0,0.0,3.1\n2025-08-04,10,27.3,0.0,0.0,4.2\n2025-08-04,11,29.1,0.0,0.0,8.0\n2025-08-04,12,30.1,0.0,0.0,8.2\n2025-08-04,13,31.2,0.0,0.0,8.5\n2025-08-04,14,31.8,0.0,0.0,8.3\n2025-08-04,15,31.3,0.0,0.0,10.1\n2025-08-04,16,30.0,0.0,0.0,13.5\n2025-08-04,17,28.8,0.0,0.0,14.2\n2025-08-04,18,28.4,0.0,0.0,10.7\n2025-08-04,19,27.6,0.0,0.0,10.3\n2025-08-04,20,26.8,0.0,0.0,9.3\n2025-08-04,21,26.7,0.0,0.0,8.7\n2025-08-04,22,26.2,0.0,0.0,5.5\n2025-08-04,23,26.0,0.0,0.0,6.2\n2025-08-05,0,25.6,0.0,0.0,6.9\n2025-08-05,1,25.1,0.0,0.0,7.8\n2025-08-05,2,24.6,0.0,0.0,7.2\n2025-08-05,3,24.3,0.0,0.0,6.5\n2025-08-05,4,24.1,0.0,0.0,7.2\n2025-08-05,5,23.9,0.0,0.0,8.7\n2025-08-05,6,23.8,0.0,0.0,9.0\n2025-08-05,7,24.6,0.0,0.0,9.1\n2025-08-05,8,25.6,0.0,0.0,8.5\n2025-08-05,9,26.7,0.0,0.0,8.1\n2025-08-05,10,28.1,0.0,0.0,7.3\n2025-08-05,11,28.8,0.0,0.0,8.1\n2025-08-05,12,29.2,0.0,0.0,8.1\n2025-08-05,13,29.6,0.0,0.0,9.6\n2025-08-05,14,29.7,0.0,0.0,10.6\n2025-08-05,15,29.6,0.0,0.0,10.5\n2025-08-05,16,29.2,0.0,0.0,9.7\n2025-08-05,17,28.7,0.0,0.0,8.2\n2025-08-05,18,27.8,0.0,0.0,8.3\n2025-08-05,19,26.9,0.0,0.0,5.3\n2025-08-05,20,26.2,0.0,0.0,3.6\n2025-08-05,21,25.9,0.0,0.0,5.0\n2025-08-05,22,26.8,0.0,0.0,1.5\n2025-08-05,23,26.5,0.0,0.0,6.1\n2025-08-06,0,26.1,0.0,0.0,6.4\n2025-08-06,1,25.9,0.0,0.0,7.2\n2025-08-06,2,25.7,0.0,0.0,12.3\n2025-08-06,3,24.6,0.4,0.0,15.4\n2025-08-06,4,25.8,0.5,0.0,16.3\n2025-08-06,5,26.5,0.0,0.0,19.7\n2025-08-06,6,26.3,0.4,0.0,22.3\n2025-08-06,7,26.4,0.8,0.0,23.2\n2025-08-06,8,26.6,0.3,0.0,21.1\n2025-08-06,9,27.0,0.2,0.0,17.2\n2025-08-06,10,28.2,0.2,0.0,18.4\n2025-08-06,11,28.7,0.5,0.0,20.9\n2025-08-06,12,27.7,0.2,0.0,16.5\n2025-08-06,13,28.1,1.0,0.0,17.9\n2025-08-06,14,26.6,1.8,0.0,14.1\n2025-08-06,15,25.5,3.5,0.0,13.4\n2025-08-06,16,25.3,1.3,0.0,12.9\n2025-08-06,17,25.4,0.3,0.0,12.3\n2025-08-06,18,26.0,0.0,0.0,13.2\n2025-08-06,19,25.6,0.0,0.0,12.1\n2025-08-06,20,25.3,0.0,0.0,11.4\n2025-08-06,21,25.6,0.0,0.0,11.4\n2025-08-06,22,25.3,0.0,0.0,10.2\n2025-08-06,23,25.4,0.0,0.0,11.8\n2025-08-07,0,25.4,0.0,0.0,12.7\n2025-08-07,1,25.2,0.0,0.0,11.6\n2025-08-07,2,24.9,0.0,0.0,10.3\n2025-08-07,3,24.6,0.0,0.0,10.2\n2025-08-07,4,24.3,0.0,0.0,9.7\n2025-08-07,5,24.4,0.0,0.0,9.0\n2025-08-07,6,24.3,0.0,0.0,8.5\n2025-08-07,7,24.8,0.0,0.0,7.3\n2025-08-07,8,25.5,0.0,0.0,8.4\n2025-08-07,9,26.5,0.0,0.0,9.7\n2025-08-07,10,28.2,0.0,0.0,12.6\n2025-08-07,11,28.9,0.0,0.0,11.7\n2025-08-07,12,29.3,0.0,0.0,11.7\n2025-08-07,13,29.3,0.0,0.0,12.3\n2025-08-07,14,29.2,0.0,0.0,12.7\n2025-08-07,15,29.1,0.0,0.0,13.0\n2025-08-07,16,28.9,0.0,0.0,13.3\n2025-08-07,17,28.5,0.0,0.0,12.6\n2025-08-07,18,27.7,0.0,0.0,11.0\n2025-08-07,19,26.9,0.0,0.0,7.9\n2025-08-07,20,26.2,0.0,0.0,5.8\n2025-08-07,21,25.8,0.0,0.0,8.4\n2025-08-07,22,25.4,0.0,0.0,3.3\n2025-08-07,23,24.9,0.0,0.0,5.8\n2025-08-08,0,24.2,0.0,0.0,6.3\n2025-08-08,1,23.6,0.0,0.0,6.0\n2025-08-08,2,23.1,0.0,0.0,5.3\n2025-08-08,3,22.7,0.0,0.0,4.3\n2025-08-08,4,22.2,0.0,0.0,3.3\n2025-08-08,5,22.2,0.0,0.0,1.6\n2025-08-08,6,22.2,0.0,0.0,0.6\n2025-08-08,7,23.2,0.0,0.0,1.6\n2025-08-08,8,24.3,0.0,0.0,1.0\n2025-08-08,9,25.8,0.0,0.0,1.3\n2025-08-08,10,27.2,0.0,0.0,3.4\n2025-08-08,11,28.0,0.0,0.0,4.8\n2025-08-08,12,28.7,0.0,0.0,6.2\n2025-08-08,13,28.6,0.1,0.0,8.2\n2025-08-08,14,28.6,0.0,0.0,8.6\n2025-08-08,15,28.7,0.0,0.0,7.8\n2025-08-08,16,28.6,0.0,0.0,9.6\n2025-08-08,17,27.8,0.0,0.0,9.6\n2025-08-08,18,26.4,0.0,0.0,8.7\n2025-08-08,19,25.5,0.0,0.0,6.2\n2025-08-08,20,24.5,0.0,0.0,3.5\n2025-08-08,21,24.5,0.0,0.0,4.6\n2025-08-08,22,24.7,0.0,0.0,2.9\n2025-08-08,23,24.3,0.0,0.0,2.3\n2025-08-09,0,24.4,0.0,0.0,3.5\n2025-08-09,1,24.0,0.0,0.0,2.2\n2025-08-09,2,23.9,0.0,0.0,4.1\n2025-08-09,3,23.6,0.3,0.0,5.3\n2025-08-09,4,23.5,1.4,0.0,4.3\n2025-08-09,5,22.9,1.4,0.0,5.8\n2025-08-09,6,22.7,0.6,0.0,7.0\n2025-08-09,7,23.0,0.1,0.0,9.9\n2025-08-09,8,24.1,0.1,0.0,11.3\n2025-08-09,9,24.9,0.0,0.0,7.4\n2025-08-09,10,25.1,0.1,0.0,3.0\n2025-08-09,11,24.7,0.8,0.0,5.3\n2025-08-09,12,24.8,0.5,0.0,1.5\n2025-08-09,13,25.5,0.0,0.0,3.3\n2025-08-09,14,26.8,0.0,0.0,2.7\n2025-08-09,15,26.5,0.0,0.0,3.6\n2025-08-09,16,25.7,0.3,0.0,3.9\n2025-08-09,17,25.3,0.5,0.0,4.8\n2025-08-09,18,25.2,0.2,0.0,5.8\n2025-08-09,19,24.6,0.0,0.0,1.9\n2025-08-09,20,23.5,0.0,0.0,2.5\n2025-08-09,21,23.2,0.0,0.0,3.3\n2025-08-09,22,23.5,0.0,0.0,1.9\n2025-08-09,23,23.3,0.0,0.0,5.7\n2025-08-10,0,22.8,0.0,0.0,8.2\n2025-08-10,1,22.5,0.0,0.0,8.5\n2025-08-10,2,22.1,0.0,0.0,7.7\n2025-08-10,3,21.9,0.0,0.0,8.6\n2025-08-10,4,21.6,0.0,0.0,8.1\n2025-08-10,5,21.3,0.0,0.0,6.7\n2025-08-10,6,20.8,0.0,0.0,3.3\n2025-08-10,7,22.0,0.0,0.0,6.0\n2025-08-10,8,23.2,0.0,0.0,6.8\n2025-08-10,9,24.7,0.0,0.0,6.9\n2025-08-10,10,26.1,0.0,0.0,6.1\n2025-08-10,11,27.4,0.0,0.0,7.8\n2025-08-10,12,28.3,0.0,0.0,8.2\n2025-08-10,13,29.1,0.0,0.0,6.8\n2025-08-10,14,30.1,0.0,0.0,7.1\n2025-08-10,15,30.4,0.0,0.0,7.6\n2025-08-10,16,29.6,0.0,0.0,9.8\n2025-08-10,17,28.9,0.0,0.0,11.3\n2025-08-10,18,27.7,0.0,0.0,11.4\n2025-08-10,19,26.4,0.0,0.0,8.9\n2025-08-10,20,25.4,0.0,0.0,6.4\n2025-08-10,21,25.0,0.0,0.0,4.8\n2025-08-10,22,24.9,0.0,0.0,5.0\n2025-08-10,23,24.7,0.0,0.0,4.7\n2025-08-11,0,24.1,0.0,0.0,1.5\n2025-08-11,1,23.7,0.0,0.0,1.0\n2025-08-11,2,23.5,0.0,0.0,7.2\n2025-08-11,3,23.3,0.0,0.0,8.4\n2025-08-11,4,22.9,0.0,0.0,8.9\n2025-08-11,5,22.7,0.0,0.0,7.8\n2025-08-11,6,22.8,0.0,0.0,6.0\n2025-08-11,7,23.2,0.0,0.0,5.7\n2025-08-11,8,23.9,0.0,0.0,5.6\n2025-08-11,9,24.3,0.0,0.0,5.2\n2025-08-11,10,26.4,0.0,0.0,5.3\n2025-08-11,11,27.1,0.0,0.0,7.3\n2025-08-11,12,27.1,0.0,0.0,7.8\n2025-08-11,13,26.6,0.0,0.0,6.0\n2025-08-11,14,27.4,0.0,0.0,6.2\n2025-08-11,15,26.9,0.0,0.0,8.6\n2025-08-11,16,27.0,0.0,0.0,8.7\n2025-08-11,17,26.8,0.0,0.0,8.0\n2025-08-11,18,26.4,0.0,0.0,8.3\n2025-08-11,19,25.5,0.1,0.0,5.8\n2025-08-11,20,24.6,0.0,0.0,3.8\n2025-08-11,21,24.2,0.0,0.0,4.1\n2025-08-11,22,23.7,0.0,0.0,2.6\n2025-08-11,23,23.4,0.0,0.0,4.2\n2025-08-12,0,22.7,0.0,0.0,2.3\n2025-08-12,1,22.3,0.0,0.0,2.4\n2025-08-12,2,21.8,0.0,0.0,1.5\n2025-08-12,3,21.3,0.0,0.0,2.0\n2025-08-12,4,21.4,0.0,0.0,4.2\n2025-08-12,5,21.1,0.0,0.0,3.9\n2025-08-12,6,21.0,0.0,0.0,4.0\n2025-08-12,7,22.3,0.0,0.0,5.6\n2025-08-12,8,23.8,0.0,0.0,4.1\n2025-08-12,9,25.3,0.0,0.0,2.5\n2025-08-12,10,27.6,0.0,0.0,6.5\n2025-08-12,11,28.8,0.0,0.0,6.4\n2025-08-12,12,29.9,0.0,0.0,6.6\n2025-08-12,13,30.9,0.0,0.0,6.1\n2025-08-12,14,31.9,0.0,0.0,8.8\n2025-08-12,15,31.5,0.0,0.0,9.3\n2025-08-12,16,30.8,0.0,0.0,7.1\n2025-08-12,17,30.1,0.0,0.0,5.4\n2025-08-12,18,29.4,0.0,0.0,6.5\n2025-08-12,19,27.9,0.0,0.0,4.4\n2025-08-12,20,26.9,0.0,0.0,8.6\n2025-08-12,21,25.8,0.0,0.0,10.8\n2025-08-12,22,25.5,0.0,0.0,7.8\n2025-08-12,23,25.1,0.0,0.0,7.9\n2025-08-13,0,24.9,0.0,0.0,7.7\n2025-08-13,1,24.5,0.1,0.0,9.0\n2025-08-13,2,24.2,0.7,0.0,9.4\n2025-08-13,3,23.5,1.0,0.0,10.7\n2025-08-13,4,23.7,0.0,0.0,12.7\n2025-08-13,5,23.7,0.5,0.0,11.8\n2025-08-13,6,23.1,4.3,0.0,9.1\n2025-08-13,7,23.2,2.0,0.0,11.2\n2025-08-13,8,23.6,2.0,0.0,9.0\n2025-08-13,9,23.8,1.9,0.0,12.3\n2025-08-13,10,23.9,4.4,0.0,16.3\n2025-08-13,11,24.3,13.3,0.0,19.5\n2025-08-13,12,25.3,7.4,0.0,24.4\n2025-08-13,13,26.5,1.0,0.0,22.0\n2025-08-13,14,26.4,1.8,0.0,17.6\n2025-08-13,15,27.3,0.5,0.0,16.9\n2025-08-13,16,27.8,0.1,0.0,17.2\n2025-08-13,17,28.3,0.1,0.0,18.6\n2025-08-13,18,27.6,0.1,0.0,20.2\n2025-08-13,19,26.8,0.3,0.0,22.1\n2025-08-13,20,26.3,0.6,0.0,19.0\n2025-08-13,21,25.6,0.5,0.0,15.0\n2025-08-13,22,24.5,1.5,0.0,15.3\n2025-08-13,23,24.5,1.1,0.0,12.0\n2025-08-14,0,24.6,0.1,0.0,15.9\n2025-08-14,1,24.4,0.8,0.0,19.8\n2025-08-14,2,23.9,3.6,0.0,12.8\n2025-08-14,3,24.4,1.2,0.0,13.5\n2025-08-14,4,24.3,3.7,0.0,15.8\n2025-08-14,5,23.8,5.2,0.0,11.5\n2025-08-14,6,23.7,3.6,0.0,9.3\n2025-08-14,7,23.8,2.8,0.0,6.9\n2025-08-14,8,23.8,4.4,0.0,8.1\n2025-08-14,9,23.8,4.1,0.0,8.0\n2025-08-14,10,26.8,0.2,0.0,11.6\n2025-08-14,11,26.4,0.2,0.0,12.2\n2025-08-14,12,26.0,0.3,0.0,11.4\n2025-08-14,13,26.6,0.1,0.0,13.9\n2025-08-14,14,26.9,0.1,0.0,12.6\n2025-08-14,15,27.3,0.0,0.0,11.1\n2025-08-14,16,26.7,0.0,0.0,9.2\n2025-08-14,17,26.1,0.0,0.0,8.7\n2025-08-14,18,26.0,0.0,0.0,7.0\n2025-08-14,19,25.4,0.0,0.0,6.3\n2025-08-14,20,24.6,0.0,0.0,4.9\n2025-08-14,21,24.1,0.0,0.0,3.9\n2025-08-14,22,24.5,0.0,0.0,2.3\n2025-08-14,23,24.3,0.0,0.0,0.7\n2025-08-15,0,24.1,0.0,0.0,0.2\n2025-08-15,1,23.9,0.0,0.0,1.5\n2025-08-15,2,23.7,0.0,0.0,3.3\n2025-08-15,3,23.7,0.0,0.0,3.9\n2025-08-15,4,23.6,0.2,0.0,4.0\n2025-08-15,5,23.5,0.1,0.0,3.5\n2025-08-15,6,23.6,0.0,0.0,4.1\n2025-08-15,7,23.9,0.0,0.0,4.4\n2025-08-15,8,25.1,0.1,0.0,3.8\n2025-08-15,9,26.5,0.1,0.0,4.5\n2025-08-15,10,27.4,0.6,0.0,7.5\n2025-08-15,11,28.3,0.2,0.0,6.2\n2025-08-15,12,29.4,0.1,0.0,6.9\n2025-08-15,13,29.9,0.0,0.0,9.4\n2025-08-15,14,29.8,0.1,0.0,11.8\n2025-08-15,15,29.6,0.0,0.0,11.7\n2025-08-15,16,29.4,0.1,0.0,11.9\n2025-08-15,17,28.7,0.0,0.0,10.4\n2025-08-15,18,27.9,0.0,0.0,9.3\n2025-08-15,19,27.0,0.0,0.0,6.9\n2025-08-15,20,26.1,0.0,0.0,5.3\n2025-08-15,21,25.8,0.0,0.0,4.6\n2025-08-15,22,25.7,0.0,0.0,3.8\n2025-08-15,23,25.3,0.0,0.0,2.2\n2025-08-16,0,24.9,0.0,0.0,2.3\n2025-08-16,1,24.6,0.0,0.0,2.5\n2025-08-16,2,24.5,0.0,0.0,2.5\n2025-08-16,3,24.9,0.0,0.0,3.8\n2025-08-16,4,25.1,0.0,0.0,4.5\n2025-08-16,5,25.0,0.0,0.0,3.4\n2025-08-16,6,24.8,0.0,0.0,5.1\n2025-08-16,7,25.3,0.1,0.0,7.8\n2025-08-16,8,26.0,0.1,0.0,7.9\n2025-08-16,9,27.5,0.0,0.0,8.5\n2025-08-16,10,28.7,0.0,0.0,10.2\n2025-08-16,11,29.0,0.1,0.0,11.1\n2025-08-16,12,29.4,0.2,0.0,12.2\n2025-08-16,13,29.7,0.1,0.0,12.6\n2025-08-16,14,29.5,0.1,0.0,15.1\n2025-08-16,15,29.1,0.1,0.0,14.8\n2025-08-16,16,28.7,0.1,0.0,14.1\n2025-08-16,17,28.3,0.0,0.0,14.2\n2025-08-16,18,27.6,0.0,0.0,13.8\n2025-08-16,19,26.7,0.0,0.0,10.7\n2025-08-16,20,26.0,0.0,0.0,8.7\n2025-08-16,21,25.6,0.0,0.0,8.0\n2025-08-16,22,25.7,0.0,0.0,7.3\n2025-08-16,23,25.4,0.0,0.0,6.9\n2025-08-17,0,25.1,0.0,0.0,7.6\n2025-08-17,1,24.8,0.0,0.0,7.2\n2025-08-17,2,24.5,0.0,0.0,6.0\n2025-08-17,3,24.4,0.0,0.0,6.2\n2025-08-17,4,24.3,0.0,0.0,7.9\n2025-08-17,5,24.0,0.0,0.0,7.1\n2025-08-17,6,23.8,0.0,0.0,7.1\n2025-08-17,7,24.6,0.0,0.0,6.7\n2025-08-17,8,25.8,0.1,0.0,7.5\n2025-08-17,9,26.9,0.4,0.0,11.9\n2025-08-17,10,28.6,0.2,0.0,12.6\n2025-08-17,11,29.1,0.0,0.0,11.8\n2025-08-17,12,29.6,0.0,0.0,11.2\n2025-08-17,13,29.7,0.0,0.0,11.0\n2025-08-17,14,29.2,0.1,0.0,11.3\n2025-08-17,15,28.8,0.0,0.0,10.7\n2025-08-17,16,28.5,0.0,0.0,9.7\n2025-08-17,17,28.5,0.0,0.0,9.2\n2025-08-17,18,28.3,0.0,0.0,7.8\n2025-08-17,19,27.3,0.0,0.0,4.4\n2025-08-17,20,26.8,0.0,0.0,6.2\n2025-08-17,21,26.4,0.0,0.0,8.3\n2025-08-17,22,26.6,0.0,0.0,4.8\n2025-08-17,23,26.3,0.0,0.0,4.6\n2025-08-18,0,26.1,0.0,0.0,4.9\n2025-08-18,1,25.7,0.0,0.0,4.9\n2025-08-18,2,25.4,0.0,0.0,5.4\n2025-08-18,3,25.2,0.0,0.0,6.3\n2025-08-18,4,25.0,0.0,0.0,7.0\n2025-08-18,5,24.8,0.0,0.0,7.2\n2025-08-18,6,24.9,0.0,0.0,8.6\n2025-08-18,7,25.4,0.0,0.0,8.8\n2025-08-18,8,25.9,0.3,0.0,10.9\n2025-08-18,9,27.3,0.1,0.0,11.7\n2025-08-18,10,29.2,0.2,0.0,15.5\n2025-08-18,11,29.3,0.7,0.0,16.6\n2025-08-18,12,29.2,0.0,0.0,15.6\n2025-08-18,13,30.2,0.0,0.0,17.0\n2025-08-18,14,30.2,0.0,0.0,18.7\n2025-08-18,15,30.0,0.0,0.0,18.6\n2025-08-18,16,29.6,0.0,0.0,18.6\n2025-08-18,17,28.9,0.0,0.0,17.2\n2025-08-18,18,28.1,0.0,0.0,16.2\n2025-08-18,19,27.4,0.0,0.0,12.9\n2025-08-18,20,26.8,0.0,0.0,13.1\n2025-08-18,21,26.6,0.0,0.0,13.5\n2025-08-18,22,27.0,0.0,0.0,12.5\n2025-08-18,23,26.8,0.1,0.0,14.3\n2025-08-19,0,26.6,0.0,0.0,12.9\n2025-08-19,1,26.4,0.0,0.0,11.8\n2025-08-19,2,26.4,0.1,0.0,11.5\n2025-08-19,3,26.3,0.0,0.0,11.1\n2025-08-19,4,26.0,0.0,0.0,10.9\n2025-08-19,5,26.0,0.1,0.0,10.7\n2025-08-19,6,25.8,0.1,0.0,10.6\n2025-08-19,7,26.3,0.0,0.0,11.2\n2025-08-19,8,27.3,0.1,0.0,14.4\n2025-08-19,9,28.4,0.0,0.0,15.6\n2025-08-19,10,28.4,0.3,0.0,16.1\n2025-08-19,11,28.1,0.8,0.0,15.0\n2025-08-19,12,29.6,0.2,0.0,17.6\n2025-08-19,13,29.8,0.1,0.0,17.3\n2025-08-19,14,29.7,0.1,0.0,16.0\n2025-08-19,15,29.7,0.1,0.0,16.2\n2025-08-19,16,29.5,0.0,0.0,15.6\n2025-08-19,17,29.1,0.0,0.0,15.1\n2025-08-19,18,28.4,0.0,0.0,13.6\n2025-08-19,19,27.6,0.0,0.0,9.7\n2025-08-19,20,26.9,0.0,0.0,9.1\n2025-08-19,21,26.7,0.0,0.0,8.9\n2025-08-19,22,26.7,0.0,0.0,8.3\n2025-08-19,23,26.0,0.5,0.0,9.5\n2025-08-20,0,25.6,0.6,0.0,8.5\n2025-08-20,1,25.7,0.5,0.0,10.0\n2025-08-20,2,25.6,0.1,0.0,11.0\n2025-08-20,3,25.4,0.3,0.0,10.8\n2025-08-20,4,25.6,0.0,0.0,10.5\n2025-08-20,5,25.5,0.0,0.0,10.5\n2025-08-20,6,25.2,0.1,0.0,9.1\n2025-08-20,7,25.1,0.5,0.0,8.6\n2025-08-20,8,26.2,0.2,0.0,8.7\n2025-08-20,9,27.0,0.1,0.0,8.1\n2025-08-20,10,29.4,0.2,0.0,11.3\n2025-08-20,11,29.6,0.3,0.0,13.9\n2025-08-20,12,29.8,0.4,0.0,13.1\n2025-08-20,13,30.1,0.4,0.0,14.8\n2025-08-20,14,30.3,0.1,0.0,14.2\n2025-08-20,15,30.4,0.0,0.0,13.6\n2025-08-20,16,29.6,0.0,0.0,14.0\n2025-08-20,17,29.3,0.0,0.0,11.8\n2025-08-20,18,28.8,0.0,0.0,11.0\n2025-08-20,19,27.8,0.0,0.0,8.8\n2025-08-20,20,27.1,0.0,0.0,6.5\n2025-08-20,21,26.9,0.0,0.0,4.5\n2025-08-20,22,27.2,0.1,0.0,5.2\n2025-08-20,23,27.0,0.0,0.0,4.8\n2025-08-21,0,26.7,0.0,0.0,4.0\n2025-08-21,1,26.5,0.0,0.0,6.9\n2025-08-21,2,26.2,0.0,0.0,7.4\n2025-08-21,3,26.0,0.0,0.0,8.2\n2025-08-21,4,25.7,0.0,0.0,7.8\n2025-08-21,5,25.5,0.1,0.0,8.0\n2025-08-21,6,25.2,0.0,0.0,7.4\n2025-08-21,7,25.8,0.0,0.0,9.2\n2025-08-21,8,26.6,0.1,0.0,9.5\n2025-08-21,9,28.0,0.1,0.0,8.5\n2025-08-21,10,29.6,0.0,0.0,7.0\n2025-08-21,11,30.4,0.0,0.0,8.7\n2025-08-21,12,30.6,0.0,0.0,9.8\n2025-08-21,13,30.8,0.0,0.0,10.4\n2025-08-21,14,31.0,0.0,0.0,11.0\n2025-08-21,15,30.8,0.0,0.0,9.9\n2025-08-21,16,30.5,0.0,0.0,9.9\n2025-08-21,17,30.1,0.0,0.0,10.0\n2025-08-21,18,29.2,0.0,0.0,8.3\n2025-08-21,19,28.1,0.0,0.0,4.9\n2025-08-21,20,27.1,0.0,0.0,2.4\n2025-08-21,21,26.8,0.0,0.0,0.7\n2025-08-21,22,26.9,0.0,0.0,2.0\n2025-08-21,23,26.5,0.0,0.0,1.8\n2025-08-22,0,26.2,0.0,0.0,0.3\n2025-08-22,1,25.8,0.0,0.0,0.6\n2025-08-22,2,25.5,0.0,0.0,1.0\n2025-08-22,3,25.2,0.0,0.0,1.5\n2025-08-22,4,25.0,0.0,0.0,1.4\n2025-08-22,5,24.7,0.0,0.0,2.0\n2025-08-22,6,24.9,0.0,0.0,4.4\n2025-08-22,7,25.3,0.0,0.0,6.5\n2025-08-22,8,26.4,0.0,0.0,6.3\n2025-08-22,9,28.1,0.0,0.0,5.4\n2025-08-22,10,29.8,0.0,0.0,7.2\n2025-08-22,11,30.4,0.0,0.0,10.3\n2025-08-22,12,30.6,0.0,0.0,12.6\n2025-08-22,13,30.6,0.0,0.0,12.4\n2025-08-22,14,30.7,0.0,0.0,12.8\n2025-08-22,15,30.4,0.0,0.0,12.9\n2025-08-22,16,30.0,0.0,0.0,12.9\n2025-08-22,17,29.4,0.0,0.0,13.4\n2025-08-22,18,28.7,0.0,0.0,12.4\n2025-08-22,19,27.8,0.0,0.0,8.6\n2025-08-22,20,27.0,0.0,0.0,5.1\n2025-08-22,21,26.8,0.0,0.0,6.0\n2025-08-22,22,27.1,0.0,0.0,5.8\n2025-08-22,23,27.1,0.0,0.0,6.7\n2025-08-23,0,26.8,0.0,0.0,5.4\n2025-08-23,1,26.4,0.0,0.0,4.8\n2025-08-23,2,25.9,0.0,0.0,5.8\n2025-08-23,3,25.5,0.0,0.0,6.4\n2025-08-23,4,25.2,0.0,0.0,7.4\n2025-08-23,5,25.0,0.0,0.0,7.3\n2025-08-23,6,24.8,0.0,0.0,7.7\n2025-08-23,7,25.5,0.0,0.0,7.8\n2025-08-23,8,27.1,0.0,0.0,7.0\n2025-08-23,9,28.8,0.0,0.0,8.5\n2025-08-23,10,30.0,0.1,0.0,8.9\n2025-08-23,11,30.7,0.1,0.0,11.4\n2025-08-23,12,30.6,0.1,0.0,12.1\n2025-08-23,13,31.5,0.0,0.0,12.9\n2025-08-23,14,31.9,0.0,0.0,13.0\n2025-08-23,15,31.9,0.0,0.0,12.6\n2025-08-23,16,31.5,0.0,0.0,11.0\n2025-08-23,17,31.0,0.0,0.0,9.7\n2025-08-23,18,30.0,0.0,0.0,8.5\n2025-08-23,19,28.9,0.0,0.0,7.0\n2025-08-23,20,28.2,0.0,0.0,4.1\n2025-08-23,21,27.8,0.0,0.0,2.5\n2025-08-23,22,27.8,0.0,0.0,4.0\n2025-08-23,23,27.5,0.0,0.0,4.1\n2025-08-24,0,27.1,0.0,0.0,4.6\n2025-08-24,1,26.8,0.0,0.0,3.7\n2025-08-24,2,26.4,0.0,0.0,2.1\n2025-08-24,3,26.2,0.0,0.0,0.0\n2025-08-24,4,25.8,0.0,0.0,0.5\n2025-08-24,5,25.5,0.0,0.0,3.3\n2025-08-24,6,25.3,0.0,0.0,6.3\n2025-08-24,7,25.9,0.0,0.0,6.4\n2025-08-24,8,27.7,0.0,0.0,5.0\n2025-08-24,9,29.6,0.1,0.0,7.0\n2025-08-24,10,30.7,0.1,0.0,3.5\n2025-08-24,11,32.3,0.0,0.0,7.9\n2025-08-24,12,33.0,0.0,0.0,10.2\n2025-08-24,13,33.3,0.0,0.0,10.4\n2025-08-24,14,33.7,0.0,0.0,10.4\n2025-08-24,15,33.6,0.0,0.0,11.0\n2025-08-24,16,32.7,0.0,0.0,11.8\n2025-08-24,17,32.1,0.0,0.0,10.2\n2025-08-24,18,31.3,0.0,0.0,7.4\n2025-08-24,19,30.2,0.0,0.0,6.3\n2025-08-24,20,29.3,0.0,0.0,5.3\n2025-08-24,21,28.7,0.0,0.0,3.8\n2025-08-24,22,28.9,0.0,0.0,0.2\n2025-08-24,23,28.4,0.0,0.0,3.8\n2025-08-25,0,27.9,0.0,0.0,9.3\n2025-08-25,1,27.0,0.0,0.0,9.5\n2025-08-25,2,26.7,0.0,0.0,8.3\n2025-08-25,3,26.5,0.1,0.0,8.0\n2025-08-25,4,26.1,0.0,0.0,8.3\n2025-08-25,5,25.7,0.0,0.0,8.8\n2025-08-25,6,25.8,0.0,0.0,10.0\n2025-08-25,7,26.6,0.0,0.0,9.8\n2025-08-25,8,27.6,0.0,0.0,13.4\n2025-08-25,9,28.4,0.3,0.0,11.0\n2025-08-25,10,28.8,0.7,0.0,12.8\n2025-08-25,11,28.1,0.4,0.0,11.4\n2025-08-25,12,30.3,0.0,0.0,12.6\n2025-08-25,13,31.6,0.2,0.0,15.3\n2025-08-25,14,32.3,0.0,0.0,16.5\n2025-08-25,15,32.2,0.0,0.0,17.0\n2025-08-25,16,31.6,0.1,0.0,17.3\n2025-08-25,17,31.0,0.0,0.0,18.4\n2025-08-25,18,29.7,0.4,0.0,16.1\n2025-08-25,19,27.7,1.2,0.0,14.8\n2025-08-25,20,26.8,0.4,0.0,15.7\n2025-08-25,21,26.7,0.1,0.0,14.1\n2025-08-25,22,26.0,1.4,0.0,16.4\n2025-08-25,23,26.0,0.7,0.0,16.2\n2025-08-26,0,25.2,1.9,0.0,17.9\n2025-08-26,1,25.5,0.1,0.0,15.1\n2025-08-26,2,25.2,0.2,0.0,14.0\n2025-08-26,3,24.5,3.7,0.0,9.0\n2025-08-26,4,24.1,2.1,0.0,10.9\n2025-08-26,5,24.1,1.8,0.0,8.0\n2025-08-26,6,24.3,1.3,0.0,7.9\n2025-08-26,7,24.1,0.5,0.0,9.9\n2025-08-26,8,24.4,0.4,0.0,9.0\n2025-08-26,9,24.6,3.2,0.0,11.5\n2025-08-26,10,25.0,0.6,0.0,12.0\n2025-08-26,11,27.9,0.0,0.0,9.4\n2025-08-26,12,28.8,0.0,0.0,10.8\n2025-08-26,13,28.8,0.1,0.0,14.1\n2025-08-26,14,28.9,0.0,0.0,15.7\n2025-08-26,15,28.7,0.0,0.0,15.9\n2025-08-26,16,28.4,0.0,0.0,15.0\n2025-08-26,17,27.9,0.0,0.0,14.0\n2025-08-26,18,27.4,0.0,0.0,12.4\n2025-08-26,19,26.7,0.0,0.0,9.6\n2025-08-26,20,26.1,0.0,0.0,7.9\n2025-08-26,21,25.9,0.0,0.0,9.4\n2025-08-26,22,25.9,0.0,0.0,7.8\n2025-08-26,23,25.2,0.0,0.0,10.2\n2025-08-27,0,24.4,0.0,0.0,7.6\n2025-08-27,1,23.7,0.0,0.0,6.3\n2025-08-27,2,22.9,0.0,0.0,0.8\n2025-08-27,3,22.7,0.0,0.0,2.8\n2025-08-27,4,22.5,0.0,0.0,3.1\n2025-08-27,5,22.7,0.0,0.0,6.2\n2025-08-27,6,22.2,0.0,0.0,5.3\n2025-08-27,7,22.2,0.0,0.0,3.0\n2025-08-27,8,23.4,0.0,0.0,4.0\n2025-08-27,9,25.2,0.0,0.0,2.8\n2025-08-27,10,27.3,0.0,0.0,1.8\n2025-08-27,11,28.2,0.0,0.0,2.6\n2025-08-27,12,28.9,0.0,0.0,2.7\n2025-08-27,13,29.1,0.0,0.0,2.6\n2025-08-27,14,29.7,0.0,0.0,3.1\n2025-08-27,15,29.4,0.0,0.0,7.3\n2025-08-27,16,28.0,0.0,0.0,11.1\n2025-08-27,17,27.2,0.0,0.0,9.6\n2025-08-27,18,26.8,0.0,0.0,5.7\n2025-08-27,19,26.1,0.0,0.0,5.2\n2025-08-27,20,25.2,0.0,0.0,2.4\n2025-08-27,21,24.8,0.0,0.0,3.9\n2025-08-27,22,25.0,0.0,0.0,2.1\n2025-08-27,23,24.6,0.0,0.0,2.2\n2025-08-28,0,24.3,0.0,0.0,2.5\n2025-08-28,1,24.0,0.0,0.0,2.0\n2025-08-28,2,23.6,0.0,0.0,6.5\n2025-08-28,3,23.6,0.0,0.0,5.9\n2025-08-28,4,23.7,0.0,0.0,9.0\n2025-08-28,5,23.6,0.0,0.0,9.6\n2025-08-28,6,23.6,0.0,0.0,10.2\n2025-08-28,7,23.7,0.0,0.0,10.6\n2025-08-28,8,24.3,0.0,0.0,10.3\n2025-08-28,9,24.6,0.0,0.0,10.5\n2025-08-28,10,28.0,0.0,0.0,9.2\n2025-08-28,11,28.6,0.0,0.0,12.8\n2025-08-28,12,29.3,0.0,0.0,13.9\n2025-08-28,13,29.7,0.0,0.0,13.4\n2025-08-28,14,29.7,0.0,0.0,13.0\n2025-08-28,15,29.7,0.2,0.0,15.6\n2025-08-28,16,28.6,0.1,0.0,14.4\n2025-08-28,17,28.6,0.1,0.0,16.2\n2025-08-28,18,28.0,0.0,0.0,12.2\n2025-08-28,19,27.4,0.0,0.0,11.4\n2025-08-28,20,27.1,0.1,0.0,11.5\n2025-08-28,21,26.8,0.3,0.0,13.6\n2025-08-28,22,26.3,0.2,0.0,12.7\n2025-08-28,23,26.5,0.3,0.0,14.6\n2025-08-29,0,26.3,0.0,0.0,13.1\n2025-08-29,1,26.1,0.0,0.0,13.3\n2025-08-29,2,26.2,0.1,0.0,14.2\n2025-08-29,3,26.2,0.4,0.0,15.4\n2025-08-29,4,26.4,0.2,0.0,16.3\n2025-08-29,5,26.4,0.0,0.0,14.8\n2025-08-29,6,26.4,0.1,0.0,14.0\n2025-08-29,7,26.8,0.1,0.0,14.5\n2025-08-29,8,27.3,0.1,0.0,17.3\n2025-08-29,9,28.6,0.2,0.0,20.6\n2025-08-29,10,29.3,0.2,0.0,20.2\n2025-08-29,11,29.1,0.3,0.0,21.5\n2025-08-29,12,29.6,0.0,0.0,21.3\n2025-08-29,13,30.0,0.1,0.0,20.7\n2025-08-29,14,29.9,0.2,0.0,21.8\n2025-08-29,15,30.0,0.4,0.0,21.3\n2025-08-29,16,29.0,0.5,0.0,18.2\n2025-08-29,17,28.9,0.6,0.0,18.0\n2025-08-29,18,28.2,0.2,0.0,15.0\n2025-08-29,19,27.4,0.1,0.0,10.5\n2025-08-29,20,26.7,0.0,0.0,8.8\n2025-08-29,21,26.4,0.0,0.0,8.4\n2025-08-29,22,24.9,3.6,0.0,5.3\n2025-08-29,23,25.0,5.6,0.0,4.9\n2025-08-30,0,24.8,0.5,0.0,5.0\n2025-08-30,1,24.6,0.7,0.0,4.2\n2025-08-30,2,24.6,0.2,0.0,0.7\n2025-08-30,3,24.3,0.0,0.0,3.5\n2025-08-30,4,23.8,0.1,0.0,2.7\n2025-08-30,5,23.6,1.9,0.0,6.5\n2025-08-30,6,23.8,0.4,0.0,6.4\n2025-08-30,7,23.7,1.1,0.0,4.2\n2025-08-30,8,25.0,0.0,0.0,3.5\n2025-08-30,9,26.4,0.2,0.0,3.3\n2025-08-30,10,28.3,0.1,0.0,6.0\n2025-08-30,11,29.6,0.0,0.0,8.1\n2025-08-30,12,30.1,0.0,0.0,10.3\n2025-08-30,13,30.4,0.0,0.0,12.0\n2025-08-30,14,30.3,0.0,0.0,11.1\n2025-08-30,15,30.3,0.0,0.0,9.3\n2025-08-30,16,30.0,0.0,0.0,9.3\n2025-08-30,17,29.3,0.0,0.0,8.5\n2025-08-30,18,28.5,0.0,0.0,6.7\n2025-08-30,19,27.2,0.0,0.0,3.7\n2025-08-30,20,26.4,0.0,0.0,2.9\n2025-08-30,21,26.1,0.0,0.0,2.6\n2025-08-30,22,26.4,0.0,0.0,2.3\n2025-08-30,23,26.2,0.0,0.0,3.2\n2025-08-31,0,25.5,0.0,0.0,2.4\n2025-08-31,1,25.2,0.0,0.0,3.4\n2025-08-31,2,25.1,0.0,0.0,5.5\n2025-08-31,3,25.0,0.0,0.0,6.7\n2025-08-31,4,24.9,0.0,0.0,7.2\n2025-08-31,5,24.9,0.0,0.0,8.4\n2025-08-31,6,24.7,0.0,0.0,8.5\n2025-08-31,7,25.0,0.0,0.0,7.7\n2025-08-31,8,25.9,1.3,0.0,5.7\n2025-08-31,9,26.4,0.3,0.0,4.1\n2025-08-31,10,28.7,0.0,0.0,8.7\n2025-08-31,11,27.9,0.2,0.0,6.4\n2025-08-31,12,28.3,0.2,0.0,12.1\n2025-08-31,13,27.2,0.9,0.0,9.2\n2025-08-31,14,29.2,0.0,0.0,10.6\n2025-08-31,15,29.5,0.0,0.0,11.3\n2025-08-31,16,29.5,2.0,0.0,13.6\n2025-08-31,17,28.9,0.1,0.0,12.2\n2025-08-31,18,28.1,0.0,0.0,13.9\n2025-08-31,19,27.3,0.0,0.0,11.8\n2025-08-31,20,26.7,0.2,0.0,12.6\n2025-08-31,21,26.4,0.0,0.0,11.4\n2025-08-31,22,26.0,1.0,0.0,11.3\n2025-08-31,23,25.6,0.1,0.0,10.1\n2025-09-01,0,25.0,0.8,0.0,7.6\n2025-09-01,1,24.4,1.2,0.0,12.6\n2025-09-01,2,24.0,2.8,0.0,15.7\n2025-09-01,3,23.9,5.4,0.0,15.4\n2025-09-01,4,23.6,1.4,0.0,10.3\n2025-09-01,5,23.2,2.5,0.0,14.3\n2025-09-01,6,23.6,0.2,0.0,14.9\n2025-09-01,7,24.1,0.1,0.0,12.9\n2025-09-01,8,24.9,0.1,0.0,16.2\n2025-09-01,9,25.4,0.2,0.0,16.3\n2025-09-01,10,26.4,0.8,0.0,15.6\n2025-09-01,11,27.0,0.4,0.0,14.9\n2025-09-01,12,26.4,0.7,0.0,16.8\n2025-09-01,13,27.6,0.1,0.0,16.9\n2025-09-01,14,26.9,4.1,0.0,17.0\n2025-09-01,15,25.6,8.2,0.0,14.8\n2025-09-01,16,26.3,0.8,0.0,17.7\n2025-09-01,17,26.1,5.2,0.0,12.4\n2025-09-01,18,26.1,0.1,0.0,11.1\n2025-09-01,19,25.7,0.0,0.0,9.8\n2025-09-01,20,25.1,0.0,0.0,8.3\n2025-09-01,21,24.8,0.0,0.0,8.2\n2025-09-01,22,24.4,0.0,0.0,2.9\n2025-09-01,23,24.3,0.0,0.0,4.8\n2025-09-02,0,23.7,0.0,0.0,8.7\n2025-09-02,1,22.5,0.0,0.0,5.0\n2025-09-02,2,21.6,0.0,0.0,7.4\n2025-09-02,3,20.9,0.0,0.0,8.5\n2025-09-02,4,20.3,0.0,0.0,5.9\n2025-09-02,5,19.7,0.0,0.0,4.6\n2025-09-02,6,19.6,0.0,0.0,4.5\n2025-09-02,7,20.7,0.0,0.0,6.8\n2025-09-02,8,21.6,0.0,0.0,7.1\n2025-09-02,9,23.5,0.0,0.0,8.4\n2025-09-02,10,25.4,0.0,0.0,3.2\n2025-09-02,11,26.7,0.0,0.0,4.3\n2025-09-02,12,27.7,0.0,0.0,6.0\n2025-09-02,13,28.5,0.0,0.0,6.7\n2025-09-02,14,28.8,0.0,0.0,5.8\n2025-09-02,15,29.0,0.0,0.0,5.7\n2025-09-02,16,28.6,0.0,0.0,7.2\n2025-09-02,17,27.8,0.0,0.0,8.1\n2025-09-02,18,26.7,0.0,0.0,6.8\n2025-09-02,19,25.3,0.0,0.0,6.6\n2025-09-02,20,24.5,0.0,0.0,6.5\n2025-09-02,21,23.8,0.0,0.0,4.1\n2025-09-02,22,23.8,0.0,0.0,3.7\n2025-09-02,23,23.7,0.0,0.0,4.9\n2025-09-03,0,23.0,0.0,0.0,5.3\n2025-09-03,1,22.5,0.0,0.0,3.5\n2025-09-03,2,22.2,0.0,0.0,3.5\n2025-09-03,3,21.7,0.0,0.0,1.7\n2025-09-03,4,21.4,0.0,0.0,1.1\n2025-09-03,5,20.9,0.0,0.0,3.1\n2025-09-03,6,21.4,0.0,0.0,4.7\n2025-09-03,7,22.0,0.0,0.0,6.8\n2025-09-03,8,23.3,0.0,0.0,5.0\n2025-09-03,9,25.3,0.0,0.0,5.2\n2025-09-03,10,27.0,0.0,0.0,6.0\n2025-09-03,11,28.2,0.0,0.0,7.1\n2025-09-03,12,29.2,0.0,0.0,8.0\n2025-09-03,13,30.1,0.0,0.0,6.8\n2025-09-03,14,30.4,0.0,0.0,6.6\n2025-09-03,15,30.5,0.0,0.0,6.1\n2025-09-03,16,30.4,0.0,0.0,4.7\n2025-09-03,17,30.4,0.0,0.0,2.7\n2025-09-03,18,29.8,0.0,0.0,1.3\n2025-09-03,19,27.7,0.0,0.0,2.6\n2025-09-03,20,26.5,0.0,0.0,3.6\n2025-09-03,21,26.3,0.0,0.0,3.7\n2025-09-03,22,26.4,0.0,0.0,0.7\n2025-09-03,23,25.9,0.0,0.0,7.0\n2025-09-04,0,25.7,0.0,0.0,5.0\n2025-09-04,1,25.2,0.1,0.0,7.2\n2025-09-04,2,24.6,2.9,0.0,7.2\n2025-09-04,3,24.1,0.3,0.0,5.0\n2025-09-04,4,23.5,1.7,0.0,8.6\n2025-09-04,5,23.3,1.8,0.0,10.5\n2025-09-04,6,23.0,3.0,0.0,10.6\n2025-09-04,7,22.7,2.9,0.0,8.9\n2025-09-04,8,22.7,3.1,0.0,9.4\n2025-09-04,9,22.8,3.0,0.0,11.3\n2025-09-04,10,23.4,1.7,0.0,11.7\n2025-09-04,11,23.8,1.7,0.0,10.0\n2025-09-04,12,24.6,1.4,0.0,15.3\n2025-09-04,13,24.8,0.1,0.0,10.9\n2025-09-04,14,27.3,0.0,0.0,9.7\n2025-09-04,15,28.4,0.0,0.0,9.1\n2025-09-04,16,29.0,0.0,0.0,8.9\n2025-09-04,17,28.9,0.0,0.0,8.6\n2025-09-04,18,28.1,0.0,0.0,4.8\n2025-09-04,19,27.0,0.0,0.0,6.8\n2025-09-04,20,26.2,0.0,0.0,7.2\n2025-09-04,21,25.8,0.0,0.0,6.5\n2025-09-04,22,25.6,0.0,0.0,9.1\n2025-09-04,23,25.3,0.0,0.0,10.2\n2025-09-05,0,24.8,0.0,0.0,9.2\n2025-09-05,1,24.4,0.0,0.0,8.1\n2025-09-05,2,24.0,0.0,0.0,7.0\n2025-09-05,3,23.5,0.0,0.0,5.8\n2025-09-05,4,23.0,0.0,0.0,6.5\n2025-09-05,5,22.6,0.0,0.0,7.8\n2025-09-05,6,22.4,0.0,0.0,7.2\n2025-09-05,7,22.8,0.0,0.0,7.2\n2025-09-05,8,24.1,0.0,0.0,5.2\n2025-09-05,9,25.2,0.0,0.0,5.2\n2025-09-05,10,27.8,0.0,0.0,7.2\n2025-09-05,11,29.0,0.0,0.0,6.8\n2025-09-05,12,30.0,0.0,0.0,7.6\n2025-09-05,13,30.7,0.1,0.0,8.9\n2025-09-05,14,31.0,0.1,0.0,9.2\n2025-09-05,15,30.7,0.1,0.0,10.3\n2025-09-05,16,30.4,0.0,0.0,9.1\n2025-09-05,17,29.6,0.0,0.0,7.8\n2025-09-05,18,28.1,0.0,0.0,6.5\n2025-09-05,19,27.1,0.0,0.0,6.6\n2025-09-05,20,26.5,0.0,0.0,4.0\n2025-09-05,21,26.1,0.0,0.0,4.1\n2025-09-05,22,26.2,0.0,0.0,2.3\n2025-09-05,23,25.9,0.0,0.0,5.3\n2025-09-06,0,25.6,0.0,0.0,7.7\n2025-09-06,1,25.4,0.2,0.0,9.2\n2025-09-06,2,24.9,0.4,0.0,14.6\n2025-09-06,3,24.5,0.1,0.0,11.3\n2025-09-06,4,24.3,0.1,0.0,11.1\n2025-09-06,5,24.2,0.0,0.0,12.2\n2025-09-06,6,24.1,0.5,0.0,13.8\n2025-09-06,7,24.1,0.9,0.0,11.4\n2025-09-06,8,25.3,0.1,0.0,13.9\n2025-09-06,9,26.1,0.2,0.0,16.5\n2025-09-06,10,25.9,1.6,0.0,17.5\n2025-09-06,11,28.1,0.3,0.0,16.7\n2025-09-06,12,28.1,0.3,0.0,19.8\n2025-09-06,13,29.6,0.6,0.0,23.8\n2025-09-06,14,29.6,0.4,0.0,22.2\n2025-09-06,15,29.9,0.0,0.0,23.0\n2025-09-06,16,29.1,0.1,0.0,23.5\n2025-09-06,17,27.6,0.6,0.0,18.3\n2025-09-06,18,26.8,4.8,0.0,19.7\n2025-09-06,19,25.3,8.7,0.0,20.2\n2025-09-06,20,24.9,17.9,0.0,18.8\n2025-09-06,21,25.2,3.5,0.0,14.4\n2025-09-06,22,24.8,1.2,0.0,18.3\n2025-09-06,23,24.5,0.3,0.0,13.9\n2025-09-07,0,24.9,0.3,0.0,12.7\n2025-09-07,1,25.1,0.2,0.0,12.6\n2025-09-07,2,24.7,0.0,0.0,11.3\n2025-09-07,3,24.0,0.0,0.0,8.6\n2025-09-07,4,23.5,0.0,0.0,6.6\n2025-09-07,5,23.3,0.0,0.0,6.2\n2025-09-07,6,23.1,0.0,0.0,5.2\n2025-09-07,7,23.1,0.0,0.0,6.4\n2025-09-07,8,23.6,0.0,0.0,8.6\n2025-09-07,9,24.5,0.0,0.0,9.2\n2025-09-07,10,25.7,0.0,0.0,7.3\n2025-09-07,11,26.1,0.0,0.0,8.2\n2025-09-07,12,27.1,0.0,0.0,8.8\n2025-09-07,13,27.4,0.0,0.0,9.4\n2025-09-07,14,26.6,0.0,0.0,10.8\n2025-09-07,15,25.8,0.0,0.0,11.3\n2025-09-07,16,25.3,0.0,0.0,10.3\n2025-09-07,17,24.8,0.0,0.0,6.9\n2025-09-07,18,24.7,0.0,0.0,5.4\n2025-09-07,19,23.9,0.0,0.0,3.5\n2025-09-07,20,23.4,0.0,0.0,4.3\n2025-09-07,21,22.9,0.0,0.0,5.1\n2025-09-07,22,22.7,0.0,0.0,4.6\n2025-09-07,23,22.3,0.0,0.0,4.7\n2025-09-08,0,22.2,0.0,0.0,4.5\n2025-09-08,1,22.2,0.0,0.0,3.9\n2025-09-08,2,22.2,0.0,0.0,5.0\n2025-09-08,3,21.5,0.0,0.0,6.0\n2025-09-08,4,20.8,0.0,0.0,5.1\n2025-09-08,5,20.6,0.0,0.0,4.5\n2025-09-08,6,20.2,0.0,0.0,3.7\n2025-09-08,7,21.2,0.0,0.0,5.1\n2025-09-08,8,21.9,0.0,0.0,5.3\n2025-09-08,9,23.6,0.0,0.0,4.1\n2025-09-08,10,24.8,0.0,0.0,2.7\n2025-09-08,11,25.5,0.0,0.0,0.6\n2025-09-08,12,25.9,0.0,0.0,2.4\n2025-09-08,13,25.8,0.0,0.0,5.3\n2025-09-08,14,26.1,0.0,0.0,5.0\n2025-09-08,15,26.9,0.0,0.0,4.6\n2025-09-08,16,26.8,0.0,0.0,3.6\n2025-09-08,17,25.7,0.0,0.0,4.3\n2025-09-08,18,24.9,0.0,0.0,3.9\n2025-09-08,19,23.9,0.0,0.0,5.2\n2025-09-08,20,23.1,0.0,0.0,9.6\n2025-09-08,21,22.1,0.0,0.0,6.1\n2025-09-08,22,22.8,0.0,0.0,5.4\n2025-09-08,23,22.5,0.4,0.0,8.6\n2025-09-09,0,22.0,0.5,0.0,7.8\n2025-09-09,1,21.9,0.0,0.0,4.4\n2025-09-09,2,21.6,0.0,0.0,3.6\n2025-09-09,3,21.5,0.0,0.0,4.6\n2025-09-09,4,21.3,0.0,0.0,6.0\n2025-09-09,5,21.0,0.0,0.0,4.3\n2025-09-09,6,20.9,0.0,0.0,3.1\n2025-09-09,7,21.1,0.0,0.0,1.6\n2025-09-09,8,21.5,0.0,0.0,4.1\n2025-09-09,9,21.6,0.0,0.0,7.4\n2025-09-09,10,23.8,0.0,0.0,3.8\n2025-09-09,11,25.2,0.0,0.0,6.5\n2025-09-09,12,25.7,0.0,0.0,5.8\n2025-09-09,13,25.7,0.0,0.0,5.2\n2025-09-09,14,26.2,0.0,0.0,4.2\n2025-09-09,15,26.4,0.0,0.0,3.3\n2025-09-09,16,26.3,0.0,0.0,4.0\n2025-09-09,17,26.3,0.0,0.0,3.3\n2025-09-09,18,25.2,0.0,0.0,2.6\n2025-09-09,19,23.9,0.0,0.0,1.6\n2025-09-09,20,22.8,0.0,0.0,3.1\n2025-09-09,21,21.7,0.0,0.0,3.8\n2025-09-09,22,21.7,0.0,0.0,0.5\n2025-09-09,23,21.0,0.0,0.0,2.7\n2025-09-10,0,20.6,0.0,0.0,1.6\n2025-09-10,1,20.3,0.0,0.0,2.5\n2025-09-10,2,20.5,0.0,0.0,4.9\n2025-09-10,3,20.4,0.0,0.0,5.1\n2025-09-10,4,20.0,0.0,0.0,4.5\n2025-09-10,5,19.5,0.0,0.0,5.5\n2025-09-10,6,19.0,0.0,0.0,5.8\n2025-09-10,7,19.0,0.0,0.0,6.9\n2025-09-10,8,19.9,0.0,0.0,4.7\n2025-09-10,9,22.1,0.0,0.0,3.8\n2025-09-10,10,24.3,0.0,0.0,6.9\n2025-09-10,11,25.8,0.0,0.0,7.3\n2025-09-10,12,27.1,0.0,0.0,6.9\n2025-09-10,13,28.0,0.0,0.0,6.9\n2025-09-10,14,28.6,0.0,0.0,6.9\n2025-09-10,15,29.0,0.0,0.0,7.0\n2025-09-10,16,29.0,0.0,0.0,6.5\n2025-09-10,17,28.4,0.0,0.0,5.7\n2025-09-10,18,26.6,0.0,0.0,8.1\n2025-09-10,19,24.8,0.0,0.0,8.2\n2025-09-10,20,23.6,0.0,0.0,5.9\n2025-09-10,21,22.6,0.0,0.0,4.6\n2025-09-10,22,22.5,0.0,0.0,4.6\n2025-09-10,23,22.0,0.0,0.0,6.5\n2025-09-11,0,21.4,0.0,0.0,7.3\n2025-09-11,1,21.1,0.0,0.0,6.2\n2025-09-11,2,19.9,0.0,0.0,4.0\n2025-09-11,3,19.4,0.0,0.0,0.9\n2025-09-11,4,19.2,0.0,0.0,4.1\n2025-09-11,5,18.6,0.0,0.0,3.1\n2025-09-11,6,18.7,0.0,0.0,4.2\n2025-09-11,7,19.5,0.0,0.0,5.5\n2025-09-11,8,21.5,0.0,0.0,4.0\n2025-09-11,9,24.0,0.0,0.0,2.9\n2025-09-11,10,26.0,0.0,0.0,7.4\n2025-09-11,11,27.3,0.0,0.0,7.0\n2025-09-11,12,28.4,0.0,0.0,7.5\n2025-09-11,13,29.2,0.0,0.0,8.4\n2025-09-11,14,29.6,0.0,0.0,10.1\n2025-09-11,15,29.8,0.0,0.0,10.1\n2025-09-11,16,29.8,0.0,0.0,8.0\n2025-09-11,17,29.1,0.0,0.0,5.3\n2025-09-11,18,27.6,0.0,0.0,3.2\n2025-09-11,19,25.5,0.0,0.0,5.9\n2025-09-11,20,24.9,0.0,0.0,9.9\n2025-09-11,21,24.3,0.0,0.0,10.3\n2025-09-11,22,24.5,0.0,0.0,10.0\n2025-09-11,23,24.2,0.0,0.0,9.6\n2025-09-12,0,23.6,0.0,0.0,10.7\n2025-09-12,1,22.8,0.0,0.0,9.2\n2025-09-12,2,22.0,0.0,0.0,10.1\n2025-09-12,3,21.6,0.0,0.0,10.0\n2025-09-12,4,21.3,0.0,0.0,8.8\n2025-09-12,5,21.2,0.0,0.0,8.7\n2025-09-12,6,20.9,0.0,0.0,8.0\n2025-09-12,7,21.2,0.0,0.0,9.0\n2025-09-12,8,22.0,0.0,0.0,9.6\n2025-09-12,9,23.0,0.0,0.0,10.1\n2025-09-12,10,23.3,0.0,0.0,8.5\n2025-09-12,11,25.2,0.0,0.0,10.8\n2025-09-12,12,25.2,0.0,0.0,9.0\n2025-09-12,13,24.9,0.0,0.0,10.3\n2025-09-12,14,25.5,0.0,0.0,11.5\n2025-09-12,15,25.6,0.0,0.0,11.4\n2025-09-12,16,24.6,0.1,0.0,11.3\n2025-09-12,17,24.0,0.1,0.0,11.2\n2025-09-12,18,23.5,0.1,0.0,11.5\n2025-09-12,19,22.8,0.6,0.0,11.6\n2025-09-12,20,22.2,0.6,0.0,13.0\n2025-09-12,21,21.4,0.8,0.0,12.3\n2025-09-12,22,21.2,2.9,0.0,15.5\n2025-09-12,23,20.0,7.4,0.0,13.1\n2025-09-13,0,20.0,7.4,0.0,11.2\n2025-09-13,1,20.1,11.5,0.0,10.4\n2025-09-13,2,20.2,8.7,0.0,14.3\n2025-09-13,3,20.1,13.3,0.0,16.6\n2025-09-13,4,20.3,7.2,0.0,17.3\n2025-09-13,5,20.4,1.6,0.0,16.7\n2025-09-13,6,20.8,0.5,0.0,13.5\n2025-09-13,7,20.9,0.1,0.0,14.1\n2025-09-13,8,21.7,0.0,0.0,8.7\n2025-09-13,9,23.1,0.0,0.0,7.6\n2025-09-13,10,23.5,0.0,0.0,7.9\n2025-09-13,11,24.1,0.0,0.0,8.9\n2025-09-13,12,24.8,0.0,0.0,10.6\n2025-09-13,13,25.7,0.0,0.0,10.3\n2025-09-13,14,26.4,0.0,0.0,9.8\n2025-09-13,15,26.8,0.0,0.0,8.2\n2025-09-13,16,26.5,0.0,0.0,6.9\n2025-09-13,17,26.1,0.0,0.0,6.2\n2025-09-13,18,25.4,0.0,0.0,6.7\n2025-09-13,19,23.9,0.0,0.0,6.7\n2025-09-13,20,23.5,0.0,0.0,7.8\n2025-09-13,21,22.9,0.0,0.0,9.1\n2025-09-13,22,22.6,0.0,0.0,7.7\n2025-09-13,23,22.1,0.0,0.0,6.4\n2025-09-14,0,21.8,0.0,0.0,6.0\n2025-09-14,1,21.5,0.0,0.0,6.3\n2025-09-14,2,21.2,0.0,0.0,5.5\n2025-09-14,3,21.1,0.0,0.0,5.2\n2025-09-14,4,20.8,0.0,0.0,4.0\n2025-09-14,5,20.9,0.0,0.0,4.0\n2025-09-14,6,20.3,0.0,0.0,4.7\n2025-09-14,7,20.6,0.0,0.0,4.4\n2025-09-14,8,22.2,0.0,0.0,0.9\n2025-09-14,9,23.5,0.0,0.0,0.6\n2025-09-14,10,24.6,0.0,0.0,1.4\n2025-09-14,11,25.8,0.0,0.0,2.9\n2025-09-14,12,26.7,0.0,0.0,4.5\n2025-09-14,13,27.1,0.0,0.0,4.9\n2025-09-14,14,27.6,0.0,0.0,5.5\n2025-09-14,15,27.5,0.0,0.0,6.7\n2025-09-14,16,26.8,0.0,0.0,9.4\n2025-09-14,17,25.9,0.0,0.0,9.6\n2025-09-14,18,24.8,0.0,0.0,7.3\n2025-09-14,19,23.5,0.0,0.0,4.9\n2025-09-14,20,23.0,0.0,0.0,4.6\n2025-09-14,21,22.6,0.0,0.0,4.3\n2025-09-14,22,22.9,0.0,0.0,3.6\n2025-09-14,23,22.7,0.0,0.0,2.4\n2025-09-15,0,22.5,0.0,0.0,2.2\n2025-09-15,1,22.1,0.0,0.0,1.8\n2025-09-15,2,21.5,0.0,0.0,3.7\n2025-09-15,3,21.0,0.0,0.0,3.6\n2025-09-15,4,20.8,0.0,0.0,3.0\n2025-09-15,5,20.3,0.0,0.0,3.1\n2025-09-15,6,20.2,0.0,0.0,3.1\n2025-09-15,7,20.7,0.0,0.0,4.9\n2025-09-15,8,22.5,0.0,0.0,4.2\n2025-09-15,9,24.4,0.0,0.0,3.9\n2025-09-15,10,25.7,0.0,0.0,1.3\n2025-09-15,11,26.7,0.0,0.0,2.2\n2025-09-15,12,27.5,0.0,0.0,2.4\n2025-09-15,13,28.0,0.0,0.0,1.6\n2025-09-15,14,28.3,0.0,0.0,1.0\n2025-09-15,15,28.5,0.0,0.0,1.8\n2025-09-15,16,28.7,0.0,0.0,1.2\n2025-09-15,17,28.3,0.0,0.0,3.9\n2025-09-15,18,26.6,0.0,0.0,6.4\n2025-09-15,19,25.0,0.0,0.0,5.8\n2025-09-15,20,24.2,0.0,0.0,4.0\n2025-09-15,21,23.7,0.0,0.0,1.3\n2025-09-15,22,23.6,0.0,0.0,0.8\n2025-09-15,23,23.2,0.0,0.0,3.0\n2025-09-16,0,22.8,0.0,0.0,4.3\n2025-09-16,1,22.4,0.0,0.0,4.7\n2025-09-16,2,22.1,0.0,0.0,8.2\n2025-09-16,3,21.6,0.0,0.0,6.2\n2025-09-16,4,21.0,0.0,0.0,3.5\n2025-09-16,5,21.2,0.0,0.0,6.5\n2025-09-16,6,21.3,0.0,0.0,5.4\n2025-09-16,7,21.6,0.0,0.0,6.8\n2025-09-16,8,23.0,0.0,0.0,6.8\n2025-09-16,9,24.0,0.3,0.0,8.5\n2025-09-16,10,25.8,0.0,0.0,8.0\n2025-09-16,11,25.5,0.3,0.0,7.3\n2025-09-16,12,26.5,0.1,0.0,2.8\n2025-09-16,13,25.9,0.3,0.0,14.2\n2025-09-16,14,26.2,0.1,0.0,11.4\n2025-09-16,15,26.7,0.1,0.0,14.8\n2025-09-16,16,26.0,0.3,0.0,15.4\n2025-09-16,17,26.1,0.4,0.0,14.7\n2025-09-16,18,25.2,0.2,0.0,11.7\n2025-09-16,19,24.8,0.1,0.0,11.8\n2025-09-16,20,23.4,6.6,0.0,13.6\n2025-09-16,21,23.4,1.8,0.0,12.9\n2025-09-16,22,23.1,0.1,0.0,5.0\n2025-09-16,23,23.1,0.2,0.0,6.4\n2025-09-17,0,22.8,0.0,0.0,3.5\n2025-09-17,1,22.5,0.2,0.0,3.1\n2025-09-17,2,22.9,1.0,0.0,2.6\n2025-09-17,3,22.5,0.2,0.0,4.0\n2025-09-17,4,22.7,0.2,0.0,8.3\n2025-09-17,5,22.6,0.1,0.0,10.4\n2025-09-17,6,22.4,0.1,0.0,11.6\n2025-09-17,7,22.0,0.2,0.0,11.0\n2025-09-17,8,22.4,1.6,0.0,12.3\n2025-09-17,9,22.6,0.5,0.0,12.3\n2025-09-17,10,23.2,7.9,0.0,4.4\n2025-09-17,11,23.1,2.1,0.0,10.7\n2025-09-17,12,22.4,14.9,0.0,14.3\n2025-09-17,13,22.9,0.5,0.0,10.4\n2025-09-17,14,23.5,0.0,0.0,14.7\n2025-09-17,15,23.7,0.0,0.0,15.6\n2025-09-17,16,22.7,0.0,0.0,14.3\n2025-09-17,17,22.8,0.0,0.0,11.3\n2025-09-17,18,22.2,0.0,0.0,11.2\n2025-09-17,19,21.4,0.0,0.0,11.4\n2025-09-17,20,21.0,0.0,0.0,9.7\n2025-09-17,21,20.6,0.0,0.0,11.6\n2025-09-17,22,21.0,0.3,0.0,9.7\n2025-09-17,23,20.6,0.1,0.0,9.6\n2025-09-18,0,20.2,0.0,0.0,8.4\n2025-09-18,1,19.7,0.0,0.0,9.4\n2025-09-18,2,19.1,0.0,0.0,7.8\n2025-09-18,3,18.9,0.0,0.0,6.5\n2025-09-18,4,18.6,0.0,0.0,7.8\n2025-09-18,5,18.5,0.0,0.0,7.2\n2025-09-18,6,18.1,0.0,0.0,5.9\n2025-09-18,7,18.5,0.0,0.0,5.4\n2025-09-18,8,19.2,0.0,0.0,5.1\n2025-09-18,9,20.4,0.0,0.0,5.1\n2025-09-18,10,22.3,0.0,0.0,7.9\n2025-09-18,11,23.1,0.1,0.0,12.3\n2025-09-18,12,23.6,0.1,0.0,13.4\n2025-09-18,13,24.2,0.1,0.0,12.5\n2025-09-18,14,24.9,0.1,0.0,12.0\n2025-09-18,15,25.3,0.0,0.0,11.7\n2025-09-18,16,24.7,0.0,0.0,13.1\n2025-09-18,17,23.5,0.0,0.0,10.9\n2025-09-18,18,23.0,0.0,0.0,9.0\n2025-09-18,19,21.7,0.0,0.0,7.2\n2025-09-18,20,21.3,0.0,0.0,8.8\n2025-09-18,21,20.8,0.0,0.0,9.8\n2025-09-18,22,20.8,0.0,0.0,11.3\n2025-09-18,23,20.3,0.0,0.0,11.9\n2025-09-19,0,20.0,0.0,0.0,12.8\n2025-09-19,1,19.7,0.0,0.0,11.7\n2025-09-19,2,19.3,0.0,0.0,10.5\n2025-09-19,3,19.0,0.0,0.0,9.5\n2025-09-19,4,18.9,0.0,0.0,9.8\n2025-09-19,5,18.8,0.0,0.0,10.6\n2025-09-19,6,18.8,0.0,0.0,10.9\n2025-09-19,7,18.9,0.0,0.0,10.9\n2025-09-19,8,19.4,0.0,0.0,11.4\n2025-09-19,9,19.8,0.0,0.0,11.3\n2025-09-19,10,19.6,0.7,0.0,13.2\n2025-09-19,11,20.3,0.1,0.0,12.6\n2025-09-19,12,20.6,0.2,0.0,12.7\n2025-09-19,13,20.6,0.3,0.0,11.7\n2025-09-19,14,20.6,0.3,0.0,9.9\n2025-09-19,15,20.7,0.6,0.0,10.5\n2025-09-19,16,20.2,1.0,0.0,11.6\n2025-09-19,17,19.9,1.0,0.0,9.7\n2025-09-19,18,19.4,1.1,0.0,10.4\n2025-09-19,19,19.0,0.8,0.0,8.6\n2025-09-19,20,18.8,1.2,0.0,11.6\n2025-09-19,21,18.6,2.1,0.0,10.3\n2025-09-19,22,18.2,3.1,0.0,9.9\n2025-09-19,23,18.1,2.6,0.0,11.5\n2025-09-20,0,18.0,4.4,0.0,16.0\n2025-09-20,1,17.8,7.7,0.0,10.5\n2025-09-20,2,17.7,6.6,0.0,12.6\n2025-09-20,3,17.6,7.0,0.0,13.1\n2025-09-20,4,17.4,7.7,0.0,10.9\n2025-09-20,5,17.2,4.9,0.0,9.9\n2025-09-20,6,17.3,0.5,0.0,12.5\n2025-09-20,7,17.1,0.0,0.0,11.3\n2025-09-20,8,17.8,0.0,0.0,8.1\n2025-09-20,9,18.6,0.0,0.0,9.3\n2025-09-20,10,19.1,0.0,0.0,8.6\n2025-09-20,11,20.5,0.0,0.0,9.7\n2025-09-20,12,21.9,0.0,0.0,11.0\n2025-09-20,13,22.2,0.0,0.0,9.3\n2025-09-20,14,22.9,0.0,0.0,8.7\n2025-09-20,15,23.0,0.0,0.0,8.5\n2025-09-20,16,22.9,0.0,0.0,8.7\n2025-09-20,17,21.6,0.1,0.0,10.5\n2025-09-20,18,21.0,0.0,0.0,8.9\n2025-09-20,19,20.1,0.0,0.0,11.8\n2025-09-20,20,19.1,0.0,0.0,12.3\n2025-09-20,21,18.2,0.0,0.0,11.3\n2025-09-20,22,18.3,0.0,0.0,10.1\n2025-09-20,23,17.8,0.0,0.0,9.8\n2025-09-21,0,17.4,0.0,0.0,9.1\n2025-09-21,1,17.1,0.0,0.0,8.9\n2025-09-21,2,17.0,0.0,0.0,8.4\n2025-09-21,3,17.0,0.0,0.0,7.9\n2025-09-21,4,16.7,0.0,0.0,7.6\n2025-09-21,5,16.3,0.0,0.0,7.4\n2025-09-21,6,16.0,0.0,0.0,8.8\n2025-09-21,7,16.2,0.0,0.0,8.3\n2025-09-21,8,17.8,0.0,0.0,6.4\n2025-09-21,9,19.4,0.0,0.0,7.6\n2025-09-21,10,21.7,0.0,0.0,10.2\n2025-09-21,11,23.0,0.0,0.0,10.8\n2025-09-21,12,23.9,0.0,0.0,9.8\n2025-09-21,13,24.9,0.0,0.0,10.0\n2025-09-21,14,25.4,0.0,0.0,9.6\n2025-09-21,15,25.6,0.0,0.0,8.7\n2025-09-21,16,25.3,0.0,0.0,7.6\n2025-09-21,17,25.0,0.0,0.0,4.4\n2025-09-21,18,23.2,0.0,0.0,4.1\n2025-09-21,19,21.5,0.0,0.0,6.3\n2025-09-21,20,20.6,0.0,0.0,10.3\n2025-09-21,21,20.0,0.0,0.0,11.7\n2025-09-21,22,20.1,0.0,0.0,11.0\n2025-09-21,23,19.6,0.0,0.0,10.5\n2025-09-22,0,19.3,0.0,0.0,9.6\n2025-09-22,1,19.1,0.0,0.0,9.3\n2025-09-22,2,18.8,0.0,0.0,9.8\n2025-09-22,3,18.6,0.0,0.0,9.6\n2025-09-22,4,18.4,0.0,0.0,7.8\n2025-09-22,5,18.1,0.0,0.0,8.4\n2025-09-22,6,17.9,0.0,0.0,9.2\n2025-09-22,7,17.8,0.0,0.0,7.6\n2025-09-22,8,18.2,0.0,0.0,7.2\n2025-09-22,9,19.4,0.0,0.0,6.6\n2025-09-22,10,19.8,0.0,0.0,8.5\n2025-09-22,11,22.0,0.0,0.0,11.2\n2025-09-22,12,22.6,0.0,0.0,11.2\n2025-09-22,13,24.1,0.0,0.0,10.4\n2025-09-22,14,24.0,0.0,0.0,9.7\n2025-09-22,15,23.0,0.0,0.0,7.3\n2025-09-22,16,22.8,0.0,0.0,5.7\n2025-09-22,17,22.6,0.0,0.0,3.4\n2025-09-22,18,21.9,0.0,0.0,4.6\n2025-09-22,19,20.6,0.0,0.0,6.4\n2025-09-22,20,20.0,0.0,0.0,6.5\n2025-09-22,21,20.2,0.0,0.0,8.9\n2025-09-22,22,20.1,0.0,0.0,7.1\n2025-09-22,23,20.0,0.0,0.0,8.4\n2025-09-23,0,19.7,0.0,0.0,9.7\n2025-09-23,1,19.5,0.0,0.0,9.5\n2025-09-23,2,19.0,0.0,0.0,9.0\n2025-09-23,3,18.5,0.0,0.0,8.2\n2025-09-23,4,18.3,0.0,0.0,6.6\n2025-09-23,5,18.1,0.0,0.0,7.3\n2025-09-23,6,18.2,0.0,0.0,7.0\n2025-09-23,7,18.1,0.0,0.0,7.0\n2025-09-23,8,19.1,0.0,0.0,7.0\n2025-09-23,9,20.5,0.0,0.0,5.8\n2025-09-23,10,22.0,0.0,0.0,9.7\n2025-09-23,11,23.1,0.0,0.0,10.1\n2025-09-23,12,23.1,0.0,0.0,8.1\n2025-09-23,13,24.6,0.0,0.0,10.5\n2025-09-23,14,24.5,0.0,0.0,11.1\n2025-09-23,15,23.4,0.0,0.0,9.3\n2025-09-23,16,23.6,0.0,0.0,8.7\n2025-09-23,17,23.1,0.0,0.0,5.8\n2025-09-23,18,22.0,0.0,0.0,5.4\n2025-09-23,19,21.6,0.0,0.0,8.5\n2025-09-23,20,21.4,0.0,0.0,10.6\n2025-09-23,21,21.3,0.0,0.0,12.5\n2025-09-23,22,21.7,0.0,0.0,12.1\n2025-09-23,23,21.1,0.0,0.0,15.9\n2025-09-24,0,20.5,0.0,0.0,17.0\n2025-09-24,1,19.9,0.0,0.0,16.9\n2025-09-24,2,19.6,0.0,0.0,16.7\n2025-09-24,3,19.3,0.0,0.0,16.8\n2025-09-24,4,19.2,0.0,0.0,16.2\n2025-09-24,5,19.0,0.0,0.0,15.7\n2025-09-24,6,19.0,0.0,0.0,17.1\n2025-09-24,7,19.3,0.0,0.0,17.2\n2025-09-24,8,20.6,0.0,0.0,14.2\n2025-09-24,9,21.9,0.0,0.0,14.7\n2025-09-24,10,22.8,0.0,0.0,13.1\n2025-09-24,11,23.9,0.0,0.0,14.1\n2025-09-24,12,24.1,0.9,0.0,15.0\n2025-09-24,13,22.5,3.2,0.0,18.4\n2025-09-24,14,22.3,4.8,0.0,14.2\n2025-09-24,15,22.6,3.8,0.0,19.5\n2025-09-24,16,22.6,5.8,0.0,18.4\n2025-09-24,17,22.5,15.8,0.0,19.8\n2025-09-24,18,21.9,3.3,0.0,15.3\n2025-09-24,19,20.3,2.7,0.0,8.4\n2025-09-24,20,20.1,4.6,0.0,8.7\n2025-09-24,21,20.4,0.7,0.0,6.1\n2025-09-24,22,21.1,0.2,0.0,3.8\n2025-09-24,23,21.0,0.1,0.0,6.3\n2025-09-25,0,20.8,0.0,0.0,6.6\n2025-09-25,1,20.5,0.0,0.0,5.5\n2025-09-25,2,20.3,0.0,0.0,3.3\n2025-09-25,3,20.1,0.0,0.0,6.9\n2025-09-25,4,19.5,0.1,0.0,7.3\n2025-09-25,5,18.9,0.0,0.0,5.6\n2025-09-25,6,18.6,0.0,0.0,6.1\n2025-09-25,7,18.6,0.2,0.0,4.5\n2025-09-25,8,19.4,0.0,0.0,4.7\n2025-09-25,9,20.8,0.0,0.0,4.5\n2025-09-25,10,21.9,0.0,0.0,2.2\n2025-09-25,11,23.1,0.0,0.0,0.9\n2025-09-25,12,24.1,0.0,0.0,1.5\n2025-09-25,13,24.5,0.0,0.0,4.3\n2025-09-25,14,25.3,0.0,0.0,5.1\n2025-09-25,15,25.3,0.0,0.0,7.4\n2025-09-25,16,24.4,0.0,0.0,9.8\n2025-09-25,17,23.5,0.0,0.0,8.5\n2025-09-25,18,22.2,0.0,0.0,5.9\n2025-09-25,19,20.9,0.0,0.0,5.2\n2025-09-25,20,20.3,0.0,0.0,4.8\n2025-09-25,21,19.8,0.0,0.0,4.9\n2025-09-25,22,19.7,0.0,0.0,4.2\n2025-09-25,23,19.4,0.0,0.0,4.3\n2025-09-26,0,19.6,0.0,0.0,6.9\n2025-09-26,1,19.5,0.0,0.0,6.3\n2025-09-26,2,19.0,0.0,0.0,6.2\n2025-09-26,3,18.0,0.0,0.0,4.6\n2025-09-26,4,18.2,0.0,0.0,6.2\n2025-09-26,5,18.0,0.0,0.0,5.9\n2025-09-26,6,17.7,0.0,0.0,7.3\n2025-09-26,7,17.6,0.0,0.0,6.7\n2025-09-26,8,17.7,0.0,0.0,5.3\n2025-09-26,9,18.2,0.0,0.0,7.5\n2025-09-26,10,20.2,0.0,0.0,7.4\n2025-09-26,11,21.9,0.0,0.0,5.1\n2025-09-26,12,23.0,0.0,0.0,2.7\n2025-09-26,13,23.9,0.0,0.0,5.1\n2025-09-26,14,24.5,0.0,0.0,6.0\n2025-09-26,15,24.8,0.0,0.0,4.7\n2025-09-26,16,24.7,0.0,0.0,5.5\n2025-09-26,17,24.0,0.0,0.0,6.5\n2025-09-26,18,22.6,0.0,0.0,6.6\n2025-09-26,19,21.5,0.0,0.0,5.6\n2025-09-26,20,20.8,0.0,0.0,5.2\n2025-09-26,21,20.2,0.0,0.0,3.7\n2025-09-26,22,20.3,0.0,0.0,2.5\n2025-09-26,23,19.8,0.0,0.0,6.1\n2025-09-27,0,19.0,0.0,0.0,1.5\n2025-09-27,1,18.4,0.0,0.0,1.3\n2025-09-27,2,18.0,0.0,0.0,1.8\n2025-09-27,3,17.3,0.0,0.0,3.1\n2025-09-27,4,17.0,0.0,0.0,3.1\n2025-09-27,5,16.8,0.0,0.0,3.6\n2025-09-27,6,16.7,0.0,0.0,4.4\n2025-09-27,7,16.8,0.0,0.0,5.7\n2025-09-27,8,18.7,0.0,0.0,5.4\n2025-09-27,9,21.3,0.0,0.0,4.9\n2025-09-27,10,23.4,0.0,0.0,8.2\n2025-09-27,11,25.1,0.0,0.0,10.0\n2025-09-27,12,26.1,0.0,0.0,11.8\n2025-09-27,13,26.3,0.0,0.0,10.7\n2025-09-27,14,26.6,0.0,0.0,10.2\n2025-09-27,15,26.7,0.0,0.0,9.1\n2025-09-27,16,26.4,0.0,0.0,9.4\n2025-09-27,17,25.9,0.0,0.0,5.9\n2025-09-27,18,24.1,0.0,0.0,4.5\n2025-09-27,19,22.3,0.0,0.0,5.2\n2025-09-27,20,22.0,0.0,0.0,4.1\n2025-09-27,21,22.0,0.0,0.0,4.7\n2025-09-27,22,21.4,0.0,0.0,4.4\n2025-09-27,23,21.5,0.0,0.0,4.2\n2025-09-28,0,21.0,0.0,0.0,4.9\n2025-09-28,1,21.0,0.0,0.0,8.2\n2025-09-28,2,20.3,0.2,0.0,8.0\n2025-09-28,3,20.1,0.0,0.0,8.0\n2025-09-28,4,19.8,0.0,0.0,6.9\n2025-09-28,5,19.9,0.0,0.0,6.4\n2025-09-28,6,20.2,0.0,0.0,6.9\n2025-09-28,7,20.1,1.1,0.0,3.4\n2025-09-28,8,20.1,4.1,0.0,2.9\n2025-09-28,9,20.1,5.8,0.0,4.3\n2025-09-28,10,19.6,6.5,0.0,7.6\n2025-09-28,11,19.7,4.4,0.0,4.3\n2025-09-28,12,19.8,1.5,0.0,6.6\n2025-09-28,13,22.3,0.3,0.0,8.5\n2025-09-28,14,22.7,0.0,0.0,9.9\n2025-09-28,15,23.1,0.0,0.0,11.2\n2025-09-28,16,22.3,0.0,0.0,13.7\n2025-09-28,17,22.0,0.0,0.0,10.8\n2025-09-28,18,21.3,0.0,0.0,9.0\n2025-09-28,19,20.5,0.0,0.0,10.0\n2025-09-28,20,19.8,0.0,0.0,9.0\n2025-09-28,21,19.3,0.0,0.0,8.1\n2025-09-28,22,18.8,0.0,0.0,8.1\n2025-09-28,23,18.7,0.0,0.0,8.0\n2025-09-29,0,18.4,0.0,0.0,9.3\n2025-09-29,1,17.7,0.0,0.0,9.3\n2025-09-29,2,17.2,0.0,0.0,9.6\n2025-09-29,3,17.2,0.0,0.0,8.6\n2025-09-29,4,16.7,0.0,0.0,8.8\n2025-09-29,5,16.7,0.0,0.0,4.8\n2025-09-29,6,16.7,0.0,0.0,3.8\n2025-09-29,7,16.7,0.0,0.0,4.4\n2025-09-29,8,17.6,0.0,0.0,3.6\n2025-09-29,9,18.3,0.0,0.0,3.8\n2025-09-29,10,20.2,0.0,0.0,4.9\n2025-09-29,11,21.0,0.0,0.0,4.4\n2025-09-29,12,21.5,0.0,0.0,3.5\n2025-09-29,13,22.8,0.0,0.0,2.8\n2025-09-29,14,23.3,0.0,0.0,4.0\n2025-09-29,15,23.2,0.0,0.0,5.4\n2025-09-29,16,22.8,0.0,0.0,6.0\n2025-09-29,17,22.1,0.0,0.0,7.1\n2025-09-29,18,20.4,0.0,0.0,9.7\n2025-09-29,19,19.2,0.0,0.0,5.0\n2025-09-29,20,18.7,0.0,0.0,3.2\n2025-09-29,21,18.6,0.0,0.0,1.8\n2025-09-29,22,18.1,0.0,0.0,1.3\n2025-09-29,23,17.7,0.0,0.0,0.9\n2025-09-30,0,17.3,0.0,0.0,0.6\n2025-09-30,1,16.9,0.0,0.0,1.3\n2025-09-30,2,16.5,0.0,0.0,2.3\n2025-09-30,3,16.3,0.0,0.0,1.5\n2025-09-30,4,16.2,0.0,0.0,1.3\n2025-09-30,5,15.7,0.0,0.0,1.4\n2025-09-30,6,15.7,0.0,0.0,3.1\n2025-09-30,7,15.7,0.0,0.0,3.6\n2025-09-30,8,16.9,0.0,0.0,4.7\n2025-09-30,9,18.2,0.0,0.0,3.5\n2025-09-30,10,19.4,0.0,0.0,2.7\n2025-09-30,11,20.4,0.0,0.0,1.9\n2025-09-30,12,21.0,0.0,0.0,0.9\n2025-09-30,13,21.7,0.0,0.0,1.9\n2025-09-30,14,22.6,0.0,0.0,3.9\n2025-09-30,15,22.3,0.0,0.0,5.0\n2025-09-30,16,21.4,0.0,0.0,6.2\n2025-09-30,17,21.1,0.0,0.0,5.2\n2025-09-30,18,20.3,0.0,0.0,5.1\n2025-09-30,19,19.6,0.0,0.0,5.2\n2025-09-30,20,18.4,0.0,0.0,2.9\n2025-09-30,21,18.1,0.0,0.0,1.7\n2025-09-30,22,17.8,0.0,0.0,1.5\n2025-09-30,23,17.3,0.0,0.0,1.6\n2025-10-01,0,17.1,0.0,0.0,1.3\n2025-10-01,1,16.7,0.0,0.0,1.3\n2025-10-01,2,16.4,0.0,0.0,0.7\n2025-10-01,3,16.1,0.0,0.0,1.0\n2025-10-01,4,15.7,0.0,0.0,2.7\n2025-10-01,5,15.6,0.0,0.0,3.6\n2025-10-01,6,15.6,0.0,0.0,4.1\n2025-10-01,7,15.6,0.0,0.0,5.0\n2025-10-01,8,16.9,0.0,0.0,5.4\n2025-10-01,9,19.0,0.0,0.0,5.1\n2025-10-01,10,21.0,0.0,0.0,6.2\n2025-10-01,11,22.2,0.1,0.0,6.3\n2025-10-01,12,22.9,0.1,0.0,7.6\n2025-10-01,13,23.3,0.1,0.0,9.4\n2025-10-01,14,23.5,0.0,0.0,11.4\n2025-10-01,15,24.0,0.1,0.0,7.9\n2025-10-01,16,22.9,0.1,0.0,6.6\n2025-10-01,17,22.3,0.9,0.0,6.7\n2025-10-01,18,21.3,0.0,0.0,4.8\n2025-10-01,19,20.1,0.0,0.0,3.5\n2025-10-01,20,19.9,0.0,0.0,1.5\n2025-10-01,21,19.1,0.0,0.0,4.2\n2025-10-01,22,18.8,0.0,0.0,4.3\n2025-10-01,23,18.3,0.0,0.0,3.0\n2025-10-02,0,17.8,0.0,0.0,2.1\n2025-10-02,1,17.6,0.0,0.0,2.2\n2025-10-02,2,17.8,0.0,0.0,2.9\n2025-10-02,3,17.5,0.0,0.0,4.7\n2025-10-02,4,17.6,0.0,0.0,4.3\n2025-10-02,5,17.4,0.0,0.0,4.6\n2025-10-02,6,17.2,0.0,0.0,5.0\n2025-10-02,7,17.1,0.0,0.0,5.7\n2025-10-02,8,17.6,0.0,0.0,5.0\n2025-10-02,9,18.8,0.0,0.0,1.6\n2025-10-02,10,21.3,0.0,0.0,1.3\n2025-10-02,11,22.6,0.0,0.0,0.4\n2025-10-02,12,23.7,0.0,0.0,3.4\n2025-10-02,13,24.6,0.0,0.0,4.2\n2025-10-02,14,24.7,0.0,0.0,3.8\n2025-10-02,15,25.0,0.0,0.0,3.6\n2025-10-02,16,24.8,0.0,0.0,4.1\n2025-10-02,17,23.7,0.0,0.0,4.1\n2025-10-02,18,22.2,0.0,0.0,3.3\n2025-10-02,19,21.4,0.0,0.0,2.4\n2025-10-02,20,21.4,0.0,0.0,1.8\n2025-10-02,21,20.5,0.0,0.0,3.2\n2025-10-02,22,20.1,0.0,0.0,2.3\n2025-10-02,23,20.3,0.0,0.0,5.6\n2025-10-03,0,20.7,0.0,0.0,8.1\n2025-10-03,1,20.9,0.0,0.0,11.1\n2025-10-03,2,20.4,0.0,0.0,11.5\n2025-10-03,3,19.7,0.0,0.0,6.8\n2025-10-03,4,19.3,0.4,0.0,9.3\n2025-10-03,5,18.8,0.3,0.0,7.7\n2025-10-03,6,18.7,0.1,0.0,9.2\n2025-10-03,7,18.8,0.0,0.0,11.0\n2025-10-03,8,19.5,0.0,0.0,12.0\n2025-10-03,9,20.0,0.0,0.0,8.6\n2025-10-03,10,20.7,0.8,0.0,13.1\n2025-10-03,11,21.2,0.2,0.0,14.8\n2025-10-03,12,22.1,0.0,0.0,16.1\n2025-10-03,13,23.4,0.0,0.0,18.7\n2025-10-03,14,23.8,0.0,0.0,19.9\n2025-10-03,15,22.0,0.0,0.0,15.7\n2025-10-03,16,22.2,0.0,0.0,14.3\n2025-10-03,17,22.0,0.0,0.0,12.6\n2025-10-03,18,21.6,0.0,0.0,10.3\n2025-10-03,19,21.3,0.0,0.0,9.4\n2025-10-03,20,21.3,0.0,0.0,8.8\n2025-10-03,21,20.8,0.0,0.0,10.2\n2025-10-03,22,20.7,0.0,0.0,9.6\n2025-10-03,23,20.4,0.0,0.0,8.6\n2025-10-04,0,20.1,0.0,0.0,8.3\n2025-10-04,1,20.2,0.0,0.0,8.2\n2025-10-04,2,20.1,0.0,0.0,8.0\n2025-10-04,3,19.5,0.0,0.0,7.3\n2025-10-04,4,19.7,0.0,0.0,8.2\n2025-10-04,5,19.7,0.0,0.0,7.6\n2025-10-04,6,19.7,0.0,0.0,6.8\n2025-10-04,7,19.7,0.1,0.0,7.0\n2025-10-04,8,20.3,0.0,0.0,6.4\n2025-10-04,9,20.4,0.0,0.0,6.1\n2025-10-04,10,21.4,0.0,0.0,7.3\n2025-10-04,11,21.9,0.0,0.0,5.7\n2025-10-04,12,22.2,0.0,0.0,6.1\n2025-10-04,13,22.7,0.0,0.0,6.0\n2025-10-04,14,23.6,0.0,0.0,7.2\n2025-10-04,15,23.8,0.0,0.0,7.4\n2025-10-04,16,23.8,0.0,0.0,6.0\n2025-10-04,17,23.4,0.0,0.0,5.4\n2025-10-04,18,22.5,0.0,0.0,4.7\n2025-10-04,19,21.8,0.0,0.0,4.2\n2025-10-04,20,21.6,0.0,0.0,3.7\n2025-10-04,21,21.8,0.0,0.0,5.6\n2025-10-04,22,21.3,0.0,0.0,2.9\n2025-10-04,23,21.0,0.0,0.0,0.8\n2025-10-05,0,20.7,0.0,0.0,1.8\n2025-10-05,1,20.4,0.0,0.0,4.5\n2025-10-05,2,20.3,0.0,0.0,3.8\n2025-10-05,3,20.1,0.0,0.0,3.1\n2025-10-05,4,20.0,0.0,0.0,3.4\n2025-10-05,5,19.9,0.0,0.0,2.5\n2025-10-05,6,19.6,0.1,0.0,1.0\n2025-10-05,7,19.7,1.1,0.0,2.8\n2025-10-05,8,20.2,0.0,0.0,6.0\n2025-10-05,9,21.0,0.0,0.0,3.1\n2025-10-05,10,21.4,0.5,0.0,2.1\n2025-10-05,11,22.0,0.2,0.0,2.8\n2025-10-05,12,22.0,0.2,0.0,7.1\n2025-10-05,13,22.5,0.1,0.0,6.3\n2025-10-05,14,23.9,0.0,0.0,3.2\n2025-10-05,15,23.0,0.1,0.0,2.5\n2025-10-05,16,21.2,1.3,0.0,5.2\n2025-10-05,17,21.6,0.2,0.0,2.9\n2025-10-05,18,21.1,0.0,0.0,1.5\n2025-10-05,19,20.6,0.0,0.0,4.5\n2025-10-05,20,20.3,0.0,0.0,3.1\n2025-10-05,21,20.8,0.0,0.0,5.5\n2025-10-05,22,20.7,0.0,0.0,1.5\n2025-10-05,23,20.1,0.0,0.0,3.4\n2025-10-06,0,20.2,0.0,0.0,3.7\n2025-10-06,1,20.3,0.0,0.0,2.1\n2025-10-06,2,20.0,0.0,0.0,4.0\n2025-10-06,3,20.6,0.0,0.0,7.4\n2025-10-06,4,20.6,0.1,0.0,8.1\n2025-10-06,5,20.5,0.1,0.0,9.4\n2025-10-06,6,20.4,0.2,0.0,9.0\n2025-10-06,7,20.1,0.8,0.0,8.3\n2025-10-06,8,20.2,1.0,0.0,10.1\n2025-10-06,9,19.9,1.2,0.0,13.8\n2025-10-06,10,20.0,0.1,0.0,16.3\n2025-10-06,11,20.0,0.3,0.0,16.9\n2025-10-06,12,19.7,0.2,0.0,16.1\n2025-10-06,13,19.3,0.3,0.0,16.1\n2025-10-06,14,19.3,0.6,0.0,16.5\n2025-10-06,15,17.5,0.8,0.0,18.2\n2025-10-06,16,17.8,1.7,0.0,19.1\n2025-10-06,17,16.7,1.1,0.0,20.6\n2025-10-06,18,16.2,3.2,0.0,22.2\n2025-10-06,19,16.1,6.4,0.0,22.5\n2025-10-06,20,16.0,5.4,0.0,21.3\n2025-10-06,21,16.0,2.7,0.0,21.1\n2025-10-06,22,16.1,1.8,0.0,19.7\n2025-10-06,23,16.3,1.6,0.0,19.2\n2025-10-07,0,16.4,2.6,0.0,18.9\n2025-10-07,1,16.6,3.3,0.0,16.9\n2025-10-07,2,16.9,0.7,0.0,15.7\n2025-10-07,3,16.3,0.5,0.0,20.4\n2025-10-07,4,17.1,0.2,0.0,17.7\n2025-10-07,5,17.1,1.1,0.0,15.6\n2025-10-07,6,17.2,0.9,0.0,16.5\n2025-10-07,7,17.6,0.3,0.0,17.2\n2025-10-07,8,18.1,0.4,0.0,16.5\n2025-10-07,9,18.6,0.2,0.0,14.4\n2025-10-07,10,19.4,0.1,0.0,14.0\n2025-10-07,11,20.1,0.1,0.0,12.5\n2025-10-07,12,20.9,0.0,0.0,12.4\n2025-10-07,13,21.4,0.0,0.0,12.1\n2025-10-07,14,21.9,0.0,0.0,11.8\n2025-10-07,15,20.5,0.0,0.0,9.4\n2025-10-07,16,20.6,0.0,0.0,7.4\n2025-10-07,17,20.1,0.1,0.0,10.0\n2025-10-07,18,19.3,0.6,0.0,11.5\n2025-10-07,19,18.9,0.2,0.0,10.6\n2025-10-07,20,18.8,0.1,0.0,9.6\n2025-10-07,21,18.7,0.0,0.0,9.9\n2025-10-07,22,18.4,0.0,0.0,10.1\n2025-10-07,23,18.1,0.0,0.0,8.6\n2025-10-08,0,18.1,0.0,0.0,7.9\n2025-10-08,1,17.6,0.0,0.0,9.5\n2025-10-08,2,17.0,0.0,0.0,11.7\n2025-10-08,3,17.5,0.0,0.0,9.4\n2025-10-08,4,17.1,0.0,0.0,11.4\n2025-10-08,5,16.5,0.0,0.0,10.8\n2025-10-08,6,16.1,0.0,0.0,10.1\n2025-10-08,7,16.0,0.0,0.0,9.2\n2025-10-08,8,17.2,0.0,0.0,9.4\n2025-10-08,9,18.5,0.0,0.0,7.6\n2025-10-08,10,20.6,0.0,0.0,7.8\n2025-10-08,11,22.5,0.0,0.0,6.9\n2025-10-08,12,23.9,0.0,0.0,6.9\n2025-10-08,13,24.8,0.0,0.0,6.4\n2025-10-08,14,25.6,0.0,0.0,7.2\n2025-10-08,15,26.0,0.0,0.0,8.3\n2025-10-08,16,25.3,0.0,0.0,11.5\n2025-10-08,17,23.9,0.0,0.0,10.0\n2025-10-08,18,22.0,0.0,0.0,7.9\n2025-10-08,19,20.9,0.0,0.0,9.7\n2025-10-08,20,19.7,0.0,0.0,12.5\n2025-10-08,21,19.6,0.0,0.0,11.7\n2025-10-08,22,18.8,0.0,0.0,9.8\n2025-10-08,23,18.2,0.0,0.0,8.3\n2025-10-09,0,17.9,0.0,0.0,7.6\n2025-10-09,1,17.8,0.0,0.0,7.2\n2025-10-09,2,18.7,0.0,0.0,10.1\n2025-10-09,3,20.3,0.0,0.0,11.2\n2025-10-09,4,20.2,0.0,0.0,10.8\n2025-10-09,5,19.4,0.0,0.0,10.4\n2025-10-09,6,19.2,0.0,0.0,11.8\n2025-10-09,7,19.0,0.0,0.0,11.8\n2025-10-09,8,19.4,0.0,0.0,12.7\n2025-10-09,9,20.4,0.0,0.0,9.3\n2025-10-09,10,20.5,0.0,0.0,10.1\n2025-10-09,11,21.7,0.0,0.0,12.4\n2025-10-09,12,22.0,0.0,0.0,12.5\n2025-10-09,13,21.9,0.0,0.0,12.9\n2025-10-09,14,21.9,0.0,0.0,12.4\n2025-10-09,15,22.3,0.0,0.0,10.8\n2025-10-09,16,21.9,0.0,0.0,8.8\n2025-10-09,17,21.2,0.0,0.0,8.9\n2025-10-09,18,20.5,0.5,0.0,10.8\n2025-10-09,19,19.8,0.3,0.0,13.2\n2025-10-09,20,19.8,0.0,0.0,14.1\n2025-10-09,21,19.3,0.0,0.0,11.8\n2025-10-09,22,19.1,0.1,0.0,12.5\n2025-10-09,23,19.0,0.0,0.0,12.7\n2025-10-10,0,18.6,0.1,0.0,11.9\n2025-10-10,1,18.2,0.2,0.0,11.3\n2025-10-10,2,17.7,0.7,0.0,12.3\n2025-10-10,3,16.9,0.6,0.0,12.3\n2025-10-10,4,17.1,2.1,0.0,13.0\n2025-10-10,5,17.5,0.0,0.0,13.4\n2025-10-10,6,17.6,0.2,0.0,13.7\n2025-10-10,7,17.7,0.2,0.0,15.9\n2025-10-10,8,18.0,0.1,0.0,16.1\n2025-10-10,9,17.3,0.2,0.0,8.1\n2025-10-10,10,17.5,1.1,0.0,11.1\n2025-10-10,11,18.4,0.9,0.0,11.6\n2025-10-10,12,19.0,0.8,0.0,11.8\n2025-10-10,13,19.8,0.2,0.0,12.1\n2025-10-10,14,19.9,0.2,0.0,12.3\n2025-10-10,15,18.9,0.2,0.0,11.2\n2025-10-10,16,18.9,0.4,0.0,10.7\n2025-10-10,17,18.4,0.8,0.0,11.3\n2025-10-10,18,18.3,0.5,0.0,12.8\n2025-10-10,19,18.2,0.5,0.0,12.1\n2025-10-10,20,18.2,1.9,0.0,12.5\n2025-10-10,21,18.2,2.3,0.0,14.6\n2025-10-10,22,18.1,4.1,0.0,14.1\n2025-10-10,23,18.2,2.0,0.0,15.0\n2025-10-11,0,18.3,1.3,0.0,14.8\n2025-10-11,1,18.4,2.8,0.0,15.6\n2025-10-11,2,18.5,1.5,0.0,15.0\n2025-10-11,3,18.4,0.9,0.0,14.9\n2025-10-11,4,18.6,0.9,0.0,13.4\n2025-10-11,5,18.7,1.7,0.0,14.9\n2025-10-11,6,19.0,0.4,0.0,18.0\n2025-10-11,7,19.3,0.1,0.0,18.8\n2025-10-11,8,20.1,0.1,0.0,18.4\n2025-10-11,9,20.3,0.1,0.0,13.3\n2025-10-11,10,22.0,0.1,0.0,14.9\n2025-10-11,11,23.1,0.0,0.0,20.2\n2025-10-11,12,24.2,0.1,0.0,22.9\n2025-10-11,13,24.6,0.0,0.0,21.5\n2025-10-11,14,24.2,0.1,0.0,22.4\n2025-10-11,15,24.1,0.0,0.0,20.4\n2025-10-11,16,23.3,0.2,0.0,15.4\n2025-10-11,17,22.3,1.2,0.0,16.2\n2025-10-11,18,22.0,1.0,0.0,16.6\n2025-10-11,19,21.7,2.1,0.0,18.1\n2025-10-11,20,22.0,0.2,0.0,12.5\n2025-10-11,21,21.5,0.0,0.0,9.4\n2025-10-11,22,20.5,0.0,0.0,10.2\n2025-10-11,23,19.3,0.0,0.0,11.5\n2025-10-12,0,18.1,0.0,0.0,10.1\n2025-10-12,1,17.2,0.0,0.0,9.4\n2025-10-12,2,16.4,0.0,0.0,10.8\n2025-10-12,3,16.5,0.0,0.0,10.9\n2025-10-12,4,16.1,0.0,0.0,9.0\n2025-10-12,5,15.9,0.0,0.0,7.6\n2025-10-12,6,15.8,0.0,0.0,6.3\n2025-10-12,7,16.0,0.2,0.0,6.3\n2025-10-12,8,16.0,0.7,0.0,7.2\n2025-10-12,9,16.7,0.1,0.0,3.8\n2025-10-12,10,18.0,0.0,0.0,4.0\n2025-10-12,11,20.4,0.0,0.0,5.4\n2025-10-12,12,22.2,0.0,0.0,4.7\n2025-10-12,13,22.6,0.1,0.0,4.9\n2025-10-12,14,22.3,0.1,0.0,3.8\n2025-10-12,15,21.5,0.4,0.0,7.6\n2025-10-12,16,21.4,0.4,0.0,9.6\n2025-10-12,17,20.8,0.3,0.0,6.7\n2025-10-12,18,19.8,0.3,0.0,8.0\n2025-10-12,19,19.2,0.0,0.0,9.6\n2025-10-12,20,18.9,0.0,0.0,8.9\n2025-10-12,21,18.7,0.0,0.0,9.5\n2025-10-12,22,18.3,0.0,0.0,9.4\n2025-10-12,23,18.1,0.0,0.0,8.5\n2025-10-13,0,18.5,0.0,0.0,9.6\n2025-10-13,1,18.6,0.1,0.0,7.2\n2025-10-13,2,18.4,0.0,0.0,8.5\n2025-10-13,3,18.3,0.0,0.0,10.2\n2025-10-13,4,17.8,0.0,0.0,8.3\n2025-10-13,5,17.4,2.5,0.0,5.4\n2025-10-13,6,17.1,2.1,0.0,10.2\n2025-10-13,7,17.1,0.1,0.0,14.2\n2025-10-13,8,16.9,0.6,0.0,15.2\n2025-10-13,9,16.8,2.2,0.0,13.2\n2025-10-13,10,17.1,0.7,0.0,13.2\n2025-10-13,11,18.0,0.6,0.0,12.3\n2025-10-13,12,18.1,0.1,0.0,13.1\n2025-10-13,13,18.2,0.0,0.0,11.3\n2025-10-13,14,18.2,0.3,0.0,8.6\n2025-10-13,15,16.2,1.3,0.0,15.3\n2025-10-13,16,15.3,2.3,0.0,18.7\n2025-10-13,17,14.7,1.8,0.0,18.2\n2025-10-13,18,14.5,1.2,0.0,17.1\n2025-10-13,19,14.5,0.6,0.0,17.1\n2025-10-13,20,14.6,0.2,0.0,17.5\n2025-10-13,21,14.0,0.3,0.0,15.4\n2025-10-13,22,14.2,0.1,0.0,15.2\n2025-10-13,23,14.3,0.0,0.0,13.2\n2025-10-14,0,14.5,0.0,0.0,11.8\n2025-10-14,1,14.6,0.0,0.0,12.2\n2025-10-14,2,14.7,0.1,0.0,12.1\n2025-10-14,3,14.5,0.1,0.0,11.5\n2025-10-14,4,14.7,0.1,0.0,11.0\n2025-10-14,5,14.8,0.2,0.0,12.0\n2025-10-14,6,14.8,0.4,0.0,12.8\n2025-10-14,7,14.8,0.2,0.0,13.9\n2025-10-14,8,15.1,0.1,0.0,14.1\n2025-10-14,9,15.2,0.2,0.0,11.7\n2025-10-14,10,16.0,0.0,0.0,10.6\n2025-10-14,11,17.3,0.0,0.0,12.6\n2025-10-14,12,17.8,0.0,0.0,12.1\n2025-10-14,13,18.9,0.0,0.0,11.4\n2025-10-14,14,18.8,0.0,0.0,10.2\n2025-10-14,15,19.8,0.0,0.0,11.6\n2025-10-14,16,19.7,0.0,0.0,7.9\n2025-10-14,17,19.4,0.0,0.0,5.9\n2025-10-14,18,19.1,0.0,0.0,9.0\n2025-10-14,19,18.9,0.0,0.0,11.5\n2025-10-14,20,18.3,0.0,0.0,11.9\n2025-10-14,21,18.0,0.1,0.0,12.3\n2025-10-14,22,17.7,0.2,0.0,12.9\n2025-10-14,23,17.7,0.1,0.0,12.3\n2025-10-15,0,17.5,0.0,0.0,11.3\n2025-10-15,1,17.1,0.0,0.0,9.5\n2025-10-15,2,16.8,0.0,0.0,8.9\n2025-10-15,3,16.4,0.1,0.0,11.6\n2025-10-15,4,16.4,0.0,0.0,11.8\n2025-10-15,5,16.2,0.0,0.0,11.6\n2025-10-15,6,16.2,0.0,0.0,12.1\n2025-10-15,7,16.3,0.0,0.0,13.0\n2025-10-15,8,17.4,0.0,0.0,13.1\n2025-10-15,9,19.0,0.0,0.0,10.4\n2025-10-15,10,20.5,0.0,0.0,11.9\n2025-10-15,11,22.1,0.0,0.0,11.1\n2025-10-15,12,23.4,0.0,0.0,11.0\n2025-10-15,13,24.2,0.0,0.0,12.6\n2025-10-15,14,24.5,0.0,0.0,12.4\n2025-10-15,15,24.0,0.0,0.0,7.2\n2025-10-15,16,23.0,0.0,0.0,4.3\n2025-10-15,17,22.9,0.0,0.0,2.7\n2025-10-15,18,20.7,1.3,0.0,2.4\n2025-10-15,19,19.4,3.5,0.0,5.0\n2025-10-15,20,18.7,3.6,0.0,8.3\n2025-10-15,21,18.6,2.0,0.0,8.9\n2025-10-15,22,18.5,1.0,0.0,6.4\n2025-10-15,23,18.2,1.6,0.0,7.1\n2025-10-16,0,18.0,2.6,0.0,10.0\n2025-10-16,1,17.8,0.8,0.0,9.0\n2025-10-16,2,17.6,0.8,0.0,8.7\n2025-10-16,3,17.5,1.3,0.0,11.5\n2025-10-16,4,17.4,0.5,0.0,9.2\n2025-10-16,5,17.1,0.1,0.0,10.3\n2025-10-16,6,16.9,0.0,0.0,10.7\n2025-10-16,7,16.7,0.0,0.0,10.6\n2025-10-16,8,17.0,0.0,0.0,9.0\n2025-10-16,9,17.8,0.0,0.0,7.7\n2025-10-16,10,19.7,0.0,0.0,9.6\n2025-10-16,11,21.3,0.0,0.0,7.9\n2025-10-16,12,22.4,0.0,0.0,8.4\n2025-10-16,13,23.1,0.0,0.0,9.4\n2025-10-16,14,23.5,0.0,0.0,10.9\n2025-10-16,15,23.4,0.0,0.0,10.5\n2025-10-16,16,23.4,0.0,0.0,7.8\n2025-10-16,17,22.9,0.0,0.0,4.2\n2025-10-16,18,21.1,0.0,0.0,7.1\n2025-10-16,19,20.4,0.0,0.0,11.0\n2025-10-16,20,19.6,0.0,0.0,15.5\n2025-10-16,21,19.0,0.0,0.0,14.5\n2025-10-16,22,18.4,0.0,0.0,14.0\n2025-10-16,23,18.1,0.0,0.0,11.8\n2025-10-17,0,17.6,0.0,0.0,10.8\n2025-10-17,1,17.2,0.0,0.0,11.0\n2025-10-17,2,16.7,0.0,0.0,9.5\n2025-10-17,3,16.4,0.0,0.0,11.9\n2025-10-17,4,16.0,0.0,0.0,10.5\n2025-10-17,5,15.8,0.0,0.0,11.1\n2025-10-17,6,15.6,0.0,0.0,11.1\n2025-10-17,7,15.5,0.0,0.0,11.4\n2025-10-17,8,16.8,0.0,0.0,12.2\n2025-10-17,9,18.4,0.0,0.0,10.8\n2025-10-17,10,20.0,0.0,0.0,13.3\n2025-10-17,11,21.4,0.0,0.0,14.6\n2025-10-17,12,22.3,0.0,0.0,15.8\n2025-10-17,13,23.1,0.0,0.0,13.8\n2025-10-17,14,23.9,0.0,0.0,11.4\n2025-10-17,15,24.2,0.0,0.0,9.4\n2025-10-17,16,22.5,0.0,0.0,4.5\n2025-10-17,17,20.7,0.0,0.0,10.2\n2025-10-17,18,19.4,0.0,0.0,10.2\n2025-10-17,19,18.9,0.5,0.0,9.6\n2025-10-17,20,18.6,0.4,0.0,6.4\n2025-10-17,21,18.5,1.1,0.0,8.0\n2025-10-17,22,18.0,1.4,0.0,11.0\n2025-10-17,23,17.6,2.1,0.0,10.6\n2025-10-18,0,17.5,0.7,0.0,10.1\n2025-10-18,1,17.3,1.1,0.0,10.3\n2025-10-18,2,17.2,1.5,0.0,12.0\n2025-10-18,3,16.8,2.8,0.0,15.3\n2025-10-18,4,16.1,3.6,0.0,18.2\n2025-10-18,5,14.8,2.4,0.0,19.3\n2025-10-18,6,14.1,1.3,0.0,17.4\n2025-10-18,7,14.0,0.2,0.0,15.7\n2025-10-18,8,14.0,0.0,0.0,16.0\n2025-10-18,9,14.1,0.0,0.0,15.6\n2025-10-18,10,15.0,0.0,0.0,14.9\n2025-10-18,11,15.0,0.0,0.0,13.7\n2025-10-18,12,16.6,0.0,0.0,12.6\n2025-10-18,13,16.5,0.0,0.0,11.4\n2025-10-18,14,17.0,0.0,0.0,13.1\n2025-10-18,15,16.6,0.0,0.0,8.9\n2025-10-18,16,16.6,0.0,0.0,8.9\n2025-10-18,17,15.9,0.0,0.0,9.0\n2025-10-18,18,14.8,0.0,0.0,7.2\n2025-10-18,19,14.3,0.0,0.0,7.5\n2025-10-18,20,13.7,0.0,0.0,8.7\n2025-10-18,21,12.9,0.0,0.0,10.3\n2025-10-18,22,12.6,0.0,0.0,9.5\n2025-10-18,23,12.1,0.0,0.0,8.6\n2025-10-19,0,12.0,0.0,0.0,7.1\n2025-10-19,1,12.1,0.0,0.0,6.7\n2025-10-19,2,12.3,0.0,0.0,5.3\n2025-10-19,3,13.6,0.0,0.0,6.5\n2025-10-19,4,14.0,0.0,0.0,5.8\n2025-10-19,5,13.2,0.0,0.0,3.7\n2025-10-19,6,13.1,0.0,0.0,1.7\n2025-10-19,7,12.7,0.0,0.0,1.2\n2025-10-19,8,14.0,0.0,0.0,2.7\n2025-10-19,9,15.3,0.0,0.0,2.8\n2025-10-19,10,17.2,0.0,0.0,3.4\n2025-10-19,11,18.0,0.3,0.0,8.0\n2025-10-19,12,17.4,0.0,0.0,12.0\n2025-10-19,13,17.3,0.0,0.0,12.0\n2025-10-19,14,18.0,0.0,0.0,12.6\n2025-10-19,15,17.5,0.0,0.0,14.0\n2025-10-19,16,17.2,0.0,0.0,11.2\n2025-10-19,17,16.4,0.0,0.0,13.6\n2025-10-19,18,14.6,0.0,0.0,17.2\n2025-10-19,19,12.3,0.0,0.0,18.6\n2025-10-19,20,10.7,0.0,0.0,19.6\n2025-10-19,21,10.0,0.0,0.0,16.0\n2025-10-19,22,9.2,0.0,0.0,15.6\n2025-10-19,23,8.3,0.0,0.0,14.6\n2025-10-20,0,7.6,0.0,0.0,14.4\n2025-10-20,1,6.8,0.0,0.0,15.5\n2025-10-20,2,6.1,0.0,0.0,16.9\n2025-10-20,3,5.2,0.0,0.0,15.4\n2025-10-20,4,4.7,0.0,0.0,13.5\n2025-10-20,5,4.4,0.0,0.0,12.5\n2025-10-20,6,4.4,0.0,0.0,12.7\n2025-10-20,7,4.5,0.0,0.0,13.4\n2025-10-20,8,5.4,0.0,0.0,12.3\n2025-10-20,9,6.8,0.0,0.0,11.3\n2025-10-20,10,8.5,0.0,0.0,10.1\n2025-10-20,11,10.9,0.0,0.0,11.0\n2025-10-20,12,12.9,0.0,0.0,11.0\n2025-10-20,13,14.0,0.0,0.0,11.2\n2025-10-20,14,14.4,0.0,0.0,11.5\n2025-10-20,15,14.2,0.0,0.0,11.3\n2025-10-20,16,13.6,0.0,0.0,13.0\n2025-10-20,17,12.6,0.0,0.0,11.8\n2025-10-20,18,10.9,0.0,0.0,13.3\n2025-10-20,19,9.6,0.0,0.0,15.5\n2025-10-20,20,8.6,0.0,0.0,15.0\n2025-10-20,21,7.9,0.0,0.0,14.5\n2025-10-20,22,7.4,0.0,0.0,14.2\n2025-10-20,23,6.8,0.0,0.0,13.1\n2025-10-21,0,6.2,0.0,0.0,12.8\n2025-10-21,1,5.6,0.0,0.0,12.6\n2025-10-21,2,5.1,0.0,0.0,11.5\n2025-10-21,3,4.7,0.0,0.0,11.5\n2025-10-21,4,4.4,0.0,0.0,12.0\n2025-10-21,5,3.9,0.0,0.0,12.1\n2025-10-21,6,3.6,0.0,0.0,12.5\n2025-10-21,7,3.6,0.0,0.0,11.6\n2025-10-21,8,4.6,0.0,0.0,8.8\n2025-10-21,9,5.6,0.0,0.0,9.9\n2025-10-21,10,7.7,0.0,0.0,9.8\n2025-10-21,11,9.4,0.0,0.0,9.5\n2025-10-21,12,11.4,0.0,0.0,9.7\n2025-10-21,13,12.6,0.0,0.0,10.1\n2025-10-21,14,13.3,0.0,0.0,9.2\n2025-10-21,15,13.3,0.0,0.0,8.0\n2025-10-21,16,12.8,0.0,0.0,8.3\n2025-10-21,17,12.3,0.0,0.0,6.9\n2025-10-21,18,10.7,0.0,0.0,11.0\n2025-10-21,19,9.3,0.0,0.0,13.3\n2025-10-21,20,8.2,0.0,0.0,12.9\n2025-10-21,21,7.6,0.0,0.0,11.0\n2025-10-21,22,6.7,0.0,0.0,10.5\n2025-10-21,23,6.4,0.0,0.0,10.4\n2025-10-22,0,6.1,0.0,0.0,10.3\n2025-10-22,1,5.8,0.0,0.0,10.3\n2025-10-22,2,5.6,0.0,0.0,9.9\n2025-10-22,3,5.6,0.0,0.0,9.5\n2025-10-22,4,5.4,0.0,0.0,8.9\n2025-10-22,5,5.3,0.0,0.0,9.1\n2025-10-22,6,5.4,0.0,0.0,9.1\n2025-10-22,7,5.5,0.0,0.0,8.9\n2025-10-22,8,6.8,0.0,0.0,9.0\n2025-10-22,9,9.6,0.0,0.0,6.8\n2025-10-22,10,12.0,0.0,0.0,7.6\n2025-10-22,11,14.0,0.0,0.0,8.1\n2025-10-22,12,15.5,0.0,0.0,8.5\n2025-10-22,13,16.7,0.0,0.0,7.9\n2025-10-22,14,17.2,0.0,0.0,6.7\n2025-10-22,15,17.6,0.0,0.0,5.6\n2025-10-22,16,17.4,0.0,0.0,5.5\n2025-10-22,17,16.6,0.0,0.0,3.3\n2025-10-22,18,15.3,0.0,0.0,7.5\n2025-10-22,19,14.2,0.0,0.0,9.9\n2025-10-22,20,13.7,0.0,0.0,6.9\n2025-10-22,21,13.3,0.0,0.0,8.2\n2025-10-22,22,13.2,0.0,0.0,8.3\n2025-10-22,23,13.6,0.0,0.0,9.0\n2025-10-23,0,13.0,0.1,0.0,9.3\n2025-10-23,1,12.4,0.0,0.0,9.3\n2025-10-23,2,12.0,0.1,0.0,8.8\n2025-10-23,3,11.5,0.2,0.0,9.5\n2025-10-23,4,11.3,0.1,0.0,9.3\n2025-10-23,5,11.5,0.1,0.0,8.6\n2025-10-23,6,11.7,0.1,0.0,8.1\n2025-10-23,7,11.7,0.1,0.0,8.1\n2025-10-23,8,12.1,0.1,0.0,7.6\n2025-10-23,9,13.3,0.0,0.0,6.3\n2025-10-23,10,14.9,0.0,0.0,6.3\n2025-10-23,11,16.5,0.0,0.0,6.9\n2025-10-23,12,17.7,0.1,0.0,4.5\n2025-10-23,13,17.9,0.1,0.0,4.4\n2025-10-23,14,18.4,0.1,0.0,5.9\n2025-10-23,15,18.7,0.0,0.0,5.2\n2025-10-23,16,18.6,0.0,0.0,7.3\n2025-10-23,17,17.4,0.0,0.0,8.0\n2025-10-23,18,15.7,0.0,0.0,9.3\n2025-10-23,19,14.8,0.0,0.0,9.7\n2025-10-23,20,13.8,0.0,0.0,10.1\n2025-10-23,21,13.3,0.0,0.0,11.8\n2025-10-23,22,12.6,0.0,0.0,9.0\n2025-10-23,23,12.5,0.0,0.0,9.2\n2025-10-24,0,12.3,0.0,0.0,10.8\n2025-10-24,1,11.8,0.0,0.0,11.1\n2025-10-24,2,11.4,0.0,0.0,10.8\n2025-10-24,3,11.1,0.0,0.0,9.8\n2025-10-24,4,10.8,0.0,0.0,8.5\n2025-10-24,5,10.7,0.0,0.0,8.8\n2025-10-24,6,11.0,0.0,0.0,9.1\n2025-10-24,7,11.0,0.0,0.0,9.3\n2025-10-24,8,12.1,0.0,0.0,10.3\n2025-10-24,9,13.3,0.0,0.0,6.2\n2025-10-24,10,15.1,0.0,0.0,7.8\n2025-10-24,11,16.8,0.0,0.0,7.6\n2025-10-24,12,18.1,0.0,0.0,5.6\n2025-10-24,13,18.7,0.0,0.0,4.4\n2025-10-24,14,19.4,0.0,0.0,3.9\n2025-10-24,15,19.5,0.0,0.0,3.8\n2025-10-24,16,19.4,0.0,0.0,2.9\n2025-10-24,17,18.0,0.0,0.0,5.6\n2025-10-24,18,16.0,0.0,0.0,10.4\n2025-10-24,19,15.2,0.0,0.0,9.0\n2025-10-24,20,14.4,0.0,0.0,8.1\n2025-10-24,21,13.8,0.0,0.0,5.5\n2025-10-24,22,13.4,0.0,0.0,3.4\n2025-10-24,23,13.0,0.0,0.0,3.3\n2025-10-25,0,12.4,0.0,0.0,3.5\n2025-10-25,1,12.4,0.0,0.0,4.2\n2025-10-25,2,11.8,0.0,0.0,4.1\n2025-10-25,3,12.4,0.0,0.0,5.4\n2025-10-25,4,11.7,0.0,0.0,3.9\n2025-10-25,5,11.7,0.0,0.0,0.4\n2025-10-25,6,10.9,0.0,0.0,3.7\n2025-10-25,7,10.6,0.0,0.0,8.2\n2025-10-25,8,10.8,0.0,0.0,8.4\n2025-10-25,9,12.5,0.0,0.0,6.2\n2025-10-25,10,14.2,0.0,0.0,8.5\n2025-10-25,11,15.5,0.0,0.0,10.2\n2025-10-25,12,16.5,0.0,0.0,10.8\n2025-10-25,13,16.9,0.0,0.0,10.8\n2025-10-25,14,17.1,0.0,0.0,11.1\n2025-10-25,15,17.0,0.0,0.0,9.8\n2025-10-25,16,16.5,0.0,0.0,11.2\n2025-10-25,17,15.3,0.0,0.0,9.2\n2025-10-25,18,13.5,0.0,0.0,7.4\n2025-10-25,19,12.7,0.0,0.0,7.9\n2025-10-25,20,12.6,0.0,0.0,9.4\n2025-10-25,21,12.4,0.0,0.0,10.9\n2025-10-25,22,12.1,0.0,0.0,10.1\n2025-10-25,23,11.8,0.0,0.0,9.3\n2025-10-26,0,11.5,0.0,0.0,9.2\n2025-10-26,1,11.1,0.0,0.0,9.3\n2025-10-26,2,10.8,0.0,0.0,8.4\n2025-10-26,3,10.6,0.0,0.0,6.7\n2025-10-26,4,10.3,0.0,0.0,4.9\n2025-10-26,5,10.3,0.0,0.0,1.9\n2025-10-26,6,10.1,0.0,0.0,2.1\n2025-10-26,7,10.1,0.0,0.0,1.3\n2025-10-26,8,10.6,0.0,0.0,2.2\n2025-10-26,9,12.0,0.0,0.0,2.2\n2025-10-26,10,12.6,0.0,0.0,0.6\n2025-10-26,11,14.1,0.0,0.0,2.4\n2025-10-26,12,14.3,0.0,0.0,5.1\n2025-10-26,13,14.3,0.0,0.0,7.0\n2025-10-26,14,15.1,0.1,0.0,9.2\n2025-10-26,15,14.8,0.1,0.0,11.1\n2025-10-26,16,14.6,0.1,0.0,11.6\n2025-10-26,17,14.1,0.0,0.0,10.4\n2025-10-26,18,13.2,0.0,0.0,8.5\n2025-10-26,19,13.4,0.0,0.0,7.4\n2025-10-26,20,13.4,0.0,0.0,8.9\n2025-10-26,21,12.7,0.0,0.0,12.5\n2025-10-26,22,12.0,2.3,0.0,15.5\n2025-10-26,23,10.9,0.0,0.0,15.2\n2025-10-27,0,10.1,0.0,0.0,14.4\n2025-10-27,1,8.9,0.0,0.0,19.6\n2025-10-27,2,7.6,0.0,0.0,18.5\n2025-10-27,3,6.2,0.0,0.0,18.1\n2025-10-27,4,5.0,0.0,0.0,17.5\n2025-10-27,5,4.1,0.0,0.0,15.6\n2025-10-27,6,3.4,0.0,0.0,14.2\n2025-10-27,7,2.8,0.0,0.0,12.7\n2025-10-27,8,3.5,0.0,0.0,11.3\n2025-10-27,9,5.1,0.0,0.0,11.5\n2025-10-27,10,6.8,0.0,0.0,11.7\n2025-10-27,11,8.1,0.0,0.0,11.8\n2025-10-27,12,9.2,0.0,0.0,14.0\n2025-10-27,13,10.0,0.0,0.0,16.0\n2025-10-27,14,10.0,0.0,0.0,19.0\n2025-10-27,15,9.9,0.0,0.0,19.6\n2025-10-27,16,10.2,0.0,0.0,17.4\n2025-10-27,17,9.6,0.0,0.0,12.7\n2025-10-27,18,8.4,0.0,0.0,14.2\n2025-10-27,19,7.9,0.0,0.0,14.3\n2025-10-27,20,7.5,0.0,0.0,14.1\n2025-10-27,21,6.9,0.0,0.0,10.4\n2025-10-27,22,6.7,0.0,0.0,11.7\n2025-10-27,23,5.9,0.0,0.0,12.8\n2025-10-28,0,4.8,0.0,0.0,7.9\n2025-10-28,1,4.5,0.0,0.0,8.7\n2025-10-28,2,3.9,0.0,0.0,9.4\n2025-10-28,3,3.5,0.0,0.0,8.5\n2025-10-28,4,2.8,0.0,0.0,7.6\n2025-10-28,5,2.3,0.0,0.0,6.2\n2025-10-28,6,1.9,0.0,0.0,5.4\n2025-10-28,7,1.5,0.0,0.0,4.7\n2025-10-28,8,2.7,0.0,0.0,5.7\n2025-10-28,9,5.2,0.0,0.0,4.8\n2025-10-28,10,7.6,0.0,0.0,4.0\n2025-10-28,11,9.1,0.0,0.0,2.5\n2025-10-28,12,10.2,0.0,0.0,2.4\n2025-10-28,13,11.0,0.0,0.0,2.9\n2025-10-28,14,11.5,0.0,0.0,3.8\n2025-10-28,15,11.9,0.0,0.0,7.2\n2025-10-28,16,11.6,0.0,0.0,7.6\n2025-10-28,17,10.6,0.0,0.0,4.9\n2025-10-28,18,9.1,0.0,0.0,3.5\n2025-10-28,19,9.0,0.0,0.0,0.2\n2025-10-28,20,8.3,0.0,0.0,3.5\n2025-10-28,21,7.3,0.0,0.0,4.9\n2025-10-28,22,7.0,0.0,0.0,5.4\n2025-10-28,23,7.0,0.0,0.0,6.7\n2025-10-29,0,6.7,0.0,0.0,8.7\n2025-10-29,1,6.7,0.0,0.0,9.1\n2025-10-29,2,6.5,0.0,0.0,9.3\n2025-10-29,3,5.7,0.0,0.0,8.5\n2025-10-29,4,5.4,0.0,0.0,9.1\n2025-10-29,5,5.4,0.0,0.0,10.2\n2025-10-29,6,5.5,0.0,0.0,10.8\n2025-10-29,7,5.4,0.0,0.0,10.5\n2025-10-29,8,6.4,0.0,0.0,10.8\n2025-10-29,9,8.6,0.0,0.0,10.2\n2025-10-29,10,10.9,0.0,0.0,10.2\n2025-10-29,11,12.8,0.0,0.0,10.3\n2025-10-29,12,14.1,0.0,0.0,9.2\n2025-10-29,13,14.9,0.0,0.0,8.6\n2025-10-29,14,15.4,0.0,0.0,7.4\n2025-10-29,15,15.7,0.0,0.0,4.8\n2025-10-29,16,15.3,0.0,0.0,4.8\n2025-10-29,17,13.9,0.0,0.0,3.6\n2025-10-29,18,12.0,0.0,0.0,3.2\n2025-10-29,19,11.2,0.0,0.0,2.5\n2025-10-29,20,10.5,0.0,0.0,1.9\n2025-10-29,21,9.8,0.0,0.0,3.3\n2025-10-29,22,9.5,0.0,0.0,5.1\n2025-10-29,23,9.1,0.0,0.0,5.8\n2025-10-30,0,8.7,0.0,0.0,5.9\n2025-10-30,1,8.6,0.0,0.0,7.1\n2025-10-30,2,8.3,0.0,0.0,6.5\n2025-10-30,3,7.5,0.0,0.0,7.3\n2025-10-30,4,7.4,0.0,0.0,8.0\n2025-10-30,5,7.1,0.0,0.0,8.9\n2025-10-30,6,7.1,0.0,0.0,10.9\n2025-10-30,7,6.8,0.0,0.0,9.6\n2025-10-30,8,7.7,0.0,0.0,8.5\n2025-10-30,9,9.9,0.0,0.0,9.0\n2025-10-30,10,12.5,0.0,0.0,8.3\n2025-10-30,11,14.2,0.0,0.0,7.2\n2025-10-30,12,15.4,0.0,0.0,3.9\n2025-10-30,13,16.2,0.0,0.0,3.3\n2025-10-30,14,16.5,0.0,0.0,4.6\n2025-10-30,15,16.4,0.0,0.0,6.1\n2025-10-30,16,15.9,0.0,0.0,6.2\n2025-10-30,17,14.7,0.0,0.0,4.7\n2025-10-30,18,13.1,0.0,0.0,1.6\n2025-10-30,19,12.9,0.0,0.0,0.0\n2025-10-30,20,12.4,0.0,0.0,0.5\n2025-10-30,21,11.9,0.0,0.0,1.7\n2025-10-30,22,11.5,0.0,0.0,2.5\n2025-10-30,23,11.2,0.0,0.0,2.9\n2025-10-31,0,11.1,0.0,0.0,1.4\n2025-10-31,1,11.0,0.0,0.0,1.0\n2025-10-31,2,10.2,0.0,0.0,3.7\n2025-10-31,3,10.5,0.0,0.0,7.0\n2025-10-31,4,10.7,0.0,0.0,6.6\n2025-10-31,5,10.7,0.0,0.0,8.2\n2025-10-31,6,10.9,0.0,0.0,7.9\n2025-10-31,7,10.9,0.0,0.0,8.4\n2025-10-31,8,11.4,0.0,0.0,8.0\n2025-10-31,9,11.9,0.0,0.0,8.5\n2025-10-31,10,13.9,0.0,0.0,7.0\n2025-10-31,11,16.2,0.0,0.0,5.2\n2025-10-31,12,17.1,0.0,0.0,3.7\n2025-10-31,13,18.1,0.0,0.0,4.1\n2025-10-31,14,18.7,0.0,0.0,3.2\n2025-10-31,15,18.8,0.0,0.0,3.3\n2025-10-31,16,18.0,0.0,0.0,4.6\n2025-10-31,17,16.3,0.0,0.0,9.7\n2025-10-31,18,14.6,0.0,0.0,11.4\n2025-10-31,19,14.1,0.0,0.0,11.5\n2025-10-31,20,13.5,0.0,0.0,10.8\n2025-10-31,21,12.7,0.0,0.0,12.7\n2025-10-31,22,12.7,0.1,0.0,12.3\n2025-10-31,23,12.3,0.4,0.0,12.4\n2025-11-01,0,11.8,0.2,0.0,6.8\n2025-11-01,1,11.7,0.1,0.0,1.8\n2025-11-01,2,11.2,0.0,0.0,4.4\n2025-11-01,3,11.2,0.0,0.0,13.3\n2025-11-01,4,10.7,0.0,0.0,13.4\n2025-11-01,5,11.0,0.0,0.0,13.0\n2025-11-01,6,10.0,0.0,0.0,10.4\n2025-11-01,7,9.2,0.0,0.0,6.9\n2025-11-01,8,10.0,0.0,0.0,8.4\n2025-11-01,9,11.5,0.0,0.0,7.7\n2025-11-01,10,13.3,0.0,0.0,12.1\n2025-11-01,11,14.2,0.0,0.0,14.8\n2025-11-01,12,14.8,0.0,0.0,15.0\n2025-11-01,13,15.2,0.0,0.0,15.6\n2025-11-01,14,15.2,0.0,0.0,16.4\n2025-11-01,15,15.2,0.0,0.0,15.7\n2025-11-01,16,14.7,0.0,0.0,16.4\n2025-11-01,17,12.2,0.2,0.0,13.0\n2025-11-01,18,11.7,0.0,0.0,2.3\n2025-11-01,19,11.8,0.0,0.0,8.1\n2025-11-01,20,12.8,0.0,0.0,17.1\n2025-11-01,21,13.0,0.0,0.0,13.3\n2025-11-01,22,11.8,0.0,0.0,10.1\n2025-11-01,23,12.2,0.0,0.0,15.3\n2025-11-02,0,10.9,0.0,0.0,11.9\n2025-11-02,1,9.9,0.0,0.0,8.7\n2025-11-02,2,10.2,0.0,0.0,9.1\n2025-11-02,3,10.0,0.0,0.0,10.4\n2025-11-02,4,8.5,0.0,0.0,8.1\n2025-11-02,5,7.7,0.0,0.0,10.0\n2025-11-02,6,7.6,0.0,0.0,12.5\n2025-11-02,7,7.4,0.0,0.0,13.9\n2025-11-02,8,7.8,0.0,0.0,15.3\n2025-11-02,9,9.1,0.0,0.0,15.7\n2025-11-02,10,9.6,0.0,0.0,20.5\n2025-11-02,11,10.1,0.0,0.0,22.5\n2025-11-02,12,10.4,0.0,0.0,21.6\n2025-11-02,13,10.4,0.0,0.0,21.7\n2025-11-02,14,10.3,0.0,0.0,21.4\n2025-11-02,15,9.8,0.0,0.0,21.6\n2025-11-02,16,9.4,0.0,0.0,20.4\n2025-11-02,17,8.4,0.0,0.0,17.3\n2025-11-02,18,7.0,0.0,0.0,19.0\n2025-11-02,19,5.6,0.0,0.0,19.6\n2025-11-02,20,5.2,0.0,0.0,19.1\n2025-11-02,21,4.8,0.0,0.0,17.6\n2025-11-02,22,3.8,0.0,0.0,16.7\n2025-11-02,23,3.1,0.0,0.0,14.0\n2025-11-03,0,2.4,0.0,0.0,12.9\n2025-11-03,1,1.8,0.0,0.0,12.5\n2025-11-03,2,1.2,0.0,0.0,10.8\n2025-11-03,3,0.5,0.0,0.0,11.1\n2025-11-03,4,-0.1,0.0,0.0,10.2\n2025-11-03,5,-0.2,0.0,0.0,10.3\n2025-11-03,6,-0.2,0.0,0.0,9.3\n2025-11-03,7,-0.1,0.0,0.0,7.8\n2025-11-03,8,0.8,0.0,0.0,6.7\n2025-11-03,9,2.5,0.0,0.0,2.3\n2025-11-03,10,5.0,0.0,0.0,1.3\n2025-11-03,11,7.2,0.0,0.0,1.1\n2025-11-03,12,8.9,0.0,0.0,2.7\n2025-11-03,13,10.0,0.0,0.0,2.2\n2025-11-03,14,10.6,0.0,0.0,2.7\n2025-11-03,15,11.4,0.0,0.0,3.6\n2025-11-03,16,11.2,0.0,0.0,4.3\n2025-11-03,17,10.1,0.0,0.0,2.3\n2025-11-03,18,8.8,0.0,0.0,1.0\n2025-11-03,19,8.1,0.0,0.0,1.2\n2025-11-03,20,7.0,0.0,0.0,3.6\n2025-11-03,21,5.8,0.0,0.0,5.2\n2025-11-03,22,5.3,0.0,0.0,4.3\n2025-11-03,23,5.0,0.0,0.0,3.6\n2025-11-04,0,4.9,0.0,0.0,3.1\n2025-11-04,1,4.7,0.0,0.0,3.4\n2025-11-04,2,4.5,0.0,0.0,4.3\n2025-11-04,3,4.4,0.0,0.0,3.1\n2025-11-04,4,4.1,0.0,0.0,3.8\n2025-11-04,5,3.5,0.0,0.0,4.0\n2025-11-04,6,3.3,0.0,0.0,3.3\n2025-11-04,7,3.3,0.0,0.0,3.6\n2025-11-04,8,4.8,0.0,0.0,4.0\n2025-11-04,9,6.8,0.0,0.0,4.2\n2025-11-04,10,9.4,0.0,0.0,2.8\n2025-11-04,11,11.4,0.0,0.0,0.4\n2025-11-04,12,12.5,0.0,0.0,3.7\n2025-11-04,13,13.1,0.0,0.0,4.0\n2025-11-04,14,13.5,0.0,0.0,4.0\n2025-11-04,15,13.7,0.0,0.0,5.5\n2025-11-04,16,13.4,0.0,0.0,6.3\n2025-11-04,17,12.0,0.0,0.0,5.2\n2025-11-04,18,10.5,0.0,0.0,6.3\n2025-11-04,19,9.7,0.0,0.0,7.0\n2025-11-04,20,8.7,0.0,0.0,6.2\n2025-11-04,21,8.0,0.0,0.0,5.0\n2025-11-04,22,7.7,0.0,0.0,4.8\n2025-11-04,23,7.5,0.0,0.0,6.6\n2025-11-05,0,7.2,0.0,0.0,7.8\n2025-11-05,1,7.0,0.0,0.0,8.0\n2025-11-05,2,6.8,0.0,0.0,7.0\n2025-11-05,3,6.1,0.0,0.0,7.5\n2025-11-05,4,6.2,0.0,0.0,6.6\n2025-11-05,5,6.1,0.0,0.0,6.6\n2025-11-05,6,5.8,0.0,0.0,5.9\n2025-11-05,7,5.2,0.0,0.0,5.0\n2025-11-05,8,6.2,0.0,0.0,6.2\n2025-11-05,9,8.3,0.0,0.0,6.5\n2025-11-05,10,10.9,0.0,0.0,5.5\n2025-11-05,11,13.1,0.0,0.0,7.6\n2025-11-05,12,14.3,0.0,0.0,7.8\n2025-11-05,13,15.1,0.0,0.0,7.9\n2025-11-05,14,15.6,0.0,0.0,8.1\n2025-11-05,15,16.1,0.0,0.0,7.2\n2025-11-05,16,15.6,0.0,0.0,7.1\n2025-11-05,17,14.0,0.0,0.0,6.4\n2025-11-05,18,12.5,0.0,0.0,6.7\n2025-11-05,19,12.0,0.0,0.0,6.7\n2025-11-05,20,11.0,0.0,0.0,6.2\n2025-11-05,21,9.9,0.0,0.0,5.1\n2025-11-05,22,9.5,0.0,0.0,4.8\n2025-11-05,23,9.0,0.0,0.0,5.3\n2025-11-06,0,8.5,0.0,0.0,5.6\n2025-11-06,1,8.0,0.0,0.0,6.3\n2025-11-06,2,7.7,0.0,0.0,6.6\n2025-11-06,3,6.8,0.0,0.0,6.4\n2025-11-06,4,6.7,0.0,0.0,6.8\n2025-11-06,5,6.7,0.0,0.0,7.5\n2025-11-06,6,6.5,0.0,0.0,7.2\n2025-11-06,7,6.1,0.0,0.0,5.9\n2025-11-06,8,6.7,0.0,0.0,8.0\n2025-11-06,9,8.2,0.0,0.0,6.9\n2025-11-06,10,11.0,0.0,0.0,4.8\n2025-11-06,11,13.2,0.0,0.0,4.1\n2025-11-06,12,14.9,0.0,0.0,5.5\n2025-11-06,13,16.0,0.0,0.0,7.4\n2025-11-06,14,16.8,0.0,0.0,9.8\n2025-11-06,15,17.2,0.0,0.0,10.0\n2025-11-06,16,16.7,0.0,0.0,9.3\n2025-11-06,17,15.4,0.0,0.0,7.2\n2025-11-06,18,13.6,0.0,0.0,6.8\n2025-11-06,19,12.9,0.0,0.0,6.0\n2025-11-06,20,12.5,0.0,0.0,6.2\n2025-11-06,21,11.7,0.0,0.0,7.2\n2025-11-06,22,11.0,0.0,0.0,8.1\n2025-11-06,23,10.3,0.0,0.0,8.9\n2025-11-07,0,9.8,0.0,0.0,8.6\n2025-11-07,1,9.1,0.0,0.0,5.1\n2025-11-07,2,8.8,0.0,0.0,5.2\n2025-11-07,3,7.7,0.0,0.0,4.9\n2025-11-07,4,7.7,0.0,0.0,5.5\n2025-11-07,5,7.6,0.0,0.0,5.8\n2025-11-07,6,7.4,0.0,0.0,5.4\n2025-11-07,7,7.5,0.0,0.0,5.4\n2025-11-07,8,7.8,0.0,0.0,4.8\n2025-11-07,9,10.5,0.0,0.0,6.4\n2025-11-07,10,13.2,0.0,0.0,4.3\n2025-11-07,11,15.4,0.0,0.0,4.9\n2025-11-07,12,16.7,0.0,0.0,2.2\n2025-11-07,13,17.6,0.0,0.0,1.7\n2025-11-07,14,17.9,0.0,0.0,4.6\n2025-11-07,15,18.4,0.0,0.0,7.0\n2025-11-07,16,18.0,0.0,0.0,8.0\n2025-11-07,17,16.3,0.0,0.0,6.8\n2025-11-07,18,14.8,0.0,0.0,7.3\n2025-11-07,19,14.0,0.0,0.0,8.4\n2025-11-07,20,13.4,0.0,0.0,10.3\n2025-11-07,21,13.5,0.0,0.0,14.7\n2025-11-07,22,12.9,0.0,0.0,14.7\n2025-11-07,23,12.5,0.0,0.0,12.5\n2025-11-08,0,12.2,0.0,0.0,13.5\n2025-11-08,1,12.2,0.0,0.0,14.8\n2025-11-08,2,12.1,0.0,0.0,13.7\n2025-11-08,3,11.4,0.0,0.0,11.1\n2025-11-08,4,11.3,0.0,0.0,10.3\n2025-11-08,5,11.7,0.0,0.0,11.2\n2025-11-08,6,12.2,0.0,0.0,12.0\n2025-11-08,7,12.3,0.0,0.0,10.4\n2025-11-08,8,12.3,0.1,0.0,10.7\n2025-11-08,9,13.3,0.0,0.0,10.2\n2025-11-08,10,13.9,0.0,0.0,8.5\n2025-11-08,11,15.1,0.0,0.0,7.6\n2025-11-08,12,14.7,0.0,0.0,6.5\n2025-11-08,13,14.9,0.0,0.0,7.2\n2025-11-08,14,15.9,0.0,0.0,5.4\n2025-11-08,15,16.3,0.0,0.0,3.8\n2025-11-08,16,15.8,0.0,0.0,5.5\n2025-11-08,17,14.9,0.0,0.0,5.7\n2025-11-08,18,14.3,0.0,0.0,5.1\n2025-11-08,19,13.8,0.0,0.0,4.9\n2025-11-08,20,13.0,0.0,0.0,4.3\n2025-11-08,21,12.9,0.0,0.0,5.6\n2025-11-08,22,12.8,0.0,0.0,6.6\n2025-11-08,23,12.2,0.0,0.0,6.5\n2025-11-09,0,11.8,0.0,0.0,5.1\n2025-11-09,1,11.2,0.0,0.0,3.8\n2025-11-09,2,10.3,0.0,0.0,1.3\n2025-11-09,3,11.4,0.0,0.0,8.4\n2025-11-09,4,11.1,0.1,0.0,10.0\n2025-11-09,5,10.0,0.0,0.0,8.4\n2025-11-09,6,9.3,0.0,0.0,6.9\n2025-11-09,7,8.0,0.0,0.0,4.4\n2025-11-09,8,9.1,0.0,0.0,6.5\n2025-11-09,9,11.2,0.0,0.0,9.0\n2025-11-09,10,12.7,0.0,0.0,9.1\n2025-11-09,11,14.2,0.0,0.0,9.1\n2025-11-09,12,15.4,0.0,0.0,9.7\n2025-11-09,13,15.9,0.0,0.0,12.6\n2025-11-09,14,15.4,0.0,0.0,18.2\n2025-11-09,15,15.1,0.0,0.0,17.4\n2025-11-09,16,14.6,0.0,0.0,18.9\n2025-11-09,17,13.9,0.0,0.0,19.1\n2025-11-09,18,13.1,0.0,0.0,19.0\n2025-11-09,19,12.9,0.0,0.0,19.4\n2025-11-09,20,12.7,0.0,0.0,20.0\n2025-11-09,21,12.5,0.0,0.0,21.7\n2025-11-09,22,11.7,0.0,0.0,18.6\n2025-11-09,23,10.9,0.0,0.0,17.5\n2025-11-10,0,10.4,0.0,0.0,18.3\n2025-11-10,1,9.1,0.0,0.0,17.9\n2025-11-10,2,8.0,0.0,0.0,14.7\n2025-11-10,3,7.7,0.0,0.0,15.0\n2025-11-10,4,7.2,0.0,0.0,13.7\n2025-11-10,5,6.6,0.0,0.0,12.0\n2025-11-10,6,6.1,0.0,0.0,12.7\n2025-11-10,7,5.6,0.0,0.0,14.5\n2025-11-10,8,5.7,0.0,0.0,15.0\n2025-11-10,9,7.4,0.0,0.0,13.4\n2025-11-10,10,8.2,0.0,0.0,15.9\n2025-11-10,11,8.8,0.0,0.0,14.7\n2025-11-10,12,9.4,0.0,0.0,14.1\n2025-11-10,13,9.9,0.0,0.0,14.2\n2025-11-10,14,10.1,0.0,0.0,13.9\n2025-11-10,15,10.4,0.0,0.0,11.5\n2025-11-10,16,10.0,0.0,0.0,11.5\n2025-11-10,17,8.9,0.0,0.0,11.3\n2025-11-10,18,8.0,0.0,0.0,10.7\n2025-11-10,19,7.6,0.0,0.0,9.3\n2025-11-10,20,6.9,0.0,0.0,10.8\n2025-11-10,21,6.3,0.0,0.0,11.3\n2025-11-10,22,5.5,0.0,0.0,12.1\n2025-11-10,23,4.8,0.0,0.0,10.3\n2025-11-11,0,4.3,0.0,0.0,7.4\n2025-11-11,1,3.8,0.0,0.0,5.3\n2025-11-11,2,3.9,0.0,0.0,7.3\n2025-11-11,3,3.1,0.0,0.0,6.9\n2025-11-11,4,2.7,0.0,0.0,5.4\n2025-11-11,5,3.4,0.0,0.0,5.2\n2025-11-11,6,3.6,0.0,0.0,4.5\n2025-11-11,7,2.8,0.0,0.0,4.1\n2025-11-11,8,4.0,0.0,0.0,4.4\n2025-11-11,9,5.8,0.0,0.0,4.7\n2025-11-11,10,8.3,0.0,0.0,3.3\n2025-11-11,11,10.1,0.0,0.0,1.6\n2025-11-11,12,11.6,0.0,0.0,2.2\n2025-11-11,13,12.6,0.0,0.0,4.3\n2025-11-11,14,13.2,0.0,0.0,6.9\n2025-11-11,15,13.2,0.0,0.0,8.4\n2025-11-11,16,12.7,0.0,0.0,9.2\n2025-11-11,17,11.7,0.0,0.0,5.4\n2025-11-11,18,10.0,0.0,0.0,2.8\n2025-11-11,19,9.8,0.0,0.0,0.7\n2025-11-11,20,8.8,0.0,0.0,1.7\n2025-11-11,21,7.7,0.0,0.0,3.3\n2025-11-11,22,7.5,0.0,0.0,4.0\n2025-11-11,23,7.5,0.0,0.0,5.4\n2025-11-12,0,7.5,0.0,0.0,5.8\n2025-11-12,1,7.3,0.0,0.0,7.9\n2025-11-12,2,6.7,0.0,0.0,7.3\n2025-11-12,3,6.1,0.0,0.0,7.3\n2025-11-12,4,5.7,0.0,0.0,6.4\n2025-11-12,5,5.3,0.0,0.0,5.9\n2025-11-12,6,4.5,0.0,0.0,4.5\n2025-11-12,7,4.5,0.0,0.0,5.3\n2025-11-12,8,5.2,0.0,0.0,7.1\n2025-11-12,9,7.7,0.0,0.0,6.4\n2025-11-12,10,10.4,0.0,0.0,3.9\n2025-11-12,11,12.5,0.0,0.0,3.6\n2025-11-12,12,13.8,0.0,0.0,2.2\n2025-11-12,13,14.6,0.0,0.0,2.9\n2025-11-12,14,15.0,0.0,0.0,5.9\n2025-11-12,15,14.8,0.0,0.0,8.7\n2025-11-12,16,14.1,0.0,0.0,9.4\n2025-11-12,17,12.7,0.0,0.0,6.6\n2025-11-12,18,11.4,0.0,0.0,6.0\n2025-11-12,19,10.2,0.0,0.0,4.9\n2025-11-12,20,9.7,0.0,0.0,4.8\n2025-11-12,21,9.0,0.0,0.0,1.0\n2025-11-12,22,8.8,0.0,0.0,0.6\n2025-11-12,23,7.9,0.0,0.0,1.7\n2025-11-13,0,7.5,0.0,0.0,1.1\n2025-11-13,1,6.8,0.0,0.0,2.4\n2025-11-13,2,6.5,0.0,0.0,6.1\n2025-11-13,3,5.3,0.0,0.0,7.7\n2025-11-13,4,5.0,0.0,0.0,10.7\n2025-11-13,5,5.1,0.0,0.0,10.0\n2025-11-13,6,5.3,0.0,0.0,10.6\n2025-11-13,7,5.7,0.0,0.0,9.6\n2025-11-13,8,6.7,0.0,0.0,10.1\n2025-11-13,9,9.3,0.0,0.0,10.3\n2025-11-13,10,12.0,0.0,0.0,7.0\n2025-11-13,11,14.2,0.0,0.0,5.2\n2025-11-13,12,15.2,0.0,0.0,7.3\n2025-11-13,13,16.0,0.0,0.0,10.3\n2025-11-13,14,16.2,0.0,0.0,12.3\n2025-11-13,15,16.0,0.0,0.0,13.3\n2025-11-13,16,15.0,0.0,0.0,13.2\n2025-11-13,17,13.9,0.0,0.0,11.4\n2025-11-13,18,12.6,0.0,0.0,10.8\n2025-11-13,19,11.7,0.0,0.0,12.8\n2025-11-13,20,10.5,0.0,0.0,14.7\n2025-11-13,21,9.4,0.0,0.0,13.0\n2025-11-13,22,8.4,0.0,0.0,11.0\n2025-11-13,23,7.5,0.0,0.0,10.9\n2025-11-14,0,6.8,0.0,0.0,9.3\n2025-11-14,1,6.2,0.0,0.0,8.5\n2025-11-14,2,5.7,0.0,0.0,8.3\n2025-11-14,3,4.7,0.0,0.0,7.5\n2025-11-14,4,4.3,0.0,0.0,7.2\n2025-11-14,5,3.8,0.0,0.0,6.4\n2025-11-14,6,3.1,0.0,0.0,4.5\n2025-11-14,7,2.6,0.0,0.0,3.5\n2025-11-14,8,3.6,0.0,0.0,2.7\n2025-11-14,9,6.2,0.0,0.0,0.8\n2025-11-14,10,8.9,0.0,0.0,3.0\n2025-11-14,11,10.9,0.0,0.0,6.3\n2025-11-14,12,12.1,0.0,0.0,9.9\n2025-11-14,13,12.8,0.0,0.0,11.6\n2025-11-14,14,13.0,0.0,0.0,12.0\n2025-11-14,15,12.9,0.0,0.0,13.0\n2025-11-14,16,12.5,0.0,0.0,12.0\n2025-11-14,17,11.4,0.0,0.0,9.1\n2025-11-14,18,10.3,0.0,0.0,7.8\n2025-11-14,19,9.6,0.0,0.0,6.3\n2025-11-14,20,8.7,0.0,0.0,7.9\n2025-11-14,21,7.7,0.0,0.0,5.9\n2025-11-14,22,7.5,0.0,0.0,9.1\n2025-11-14,23,6.2,0.0,0.0,7.9\n2025-11-15,0,5.5,0.0,0.0,6.3\n2025-11-15,1,4.9,0.0,0.0,6.5\n2025-11-15,2,4.5,0.0,0.0,5.9\n2025-11-15,3,3.0,0.0,0.0,4.4\n2025-11-15,4,2.5,0.0,0.0,3.8\n2025-11-15,5,2.5,0.0,0.0,1.3\n2025-11-15,6,2.2,0.0,0.0,2.8\n2025-11-15,7,2.5,0.0,0.0,4.7\n2025-11-15,8,3.4,0.0,0.0,6.2\n2025-11-15,9,6.1,0.0,0.0,6.4\n2025-11-15,10,8.8,0.0,0.0,4.0\n2025-11-15,11,11.1,0.0,0.0,3.1\n2025-11-15,12,12.8,0.0,0.0,1.7\n2025-11-15,13,14.0,0.0,0.0,1.7\n2025-11-15,14,14.7,0.0,0.0,0.9\n2025-11-15,15,14.9,0.0,0.0,2.0\n2025-11-15,16,14.7,0.0,0.0,2.5\n2025-11-15,17,13.0,0.0,0.0,2.4\n2025-11-15,18,11.7,0.0,0.0,1.8\n2025-11-15,19,10.1,0.0,0.0,0.8\n2025-11-15,20,9.2,0.0,0.0,0.9\n2025-11-15,21,8.8,0.0,0.0,1.1\n2025-11-15,22,7.9,0.0,0.0,2.8\n2025-11-15,23,7.1,0.0,0.0,4.6\n2025-11-16,0,7.4,0.0,0.0,5.4\n2025-11-16,1,7.3,0.0,0.0,6.5\n2025-11-16,2,6.7,0.0,0.0,7.5\n2025-11-16,3,6.5,0.0,0.0,7.8\n2025-11-16,4,6.2,0.0,0.0,7.3\n2025-11-16,5,6.0,0.0,0.0,8.9\n2025-11-16,6,5.8,0.0,0.0,9.4\n2025-11-16,7,5.5,0.0,0.0,9.0\n2025-11-16,8,6.3,0.0,0.0,10.2\n2025-11-16,9,7.8,0.0,0.0,10.3\n2025-11-16,10,10.3,0.0,0.0,8.9\n2025-11-16,11,12.9,0.0,0.0,10.8\n2025-11-16,12,14.6,0.0,0.0,10.8\n2025-11-16,13,15.6,0.0,0.0,11.1\n2025-11-16,14,15.9,0.0,0.0,13.3\n2025-11-16,15,15.9,0.0,0.0,16.0\n2025-11-16,16,15.3,0.0,0.0,16.8\n2025-11-16,17,14.8,0.0,0.0,13.5\n2025-11-16,18,13.7,0.0,0.0,12.4\n2025-11-16,19,13.3,0.3,0.0,14.4\n2025-11-16,20,13.4,0.6,0.0,14.8\n2025-11-16,21,13.0,0.0,0.0,13.1\n2025-11-16,22,12.0,0.0,0.0,20.2\n2025-11-16,23,10.7,0.0,0.0,17.4\n2025-11-17,0,9.7,0.0,0.0,18.8\n2025-11-17,1,8.1,0.0,0.0,20.8\n2025-11-17,2,6.5,0.0,0.0,21.7\n2025-11-17,3,6.0,0.0,0.0,23.2\n2025-11-17,4,5.0,0.0,0.0,19.8\n2025-11-17,5,4.2,0.0,0.0,18.1\n2025-11-17,6,3.7,0.0,0.0,17.7\n2025-11-17,7,3.1,0.0,0.0,20.0\n2025-11-17,8,2.5,0.0,0.0,20.8\n2025-11-17,9,2.9,0.0,0.0,18.9\n2025-11-17,10,3.2,0.0,0.0,18.4\n2025-11-17,11,3.9,0.0,0.0,18.5\n2025-11-17,12,3.9,0.0,0.0,18.1\n2025-11-17,13,4.5,0.0,0.0,17.7\n2025-11-17,14,4.9,0.0,0.0,18.3\n2025-11-17,15,5.0,0.0,0.0,18.3\n2025-11-17,16,4.5,0.0,0.0,18.4\n2025-11-17,17,3.5,0.0,0.0,16.6\n2025-11-17,18,2.6,0.0,0.0,18.1\n2025-11-17,19,2.0,0.0,0.0,18.4\n2025-11-17,20,1.4,0.0,0.0,17.5\n2025-11-17,21,0.9,0.0,0.0,17.1\n2025-11-17,22,0.6,0.0,0.0,16.2\n2025-11-17,23,0.1,0.0,0.0,14.5\n2025-11-18,0,-0.4,0.0,0.0,14.3\n2025-11-18,1,-0.7,0.0,0.0,14.1\n2025-11-18,2,-1.2,0.0,0.0,13.9\n2025-11-18,3,-1.5,0.0,0.0,12.4\n2025-11-18,4,-1.8,0.0,0.0,12.3\n2025-11-18,5,-2.1,0.0,0.0,10.7\n2025-11-18,6,-2.4,0.0,0.0,11.4\n2025-11-18,7,-2.6,0.0,0.0,10.8\n2025-11-18,8,-2.2,0.0,0.0,8.1\n2025-11-18,9,-0.6,0.0,0.0,7.4\n2025-11-18,10,1.1,0.0,0.0,9.6\n2025-11-18,11,2.5,0.0,0.0,11.1\n2025-11-18,12,3.6,0.0,0.0,10.9\n2025-11-18,13,4.4,0.0,0.0,11.6\n2025-11-18,14,5.0,0.0,0.0,10.6\n2025-11-18,15,4.6,0.0,0.0,9.2\n2025-11-18,16,4.3,0.0,0.0,9.2\n2025-11-18,17,3.9,0.0,0.0,9.2\n2025-11-18,18,3.5,0.0,0.0,9.4\n2025-11-18,19,2.6,0.0,0.0,8.8\n2025-11-18,20,2.5,0.0,0.0,11.9\n2025-11-18,21,3.8,0.0,0.0,12.5\n2025-11-18,22,3.0,0.0,0.0,13.7\n2025-11-18,23,2.3,0.0,0.0,10.8\n2025-11-19,0,1.2,0.0,0.0,11.0\n2025-11-19,1,0.6,0.0,0.0,9.3\n2025-11-19,2,0.0,0.0,0.0,5.8\n2025-11-19,3,-1.2,0.0,0.0,4.6\n2025-11-19,4,-1.5,0.0,0.0,3.7\n2025-11-19,5,-1.8,0.0,0.0,3.8\n2025-11-19,6,-2.1,0.0,0.0,3.3\n2025-11-19,7,-2.0,0.0,0.0,2.1\n2025-11-19,8,-1.1,0.0,0.0,2.3\n2025-11-19,9,1.1,0.0,0.0,2.0\n2025-11-19,10,2.8,0.0,0.0,0.8\n2025-11-19,11,4.3,0.0,0.0,2.8\n2025-11-19,12,5.4,0.0,0.0,5.1\n2025-11-19,13,6.1,0.0,0.0,7.4\n2025-11-19,14,6.9,0.0,0.0,10.2\n2025-11-19,15,6.4,0.0,0.0,8.5\n2025-11-19,16,6.2,0.0,0.0,10.1\n2025-11-19,17,5.2,0.0,0.0,9.1\n2025-11-19,18,4.1,0.0,0.0,8.7\n2025-11-19,19,3.5,0.0,0.0,6.5\n2025-11-19,20,2.7,0.0,0.0,3.7\n2025-11-19,21,2.2,0.0,0.0,5.4\n2025-11-19,22,2.2,0.0,0.0,4.1\n2025-11-19,23,1.6,0.0,0.0,2.8\n2025-11-20,0,1.7,0.0,0.0,5.3\n2025-11-20,1,1.4,0.0,0.0,10.6\n2025-11-20,2,2.5,0.0,0.0,10.1\n2025-11-20,3,2.6,0.0,0.0,8.3\n2025-11-20,4,2.7,0.0,0.0,9.5\n2025-11-20,5,3.1,0.0,0.0,11.9\n2025-11-20,6,2.8,0.0,0.0,13.1\n2025-11-20,7,3.4,0.0,0.0,14.2\n2025-11-20,8,4.2,0.0,0.0,13.9\n2025-11-20,9,4.8,0.0,0.0,15.0\n2025-11-20,10,8.1,0.0,0.0,9.1\n2025-11-20,11,11.4,0.0,0.0,6.5\n2025-11-20,12,12.9,0.0,0.0,21.4\n2025-11-20,13,13.4,0.0,0.0,24.5\n2025-11-20,14,13.2,0.0,0.0,25.7\n2025-11-20,15,12.6,0.0,0.0,21.8\n2025-11-20,16,12.1,0.0,0.0,22.9\n2025-11-20,17,10.9,0.0,0.0,18.4\n2025-11-20,18,9.4,0.0,0.0,16.4\n2025-11-20,19,8.3,0.0,0.0,16.4\n2025-11-20,20,6.9,0.0,0.0,16.6\n2025-11-20,21,6.4,0.0,0.0,16.4\n2025-11-20,22,5.6,0.0,0.0,15.5\n2025-11-20,23,4.9,0.0,0.0,14.2\n2025-11-21,0,4.2,0.0,0.0,11.6\n2025-11-21,1,3.3,0.0,0.0,9.5\n2025-11-21,2,2.6,0.0,0.0,8.7\n2025-11-21,3,2.1,0.0,0.0,8.1\n2025-11-21,4,1.7,0.0,0.0,7.8\n2025-11-21,5,1.2,0.0,0.0,7.4\n2025-11-21,6,0.9,0.0,0.0,4.8\n2025-11-21,7,0.5,0.0,0.0,1.9\n2025-11-21,8,1.1,0.0,0.0,1.9\n2025-11-21,9,3.0,0.0,0.0,3.2\n2025-11-21,10,5.5,0.0,0.0,3.1\n2025-11-21,11,7.6,0.0,0.0,4.9\n2025-11-21,12,9.1,0.0,0.0,9.8\n2025-11-21,13,10.2,0.0,0.0,12.7\n2025-11-21,14,10.9,0.0,0.0,19.6\n2025-11-21,15,10.7,0.0,0.0,19.0\n2025-11-21,16,10.9,0.0,0.0,20.9\n2025-11-21,17,10.7,0.0,0.0,14.8\n2025-11-21,18,10.0,0.0,0.0,14.0\n2025-11-21,19,9.6,0.0,0.0,13.9\n2025-11-21,20,9.5,0.0,0.0,9.1\n2025-11-21,21,10.5,0.0,0.0,11.4\n2025-11-21,22,10.5,0.0,0.0,11.5\n2025-11-21,23,10.0,0.0,0.0,9.9\n2025-11-22,0,9.6,0.0,0.0,9.7\n2025-11-22,1,9.2,0.0,0.0,7.4\n2025-11-22,2,8.4,0.0,0.0,5.9\n2025-11-22,3,6.7,0.0,0.0,4.8\n2025-11-22,4,5.6,0.0,0.0,4.3\n2025-11-22,5,5.2,0.0,0.0,5.1\n2025-11-22,6,4.7,0.0,0.0,3.7\n2025-11-22,7,4.0,0.0,0.0,2.2\n2025-11-22,8,4.5,0.0,0.0,4.9\n2025-11-22,9,6.3,0.0,0.0,6.1\n2025-11-22,10,9.2,0.0,0.0,4.2\n2025-11-22,11,11.3,0.0,0.0,4.7\n2025-11-22,12,12.8,0.0,0.0,6.4\n2025-11-22,13,13.5,0.0,0.0,13.4\n2025-11-22,14,13.5,0.0,0.0,14.8\n2025-11-22,15,13.4,0.0,0.0,14.3\n2025-11-22,16,13.0,0.0,0.0,13.4\n2025-11-22,17,12.0,0.0,0.0,10.5\n2025-11-22,18,10.7,0.0,0.0,6.9\n2025-11-22,19,9.7,0.0,0.0,5.4\n2025-11-22,20,9.0,0.0,0.0,4.9\n2025-11-22,21,8.7,0.0,0.0,7.1\n2025-11-22,22,8.2,0.0,0.0,9.9\n2025-11-22,23,7.7,0.0,0.0,10.6\n2025-11-23,0,7.3,0.0,0.0,10.3\n2025-11-23,1,6.9,0.0,0.0,10.7\n2025-11-23,2,6.8,0.0,0.0,10.8\n2025-11-23,3,6.4,0.0,0.0,10.8\n2025-11-23,4,6.8,0.0,0.0,9.9\n2025-11-23,5,6.5,0.0,0.0,10.4\n2025-11-23,6,6.7,0.0,0.0,11.2\n2025-11-23,7,6.7,0.0,0.0,13.3\n2025-11-23,8,7.1,0.0,0.0,13.7\n2025-11-23,9,8.0,0.0,0.0,14.8\n2025-11-23,10,10.1,0.0,0.0,11.6\n2025-11-23,11,12.3,0.0,0.0,12.0\n2025-11-23,12,14.1,0.0,0.0,15.2\n2025-11-23,13,13.9,0.0,0.0,16.2\n2025-11-23,14,14.2,0.2,0.0,17.4\n2025-11-23,15,14.0,0.0,0.0,18.4\n2025-11-23,16,14.1,0.1,0.0,16.8\n2025-11-23,17,13.3,0.0,0.0,12.1\n2025-11-23,18,11.5,0.0,0.0,14.6\n2025-11-23,19,11.3,0.0,0.0,13.5\n2025-11-23,20,11.0,0.0,0.0,13.8\n2025-11-23,21,11.0,0.0,0.0,13.4\n2025-11-23,22,10.8,0.0,0.0,13.0\n2025-11-23,23,10.5,0.0,0.0,12.4\n2025-11-24,0,10.3,0.0,0.0,11.8\n2025-11-24,1,10.0,0.0,0.0,11.9\n2025-11-24,2,10.0,0.0,0.0,12.8\n2025-11-24,3,9.1,0.0,0.0,12.3\n2025-11-24,4,9.6,0.0,0.0,12.9\n2025-11-24,5,9.6,0.0,0.0,12.5\n2025-11-24,6,9.5,0.0,0.0,12.6\n2025-11-24,7,9.2,0.0,0.0,12.4\n2025-11-24,8,8.8,0.0,0.0,12.9\n2025-11-24,9,10.3,0.0,0.0,14.7\n2025-11-24,10,11.8,0.0,0.0,12.8\n2025-11-24,11,13.3,0.5,0.0,12.6\n2025-11-24,12,14.5,0.3,0.0,17.5\n2025-11-24,13,15.4,0.1,0.0,18.4\n2025-11-24,14,15.1,0.0,0.0,17.1\n2025-11-24,15,14.9,0.0,0.0,14.4\n2025-11-24,16,14.0,0.0,0.0,13.6\n2025-11-24,17,12.5,0.0,0.0,9.7\n2025-11-24,18,11.1,0.0,0.0,8.9\n2025-11-24,19,10.7,0.0,0.0,9.1\n2025-11-24,20,10.3,0.0,0.0,8.4\n2025-11-24,21,9.5,0.0,0.0,5.3\n2025-11-24,22,9.7,0.0,0.0,5.1\n2025-11-24,23,9.5,0.0,0.0,5.6\n2025-11-25,0,8.5,0.0,0.0,4.6\n2025-11-25,1,8.3,0.0,0.0,2.9\n2025-11-25,2,8.0,0.0,0.0,0.8\n2025-11-25,3,6.9,0.0,0.0,6.1\n2025-11-25,4,6.8,0.0,0.0,9.9\n2025-11-25,5,7.2,2.2,0.0,11.7\n2025-11-25,6,8.4,1.4,0.0,12.1\n2025-11-25,7,8.4,1.0,0.0,7.1\n2025-11-25,8,8.6,0.2,0.0,14.2\n2025-11-25,9,7.7,0.0,0.0,19.3\n2025-11-25,10,7.9,0.0,0.0,19.6\n2025-11-25,11,8.6,0.0,0.0,20.8\n2025-11-25,12,9.0,0.0,0.0,23.1\n2025-11-25,13,8.7,0.2,0.0,24.4\n2025-11-25,14,8.8,0.1,0.0,21.2\n2025-11-25,15,7.8,0.0,0.0,21.0\n2025-11-25,16,7.6,0.0,0.0,19.7\n2025-11-25,17,6.6,0.0,0.0,16.3\n2025-11-25,18,5.8,0.0,0.0,12.2\n2025-11-25,19,5.3,0.0,0.0,13.1\n2025-11-25,20,5.2,0.0,0.0,11.9\n2025-11-25,21,4.7,0.0,0.0,8.6\n2025-11-25,22,4.5,0.0,0.0,6.6\n2025-11-25,23,4.4,0.0,0.0,6.4\n2025-11-26,0,4.1,0.0,0.0,6.1\n2025-11-26,1,3.4,0.0,0.0,4.0\n2025-11-26,2,2.9,0.0,0.0,3.5\n2025-11-26,3,3.8,0.0,0.0,6.1\n2025-11-26,4,4.1,0.0,0.0,6.6\n2025-11-26,5,4.2,0.0,0.0,6.4\n2025-11-26,6,4.0,0.0,0.0,6.6\n2025-11-26,7,3.7,0.0,0.0,6.4\n2025-11-26,8,4.4,0.0,0.0,6.2\n2025-11-26,9,5.5,0.0,0.0,2.6\n2025-11-26,10,6.6,0.0,0.0,1.7\n2025-11-26,11,7.4,0.0,0.0,1.3\n2025-11-26,12,8.1,0.0,0.0,1.1\n2025-11-26,13,9.2,0.0,0.0,8.6\n2025-11-26,14,9.2,0.0,0.0,10.9\n2025-11-26,15,8.9,0.0,0.0,8.0\n2025-11-26,16,9.2,0.0,0.0,6.7\n2025-11-26,17,8.4,0.0,0.0,3.9\n2025-11-26,18,6.7,0.0,0.0,2.0\n2025-11-26,19,5.9,0.0,0.0,3.3\n2025-11-26,20,5.2,0.0,0.0,5.1\n2025-11-26,21,4.6,0.0,0.0,6.7\n2025-11-26,22,4.6,0.0,0.0,8.2\n2025-11-26,23,4.7,0.0,0.0,10.2\n2025-11-27,0,4.4,0.0,0.0,10.8\n2025-11-27,1,4.1,0.0,0.0,11.6\n2025-11-27,2,4.8,0.0,0.0,14.3\n2025-11-27,3,4.8,0.0,0.0,14.4\n2025-11-27,4,4.3,0.0,0.0,16.2\n2025-11-27,5,4.8,0.0,0.0,17.5\n2025-11-27,6,4.8,0.0,0.0,19.6\n2025-11-27,7,4.7,0.0,0.0,20.7\n2025-11-27,8,4.6,0.5,0.0,17.3\n2025-11-27,9,4.9,1.5,0.0,17.7\n2025-11-27,10,5.1,4.3,0.0,19.8\n2025-11-27,11,6.4,4.2,0.0,18.8\n2025-11-27,12,8.2,0.2,0.0,16.1\n2025-11-27,13,8.8,0.1,0.0,22.7\n2025-11-27,14,9.2,0.0,0.0,19.4\n2025-11-27,15,6.2,0.1,0.0,18.8\n2025-11-27,16,7.2,0.0,0.0,26.2\n2025-11-27,17,6.5,0.0,0.0,24.1\n2025-11-27,18,6.0,0.0,0.0,23.3\n2025-11-27,19,4.8,0.0,0.0,22.0\n2025-11-27,20,3.9,0.0,0.0,20.7\n2025-11-27,21,4.3,0.0,0.0,18.9\n2025-11-27,22,3.7,0.0,0.0,19.2\n2025-11-27,23,3.0,0.0,0.0,19.0\n2025-11-28,0,2.2,0.0,0.0,17.6\n2025-11-28,1,1.3,0.0,0.0,15.3\n2025-11-28,2,0.7,0.0,0.0,15.1\n2025-11-28,3,0.6,0.0,0.0,14.6\n2025-11-28,4,0.2,0.0,0.0,12.3\n2025-11-28,5,-0.5,0.0,0.0,9.9\n2025-11-28,6,-0.9,0.0,0.0,8.0\n2025-11-28,7,-1.1,0.0,0.0,6.7\n2025-11-28,8,-1.0,0.0,0.0,6.2\n2025-11-28,9,0.3,0.0,0.0,4.9\n2025-11-28,10,1.8,0.0,0.0,4.1\n2025-11-28,11,3.1,0.0,0.0,4.0\n2025-11-28,12,4.1,0.0,0.0,4.5\n2025-11-28,13,4.9,0.0,0.0,4.7\n2025-11-28,14,5.4,0.0,0.0,4.6\n2025-11-28,15,5.5,0.0,0.0,6.1\n2025-11-28,16,5.2,0.0,0.0,5.4\n2025-11-28,17,4.0,0.0,0.0,2.8\n2025-11-28,18,3.2,0.0,0.0,1.9\n2025-11-28,19,3.0,0.0,0.0,0.5\n2025-11-28,20,2.1,0.0,0.0,2.1\n2025-11-28,21,1.5,0.0,0.0,6.5\n2025-11-28,22,1.3,0.0,0.0,7.0\n2025-11-28,23,0.8,0.0,0.0,7.6\n2025-11-29,0,0.5,0.0,0.0,7.7\n2025-11-29,1,0.2,0.0,0.0,7.3\n2025-11-29,2,0.1,0.0,0.0,8.1\n2025-11-29,3,-0.5,0.0,0.0,9.4\n2025-11-29,4,-0.5,0.0,0.0,9.4\n2025-11-29,5,-0.5,0.0,0.0,11.3\n2025-11-29,6,-0.3,0.0,0.0,12.4\n2025-11-29,7,-0.3,0.0,0.0,12.0\n2025-11-29,8,-0.3,0.0,0.0,11.6\n2025-11-29,9,1.4,0.0,0.0,13.2\n2025-11-29,10,3.5,0.0,0.0,12.5\n2025-11-29,11,5.8,0.0,0.0,12.6\n2025-11-29,12,7.6,0.0,0.0,14.5\n2025-11-29,13,9.3,0.0,0.0,15.0\n2025-11-29,14,10.6,0.0,0.0,15.1\n2025-11-29,15,10.3,0.0,0.0,11.8\n2025-11-29,16,9.5,0.0,0.0,10.2\n2025-11-29,17,8.4,0.0,0.0,11.4\n2025-11-29,18,8.4,0.1,0.0,15.3\n2025-11-29,19,8.6,0.1,0.0,16.8\n2025-11-29,20,8.5,0.1,0.0,14.1\n2025-11-29,21,9.3,0.0,0.0,13.1\n2025-11-29,22,10.2,0.1,0.0,12.8\n2025-11-29,23,10.6,0.2,0.0,12.2\n2025-11-30,0,10.4,0.1,0.0,10.3\n2025-11-30,1,10.2,0.1,0.0,8.0\n2025-11-30,2,10.0,0.1,0.0,6.3\n2025-11-30,3,8.7,0.0,0.0,3.6\n2025-11-30,4,8.2,0.1,0.0,3.1\n2025-11-30,5,7.6,0.1,0.0,3.3\n2025-11-30,6,7.3,0.0,0.0,5.1\n2025-11-30,7,7.4,0.0,0.0,6.7\n2025-11-30,8,8.0,0.0,0.0,9.3\n2025-11-30,9,8.3,0.0,0.0,11.5\n2025-11-30,10,9.3,0.0,0.0,10.9\n2025-11-30,11,11.4,0.0,0.0,11.7\n2025-11-30,12,13.4,0.1,0.0,14.5\n2025-11-30,13,14.4,0.0,0.0,13.8\n2025-11-30,14,14.7,0.0,0.0,11.5\n2025-11-30,15,15.3,0.0,0.0,14.4\n2025-11-30,16,14.8,0.0,0.0,12.4\n2025-11-30,17,13.3,0.0,0.0,12.2\n2025-11-30,18,12.2,0.0,0.0,13.7\n2025-11-30,19,11.8,0.0,0.0,15.3\n2025-11-30,20,11.6,0.0,0.0,15.1\n2025-11-30,21,11.7,0.0,0.0,14.6\n2025-11-30,22,11.8,0.1,0.0,14.8\n2025-11-30,23,11.3,0.1,0.0,15.8\n2025-12-01,0,10.7,0.0,0.0,17.7\n2025-12-01,1,9.3,0.0,0.0,18.7\n2025-12-01,2,7.3,0.0,0.0,15.3\n2025-12-01,3,5.6,0.0,0.0,13.8\n2025-12-01,4,4.7,0.0,0.0,11.1\n2025-12-01,5,4.1,0.0,0.0,11.7\n2025-12-01,6,3.7,0.0,0.0,13.4\n2025-12-01,7,2.9,0.0,0.0,13.3\n2025-12-01,8,2.3,0.0,0.0,12.2\n2025-12-01,9,3.5,0.0,0.0,11.3\n2025-12-01,10,4.7,0.0,0.0,12.0\n2025-12-01,11,5.8,0.0,0.0,10.8\n2025-12-01,12,7.0,0.0,0.0,10.4\n2025-12-01,13,7.8,0.0,0.0,10.8\n2025-12-01,14,8.0,0.0,0.0,12.2\n2025-12-01,15,7.5,0.0,0.0,13.9\n2025-12-01,16,6.9,0.0,0.0,12.0\n2025-12-01,17,5.9,0.0,0.0,8.4\n2025-12-01,18,5.0,0.0,0.0,7.3\n2025-12-01,19,4.7,0.0,0.0,7.3\n2025-12-01,20,4.0,0.0,0.0,12.6\n2025-12-01,21,2.7,0.0,0.0,12.6\n2025-12-01,22,2.0,0.0,0.0,12.6\n2025-12-01,23,1.5,0.0,0.0,13.3\n2025-12-02,0,1.0,0.0,0.0,10.8\n2025-12-02,1,0.7,0.0,0.0,8.3\n2025-12-02,2,0.6,0.0,0.0,7.7\n2025-12-02,3,0.8,0.0,0.0,8.6\n2025-12-02,4,1.0,0.0,0.0,7.7\n2025-12-02,5,0.4,0.0,0.0,12.0\n2025-12-02,6,0.1,0.0,0.0,15.5\n2025-12-02,7,-1.0,0.0,0.0,14.1\n2025-12-02,8,-1.7,0.0,0.0,14.5\n2025-12-02,9,-0.5,0.0,0.0,12.1\n2025-12-02,10,0.7,0.0,0.0,12.8\n2025-12-02,11,1.6,0.0,0.0,15.9\n2025-12-02,12,1.9,0.0,0.0,16.8\n2025-12-02,13,1.9,0.0,0.0,16.9\n2025-12-02,14,1.6,0.0,0.0,15.3\n2025-12-02,15,1.6,0.0,0.0,15.1\n2025-12-02,16,0.8,0.0,0.0,13.8\n2025-12-02,17,-0.3,0.0,0.0,12.4\n2025-12-02,18,-1.6,0.0,0.0,14.9\n2025-12-02,19,-2.8,0.0,0.0,15.7\n2025-12-02,20,-4.0,0.0,0.0,14.7\n2025-12-02,21,-4.0,0.0,0.0,17.3\n2025-12-02,22,-4.8,0.0,0.0,18.6\n2025-12-02,23,-5.5,0.0,0.0,16.4\n2025-12-03,0,-5.7,0.0,0.0,16.5\n2025-12-03,1,-5.9,0.0,0.0,16.9\n2025-12-03,2,-6.2,0.0,0.0,17.2\n2025-12-03,3,-6.7,0.0,0.0,20.1\n2025-12-03,4,-7.3,0.0,0.0,16.3\n2025-12-03,5,-7.4,0.0,0.0,14.1\n2025-12-03,6,-7.4,0.0,0.0,11.4\n2025-12-03,7,-7.4,0.0,0.0,9.5\n2025-12-03,8,-7.5,0.0,0.0,10.4\n2025-12-03,9,-7.1,0.0,0.0,10.7\n2025-12-03,10,-6.5,0.0,0.0,13.4\n2025-12-03,11,-5.9,0.0,0.0,13.0\n2025-12-03,12,-5.1,0.0,0.0,13.2\n2025-12-03,13,-4.3,0.0,0.0,15.3\n2025-12-03,14,-3.5,0.0,0.0,20.8\n2025-12-03,15,-4.1,0.0,0.0,20.7\n2025-12-03,16,-4.4,0.0,0.0,19.4\n2025-12-03,17,-4.2,0.0,0.0,20.0\n2025-12-03,18,-5.0,0.0,0.0,15.3\n2025-12-03,19,-5.8,0.0,0.0,13.2\n2025-12-03,20,-6.2,0.0,0.0,12.7\n2025-12-03,21,-7.4,0.0,0.0,13.5\n2025-12-03,22,-7.9,0.0,0.0,12.7\n2025-12-03,23,-8.1,0.0,0.0,10.5\n2025-12-04,0,-8.0,0.0,0.0,9.4\n2025-12-04,1,-7.8,0.0,0.0,7.1\n2025-12-04,2,-7.7,0.0,0.0,6.7\n2025-12-04,3,-8.2,0.0,0.0,7.6\n2025-12-04,4,-8.4,0.0,0.0,6.6\n2025-12-04,5,-8.7,0.0,0.0,5.1\n2025-12-04,6,-8.8,0.0,0.0,4.8\n2025-12-04,7,-8.5,0.0,0.0,6.3\n2025-12-04,8,-8.1,0.0,0.0,6.7\n2025-12-04,9,-8.3,0.0,0.0,5.2\n2025-12-04,10,-6.1,0.0,0.0,4.1\n2025-12-04,11,-4.2,0.0,0.0,3.3\n2025-12-04,12,-2.5,0.0,0.0,2.7\n2025-12-04,13,-1.0,0.0,0.0,6.2\n2025-12-04,14,-0.1,0.0,0.0,8.6\n2025-12-04,15,0.6,0.0,0.0,10.0\n2025-12-04,16,0.7,0.0,0.0,11.0\n2025-12-04,17,0.6,0.0,0.0,13.2\n2025-12-04,18,0.2,0.2,0.14,15.4\n2025-12-04,19,2.2,0.8,0.56,13.9\n2025-12-04,20,2.7,0.1,0.0,25.0\n2025-12-04,21,0.0,0.0,0.0,20.2\n2025-12-04,22,-3.3,0.0,0.0,20.7\n2025-12-04,23,-4.8,0.0,0.0,19.5\n2025-12-05,0,-5.5,0.0,0.0,17.0\n2025-12-05,1,-5.7,0.0,0.0,14.2\n2025-12-05,2,-5.9,0.0,0.0,12.1\n2025-12-05,3,-6.0,0.0,0.0,13.7\n2025-12-05,4,-6.6,0.0,0.0,10.4\n2025-12-05,5,-6.8,0.0,0.0,10.4\n2025-12-05,6,-6.7,0.0,0.0,10.6\n2025-12-05,7,-7.2,0.0,0.0,8.1\n2025-12-05,8,-8.4,0.0,0.0,6.1\n2025-12-05,9,-6.2,0.0,0.0,5.5\n2025-12-05,10,-4.1,0.0,0.0,6.5\n2025-12-05,11,-2.8,0.0,0.0,4.8\n2025-12-05,12,-1.5,0.0,0.0,3.9\n2025-12-05,13,-0.4,0.0,0.0,2.7\n2025-12-05,14,0.4,0.0,0.0,3.2\n2025-12-05,15,0.6,0.0,0.0,4.1\n2025-12-05,16,0.6,0.0,0.0,5.0\n2025-12-05,17,-1.1,0.0,0.0,2.3\n2025-12-05,18,-1.7,0.0,0.0,2.5\n2025-12-05,19,-2.3,0.0,0.0,3.0\n2025-12-05,20,-2.0,0.0,0.0,6.6\n2025-12-05,21,-3.4,0.0,0.0,8.9\n2025-12-05,22,-3.1,0.0,0.0,9.7\n2025-12-05,23,-2.2,0.0,0.0,10.8\n2025-12-06,0,-1.9,0.0,0.0,11.8\n2025-12-06,1,-1.8,0.0,0.0,10.4\n2025-12-06,2,-1.3,0.0,0.0,11.9\n2025-12-06,3,-1.6,0.0,0.0,11.1\n2025-12-06,4,-1.2,0.0,0.0,11.1\n2025-12-06,5,-1.2,0.0,0.0,12.1\n2025-12-06,6,-1.2,0.0,0.0,13.9\n2025-12-06,7,-1.4,0.0,0.0,16.0\n2025-12-06,8,-1.2,0.0,0.0,17.0\n2025-12-06,9,0.9,0.0,0.0,19.0\n2025-12-06,10,2.0,0.0,0.0,14.4\n2025-12-06,11,3.5,0.0,0.0,14.6\n2025-12-06,12,5.0,0.0,0.0,17.5\n2025-12-06,13,6.1,0.0,0.0,16.5\n2025-12-06,14,6.1,0.1,0.0,19.4\n2025-12-06,15,3.9,0.3,0.0,15.8\n2025-12-06,16,4.0,0.7,0.0,16.0\n2025-12-06,17,4.7,0.1,0.0,17.8\n2025-12-06,18,4.1,0.0,0.0,18.0\n2025-12-06,19,3.7,0.0,0.0,18.0\n2025-12-06,20,3.5,0.0,0.0,16.1\n2025-12-06,21,3.1,0.0,0.0,15.4\n2025-12-06,22,3.8,0.0,0.0,10.6\n2025-12-06,23,3.7,0.0,0.0,6.1\n2025-12-07,0,4.3,0.0,0.0,9.9\n2025-12-07,1,5.1,0.0,0.0,11.1\n2025-12-07,2,3.5,0.0,0.0,8.4\n2025-12-07,3,2.4,0.0,0.0,6.5\n2025-12-07,4,2.1,0.0,0.0,5.6\n2025-12-07,5,1.5,0.0,0.0,4.2\n2025-12-07,6,1.2,0.0,0.0,2.0\n2025-12-07,7,0.3,0.0,0.0,3.6\n2025-12-07,8,-0.7,0.0,0.0,5.6\n2025-12-07,9,2.6,0.0,0.0,8.7\n2025-12-07,10,5.5,0.0,0.0,9.3\n2025-12-07,11,7.9,0.0,0.0,6.4\n2025-12-07,12,9.9,0.0,0.0,7.6\n2025-12-07,13,11.3,0.0,0.0,10.6\n2025-12-07,14,11.9,0.0,0.0,17.6\n2025-12-07,15,11.9,0.0,0.0,13.2\n2025-12-07,16,10.9,0.0,0.0,12.3\n2025-12-07,17,9.8,0.1,0.0,14.1\n2025-12-07,18,9.4,0.0,0.0,17.0\n2025-12-07,19,9.0,0.0,0.0,16.5\n2025-12-07,20,8.8,0.0,0.0,18.2\n2025-12-07,21,7.8,0.0,0.0,16.2\n2025-12-07,22,7.8,0.0,0.0,15.6\n2025-12-07,23,6.9,0.0,0.0,16.7\n2025-12-08,0,5.7,0.0,0.0,18.6\n2025-12-08,1,3.9,0.0,0.0,15.9\n2025-12-08,2,2.6,0.0,0.0,15.5\n2025-12-08,3,2.9,0.0,0.0,12.7\n2025-12-08,4,3.2,0.0,0.0,14.6\n2025-12-08,5,2.6,0.0,0.0,12.8\n2025-12-08,6,1.7,0.0,0.0,16.1\n2025-12-08,7,0.2,0.0,0.0,14.4\n2025-12-08,8,-0.6,0.0,0.0,13.8\n2025-12-08,9,0.9,0.0,0.0,10.4\n2025-12-08,10,2.2,0.0,0.0,10.4\n2025-12-08,11,3.6,0.0,0.0,10.4\n2025-12-08,12,4.7,0.0,0.0,11.6\n2025-12-08,13,5.3,0.0,0.0,12.2\n2025-12-08,14,5.6,0.0,0.0,11.8\n2025-12-08,15,5.6,0.0,0.0,11.2\n2025-12-08,16,5.0,0.0,0.0,10.7\n2025-12-08,17,3.5,0.0,0.0,8.7\n2025-12-08,18,2.2,0.0,0.0,8.4\n2025-12-08,19,1.6,0.0,0.0,5.8\n2025-12-08,20,1.1,0.0,0.0,8.9\n2025-12-08,21,0.2,0.0,0.0,10.8\n2025-12-08,22,-0.5,0.0,0.0,9.5\n2025-12-08,23,-1.1,0.0,0.0,7.7\n2025-12-09,0,-1.5,0.0,0.0,6.5\n2025-12-09,1,-1.7,0.0,0.0,6.2\n2025-12-09,2,-1.7,0.0,0.0,6.4\n2025-12-09,3,-2.3,0.0,0.0,6.4\n2025-12-09,4,-2.2,0.0,0.0,6.5\n2025-12-09,5,-2.0,0.0,0.0,6.7\n2025-12-09,6,-2.3,0.0,0.0,6.6\n2025-12-09,7,-2.3,0.0,0.0,6.7\n2025-12-09,8,-1.9,0.0,0.0,7.9\n2025-12-09,9,-0.5,0.0,0.0,8.1\n2025-12-09,10,1.9,0.0,0.0,7.8\n2025-12-09,11,3.5,0.0,0.0,8.0\n2025-12-09,12,4.5,0.0,0.0,6.3\n2025-12-09,13,5.9,0.0,0.0,6.3\n2025-12-09,14,6.6,0.0,0.0,6.8\n2025-12-09,15,6.3,0.0,0.0,9.9\n2025-12-09,16,5.7,0.0,0.0,5.4\n2025-12-09,17,3.5,0.0,0.0,2.3\n2025-12-09,18,2.8,0.0,0.0,1.9\n2025-12-09,19,2.9,0.0,0.0,0.6\n2025-12-09,20,1.9,0.0,0.0,1.5\n2025-12-09,21,3.0,0.0,0.0,3.0\n2025-12-09,22,3.0,0.0,0.0,2.2\n2025-12-09,23,2.2,0.0,0.0,7.2\n2025-12-10,0,0.8,0.0,0.0,9.4\n2025-12-10,1,0.2,0.0,0.0,9.3\n2025-12-10,2,0.5,0.0,0.0,9.8\n2025-12-10,3,0.3,0.0,0.0,9.3\n2025-12-10,4,0.4,0.0,0.0,8.5\n2025-12-10,5,0.4,0.0,0.0,9.4\n2025-12-10,6,0.4,0.0,0.0,11.1\n2025-12-10,7,0.5,0.0,0.0,11.7\n2025-12-10,8,0.7,0.0,0.0,13.1\n2025-12-10,9,2.2,0.0,0.0,12.7\n2025-12-10,10,4.1,0.0,0.0,10.9\n2025-12-10,11,5.9,0.0,0.0,12.1\n2025-12-10,12,7.6,0.0,0.0,11.1\n2025-12-10,13,8.3,0.0,0.0,10.1\n2025-12-10,14,8.4,0.0,0.0,10.2\n2025-12-10,15,8.0,0.0,0.0,5.5\n2025-12-10,16,6.7,0.1,0.0,3.5\n2025-12-10,17,5.3,0.0,0.0,5.4\n2025-12-10,18,5.0,0.0,0.0,10.3\n2025-12-10,19,5.4,0.0,0.0,14.1\n2025-12-10,20,5.4,0.0,0.0,11.9\n2025-12-10,21,5.7,0.0,0.0,7.9\n2025-12-10,22,5.3,0.0,0.0,4.9\n2025-12-10,23,4.4,0.0,0.0,5.2\n2025-12-11,0,3.8,0.0,0.0,7.3\n2025-12-11,1,3.6,0.0,0.0,10.7\n2025-12-11,2,4.8,0.0,0.0,14.6\n2025-12-11,3,4.2,0.6,0.0,11.0\n2025-12-11,4,4.8,2.5,0.0,5.4\n2025-12-11,5,4.7,2.2,0.0,5.8\n2025-12-11,6,4.6,4.5,0.0,9.5\n2025-12-11,7,4.7,0.5,0.0,12.1\n2025-12-11,8,3.4,0.0,0.0,4.0\n2025-12-11,9,4.9,0.0,0.0,3.1\n2025-12-11,10,6.1,0.0,0.0,3.6\n2025-12-11,11,7.4,0.0,0.0,16.2\n2025-12-11,12,7.3,0.0,0.0,19.9\n2025-12-11,13,6.8,0.0,0.0,19.9\n2025-12-11,14,6.4,0.0,0.0,20.2\n2025-12-11,15,6.5,0.0,0.0,19.5\n2025-12-11,16,5.0,0.0,0.0,20.1\n2025-12-11,17,2.9,0.0,0.0,18.7\n2025-12-11,18,1.1,0.0,0.0,18.9\n2025-12-11,19,-0.6,0.0,0.0,19.9\n2025-12-11,20,-2.2,0.0,0.0,18.3\n2025-12-11,21,-2.3,0.0,0.0,14.0\n2025-12-11,22,-3.0,0.0,0.0,13.2\n2025-12-11,23,-3.8,0.0,0.0,13.4\n2025-12-12,0,-4.3,0.0,0.0,12.3\n2025-12-12,1,-4.3,0.0,0.0,12.3\n2025-12-12,2,-4.5,0.0,0.0,12.8\n2025-12-12,3,-4.3,0.0,0.0,11.9\n2025-12-12,4,-4.2,0.0,0.0,12.0\n2025-12-12,5,-3.9,0.0,0.0,10.8\n2025-12-12,6,-3.8,0.0,0.0,11.1\n2025-12-12,7,-3.7,0.0,0.0,9.5\n2025-12-12,8,-3.5,0.0,0.0,8.0\n2025-12-12,9,-2.6,0.0,0.0,7.8\n2025-12-12,10,0.0,0.0,0.0,7.4\n2025-12-12,11,2.2,0.0,0.0,7.5\n2025-12-12,12,3.8,0.0,0.0,7.3\n2025-12-12,13,4.7,0.0,0.0,7.2\n2025-12-12,14,5.0,0.0,0.0,6.7\n2025-12-12,15,5.1,0.0,0.0,8.4\n2025-12-12,16,4.3,0.0,0.0,7.6\n2025-12-12,17,3.2,0.0,0.0,7.5\n2025-12-12,18,1.9,0.0,0.0,7.6\n2025-12-12,19,1.4,0.0,0.0,7.8\n2025-12-12,20,1.1,0.0,0.0,7.8\n2025-12-12,21,2.3,0.0,0.0,8.8\n2025-12-12,22,2.4,0.0,0.0,9.0\n2025-12-12,23,3.1,0.0,0.0,11.5\n2025-12-13,0,2.4,0.0,0.0,11.4\n2025-12-13,1,2.2,0.0,0.0,12.1\n2025-12-13,2,2.0,0.0,0.0,13.9\n2025-12-13,3,2.6,0.0,0.0,12.1\n2025-12-13,4,3.1,0.0,0.0,12.6\n2025-12-13,5,3.0,0.0,0.0,12.1\n2025-12-13,6,3.0,0.0,0.0,10.7\n2025-12-13,7,3.2,0.0,0.0,10.0\n2025-12-13,8,3.2,0.0,0.0,8.5\n2025-12-13,9,3.2,0.2,0.0,5.2\n2025-12-13,10,2.5,0.8,0.0,7.3\n2025-12-13,11,2.2,0.8,0.0,7.7\n2025-12-13,12,2.7,0.5,0.0,3.1\n2025-12-13,13,2.6,0.5,0.0,2.6\n2025-12-13,14,2.5,0.5,0.0,3.4\n2025-12-13,15,1.7,0.9,0.0,2.9\n2025-12-13,16,1.4,1.2,0.42,5.3\n2025-12-13,17,1.1,1.4,0.63,7.6\n2025-12-13,18,1.0,0.8,0.35,9.9\n2025-12-13,19,0.8,0.5,0.21,11.0\n2025-12-13,20,0.1,0.2,0.14,14.0\n2025-12-13,21,-0.6,0.0,0.0,11.0\n2025-12-13,22,-0.7,0.0,0.0,9.8\n2025-12-13,23,-1.1,0.0,0.0,9.7\n2025-12-14,0,-1.0,0.0,0.0,8.5\n2025-12-14,1,-2.2,0.0,0.0,7.0\n2025-12-14,2,-2.3,0.0,0.0,7.8\n2025-12-14,3,-1.6,0.0,0.0,10.7\n2025-12-14,4,-0.5,0.0,0.0,13.3\n2025-12-14,5,-0.6,0.0,0.0,14.2\n2025-12-14,6,-1.6,0.0,0.0,13.6\n2025-12-14,7,-1.9,0.0,0.0,12.9\n2025-12-14,8,-1.6,0.0,0.0,14.6\n2025-12-14,9,-0.6,0.0,0.0,12.5\n2025-12-14,10,0.7,0.0,0.0,16.5\n2025-12-14,11,1.5,0.0,0.0,17.0\n2025-12-14,12,2.3,0.0,0.0,17.7\n2025-12-14,13,2.6,0.0,0.0,17.5\n2025-12-14,14,2.9,0.0,0.0,17.1\n2025-12-14,15,2.7,0.0,0.0,16.6\n2025-12-14,16,2.5,0.0,0.0,16.3\n2025-12-14,17,1.8,0.0,0.0,10.9\n2025-12-14,18,0.9,0.0,0.0,10.8\n2025-12-14,19,0.6,0.0,0.0,13.4\n2025-12-14,20,-0.3,0.0,0.0,12.3\n2025-12-14,21,-1.2,0.0,0.0,10.0\n2025-12-14,22,-1.0,0.0,0.0,9.0\n2025-12-14,23,-1.1,0.0,0.0,7.0\n2025-12-15,0,-1.6,0.0,0.0,4.9\n2025-12-15,1,-2.0,0.0,0.0,3.4\n2025-12-15,2,-1.6,0.0,0.0,4.4\n2025-12-15,3,-2.3,0.0,0.0,6.2\n2025-12-15,4,-2.7,0.0,0.0,6.0\n2025-12-15,5,-2.6,0.0,0.0,7.0\n2025-12-15,6,-2.5,0.0,0.0,7.2\n2025-12-15,7,-2.3,0.0,0.0,6.2\n2025-12-15,8,-2.7,0.0,0.0,5.8\n2025-12-15,9,-1.1,0.0,0.0,5.5\n2025-12-15,10,0.3,0.0,0.0,6.1\n2025-12-15,11,2.2,0.0,0.0,6.7\n2025-12-15,12,3.7,0.0,0.0,6.9\n2025-12-15,13,4.5,0.0,0.0,7.8\n2025-12-15,14,4.8,0.0,0.0,8.3\n2025-12-15,15,4.3,0.0,0.0,7.5\n2025-12-15,16,3.8,0.0,0.0,5.5\n2025-12-15,17,2.8,0.0,0.0,7.2\n2025-12-15,18,2.0,0.0,0.0,10.4\n2025-12-15,19,1.4,0.0,0.0,10.5\n2025-12-15,20,1.1,0.0,0.0,11.7\n2025-12-15,21,0.8,0.0,0.0,11.6\n2025-12-15,22,1.6,0.0,0.0,10.7\n2025-12-15,23,1.3,0.0,0.0,10.9\n2025-12-16,0,1.4,0.0,0.0,11.2\n2025-12-16,1,1.6,0.0,0.0,10.7\n2025-12-16,2,1.9,0.0,0.0,10.5\n2025-12-16,3,1.7,0.0,0.0,10.9\n2025-12-16,4,1.2,0.0,0.0,12.3\n2025-12-16,5,1.6,0.0,0.0,13.1\n2025-12-16,6,2.0,0.0,0.0,13.9\n2025-12-16,7,1.8,0.0,0.0,13.8\n2025-12-16,8,2.0,0.0,0.0,14.0\n2025-12-16,9,2.9,0.0,0.0,12.4\n2025-12-16,10,3.2,0.2,0.0,11.8\n2025-12-16,11,3.6,0.3,0.0,11.1\n2025-12-16,12,3.6,0.4,0.0,8.8\n2025-12-16,13,3.9,0.4,0.0,8.4\n2025-12-16,14,4.5,0.1,0.0,9.5\n2025-12-16,15,4.2,0.0,0.0,7.8\n2025-12-16,16,3.3,0.4,0.0,5.8\n2025-12-16,17,2.2,0.3,0.0,4.5\n2025-12-16,18,1.8,0.0,0.0,5.4\n2025-12-16,19,1.3,0.0,0.0,4.3\n2025-12-16,20,0.3,0.0,0.0,3.7\n2025-12-16,21,0.7,0.0,0.0,2.3\n2025-12-16,22,1.2,0.0,0.0,3.6\n2025-12-16,23,1.0,0.0,0.0,3.6\n2025-12-17,0,0.6,0.0,0.0,4.0\n2025-12-17,1,0.4,0.0,0.0,5.0\n2025-12-17,2,-0.1,0.0,0.0,3.9\n2025-12-17,3,0.2,0.0,0.0,4.9\n2025-12-17,4,1.2,0.0,0.0,7.5\n2025-12-17,5,1.7,0.0,0.0,9.2\n2025-12-17,6,1.3,0.0,0.0,9.4\n2025-12-17,7,0.1,0.0,0.0,9.8\n2025-12-17,8,-0.6,0.0,0.0,10.4\n2025-12-17,9,0.5,0.0,0.0,11.7\n2025-12-17,10,1.9,0.0,0.0,9.4\n2025-12-17,11,3.4,0.0,0.0,10.2\n2025-12-17,12,4.6,0.0,0.0,9.9\n2025-12-17,13,5.4,0.0,0.0,9.8\n2025-12-17,14,5.9,0.0,0.0,10.4\n2025-12-17,15,6.1,0.0,0.0,10.2\n2025-12-17,16,5.5,0.0,0.0,11.3\n2025-12-17,17,4.0,0.0,0.0,9.1\n2025-12-17,18,2.8,0.0,0.0,8.2\n2025-12-17,19,2.1,0.0,0.0,8.3\n2025-12-17,20,1.1,0.0,0.0,9.2\n2025-12-17,21,0.6,0.0,0.0,7.9\n2025-12-17,22,0.1,0.0,0.0,8.2\n2025-12-17,23,-0.6,0.0,0.0,6.8\n2025-12-18,0,-1.1,0.0,0.0,7.0\n2025-12-18,1,-1.5,0.0,0.0,8.4\n2025-12-18,2,-1.9,0.0,0.0,8.0\n2025-12-18,3,-2.2,0.0,0.0,7.4\n2025-12-18,4,-2.5,0.0,0.0,5.6\n2025-12-18,5,-2.9,0.0,0.0,3.4\n2025-12-18,6,-2.9,0.0,0.0,2.6\n2025-12-18,7,-2.7,0.0,0.0,2.8\n2025-12-18,8,-2.5,0.0,0.0,3.0\n2025-12-18,9,-1.0,0.0,0.0,5.7\n2025-12-18,10,1.6,0.0,0.0,3.9\n2025-12-18,11,3.5,0.0,0.0,3.5\n2025-12-18,12,4.7,0.0,0.0,3.1\n2025-12-18,13,5.2,0.0,0.0,3.3\n2025-12-18,14,5.4,0.0,0.0,3.6\n2025-12-18,15,5.3,0.0,0.0,6.0\n2025-12-18,16,4.6,0.0,0.0,4.2\n2025-12-18,17,2.1,0.0,0.0,3.6\n2025-12-18,18,1.1,0.0,0.0,1.9\n2025-12-18,19,0.5,0.0,0.0,2.0\n2025-12-18,20,0.2,0.0,0.0,4.5\n2025-12-18,21,0.2,0.0,0.0,5.6\n2025-12-18,22,0.4,0.0,0.0,7.9\n2025-12-18,23,0.1,0.0,0.0,6.2\n2025-12-19,0,-0.1,0.0,0.0,6.9\n2025-12-19,1,-0.2,0.0,0.0,7.2\n2025-12-19,2,-0.4,0.0,0.0,7.6\n2025-12-19,3,-1.0,0.0,0.0,6.7\n2025-12-19,4,-1.7,0.0,0.0,5.5\n2025-12-19,5,-1.9,0.0,0.0,7.0\n2025-12-19,6,-1.5,0.0,0.0,7.3\n2025-12-19,7,-1.0,0.0,0.0,9.3\n2025-12-19,8,-0.5,0.0,0.0,11.5\n2025-12-19,9,1.1,0.0,0.0,9.9\n2025-12-19,10,3.5,0.0,0.0,11.7\n2025-12-19,11,5.0,0.0,0.0,9.8\n2025-12-19,12,5.8,0.0,0.0,8.8\n2025-12-19,13,8.0,0.0,0.0,8.9\n2025-12-19,14,9.0,0.0,0.0,8.9\n2025-12-19,15,7.8,0.0,0.0,7.0\n2025-12-19,16,7.4,0.0,0.0,8.2\n2025-12-19,17,5.6,0.0,0.0,10.9\n2025-12-19,18,4.4,0.0,0.0,11.3\n2025-12-19,19,4.1,0.0,0.0,11.7\n2025-12-19,20,4.1,0.0,0.0,11.5\n2025-12-19,21,5.0,0.0,0.0,10.3\n2025-12-19,22,5.6,0.0,0.0,11.1\n2025-12-19,23,5.6,0.0,0.0,11.1\n2025-12-20,0,5.4,0.0,0.0,10.8\n2025-12-20,1,5.7,0.0,0.0,10.1\n2025-12-20,2,6.0,0.0,0.0,10.0\n2025-12-20,3,5.9,0.0,0.0,12.3\n2025-12-20,4,6.7,0.0,0.0,11.8\n2025-12-20,5,7.4,0.5,0.0,12.4\n2025-12-20,6,7.3,0.1,0.0,11.1\n2025-12-20,7,7.2,0.0,0.0,10.6\n2025-12-20,8,7.6,1.1,0.0,9.1\n2025-12-20,9,7.9,1.1,0.0,8.4\n2025-12-20,10,9.0,0.4,0.0,6.3\n2025-12-20,11,10.2,0.1,0.0,7.1\n2025-12-20,12,10.5,0.0,0.0,12.5\n2025-12-20,13,10.3,0.1,0.0,12.3\n2025-12-20,14,9.8,0.1,0.0,12.9\n2025-12-20,15,8.9,0.0,0.0,12.7\n2025-12-20,16,8.8,0.0,0.0,12.1\n2025-12-20,17,8.1,0.0,0.0,12.1\n2025-12-20,18,6.8,0.0,0.0,14.8\n2025-12-20,19,4.8,0.0,0.0,16.2\n2025-12-20,20,3.2,0.0,0.0,16.3\n2025-12-20,21,3.1,0.0,0.0,16.5\n2025-12-20,22,2.3,0.0,0.0,18.9\n2025-12-20,23,1.1,0.0,0.0,18.0\n2025-12-21,0,0.5,0.0,0.0,16.6\n2025-12-21,1,-0.3,0.0,0.0,16.8\n2025-12-21,2,-1.2,0.0,0.0,16.1\n2025-12-21,3,-1.4,0.0,0.0,17.0\n2025-12-21,4,-2.0,0.0,0.0,16.6\n2025-12-21,5,-2.8,0.0,0.0,14.9\n2025-12-21,6,-3.2,0.0,0.0,13.8\n2025-12-21,7,-2.9,0.0,0.0,15.7\n2025-12-21,8,-3.5,0.0,0.0,15.3\n2025-12-21,9,-3.2,0.0,0.0,15.3\n2025-12-21,10,-2.5,0.0,0.0,15.2\n2025-12-21,11,-1.6,0.0,0.0,13.5\n2025-12-21,12,-0.7,0.0,0.0,12.1\n2025-12-21,13,0.1,0.0,0.0,13.2\n2025-12-21,14,0.5,0.0,0.0,13.6\n2025-12-21,15,0.8,0.0,0.0,11.9\n2025-12-21,16,0.3,0.0,0.0,11.4\n2025-12-21,17,-0.6,0.0,0.0,9.4\n2025-12-21,18,-1.8,0.0,0.0,9.3\n2025-12-21,19,-2.4,0.0,0.0,10.3\n2025-12-21,20,-2.9,0.0,0.0,11.5\n2025-12-21,21,-3.1,0.0,0.0,12.0\n2025-12-21,22,-3.7,0.0,0.0,12.2\n2025-12-21,23,-4.2,0.0,0.0,11.3\n2025-12-22,0,-4.6,0.0,0.0,9.4\n2025-12-22,1,-4.8,0.0,0.0,7.2\n2025-12-22,2,-4.8,0.0,0.0,6.0\n2025-12-22,3,-5.2,0.0,0.0,6.4\n2025-12-22,4,-5.3,0.0,0.0,6.4\n2025-12-22,5,-5.2,0.0,0.0,6.5\n2025-12-22,6,-5.3,0.0,0.0,6.8\n2025-12-22,7,-5.4,0.0,0.0,7.0\n2025-12-22,8,-5.3,0.0,0.0,6.5\n2025-12-22,9,-4.5,0.0,0.0,6.5\n2025-12-22,10,-2.6,0.0,0.0,4.5\n2025-12-22,11,-0.8,0.0,0.0,3.7\n2025-12-22,12,0.8,0.0,0.0,3.5\n2025-12-22,13,2.0,0.0,0.0,3.1\n2025-12-22,14,2.7,0.0,0.0,3.2\n2025-12-22,15,2.9,0.0,0.0,4.6\n2025-12-22,16,2.4,0.0,0.0,4.1\n2025-12-22,17,0.5,0.0,0.0,6.7\n2025-12-22,18,-0.8,0.0,0.0,7.9\n2025-12-22,19,-1.6,0.0,0.0,5.6\n2025-12-22,20,-1.7,0.0,0.0,5.7\n2025-12-22,21,-2.3,0.0,0.0,4.9\n2025-12-22,22,-2.1,0.0,0.0,4.5\n2025-12-22,23,-1.2,0.0,0.0,4.2\n2025-12-23,0,-0.7,0.0,0.0,5.2\n2025-12-23,1,0.5,0.0,0.0,8.4\n2025-12-23,2,0.3,0.0,0.0,8.0\n2025-12-23,3,-0.5,0.0,0.0,8.3\n2025-12-23,4,-0.4,0.0,0.0,11.3\n2025-12-23,5,0.6,0.0,0.0,11.4\n2025-12-23,6,1.4,0.0,0.0,12.3\n2025-12-23,7,1.5,0.0,0.0,11.4\n2025-12-23,8,1.6,0.0,0.0,10.1\n2025-12-23,9,2.4,0.0,0.0,13.2\n2025-12-23,10,2.8,0.1,0.0,14.6\n2025-12-23,11,3.2,0.1,0.0,14.3\n2025-12-23,12,3.7,0.4,0.0,9.6\n2025-12-23,13,2.2,0.8,0.0,6.4\n2025-12-23,14,0.8,0.9,0.42,7.8\n2025-12-23,15,0.7,1.6,0.14,11.3\n2025-12-23,16,1.0,1.7,0.0,8.3\n2025-12-23,17,0.8,1.2,0.0,8.2\n2025-12-23,18,0.7,1.2,0.0,7.8\n2025-12-23,19,0.7,1.1,0.0,6.6\n2025-12-23,20,0.7,0.8,0.0,7.0\n2025-12-23,21,1.2,0.8,0.0,7.5\n2025-12-23,22,1.1,0.2,0.0,7.4\n2025-12-23,23,0.8,0.0,0.0,9.8\n2025-12-24,0,0.9,0.0,0.0,10.9\n2025-12-24,1,0.4,0.0,0.0,9.2\n2025-12-24,2,-0.6,0.0,0.0,8.6\n2025-12-24,3,0.1,0.0,0.0,10.7\n2025-12-24,4,0.8,0.0,0.0,9.4\n2025-12-24,5,1.0,0.0,0.0,9.4\n2025-12-24,6,1.0,0.0,0.0,9.2\n2025-12-24,7,1.1,0.0,0.0,11.9\n2025-12-24,8,0.9,0.0,0.0,12.2\n2025-12-24,9,0.7,0.0,0.0,11.4\n2025-12-24,10,1.2,0.0,0.0,10.9\n2025-12-24,11,1.6,0.0,0.0,11.0\n2025-12-24,12,2.1,0.0,0.0,11.7\n2025-12-24,13,3.0,0.0,0.0,11.8\n2025-12-24,14,3.6,0.0,0.0,11.3\n2025-12-24,15,3.6,0.0,0.0,10.1\n2025-12-24,16,3.3,0.0,0.0,10.4\n2025-12-24,17,2.1,0.0,0.0,8.6\n2025-12-24,18,0.7,0.0,0.0,10.2\n2025-12-24,19,-0.4,0.0,0.0,9.1\n2025-12-24,20,-1.3,0.0,0.0,11.1\n2025-12-24,21,-1.8,0.0,0.0,11.2\n2025-12-24,22,-2.2,0.0,0.0,10.5\n2025-12-24,23,-2.8,0.0,0.0,11.2\n2025-12-25,0,-3.3,0.0,0.0,12.2\n2025-12-25,1,-3.4,0.0,0.0,12.6\n2025-12-25,2,-3.6,0.0,0.0,13.2\n2025-12-25,3,-3.5,0.0,0.0,11.3\n2025-12-25,4,-3.7,0.0,0.0,11.6\n2025-12-25,5,-3.9,0.0,0.0,12.5\n2025-12-25,6,-4.1,0.0,0.0,12.7\n2025-12-25,7,-4.3,0.0,0.0,13.9\n2025-12-25,8,-4.6,0.0,0.0,14.0\n2025-12-25,9,-3.2,0.0,0.0,10.6\n2025-12-25,10,-1.8,0.0,0.0,14.7\n2025-12-25,11,-1.1,0.0,0.0,18.3\n2025-12-25,12,-0.7,0.0,0.0,18.6\n2025-12-25,13,-0.5,0.0,0.0,20.1\n2025-12-25,14,-0.6,0.0,0.0,22.4\n2025-12-25,15,-1.6,0.0,0.0,22.8\n2025-12-25,16,-2.7,0.0,0.0,22.7\n2025-12-25,17,-3.9,0.0,0.0,21.2\n2025-12-25,18,-5.2,0.0,0.0,23.6\n2025-12-25,19,-6.4,0.0,0.0,23.6\n2025-12-25,20,-7.2,0.0,0.0,21.1\n2025-12-25,21,-7.6,0.0,0.0,21.0\n2025-12-25,22,-8.7,0.0,0.0,21.2\n2025-12-25,23,-9.1,0.0,0.0,19.8\n2025-12-26,0,-9.3,0.0,0.0,20.3\n2025-12-26,1,-9.7,0.0,0.0,20.4\n2025-12-26,2,-10.4,0.0,0.0,19.6\n2025-12-26,3,-10.4,0.0,0.0,20.7\n2025-12-26,4,-10.7,0.0,0.0,19.0\n2025-12-26,5,-11.0,0.0,0.0,17.3\n2025-12-26,6,-11.1,0.0,0.0,16.3\n2025-12-26,7,-11.2,0.0,0.0,16.4\n2025-12-26,8,-11.3,0.0,0.0,16.1\n2025-12-26,9,-11.2,0.0,0.0,13.6\n2025-12-26,10,-10.1,0.0,0.0,12.5\n2025-12-26,11,-8.6,0.0,0.0,12.2\n2025-12-26,12,-7.2,0.0,0.0,12.9\n2025-12-26,13,-6.0,0.0,0.0,14.2\n2025-12-26,14,-5.4,0.0,0.0,15.3\n2025-12-26,15,-5.1,0.0,0.0,13.0\n2025-12-26,16,-5.4,0.0,0.0,13.8\n2025-12-26,17,-6.1,0.0,0.0,12.4\n2025-12-26,18,-6.5,0.0,0.0,9.5\n2025-12-26,19,-6.7,0.0,0.0,7.7\n2025-12-26,20,-6.9,0.0,0.0,5.7\n2025-12-26,21,-7.2,0.0,0.0,5.8\n2025-12-26,22,-7.3,0.0,0.0,6.5\n2025-12-26,23,-7.4,0.0,0.0,6.9\n2025-12-27,0,-7.3,0.0,0.0,6.4\n2025-12-27,1,-7.3,0.0,0.0,5.8\n2025-12-27,2,-7.1,0.0,0.0,5.6\n2025-12-27,3,-7.5,0.0,0.0,5.1\n2025-12-27,4,-7.2,0.0,0.0,4.6\n2025-12-27,5,-6.8,0.0,0.0,7.4\n2025-12-27,6,-6.6,0.0,0.0,4.4\n2025-12-27,7,-6.2,0.0,0.0,5.6\n2025-12-27,8,-6.1,0.0,0.0,7.1\n2025-12-27,9,-6.1,0.0,0.0,5.8\n2025-12-27,10,-4.1,0.0,0.0,6.5\n2025-12-27,11,-2.5,0.0,0.0,6.9\n2025-12-27,12,0.3,0.0,0.0,7.4\n2025-12-27,13,1.8,0.0,0.0,8.3\n2025-12-27,14,2.9,0.0,0.0,10.0\n2025-12-27,15,0.8,0.0,0.0,5.4\n2025-12-27,16,1.0,0.0,0.0,3.8\n2025-12-27,17,0.3,0.0,0.0,4.4\n2025-12-27,18,-0.3,0.0,0.0,7.1\n2025-12-27,19,-1.3,0.0,0.0,8.7\n2025-12-27,20,-1.8,0.0,0.0,10.2\n2025-12-27,21,-1.1,0.0,0.0,7.9\n2025-12-27,22,-0.7,0.0,0.0,7.8\n2025-12-27,23,-0.3,0.0,0.0,7.9\n2025-12-28,0,-1.1,0.0,0.0,8.6\n2025-12-28,1,-1.5,0.0,0.0,10.0\n2025-12-28,2,-1.6,0.0,0.0,10.4\n2025-12-28,3,-1.9,0.0,0.0,8.3\n2025-12-28,4,-1.6,0.0,0.0,8.2\n2025-12-28,5,-1.7,0.0,0.0,8.2\n2025-12-28,6,-1.5,0.0,0.0,8.6\n2025-12-28,7,-1.2,0.0,0.0,8.2\n2025-12-28,8,-1.3,0.0,0.0,8.4\n2025-12-28,9,-0.4,0.0,0.0,10.6\n2025-12-28,10,2.1,0.0,0.0,10.0\n2025-12-28,11,4.1,0.0,0.0,10.7\n2025-12-28,12,5.6,0.0,0.0,11.9\n2025-12-28,13,5.7,0.0,0.0,11.0\n2025-12-28,14,4.7,0.6,0.0,10.5\n2025-12-28,15,5.6,0.3,0.0,9.3\n2025-12-28,16,5.2,0.0,0.0,7.2\n2025-12-28,17,4.0,0.4,0.0,8.5\n2025-12-28,18,3.1,0.6,0.0,10.6\n2025-12-28,19,2.6,0.0,0.0,12.9\n2025-12-28,20,2.7,0.0,0.0,13.5\n2025-12-28,21,2.4,0.0,0.0,13.6\n2025-12-28,22,3.0,0.0,0.0,12.1\n2025-12-28,23,2.7,0.0,0.0,10.4\n2025-12-29,0,2.4,0.0,0.0,9.7\n2025-12-29,1,2.2,0.0,0.0,9.5\n2025-12-29,2,2.4,0.0,0.0,10.2\n2025-12-29,3,2.1,0.0,0.0,10.0\n2025-12-29,4,2.1,0.3,0.0,11.0\n2025-12-29,5,1.7,0.0,0.0,11.3\n2025-12-29,6,1.7,0.3,0.0,9.3\n2025-12-29,7,2.4,0.5,0.0,5.3\n2025-12-29,8,2.8,0.0,0.0,6.0\n2025-12-29,9,5.0,0.0,0.0,9.6\n2025-12-29,10,6.8,0.0,0.0,7.6\n2025-12-29,11,7.7,0.0,0.0,10.8\n2025-12-29,12,8.1,0.1,0.0,12.5\n2025-12-29,13,8.2,0.1,0.0,15.2\n2025-12-29,14,8.1,0.0,0.0,16.6\n2025-12-29,15,7.7,0.0,0.0,15.7\n2025-12-29,16,6.6,0.0,0.0,15.5\n2025-12-29,17,5.3,0.0,0.0,14.5\n2025-12-29,18,4.3,0.0,0.0,15.1\n2025-12-29,19,3.0,0.0,0.0,15.5\n2025-12-29,20,1.3,0.0,0.0,14.8\n2025-12-29,21,0.7,0.0,0.0,12.8\n2025-12-29,22,-0.1,0.0,0.0,12.0\n2025-12-29,23,-1.1,0.0,0.0,12.2\n2025-12-30,0,-2.1,0.0,0.0,12.4\n2025-12-30,1,-2.6,0.0,0.0,11.8\n2025-12-30,2,-2.9,0.0,0.0,11.4\n2025-12-30,3,-3.1,0.0,0.0,11.8\n2025-12-30,4,-3.5,0.0,0.0,11.7\n2025-12-30,5,-3.9,0.0,0.0,10.7\n2025-12-30,6,-4.1,0.0,0.0,9.6\n2025-12-30,7,-4.2,0.0,0.0,9.9\n2025-12-30,8,-4.2,0.0,0.0,10.3\n2025-12-30,9,-3.0,0.0,0.0,7.8\n2025-12-30,10,-1.4,0.0,0.0,8.0\n2025-12-30,11,0.1,0.0,0.0,9.1\n2025-12-30,12,1.1,0.0,0.0,10.3\n2025-12-30,13,1.7,0.0,0.0,10.9\n2025-12-30,14,2.2,0.0,0.0,10.1\n2025-12-30,15,2.4,0.0,0.0,10.9\n2025-12-30,16,1.8,0.0,0.0,11.7\n2025-12-30,17,0.5,0.0,0.0,8.3\n2025-12-30,18,-1.1,0.0,0.0,8.4\n2025-12-30,19,-2.1,0.0,0.0,10.6\n2025-12-30,20,-2.2,0.0,0.0,10.9\n2025-12-30,21,-1.0,0.0,0.0,11.2\n2025-12-30,22,-1.2,0.0,0.0,10.5\n2025-12-30,23,-1.4,0.0,0.0,10.7\n2025-12-31,0,-1.7,0.0,0.0,11.4\n2025-12-31,1,-1.9,0.0,0.0,12.5\n2025-12-31,2,-3.6,0.0,0.0,17.6\n2025-12-31,3,-4.6,0.0,0.0,15.8\n2025-12-31,4,-5.8,0.0,0.0,14.8\n2025-12-31,5,-6.6,0.0,0.0,15.7\n2025-12-31,6,-7.3,0.0,0.0,16.0\n2025-12-31,7,-7.6,0.0,0.0,15.8\n2025-12-31,8,-7.7,0.0,0.0,16.5\n2025-12-31,9,-7.4,0.0,0.0,15.1\n2025-12-31,10,-6.5,0.0,0.0,15.8\n2025-12-31,11,-5.4,0.0,0.0,15.3\n2025-12-31,12,-4.5,0.0,0.0,15.7\n2025-12-31,13,-4.0,0.0,0.0,16.3\n2025-12-31,14,-3.7,0.0,0.0,16.7\n2025-12-31,15,-3.8,0.0,0.0,15.3\n2025-12-31,16,-4.2,0.0,0.0,14.8\n2025-12-31,17,-4.9,0.0,0.0,13.1\n2025-12-31,18,-5.6,0.0,0.0,13.9\n2025-12-31,19,-6.4,0.0,0.0,16.0\n2025-12-31,20,-7.1,0.0,0.0,16.8\n2025-12-31,21,-7.3,0.0,0.0,15.3\n2025-12-31,22,-7.9,0.0,0.0,14.8\n2025-12-31,23,-8.1,0.0,0.0,14.3\n2026-01-01,0,-8.2,0.0,0.0,14.3\n2026-01-01,1,-8.3,0.0,0.0,14.1\n2026-01-01,2,-8.4,0.0,0.0,15.2\n2026-01-01,3,-8.7,0.0,0.0,14.3\n2026-01-01,4,-8.9,0.0,0.0,14.4\n2026-01-01,5,-9.3,0.0,0.0,15.0\n2026-01-01,6,-9.6,0.0,0.0,13.7\n2026-01-01,7,-9.8,0.0,0.0,13.3\n2026-01-01,8,-9.9,0.0,0.0,12.3\n2026-01-01,9,-10.2,0.0,0.0,10.2\n2026-01-01,10,-9.0,0.0,0.0,9.4\n2026-01-01,11,-7.3,0.0,0.0,10.1\n2026-01-01,12,-5.7,0.0,0.0,11.9\n2026-01-01,13,-4.6,0.0,0.0,13.5\n2026-01-01,14,-4.1,0.0,0.0,14.0\n2026-01-01,15,-3.9,0.0,0.0,13.2\n2026-01-01,16,-4.3,0.0,0.0,13.9\n2026-01-01,17,-4.9,0.0,0.0,14.9\n2026-01-01,18,-5.5,0.0,0.0,14.4\n2026-01-01,19,-6.1,0.0,0.0,15.0\n2026-01-01,20,-7.0,0.0,0.0,18.7\n2026-01-01,21,-7.5,0.0,0.0,16.5\n2026-01-01,22,-7.9,0.0,0.0,16.9\n2026-01-01,23,-8.6,0.0,0.0,17.6\n2026-01-02,0,-9.1,0.0,0.0,15.6\n2026-01-02,1,-9.7,0.0,0.0,15.2\n2026-01-02,2,-10.1,0.0,0.0,15.5\n2026-01-02,3,-10.8,0.0,0.0,12.9\n2026-01-02,4,-11.2,0.0,0.0,13.3\n2026-01-02,5,-11.4,0.0,0.0,13.5\n2026-01-02,6,-11.6,0.0,0.0,12.2\n2026-01-02,7,-11.7,0.0,0.0,11.1\n2026-01-02,8,-11.6,0.0,0.0,11.8\n2026-01-02,9,-11.8,0.0,0.0,7.6\n2026-01-02,10,-10.3,0.0,0.0,6.3\n2026-01-02,11,-8.7,0.0,0.0,4.3\n2026-01-02,12,-7.3,0.0,0.0,7.2\n2026-01-02,13,-6.0,0.0,0.0,9.8\n2026-01-02,14,-5.4,0.0,0.0,11.7\n2026-01-02,15,-5.1,0.0,0.0,10.5\n2026-01-02,16,-5.1,0.0,0.0,11.5\n2026-01-02,17,-5.5,0.0,0.0,11.1\n2026-01-02,18,-5.7,0.0,0.0,9.8\n2026-01-02,19,-6.1,0.0,0.0,13.9\n2026-01-02,20,-6.7,0.0,0.0,14.7\n2026-01-02,21,-7.7,0.0,0.0,11.3\n2026-01-02,22,-8.1,0.0,0.0,9.1\n2026-01-02,23,-8.2,0.0,0.0,6.2\n2026-01-03,0,-8.2,0.0,0.0,3.7\n2026-01-03,1,-8.1,0.0,0.0,4.6\n2026-01-03,2,-8.0,0.0,0.0,4.9\n2026-01-03,3,-9.3,0.0,0.0,2.4\n2026-01-03,4,-9.6,0.0,0.0,3.1\n2026-01-03,5,-9.5,0.0,0.0,3.1\n2026-01-03,6,-9.7,0.0,0.0,2.5\n2026-01-03,7,-9.6,0.0,0.0,2.2\n2026-01-03,8,-9.4,0.0,0.0,2.6\n2026-01-03,9,-8.1,0.0,0.0,6.4\n2026-01-03,10,-5.9,0.0,0.0,4.5\n2026-01-03,11,-3.5,0.0,0.0,3.2\n2026-01-03,12,-2.2,0.0,0.0,1.9\n2026-01-03,13,-1.0,0.0,0.0,2.4\n2026-01-03,14,-0.7,0.0,0.0,4.8\n2026-01-03,15,-0.8,0.0,0.0,4.2\n2026-01-03,16,-0.9,0.0,0.0,3.4\n2026-01-03,17,-1.0,0.0,0.0,1.6\n2026-01-03,18,-1.0,0.0,0.0,2.2\n2026-01-03,19,-1.8,0.0,0.0,3.2\n2026-01-03,20,-1.7,0.0,0.0,2.7\n2026-01-03,21,-1.4,0.0,0.0,5.8\n2026-01-03,22,-1.7,0.0,0.0,5.6\n2026-01-03,23,-1.3,0.0,0.0,4.7\n2026-01-04,0,-1.7,0.0,0.0,4.5\n2026-01-04,1,-2.3,0.0,0.0,3.9\n2026-01-04,2,-1.5,0.0,0.0,5.2\n2026-01-04,3,-2.9,0.0,0.0,3.3\n2026-01-04,4,-3.4,0.0,0.0,4.6\n2026-01-04,5,-3.9,0.0,0.0,7.2\n2026-01-04,6,-3.8,0.0,0.0,7.6\n2026-01-04,7,-3.0,0.0,0.0,6.2\n2026-01-04,8,-2.4,0.0,0.0,6.0\n2026-01-04,9,-2.8,0.0,0.0,4.8\n2026-01-04,10,-1.2,0.0,0.0,4.3\n2026-01-04,11,0.4,0.0,0.0,6.1\n2026-01-04,12,1.5,0.0,0.0,8.5\n2026-01-04,13,2.2,0.0,0.0,9.5\n2026-01-04,14,2.4,0.0,0.0,11.6\n2026-01-04,15,2.5,0.0,0.0,12.2\n2026-01-04,16,1.5,0.0,0.0,12.1\n2026-01-04,17,0.6,0.0,0.0,8.8\n2026-01-04,18,-0.1,0.0,0.0,8.2\n2026-01-04,19,0.2,0.0,0.0,8.6\n2026-01-04,20,0.5,0.0,0.0,7.3\n2026-01-04,21,0.6,0.0,0.0,8.4\n2026-01-04,22,0.3,0.0,0.0,10.1\n2026-01-04,23,-0.9,0.0,0.0,10.6\n2026-01-05,0,-1.5,0.0,0.0,10.2\n2026-01-05,1,-1.8,0.0,0.0,10.0\n2026-01-05,2,-2.0,0.0,0.0,10.7\n2026-01-05,3,-2.1,0.0,0.0,10.0\n2026-01-05,4,-3.0,0.0,0.0,9.2\n2026-01-05,5,-3.3,0.0,0.0,8.1\n2026-01-05,6,-3.2,0.0,0.0,8.9\n2026-01-05,7,-3.1,0.0,0.0,12.0\n2026-01-05,8,-3.8,0.0,0.0,13.1\n2026-01-05,9,-3.2,0.0,0.0,10.9\n2026-01-05,10,-2.3,0.0,0.0,12.8\n2026-01-05,11,-1.2,0.0,0.0,13.9\n2026-01-05,12,-0.3,0.0,0.0,15.4\n2026-01-05,13,0.4,0.0,0.0,16.1\n2026-01-05,14,0.6,0.0,0.0,17.2\n2026-01-05,15,0.2,0.0,0.0,16.2\n2026-01-05,16,-0.4,0.0,0.0,15.2\n2026-01-05,17,-1.5,0.0,0.0,11.9\n2026-01-05,18,-2.7,0.0,0.0,12.4\n2026-01-05,19,-3.2,0.0,0.0,13.4\n2026-01-05,20,-3.8,0.0,0.0,13.2\n2026-01-05,21,-3.9,0.0,0.0,12.5\n2026-01-05,22,-4.3,0.0,0.0,11.3\n2026-01-05,23,-4.6,0.0,0.0,9.9\n2026-01-06,0,-4.7,0.0,0.0,8.2\n2026-01-06,1,-4.7,0.0,0.0,7.9\n2026-01-06,2,-4.7,0.0,0.0,7.7\n2026-01-06,3,-5.4,0.0,0.0,7.3\n2026-01-06,4,-5.8,0.0,0.0,4.6\n2026-01-06,5,-6.4,0.0,0.0,2.6\n2026-01-06,6,-6.6,0.0,0.0,2.3\n2026-01-06,7,-6.9,0.0,0.0,1.7\n2026-01-06,8,-7.0,0.0,0.0,2.4\n2026-01-06,9,-6.0,0.0,0.0,5.1\n2026-01-06,10,-3.3,0.0,0.0,4.1\n2026-01-06,11,-1.1,0.0,0.0,4.4\n2026-01-06,12,0.7,0.0,0.0,4.5\n2026-01-06,13,2.0,0.0,0.0,5.9\n2026-01-06,14,2.7,0.0,0.0,8.7\n2026-01-06,15,2.4,0.0,0.0,9.5\n2026-01-06,16,2.1,0.0,0.0,7.4\n2026-01-06,17,1.1,0.0,0.0,8.8\n2026-01-06,18,-0.4,0.0,0.0,11.7\n2026-01-06,19,-0.8,0.0,0.0,11.1\n2026-01-06,20,-0.3,0.0,0.0,8.3\n2026-01-06,21,-0.6,0.0,0.0,3.9\n2026-01-06,22,-0.2,0.0,0.0,4.5\n2026-01-06,23,0.2,0.0,0.0,6.1\n2026-01-07,0,1.9,0.0,0.0,10.5\n2026-01-07,1,2.4,0.0,0.0,11.4\n2026-01-07,2,2.4,0.0,0.0,11.8\n2026-01-07,3,1.6,0.0,0.0,10.7\n2026-01-07,4,1.5,0.0,0.0,10.5\n2026-01-07,5,1.9,0.0,0.0,12.9\n2026-01-07,6,2.3,0.1,0.07,14.6\n2026-01-07,7,1.9,0.0,0.0,15.6\n2026-01-07,8,-0.5,0.0,0.0,14.5\n2026-01-07,9,-0.7,0.0,0.0,14.7\n2026-01-07,10,-0.1,0.0,0.0,16.3\n2026-01-07,11,0.6,0.0,0.0,17.8\n2026-01-07,12,1.1,0.0,0.0,17.1\n2026-01-07,13,1.3,0.0,0.0,18.0\n2026-01-07,14,1.3,0.0,0.0,18.5\n2026-01-07,15,1.0,0.0,0.0,18.3\n2026-01-07,16,0.3,0.0,0.0,18.4\n2026-01-07,17,-0.9,0.0,0.0,17.1\n2026-01-07,18,-2.2,0.0,0.0,17.2\n2026-01-07,19,-3.0,0.0,0.0,17.4\n2026-01-07,20,-4.1,0.0,0.0,18.8\n2026-01-07,21,-4.2,0.0,0.0,17.3\n2026-01-07,22,-4.7,0.0,0.0,17.6\n2026-01-07,23,-4.9,0.0,0.0,17.4\n2026-01-08,0,-5.1,0.0,0.0,17.2\n2026-01-08,1,-5.3,0.0,0.0,17.2\n2026-01-08,2,-5.9,0.0,0.0,17.6\n2026-01-08,3,-5.8,0.0,0.0,15.3\n2026-01-08,4,-6.3,0.0,0.0,14.4\n2026-01-08,5,-6.6,0.0,0.0,14.7\n2026-01-08,6,-6.9,0.0,0.0,15.0\n2026-01-08,7,-7.1,0.0,0.0,13.3\n2026-01-08,8,-7.2,0.0,0.0,13.0\n2026-01-08,9,-7.4,0.0,0.0,14.0\n2026-01-08,10,-6.2,0.0,0.0,13.4\n2026-01-08,11,-5.1,0.0,0.0,12.5\n2026-01-08,12,-4.1,0.0,0.0,11.7\n2026-01-08,13,-3.3,0.0,0.0,10.8\n2026-01-08,14,-2.8,0.0,0.0,10.1\n2026-01-08,15,-2.8,0.0,0.0,9.2\n2026-01-08,16,-3.1,0.0,0.0,10.3\n2026-01-08,17,-3.7,0.0,0.0,9.8\n2026-01-08,18,-4.4,0.0,0.0,6.6\n2026-01-08,19,-4.4,0.0,0.0,3.9\n2026-01-08,20,-4.1,0.0,0.0,0.9\n2026-01-08,21,-4.1,0.0,0.0,0.8\n2026-01-08,22,-4.5,0.0,0.0,0.9\n2026-01-08,23,-4.9,0.0,0.0,0.9\n2026-01-09,0,-5.3,0.0,0.0,3.0\n2026-01-09,1,-5.7,0.0,0.0,4.0\n2026-01-09,2,-5.5,0.0,0.0,5.0\n2026-01-09,3,-6.2,0.0,0.0,9.1\n2026-01-09,4,-6.2,0.0,0.0,10.7\n2026-01-09,5,-6.5,0.0,0.0,12.5\n2026-01-09,6,-5.4,0.0,0.0,13.6\n2026-01-09,7,-4.1,0.0,0.0,14.6\n2026-01-09,8,-4.4,0.0,0.0,15.1\n2026-01-09,9,-4.0,0.0,0.0,16.6\n2026-01-09,10,-1.8,0.0,0.0,12.6\n2026-01-09,11,0.8,0.0,0.0,11.8\n2026-01-09,12,2.5,0.0,0.0,11.5\n2026-01-09,13,3.4,0.0,0.0,13.8\n2026-01-09,14,3.6,0.0,0.0,14.9\n2026-01-09,15,3.1,0.1,0.0,15.4\n2026-01-09,16,3.3,0.2,0.07,13.8\n2026-01-09,17,2.5,0.1,0.0,13.1\n2026-01-09,18,2.7,0.0,0.0,13.5\n2026-01-09,19,2.9,0.1,0.0,14.3\n2026-01-09,20,1.7,0.0,0.0,14.8\n2026-01-09,21,0.9,0.0,0.0,15.5\n2026-01-09,22,1.6,0.0,0.0,15.7\n2026-01-09,23,1.5,0.0,0.0,15.9\n2026-01-10,0,1.8,0.0,0.0,17.2\n2026-01-10,1,3.5,1.3,0.14,23.9\n2026-01-10,2,4.7,2.9,0.7,31.1\n2026-01-10,3,4.1,0.1,0.0,27.3\n2026-01-10,4,5.0,0.0,0.0,26.6\n2026-01-10,5,3.5,2.8,0.35,24.3\n2026-01-10,6,4.3,0.3,0.0,20.6\n2026-01-10,7,4.2,0.4,0.0,23.5\n2026-01-10,8,4.5,0.0,0.0,30.8\n2026-01-10,9,3.9,0.1,0.0,29.6\n2026-01-10,10,4.1,0.1,0.0,36.7\n2026-01-10,11,4.1,0.0,0.0,35.7\n2026-01-10,12,3.5,0.5,0.35,39.0\n2026-01-10,13,2.3,0.4,0.28,36.8\n2026-01-10,14,2.5,0.0,0.0,36.3\n2026-01-10,15,1.2,0.0,0.0,37.8\n2026-01-10,16,0.1,0.0,0.0,33.3\n2026-01-10,17,-0.9,0.0,0.0,28.9\n2026-01-10,18,-1.9,0.0,0.0,25.9\n2026-01-10,19,-3.0,0.0,0.0,26.5\n2026-01-10,20,-3.5,0.0,0.0,23.7\n2026-01-10,21,-4.0,0.0,0.0,24.2\n2026-01-10,22,-4.1,0.0,0.0,23.7\n2026-01-10,23,-4.4,0.0,0.0,21.7\n2026-01-11,0,-5.6,0.0,0.0,20.0\n2026-01-11,1,-6.3,0.0,0.0,19.8\n2026-01-11,2,-6.8,0.0,0.0,19.0\n2026-01-11,3,-6.4,0.0,0.0,19.3\n2026-01-11,4,-6.8,0.0,0.0,16.3\n2026-01-11,5,-7.2,0.0,0.0,13.9\n2026-01-11,6,-7.6,0.0,0.0,14.8\n2026-01-11,7,-8.0,0.0,0.0,13.8\n2026-01-11,8,-8.5,0.0,0.0,13.1\n2026-01-11,9,-7.7,0.0,0.0,13.4\n2026-01-11,10,-6.3,0.0,0.0,16.3\n2026-01-11,11,-5.5,0.0,0.0,18.2\n2026-01-11,12,-4.8,0.0,0.0,18.1\n2026-01-11,13,-4.3,0.0,0.0,17.7\n2026-01-11,14,-4.3,0.0,0.0,17.8\n2026-01-11,15,-4.3,0.0,0.0,21.0\n2026-01-11,16,-4.6,0.0,0.0,22.1\n2026-01-11,17,-5.4,0.0,0.0,18.0\n2026-01-11,18,-6.0,0.0,0.0,13.9\n2026-01-11,19,-6.4,0.0,0.0,13.8\n2026-01-11,20,-7.4,0.0,0.0,14.8\n2026-01-11,21,-7.7,0.0,0.0,10.8\n2026-01-11,22,-8.1,0.0,0.0,8.4\n2026-01-11,23,-8.5,0.0,0.0,5.7\n2026-01-12,0,-8.9,0.0,0.0,2.4\n2026-01-12,1,-9.0,0.0,0.0,3.1\n2026-01-12,2,-9.2,0.0,0.0,4.3\n2026-01-12,3,-10.0,0.0,0.0,6.4\n2026-01-12,4,-10.2,0.0,0.0,6.9\n2026-01-12,5,-10.3,0.0,0.0,8.2\n2026-01-12,6,-9.7,0.0,0.0,12.2\n2026-01-12,7,-9.4,0.0,0.0,10.2\n2026-01-12,8,-8.0,0.0,0.0,14.4\n2026-01-12,9,-7.2,0.0,0.0,11.8\n2026-01-12,10,-6.1,0.0,0.0,11.6\n2026-01-12,11,-4.3,0.0,0.0,13.5\n2026-01-12,12,-2.8,0.0,0.0,15.0\n2026-01-12,13,-1.4,0.0,0.0,14.2\n2026-01-12,14,-0.4,0.0,0.0,14.0\n2026-01-12,15,-1.4,0.0,0.0,14.4\n2026-01-12,16,-1.8,0.2,0.14,13.9\n2026-01-12,17,-1.8,0.2,0.14,14.5\n2026-01-12,18,-1.4,0.0,0.0,14.0\n2026-01-12,19,-0.8,0.1,0.07,12.0\n2026-01-12,20,-0.3,0.1,0.07,13.7\n2026-01-12,21,0.0,0.2,0.07,16.3\n2026-01-12,22,0.3,0.7,0.42,15.5\n2026-01-12,23,0.7,1.3,0.63,9.5\n2026-01-13,0,2.7,0.1,0.0,17.2\n2026-01-13,1,2.1,0.0,0.0,21.3\n2026-01-13,2,0.8,0.0,0.0,20.8\n2026-01-13,3,-0.3,0.0,0.0,16.4\n2026-01-13,4,-1.5,0.0,0.0,13.9\n2026-01-13,5,-2.5,0.0,0.0,14.3\n2026-01-13,6,-3.4,0.0,0.0,15.8\n2026-01-13,7,-4.3,0.0,0.0,14.5\n2026-01-13,8,-4.8,0.0,0.0,16.4\n2026-01-13,9,-3.2,0.0,0.0,17.7\n2026-01-13,10,-2.6,0.0,0.0,20.4\n2026-01-13,11,-2.6,0.0,0.0,19.1\n2026-01-13,12,-2.4,0.0,0.0,17.7\n2026-01-13,13,-2.0,0.0,0.0,23.7\n2026-01-13,14,-2.1,0.0,0.0,24.2\n2026-01-13,15,-2.6,0.0,0.0,21.9\n2026-01-13,16,-3.1,0.0,0.0,20.5\n2026-01-13,17,-3.8,0.0,0.0,15.9\n2026-01-13,18,-4.7,0.0,0.0,14.2\n2026-01-13,19,-5.3,0.0,0.0,13.4\n2026-01-13,20,-5.8,0.0,0.0,13.0\n2026-01-13,21,-6.3,0.0,0.0,13.8\n2026-01-13,22,-6.7,0.0,0.0,12.3\n2026-01-13,23,-7.3,0.0,0.0,10.0\n2026-01-14,0,-7.8,0.0,0.0,7.9\n2026-01-14,1,-8.1,0.0,0.0,6.3\n2026-01-14,2,-8.1,0.0,0.0,5.4\n2026-01-14,3,-8.7,0.0,0.0,4.7\n2026-01-14,4,-8.8,0.0,0.0,3.1\n2026-01-14,5,-9.0,0.0,0.0,4.2\n2026-01-14,6,-9.0,0.0,0.0,6.0\n2026-01-14,7,-9.1,0.0,0.0,7.6\n2026-01-14,8,-9.0,0.0,0.0,7.1\n2026-01-14,9,-8.4,0.0,0.0,6.3\n2026-01-14,10,-5.7,0.0,0.0,6.0\n2026-01-14,11,-3.5,0.0,0.0,6.0\n2026-01-14,12,-1.6,0.0,0.0,4.5\n2026-01-14,13,-0.1,0.0,0.0,4.2\n2026-01-14,14,1.0,0.0,0.0,4.0\n2026-01-14,15,0.6,0.0,0.0,3.3\n2026-01-14,16,0.9,0.0,0.0,5.9\n2026-01-14,17,0.3,0.0,0.0,6.1\n2026-01-14,18,-0.6,0.0,0.0,9.6\n2026-01-14,19,-0.4,0.0,0.0,12.3\n2026-01-14,20,0.5,0.0,0.0,11.9\n2026-01-14,21,0.5,0.0,0.0,10.6\n2026-01-14,22,0.5,0.1,0.0,12.8\n2026-01-14,23,0.8,0.0,0.0,13.6\n2026-01-15,0,1.5,0.0,0.0,12.2\n2026-01-15,1,2.3,0.0,0.0,12.6\n2026-01-15,2,2.9,0.0,0.0,13.2\n2026-01-15,3,2.1,0.0,0.0,11.7\n2026-01-15,4,2.6,0.0,0.0,12.0\n2026-01-15,5,3.3,0.0,0.0,14.2\n2026-01-15,6,2.8,0.0,0.0,13.4\n2026-01-15,7,2.9,0.0,0.0,11.7\n2026-01-15,8,3.0,0.0,0.0,11.2\n2026-01-15,9,2.5,0.0,0.0,10.7\n2026-01-15,10,4.7,0.0,0.0,8.0\n2026-01-15,11,6.1,0.0,0.0,5.6\n2026-01-15,12,7.0,0.0,0.0,10.5\n2026-01-15,13,7.4,0.0,0.0,11.3\n2026-01-15,14,7.4,0.0,0.0,11.4\n2026-01-15,15,7.5,0.0,0.0,9.9\n2026-01-15,16,6.8,0.0,0.0,10.4\n2026-01-15,17,4.7,0.0,0.0,11.2\n2026-01-15,18,2.6,0.0,0.0,11.8\n2026-01-15,19,1.5,0.0,0.0,10.8\n2026-01-15,20,1.0,0.0,0.0,8.8\n2026-01-15,21,0.9,0.0,0.0,8.3\n2026-01-15,22,0.7,0.0,0.0,8.9\n2026-01-15,23,-0.3,0.0,0.0,7.7\n2026-01-16,0,-2.0,0.0,0.0,4.5\n2026-01-16,1,-2.1,0.0,0.0,1.6\n2026-01-16,2,-2.1,0.0,0.0,0.2\n2026-01-16,3,-2.2,0.0,0.0,1.3\n2026-01-16,4,-2.2,0.0,0.0,2.6\n2026-01-16,5,-1.5,0.0,0.0,4.8\n2026-01-16,6,-1.3,0.0,0.0,4.7\n2026-01-16,7,-1.1,0.0,0.0,6.7\n2026-01-16,8,-1.0,0.0,0.0,7.7\n2026-01-16,9,-0.2,0.0,0.0,7.3\n2026-01-16,10,0.7,0.0,0.0,5.3\n2026-01-16,11,1.9,0.0,0.0,6.1\n2026-01-16,12,3.8,0.0,0.0,3.1\n2026-01-16,13,5.5,0.0,0.0,2.5\n2026-01-16,14,6.9,0.0,0.0,2.9\n2026-01-16,15,7.5,0.0,0.0,3.1\n2026-01-16,16,7.6,0.0,0.0,1.6\n2026-01-16,17,5.5,0.0,0.0,4.7\n2026-01-16,18,3.1,0.0,0.0,7.0\n2026-01-16,19,1.8,0.0,0.0,5.9\n2026-01-16,20,0.8,0.0,0.0,4.7\n2026-01-16,21,1.2,0.0,0.0,4.2\n2026-01-16,22,2.5,0.0,0.0,8.1\n2026-01-16,23,2.9,0.0,0.0,11.2\n2026-01-17,0,2.3,0.1,0.0,11.1\n2026-01-17,1,1.3,0.0,0.0,12.6\n2026-01-17,2,-1.1,0.0,0.0,15.2\n2026-01-17,3,-1.5,0.0,0.0,11.1\n2026-01-17,4,-2.3,0.0,0.0,12.7\n2026-01-17,5,-3.2,0.0,0.0,12.8\n2026-01-17,6,-3.9,0.0,0.0,13.9\n2026-01-17,7,-4.6,0.0,0.0,13.3\n2026-01-17,8,-5.3,0.0,0.0,14.0\n2026-01-17,9,-4.6,0.0,0.0,11.0\n2026-01-17,10,-3.4,0.0,0.0,9.4\n2026-01-17,11,-2.0,0.0,0.0,7.6\n2026-01-17,12,-0.6,0.0,0.0,7.9\n2026-01-17,13,0.7,0.0,0.0,7.7\n2026-01-17,14,1.7,0.0,0.0,10.5\n2026-01-17,15,2.1,0.0,0.0,6.9\n2026-01-17,16,2.2,0.0,0.0,7.5\n2026-01-17,17,1.3,0.0,0.0,6.8\n2026-01-17,18,-0.6,0.0,0.0,7.5\n2026-01-17,19,-1.5,0.0,0.0,8.3\n2026-01-17,20,-2.2,0.0,0.0,8.3\n2026-01-17,21,-3.0,0.0,0.0,7.5\n2026-01-17,22,-3.5,0.0,0.0,8.9\n2026-01-17,23,-4.2,0.0,0.0,9.9\n2026-01-18,0,-4.5,0.0,0.0,9.6\n2026-01-18,1,-4.5,0.0,0.0,8.1\n2026-01-18,2,-4.4,0.0,0.0,6.2\n2026-01-18,3,-4.4,0.0,0.0,8.0\n2026-01-18,4,-4.4,0.0,0.0,5.7\n2026-01-18,5,-4.1,0.0,0.0,5.8\n2026-01-18,6,-4.0,0.0,0.0,5.7\n2026-01-18,7,-2.5,0.0,0.0,4.8\n2026-01-18,8,-2.1,0.0,0.0,6.0\n2026-01-18,9,-2.2,0.0,0.0,6.8\n2026-01-18,10,-1.1,0.0,0.0,7.3\n2026-01-18,11,-0.6,0.0,0.0,7.9\n2026-01-18,12,0.7,0.0,0.0,7.2\n2026-01-18,13,2.3,0.0,0.0,7.1\n2026-01-18,14,2.9,0.0,0.0,6.9\n2026-01-18,15,2.0,0.0,0.0,8.4\n2026-01-18,16,2.9,0.0,0.0,10.3\n2026-01-18,17,1.9,0.0,0.0,10.6\n2026-01-18,18,0.7,0.0,0.0,11.9\n2026-01-18,19,0.3,0.0,0.0,12.5\n2026-01-18,20,-0.1,0.0,0.0,11.8\n2026-01-18,21,0.2,0.1,0.07,9.4\n2026-01-18,22,-0.1,0.2,0.14,6.4\n2026-01-18,23,-0.5,0.3,0.21,8.9\n2026-01-19,0,-0.8,0.3,0.21,9.2\n2026-01-19,1,-1.1,0.1,0.07,9.6\n2026-01-19,2,-1.6,0.3,0.21,11.5\n2026-01-19,3,-1.5,0.3,0.21,11.8\n2026-01-19,4,-2.3,0.6,0.42,11.0\n2026-01-19,5,-2.7,0.3,0.21,12.1\n2026-01-19,6,-3.0,0.2,0.14,12.2\n2026-01-19,7,-3.4,0.1,0.07,15.2\n2026-01-19,8,-4.3,0.0,0.0,17.7\n2026-01-19,9,-4.1,0.0,0.0,13.2\n2026-01-19,10,-4.1,0.0,0.0,14.1\n2026-01-19,11,-3.3,0.0,0.0,14.8\n2026-01-19,12,-2.8,0.0,0.0,18.1\n2026-01-19,13,-2.5,0.0,0.0,18.9\n2026-01-19,14,-2.5,0.0,0.0,18.7\n2026-01-19,15,-3.3,0.0,0.0,19.9\n2026-01-19,16,-4.0,0.0,0.0,21.1\n2026-01-19,17,-5.0,0.0,0.0,19.5\n2026-01-19,18,-6.7,0.0,0.0,22.6\n2026-01-19,19,-8.2,0.0,0.0,23.7\n2026-01-19,20,-9.2,0.0,0.0,22.8\n2026-01-19,21,-9.1,0.0,0.0,19.4\n2026-01-19,22,-9.5,0.0,0.0,17.9\n2026-01-19,23,-10.1,0.0,0.0,19.2\n2026-01-20,0,-10.7,0.0,0.0,20.0\n2026-01-20,1,-11.1,0.0,0.0,18.7\n2026-01-20,2,-11.6,0.0,0.0,16.8\n2026-01-20,3,-11.5,0.0,0.0,14.7\n2026-01-20,4,-11.7,0.0,0.0,15.0\n2026-01-20,5,-12.4,0.0,0.0,13.9\n2026-01-20,6,-13.1,0.0,0.0,13.5\n2026-01-20,7,-13.4,0.0,0.0,13.0\n2026-01-20,8,-13.6,0.0,0.0,11.7\n2026-01-20,9,-12.9,0.0,0.0,9.8\n2026-01-20,10,-11.8,0.0,0.0,9.5\n2026-01-20,11,-10.4,0.0,0.0,8.9\n2026-01-20,12,-9.1,0.0,0.0,9.4\n2026-01-20,13,-7.7,0.0,0.0,9.5\n2026-01-20,14,-6.5,0.0,0.0,10.4\n2026-01-20,15,-6.2,0.0,0.0,11.3\n2026-01-20,16,-6.1,0.0,0.0,12.0\n2026-01-20,17,-6.6,0.0,0.0,11.7\n2026-01-20,18,-7.4,0.0,0.0,10.9\n2026-01-20,19,-8.0,0.0,0.0,12.9\n2026-01-20,20,-8.8,0.0,0.0,14.0\n2026-01-20,21,-8.5,0.0,0.0,11.7\n2026-01-20,22,-8.9,0.0,0.0,13.5\n2026-01-20,23,-9.7,0.0,0.0,13.3\n2026-01-21,0,-10.2,0.0,0.0,12.7\n2026-01-21,1,-10.4,0.0,0.0,12.8\n2026-01-21,2,-10.7,0.0,0.0,12.3\n2026-01-21,3,-10.8,0.0,0.0,11.4\n2026-01-21,4,-11.0,0.0,0.0,12.0\n2026-01-21,5,-11.4,0.0,0.0,12.8\n2026-01-21,6,-12.0,0.0,0.0,13.2\n2026-01-21,7,-12.6,0.0,0.0,13.1\n2026-01-21,8,-13.0,0.0,0.0,13.0\n2026-01-21,9,-11.7,0.0,0.0,13.1\n2026-01-21,10,-10.5,0.0,0.0,14.1\n2026-01-21,11,-9.4,0.0,0.0,13.6\n2026-01-21,12,-8.2,0.0,0.0,13.9\n2026-01-21,13,-7.3,0.0,0.0,15.5\n2026-01-21,14,-6.5,0.0,0.0,17.1\n2026-01-21,15,-6.7,0.0,0.0,15.3\n2026-01-21,16,-6.5,0.0,0.0,16.2\n2026-01-21,17,-7.2,0.0,0.0,16.2\n2026-01-21,18,-8.2,0.0,0.0,15.4\n2026-01-21,19,-9.1,0.0,0.0,17.0\n2026-01-21,20,-10.4,0.0,0.0,17.6\n2026-01-21,21,-10.3,0.0,0.0,15.6\n2026-01-21,22,-10.8,0.0,0.0,16.5\n2026-01-21,23,-11.0,0.0,0.0,15.0\n2026-01-22,0,-11.2,0.0,0.0,15.0\n2026-01-22,1,-11.4,0.0,0.0,15.0\n2026-01-22,2,-11.6,0.0,0.0,15.3\n2026-01-22,3,-12.0,0.0,0.0,13.9\n2026-01-22,4,-12.4,0.0,0.0,14.4\n2026-01-22,5,-12.7,0.0,0.0,14.2\n2026-01-22,6,-13.1,0.0,0.0,14.9\n2026-01-22,7,-13.3,0.0,0.0,14.6\n2026-01-22,8,-13.5,0.0,0.0,13.2\n2026-01-22,9,-12.4,0.0,0.0,10.8\n2026-01-22,10,-10.7,0.0,0.0,10.7\n2026-01-22,11,-9.0,0.0,0.0,10.7\n2026-01-22,12,-7.5,0.0,0.0,11.8\n2026-01-22,13,-6.0,0.0,0.0,15.0\n2026-01-22,14,-5.4,0.0,0.0,16.9\n2026-01-22,15,-5.7,0.0,0.0,16.7\n2026-01-22,16,-5.7,0.0,0.0,16.5\n2026-01-22,17,-5.3,0.0,0.0,19.1\n2026-01-22,18,-6.7,0.0,0.0,11.5\n2026-01-22,19,-6.8,0.0,0.0,12.4\n2026-01-22,20,-7.3,0.0,0.0,13.8\n2026-01-22,21,-7.5,0.0,0.0,13.1\n2026-01-22,22,-8.4,0.0,0.0,12.4\n2026-01-22,23,-9.4,0.0,0.0,10.5\n2026-01-23,0,-10.1,0.0,0.0,8.7\n2026-01-23,1,-10.4,0.0,0.0,7.3\n2026-01-23,2,-10.1,0.0,0.0,7.1\n2026-01-23,3,-11.1,0.0,0.0,1.2\n2026-01-23,4,-11.9,0.0,0.0,2.6\n2026-01-23,5,-11.9,0.0,0.0,3.3\n2026-01-23,6,-11.6,0.0,0.0,4.7\n2026-01-23,7,-11.6,0.0,0.0,7.5\n2026-01-23,8,-11.8,0.0,0.0,4.4\n2026-01-23,9,-10.6,0.0,0.0,4.5\n2026-01-23,10,-8.4,0.0,0.0,3.2\n2026-01-23,11,-6.5,0.0,0.0,2.8\n2026-01-23,12,-5.1,0.0,0.0,2.2\n2026-01-23,13,-4.2,0.0,0.0,1.5\n2026-01-23,14,-2.1,0.0,0.0,1.7\n2026-01-23,15,-1.4,0.0,0.0,5.9\n2026-01-23,16,-1.5,0.0,0.0,7.5\n2026-01-23,17,-2.4,0.0,0.0,6.1\n2026-01-23,18,-4.4,0.0,0.0,1.3\n2026-01-23,19,-4.8,0.0,0.0,2.8\n2026-01-23,20,-3.7,0.0,0.0,9.6\n2026-01-23,21,-2.6,0.4,0.28,11.7\n2026-01-23,22,-2.2,0.8,0.56,7.2\n2026-01-23,23,-2.9,0.3,0.21,3.9\n2026-01-24,0,-3.0,0.0,0.0,12.9\n2026-01-24,1,-3.9,0.0,0.0,18.4\n2026-01-24,2,-6.2,0.0,0.0,16.8\n2026-01-24,3,-7.7,0.0,0.0,13.6\n2026-01-24,4,-8.8,0.0,0.0,11.2\n2026-01-24,5,-9.3,0.0,0.0,11.6\n2026-01-24,6,-10.3,0.0,0.0,11.8\n2026-01-24,7,-11.0,0.0,0.0,10.3\n2026-01-24,8,-11.6,0.0,0.0,7.8\n2026-01-24,9,-8.7,0.0,0.0,8.7\n2026-01-24,10,-6.3,0.0,0.0,11.7\n2026-01-24,11,-5.2,0.0,0.0,11.0\n2026-01-24,12,-4.2,0.0,0.0,11.8\n2026-01-24,13,-3.4,0.0,0.0,12.9\n2026-01-24,14,-3.1,0.0,0.0,15.8\n2026-01-24,15,-3.6,0.0,0.0,14.8\n2026-01-24,16,-3.5,0.0,0.0,15.3\n2026-01-24,17,-3.8,0.0,0.0,12.1\n2026-01-24,18,-4.8,0.0,0.0,7.6\n2026-01-24,19,-4.7,0.0,0.0,10.7\n2026-01-24,20,-4.3,0.0,0.0,17.3\n2026-01-24,21,-5.1,0.0,0.0,13.9\n2026-01-24,22,-6.1,0.0,0.0,14.1\n2026-01-24,23,-7.1,0.0,0.0,14.4\n2026-01-25,0,-8.0,0.0,0.0,11.9\n2026-01-25,1,-8.3,0.0,0.0,10.1\n2026-01-25,2,-7.9,0.0,0.0,12.2\n2026-01-25,3,-8.1,0.0,0.0,8.8\n2026-01-25,4,-8.0,0.0,0.0,12.5\n2026-01-25,5,-8.6,0.0,0.0,13.3\n2026-01-25,6,-9.3,0.0,0.0,13.4\n2026-01-25,7,-9.9,0.0,0.0,12.3\n2026-01-25,8,-10.4,0.0,0.0,11.9\n2026-01-25,9,-8.7,0.0,0.0,11.8\n2026-01-25,10,-7.4,0.0,0.0,14.8\n2026-01-25,11,-6.5,0.0,0.0,14.2\n2026-01-25,12,-5.1,0.0,0.0,9.7\n2026-01-25,13,-4.1,0.0,0.0,10.7\n2026-01-25,14,-3.6,0.0,0.0,11.8\n2026-01-25,15,-4.0,0.0,0.0,14.5\n2026-01-25,16,-4.3,0.0,0.0,13.5\n2026-01-25,17,-4.7,0.0,0.0,11.6\n2026-01-25,18,-5.8,0.0,0.0,4.7\n2026-01-25,19,-6.3,0.0,0.0,2.1\n2026-01-25,20,-6.6,0.0,0.0,1.7\n2026-01-25,21,-7.1,0.0,0.0,4.6\n2026-01-25,22,-7.1,0.0,0.0,8.4\n2026-01-25,23,-7.9,0.0,0.0,7.9\n2026-01-26,0,-8.6,0.0,0.0,7.3\n2026-01-26,1,-9.3,0.0,0.0,4.2\n2026-01-26,2,-10.1,0.0,0.0,3.3\n2026-01-26,3,-10.8,0.0,0.0,2.8\n2026-01-26,4,-10.9,0.0,0.0,1.9\n2026-01-26,5,-10.8,0.0,0.0,5.5\n2026-01-26,6,-10.6,0.0,0.0,6.5\n2026-01-26,7,-10.3,0.0,0.0,3.6\n2026-01-26,8,-10.2,0.0,0.0,4.6\n2026-01-26,9,-9.2,0.0,0.0,4.3\n2026-01-26,10,-6.9,0.0,0.0,4.0\n2026-01-26,11,-5.0,0.0,0.0,4.4\n2026-01-26,12,-3.8,0.0,0.0,4.8\n2026-01-26,13,-4.0,0.0,0.0,4.7\n2026-01-26,14,-3.7,0.0,0.0,4.1\n2026-01-26,15,-2.2,0.0,0.0,6.5\n2026-01-26,16,-2.0,0.0,0.0,6.9\n2026-01-26,17,-2.5,0.0,0.0,6.7\n2026-01-26,18,-3.7,0.0,0.0,6.5\n2026-01-26,19,-4.3,0.0,0.0,7.2\n2026-01-26,20,-4.5,0.0,0.0,6.5\n2026-01-26,21,-4.9,0.0,0.0,5.4\n2026-01-26,22,-2.5,0.0,0.0,7.9\n2026-01-26,23,-1.6,0.0,0.0,12.7\n2026-01-27,0,-2.3,0.0,0.0,13.6\n2026-01-27,1,-3.7,0.0,0.0,14.6\n2026-01-27,2,-5.5,0.0,0.0,12.3\n2026-01-27,3,-6.1,0.0,0.0,10.9\n2026-01-27,4,-5.6,0.0,0.0,11.6\n2026-01-27,5,-6.2,0.0,0.0,13.0\n2026-01-27,6,-6.6,0.0,0.0,14.7\n2026-01-27,7,-7.3,0.0,0.0,14.1\n2026-01-27,8,-7.7,0.0,0.0,13.0\n2026-01-27,9,-6.6,0.0,0.0,11.8\n2026-01-27,10,-5.0,0.0,0.0,14.9\n2026-01-27,11,-3.8,0.0,0.0,14.7\n2026-01-27,12,-2.8,0.0,0.0,14.0\n2026-01-27,13,-2.1,0.0,0.0,14.8\n2026-01-27,14,-1.6,0.0,0.0,15.4\n2026-01-27,15,-1.5,0.0,0.0,13.5\n2026-01-27,16,-1.8,0.0,0.0,14.5\n2026-01-27,17,-2.4,0.0,0.0,14.2\n2026-01-27,18,-3.5,0.0,0.0,12.0\n2026-01-27,19,-4.2,0.0,0.0,13.6\n2026-01-27,20,-5.1,0.0,0.0,16.5\n2026-01-27,21,-5.3,0.0,0.0,16.1\n2026-01-27,22,-6.4,0.0,0.0,16.4\n2026-01-27,23,-6.8,0.0,0.0,14.4\n2026-01-28,0,-6.9,0.0,0.0,14.4\n2026-01-28,1,-7.0,0.0,0.0,14.3\n2026-01-28,2,-7.2,0.0,0.0,14.3\n2026-01-28,3,-7.4,0.0,0.0,14.2\n2026-01-28,4,-7.8,0.0,0.0,14.0\n2026-01-28,5,-8.1,0.0,0.0,13.8\n2026-01-28,6,-8.3,0.0,0.0,14.2\n2026-01-28,7,-8.5,0.0,0.0,14.0\n2026-01-28,8,-8.5,0.0,0.0,13.9\n2026-01-28,9,-8.1,0.0,0.0,11.8\n2026-01-28,10,-6.6,0.0,0.0,13.9\n2026-01-28,11,-5.2,0.0,0.0,13.5\n2026-01-28,12,-3.9,0.0,0.0,13.3\n2026-01-28,13,-2.9,0.0,0.0,13.9\n2026-01-28,14,-2.3,0.0,0.0,14.6\n2026-01-28,15,-2.1,0.0,0.0,12.6\n2026-01-28,16,-2.1,0.0,0.0,11.7\n2026-01-28,17,-2.6,0.0,0.0,10.7\n2026-01-28,18,-3.8,0.0,0.0,10.5\n2026-01-28,19,-4.2,0.0,0.0,12.3\n2026-01-28,20,-4.0,0.0,0.0,10.3\n2026-01-28,21,-4.3,0.0,0.0,9.3\n2026-01-28,22,-4.4,0.0,0.0,11.4\n2026-01-28,23,-4.9,0.0,0.0,12.6\n2026-01-29,0,-5.8,0.0,0.0,12.0\n2026-01-29,1,-7.0,0.0,0.0,9.9\n2026-01-29,2,-7.7,0.0,0.0,7.9\n2026-01-29,3,-8.2,0.0,0.0,6.2\n2026-01-29,4,-8.6,0.0,0.0,3.5\n2026-01-29,5,-8.1,0.0,0.0,3.4\n2026-01-29,6,-8.7,0.0,0.0,4.2\n2026-01-29,7,-8.9,0.0,0.0,4.4\n2026-01-29,8,-8.8,0.0,0.0,6.9\n2026-01-29,9,-7.4,0.0,0.0,3.4\n2026-01-29,10,-5.4,0.0,0.0,7.7\n2026-01-29,11,-3.9,0.0,0.0,10.9\n2026-01-29,12,-2.8,0.0,0.0,11.7\n2026-01-29,13,-2.0,0.0,0.0,12.8\n2026-01-29,14,-1.7,0.0,0.0,14.6\n2026-01-29,15,-1.8,0.0,0.0,14.3\n2026-01-29,16,-2.0,0.0,0.0,15.2\n2026-01-29,17,-2.3,0.0,0.0,15.8\n2026-01-29,18,-3.1,0.0,0.0,14.8\n2026-01-29,19,-3.9,0.0,0.0,16.9\n2026-01-29,20,-5.0,0.0,0.0,18.2\n2026-01-29,21,-5.8,0.0,0.0,20.3\n2026-01-29,22,-6.7,0.0,0.0,17.3\n2026-01-29,23,-7.0,0.0,0.0,16.0\n2026-01-30,0,-7.2,0.0,0.0,12.2\n2026-01-30,1,-7.4,0.0,0.0,12.3\n2026-01-30,2,-7.5,0.0,0.0,13.4\n2026-01-30,3,-7.9,0.0,0.0,13.9\n2026-01-30,4,-8.4,0.0,0.0,13.5\n2026-01-30,5,-8.8,0.0,0.0,13.6\n2026-01-30,6,-9.0,0.0,0.0,13.4\n2026-01-30,7,-9.4,0.0,0.0,12.9\n2026-01-30,8,-9.6,0.0,0.0,12.3\n2026-01-30,9,-9.2,0.0,0.0,11.4\n2026-01-30,10,-8.1,0.0,0.0,9.6\n2026-01-30,11,-6.7,0.0,0.0,9.4\n2026-01-30,12,-5.3,0.0,0.0,9.2\n2026-01-30,13,-4.7,0.0,0.0,9.7\n2026-01-30,14,-3.7,0.0,0.0,10.5\n2026-01-30,15,-3.7,0.0,0.0,10.2\n2026-01-30,16,-3.4,0.0,0.0,9.7\n2026-01-30,17,-3.6,0.0,0.0,9.2\n2026-01-30,18,-4.7,0.0,0.0,8.2\n2026-01-30,19,-5.4,0.0,0.0,9.7\n2026-01-30,20,-6.0,0.0,0.0,10.6\n2026-01-30,21,-6.7,0.0,0.0,9.3\n2026-01-30,22,-7.3,0.0,0.0,10.8\n2026-01-30,23,-7.9,0.0,0.0,9.1\n2026-01-31,0,-8.4,0.0,0.0,8.2\n2026-01-31,1,-8.7,0.0,0.0,7.1\n2026-01-31,2,-8.9,0.0,0.0,6.8\n2026-01-31,3,-9.4,0.0,0.0,8.0\n2026-01-31,4,-9.9,0.0,0.0,7.5\n2026-01-31,5,-10.0,0.0,0.0,5.5\n2026-01-31,6,-10.1,0.0,0.0,5.5\n2026-01-31,7,-10.1,0.0,0.0,5.7\n2026-01-31,8,-10.1,0.0,0.0,6.3\n2026-01-31,9,-8.9,0.0,0.0,5.7\n2026-01-31,10,-6.9,0.0,0.0,4.7\n2026-01-31,11,-4.8,0.0,0.0,5.0\n2026-01-31,12,-3.0,0.0,0.0,6.3\n2026-01-31,13,-1.8,0.0,0.0,9.6\n2026-01-31,14,-1.1,0.0,0.0,11.2\n2026-01-31,15,-1.2,0.0,0.0,12.3\n2026-01-31,16,-1.4,0.0,0.0,13.0\n2026-01-31,17,-1.8,0.0,0.0,12.4\n2026-01-31,18,-2.7,0.0,0.0,9.9\n2026-01-31,19,-3.0,0.0,0.0,9.4\n2026-01-31,20,-1.9,0.0,0.0,11.2\n2026-01-31,21,-1.3,0.0,0.0,11.6\n2026-01-31,22,-1.9,0.0,0.0,9.3\n2026-01-31,23,-3.2,0.0,0.0,7.7\n2026-02-01,0,-2.6,0.0,0.0,8.9\n2026-02-01,1,-2.8,0.0,0.0,9.3\n2026-02-01,2,-3.4,0.0,0.0,11.0\n2026-02-01,3,-3.9,0.0,0.0,10.4\n2026-02-01,4,-5.6,0.0,0.0,10.1\n2026-02-01,5,-6.6,0.0,0.0,8.1\n2026-02-01,6,-7.1,0.0,0.0,6.2\n2026-02-01,7,-7.4,0.0,0.0,5.1\n2026-02-01,8,-7.4,0.0,0.0,4.6\n2026-02-01,9,-5.6,0.0,0.0,3.8\n2026-02-01,10,-3.5,0.0,0.0,5.2\n2026-02-01,11,-1.8,0.0,0.0,6.5\n2026-02-01,12,-0.6,0.0,0.0,7.6\n2026-02-01,13,0.2,0.0,0.0,10.8\n2026-02-01,14,0.2,0.0,0.0,13.3\n2026-02-01,15,0.1,0.0,0.0,13.5\n2026-02-01,16,-0.3,0.0,0.0,12.2\n2026-02-01,17,-1.0,0.0,0.0,8.8\n2026-02-01,18,-3.0,0.0,0.0,3.2\n2026-02-01,19,-2.0,0.0,0.0,3.5\n2026-02-01,20,-1.7,0.0,0.0,6.6\n2026-02-01,21,-1.8,0.0,0.0,6.9\n2026-02-01,22,-2.2,0.0,0.0,9.0\n2026-02-01,23,-2.6,0.0,0.0,10.8\n2026-02-02,0,-1.6,0.1,0.07,4.1\n2026-02-02,1,-1.8,0.4,0.28,4.0\n2026-02-02,2,-3.0,0.2,0.14,12.5\n2026-02-02,3,-2.8,0.1,0.07,6.8\n2026-02-02,4,-3.3,0.1,0.07,7.4\n2026-02-02,5,-3.5,0.0,0.0,7.5\n2026-02-02,6,-4.8,0.0,0.0,10.5\n2026-02-02,7,-5.3,0.0,0.0,11.4\n2026-02-02,8,-4.8,0.0,0.0,10.1\n2026-02-02,9,-4.3,0.0,0.0,13.0\n2026-02-02,10,-2.5,0.0,0.0,10.9\n2026-02-02,11,-1.2,0.0,0.0,14.6\n2026-02-02,12,-0.7,0.0,0.0,15.3\n2026-02-02,13,-0.4,0.0,0.0,15.0\n2026-02-02,14,-0.1,0.0,0.0,14.9\n2026-02-02,15,-0.6,0.0,0.0,13.2\n2026-02-02,16,-0.5,0.0,0.0,13.4\n2026-02-02,17,-1.2,0.0,0.0,14.0\n2026-02-02,18,-2.1,0.0,0.0,11.4\n2026-02-02,19,-2.4,0.0,0.0,9.9\n2026-02-02,20,-2.2,0.0,0.0,12.4\n2026-02-02,21,-2.6,0.0,0.0,12.0\n2026-02-02,22,-3.3,0.0,0.0,12.3\n2026-02-02,23,-4.0,0.0,0.0,11.0\n2026-02-03,0,-4.6,0.0,0.0,9.1\n2026-02-03,1,-5.0,0.0,0.0,7.7\n2026-02-03,2,-5.0,0.0,0.0,7.0\n2026-02-03,3,-5.6,0.0,0.0,6.5\n2026-02-03,4,-6.1,0.0,0.0,3.5\n2026-02-03,5,-6.7,0.0,0.0,2.2\n2026-02-03,6,-7.0,0.0,0.0,1.7\n2026-02-03,7,-7.1,0.0,0.0,2.0\n2026-02-03,8,-7.3,0.0,0.0,3.8\n2026-02-03,9,-6.0,0.0,0.0,5.6\n2026-02-03,10,-3.1,0.0,0.0,4.2\n2026-02-03,11,-0.8,0.0,0.0,3.4\n2026-02-03,12,1.0,0.0,0.0,3.0\n2026-02-03,13,2.3,0.0,0.0,4.8\n2026-02-03,14,2.9,0.0,0.0,8.3\n2026-02-03,15,3.2,0.0,0.0,12.4\n2026-02-03,16,2.5,0.0,0.0,9.6\n2026-02-03,17,1.8,0.0,0.0,4.3\n2026-02-03,18,-0.2,0.0,0.0,0.9\n2026-02-03,19,-0.6,0.0,0.0,0.9\n2026-02-03,20,-1.0,0.0,0.0,1.4\n2026-02-03,21,-0.2,0.0,0.0,0.4\n2026-02-03,22,-0.1,0.0,0.0,2.7\n2026-02-03,23,-0.9,0.0,0.0,7.8\n2026-02-04,0,-1.3,0.0,0.0,8.4\n2026-02-04,1,-1.4,0.0,0.0,5.8\n2026-02-04,2,-1.9,0.0,0.0,8.2\n2026-02-04,3,-1.8,0.0,0.0,8.9\n2026-02-04,4,-1.2,0.0,0.0,9.2\n2026-02-04,5,-1.1,0.0,0.0,9.8\n2026-02-04,6,-1.4,0.0,0.0,10.7\n2026-02-04,7,-1.6,0.0,0.0,11.5\n2026-02-04,8,-1.4,0.0,0.0,11.8\n2026-02-04,9,0.7,0.0,0.0,11.1\n2026-02-04,10,2.5,0.0,0.0,9.5\n2026-02-04,11,4.7,0.0,0.0,9.6\n2026-02-04,12,5.1,0.0,0.0,9.4\n2026-02-04,13,5.0,0.0,0.0,8.5\n2026-02-04,14,6.8,0.0,0.0,12.7\n2026-02-04,15,6.3,0.0,0.0,11.7\n2026-02-04,16,6.0,0.0,0.0,8.9\n2026-02-04,17,5.1,0.0,0.0,4.3\n2026-02-04,18,2.2,0.0,0.0,4.0\n2026-02-04,19,1.2,0.0,0.0,5.1\n2026-02-04,20,1.6,0.0,0.0,8.2\n2026-02-04,21,0.6,0.0,0.0,7.0\n2026-02-04,22,0.4,0.0,0.0,7.9\n2026-02-04,23,0.2,0.0,0.0,8.2\n2026-02-05,0,0.1,0.0,0.0,8.1\n2026-02-05,1,0.3,0.0,0.0,7.9\n2026-02-05,2,0.4,0.0,0.0,9.0\n2026-02-05,3,0.1,0.0,0.0,8.3\n2026-02-05,4,0.5,0.0,0.0,7.3\n2026-02-05,5,0.9,0.0,0.0,7.3\n2026-02-05,6,1.4,0.0,0.0,6.6\n2026-02-05,7,1.7,0.0,0.0,7.8\n2026-02-05,8,1.7,0.0,0.0,7.8\n2026-02-05,9,2.2,0.0,0.0,6.5\n2026-02-05,10,3.1,0.0,0.0,6.6\n2026-02-05,11,5.2,0.0,0.0,6.9\n2026-02-05,12,6.6,0.0,0.0,8.7\n2026-02-05,13,7.6,0.0,0.0,14.1\n2026-02-05,14,7.5,0.0,0.0,16.7\n2026-02-05,15,6.4,0.0,0.0,15.7\n2026-02-05,16,6.2,0.0,0.0,13.1\n2026-02-05,17,4.9,0.0,0.0,14.3\n2026-02-05,18,3.6,0.0,0.0,19.2\n2026-02-05,19,1.7,0.0,0.0,17.7\n2026-02-05,20,-0.6,0.0,0.0,19.9\n2026-02-05,21,-0.8,0.0,0.0,19.2\n2026-02-05,22,-2.0,0.0,0.0,20.2\n2026-02-05,23,-3.2,0.0,0.0,19.2\n2026-02-06,0,-4.0,0.0,0.0,19.3\n2026-02-06,1,-4.7,0.0,0.0,19.8\n2026-02-06,2,-5.7,0.0,0.0,21.3\n2026-02-06,3,-5.4,0.0,0.0,20.3\n2026-02-06,4,-6.6,0.0,0.0,19.5\n2026-02-06,5,-7.6,0.0,0.0,18.8\n2026-02-06,6,-8.4,0.0,0.0,18.0\n2026-02-06,7,-9.2,0.0,0.0,17.4\n2026-02-06,8,-9.5,0.0,0.0,17.8\n2026-02-06,9,-8.5,0.0,0.0,14.3\n2026-02-06,10,-8.0,0.0,0.0,15.1\n2026-02-06,11,-6.9,0.0,0.0,15.2\n2026-02-06,12,-6.0,0.0,0.0,15.3\n2026-02-06,13,-5.1,0.0,0.0,14.5\n2026-02-06,14,-4.4,0.0,0.0,14.2\n2026-02-06,15,-4.1,0.0,0.0,11.1\n2026-02-06,16,-4.1,0.0,0.0,11.3\n2026-02-06,17,-4.7,0.0,0.0,12.0\n2026-02-06,18,-5.8,0.0,0.0,13.1\n2026-02-06,19,-6.7,0.0,0.0,15.7\n2026-02-06,20,-7.4,0.0,0.0,16.9\n2026-02-06,21,-7.5,0.0,0.0,13.6\n2026-02-06,22,-8.0,0.0,0.0,13.4\n2026-02-06,23,-8.5,0.0,0.0,14.0\n2026-02-07,0,-8.9,0.0,0.0,14.2\n2026-02-07,1,-9.1,0.0,0.0,14.8\n2026-02-07,2,-9.3,0.0,0.0,14.3\n2026-02-07,3,-9.9,0.0,0.0,12.0\n2026-02-07,4,-10.2,0.0,0.0,11.3\n2026-02-07,5,-10.2,0.0,0.0,12.5\n2026-02-07,6,-10.4,0.0,0.0,11.9\n2026-02-07,7,-10.5,0.0,0.0,12.2\n2026-02-07,8,-10.9,0.0,0.0,12.2\n2026-02-07,9,-10.8,0.0,0.0,10.0\n2026-02-07,10,-9.2,0.0,0.0,11.1\n2026-02-07,11,-7.6,0.0,0.0,12.3\n2026-02-07,12,-6.3,0.0,0.0,14.5\n2026-02-07,13,-5.7,0.0,0.0,17.2\n2026-02-07,14,-5.1,0.0,0.0,17.6\n2026-02-07,15,-5.3,0.0,0.0,16.5\n2026-02-07,16,-5.1,0.0,0.0,19.8\n2026-02-07,17,-6.0,0.0,0.0,22.3\n2026-02-07,18,-7.0,0.0,0.0,19.5\n2026-02-07,19,-7.6,0.0,0.0,18.6\n2026-02-07,20,-8.7,0.0,0.0,20.5\n2026-02-07,21,-8.9,0.0,0.0,18.6\n2026-02-07,22,-9.7,0.0,0.0,19.0\n2026-02-07,23,-10.0,0.0,0.0,16.1\n2026-02-08,0,-10.2,0.0,0.0,16.8\n2026-02-08,1,-10.2,0.0,0.0,17.8\n2026-02-08,2,-10.5,0.0,0.0,18.8\n2026-02-08,3,-10.7,0.0,0.0,16.2\n2026-02-08,4,-11.2,0.0,0.0,16.3\n2026-02-08,5,-11.5,0.0,0.0,15.9\n2026-02-08,6,-11.9,0.0,0.0,16.5\n2026-02-08,7,-12.2,0.0,0.0,16.0\n2026-02-08,8,-12.5,0.0,0.0,14.8\n2026-02-08,9,-11.9,0.0,0.0,13.6\n2026-02-08,10,-10.7,0.0,0.0,13.4\n2026-02-08,11,-9.2,0.0,0.0,12.6\n2026-02-08,12,-7.7,0.0,0.0,12.3\n2026-02-08,13,-6.2,0.0,0.0,12.9\n2026-02-08,14,-4.9,0.0,0.0,13.5\n2026-02-08,15,-5.2,0.0,0.0,11.3\n2026-02-08,16,-5.2,0.0,0.0,14.1\n2026-02-08,17,-5.2,0.0,0.0,13.9\n2026-02-08,18,-5.3,0.0,0.0,12.4\n2026-02-08,19,-5.6,0.0,0.0,12.7\n2026-02-08,20,-6.2,0.0,0.0,14.6\n2026-02-08,21,-6.4,0.0,0.0,13.9\n2026-02-08,22,-7.0,0.0,0.0,13.8\n2026-02-08,23,-7.8,0.0,0.0,12.3\n2026-02-09,0,-8.4,0.0,0.0,10.5\n2026-02-09,1,-8.8,0.0,0.0,7.7\n2026-02-09,2,-9.2,0.0,0.0,3.3\n2026-02-09,3,-9.8,0.0,0.0,2.8\n2026-02-09,4,-10.0,0.0,0.0,2.9\n2026-02-09,5,-9.8,0.0,0.0,3.9\n2026-02-09,6,-9.9,0.0,0.0,5.8\n2026-02-09,7,-10.0,0.0,0.0,6.4\n2026-02-09,8,-9.8,0.0,0.0,8.2\n2026-02-09,9,-8.2,0.0,0.0,6.5\n2026-02-09,10,-5.2,0.0,0.0,5.6\n2026-02-09,11,-2.7,0.0,0.0,2.8\n2026-02-09,12,-0.8,0.0,0.0,1.6\n2026-02-09,13,0.5,0.0,0.0,3.8\n2026-02-09,14,1.2,0.0,0.0,5.0\n2026-02-09,15,1.4,0.0,0.0,9.4\n2026-02-09,16,1.4,0.0,0.0,8.6\n2026-02-09,17,0.8,0.0,0.0,6.6\n2026-02-09,18,-1.6,0.0,0.0,3.3\n2026-02-09,19,-2.1,0.0,0.0,2.9\n2026-02-09,20,-1.9,0.0,0.0,1.8\n2026-02-09,21,-2.5,0.0,0.0,0.6\n2026-02-09,22,-2.6,0.0,0.0,2.1\n2026-02-09,23,-2.9,0.0,0.0,3.7\n2026-02-10,0,-2.7,0.0,0.0,7.3\n2026-02-10,1,-2.5,0.0,0.0,9.8\n2026-02-10,2,-2.0,0.0,0.0,10.1\n2026-02-10,3,-2.5,0.0,0.0,8.0\n2026-02-10,4,-2.6,0.0,0.0,8.3\n2026-02-10,5,-2.4,0.0,0.0,8.8\n2026-02-10,6,-2.2,0.0,0.0,9.1\n2026-02-10,7,-1.9,0.0,0.0,10.4\n2026-02-10,8,-1.6,0.0,0.0,9.8\n2026-02-10,9,-1.2,0.0,0.0,9.2\n2026-02-10,10,-0.4,0.1,0.07,10.2\n2026-02-10,11,1.0,0.1,0.07,8.9\n2026-02-10,12,3.3,0.0,0.0,9.4\n2026-02-10,13,4.6,0.0,0.0,9.2\n2026-02-10,14,5.4,0.0,0.0,10.9\n2026-02-10,15,4.5,0.0,0.0,7.6\n2026-02-10,16,3.2,0.1,0.0,5.4\n2026-02-10,17,2.2,0.1,0.0,4.0\n2026-02-10,18,1.4,0.1,0.0,3.9\n2026-02-10,19,1.5,0.0,0.0,6.3\n2026-02-10,20,0.8,0.0,0.0,5.2\n2026-02-10,21,0.6,0.0,0.0,3.7\n2026-02-10,22,0.2,0.0,0.0,2.0\n2026-02-10,23,1.0,0.0,0.0,0.7\n2026-02-11,0,0.9,0.0,0.0,3.5\n2026-02-11,1,-0.4,0.0,0.0,6.0\n2026-02-11,2,-0.5,0.0,0.0,4.3\n2026-02-11,3,0.1,0.0,0.0,1.3\n2026-02-11,4,0.3,0.0,0.0,4.8\n2026-02-11,5,0.3,0.0,0.0,5.4\n2026-02-11,6,0.3,0.0,0.0,6.3\n2026-02-11,7,0.4,0.0,0.0,5.7\n2026-02-11,8,0.5,0.0,0.0,3.1\n2026-02-11,9,0.6,0.0,0.0,3.9\n2026-02-11,10,1.6,0.0,0.0,6.6\n2026-02-11,11,1.8,0.0,0.0,7.9\n2026-02-11,12,2.5,0.0,0.0,8.9\n2026-02-11,13,3.7,0.1,0.0,11.9\n2026-02-11,14,4.2,0.1,0.0,13.9\n2026-02-11,15,4.4,0.0,0.0,12.0\n2026-02-11,16,3.9,0.0,0.0,12.6\n2026-02-11,17,2.7,0.0,0.0,12.4\n2026-02-11,18,0.9,0.0,0.0,10.1\n2026-02-11,19,-0.1,0.0,0.0,10.4\n2026-02-11,20,-0.2,0.0,0.0,9.7\n2026-02-11,21,-0.3,0.0,0.0,9.6\n2026-02-11,22,-0.4,0.0,0.0,8.1\n2026-02-11,23,-0.9,0.0,0.0,6.5\n2026-02-12,0,-1.3,0.0,0.0,5.9\n2026-02-12,1,-1.6,0.0,0.0,6.7\n2026-02-12,2,-1.8,0.0,0.0,6.1\n2026-02-12,3,-2.8,0.0,0.0,4.5\n2026-02-12,4,-3.0,0.0,0.0,1.1\n2026-02-12,5,-3.2,0.0,0.0,1.5\n2026-02-12,6,-3.2,0.0,0.0,3.1\n2026-02-12,7,-3.2,0.0,0.0,2.6\n2026-02-12,8,-2.8,0.0,0.0,6.6\n2026-02-12,9,-1.0,0.0,0.0,7.6\n2026-02-12,10,1.8,0.0,0.0,6.3\n2026-02-12,11,4.0,0.0,0.0,6.6\n2026-02-12,12,5.6,0.0,0.0,9.1\n2026-02-12,13,6.4,0.0,0.0,11.4\n2026-02-12,14,6.7,0.0,0.0,12.4\n2026-02-12,15,6.5,0.0,0.0,12.6\n2026-02-12,16,6.2,0.0,0.0,11.5\n2026-02-12,17,5.3,0.0,0.0,9.0\n2026-02-12,18,3.4,0.0,0.0,10.8\n2026-02-12,19,2.5,0.0,0.0,13.0\n2026-02-12,20,1.8,0.0,0.0,12.6\n2026-02-12,21,1.0,0.0,0.0,10.0\n2026-02-12,22,1.0,0.0,0.0,9.2\n2026-02-12,23,0.5,0.0,0.0,9.0\n2026-02-13,0,0.0,0.0,0.0,8.6\n2026-02-13,1,-0.2,0.0,0.0,8.9\n2026-02-13,2,-0.3,0.0,0.0,9.2\n2026-02-13,3,0.6,0.0,0.0,8.4\n2026-02-13,4,1.0,0.0,0.0,9.0\n2026-02-13,5,1.2,0.0,0.0,10.1\n2026-02-13,6,1.1,0.0,0.0,10.2\n2026-02-13,7,0.9,0.0,0.0,10.3\n2026-02-13,8,0.5,0.0,0.0,11.5\n2026-02-13,9,2.2,0.0,0.0,10.5\n2026-02-13,10,3.6,0.0,0.0,11.2\n2026-02-13,11,5.5,0.0,0.0,10.8\n2026-02-13,12,7.2,0.0,0.0,11.0\n2026-02-13,13,8.4,0.0,0.0,11.8\n2026-02-13,14,9.1,0.0,0.0,12.6\n2026-02-13,15,8.6,0.0,0.0,11.3\n2026-02-13,16,8.6,0.0,0.0,9.2\n2026-02-13,17,7.7,0.0,0.0,6.0\n2026-02-13,18,5.3,0.0,0.0,2.7\n2026-02-13,19,3.7,0.0,0.0,5.5\n2026-02-13,20,4.0,0.0,0.0,10.5\n2026-02-13,21,2.8,0.0,0.0,5.9\n2026-02-13,22,2.8,0.0,0.0,6.8\n2026-02-13,23,2.1,0.0,0.0,8.2\n2026-02-14,0,1.6,0.0,0.0,8.3\n2026-02-14,1,1.3,0.0,0.0,8.1\n2026-02-14,2,1.0,0.0,0.0,7.6\n2026-02-14,3,0.4,0.0,0.0,6.5\n2026-02-14,4,0.8,0.0,0.0,5.7\n2026-02-14,5,1.5,0.0,0.0,5.9\n2026-02-14,6,1.6,0.0,0.0,7.1\n2026-02-14,7,1.7,0.0,0.0,7.6\n2026-02-14,8,1.8,0.0,0.0,7.9\n2026-02-14,9,2.9,0.0,0.0,6.6\n2026-02-14,10,5.3,0.0,0.0,5.5\n2026-02-14,11,7.1,0.0,0.0,5.5\n2026-02-14,12,7.8,0.0,0.0,3.7\n2026-02-14,13,9.4,0.0,0.0,3.0\n2026-02-14,14,9.0,0.0,0.0,7.0\n2026-02-14,15,10.7,0.0,0.0,6.9\n2026-02-14,16,10.0,0.0,0.0,6.6\n2026-02-14,17,9.3,0.0,0.0,4.1\n2026-02-14,18,7.3,0.0,0.0,3.8\n2026-02-14,19,5.5,0.0,0.0,1.7\n2026-02-14,20,6.0,0.0,0.0,1.3\n2026-02-14,21,4.2,0.0,0.0,3.8\n2026-02-14,22,5.1,0.0,0.0,2.7\n2026-02-14,23,5.0,0.0,0.0,3.1\n2026-02-15,0,4.8,0.0,0.0,1.9\n2026-02-15,1,4.2,0.0,0.0,2.2\n2026-02-15,2,4.4,0.0,0.0,4.4\n2026-02-15,3,3.8,0.0,0.0,3.2\n2026-02-15,4,3.9,0.0,0.0,6.6\n2026-02-15,5,3.8,0.0,0.0,7.1\n2026-02-15,6,3.6,0.0,0.0,9.6\n2026-02-15,7,3.2,0.0,0.0,10.0\n2026-02-15,8,2.8,0.0,0.0,8.0\n2026-02-15,9,3.8,0.0,0.0,7.5\n2026-02-15,10,5.8,0.0,0.0,12.6\n2026-02-15,11,7.5,0.0,0.0,12.0\n2026-02-15,12,8.7,0.0,0.0,11.2\n2026-02-15,13,9.4,0.0,0.0,13.2\n2026-02-15,14,8.6,0.0,0.0,17.9\n2026-02-15,15,7.5,0.0,0.0,19.3\n2026-02-15,16,6.7,0.0,0.0,17.2\n2026-02-15,17,5.6,0.0,0.0,15.0\n2026-02-15,18,4.0,0.0,0.0,13.8\n2026-02-15,19,2.6,0.0,0.0,16.8\n2026-02-15,20,1.2,0.0,0.0,17.4\n2026-02-15,21,0.5,0.0,0.0,16.8\n2026-02-15,22,-0.3,0.0,0.0,15.5\n2026-02-15,23,-1.0,0.0,0.0,14.2\n2026-02-16,0,-1.5,0.0,0.0,13.0\n2026-02-16,1,-1.9,0.0,0.0,12.0\n2026-02-16,2,-2.2,0.0,0.0,12.8\n2026-02-16,3,-1.9,0.0,0.0,14.1\n2026-02-16,4,-1.9,0.0,0.0,12.9\n2026-02-16,5,-2.2,0.0,0.0,11.1\n2026-02-16,6,-2.7,0.0,0.0,10.8\n2026-02-16,7,-3.2,0.0,0.0,12.0\n2026-02-16,8,-3.4,0.0,0.0,11.6\n2026-02-16,9,-2.0,0.0,0.0,9.7\n2026-02-16,10,-0.2,0.0,0.0,8.5\n2026-02-16,11,1.8,0.0,0.0,7.2\n2026-02-16,12,3.6,0.0,0.0,8.4\n2026-02-16,13,5.0,0.0,0.0,9.1\n2026-02-16,14,6.0,0.0,0.0,9.9\n2026-02-16,15,5.8,0.0,0.0,9.4\n2026-02-16,16,5.7,0.0,0.0,9.7\n2026-02-16,17,4.7,0.0,0.0,10.7\n2026-02-16,18,2.7,0.0,0.0,11.4\n2026-02-16,19,1.5,0.0,0.0,11.3\n2026-02-16,20,0.9,0.0,0.0,12.3\n2026-02-16,21,-0.3,0.0,0.0,11.3\n2026-02-16,22,-1.4,0.0,0.0,9.6\n2026-02-16,23,-2.3,0.0,0.0,9.4\n2026-02-17,0,-2.9,0.0,0.0,8.4\n2026-02-17,1,-3.3,0.0,0.0,6.9\n2026-02-17,2,-3.6,0.0,0.0,6.0\n2026-02-17,3,-3.6,0.0,0.0,4.4\n2026-02-17,4,-3.8,0.0,0.0,7.0\n2026-02-17,5,-3.9,0.0,0.0,7.0\n2026-02-17,6,-4.0,0.0,0.0,5.0\n2026-02-17,7,-4.2,0.0,0.0,3.5\n2026-02-17,8,-3.5,0.0,0.0,3.4\n2026-02-17,9,-2.1,0.0,0.0,1.7\n2026-02-17,10,0.4,0.0,0.0,1.0\n2026-02-17,11,2.4,0.0,0.0,2.5\n2026-02-17,12,3.8,0.0,0.0,6.6\n2026-02-17,13,4.2,0.0,0.0,10.8\n2026-02-17,14,3.6,0.0,0.0,13.8\n2026-02-17,15,3.4,0.0,0.0,13.9\n2026-02-17,16,2.9,0.0,0.0,13.8\n2026-02-17,17,2.2,0.0,0.0,13.3\n2026-02-17,18,0.8,0.0,0.0,11.0\n2026-02-17,19,-0.1,0.0,0.0,10.8\n2026-02-17,20,-0.1,0.0,0.0,9.8\n2026-02-17,21,0.3,0.0,0.0,11.8\n2026-02-17,22,0.7,0.0,0.0,12.2\n2026-02-17,23,0.4,0.0,0.0,11.7\n2026-02-18,0,0.4,0.0,0.0,12.8\n2026-02-18,1,0.9,0.0,0.0,14.8\n2026-02-18,2,1.4,0.0,0.0,15.9\n2026-02-18,3,1.4,0.0,0.0,14.8\n2026-02-18,4,1.3,0.0,0.0,13.1\n2026-02-18,5,1.2,0.0,0.0,11.7\n2026-02-18,6,0.7,0.0,0.0,9.1\n2026-02-18,7,0.3,0.0,0.0,9.4\n2026-02-18,8,0.2,0.0,0.0,11.9\n2026-02-18,9,1.1,0.0,0.0,7.0\n2026-02-18,10,3.4,0.0,0.0,12.4\n2026-02-18,11,4.5,0.0,0.0,15.7\n2026-02-18,12,5.1,0.0,0.0,16.1\n2026-02-18,13,5.5,0.0,0.0,16.5\n2026-02-18,14,5.5,0.0,0.0,15.8\n2026-02-18,15,5.1,0.0,0.0,15.3\n2026-02-18,16,4.4,0.0,0.0,17.2\n2026-02-18,17,3.3,0.0,0.0,15.3\n2026-02-18,18,1.4,0.0,0.0,13.3\n2026-02-18,19,0.4,0.0,0.0,11.0\n2026-02-18,20,-0.3,0.0,0.0,10.7\n2026-02-18,21,-0.5,0.0,0.0,10.6\n2026-02-18,22,-1.1,0.0,0.0,8.5\n2026-02-18,23,-1.9,0.0,0.0,5.6\n2026-02-19,0,-2.3,0.0,0.0,4.4\n2026-02-19,1,-2.4,0.0,0.0,6.2\n2026-02-19,2,-2.3,0.0,0.0,6.2\n2026-02-19,3,-2.6,0.0,0.0,5.2\n2026-02-19,4,-3.0,0.0,0.0,5.0\n2026-02-19,5,-3.6,0.0,0.0,4.2\n2026-02-19,6,-4.0,0.0,0.0,2.3\n2026-02-19,7,-3.9,0.0,0.0,2.9\n2026-02-19,8,-3.4,0.0,0.0,6.8\n2026-02-19,9,-1.7,0.0,0.0,5.6\n2026-02-19,10,0.8,0.0,0.0,5.8\n2026-02-19,11,2.5,0.0,0.0,8.2\n2026-02-19,12,3.5,0.0,0.0,10.2\n2026-02-19,13,4.2,0.0,0.0,10.6\n2026-02-19,14,4.2,0.0,0.0,11.3\n2026-02-19,15,3.9,0.0,0.0,12.3\n2026-02-19,16,3.4,0.0,0.0,10.0\n2026-02-19,17,2.8,0.0,0.0,6.4\n2026-02-19,18,0.6,0.0,0.0,2.1\n2026-02-19,19,-0.5,0.0,0.0,2.4\n2026-02-19,20,-0.6,0.0,0.0,2.2\n2026-02-19,21,-0.4,0.0,0.0,2.0\n2026-02-19,22,-0.1,0.0,0.0,4.2\n2026-02-19,23,-0.5,0.0,0.0,6.6\n2026-02-20,0,-1.0,0.0,0.0,7.4\n2026-02-20,1,-1.2,0.0,0.0,8.0\n2026-02-20,2,-1.3,0.0,0.0,10.0\n2026-02-20,3,-1.7,0.0,0.0,10.0\n2026-02-20,4,-1.7,0.0,0.0,11.4\n2026-02-20,5,-1.6,0.0,0.0,11.7\n2026-02-20,6,-1.4,0.0,0.0,11.8\n2026-02-20,7,-1.2,0.0,0.0,12.0\n2026-02-20,8,-0.2,0.0,0.0,13.0\n2026-02-20,9,1.5,0.0,0.0,12.1\n2026-02-20,10,4.4,0.0,0.0,11.8\n2026-02-20,11,6.8,0.0,0.0,11.9\n2026-02-20,12,8.8,0.0,0.0,11.7\n2026-02-20,13,10.1,0.0,0.0,12.8\n2026-02-20,14,10.9,0.0,0.0,14.0\n2026-02-20,15,10.7,0.0,0.0,13.2\n2026-02-20,16,10.2,0.0,0.0,12.9\n2026-02-20,17,9.1,0.0,0.0,10.6\n2026-02-20,18,6.9,0.0,0.0,12.5\n2026-02-20,19,5.0,0.0,0.0,13.4\n2026-02-20,20,4.0,0.0,0.0,13.4\n2026-02-20,21,3.7,0.0,0.0,13.2\n2026-02-20,22,3.9,0.0,0.0,13.6\n2026-02-20,23,3.6,0.0,0.0,14.4\n2026-02-21,0,3.4,0.0,0.0,14.6\n2026-02-21,1,3.1,0.0,0.0,14.8\n2026-02-21,2,3.0,0.0,0.0,14.9\n2026-02-21,3,2.0,0.0,0.0,13.7\n2026-02-21,4,2.1,0.0,0.0,14.4\n2026-02-21,5,2.3,0.0,0.0,15.8\n2026-02-21,6,2.6,0.0,0.0,16.9\n2026-02-21,7,2.9,0.0,0.0,17.4\n2026-02-21,8,3.6,0.0,0.0,17.3\n2026-02-21,9,5.0,0.0,0.0,16.9\n2026-02-21,10,7.8,0.0,0.0,15.1\n2026-02-21,11,10.5,0.0,0.0,17.9\n2026-02-21,12,12.8,0.0,0.0,23.0\n2026-02-21,13,14.0,0.0,0.0,25.7\n2026-02-21,14,14.6,0.0,0.0,26.6\n2026-02-21,15,15.0,0.0,0.0,25.5\n2026-02-21,16,14.4,0.0,0.0,23.9\n2026-02-21,17,13.2,0.0,0.0,24.6\n2026-02-21,18,11.1,0.0,0.0,26.3\n2026-02-21,19,9.8,0.0,0.0,25.8\n2026-02-21,20,9.2,0.0,0.0,24.0\n2026-02-21,21,9.4,0.0,0.0,19.6\n2026-02-21,22,9.6,0.0,0.0,19.9\n2026-02-21,23,9.6,0.0,0.0,20.2\n2026-02-22,0,9.4,0.0,0.0,21.3\n2026-02-22,1,9.1,0.0,0.0,21.9\n2026-02-22,2,9.1,0.0,0.0,23.5\n2026-02-22,3,9.3,0.0,0.0,23.8\n2026-02-22,4,9.3,0.0,0.0,22.3\n2026-02-22,5,9.2,0.0,0.0,22.6\n2026-02-22,6,8.9,0.0,0.0,20.5\n2026-02-22,7,8.8,0.0,0.0,19.6\n2026-02-22,8,8.7,0.0,0.0,21.7\n2026-02-22,9,9.4,0.0,0.0,19.6\n2026-02-22,10,9.3,0.0,0.0,20.6\n2026-02-22,11,9.4,0.0,0.0,15.2\n2026-02-22,12,9.4,0.0,0.0,19.2\n2026-02-22,13,8.6,0.1,0.0,22.9\n2026-02-22,14,8.9,0.0,0.0,27.5\n2026-02-22,15,7.7,0.0,0.0,27.4\n2026-02-22,16,6.3,0.0,0.0,27.5\n2026-02-22,17,5.1,0.0,0.0,25.7\n2026-02-22,18,3.7,0.0,0.0,21.7\n2026-02-22,19,2.6,0.0,0.0,20.8\n2026-02-22,20,2.1,0.0,0.0,20.2\n2026-02-22,21,1.9,0.0,0.0,18.9\n2026-02-22,22,1.6,0.0,0.0,18.1\n2026-02-22,23,1.2,0.0,0.0,18.0\n2026-02-23,0,0.5,0.0,0.0,16.9\n2026-02-23,1,-0.2,0.0,0.0,14.4\n2026-02-23,2,-0.9,0.0,0.0,11.0\n2026-02-23,3,-1.3,0.0,0.0,8.6\n2026-02-23,4,-1.1,0.0,0.0,9.0\n2026-02-23,5,-1.0,0.0,0.0,9.2\n2026-02-23,6,-0.6,0.0,0.0,10.2\n2026-02-23,7,-0.4,0.0,0.0,9.6\n2026-02-23,8,0.4,0.0,0.0,10.5\n2026-02-23,9,1.8,0.0,0.0,12.9\n2026-02-23,10,1.7,0.0,0.0,12.8\n2026-02-23,11,3.0,0.0,0.0,11.6\n2026-02-23,12,3.9,0.0,0.0,12.6\n2026-02-23,13,4.5,0.0,0.0,12.5\n2026-02-23,14,4.6,0.0,0.0,13.1\n2026-02-23,15,4.6,0.0,0.0,12.2\n2026-02-23,16,3.8,0.0,0.0,13.5\n2026-02-23,17,2.3,0.0,0.0,14.9\n2026-02-23,18,1.7,0.0,0.0,11.8\n2026-02-23,19,1.1,0.0,0.0,9.2\n2026-02-23,20,1.0,0.0,0.0,8.5\n2026-02-23,21,1.1,0.0,0.0,7.7\n2026-02-23,22,1.2,0.0,0.0,6.1\n2026-02-23,23,0.8,0.0,0.0,3.9\n2026-02-24,0,0.6,0.0,0.0,6.0\n2026-02-24,1,0.0,0.0,0.0,8.3\n2026-02-24,2,-0.3,0.0,0.0,9.1\n2026-02-24,3,-0.2,0.0,0.0,8.2\n2026-02-24,4,-0.2,0.1,0.07,7.9\n2026-02-24,5,-0.5,0.2,0.14,4.9\n2026-02-24,6,-0.7,0.2,0.14,2.9\n2026-02-24,7,-0.9,0.3,0.21,2.1\n2026-02-24,8,-0.9,0.2,0.14,3.3\n2026-02-24,9,0.1,0.2,0.14,4.8\n2026-02-24,10,0.1,0.1,0.07,7.2\n2026-02-24,11,0.5,0.1,0.07,6.1\n2026-02-24,12,1.6,0.1,0.07,6.3\n2026-02-24,13,3.0,0.0,0.0,6.9\n2026-02-24,14,3.3,0.0,0.0,6.8\n2026-02-24,15,3.9,0.0,0.0,4.4\n2026-02-24,16,4.5,0.0,0.0,6.4\n2026-02-24,17,4.5,0.0,0.0,7.3\n2026-02-24,18,3.5,0.0,0.0,7.2\n2026-02-24,19,1.9,0.0,0.0,7.1\n2026-02-24,20,1.3,0.0,0.0,6.7\n2026-02-24,21,0.5,0.0,0.0,9.4\n2026-02-24,22,-0.1,0.0,0.0,8.9\n2026-02-24,23,-0.4,0.0,0.0,7.9\n2026-02-25,0,-0.7,0.0,0.0,7.5\n2026-02-25,1,-1.1,0.0,0.0,7.4\n2026-02-25,2,-1.1,0.0,0.0,8.2\n2026-02-25,3,-1.2,0.0,0.0,9.8\n2026-02-25,4,-1.2,0.0,0.0,9.8\n2026-02-25,5,-1.3,0.0,0.0,9.9\n2026-02-25,6,-1.5,0.0,0.0,9.4\n2026-02-25,7,-1.7,0.0,0.0,6.6\n2026-02-25,8,-1.0,0.0,0.0,5.3\n2026-02-25,9,1.5,0.0,0.0,5.2\n2026-02-25,10,5.1,0.0,0.0,3.9\n2026-02-25,11,8.0,0.0,0.0,4.6\n2026-02-25,12,10.0,0.0,0.0,5.7\n2026-02-25,13,11.5,0.0,0.0,5.1\n2026-02-25,14,12.2,0.0,0.0,5.6\n2026-02-25,15,12.4,0.0,0.0,5.3\n2026-02-25,16,12.6,0.0,0.0,6.7\n2026-02-25,17,11.4,0.0,0.0,4.4\n2026-02-25,18,8.2,0.0,0.0,2.6\n2026-02-25,19,6.1,0.0,0.0,1.2\n2026-02-25,20,4.3,0.0,0.0,3.1\n2026-02-25,21,5.9,0.0,0.0,5.9\n2026-02-25,22,6.2,0.0,0.0,8.6\n2026-02-25,23,5.5,0.0,0.0,11.8\n2026-02-26,0,4.4,0.0,0.0,10.9\n2026-02-26,1,3.4,0.0,0.0,10.6\n2026-02-26,2,2.6,0.0,0.0,8.9\n2026-02-26,3,2.5,0.0,0.0,6.7\n2026-02-26,4,2.1,0.0,0.0,7.5\n2026-02-26,5,1.3,0.0,0.0,9.4\n2026-02-26,6,0.9,0.0,0.0,10.1\n2026-02-26,7,0.6,0.0,0.0,8.5\n2026-02-26,8,1.1,0.0,0.0,8.3\n2026-02-26,9,3.3,0.0,0.0,7.5\n2026-02-26,10,5.8,0.0,0.0,5.1\n2026-02-26,11,8.0,0.0,0.0,4.3\n2026-02-26,12,9.9,0.0,0.0,6.0\n2026-02-26,13,10.9,0.0,0.0,6.0\n2026-02-26,14,11.5,0.0,0.0,7.3\n2026-02-26,15,11.3,0.0,0.0,6.5\n2026-02-26,16,11.1,0.0,0.0,6.3\n2026-02-26,17,10.7,0.0,0.0,6.2\n2026-02-26,18,8.9,0.0,0.0,3.5\n2026-02-26,19,7.5,0.0,0.0,4.8\n2026-02-26,20,7.1,0.0,0.0,4.2\n2026-02-26,21,6.6,0.0,0.0,3.1\n2026-02-26,22,7.0,0.0,0.0,5.2\n2026-02-26,23,6.7,0.0,0.0,5.7\n2026-02-27,0,6.5,0.0,0.0,6.4\n2026-02-27,1,6.0,0.0,0.0,3.7\n2026-02-27,2,5.2,0.0,0.0,5.9\n2026-02-27,3,5.9,0.0,0.0,10.5\n2026-02-27,4,5.3,0.0,0.0,12.7\n2026-02-27,5,4.7,0.0,0.0,15.5\n2026-02-27,6,4.6,0.0,0.0,15.4\n2026-02-27,7,4.3,0.0,0.0,13.7\n2026-02-27,8,4.5,0.0,0.0,11.0\n2026-02-27,9,6.7,0.0,0.0,9.7\n2026-02-27,10,8.6,0.0,0.0,11.9\n2026-02-27,11,10.2,0.0,0.0,11.9\n2026-02-27,12,11.6,0.0,0.0,9.5\n2026-02-27,13,12.2,0.0,0.0,9.3\n2026-02-27,14,12.2,0.0,0.0,11.2\n2026-02-27,15,12.8,0.0,0.0,9.5\n2026-02-27,16,12.9,0.0,0.0,9.2\n2026-02-27,17,12.5,0.0,0.0,10.3\n2026-02-27,18,11.3,0.0,0.0,5.9\n2026-02-27,19,9.1,0.0,0.0,1.3\n2026-02-27,20,9.2,0.0,0.0,10.1\n2026-02-27,21,7.6,0.0,0.0,12.2\n2026-02-27,22,7.2,0.0,0.0,10.3\n2026-02-27,23,6.2,0.0,0.0,5.7\n2026-02-28,0,5.2,0.0,0.0,5.7\n2026-02-28,1,4.1,0.0,0.0,4.5\n2026-02-28,2,3.2,0.0,0.0,4.3\n2026-02-28,3,2.8,0.0,0.0,6.5\n2026-02-28,4,2.7,0.0,0.0,6.2\n2026-02-28,5,2.1,0.0,0.0,5.9\n2026-02-28,6,1.5,0.0,0.0,5.3\n2026-02-28,7,0.8,0.0,0.0,4.7\n2026-02-28,8,1.2,0.0,0.0,6.1\n2026-02-28,9,3.2,0.0,0.0,3.9\n2026-02-28,10,6.7,0.0,0.0,3.6\n2026-02-28,11,9.5,0.0,0.0,6.2\n2026-02-28,12,11.4,0.0,0.0,8.0\n2026-02-28,13,12.7,0.0,0.0,9.3\n2026-02-28,14,13.6,0.0,0.0,8.7\n2026-02-28,15,14.5,0.0,0.0,2.1\n2026-02-28,16,14.3,0.0,0.0,3.8\n2026-02-28,17,13.3,0.0,0.0,5.8\n2026-02-28,18,11.3,0.0,0.0,5.2\n2026-02-28,19,9.3,0.0,0.0,7.2\n2026-02-28,20,7.9,0.0,0.0,4.2\n2026-02-28,21,6.7,0.0,0.0,3.5\n2026-02-28,22,7.6,0.0,0.0,10.0\n2026-02-28,23,7.2,0.0,0.0,12.7\n2026-03-01,0,6.3,0.0,0.0,14.3\n2026-03-01,1,5.5,0.0,0.0,12.8\n2026-03-01,2,4.9,0.0,0.0,12.4\n2026-03-01,3,5.0,0.0,0.0,12.5\n2026-03-01,4,5.2,0.0,0.0,15.2\n2026-03-01,5,5.2,0.0,0.0,14.1\n2026-03-01,6,5.1,0.0,0.0,13.1\n2026-03-01,7,5.2,0.0,0.0,12.9\n2026-03-01,8,5.3,0.0,0.0,11.4\n2026-03-01,9,6.0,0.0,0.0,8.9\n2026-03-01,10,7.4,0.0,0.0,10.8\n2026-03-01,11,10.2,0.0,0.0,12.8\n2026-03-01,12,11.8,0.0,0.0,9.2\n2026-03-01,13,12.7,0.0,0.0,9.1\n2026-03-01,14,12.8,0.0,0.0,8.3\n2026-03-01,15,12.0,0.0,0.0,12.9\n2026-03-01,16,12.2,0.0,0.0,13.0\n2026-03-01,17,11.0,0.0,0.0,11.0\n2026-03-01,18,9.8,0.0,0.0,11.5\n2026-03-01,19,9.2,0.0,0.0,12.0\n2026-03-01,20,8.3,0.0,0.0,15.0\n2026-03-01,21,7.9,0.1,0.0,12.9\n2026-03-01,22,7.5,0.1,0.0,13.5\n2026-03-01,23,6.8,0.1,0.0,14.2\n2026-03-02,0,6.2,0.1,0.0,14.1\n2026-03-02,1,6.0,0.0,0.0,13.6\n2026-03-02,2,5.9,0.0,0.0,13.7\n2026-03-02,3,6.4,0.0,0.0,15.1\n2026-03-02,4,6.2,0.0,0.0,15.7\n2026-03-02,5,5.9,0.0,0.0,15.8\n2026-03-02,6,5.7,0.1,0.0,16.9\n2026-03-02,7,5.5,0.3,0.0,16.7\n2026-03-02,8,5.2,0.4,0.0,17.2\n2026-03-02,9,5.0,0.5,0.0,16.8\n2026-03-02,10,4.7,1.4,0.0,19.1\n2026-03-02,11,4.1,1.3,0.0,18.7\n2026-03-02,12,3.9,0.9,0.0,17.3\n2026-03-02,13,4.4,0.6,0.0,19.8\n2026-03-02,14,4.2,0.9,0.0,18.9\n2026-03-02,15,4.5,0.9,0.0,19.5\n2026-03-02,16,4.4,0.9,0.0,20.2\n2026-03-02,17,4.0,1.0,0.0,20.4\n2026-03-02,18,3.5,0.7,0.0,19.3\n2026-03-02,19,3.0,0.7,0.0,18.2\n2026-03-02,20,2.3,0.8,0.0,14.8\n2026-03-02,21,2.1,1.0,0.21,15.8\n2026-03-02,22,1.9,0.7,0.0,15.3\n2026-03-02,23,1.7,0.7,0.0,13.5\n2026-03-03,0,1.7,0.5,0.0,11.7\n2026-03-03,1,1.9,0.3,0.0,13.7\n2026-03-03,2,1.8,0.1,0.0,13.5\n2026-03-03,3,1.2,0.2,0.0,14.7\n2026-03-03,4,1.4,0.3,0.0,15.1\n2026-03-03,5,1.6,0.0,0.0,14.6\n2026-03-03,6,1.8,0.0,0.0,13.0\n2026-03-03,7,1.9,0.0,0.0,12.3\n2026-03-03,8,2.3,0.0,0.0,13.0\n2026-03-03,9,3.5,0.0,0.0,16.2\n2026-03-03,10,4.8,0.0,0.0,11.9\n2026-03-03,11,5.5,0.0,0.0,11.7\n2026-03-03,12,7.1,0.0,0.0,12.7\n2026-03-03,13,8.6,0.0,0.0,12.1\n2026-03-03,14,10.0,0.0,0.0,10.1\n2026-03-03,15,10.7,0.0,0.0,11.3\n2026-03-03,16,10.3,0.0,0.0,9.3\n2026-03-03,17,9.7,0.0,0.0,9.9\n2026-03-03,18,9.0,0.0,0.0,8.8\n2026-03-03,19,7.7,0.0,0.0,6.2\n2026-03-03,20,6.4,0.0,0.0,10.6\n2026-03-03,21,5.0,0.0,0.0,13.0\n2026-03-03,22,4.7,0.0,0.0,10.4\n2026-03-03,23,4.3,0.0,0.0,9.0\n2026-03-04,0,3.9,0.0,0.0,7.8\n2026-03-04,1,3.2,0.0,0.0,5.2\n2026-03-04,2,3.1,0.0,0.0,3.1\n2026-03-04,3,2.8,0.0,0.0,7.2\n2026-03-04,4,1.6,0.0,0.0,8.9\n2026-03-04,5,0.8,0.0,0.0,10.1\n2026-03-04,6,0.2,0.0,0.0,9.2\n2026-03-04,7,-0.4,0.0,0.0,8.1\n2026-03-04,8,0.3,0.0,0.0,8.8\n2026-03-04,9,2.3,0.0,0.0,4.8\n2026-03-04,10,4.9,0.0,0.0,6.1\n2026-03-04,11,6.8,0.0,0.0,7.6\n2026-03-04,12,8.4,0.0,0.0,9.0\n2026-03-04,13,9.4,0.0,0.0,11.0\n2026-03-04,14,9.7,0.0,0.0,12.5\n2026-03-04,15,9.7,0.0,0.0,11.9\n2026-03-04,16,8.9,0.0,0.0,14.5\n2026-03-04,17,7.9,0.0,0.0,12.6\n2026-03-04,18,6.4,0.0,0.0,9.0\n2026-03-04,19,4.7,0.0,0.0,10.0\n2026-03-04,20,3.9,0.0,0.0,10.5\n2026-03-04,21,3.9,0.0,0.0,8.9\n2026-03-04,22,3.6,0.0,0.0,8.5\n2026-03-04,23,3.1,0.0,0.0,8.1\n2026-03-05,0,2.4,0.0,0.0,8.5\n2026-03-05,1,1.6,0.0,0.0,7.5\n2026-03-05,2,1.2,0.0,0.0,7.3\n2026-03-05,3,1.1,0.0,0.0,6.1\n2026-03-05,4,0.9,0.0,0.0,6.8\n2026-03-05,5,-0.1,0.0,0.0,2.8\n2026-03-05,6,-0.4,0.0,0.0,0.9\n2026-03-05,7,-0.9,0.0,0.0,1.1\n2026-03-05,8,0.2,0.0,0.0,3.4\n2026-03-05,9,2.7,0.0,0.0,5.8\n2026-03-05,10,6.0,0.0,0.0,4.1\n2026-03-05,11,8.6,0.0,0.0,2.7\n2026-03-05,12,10.0,0.0,0.0,3.5\n2026-03-05,13,11.4,0.0,0.0,6.2\n2026-03-05,14,11.6,0.0,0.0,9.4\n2026-03-05,15,11.4,0.0,0.0,8.0\n2026-03-05,16,9.5,0.0,0.0,8.3\n2026-03-05,17,7.7,0.1,0.0,9.2\n2026-03-05,18,6.1,0.5,0.0,11.9\n2026-03-05,19,4.8,1.3,0.0,7.7\n2026-03-05,20,2.5,1.6,0.14,9.5\n2026-03-05,21,1.6,2.7,1.26,12.5\n2026-03-05,22,1.1,2.8,1.54,10.1\n2026-03-05,23,0.8,1.5,0.77,9.6\n2026-03-06,0,1.0,0.4,0.07,8.0\n2026-03-06,1,0.9,0.0,0.0,7.1\n2026-03-06,2,0.9,0.0,0.0,8.3\n2026-03-06,3,0.8,0.0,0.0,8.1\n2026-03-06,4,0.1,0.0,0.0,8.5\n2026-03-06,5,0.6,0.0,0.0,9.3\n2026-03-06,6,0.8,0.0,0.0,9.0\n2026-03-06,7,0.5,0.0,0.0,9.7\n2026-03-06,8,1.1,0.0,0.0,10.4\n2026-03-06,9,1.5,0.0,0.0,12.2\n2026-03-06,10,2.4,0.0,0.0,15.3\n2026-03-06,11,3.4,0.0,0.0,15.3\n2026-03-06,12,4.4,0.0,0.0,20.2\n2026-03-06,13,3.8,0.0,0.0,20.9\n2026-03-06,14,3.6,0.0,0.0,21.9\n2026-03-06,15,2.4,0.0,0.0,21.0\n2026-03-06,16,2.6,0.0,0.0,20.9\n2026-03-06,17,2.5,0.0,0.0,20.3\n2026-03-06,18,1.8,0.0,0.0,18.5\n2026-03-06,19,1.2,0.0,0.0,16.7\n2026-03-06,20,0.8,0.0,0.0,15.2\n2026-03-06,21,-0.3,0.0,0.0,12.3\n2026-03-06,22,-0.9,0.0,0.0,12.1\n2026-03-06,23,-1.2,0.0,0.0,10.3\n2026-03-07,0,-1.3,0.0,0.0,10.1\n2026-03-07,1,-1.3,0.0,0.0,10.8\n2026-03-07,2,-1.5,0.0,0.0,12.0\n2026-03-07,3,-1.5,0.0,0.0,11.2\n2026-03-07,4,-1.9,0.0,0.0,12.3\n2026-03-07,5,-2.6,0.0,0.0,11.9\n2026-03-07,6,-3.1,0.0,0.0,10.7\n2026-03-07,7,-3.5,0.0,0.0,10.7\n2026-03-07,8,-2.8,0.0,0.0,11.0\n2026-03-07,9,-1.4,0.0,0.0,10.1\n2026-03-07,10,0.0,0.0,0.0,10.5\n2026-03-07,11,1.2,0.0,0.0,12.0\n2026-03-07,12,2.1,0.0,0.0,14.2\n2026-03-07,13,2.6,0.0,0.0,15.2\n2026-03-07,14,3.5,0.0,0.0,15.9\n2026-03-07,15,3.3,0.0,0.0,16.3\n2026-03-07,16,3.5,0.0,0.0,16.7\n2026-03-07,17,3.6,0.0,0.0,15.3\n2026-03-07,18,2.8,0.0,0.0,10.5\n2026-03-07,19,1.0,0.0,0.0,10.8\n2026-03-07,20,0.0,0.0,0.0,13.6\n2026-03-07,21,-0.7,0.0,0.0,12.7\n2026-03-07,22,-1.4,0.0,0.0,11.0\n2026-03-07,23,-2.1,0.0,0.0,9.4\n2026-03-08,0,-2.6,0.0,0.0,7.8\n2026-03-08,1,-2.9,0.0,0.0,7.2\n2026-03-08,2,-3.4,0.0,0.0,7.9\n2026-03-08,3,-3.7,0.0,0.0,5.4\n2026-03-08,4,-3.9,0.0,0.0,3.3\n2026-03-08,5,-3.8,0.0,0.0,0.2\n2026-03-08,6,-3.9,0.0,0.0,2.0\n2026-03-08,7,-3.8,0.0,0.0,1.6\n2026-03-08,8,-2.6,0.0,0.0,1.6\n2026-03-08,9,-1.4,0.0,0.0,1.7\n2026-03-08,10,1.0,0.0,0.0,2.7\n2026-03-08,11,3.0,0.0,0.0,3.6\n2026-03-08,12,4.2,0.0,0.0,7.3\n2026-03-08,13,4.5,0.0,0.0,9.6\n2026-03-08,14,4.3,0.0,0.0,11.6\n2026-03-08,15,4.1,0.0,0.0,10.4\n2026-03-08,16,3.5,0.0,0.0,10.9\n2026-03-08,17,3.3,0.0,0.0,8.8\n2026-03-08,18,2.1,0.0,0.0,9.0\n2026-03-08,19,0.5,0.0,0.0,10.0\n2026-03-08,20,0.4,0.0,0.0,10.1\n2026-03-08,21,0.8,0.0,0.0,9.4\n2026-03-08,22,1.5,0.0,0.0,9.2\n2026-03-08,23,1.6,0.0,0.0,10.4\n2026-03-09,0,1.5,0.0,0.0,9.6\n2026-03-09,1,1.3,0.0,0.0,8.0\n2026-03-09,2,1.7,0.0,0.0,6.7\n2026-03-09,3,1.9,0.0,0.0,7.1\n2026-03-09,4,2.1,0.0,0.0,6.1\n2026-03-09,5,1.9,0.0,0.0,5.4\n2026-03-09,6,1.7,0.0,0.0,4.8\n2026-03-09,7,0.9,0.0,0.0,7.3\n2026-03-09,8,1.2,0.0,0.0,10.0\n2026-03-09,9,2.5,0.0,0.0,8.1\n2026-03-09,10,4.1,0.0,0.0,10.7\n2026-03-09,11,5.5,0.0,0.0,13.2\n2026-03-09,12,6.6,0.0,0.0,16.8\n2026-03-09,13,6.5,0.0,0.0,17.1\n2026-03-09,14,6.5,0.0,0.0,17.2\n2026-03-09,15,6.7,0.0,0.0,16.6\n2026-03-09,16,6.6,0.0,0.0,15.6\n2026-03-09,17,6.1,0.0,0.0,13.2\n2026-03-09,18,5.6,0.0,0.0,8.6\n2026-03-09,19,3.8,0.0,0.0,6.8\n2026-03-09,20,3.4,0.0,0.0,12.1\n2026-03-09,21,2.3,0.0,0.0,12.6\n2026-03-09,22,1.7,0.0,0.0,11.3\n2026-03-09,23,0.9,0.0,0.0,8.0\n2026-03-10,0,0.3,0.0,0.0,5.5\n2026-03-10,1,-0.3,0.0,0.0,5.1\n2026-03-10,2,-1.1,0.0,0.0,4.6\n2026-03-10,3,-1.3,0.0,0.0,3.2\n2026-03-10,4,-1.6,0.0,0.0,2.3\n2026-03-10,5,-1.8,0.0,0.0,0.6\n2026-03-10,6,-2.2,0.0,0.0,0.9\n2026-03-10,7,-2.8,0.0,0.0,2.7\n2026-03-10,8,-1.0,0.0,0.0,5.4\n2026-03-10,9,1.7,0.0,0.0,3.8\n2026-03-10,10,4.3,0.0,0.0,5.0\n2026-03-10,11,5.9,0.0,0.0,6.8\n2026-03-10,12,6.6,0.0,0.0,10.0\n2026-03-10,13,6.4,0.0,0.0,11.8\n2026-03-10,14,6.3,0.0,0.0,11.9\n2026-03-10,15,6.2,0.0,0.0,10.1\n2026-03-10,16,6.1,0.0,0.0,9.4\n2026-03-10,17,5.2,0.0,0.0,10.3\n2026-03-10,18,3.6,0.0,0.0,7.7\n2026-03-10,19,1.1,0.0,0.0,5.0\n2026-03-10,20,0.6,0.0,0.0,2.1\n2026-03-10,21,1.1,0.0,0.0,2.1\n2026-03-10,22,1.2,0.0,0.0,0.9\n2026-03-10,23,0.2,0.0,0.0,2.9\n2026-03-11,0,-0.4,0.0,0.0,2.2\n2026-03-11,1,-0.9,0.0,0.0,2.5\n2026-03-11,2,-1.0,0.0,0.0,2.2\n2026-03-11,3,-0.6,0.0,0.0,3.7\n2026-03-11,4,-1.2,0.0,0.0,3.9\n2026-03-11,5,-0.2,0.0,0.0,3.1\n2026-03-11,6,-0.1,0.0,0.0,1.5\n2026-03-11,7,0.0,0.0,0.0,1.4\n2026-03-11,8,0.9,0.0,0.0,1.2\n2026-03-11,9,1.8,0.0,0.0,2.5\n2026-03-11,10,3.5,0.0,0.0,1.9\n2026-03-11,11,4.7,0.0,0.0,3.1\n2026-03-11,12,5.6,0.0,0.0,4.2\n2026-03-11,13,7.2,0.0,0.0,5.8\n2026-03-11,14,8.1,0.0,0.0,8.2\n2026-03-11,15,8.0,0.0,0.0,9.0\n2026-03-11,16,6.7,0.0,0.0,10.9\n2026-03-11,17,6.4,0.0,0.0,10.4\n2026-03-11,18,4.7,0.0,0.0,11.1\n2026-03-11,19,2.9,0.0,0.0,11.1\n2026-03-11,20,2.2,0.0,0.0,9.8\n2026-03-11,21,3.3,0.0,0.0,8.8\n2026-03-11,22,3.2,0.0,0.0,8.0\n2026-03-11,23,2.9,0.0,0.0,8.4\n2026-03-12,0,2.4,0.0,0.0,8.4\n2026-03-12,1,1.9,0.0,0.0,7.7\n2026-03-12,2,1.3,0.0,0.0,6.9\n2026-03-12,3,0.7,0.0,0.0,6.5\n2026-03-12,4,0.2,0.0,0.0,7.4\n2026-03-12,5,-0.4,0.0,0.0,6.7\n2026-03-12,6,-0.7,0.0,0.0,7.2\n2026-03-12,7,-0.9,0.0,0.0,6.9\n2026-03-12,8,0.1,0.0,0.0,8.1\n2026-03-12,9,2.1,0.0,0.0,4.9\n2026-03-12,10,5.5,0.0,0.0,5.8\n2026-03-12,11,8.2,0.0,0.0,7.1\n2026-03-12,12,9.8,0.0,0.0,8.1\n2026-03-12,13,10.6,0.0,0.0,7.6\n2026-03-12,14,11.5,0.0,0.0,10.0\n2026-03-12,15,11.9,0.0,0.0,9.6\n2026-03-12,16,11.6,0.0,0.0,8.1\n2026-03-12,17,9.7,0.0,0.0,11.9\n2026-03-12,18,7.2,0.0,0.0,11.4\n2026-03-12,19,5.7,0.0,0.0,12.1\n2026-03-12,20,4.8,0.0,0.0,8.4\n2026-03-12,21,5.6,0.0,0.0,7.9\n2026-03-12,22,5.7,0.0,0.0,12.6\n2026-03-12,23,4.8,0.0,0.0,13.1\n2026-03-13,0,3.8,0.0,0.0,12.1\n2026-03-13,1,3.0,0.0,0.0,11.8\n2026-03-13,2,2.5,0.0,0.0,11.7\n2026-03-13,3,2.4,0.0,0.0,8.3\n2026-03-13,4,2.6,0.0,0.0,9.5\n2026-03-13,5,2.4,0.0,0.0,9.2\n2026-03-13,6,2.1,0.0,0.0,10.7\n2026-03-13,7,2.0,0.0,0.0,12.0\n2026-03-13,8,2.8,0.0,0.0,12.0\n2026-03-13,9,4.2,0.0,0.0,7.1\n2026-03-13,10,6.2,0.0,0.0,11.0\n2026-03-13,11,7.7,0.0,0.0,10.5\n2026-03-13,12,9.1,0.0,0.0,10.9\n2026-03-13,13,10.1,0.0,0.0,10.8\n2026-03-13,14,10.6,0.0,0.0,10.8\n2026-03-13,15,11.1,0.0,0.0,10.4\n2026-03-13,16,11.3,0.0,0.0,8.0\n2026-03-13,17,10.6,0.0,0.0,2.7\n2026-03-13,18,8.7,0.0,0.0,7.6\n2026-03-13,19,6.6,0.0,0.0,10.9\n2026-03-13,20,5.5,0.0,0.0,7.6\n2026-03-13,21,4.7,0.0,0.0,4.1\n2026-03-13,22,4.7,0.0,0.0,3.9\n2026-03-13,23,4.7,0.0,0.0,6.9\n2026-03-14,0,4.5,0.0,0.0,7.3\n2026-03-14,1,3.8,0.0,0.0,9.7\n2026-03-14,2,2.8,0.0,0.0,10.7\n2026-03-14,3,2.9,0.0,0.0,8.4\n2026-03-14,4,3.7,0.0,0.0,7.8\n2026-03-14,5,3.8,0.0,0.0,8.1\n2026-03-14,6,3.6,0.0,0.0,7.4\n2026-03-14,7,3.0,0.0,0.0,5.3\n2026-03-14,8,3.7,0.0,0.0,5.1\n2026-03-14,9,4.5,0.0,0.0,3.5\n2026-03-14,10,6.9,0.0,0.0,4.7\n2026-03-14,11,8.8,0.0,0.0,4.6\n2026-03-14,12,9.8,0.0,0.0,6.3\n2026-03-14,13,10.5,0.0,0.0,8.2\n2026-03-14,14,11.2,0.0,0.0,11.0\n2026-03-14,15,10.4,0.0,0.0,14.3\n2026-03-14,16,9.1,0.0,0.0,13.9\n2026-03-14,17,8.3,0.0,0.0,12.2\n2026-03-14,18,7.4,0.0,0.0,9.6\n2026-03-14,19,6.5,0.0,0.0,7.7\n2026-03-14,20,5.9,0.0,0.0,7.2\n2026-03-14,21,4.8,0.0,0.0,6.7\n2026-03-14,22,4.7,0.0,0.0,5.7\n2026-03-14,23,4.7,0.0,0.0,6.6\n2026-03-15,0,4.0,0.0,0.0,3.9\n2026-03-15,1,3.6,0.0,0.0,4.0\n2026-03-15,2,4.1,0.0,0.0,3.4\n2026-03-15,3,4.8,0.0,0.0,2.4\n2026-03-15,4,5.1,0.0,0.0,1.8\n2026-03-15,5,5.1,0.0,0.0,2.0\n2026-03-15,6,5.0,0.1,0.0,2.7\n2026-03-15,7,4.7,0.1,0.0,2.6\n2026-03-15,8,5.1,0.1,0.0,3.1\n2026-03-15,9,5.6,0.2,0.0,2.4\n2026-03-15,10,7.0,0.1,0.0,3.1\n2026-03-15,11,8.1,0.0,0.0,4.3\n2026-03-15,12,8.7,0.0,0.0,7.0\n2026-03-15,13,9.2,0.0,0.0,8.7\n2026-03-15,14,9.1,0.0,0.0,9.6\n2026-03-15,15,8.6,0.0,0.0,11.2\n2026-03-15,16,8.8,0.0,0.0,11.2\n2026-03-15,17,7.5,0.0,0.0,13.3\n2026-03-15,18,5.9,0.0,0.0,11.1\n2026-03-15,19,4.3,0.0,0.0,8.6\n2026-03-15,20,3.7,0.0,0.0,7.9\n2026-03-15,21,3.5,0.0,0.0,7.5\n2026-03-15,22,4.0,0.0,0.0,9.5\n2026-03-15,23,3.6,0.0,0.0,8.7\n2026-03-16,0,2.6,0.0,0.0,5.7\n2026-03-16,1,1.6,0.0,0.0,3.2\n2026-03-16,2,1.1,0.0,0.0,2.0\n2026-03-16,3,1.2,0.0,0.0,1.8\n2026-03-16,4,1.2,0.0,0.0,3.1\n2026-03-16,5,0.7,0.0,0.0,4.1\n2026-03-16,6,0.6,0.0,0.0,3.3\n2026-03-16,7,0.2,0.0,0.0,3.1\n2026-03-16,8,1.7,0.0,0.0,3.0\n2026-03-16,9,3.7,0.0,0.0,3.4\n2026-03-16,10,6.1,0.0,0.0,2.8\n2026-03-16,11,8.3,0.0,0.0,2.5\n2026-03-16,12,10.1,0.0,0.0,4.8\n2026-03-16,13,10.7,0.0,0.0,8.2\n2026-03-16,14,11.2,0.0,0.0,8.4\n2026-03-16,15,11.1,0.0,0.0,8.6\n2026-03-16,16,10.6,0.0,0.0,10.5\n2026-03-16,17,9.9,0.0,0.0,10.3\n2026-03-16,18,8.7,0.0,0.0,8.4\n2026-03-16,19,6.3,0.0,0.0,8.5\n2026-03-16,20,5.1,0.0,0.0,5.1\n2026-03-16,21,3.9,0.0,0.0,3.9\n2026-03-16,22,4.1,0.0,0.0,4.0\n2026-03-16,23,3.2,0.0,0.0,2.9\n2026-03-17,0,2.7,0.0,0.0,1.6\n2026-03-17,1,2.3,0.0,0.0,0.9\n2026-03-17,2,2.0,0.0,0.0,0.2\n2026-03-17,3,1.4,0.0,0.0,0.4\n2026-03-17,4,1.1,0.0,0.0,0.4\n2026-03-17,5,0.2,0.0,0.0,1.8\n2026-03-17,6,-0.6,0.0,0.0,2.6\n2026-03-17,7,-1.0,0.0,0.0,2.3\n2026-03-17,8,1.2,0.0,0.0,3.2\n2026-03-17,9,4.0,0.0,0.0,3.7\n2026-03-17,10,7.1,0.0,0.0,1.6\n2026-03-17,11,9.4,0.0,0.0,2.3\n2026-03-17,12,11.0,0.0,0.0,4.4\n2026-03-17,13,11.7,0.0,0.0,6.1\n2026-03-17,14,12.0,0.0,0.0,7.2\n2026-03-17,15,12.4,0.0,0.0,7.6\n2026-03-17,16,12.1,0.0,0.0,8.3\n2026-03-17,17,11.1,0.0,0.0,8.3\n2026-03-17,18,8.7,0.0,0.0,3.8\n2026-03-17,19,5.9,0.0,0.0,5.4\n2026-03-17,20,4.8,0.0,0.0,5.1\n2026-03-17,21,5.3,0.0,0.0,2.7\n2026-03-17,22,5.5,0.0,0.0,4.7\n2026-03-17,23,4.9,0.0,0.0,6.2\n2026-03-18,0,4.7,0.0,0.0,4.6\n2026-03-18,1,5.2,0.0,0.0,5.6\n2026-03-18,2,5.2,0.0,0.0,5.0\n2026-03-18,3,5.2,0.0,0.0,6.3\n2026-03-18,4,5.4,0.0,0.0,6.5\n2026-03-18,5,5.5,0.3,0.0,5.6\n2026-03-18,6,4.8,0.8,0.0,3.2\n2026-03-18,7,4.8,1.0,0.0,2.4\n2026-03-18,8,5.0,0.9,0.0,3.2\n2026-03-18,9,5.1,0.8,0.0,9.3\n2026-03-18,10,4.6,0.6,0.0,12.5\n2026-03-18,11,4.5,0.6,0.0,16.6\n2026-03-18,12,3.9,0.4,0.0,16.8\n2026-03-18,13,3.8,0.4,0.0,16.4\n2026-03-18,14,4.4,0.1,0.0,9.4\n2026-03-18,15,8.0,0.0,0.0,12.4\n2026-03-18,16,9.2,0.0,0.0,12.4\n2026-03-18,17,8.9,0.0,0.0,14.7\n2026-03-18,18,8.3,0.0,0.0,9.9\n2026-03-18,19,5.7,0.0,0.0,14.4\n2026-03-18,20,5.2,0.0,0.0,12.8\n2026-03-18,21,4.9,0.0,0.0,13.7\n2026-03-18,22,4.9,0.0,0.0,14.0\n2026-03-18,23,4.1,0.0,0.0,13.1\n2026-03-19,0,3.3,0.0,0.0,12.9\n2026-03-19,1,2.8,0.0,0.0,13.7\n2026-03-19,2,2.1,0.0,0.0,13.8\n2026-03-19,3,1.7,0.0,0.0,10.7\n2026-03-19,4,1.2,0.0,0.0,10.8\n2026-03-19,5,0.4,0.0,0.0,10.4\n2026-03-19,6,-0.3,0.0,0.0,11.4\n2026-03-19,7,-0.7,0.0,0.0,12.0\n2026-03-19,8,0.6,0.0,0.0,9.6\n2026-03-19,9,2.7,0.0,0.0,8.3\n2026-03-19,10,5.2,0.0,0.0,6.2\n2026-03-19,11,7.4,0.0,0.0,5.3\n2026-03-19,12,9.3,0.0,0.0,6.5\n2026-03-19,13,10.0,0.0,0.0,11.6\n2026-03-19,14,9.6,0.0,0.0,16.1\n2026-03-19,15,8.8,0.0,0.0,18.4\n2026-03-19,16,8.0,0.0,0.0,18.6\n2026-03-19,17,6.9,0.0,0.0,17.5\n2026-03-19,18,5.6,0.0,0.0,14.8\n2026-03-19,19,4.4,0.0,0.0,13.3\n2026-03-19,20,4.0,0.0,0.0,11.5\n2026-03-19,21,3.7,0.0,0.0,10.7\n2026-03-19,22,3.8,0.0,0.0,9.6\n2026-03-19,23,3.5,0.0,0.0,9.9\n2026-03-20,0,3.3,0.0,0.0,10.0\n2026-03-20,1,3.2,0.0,0.0,9.7\n2026-03-20,2,3.0,0.0,0.0,9.9\n2026-03-20,3,2.3,0.0,0.0,10.1\n2026-03-20,4,2.2,0.0,0.0,8.7\n2026-03-20,5,1.8,0.0,0.0,7.8\n2026-03-20,6,1.5,0.0,0.0,10.3\n2026-03-20,7,1.3,0.0,0.0,10.7\n2026-03-20,8,3.3,0.0,0.0,11.4\n2026-03-20,9,5.0,0.0,0.0,10.6\n2026-03-20,10,7.5,0.0,0.0,15.6\n2026-03-20,11,8.5,0.0,0.0,14.4\n2026-03-20,12,9.6,0.0,0.0,11.9\n2026-03-20,13,10.1,0.0,0.0,12.0\n2026-03-20,14,10.4,0.0,0.0,13.2\n2026-03-20,15,10.0,0.0,0.0,13.2\n2026-03-20,16,10.0,0.0,0.0,12.8\n2026-03-20,17,9.3,0.0,0.0,12.0\n2026-03-20,18,7.6,0.0,0.0,10.4\n2026-03-20,19,5.7,0.0,0.0,8.2\n2026-03-20,20,4.9,0.0,0.0,6.2\n2026-03-20,21,4.4,0.0,0.0,5.3\n2026-03-20,22,4.8,0.0,0.0,6.3\n2026-03-20,23,4.1,0.0,0.0,5.0\n2026-03-21,0,3.0,0.0,0.0,3.2\n2026-03-21,1,2.2,0.0,0.0,3.1\n2026-03-21,2,2.0,0.0,0.0,4.9\n2026-03-21,3,1.5,0.0,0.0,8.5\n2026-03-21,4,1.1,0.0,0.0,7.4\n2026-03-21,5,0.6,0.0,0.0,6.4\n2026-03-21,6,-0.2,0.0,0.0,6.6\n2026-03-21,7,-0.7,0.0,0.0,7.1\n2026-03-21,8,1.1,0.0,0.0,7.1\n2026-03-21,9,3.1,0.0,0.0,6.1\n2026-03-21,10,6.4,0.0,0.0,5.8\n2026-03-21,11,8.9,0.0,0.0,6.2\n2026-03-21,12,10.4,0.0,0.0,7.7\n2026-03-21,13,11.0,0.0,0.0,9.8\n2026-03-21,14,10.9,0.0,0.0,11.2\n2026-03-21,15,10.9,0.0,0.0,11.2\n2026-03-21,16,10.4,0.0,0.0,10.4\n2026-03-21,17,9.4,0.0,0.0,9.3\n2026-03-21,18,7.8,0.0,0.0,7.8\n2026-03-21,19,5.6,0.0,0.0,4.2\n2026-03-21,20,4.7,0.0,0.0,1.2\n2026-03-21,21,4.3,0.0,0.0,1.4\n2026-03-21,22,4.5,0.0,0.0,1.3\n2026-03-21,23,3.9,0.0,0.0,1.5\n2026-03-22,0,3.5,0.0,0.0,2.7\n2026-03-22,1,3.0,0.0,0.0,4.0\n2026-03-22,2,3.1,0.0,0.0,2.2\n2026-03-22,3,2.6,0.0,0.0,2.8\n2026-03-22,4,2.8,0.0,0.0,4.0\n2026-03-22,5,3.0,0.0,0.0,5.3\n2026-03-22,6,2.8,0.0,0.0,4.6\n2026-03-22,7,2.7,0.0,0.0,3.6\n2026-03-22,8,4.2,0.0,0.0,3.3\n2026-03-22,9,7.0,0.0,0.0,1.9\n2026-03-22,10,9.7,0.0,0.0,2.7\n2026-03-22,11,12.3,0.0,0.0,3.9\n2026-03-22,12,14.1,0.0,0.0,3.7\n2026-03-22,13,15.6,0.0,0.0,3.0\n2026-03-22,14,16.2,0.0,0.0,7.5\n2026-03-22,15,15.7,0.0,0.0,13.9\n2026-03-22,16,14.9,0.0,0.0,13.9\n2026-03-22,17,13.8,0.0,0.0,12.8\n2026-03-22,18,11.8,0.0,0.0,11.3\n2026-03-22,19,9.2,0.0,0.0,7.5\n2026-03-22,20,7.7,0.0,0.0,3.2\n2026-03-22,21,7.5,0.0,0.0,2.0\n2026-03-22,22,7.3,0.0,0.0,4.7\n2026-03-22,23,6.5,0.0,0.0,7.1\n2026-03-23,0,6.0,0.0,0.0,10.3\n2026-03-23,1,5.2,0.0,0.0,10.4\n2026-03-23,2,4.6,0.0,0.0,10.8\n2026-03-23,3,4.0,0.0,0.0,8.7\n2026-03-23,4,3.8,0.0,0.0,7.1\n2026-03-23,5,3.1,0.0,0.0,5.9\n2026-03-23,6,2.4,0.0,0.0,4.8\n2026-03-23,7,2.5,0.0,0.0,5.3\n2026-03-23,8,4.0,0.0,0.0,7.1\n2026-03-23,9,7.2,0.0,0.0,4.7\n2026-03-23,10,9.9,0.0,0.0,5.7\n2026-03-23,11,12.4,0.0,0.0,6.2\n2026-03-23,12,14.3,0.0,0.0,6.8\n2026-03-23,13,15.8,0.0,0.0,5.4\n2026-03-23,14,16.7,0.0,0.0,1.7\n2026-03-23,15,16.7,0.0,0.0,8.0\n2026-03-23,16,16.2,0.0,0.0,10.6\n2026-03-23,17,14.9,0.0,0.0,12.6\n2026-03-23,18,12.9,0.0,0.0,9.7\n2026-03-23,19,10.6,0.0,0.0,8.1\n2026-03-23,20,9.3,0.0,0.0,5.6\n2026-03-23,21,8.3,0.0,0.0,3.6\n2026-03-23,22,8.7,0.0,0.0,3.0\n2026-03-23,23,8.4,0.0,0.0,4.4\n2026-03-24,0,8.0,0.0,0.0,5.3\n2026-03-24,1,7.7,0.0,0.0,9.5\n2026-03-24,2,6.7,0.0,0.0,10.7\n2026-03-24,3,6.7,0.0,0.0,11.4\n2026-03-24,4,6.7,0.0,0.0,12.1\n2026-03-24,5,6.6,0.0,0.0,12.2\n2026-03-24,6,6.2,0.0,0.0,12.8\n2026-03-24,7,6.0,0.0,0.0,12.6\n2026-03-24,8,7.4,0.0,0.0,11.1\n2026-03-24,9,9.8,0.0,0.0,10.7\n2026-03-24,10,12.1,0.0,0.0,9.5\n2026-03-24,11,13.9,0.0,0.0,6.9\n2026-03-24,12,15.2,0.0,0.0,7.3\n2026-03-24,13,15.8,0.0,0.0,12.0\n2026-03-24,14,15.4,0.0,0.0,12.9\n2026-03-24,15,15.1,0.0,0.0,11.9\n2026-03-24,16,14.6,0.0,0.0,10.0\n2026-03-24,17,14.2,0.0,0.0,7.8\n2026-03-24,18,13.4,0.0,0.0,7.1\n2026-03-24,19,12.2,0.0,0.0,5.8\n2026-03-24,20,11.8,0.0,0.0,5.6\n2026-03-24,21,11.7,0.0,0.0,5.2\n2026-03-24,22,11.4,0.0,0.0,2.7\n2026-03-24,23,10.9,0.0,0.0,3.4\n2026-03-25,0,9.8,0.0,0.0,2.2\n2026-03-25,1,9.3,0.0,0.0,1.5\n2026-03-25,2,8.8,0.0,0.0,2.1\n2026-03-25,3,8.3,0.0,0.0,4.2\n2026-03-25,4,8.3,0.0,0.0,4.7\n2026-03-25,5,7.8,0.0,0.0,4.6\n2026-03-25,6,7.5,0.0,0.0,4.7\n2026-03-25,7,6.8,0.0,0.0,4.0\n2026-03-25,8,8.3,0.0,0.0,5.2\n2026-03-25,9,10.5,0.0,0.0,5.1\n2026-03-25,10,12.7,0.0,0.0,6.5\n2026-03-25,11,14.5,0.0,0.0,6.2\n2026-03-25,12,16.0,0.0,0.0,5.8\n2026-03-25,13,16.4,0.0,0.0,10.4\n2026-03-25,14,15.9,0.0,0.0,13.3\n2026-03-25,15,15.2,0.0,0.0,15.7\n2026-03-25,16,13.9,0.0,0.0,15.3\n2026-03-25,17,12.8,0.0,0.0,13.1\n2026-03-25,18,10.8,0.0,0.0,11.0\n2026-03-25,19,8.8,0.0,0.0,8.5\n2026-03-25,20,7.5,0.0,0.0,8.2\n2026-03-25,21,8.0,0.0,0.0,8.8\n2026-03-25,22,7.7,0.0,0.0,9.1\n2026-03-25,23,7.1,0.0,0.0,7.8\n2026-03-26,0,6.7,0.0,0.0,4.1\n2026-03-26,1,6.2,0.0,0.0,3.7\n2026-03-26,2,5.7,0.0,0.0,5.5\n2026-03-26,3,5.0,0.0,0.0,3.6\n2026-03-26,4,5.1,0.0,0.0,1.1\n2026-03-26,5,4.9,0.0,0.0,1.1\n2026-03-26,6,4.7,0.0,0.0,3.7\n2026-03-26,7,4.4,0.0,0.0,5.8\n2026-03-26,8,5.7,0.0,0.0,3.7\n2026-03-26,9,7.4,0.0,0.0,2.4\n2026-03-26,10,10.4,0.0,0.0,3.9\n2026-03-26,11,13.0,0.0,0.0,5.1\n2026-03-26,12,14.7,0.0,0.0,7.7\n2026-03-26,13,15.6,0.0,0.0,9.3\n2026-03-26,14,16.3,0.0,0.0,10.5\n2026-03-26,15,15.3,0.0,0.0,12.4\n2026-03-26,16,14.1,0.0,0.0,12.3\n2026-03-26,17,12.6,0.0,0.0,10.3\n2026-03-26,18,10.8,0.0,0.0,9.5\n2026-03-26,19,8.8,0.0,0.0,7.7\n2026-03-26,20,7.8,0.0,0.0,5.3\n2026-03-26,21,7.5,0.0,0.0,3.2\n2026-03-26,22,7.2,0.0,0.0,2.7\n2026-03-26,23,6.8,0.0,0.0,0.6\n2026-03-27,0,6.2,0.0,0.0,2.2\n2026-03-27,1,6.0,0.0,0.0,5.0\n2026-03-27,2,5.7,0.0,0.0,6.8\n2026-03-27,3,5.6,0.0,0.0,7.6\n2026-03-27,4,5.6,0.0,0.0,7.7\n2026-03-27,5,5.4,0.0,0.0,6.8\n2026-03-27,6,5.1,0.0,0.0,6.5\n2026-03-27,7,5.2,0.0,0.0,8.8\n2026-03-27,8,7.4,0.0,0.0,8.7\n2026-03-27,9,8.6,0.0,0.0,3.6\n2026-03-27,10,10.9,0.0,0.0,3.3\n2026-03-27,11,12.9,0.0,0.0,6.1\n2026-03-27,12,14.3,0.0,0.0,9.0\n2026-03-27,13,14.7,0.0,0.0,10.8\n2026-03-27,14,15.0,0.0,0.0,11.9\n2026-03-27,15,14.5,0.0,0.0,11.9\n2026-03-27,16,14.1,0.0,0.0,11.9\n2026-03-27,17,13.0,0.0,0.0,12.4\n2026-03-27,18,11.3,0.0,0.0,10.7\n2026-03-27,19,9.3,0.0,0.0,10.1\n2026-03-27,20,8.0,0.0,0.0,8.1\n2026-03-27,21,7.4,0.0,0.0,7.8\n2026-03-27,22,7.0,0.0,0.0,8.3\n2026-03-27,23,6.5,0.0,0.0,6.2\n2026-03-28,0,6.0,0.0,0.0,4.3\n2026-03-28,1,5.7,0.0,0.0,2.8\n2026-03-28,2,5.3,0.0,0.0,3.1\n2026-03-28,3,5.2,0.0,0.0,5.8\n2026-03-28,4,5.2,0.0,0.0,4.2\n2026-03-28,5,4.6,0.0,0.0,2.8\n2026-03-28,6,4.3,0.0,0.0,2.4\n2026-03-28,7,4.3,0.0,0.0,4.4\n2026-03-28,8,5.5,0.0,0.0,4.0\n2026-03-28,9,7.1,0.0,0.0,4.1\n2026-03-28,10,9.9,0.0,0.0,1.9\n2026-03-28,11,12.6,0.0,0.0,0.5\n2026-03-28,12,14.6,0.0,0.0,2.6\n2026-03-28,13,16.1,0.0,0.0,4.5\n2026-03-28,14,17.1,0.0,0.0,6.0\n2026-03-28,15,16.7,0.0,0.0,9.6\n2026-03-28,16,16.1,0.0,0.0,10.7\n2026-03-28,17,14.4,0.0,0.0,10.4\n2026-03-28,18,12.5,0.0,0.0,7.4\n2026-03-28,19,10.2,0.0,0.0,6.5\n2026-03-28,20,8.7,0.0,0.0,6.7\n2026-03-28,21,7.8,0.0,0.0,3.6\n2026-03-28,22,7.5,0.0,0.0,2.2\n2026-03-28,23,7.2,0.0,0.0,0.2\n2026-03-29,0,6.6,0.0,0.0,2.2\n2026-03-29,1,5.9,0.0,0.0,3.1\n2026-03-29,2,5.6,0.0,0.0,5.4\n2026-03-29,3,5.1,0.0,0.0,7.5\n2026-03-29,4,4.8,0.0,0.0,8.6\n2026-03-29,5,5.0,0.0,0.0,6.7\n2026-03-29,6,4.7,0.0,0.0,3.5\n2026-03-29,7,5.3,0.0,0.0,6.0\n2026-03-29,8,8.0,0.0,0.0,5.9\n2026-03-29,9,11.1,0.0,0.0,4.4\n2026-03-29,10,13.6,0.0,0.0,4.0\n2026-03-29,11,15.7,0.0,0.0,4.2\n2026-03-29,12,17.5,0.0,0.0,4.1\n2026-03-29,13,17.9,0.0,0.0,7.2\n2026-03-29,14,18.4,0.0,0.0,11.1\n2026-03-29,15,17.3,0.0,0.0,12.5\n2026-03-29,16,15.8,0.0,0.0,10.3\n2026-03-29,17,15.5,0.0,0.0,8.0\n2026-03-29,18,14.9,0.0,0.0,4.7\n2026-03-29,19,13.2,0.0,0.0,2.7\n2026-03-29,20,12.8,0.0,0.0,1.2\n2026-03-29,21,10.3,0.0,0.0,10.6\n2026-03-29,22,10.1,0.0,0.0,7.0\n2026-03-29,23,10.4,0.0,0.0,6.6\n2026-03-30,0,9.8,0.0,0.0,7.7\n2026-03-30,1,10.4,0.0,0.0,9.0\n2026-03-30,2,10.3,0.0,0.0,6.5\n2026-03-30,3,9.7,0.0,0.0,6.6\n2026-03-30,4,9.5,0.0,0.0,5.1\n2026-03-30,5,9.7,0.0,0.0,5.4\n2026-03-30,6,9.2,0.0,0.0,3.7\n2026-03-30,7,9.4,0.0,0.0,3.5\n2026-03-30,8,10.9,0.0,0.0,3.6\n2026-03-30,9,12.0,0.0,0.0,3.9\n2026-03-30,10,13.0,0.0,0.0,4.2\n2026-03-30,11,13.9,0.0,0.0,4.1\n2026-03-30,12,15.6,0.0,0.0,3.9\n2026-03-30,13,16.9,0.0,0.0,3.2\n2026-03-30,14,17.7,0.0,0.0,4.2\n2026-03-30,15,19.0,0.0,0.0,3.2\n2026-03-30,16,18.9,0.0,0.0,4.1\n2026-03-30,17,18.9,0.0,0.0,5.9\n2026-03-30,18,17.6,0.0,0.0,8.1\n2026-03-30,19,15.9,0.0,0.0,6.6\n2026-03-30,20,14.6,0.0,0.0,3.3\n2026-03-30,21,14.5,0.3,0.0,6.4\n2026-03-30,22,13.4,0.5,0.0,0.4\n2026-03-30,23,12.2,1.2,0.0,3.6\n2026-03-31,0,11.6,1.3,0.0,4.3\n2026-03-31,1,11.3,1.4,0.0,6.0\n2026-03-31,2,10.7,1.6,0.0,13.0\n2026-03-31,3,11.0,1.3,0.0,17.0\n2026-03-31,4,10.6,0.9,0.0,17.7\n2026-03-31,5,10.4,0.6,0.0,15.5\n2026-03-31,6,10.2,0.4,0.0,15.0\n2026-03-31,7,10.0,0.1,0.0,14.5\n2026-03-31,8,10.5,0.1,0.0,12.1\n2026-03-31,9,11.5,0.0,0.0,12.6\n2026-03-31,10,13.8,0.0,0.0,11.1\n2026-03-31,11,15.7,0.0,0.0,10.9\n2026-03-31,12,17.4,0.0,0.0,9.8\n2026-03-31,13,17.9,0.0,0.0,7.3\n2026-03-31,14,18.5,0.0,0.0,6.5\n2026-03-31,15,17.0,0.0,0.0,7.9\n2026-03-31,16,17.4,0.0,0.0,5.5\n2026-03-31,17,17.0,0.0,0.0,1.5\n2026-03-31,18,16.1,0.0,0.0,2.7\n2026-03-31,19,14.2,0.0,0.0,4.9\n2026-03-31,20,12.8,0.0,0.0,7.7\n2026-03-31,21,12.7,0.0,0.0,3.9\n2026-03-31,22,13.0,0.0,0.0,3.9\n2026-03-31,23,12.0,0.0,0.0,1.0\n2026-04-01,0,11.4,0.0,0.0,1.5\n2026-04-01,1,11.0,0.0,0.0,4.3\n2026-04-01,2,11.1,0.0,0.0,8.1\n2026-04-01,3,10.8,0.0,0.0,5.2\n2026-04-01,4,10.9,0.0,0.0,4.4\n2026-04-01,5,9.8,0.0,0.0,4.3\n2026-04-01,6,9.8,0.0,0.0,6.0\n2026-04-01,7,9.9,0.0,0.0,7.0\n2026-04-01,8,11.3,0.0,0.0,5.0\n2026-04-01,9,11.8,0.0,0.0,4.7\n2026-04-01,10,13.2,0.0,0.0,2.9\n2026-04-01,11,15.0,0.0,0.0,6.5\n2026-04-01,12,15.8,0.0,0.0,8.3\n2026-04-01,13,14.9,0.0,0.0,11.0\n2026-04-01,14,14.4,0.0,0.0,11.9\n2026-04-01,15,12.7,0.1,0.0,14.4\n2026-04-01,16,12.8,0.0,0.0,11.5\n2026-04-01,17,12.6,0.0,0.0,11.1\n2026-04-01,18,10.9,0.0,0.0,10.4\n2026-04-01,19,9.0,0.0,0.0,8.8\n2026-04-01,20,8.6,0.0,0.0,7.6\n2026-04-01,21,9.3,0.0,0.0,10.7\n2026-04-01,22,9.6,0.0,0.0,10.8\n2026-04-01,23,9.3,0.0,0.0,10.6\n2026-04-02,0,8.3,0.0,0.0,8.4\n2026-04-02,1,7.7,0.0,0.0,7.6\n2026-04-02,2,7.2,0.0,0.0,7.2\n2026-04-02,3,6.7,0.0,0.0,7.4\n2026-04-02,4,6.5,0.0,0.0,6.7\n2026-04-02,5,6.2,0.0,0.0,5.9\n2026-04-02,6,5.9,0.0,0.0,5.2\n2026-04-02,7,6.2,0.0,0.0,5.1\n2026-04-02,8,8.0,0.0,0.0,4.8\n2026-04-02,9,9.7,0.0,0.0,7.5\n2026-04-02,10,11.3,0.0,0.0,10.5\n2026-04-02,11,12.4,0.0,0.0,10.5\n2026-04-02,12,13.8,0.0,0.0,9.4\n2026-04-02,13,14.4,0.0,0.0,12.5\n2026-04-02,14,14.1,0.0,0.0,14.9\n2026-04-02,15,13.5,0.0,0.0,16.0\n2026-04-02,16,12.8,0.0,0.0,15.2\n2026-04-02,17,11.9,0.0,0.0,14.2\n2026-04-02,18,10.2,0.0,0.0,12.4\n2026-04-02,19,8.3,0.0,0.0,12.9\n2026-04-02,20,7.8,0.0,0.0,11.1\n2026-04-02,21,8.0,0.0,0.0,9.2\n2026-04-02,22,8.1,0.0,0.0,9.6\n2026-04-02,23,7.7,0.0,0.0,7.2\n2026-04-03,0,7.4,0.0,0.0,5.5\n2026-04-03,1,7.3,0.0,0.0,6.7\n2026-04-03,2,7.4,0.0,0.0,7.2\n2026-04-03,3,7.4,0.0,0.0,7.3\n2026-04-03,4,7.4,0.0,0.0,9.4\n2026-04-03,5,7.1,0.0,0.0,8.2\n2026-04-03,6,6.9,0.0,0.0,8.9\n2026-04-03,7,7.2,0.0,0.0,8.6\n2026-04-03,8,8.4,0.0,0.0,9.1\n2026-04-03,9,10.3,0.0,0.0,7.8\n2026-04-03,10,12.8,0.0,0.0,7.8\n2026-04-03,11,15.6,0.0,0.0,7.9\n2026-04-03,12,17.7,0.0,0.0,8.6\n2026-04-03,13,19.3,0.0,0.0,10.3\n2026-04-03,14,20.1,0.0,0.0,14.1\n2026-04-03,15,20.1,0.0,0.0,13.7\n2026-04-03,16,20.3,0.0,0.0,13.9\n2026-04-03,17,17.4,0.0,0.0,12.5\n2026-04-03,18,15.0,0.0,0.0,9.0\n2026-04-03,19,14.0,0.0,0.0,5.9\n2026-04-03,20,13.5,0.0,0.0,7.2\n2026-04-03,21,13.1,0.0,0.0,7.1\n2026-04-03,22,13.5,0.0,0.0,7.4\n2026-04-03,23,12.7,0.0,0.0,7.1\n2026-04-04,0,12.0,0.0,0.0,6.0\n2026-04-04,1,12.5,0.0,0.0,9.3\n2026-04-04,2,12.9,0.0,0.0,12.0\n2026-04-04,3,12.8,0.0,0.0,7.5\n2026-04-04,4,12.9,0.0,0.0,3.9\n2026-04-04,5,12.8,0.0,0.0,6.1\n2026-04-04,6,12.2,0.1,0.0,9.8\n2026-04-04,7,11.5,0.3,0.0,12.7\n2026-04-04,8,10.9,0.8,0.0,13.2\n2026-04-04,9,11.9,0.9,0.0,9.0\n2026-04-04,10,14.3,0.0,0.0,7.6\n2026-04-04,11,16.9,0.0,0.0,7.7\n2026-04-04,12,19.2,0.0,0.0,9.7\n2026-04-04,13,18.6,0.0,0.0,19.1\n2026-04-04,14,16.1,0.0,0.0,24.4\n2026-04-04,15,15.8,0.0,0.0,21.4\n2026-04-04,16,14.8,0.0,0.0,19.4\n2026-04-04,17,13.1,0.0,0.0,15.8\n2026-04-04,18,10.9,0.0,0.0,13.3\n2026-04-04,19,9.2,0.0,0.0,11.1\n2026-04-04,20,8.4,0.0,0.0,10.3\n2026-04-04,21,9.1,0.0,0.0,9.9\n2026-04-04,22,9.1,0.0,0.0,9.9\n2026-04-04,23,8.8,0.0,0.0,9.3\n2026-04-05,0,7.6,0.0,0.0,11.1\n2026-04-05,1,7.0,0.0,0.0,10.7\n2026-04-05,2,6.5,0.0,0.0,10.4\n2026-04-05,3,6.7,0.0,0.0,10.1\n2026-04-05,4,7.1,0.0,0.0,9.1\n2026-04-05,5,7.1,0.0,0.0,6.0\n2026-04-05,6,7.0,0.0,0.0,3.1\n2026-04-05,7,6.7,0.0,0.0,4.0\n2026-04-05,8,7.7,0.0,0.0,6.1\n2026-04-05,9,8.3,0.0,0.0,8.7\n2026-04-05,10,10.0,0.0,0.0,9.7\n2026-04-05,11,11.2,0.0,0.0,9.8\n2026-04-05,12,12.1,0.0,0.0,9.6\n2026-04-05,13,13.6,0.0,0.0,9.1\n2026-04-05,14,14.4,0.0,0.0,12.0\n2026-04-05,15,14.3,0.0,0.0,11.9\n2026-04-05,16,13.7,0.0,0.0,11.0\n2026-04-05,17,12.7,0.0,0.0,9.8\n2026-04-05,18,11.4,0.0,0.0,7.1\n2026-04-05,19,10.1,0.0,0.0,4.7\n2026-04-05,20,9.6,0.0,0.0,1.3\n2026-04-05,21,10.2,0.0,0.0,9.9\n2026-04-05,22,7.9,1.7,0.0,17.7\n2026-04-05,23,7.5,1.4,0.0,3.2\n2026-04-06,0,8.2,0.1,0.0,5.9\n2026-04-06,1,8.6,0.0,0.0,8.3\n2026-04-06,2,8.1,0.0,0.0,10.0\n2026-04-06,3,8.6,0.0,0.0,7.1\n2026-04-06,4,9.4,0.0,0.0,3.6\n2026-04-06,5,9.4,0.0,0.0,3.4\n2026-04-06,6,9.3,0.5,0.0,0.3\n2026-04-06,7,9.6,0.0,0.0,4.0\n2026-04-06,8,10.3,0.0,0.0,4.2\n2026-04-06,9,9.3,0.0,0.0,4.4\n2026-04-06,10,10.9,0.1,0.0,6.4\n2026-04-06,11,10.3,0.2,0.0,5.3\n2026-04-06,12,11.9,0.0,0.0,6.7\n2026-04-06,13,12.5,0.1,0.0,13.6\n2026-04-06,14,12.4,0.0,0.0,16.9\n2026-04-06,15,12.1,0.0,0.0,13.6\n2026-04-06,16,13.0,0.0,0.0,14.2\n2026-04-06,17,13.4,0.0,0.0,14.4\n2026-04-06,18,11.6,0.0,0.0,15.0\n2026-04-06,19,9.6,0.0,0.0,16.4\n2026-04-06,20,8.5,0.0,0.0,17.0\n2026-04-06,21,7.7,0.0,0.0,16.2\n2026-04-06,22,6.6,0.0,0.0,12.9\n2026-04-06,23,5.8,0.0,0.0,10.7\n2026-04-07,0,5.6,0.0,0.0,11.5\n2026-04-07,1,5.2,0.0,0.0,11.2\n2026-04-07,2,4.7,0.0,0.0,9.9\n2026-04-07,3,3.9,0.0,0.0,9.9\n2026-04-07,4,3.6,0.0,0.0,9.2\n2026-04-07,5,3.3,0.0,0.0,9.7\n2026-04-07,6,3.2,0.0,0.0,10.5\n2026-04-07,7,3.6,0.0,0.0,11.5\n2026-04-07,8,5.4,0.0,0.0,10.3\n2026-04-07,9,7.6,0.0,0.0,11.6\n2026-04-07,10,9.2,0.0,0.0,13.0\n2026-04-07,11,10.4,0.0,0.0,16.6\n2026-04-07,12,10.8,0.0,0.0,17.0\n2026-04-07,13,10.8,0.0,0.0,17.2\n2026-04-07,14,10.7,0.0,0.0,19.3\n2026-04-07,15,10.5,0.0,0.0,19.2\n2026-04-07,16,10.5,0.0,0.0,19.5\n2026-04-07,17,9.7,0.0,0.0,19.3\n2026-04-07,18,8.3,0.0,0.0,15.7\n2026-04-07,19,7.0,0.0,0.0,14.0\n2026-04-07,20,6.8,0.0,0.0,14.4\n2026-04-07,21,6.3,0.0,0.0,14.1\n2026-04-07,22,6.3,0.0,0.0,12.5\n2026-04-07,23,6.0,0.0,0.0,9.0\n2026-04-08,0,5.3,0.0,0.0,4.8\n2026-04-08,1,4.2,0.0,0.0,3.9\n2026-04-08,2,3.7,0.0,0.0,3.3\n2026-04-08,3,3.1,0.0,0.0,2.3\n2026-04-08,4,3.0,0.0,0.0,1.3\n2026-04-08,5,2.6,0.0,0.0,0.9\n2026-04-08,6,2.1,0.0,0.0,1.9\n2026-04-08,7,2.8,0.0,0.0,5.6\n2026-04-08,8,5.1,0.0,0.0,4.6\n2026-04-08,9,7.2,0.0,0.0,4.6\n2026-04-08,10,9.3,0.0,0.0,5.0\n2026-04-08,11,10.9,0.0,0.0,6.4\n2026-04-08,12,11.8,0.0,0.0,10.6\n2026-04-08,13,12.4,0.0,0.0,10.1\n2026-04-08,14,13.2,0.0,0.0,8.7\n2026-04-08,15,13.4,0.0,0.0,9.7\n2026-04-08,16,13.3,0.0,0.0,9.9\n2026-04-08,17,12.6,0.0,0.0,9.7\n2026-04-08,18,11.2,0.0,0.0,9.1\n2026-04-08,19,9.2,0.0,0.0,6.9\n2026-04-08,20,7.9,0.0,0.0,5.2\n2026-04-08,21,7.3,0.0,0.0,2.0\n2026-04-08,22,7.8,0.0,0.0,0.7\n2026-04-08,23,7.3,0.0,0.0,1.7\n2026-04-09,0,6.7,0.0,0.0,2.4\n2026-04-09,1,6.4,0.0,0.0,6.1\n2026-04-09,2,6.2,0.0,0.0,8.9\n2026-04-09,3,6.1,0.0,0.0,8.7\n2026-04-09,4,6.7,0.0,0.0,8.5\n2026-04-09,5,7.0,0.0,0.0,8.0\n2026-04-09,6,7.5,0.0,0.0,10.0\n2026-04-09,7,8.4,0.0,0.0,10.3\n2026-04-09,8,9.5,0.0,0.0,10.4\n2026-04-09,9,9.7,0.0,0.0,10.3\n2026-04-09,10,10.4,0.5,0.0,11.7\n2026-04-09,11,10.1,0.4,0.0,14.6\n2026-04-09,12,8.8,1.3,0.0,17.4\n2026-04-09,13,9.2,2.4,0.0,16.6\n2026-04-09,14,9.3,3.8,0.0,14.7\n2026-04-09,15,9.7,3.4,0.0,19.1\n2026-04-09,16,10.4,0.6,0.0,18.2\n2026-04-09,17,10.9,0.3,0.0,17.3\n2026-04-09,18,11.3,0.3,0.0,16.6\n2026-04-09,19,11.5,0.8,0.0,16.5\n2026-04-09,20,11.7,3.4,0.0,14.2\n2026-04-09,21,11.7,1.9,0.0,8.2\n2026-04-09,22,11.8,1.7,0.0,4.9\n2026-04-09,23,11.9,2.3,0.0,5.1\n2026-04-10,0,11.8,1.3,0.0,9.1\n2026-04-10,1,11.3,0.9,0.0,13.3\n2026-04-10,2,10.7,0.6,0.0,15.3\n2026-04-10,3,11.2,0.6,0.0,10.8\n2026-04-10,4,10.7,0.7,0.0,15.3\n2026-04-10,5,9.9,0.5,0.0,17.4\n2026-04-10,6,9.2,0.3,0.0,19.7\n2026-04-10,7,8.9,0.1,0.0,15.6\n2026-04-10,8,9.4,0.1,0.0,18.0\n2026-04-10,9,9.1,0.0,0.0,16.7\n2026-04-10,10,9.1,0.1,0.0,16.8\n2026-04-10,11,9.3,0.0,0.0,17.2\n2026-04-10,12,9.9,0.0,0.0,19.0\n2026-04-10,13,11.0,0.0,0.0,21.5\n2026-04-10,14,11.1,0.0,0.0,22.8\n2026-04-10,15,10.4,0.0,0.0,22.9\n2026-04-10,16,10.1,0.0,0.0,20.7\n2026-04-10,17,9.6,0.0,0.0,20.6\n2026-04-10,18,9.2,0.0,0.0,20.8\n2026-04-10,19,8.4,0.0,0.0,18.9\n2026-04-10,20,8.4,0.0,0.0,17.7\n2026-04-10,21,8.4,0.0,0.0,14.9\n2026-04-10,22,8.0,0.0,0.0,14.2\n2026-04-10,23,8.0,0.0,0.0,13.3\n2026-04-11,0,7.8,0.0,0.0,10.5\n2026-04-11,1,7.8,0.0,0.0,8.9\n2026-04-11,2,7.7,0.0,0.0,7.5\n2026-04-11,3,7.8,0.0,0.0,7.5\n2026-04-11,4,7.9,0.0,0.0,8.1\n2026-04-11,5,8.0,0.0,0.0,10.2\n2026-04-11,6,7.8,0.0,0.0,8.0\n2026-04-11,7,8.1,0.0,0.0,7.8\n2026-04-11,8,9.0,0.0,0.0,6.2\n2026-04-11,9,9.0,0.0,0.0,4.0\n2026-04-11,10,10.7,0.0,0.0,6.7\n2026-04-11,11,11.5,0.0,0.0,10.9\n2026-04-11,12,12.7,0.0,0.0,11.9\n2026-04-11,13,13.8,0.0,0.0,14.5\n2026-04-11,14,14.6,0.0,0.0,14.2\n2026-04-11,15,13.9,0.0,0.0,13.9\n2026-04-11,16,13.6,0.0,0.0,14.9\n2026-04-11,17,12.7,0.0,0.0,13.1\n2026-04-11,18,11.2,0.0,0.0,10.3\n2026-04-11,19,9.8,0.0,0.0,8.8\n2026-04-11,20,8.6,0.0,0.0,9.4\n2026-04-11,21,8.2,0.0,0.0,4.8\n2026-04-11,22,8.8,0.0,0.0,5.8\n2026-04-11,23,8.6,0.0,0.0,3.9\n2026-04-12,0,8.4,0.0,0.0,2.8\n2026-04-12,1,7.9,0.0,0.0,1.7\n2026-04-12,2,7.4,0.0,0.0,1.1\n2026-04-12,3,6.3,0.0,0.0,1.7\n2026-04-12,4,6.0,0.0,0.0,1.3\n2026-04-12,5,5.9,0.0,0.0,2.6\n2026-04-12,6,5.9,0.0,0.0,5.3\n2026-04-12,7,7.1,0.0,0.0,6.6\n2026-04-12,8,9.3,0.0,0.0,7.6\n2026-04-12,9,11.5,0.0,0.0,4.5\n2026-04-12,10,13.7,0.0,0.0,3.4\n2026-04-12,11,15.8,0.0,0.0,2.8\n2026-04-12,12,17.7,0.0,0.0,1.8\n2026-04-12,13,18.7,0.0,0.0,4.7\n2026-04-12,14,19.6,0.0,0.0,4.6\n2026-04-12,15,20.5,0.0,0.0,5.5\n2026-04-12,16,20.5,0.0,0.0,4.9\n2026-04-12,17,19.0,0.0,0.0,8.9\n2026-04-12,18,15.7,0.0,0.0,8.5\n2026-04-12,19,12.8,0.0,0.0,7.2\n2026-04-12,20,11.7,0.0,0.0,5.4\n2026-04-12,21,11.6,0.0,0.0,6.3\n2026-04-12,22,11.4,0.0,0.0,6.2\n2026-04-12,23,10.3,0.0,0.0,4.1\n2026-04-13,0,10.4,0.0,0.0,2.1\n2026-04-13,1,10.0,0.0,0.0,5.1\n2026-04-13,2,9.4,0.0,0.0,8.5\n2026-04-13,3,8.5,0.0,0.0,4.7\n2026-04-13,4,8.5,0.0,0.0,0.9\n2026-04-13,5,7.8,0.0,0.0,1.1\n2026-04-13,6,7.0,0.0,0.0,3.4\n2026-04-13,7,8.1,0.0,0.0,4.3\n2026-04-13,8,10.6,0.0,0.0,5.8\n2026-04-13,9,13.6,0.0,0.0,2.6\n2026-04-13,10,16.2,0.0,0.0,4.2\n2026-04-13,11,18.4,0.0,0.0,4.0\n2026-04-13,12,20.2,0.0,0.0,6.0\n2026-04-13,13,21.7,0.0,0.0,6.1\n2026-04-13,14,22.9,0.0,0.0,5.1\n2026-04-13,15,23.8,0.0,0.0,4.4\n2026-04-13,16,23.3,0.0,0.0,6.8\n2026-04-13,17,22.2,0.0,0.0,7.5\n2026-04-13,18,20.0,0.0,0.0,8.4\n2026-04-13,19,17.1,0.0,0.0,7.7\n2026-04-13,20,15.4,0.0,0.0,7.6\n2026-04-13,21,14.5,0.0,0.0,8.8\n2026-04-13,22,14.6,0.0,0.0,5.6\n2026-04-13,23,14.8,0.0,0.0,7.0\n2026-04-14,0,14.2,0.0,0.0,5.5\n2026-04-14,1,13.4,0.0,0.0,10.7\n2026-04-14,2,13.6,0.0,0.0,12.9\n2026-04-14,3,13.9,0.0,0.0,9.7\n2026-04-14,4,13.5,0.0,0.0,8.0\n2026-04-14,5,12.9,0.0,0.0,7.3\n2026-04-14,6,12.1,0.0,0.0,6.6\n2026-04-14,7,12.5,0.0,0.0,7.8\n2026-04-14,8,14.5,0.0,0.0,8.6\n2026-04-14,9,17.2,0.0,0.0,6.1\n2026-04-14,10,19.8,0.0,0.0,6.1\n2026-04-14,11,21.6,0.0,0.0,5.4\n2026-04-14,12,22.9,0.0,0.0,3.9\n2026-04-14,13,23.6,0.0,0.0,4.0\n2026-04-14,14,24.2,0.0,0.0,4.4\n2026-04-14,15,24.7,0.0,0.0,5.5\n2026-04-14,16,23.6,0.0,0.0,9.0\n2026-04-14,17,22.3,0.0,0.0,10.5\n2026-04-14,18,19.8,0.0,0.0,9.1\n2026-04-14,19,16.9,0.0,0.0,6.1\n2026-04-14,20,14.8,0.0,0.0,3.6\n2026-04-14,21,13.5,0.0,0.0,1.5\n2026-04-14,22,14.0,0.0,0.0,0.8\n2026-04-14,23,13.2,0.0,0.0,10.7\n2026-04-15,0,12.9,0.0,0.0,7.3\n2026-04-15,1,12.8,0.0,0.0,9.4\n2026-04-15,2,13.3,0.0,0.0,11.6\n2026-04-15,3,12.7,0.0,0.0,6.4\n2026-04-15,4,12.7,0.0,0.0,0.2\n2026-04-15,5,11.8,0.0,0.0,5.6\n2026-04-15,6,10.8,0.0,0.0,1.5\n2026-04-15,7,12.0,0.0,0.0,4.2\n2026-04-15,8,14.6,0.0,0.0,2.9\n2026-04-15,9,16.3,0.0,0.0,1.3\n2026-04-15,10,18.3,0.0,0.0,2.3\n2026-04-15,11,20.1,0.0,0.0,3.8\n2026-04-15,12,21.2,0.0,0.0,6.5\n2026-04-15,13,22.0,0.0,0.0,8.3\n2026-04-15,14,21.9,0.0,0.0,11.7\n2026-04-15,15,21.6,0.0,0.0,13.6\n2026-04-15,16,20.3,0.0,0.0,12.8\n2026-04-15,17,18.6,0.0,0.0,10.3\n2026-04-15,18,17.1,0.0,0.0,7.9\n2026-04-15,19,15.1,0.0,0.0,5.4\n2026-04-15,20,13.6,0.0,0.0,5.5\n2026-04-15,21,12.8,0.0,0.0,3.4\n2026-04-15,22,13.4,0.0,0.0,8.6\n2026-04-15,23,15.9,0.0,0.0,16.1\n2026-04-16,0,14.6,0.0,0.0,18.3\n2026-04-16,1,13.0,0.0,0.0,14.7\n2026-04-16,2,12.0,0.0,0.0,13.7\n2026-04-16,3,12.3,0.0,0.0,16.1\n2026-04-16,4,12.1,0.0,0.0,12.3\n2026-04-16,5,11.4,0.0,0.0,7.9\n2026-04-16,6,10.8,0.0,0.0,8.5\n2026-04-16,7,11.4,0.0,0.0,10.3\n2026-04-16,8,13.7,0.0,0.0,7.3\n2026-04-16,9,16.7,0.0,0.0,9.4\n2026-04-16,10,19.1,0.0,0.0,10.8\n2026-04-16,11,20.9,0.0,0.0,10.8\n2026-04-16,12,22.8,0.0,0.0,11.4\n2026-04-16,13,24.2,0.0,0.0,9.8\n2026-04-16,14,25.2,0.0,0.0,8.8\n2026-04-16,15,25.7,0.0,0.0,3.8\n2026-04-16,16,24.3,0.0,0.0,9.2\n2026-04-16,17,22.1,0.0,0.0,13.4\n2026-04-16,18,20.1,0.0,0.0,11.2\n2026-04-16,19,17.6,0.0,0.0,4.6\n2026-04-16,20,16.7,0.0,0.0,1.2\n2026-04-16,21,16.8,0.0,0.0,3.5\n2026-04-16,22,17.1,0.0,0.0,2.7\n2026-04-16,23,16.8,0.0,0.0,5.1\n2026-04-17,0,16.6,0.0,0.0,9.7\n2026-04-17,1,15.2,0.0,0.0,7.5\n2026-04-17,2,14.4,0.0,0.0,8.7\n2026-04-17,3,14.9,0.0,0.0,8.1\n2026-04-17,4,14.8,0.0,0.0,6.9\n2026-04-17,5,13.5,0.0,0.0,3.6\n2026-04-17,6,13.4,0.0,0.0,7.8\n2026-04-17,7,13.5,0.0,0.0,5.6\n2026-04-17,8,13.8,0.0,0.0,7.7\n2026-04-17,9,15.6,0.0,0.0,5.4\n2026-04-17,10,16.9,0.0,0.0,4.2\n2026-04-17,11,18.8,0.0,0.0,6.0\n2026-04-17,12,19.6,0.0,0.0,10.0\n2026-04-17,13,17.6,0.0,0.0,13.3\n2026-04-17,14,18.5,0.0,0.0,14.3\n2026-04-17,15,16.7,0.0,0.0,14.5\n2026-04-17,16,16.8,0.0,0.0,12.7\n2026-04-17,17,16.2,0.0,0.0,12.0\n2026-04-17,18,15.0,0.0,0.0,10.0\n2026-04-17,19,13.5,0.0,0.0,6.9\n2026-04-17,20,12.5,0.0,0.0,6.9\n2026-04-17,21,12.3,0.0,0.0,4.3\n2026-04-17,22,12.5,0.0,0.0,4.1\n2026-04-17,23,12.0,0.0,0.0,4.0\n2026-04-18,0,11.7,0.0,0.0,5.2\n2026-04-18,1,11.5,0.0,0.0,7.4\n2026-04-18,2,11.1,0.0,0.0,6.5\n2026-04-18,3,10.3,0.0,0.0,4.0\n2026-04-18,4,10.2,0.0,0.0,3.3\n2026-04-18,5,9.8,0.0,0.0,2.0\n2026-04-18,6,9.3,0.0,0.0,1.7\n2026-04-18,7,10.4,0.0,0.0,3.4\n2026-04-18,8,13.2,0.0,0.0,3.5\n2026-04-18,9,15.2,0.0,0.0,4.2\n2026-04-18,10,17.8,0.0,0.0,5.9\n2026-04-18,11,20.0,0.0,0.0,6.0\n2026-04-18,12,22.0,0.0,0.0,4.7\n2026-04-18,13,23.6,0.0,0.0,3.3\n2026-04-18,14,24.8,0.0,0.0,4.2\n2026-04-18,15,24.6,0.0,0.0,8.8\n2026-04-18,16,23.5,0.0,0.0,11.4\n2026-04-18,17,22.2,0.0,0.0,11.0\n2026-04-18,18,20.7,0.0,0.0,8.9\n2026-04-18,19,18.4,0.0,0.0,8.5\n2026-04-18,20,16.1,0.0,0.0,8.5\n2026-04-18,21,14.9,0.0,0.0,5.2\n2026-04-18,22,14.5,0.0,0.0,5.2\n2026-04-18,23,13.7,0.0,0.0,2.0\n2026-04-19,0,12.7,0.0,0.0,3.8\n2026-04-19,1,11.8,0.0,0.0,6.1\n2026-04-19,2,11.8,0.0,0.0,6.0\n2026-04-19,3,12.3,0.0,0.0,6.6\n2026-04-19,4,12.6,0.0,0.0,5.1\n2026-04-19,5,12.4,0.0,0.0,4.8\n2026-04-19,6,12.3,0.0,0.0,1.8\n2026-04-19,7,13.3,0.0,0.0,5.2\n2026-04-19,8,15.0,0.0,0.0,3.7\n2026-04-19,9,17.8,0.0,0.0,1.9\n2026-04-19,10,21.7,0.0,0.0,0.0\n2026-04-19,11,24.2,0.0,0.0,2.6\n2026-04-19,12,26.0,0.0,0.0,7.4\n2026-04-19,13,26.1,0.0,0.0,11.7\n2026-04-19,14,26.0,0.0,0.0,12.3\n2026-04-19,15,25.7,0.0,0.0,11.9\n2026-04-19,16,25.2,0.0,0.0,11.5\n2026-04-19,17,23.7,0.0,0.0,12.2\n2026-04-19,18,21.1,0.0,0.0,9.8\n2026-04-19,19,18.0,0.0,0.0,7.6\n2026-04-19,20,15.1,0.0,0.0,7.7\n2026-04-19,21,13.6,0.0,0.0,6.7\n2026-04-19,22,11.8,0.0,0.0,8.5\n2026-04-19,23,9.8,0.0,0.0,7.0\n2026-04-20,0,9.8,0.0,0.0,4.8\n2026-04-20,1,9.6,0.0,0.0,6.0\n2026-04-20,2,9.2,0.0,0.0,6.7\n2026-04-20,3,10.1,0.0,0.0,8.2\n2026-04-20,4,10.1,0.0,0.0,7.8\n2026-04-20,5,9.9,0.0,0.0,6.5\n2026-04-20,6,9.8,0.0,0.0,6.1\n2026-04-20,7,9.7,0.0,0.0,7.0\n2026-04-20,8,10.2,0.0,0.0,5.9\n2026-04-20,9,12.9,1.0,0.0,5.5\n2026-04-20,10,13.9,0.0,0.0,10.0\n2026-04-20,11,15.0,0.1,0.0,17.2\n2026-04-20,12,13.9,0.3,0.0,18.5\n2026-04-20,13,14.0,0.1,0.0,15.2\n2026-04-20,14,13.7,0.2,0.0,20.7\n2026-04-20,15,11.8,0.0,0.0,18.7\n2026-04-20,16,11.2,0.1,0.0,20.0\n2026-04-20,17,12.0,0.0,0.0,21.4\n2026-04-20,18,11.3,0.0,0.0,18.8\n2026-04-20,19,10.9,0.0,0.0,20.9\n2026-04-20,20,10.1,0.0,0.0,18.7\n2026-04-20,21,9.3,0.0,0.0,16.9\n2026-04-20,22,8.7,0.0,0.0,12.7\n2026-04-20,23,8.1,0.0,0.0,14.6\n2026-04-21,0,7.9,0.0,0.0,14.6\n2026-04-21,1,7.7,0.0,0.0,11.9\n2026-04-21,2,7.4,0.0,0.0,8.3\n2026-04-21,3,6.6,0.0,0.0,7.8\n2026-04-21,4,6.2,0.0,0.0,4.7\n2026-04-21,5,6.2,0.0,0.0,6.4\n2026-04-21,6,6.2,0.0,0.0,8.1\n2026-04-21,7,7.2,0.0,0.0,10.2\n2026-04-21,8,9.6,0.0,0.0,10.4\n2026-04-21,9,11.5,0.0,0.0,12.8\n2026-04-21,10,13.4,0.0,0.0,15.3\n2026-04-21,11,15.0,0.0,0.0,17.5\n2026-04-21,12,15.7,0.0,0.0,19.0\n2026-04-21,13,16.1,0.0,0.0,18.0\n2026-04-21,14,15.7,0.0,0.0,17.6\n2026-04-21,15,14.6,0.0,0.0,16.5\n2026-04-21,16,13.8,0.0,0.0,17.2\n2026-04-21,17,12.9,0.0,0.0,17.0\n2026-04-21,18,12.2,0.0,0.0,13.7\n2026-04-21,19,11.3,0.0,0.0,10.9\n2026-04-21,20,10.7,0.0,0.0,11.0\n2026-04-21,21,10.3,0.0,0.0,11.1\n2026-04-21,22,10.0,0.0,0.0,8.9\n2026-04-21,23,9.8,0.0,0.0,7.9\n2026-04-22,0,9.7,0.0,0.0,7.2\n2026-04-22,1,9.4,0.0,0.0,6.7\n2026-04-22,2,9.3,0.0,0.0,5.7\n2026-04-22,3,8.9,0.0,0.0,6.1\n2026-04-22,4,8.8,0.0,0.0,6.5\n2026-04-22,5,8.6,0.0,0.0,5.9\n2026-04-22,6,8.5,0.0,0.0,6.3\n2026-04-22,7,9.1,0.0,0.0,6.0\n2026-04-22,8,11.3,0.0,0.0,4.5\n2026-04-22,9,12.5,0.0,0.0,4.7\n2026-04-22,10,13.5,0.0,0.0,5.8\n2026-04-22,11,15.1,0.0,0.0,10.1\n2026-04-22,12,15.4,0.0,0.0,8.2\n2026-04-22,13,16.0,0.0,0.0,8.6\n2026-04-22,14,17.7,0.0,0.0,10.8\n2026-04-22,15,17.4,0.0,0.0,13.5\n2026-04-22,16,17.3,0.0,0.0,14.6\n2026-04-22,17,16.4,0.0,0.0,12.5\n2026-04-22,18,15.2,0.0,0.0,10.7\n2026-04-22,19,13.6,0.0,0.0,9.4\n2026-04-22,20,12.4,0.0,0.0,10.6\n2026-04-22,21,11.9,0.0,0.0,11.1\n2026-04-22,22,12.5,0.0,0.0,11.5\n2026-04-22,23,12.3,0.0,0.0,10.8\n2026-04-23,0,11.7,0.0,0.0,9.5\n2026-04-23,1,11.3,0.0,0.0,11.9\n2026-04-23,2,11.1,0.0,0.0,11.5\n2026-04-23,3,10.4,0.0,0.0,7.2\n2026-04-23,4,10.4,0.0,0.0,8.9\n2026-04-23,5,9.8,0.0,0.0,11.1\n2026-04-23,6,9.7,0.0,0.0,10.8\n2026-04-23,7,11.0,0.0,0.0,12.3\n2026-04-23,8,13.2,0.0,0.0,12.9\n2026-04-23,9,15.3,0.0,0.0,15.9\n2026-04-23,10,17.3,0.0,0.0,14.9\n2026-04-23,11,19.7,0.0,0.0,13.8\n2026-04-23,12,21.4,0.0,0.0,13.9\n2026-04-23,13,22.3,0.0,0.0,13.1\n2026-04-23,14,22.6,0.0,0.0,12.2\n2026-04-23,15,23.1,0.0,0.0,11.8\n2026-04-23,16,23.4,0.0,0.0,11.0\n2026-04-23,17,22.7,0.0,0.0,8.8\n2026-04-23,18,21.6,0.0,0.0,8.7\n2026-04-23,19,20.1,0.0,0.0,10.7\n2026-04-23,20,18.7,0.0,0.0,14.6\n2026-04-23,21,17.9,0.0,0.0,12.0\n2026-04-23,22,16.9,0.0,0.0,12.4\n2026-04-23,23,15.5,0.0,0.0,11.2\n2026-04-24,0,14.6,0.0,0.0,10.0\n2026-04-24,1,13.6,0.0,0.0,9.3\n2026-04-24,2,12.6,0.0,0.0,11.3\n2026-04-24,3,12.2,0.0,0.0,11.4\n2026-04-24,4,11.6,0.0,0.0,9.1\n2026-04-24,5,11.0,0.0,0.0,7.9\n2026-04-24,6,10.4,0.0,0.0,7.4\n2026-04-24,7,11.1,0.0,0.0,8.3\n2026-04-24,8,13.8,0.0,0.0,5.8\n2026-04-24,9,16.5,0.0,0.0,7.4\n2026-04-24,10,19.2,0.0,0.0,5.2\n2026-04-24,11,21.2,0.0,0.0,3.6\n2026-04-24,12,22.9,0.0,0.0,2.6\n2026-04-24,13,24.3,0.0,0.0,1.4\n2026-04-24,14,25.1,0.0,0.0,4.0\n2026-04-24,15,24.4,0.0,0.0,11.0\n2026-04-24,16,23.5,0.0,0.0,13.3\n2026-04-24,17,22.1,0.0,0.0,12.3\n2026-04-24,18,20.8,0.0,0.0,9.5\n2026-04-24,19,18.9,0.0,0.0,8.4\n2026-04-24,20,17.0,0.0,0.0,9.1\n2026-04-24,21,15.5,0.0,0.0,5.0\n2026-04-24,22,14.5,0.0,0.0,4.1\n2026-04-24,23,13.3,0.0,0.0,2.8\n2026-04-25,0,12.7,0.0,0.0,3.9\n2026-04-25,1,12.3,0.0,0.0,1.8\n2026-04-25,2,11.6,0.0,0.0,2.0\n2026-04-25,3,10.7,0.0,0.0,5.9\n2026-04-25,4,9.8,0.0,0.0,6.5\n2026-04-25,5,9.2,0.0,0.0,6.3\n2026-04-25,6,8.8,0.0,0.0,5.7\n2026-04-25,7,10.3,0.0,0.0,7.8\n2026-04-25,8,13.8,0.0,0.0,5.1\n2026-04-25,9,16.5,0.0,0.0,3.5\n2026-04-25,10,19.4,0.0,0.0,4.0\n2026-04-25,11,21.7,0.0,0.0,3.1\n2026-04-25,12,23.6,0.0,0.0,6.0\n2026-04-25,13,24.2,0.0,0.0,11.9\n2026-04-25,14,23.8,0.0,0.0,13.5\n2026-04-25,15,23.1,0.0,0.0,12.8\n2026-04-25,16,22.1,0.0,0.0,12.7\n2026-04-25,17,20.7,0.0,0.0,12.0\n2026-04-25,18,19.0,0.0,0.0,10.4\n2026-04-25,19,16.9,0.0,0.0,7.1\n2026-04-25,20,15.2,0.0,0.0,6.2\n2026-04-25,21,14.9,0.0,0.0,4.8\n2026-04-25,22,15.0,0.0,0.0,6.6\n2026-04-25,23,14.4,0.0,0.0,6.2\n2026-04-26,0,13.2,0.0,0.0,4.7\n2026-04-26,1,12.8,0.0,0.0,0.9\n2026-04-26,2,11.8,0.0,0.0,3.0\n2026-04-26,3,10.5,0.0,0.0,1.8\n2026-04-26,4,10.0,0.0,0.0,0.9\n2026-04-26,5,9.8,0.0,0.0,0.4\n2026-04-26,6,9.3,0.0,0.0,2.2\n2026-04-26,7,10.9,0.0,0.0,6.0\n2026-04-26,8,13.5,0.0,0.0,3.9\n2026-04-26,9,16.0,0.0,0.0,4.2\n2026-04-26,10,18.4,0.0,0.0,4.9\n2026-04-26,11,20.4,0.0,0.0,4.2\n2026-04-26,12,21.9,0.0,0.0,6.9\n2026-04-26,13,22.4,0.0,0.0,11.0\n2026-04-26,14,22.3,0.0,0.0,13.9\n2026-04-26,15,21.5,0.0,0.0,15.8\n2026-04-26,16,20.8,0.0,0.0,15.2\n2026-04-26,17,19.9,0.0,0.0,13.3\n2026-04-26,18,18.6,0.0,0.0,10.4\n2026-04-26,19,16.5,0.0,0.0,9.3\n2026-04-26,20,14.8,0.0,0.0,7.7\n2026-04-26,21,14.0,0.0,0.0,7.8\n2026-04-26,22,13.8,0.0,0.0,8.9\n2026-04-26,23,13.3,0.0,0.0,7.7\n2026-04-27,0,12.9,0.0,0.0,5.9\n2026-04-27,1,12.4,0.0,0.0,4.9\n2026-04-27,2,12.0,0.0,0.0,4.0\n2026-04-27,3,11.5,0.0,0.0,4.2\n2026-04-27,4,11.1,0.0,0.0,2.8\n2026-04-27,5,11.2,0.0,0.0,4.1\n2026-04-27,6,10.9,0.0,0.0,6.1\n2026-04-27,7,11.7,0.0,0.0,1.7\n2026-04-27,8,13.3,0.0,0.0,1.9\n2026-04-27,9,15.2,0.0,0.0,2.1\n2026-04-27,10,18.0,0.0,0.0,2.2\n2026-04-27,11,19.9,0.0,0.0,7.3\n2026-04-27,12,20.2,0.0,0.0,13.5\n2026-04-27,13,19.7,0.0,0.0,14.5\n2026-04-27,14,19.8,0.0,0.0,10.8\n2026-04-27,15,18.3,0.0,0.0,9.5\n2026-04-27,16,17.2,0.0,0.0,6.2\n2026-04-27,17,16.2,0.0,0.0,11.9\n2026-04-27,18,12.5,1.5,0.0,8.9\n2026-04-27,19,12.8,0.0,0.0,4.2\n2026-04-27,20,11.9,1.8,0.0,3.0\n2026-04-27,21,11.8,0.0,0.0,7.8\n2026-04-27,22,12.0,1.8,0.0,4.4\n2026-04-27,23,11.9,0.0,0.0,7.7\n2026-04-28,0,12.2,0.0,0.0,10.7\n2026-04-28,1,12.2,0.0,0.0,15.2\n2026-04-28,2,11.9,0.0,0.0,12.9\n2026-04-28,3,10.9,0.0,0.0,12.5\n2026-04-28,4,11.0,0.1,0.0,14.3\n2026-04-28,5,10.8,0.1,0.0,14.2\n2026-04-28,6,10.6,0.0,0.0,15.0\n2026-04-28,7,10.3,0.0,0.0,16.4\n2026-04-28,8,11.0,0.0,0.0,14.9\n2026-04-28,9,12.1,0.0,0.0,15.0\n2026-04-28,10,13.1,0.0,0.0,15.6\n2026-04-28,11,12.6,0.1,0.0,18.4\n2026-04-28,12,13.8,0.1,0.0,18.0\n2026-04-28,13,14.4,0.0,0.0,17.5\n2026-04-28,14,15.0,0.0,0.0,17.5\n2026-04-28,15,14.7,0.0,0.0,15.4\n2026-04-28,16,14.6,0.0,0.0,15.3\n2026-04-28,17,13.8,0.0,0.0,15.8\n2026-04-28,18,13.0,0.0,0.0,13.1\n2026-04-28,19,12.3,0.0,0.0,9.0\n2026-04-28,20,11.8,0.0,0.0,3.8\n2026-04-28,21,12.7,0.0,0.0,9.3\n2026-04-28,22,12.3,0.0,0.0,8.6\n2026-04-28,23,11.1,0.0,0.0,5.4\n2026-04-29,0,10.4,0.0,0.0,3.9\n2026-04-29,1,9.9,0.0,0.0,5.2\n2026-04-29,2,9.7,0.0,0.0,6.1\n2026-04-29,3,9.2,0.0,0.0,6.2\n2026-04-29,4,8.7,0.0,0.0,6.3\n2026-04-29,5,8.2,0.0,0.0,7.1\n2026-04-29,6,7.9,0.0,0.0,6.2\n2026-04-29,7,9.0,0.0,0.0,5.1\n2026-04-29,8,11.2,0.0,0.0,2.9\n2026-04-29,9,12.9,0.0,0.0,5.0\n2026-04-29,10,14.8,0.0,0.0,5.8\n2026-04-29,11,16.5,0.0,0.0,7.5\n2026-04-29,12,17.9,0.0,0.0,8.1\n2026-04-29,13,18.4,0.0,0.0,9.3\n2026-04-29,14,18.6,0.0,0.0,11.6\n2026-04-29,15,18.6,0.0,0.0,11.7\n2026-04-29,16,18.0,0.0,0.0,13.9\n2026-04-29,17,16.5,0.0,0.0,14.8\n2026-04-29,18,15.1,0.0,0.0,12.1\n2026-04-29,19,13.6,0.0,0.0,10.7\n2026-04-29,20,12.5,0.0,0.0,6.7\n2026-04-29,21,12.1,0.0,0.0,3.4\n2026-04-29,22,12.2,0.0,0.0,3.1\n2026-04-29,23,12.0,0.0,0.0,4.8\n2026-04-30,0,11.4,0.0,0.0,3.0\n2026-04-30,1,11.0,0.0,0.0,3.1\n2026-04-30,2,10.5,0.0,0.0,4.2\n2026-04-30,3,9.9,0.0,0.0,3.1\n2026-04-30,4,9.5,0.0,0.0,2.5\n2026-04-30,5,9.2,0.0,0.0,1.1\n2026-04-30,6,8.7,0.0,0.0,2.8\n2026-04-30,7,10.4,0.0,0.0,3.6\n2026-04-30,8,13.2,0.0,0.0,1.9\n2026-04-30,9,15.3,0.0,0.0,1.7\n2026-04-30,10,17.8,0.0,0.0,0.9\n2026-04-30,11,19.2,0.0,0.0,4.8\n2026-04-30,12,20.2,0.0,0.0,7.7\n2026-04-30,13,21.0,0.0,0.0,10.3\n2026-04-30,14,19.9,0.0,0.0,16.6\n2026-04-30,15,19.7,0.0,0.0,18.0\n2026-04-30,16,19.0,0.0,0.0,17.2\n2026-04-30,17,19.0,0.0,0.0,16.4\n2026-04-30,18,17.7,0.0,0.0,16.7\n2026-04-30,19,15.9,0.0,0.0,11.8\n2026-04-30,20,14.8,0.0,0.0,13.8\n2026-04-30,21,14.6,0.0,0.0,13.5\n2026-04-30,22,15.2,0.0,0.0,11.4\n2026-04-30,23,14.9,0.0,0.0,12.5\n2026-05-01,0,14.7,0.0,0.0,11.9\n2026-05-01,1,14.4,0.0,0.0,11.6\n2026-05-01,2,13.9,0.0,0.0,10.2\n2026-05-01,3,13.0,0.0,0.0,11.0\n2026-05-01,4,12.6,0.0,0.0,10.7\n2026-05-01,5,12.4,0.0,0.0,11.2\n2026-05-01,6,11.8,0.0,0.0,11.3\n2026-05-01,7,12.6,0.0,0.0,9.8\n2026-05-01,8,13.1,0.0,0.0,12.1\n2026-05-01,9,15.6,0.0,0.0,15.1\n2026-05-01,10,17.1,0.0,0.0,15.3\n2026-05-01,11,18.5,0.0,0.0,18.4\n2026-05-01,12,19.3,0.0,0.0,21.8\n2026-05-01,13,19.8,0.0,0.0,22.0\n2026-05-01,14,20.3,0.0,0.0,21.6\n2026-05-01,15,20.2,0.0,0.0,22.3\n2026-05-01,16,20.1,0.0,0.0,20.7\n2026-05-01,17,18.9,0.0,0.0,20.0\n2026-05-01,18,16.7,0.0,0.0,19.5\n2026-05-01,19,15.1,0.0,0.0,15.9\n2026-05-01,20,14.3,0.0,0.0,16.8\n2026-05-01,21,14.1,0.0,0.0,12.7\n2026-05-01,22,13.8,0.0,0.0,11.5\n2026-05-01,23,13.4,0.0,0.0,9.3\n2026-05-02,0,12.7,0.0,0.0,11.9\n2026-05-02,1,12.2,0.0,0.0,10.2\n2026-05-02,2,12.0,0.0,0.0,9.4\n2026-05-02,3,11.7,0.0,0.0,8.3\n2026-05-02,4,11.8,0.0,0.0,7.4\n2026-05-02,5,11.9,0.0,0.0,6.9\n2026-05-02,6,12.0,0.0,0.0,7.4\n2026-05-02,7,13.0,0.0,0.0,8.0\n2026-05-02,8,14.7,0.0,0.0,8.2\n2026-05-02,9,16.3,0.0,0.0,8.3\n2026-05-02,10,17.2,0.0,0.0,8.7\n2026-05-02,11,18.6,0.0,0.0,10.6\n2026-05-02,12,19.5,0.0,0.0,11.5\n2026-05-02,13,20.5,0.0,0.0,10.5\n2026-05-02,14,20.1,0.0,0.0,14.6\n2026-05-02,15,19.1,0.0,0.0,12.9\n2026-05-02,16,18.7,0.0,0.0,11.7\n2026-05-02,17,18.1,0.0,0.0,10.6\n2026-05-02,18,16.4,0.0,0.0,10.1\n2026-05-02,19,15.7,0.0,0.0,6.9\n2026-05-02,20,15.1,0.1,0.0,8.1\n2026-05-02,21,15.2,0.1,0.0,3.9\n2026-05-02,22,14.9,0.4,0.0,2.4\n2026-05-02,23,14.3,0.4,0.0,0.5\n2026-05-03,0,14.2,0.9,0.0,2.4\n2026-05-03,1,14.2,0.5,0.0,2.4\n2026-05-03,2,14.0,0.6,0.0,2.7\n2026-05-03,3,13.2,0.4,0.0,5.8\n2026-05-03,4,12.9,0.6,0.0,9.4\n2026-05-03,5,12.3,0.8,0.0,8.9\n2026-05-03,6,11.9,0.8,0.0,7.4\n2026-05-03,7,11.9,1.1,0.0,8.2\n2026-05-03,8,11.9,1.0,0.0,8.9\n2026-05-03,9,12.4,0.7,0.0,8.8\n2026-05-03,10,13.1,0.3,0.0,10.0\n2026-05-03,11,13.3,0.1,0.0,9.2\n2026-05-03,12,13.5,0.0,0.0,11.8\n2026-05-03,13,13.4,0.0,0.0,11.1\n2026-05-03,14,13.7,0.0,0.0,9.6\n2026-05-03,15,13.7,0.0,0.0,10.7\n2026-05-03,16,14.1,0.1,0.0,11.6\n2026-05-03,17,12.6,0.5,0.0,22.0\n2026-05-03,18,11.8,0.2,0.0,17.9\n2026-05-03,19,11.7,0.0,0.0,13.9\n2026-05-03,20,11.5,0.0,0.0,12.7\n2026-05-03,21,11.7,0.6,0.0,13.3\n2026-05-03,22,11.1,1.4,0.0,18.6\n2026-05-03,23,10.9,0.3,0.0,17.3\n2026-05-04,0,11.0,0.2,0.0,16.7\n2026-05-04,1,10.2,0.0,0.0,23.9\n2026-05-04,2,9.2,0.0,0.0,24.4\n2026-05-04,3,9.4,0.0,0.0,22.7\n2026-05-04,4,9.5,0.0,0.0,19.3\n2026-05-04,5,9.4,0.0,0.0,14.3\n2026-05-04,6,9.5,0.0,0.0,12.3\n2026-05-04,7,10.9,0.0,0.0,11.8\n2026-05-04,8,12.7,0.0,0.0,13.0\n2026-05-04,9,13.5,0.0,0.0,12.2\n2026-05-04,10,15.2,0.0,0.0,13.0\n2026-05-04,11,16.7,0.0,0.0,14.8\n2026-05-04,12,17.5,0.0,0.0,15.9\n2026-05-04,13,17.5,0.0,0.0,20.0\n2026-05-04,14,17.6,0.0,0.0,21.4\n2026-05-04,15,17.4,0.0,0.0,20.8\n2026-05-04,16,16.5,0.0,0.0,19.5\n2026-05-04,17,16.1,0.0,0.0,19.0\n2026-05-04,18,14.6,0.0,0.0,16.7\n2026-05-04,19,13.3,0.0,0.0,16.1\n2026-05-04,20,12.1,0.0,0.0,12.1\n2026-05-04,21,11.9,0.0,0.0,11.4\n2026-05-04,22,11.4,0.0,0.0,9.0\n2026-05-04,23,10.9,0.0,0.0,8.0\n2026-05-05,0,10.4,0.0,0.0,7.8\n2026-05-05,1,9.9,0.0,0.0,7.7\n2026-05-05,2,9.6,0.0,0.0,7.6\n2026-05-05,3,8.7,0.0,0.0,7.8\n2026-05-05,4,8.4,0.0,0.0,5.6\n2026-05-05,5,8.0,0.0,0.0,6.7\n2026-05-05,6,7.6,0.0,0.0,8.8\n2026-05-05,7,8.7,0.0,0.0,9.7\n2026-05-05,8,11.1,0.0,0.0,8.0\n2026-05-05,9,13.1,0.0,0.0,6.5\n2026-05-05,10,15.5,0.0,0.0,7.1\n2026-05-05,11,17.2,0.0,0.0,9.3\n2026-05-05,12,18.3,0.0,0.0,12.0\n2026-05-05,13,18.9,0.0,0.0,12.9\n2026-05-05,14,19.2,0.0,0.0,13.2\n2026-05-05,15,18.8,0.0,0.0,13.9\n2026-05-05,16,18.2,0.0,0.0,13.3\n2026-05-05,17,17.2,0.0,0.0,12.6\n2026-05-05,18,15.8,0.0,0.0,11.3\n2026-05-05,19,14.2,0.0,0.0,9.6\n2026-05-05,20,12.9,0.0,0.0,5.8\n2026-05-05,21,12.3,0.0,0.0,5.5\n2026-05-05,22,12.5,0.0,0.0,7.3\n2026-05-05,23,11.7,0.0,0.0,8.7\n2026-05-06,0,11.0,0.0,0.0,10.9\n2026-05-06,1,10.5,0.0,0.0,11.8\n2026-05-06,2,10.2,0.0,0.0,11.0\n2026-05-06,3,9.4,0.0,0.0,10.7\n2026-05-06,4,9.3,0.0,0.0,10.7\n2026-05-06,5,9.2,0.0,0.0,11.1\n2026-05-06,6,9.4,0.0,0.0,11.1\n2026-05-06,7,10.8,0.0,0.0,12.0\n2026-05-06,8,13.3,0.0,0.0,11.1\n2026-05-06,9,15.1,0.0,0.0,9.7\n2026-05-06,10,17.5,0.0,0.0,8.9\n2026-05-06,11,19.5,0.0,0.0,10.1\n2026-05-06,12,21.0,0.0,0.0,12.2\n2026-05-06,13,21.8,0.0,0.0,14.8\n2026-05-06,14,21.7,0.0,0.0,15.5\n2026-05-06,15,20.9,0.0,0.0,15.7\n2026-05-06,16,20.4,0.0,0.0,15.7\n2026-05-06,17,19.4,0.0,0.0,15.1\n2026-05-06,18,18.0,0.0,0.0,12.5\n2026-05-06,19,16.7,0.0,0.0,7.8\n2026-05-06,20,15.6,0.0,0.0,7.6\n2026-05-06,21,14.9,0.0,0.0,6.8\n2026-05-06,22,15.1,0.0,0.0,6.2\n2026-05-06,23,14.5,0.0,0.0,5.7\n2026-05-07,0,14.4,0.0,0.0,7.1\n2026-05-07,1,13.4,0.0,0.0,9.2\n2026-05-07,2,12.9,0.0,0.0,11.8\n2026-05-07,3,12.4,0.0,0.0,9.6\n2026-05-07,4,12.2,0.0,0.0,7.7\n2026-05-07,5,11.9,0.0,0.0,7.9\n2026-05-07,6,11.8,0.0,0.0,9.5\n2026-05-07,7,13.4,0.0,0.0,9.2\n2026-05-07,8,15.4,0.0,0.0,7.4\n2026-05-07,9,16.1,0.0,0.0,9.5\n2026-05-07,10,18.7,0.0,0.0,19.3\n2026-05-07,11,18.2,0.0,0.0,21.3\n2026-05-07,12,18.9,0.0,0.0,24.4\n2026-05-07,13,18.1,0.1,0.0,24.3\n2026-05-07,14,16.5,0.2,0.0,22.1\n2026-05-07,15,14.4,1.1,0.0,17.4\n2026-05-07,16,13.6,1.6,0.0,15.8\n2026-05-07,17,13.3,0.7,0.0,12.7\n2026-05-07,18,13.1,0.2,0.0,11.6\n2026-05-07,19,12.9,0.0,0.0,8.4\n2026-05-07,20,12.0,0.0,0.0,8.6\n2026-05-07,21,12.7,0.0,0.0,11.0\n2026-05-07,22,12.5,0.0,0.0,12.8\n2026-05-07,23,12.8,0.0,0.0,12.6\n2026-05-08,0,12.4,0.0,0.0,12.6\n2026-05-08,1,11.9,0.0,0.0,13.6\n2026-05-08,2,11.6,0.0,0.0,14.9\n2026-05-08,3,12.1,0.0,0.0,17.6\n2026-05-08,4,11.7,0.0,0.0,16.7\n2026-05-08,5,11.2,0.0,0.0,14.9\n2026-05-08,6,11.0,0.0,0.0,15.5\n2026-05-08,7,12.3,0.0,0.0,15.1\n2026-05-08,8,14.1,0.0,0.0,14.1\n2026-05-08,9,15.3,0.0,0.0,12.3\n2026-05-08,10,16.6,0.0,0.0,14.6\n2026-05-08,11,17.4,0.1,0.0,19.1\n2026-05-08,12,17.4,0.3,0.0,24.3\n2026-05-08,13,18.4,0.3,0.0,25.7\n2026-05-08,14,18.9,0.0,0.0,25.0\n2026-05-08,15,19.2,0.0,0.0,24.8\n2026-05-08,16,19.1,0.0,0.0,22.0\n2026-05-08,17,18.6,0.0,0.0,17.5\n2026-05-08,18,17.8,0.0,0.0,14.9\n2026-05-08,19,16.3,0.0,0.0,11.4\n2026-05-08,20,14.9,0.0,0.0,13.8\n2026-05-08,21,14.6,0.0,0.0,13.8\n2026-05-08,22,14.6,0.0,0.0,12.7\n2026-05-08,23,13.7,0.0,0.0,11.0\n2026-05-09,0,12.9,0.0,0.0,8.3\n2026-05-09,1,11.7,0.0,0.0,4.8\n2026-05-09,2,11.1,0.0,0.0,2.8\n2026-05-09,3,10.5,0.0,0.0,1.0\n2026-05-09,4,10.0,0.0,0.0,1.3\n2026-05-09,5,9.5,0.0,0.0,1.7\n2026-05-09,6,9.4,0.0,0.0,0.9\n2026-05-09,7,11.0,0.0,0.0,0.3\n2026-05-09,8,13.6,0.0,0.0,1.8\n2026-05-09,9,15.6,0.0,0.0,2.3\n2026-05-09,10,17.9,0.0,0.0,3.8\n2026-05-09,11,19.5,0.0,0.0,7.6\n2026-05-09,12,20.3,0.0,0.0,10.9\n2026-05-09,13,20.6,0.0,0.0,12.0\n2026-05-09,14,20.4,0.0,0.0,14.6\n2026-05-09,15,19.8,0.0,0.0,14.9\n2026-05-09,16,19.3,0.0,0.0,12.6\n2026-05-09,17,18.8,0.0,0.0,10.4\n2026-05-09,18,17.8,0.0,0.0,8.8\n2026-05-09,19,15.8,0.0,0.0,8.1\n2026-05-09,20,14.0,0.0,0.0,5.8\n2026-05-09,21,12.8,0.0,0.0,3.0\n2026-05-09,22,13.0,0.0,0.0,0.6\n2026-05-09,23,12.4,0.0,0.0,1.2\n2026-05-10,0,11.6,0.0,0.0,3.0\n2026-05-10,1,11.1,0.0,0.0,3.5\n2026-05-10,2,10.3,0.0,0.0,4.1\n2026-05-10,3,9.9,0.0,0.0,5.3\n2026-05-10,4,9.9,0.0,0.0,5.1\n2026-05-10,5,10.5,0.0,0.0,7.8\n2026-05-10,6,10.6,0.0,0.0,10.1\n2026-05-10,7,12.2,0.0,0.0,9.3\n2026-05-10,8,15.1,0.0,0.0,6.5\n2026-05-10,9,17.5,0.0,0.0,9.3\n2026-05-10,10,19.4,0.0,0.0,10.9\n2026-05-10,11,21.1,0.0,0.0,11.3\n2026-05-10,12,22.0,0.0,0.0,12.4\n2026-05-10,13,22.7,0.0,0.0,12.2\n2026-05-10,14,22.6,0.0,0.0,13.2\n2026-05-10,15,21.5,0.0,0.0,12.4\n2026-05-10,16,20.3,0.0,0.0,12.3\n2026-05-10,17,19.3,0.0,0.0,11.2\n2026-05-10,18,18.4,0.0,0.0,8.1\n2026-05-10,19,16.9,0.0,0.0,8.6\n2026-05-10,20,15.1,0.0,0.0,9.5\n2026-05-10,21,14.5,0.0,0.0,5.6\n2026-05-10,22,14.8,0.0,0.0,7.1\n2026-05-10,23,14.0,0.0,0.0,8.7\n2026-05-11,0,13.1,0.0,0.0,11.3\n2026-05-11,1,12.7,0.0,0.0,11.7\n2026-05-11,2,12.5,0.0,0.0,10.3\n2026-05-11,3,11.7,0.0,0.0,8.0\n2026-05-11,4,11.6,0.0,0.0,9.2\n2026-05-11,5,11.6,0.0,0.0,10.2\n2026-05-11,6,12.0,0.0,0.0,11.2\n2026-05-11,7,13.6,0.0,0.0,10.2\n2026-05-11,8,15.7,0.0,0.0,11.4\n2026-05-11,9,17.1,0.1,0.0,10.0\n2026-05-11,10,17.6,0.2,0.0,9.4\n2026-05-11,11,20.3,0.0,0.0,9.5\n2026-05-11,12,21.7,0.0,0.0,13.3\n2026-05-11,13,21.9,0.0,0.0,16.3\n2026-05-11,14,18.3,1.1,0.0,18.4\n2026-05-11,15,17.3,0.8,0.0,13.4\n2026-05-11,16,17.9,0.3,0.0,14.1\n2026-05-11,17,18.4,0.0,0.0,13.1\n2026-05-11,18,17.5,0.0,0.0,7.4\n2026-05-11,19,15.8,0.1,0.0,7.4\n2026-05-11,20,15.3,0.1,0.0,11.2\n2026-05-11,21,15.8,0.1,0.0,1.1\n2026-05-11,22,15.8,0.0,0.0,1.1\n2026-05-11,23,15.9,0.0,0.0,3.7\n2026-05-12,0,15.3,0.0,0.0,2.4\n2026-05-12,1,14.1,0.5,0.0,3.8\n2026-05-12,2,15.2,0.0,0.0,6.7\n2026-05-12,3,14.6,1.5,0.0,7.5\n2026-05-12,4,15.3,0.3,0.0,9.3\n2026-05-12,5,15.1,0.0,0.0,7.9\n2026-05-12,6,14.5,0.0,0.0,6.1\n2026-05-12,7,15.3,0.0,0.0,6.4\n2026-05-12,8,16.6,0.0,0.0,5.5\n2026-05-12,9,17.3,0.1,0.0,1.6\n2026-05-12,10,20.1,0.0,0.0,1.2\n2026-05-12,11,21.4,0.1,0.0,1.3\n2026-05-12,12,22.7,0.0,0.0,3.7\n2026-05-12,13,23.6,0.2,0.0,6.5\n2026-05-12,14,22.9,0.3,0.0,10.6\n2026-05-12,15,22.5,0.1,0.0,9.3\n2026-05-12,16,22.2,0.0,0.0,11.7\n2026-05-12,17,21.4,0.0,0.0,12.3\n2026-05-12,18,20.2,0.0,0.0,10.1\n2026-05-12,19,18.3,0.0,0.0,9.8\n2026-05-12,20,16.6,0.0,0.0,7.9\n2026-05-12,21,16.2,0.0,0.0,7.2\n2026-05-12,22,15.2,0.0,0.0,6.7\n2026-05-12,23,14.5,0.0,0.0,6.0\n2026-05-13,0,13.7,0.0,0.0,2.8\n2026-05-13,1,13.2,0.0,0.0,4.5\n2026-05-13,2,12.7,0.0,0.0,3.9\n2026-05-13,3,12.7,0.0,0.0,4.9\n2026-05-13,4,12.4,0.0,0.0,5.2\n2026-05-13,5,12.1,0.0,0.0,4.5\n2026-05-13,6,12.0,0.0,0.0,3.8\n2026-05-13,7,12.5,0.0,0.0,3.4\n2026-05-13,8,13.9,0.0,0.0,1.5\n2026-05-13,9,16.5,0.0,0.0,1.0\n2026-05-13,10,19.2,0.0,0.0,2.6\n2026-05-13,11,21.0,0.0,0.0,6.5\n2026-05-13,12,21.7,0.0,0.0,8.1\n2026-05-13,13,21.3,0.0,0.0,12.3\n2026-05-13,14,20.0,0.0,0.0,15.5\n2026-05-13,15,20.7,0.0,0.0,13.0\n2026-05-13,16,21.1,0.0,0.0,10.1\n2026-05-13,17,19.7,0.0,0.0,8.9\n2026-05-13,18,19.6,0.0,0.0,7.5\n2026-05-13,19,18.1,0.0,0.0,6.7\n2026-05-13,20,16.1,0.0,0.0,6.3\n2026-05-13,21,15.4,0.0,0.0,7.7\n2026-05-13,22,14.5,0.0,0.0,7.5\n2026-05-13,23,14.2,0.0,0.0,5.7\n2026-05-14,0,14.1,0.0,0.0,4.9\n2026-05-14,1,13.7,0.0,0.0,3.6\n2026-05-14,2,13.5,0.0,0.0,2.9\n2026-05-14,3,13.5,0.0,0.0,3.4\n2026-05-14,4,13.1,0.0,0.0,4.4\n2026-05-14,5,13.0,0.0,0.0,5.6\n2026-05-14,6,13.1,0.0,0.0,7.6\n2026-05-14,7,15.4,0.0,0.0,5.9\n2026-05-14,8,19.7,0.0,0.0,4.2\n2026-05-14,9,22.7,0.0,0.0,7.2\n2026-05-14,10,24.8,0.0,0.0,7.4\n2026-05-14,11,27.2,0.0,0.0,8.1\n2026-05-14,12,28.7,0.0,0.0,5.1\n2026-05-14,13,29.2,0.0,0.0,5.6\n2026-05-14,14,29.4,0.0,0.0,7.6\n2026-05-14,15,29.2,0.2,0.0,9.9\n2026-05-14,16,28.8,0.1,0.0,13.0\n2026-05-14,17,28.3,0.0,0.0,13.5\n2026-05-14,18,26.9,0.0,0.0,10.8\n2026-05-14,19,25.0,0.0,0.0,8.0\n2026-05-14,20,23.3,0.0,0.0,4.4\n2026-05-14,21,23.3,0.0,0.0,6.3\n2026-05-14,22,22.2,0.0,0.0,1.7\n2026-05-14,23,20.7,0.0,0.0,10.0\n2026-05-15,0,19.8,0.0,0.0,10.0\n2026-05-15,1,19.2,0.0,0.0,12.0\n2026-05-15,2,19.2,0.0,0.0,11.8\n2026-05-15,3,18.1,0.0,0.0,8.3\n2026-05-15,4,17.2,0.0,0.0,6.4\n2026-05-15,5,17.3,0.0,0.0,8.2\n2026-05-15,6,17.7,0.0,0.0,9.6\n2026-05-15,7,19.3,0.0,0.0,8.0\n2026-05-15,8,21.9,0.0,0.0,8.0\n2026-05-15,9,23.8,0.0,0.0,3.9\n2026-05-15,10,25.8,0.0,0.0,1.0\n2026-05-15,11,27.7,0.0,0.0,1.5\n2026-05-15,12,28.7,0.0,0.0,7.2\n2026-05-15,13,28.6,0.0,0.0,12.8\n2026-05-15,14,28.2,0.0,0.0,16.0\n2026-05-15,15,28.2,0.0,0.0,15.2\n2026-05-15,16,27.5,0.0,0.0,14.3\n2026-05-15,17,27.0,0.0,0.0,13.2\n2026-05-15,18,25.8,0.0,0.0,12.3\n2026-05-15,19,24.0,0.0,0.0,9.8\n2026-05-15,20,22.1,0.0,0.0,8.3\n2026-05-15,21,20.9,0.0,0.0,8.5\n2026-05-15,22,20.1,0.0,0.0,9.3\n2026-05-15,23,20.2,0.0,0.0,7.7\n2026-05-16,0,20.3,0.0,0.0,8.1\n2026-05-16,1,19.9,0.0,0.0,7.4\n2026-05-16,2,19.0,0.0,0.0,3.0\n2026-05-16,3,17.8,0.0,0.0,2.6\n2026-05-16,4,16.7,0.0,0.0,1.6\n2026-05-16,5,16.0,0.0,0.0,1.9\n2026-05-16,6,15.9,0.0,0.0,3.9\n2026-05-16,7,18.0,0.0,0.0,2.9\n2026-05-16,8,20.8,0.0,0.0,3.8\n2026-05-16,9,23.2,0.0,0.0,4.6\n2026-05-16,10,25.0,0.0,0.0,6.5\n2026-05-16,11,26.2,0.0,0.0,8.7\n2026-05-16,12,27.0,0.0,0.0,11.6\n2026-05-16,13,27.2,0.0,0.0,13.4\n2026-05-16,14,27.0,0.0,0.0,14.8\n2026-05-16,15,27.2,0.0,0.0,14.7\n2026-05-16,16,26.4,0.0,0.0,15.6\n2026-05-16,17,25.5,0.0,0.0,14.8\n2026-05-16,18,24.2,0.0,0.0,12.6\n2026-05-16,19,22.6,0.0,0.0,9.3\n2026-05-16,20,20.9,0.0,0.0,7.0\n2026-05-16,21,20.4,0.0,0.0,7.6\n2026-05-16,22,19.5,0.0,0.0,7.2\n2026-05-16,23,19.0,0.0,0.0,6.1\n2026-05-17,0,18.5,0.0,0.0,4.9\n2026-05-17,1,18.0,0.0,0.0,3.8\n2026-05-17,2,17.4,0.0,0.0,3.8\n2026-05-17,3,16.4,0.0,0.0,2.4\n2026-05-17,4,15.7,0.0,0.0,1.8\n2026-05-17,5,14.8,0.0,0.0,3.1\n2026-05-17,6,14.6,0.0,0.0,3.9\n2026-05-17,7,16.4,0.0,0.0,3.3\n2026-05-17,8,19.0,0.0,0.0,4.2\n2026-05-17,9,21.6,0.0,0.0,4.9\n2026-05-17,10,23.5,0.0,0.0,7.3\n2026-05-17,11,25.1,0.0,0.0,7.8\n2026-05-17,12,26.3,0.0,0.0,8.7\n2026-05-17,13,27.0,0.0,0.0,10.4\n2026-05-17,14,27.0,0.0,0.0,12.3\n2026-05-17,15,27.4,0.0,0.0,12.7\n2026-05-17,16,26.3,0.0,0.0,13.2\n2026-05-17,17,25.7,0.0,0.0,12.2\n2026-05-17,18,24.7,0.0,0.0,10.4\n2026-05-17,19,23.1,0.0,0.0,9.2\n2026-05-17,20,21.2,0.0,0.0,6.7\n2026-05-17,21,20.7,0.0,0.0,6.6\n2026-05-17,22,19.7,0.0,0.0,5.0\n2026-05-17,23,18.9,0.0,0.0,4.1\n2026-05-18,0,18.2,0.0,0.0,4.0\n2026-05-18,1,17.6,0.0,0.0,3.9\n2026-05-18,2,17.4,0.0,0.0,4.5\n2026-05-18,3,16.6,0.0,0.0,3.0\n2026-05-18,4,16.0,0.0,0.0,1.8\n2026-05-18,5,15.6,0.0,0.0,3.3\n2026-05-18,6,15.7,0.0,0.0,3.3\n2026-05-18,7,18.2,0.0,0.0,3.4\n2026-05-18,8,21.7,0.0,0.0,2.7\n2026-05-18,9,24.0,0.0,0.0,2.6\n2026-05-18,10,26.4,0.0,0.0,5.2\n2026-05-18,11,27.8,0.0,0.0,9.5\n2026-05-18,12,28.1,0.0,0.0,12.5\n2026-05-18,13,28.3,0.0,0.0,14.0\n2026-05-18,14,28.2,0.0,0.0,13.5\n2026-05-18,15,28.4,0.0,0.0,12.9\n2026-05-18,16,27.4,0.0,0.0,13.1\n2026-05-18,17,26.7,0.0,0.0,11.6\n2026-05-18,18,25.3,0.0,0.0,10.8\n2026-05-18,19,23.8,0.0,0.0,8.1\n2026-05-18,20,22.8,0.0,0.0,7.4\n2026-05-18,21,21.9,0.0,0.0,6.3\n2026-05-18,22,21.4,0.0,0.0,3.3\n2026-05-18,23,20.9,0.0,0.0,1.2\n2026-05-19,0,20.2,0.0,0.0,4.4\n2026-05-19,1,20.0,0.0,0.0,7.2\n2026-05-19,2,19.8,0.0,0.0,7.9\n2026-05-19,3,18.2,0.0,0.0,6.7\n2026-05-19,4,18.0,0.0,0.0,5.5\n2026-05-19,5,18.3,0.0,0.0,6.1\n2026-05-19,6,18.5,0.0,0.0,5.1\n2026-05-19,7,19.4,0.0,0.0,5.9\n2026-05-19,8,20.8,0.0,0.0,5.5\n2026-05-19,9,22.8,0.0,0.0,6.3\n2026-05-19,10,24.0,0.0,0.0,6.5\n2026-05-19,11,24.2,0.0,0.0,5.6\n2026-05-19,12,24.4,0.0,0.0,4.9\n2026-05-19,13,24.2,0.0,0.0,6.5\n2026-05-19,14,24.1,0.0,0.0,6.9\n2026-05-19,15,23.6,0.0,0.0,5.6\n2026-05-19,16,23.7,0.0,0.0,6.9\n2026-05-19,17,23.9,0.0,0.0,9.6\n2026-05-19,18,23.0,0.0,0.0,12.6\n2026-05-19,19,22.6,0.0,0.0,14.7\n2026-05-19,20,21.4,0.3,0.0,11.3\n2026-05-19,21,21.7,0.0,0.0,12.5\n2026-05-19,22,21.7,0.0,0.0,10.2\n2026-05-19,23,21.9,0.0,0.0,10.7\n2026-05-20,0,22.1,0.0,0.0,11.8\n2026-05-20,1,22.1,0.0,0.0,12.5\n2026-05-20,2,22.0,0.0,0.0,13.4\n2026-05-20,3,21.3,0.0,0.0,11.4\n2026-05-20,4,21.5,0.0,0.0,12.2\n2026-05-20,5,21.3,0.0,0.0,18.6\n2026-05-20,6,18.7,0.2,0.0,16.2\n2026-05-20,7,17.8,0.3,0.0,14.4\n2026-05-20,8,18.1,0.1,0.0,16.8\n2026-05-20,9,16.5,1.0,0.0,17.6\n2026-05-20,10,16.8,3.2,0.0,17.1\n2026-05-20,11,16.9,2.5,0.0,14.9\n2026-05-20,12,16.2,2.6,0.0,19.3\n2026-05-20,13,16.1,3.2,0.0,20.2\n2026-05-20,14,16.1,3.1,0.0,21.1\n2026-05-20,15,15.8,2.8,0.0,18.9\n2026-05-20,16,15.8,4.6,0.0,18.9\n2026-05-20,17,15.9,5.7,0.0,17.0\n2026-05-20,18,16.1,5.1,0.0,15.0\n2026-05-20,19,16.0,5.7,0.0,19.5\n2026-05-20,20,15.7,2.2,0.0,23.3\n2026-05-20,21,16.0,0.9,0.0,22.5\n2026-05-20,22,15.8,4.4,0.0,23.6\n2026-05-20,23,15.6,1.2,0.0,20.9\n2026-05-21,0,15.5,0.8,0.0,19.6\n2026-05-21,1,15.5,0.5,0.0,18.2\n2026-05-21,2,15.5,0.6,0.0,17.3\n2026-05-21,3,15.4,0.4,0.0,17.7\n2026-05-21,4,15.6,0.3,0.0,16.6\n2026-05-21,5,15.6,1.1,0.0,16.4\n2026-05-21,6,15.7,0.9,0.0,17.9\n2026-05-21,7,16.2,0.0,0.0,17.2\n2026-05-21,8,17.0,0.1,0.0,14.8\n2026-05-21,9,16.7,0.1,0.0,13.9\n2026-05-21,10,17.5,0.2,0.0,13.3\n2026-05-21,11,18.0,0.4,0.0,14.2\n2026-05-21,12,18.6,0.5,0.0,14.8\n2026-05-21,13,18.4,0.5,0.0,11.4\n2026-05-21,14,18.7,0.5,0.0,11.9\n2026-05-21,15,18.9,0.5,0.0,11.9\n2026-05-21,16,19.6,0.0,0.0,12.1\n2026-05-21,17,19.8,0.0,0.0,11.0\n2026-05-21,18,20.1,0.0,0.0,12.6\n2026-05-21,19,19.4,0.0,0.0,15.1\n2026-05-21,20,18.3,0.0,0.0,15.5\n2026-05-21,21,18.6,0.0,0.0,14.2\n2026-05-21,22,18.2,0.0,0.0,12.1\n2026-05-21,23,18.0,0.0,0.0,11.5\n2026-05-22,0,17.5,0.0,0.0,11.6\n2026-05-22,1,17.3,0.0,0.0,12.3\n2026-05-22,2,16.9,0.0,0.0,12.5\n2026-05-22,3,16.6,0.0,0.0,15.2\n2026-05-22,4,15.7,0.0,0.0,11.0\n2026-05-22,5,15.2,0.0,0.0,11.3\n2026-05-22,6,15.3,0.0,0.0,11.8\n2026-05-22,7,16.3,0.0,0.0,9.1\n2026-05-22,8,17.4,0.0,0.0,8.8\n2026-05-22,9,18.9,0.0,0.0,8.7\n2026-05-22,10,20.5,0.0,0.0,8.1\n2026-05-22,11,22.0,0.0,0.0,8.3\n2026-05-22,12,23.1,0.0,0.0,11.3\n2026-05-22,13,23.8,0.0,0.0,12.1\n2026-05-22,14,24.3,0.0,0.0,13.0\n2026-05-22,15,25.4,0.0,0.0,12.0\n2026-05-22,16,25.5,0.0,0.0,10.9\n2026-05-22,17,25.0,0.0,0.0,9.2\n2026-05-22,18,24.5,0.0,0.0,7.3\n2026-05-22,19,21.6,0.0,0.0,3.0\n2026-05-22,20,19.9,0.0,0.0,0.4\n2026-05-22,21,19.3,0.0,0.0,6.3\n2026-05-22,22,18.7,0.0,0.0,11.8\n2026-05-22,23,17.9,0.0,0.0,11.6\n2026-05-23,0,17.8,0.0,0.0,12.5\n2026-05-23,1,16.8,0.0,0.0,11.3\n2026-05-23,2,16.1,0.0,0.0,11.6\n2026-05-23,3,16.2,0.0,0.0,12.3\n2026-05-23,4,15.8,0.0,0.0,13.9\n2026-05-23,5,15.6,0.0,0.0,14.0\n2026-05-23,6,15.4,0.0,0.0,10.7\n2026-05-23,7,15.9,0.0,0.0,11.6\n2026-05-23,8,15.9,0.0,0.0,7.0\n2026-05-23,9,16.8,0.1,0.0,10.2\n2026-05-23,10,18.2,0.1,0.0,8.1\n2026-05-23,11,19.1,0.5,0.0,6.9\n2026-05-23,12,21.0,0.1,0.0,4.4\n2026-05-23,13,21.7,0.0,0.0,2.6\n2026-05-23,14,22.3,0.0,0.0,1.9\n2026-05-23,15,22.3,0.0,0.0,8.1\n2026-05-23,16,21.7,0.0,0.0,11.7\n2026-05-23,17,21.0,0.0,0.0,11.0\n2026-05-23,18,20.4,0.0,0.0,10.1\n2026-05-23,19,19.4,0.0,0.0,7.8\n2026-05-23,20,17.6,0.0,0.0,6.9\n2026-05-23,21,17.2,0.0,0.0,7.2\n2026-05-23,22,16.7,0.0,0.0,5.5\n2026-05-23,23,16.5,0.0,0.0,5.9\n2026-05-24,0,16.2,0.0,0.0,3.6\n2026-05-24,1,15.8,0.0,0.0,3.2\n2026-05-24,2,15.3,0.0,0.0,3.9\n2026-05-24,3,15.5,0.0,0.0,7.6\n2026-05-24,4,15.0,0.0,0.0,4.4\n2026-05-24,5,14.6,0.0,0.0,3.8\n2026-05-24,6,15.0,0.0,0.0,3.0\n2026-05-24,7,16.5,0.0,0.0,2.7\n2026-05-24,8,18.8,0.0,0.0,2.9\n2026-05-24,9,20.9,0.0,0.0,4.0\n2026-05-24,10,22.5,0.0,0.0,2.4\n2026-05-24,11,23.6,0.0,0.0,3.6\n2026-05-24,12,24.5,0.0,0.0,3.5\n2026-05-24,13,25.4,0.0,0.0,3.8\n2026-05-24,14,26.1,0.0,0.0,3.5\n2026-05-24,15,26.2,0.0,0.0,6.8\n2026-05-24,16,25.8,0.0,0.0,7.5\n2026-05-24,17,24.6,0.0,0.0,10.6\n2026-05-24,18,23.0,0.0,0.0,9.9\n2026-05-24,19,21.6,0.0,0.0,8.2\n2026-05-24,20,19.9,0.0,0.0,6.3\n2026-05-24,21,19.5,0.0,0.0,6.0\n2026-05-24,22,19.1,0.0,0.0,7.1\n2026-05-24,23,18.4,0.0,0.0,5.6\n2026-05-25,0,17.8,0.0,0.0,3.7\n2026-05-25,1,17.5,0.0,0.0,3.9\n2026-05-25,2,17.2,0.0,0.0,4.3\n2026-05-25,3,16.9,0.0,0.0,4.8\n2026-05-25,4,16.7,0.0,0.0,6.0\n2026-05-25,5,16.6,0.0,0.0,6.1\n2026-05-25,6,17.4,0.0,0.0,7.7\n2026-05-25,7,19.3,0.0,0.0,6.8\n2026-05-25,8,21.3,0.0,0.0,6.6\n2026-05-25,9,23.0,0.0,0.0,5.3\n2026-05-25,10,24.7,0.0,0.0,4.6\n2026-05-25,11,26.3,0.0,0.0,4.9\n2026-05-25,12,27.1,0.0,0.0,7.7\n2026-05-25,13,27.1,0.0,0.0,8.4\n2026-05-25,14,27.5,0.0,0.0,6.5\n2026-05-25,15,28.1,0.0,0.0,7.3\n2026-05-25,16,27.6,0.0,0.0,9.4\n2026-05-25,17,26.9,0.0,0.0,8.6\n2026-05-25,18,26.4,0.0,0.0,5.7\n2026-05-25,19,24.9,0.0,0.0,9.0\n2026-05-25,20,23.6,0.0,0.0,6.0\n2026-05-25,21,23.2,0.0,0.0,7.1\n2026-05-25,22,22.8,0.0,0.0,7.6\n2026-05-25,23,22.3,0.0,0.0,6.3\n2026-05-26,0,21.7,0.0,0.0,6.5\n2026-05-26,1,21.4,0.0,0.0,5.9\n2026-05-26,2,21.7,0.0,0.0,6.6\n2026-05-26,3,20.8,0.0,0.0,4.3\n2026-05-26,4,20.5,0.0,0.0,3.4\n2026-05-26,5,20.7,0.0,0.0,2.9\n2026-05-26,6,20.7,0.0,0.0,5.2\n2026-05-26,7,21.4,0.0,0.0,6.7\n2026-05-26,8,22.9,0.0,0.0,7.4\n2026-05-26,9,24.0,0.0,0.0,4.8\n2026-05-26,10,25.1,0.0,0.0,4.5\n2026-05-26,11,26.5,0.0,0.0,4.6\n2026-05-26,12,27.5,0.1,0.0,3.5\n2026-05-26,13,28.2,0.2,0.0,6.2\n2026-05-26,14,27.0,0.2,0.0,9.8\n2026-05-26,15,26.8,0.2,0.0,8.6\n2026-05-26,16,26.7,0.4,0.0,4.8\n2026-05-26,17,25.7,0.5,0.0,5.6\n2026-05-26,18,24.5,0.3,0.0,9.1\n2026-05-26,19,23.6,1.3,0.0,7.2\n2026-05-26,20,23.1,0.8,0.0,7.0\n2026-05-26,21,22.4,0.6,0.0,9.2\n2026-05-26,22,22.1,0.0,0.0,10.9\n2026-05-26,23,22.1,0.0,0.0,11.9\n2026-05-27,0,21.8,0.0,0.0,10.4\n2026-05-27,1,21.5,0.0,0.0,8.6\n2026-05-27,2,21.3,0.0,0.0,11.1\n2026-05-27,3,21.9,0.0,0.0,14.0\n2026-05-27,4,21.9,0.0,0.0,13.0\n2026-05-27,5,20.7,3.4,0.0,11.2\n2026-05-27,6,20.5,5.5,0.0,11.6\n2026-05-27,7,20.4,2.6,0.0,11.7\n2026-05-27,8,20.5,0.4,0.0,12.1\n2026-05-27,9,20.9,0.1,0.0,9.3\n2026-05-27,10,21.5,0.1,0.0,9.9\n2026-05-27,11,22.4,0.0,0.0,10.4\n2026-05-27,12,23.4,0.0,0.0,8.7\n2026-05-27,13,24.3,0.0,0.0,4.9\n2026-05-27,14,24.8,0.0,0.0,3.0\n2026-05-27,15,23.9,1.2,0.0,0.6\n2026-05-27,16,22.9,1.2,0.0,4.7\n2026-05-27,17,22.5,0.9,0.0,5.1\n2026-05-27,18,22.3,0.4,0.0,4.7\n2026-05-27,19,22.2,0.0,0.0,6.6\n2026-05-27,20,21.6,0.0,0.0,8.9\n2026-05-27,21,20.5,0.0,0.0,6.8\n2026-05-27,22,20.1,0.3,0.0,7.8\n2026-05-27,23,20.0,0.1,0.0,6.2\n2026-05-28,0,20.1,0.0,0.0,3.2\n2026-05-28,1,20.2,0.2,0.0,1.4\n2026-05-28,2,20.0,0.4,0.0,6.4\n2026-05-28,3,19.2,0.0,0.0,6.3\n2026-05-28,4,19.1,0.0,0.0,6.4\n2026-05-28,5,19.1,0.0,0.0,7.4\n2026-05-28,6,19.0,0.0,0.0,5.2\n2026-05-28,7,19.2,0.0,0.0,4.8\n2026-05-28,8,19.7,0.0,0.0,7.4\n2026-05-28,9,19.5,0.0,0.0,9.6\n2026-05-28,10,20.2,0.0,0.0,9.4\n2026-05-28,11,20.6,0.0,0.0,9.3\n2026-05-28,12,21.2,0.1,0.0,10.1\n2026-05-28,13,22.3,0.1,0.0,10.4\n2026-05-28,14,22.9,0.1,0.0,11.4\n2026-05-28,15,22.6,0.0,0.0,12.9\n2026-05-28,16,21.5,0.0,0.0,13.1\n2026-05-28,17,20.9,0.0,0.0,12.6\n2026-05-28,18,19.3,0.0,0.0,13.0\n2026-05-28,19,17.8,0.0,0.0,14.1\n2026-05-28,20,17.3,0.4,0.0,10.4\n2026-05-28,21,17.4,0.0,0.0,7.9\n2026-05-28,22,17.2,0.0,0.0,5.0\n2026-05-28,23,17.1,0.0,0.0,5.6\n2026-05-29,0,16.7,0.0,0.0,5.9\n2026-05-29,1,16.7,0.0,0.0,7.1\n2026-05-29,2,16.7,0.0,0.0,6.7\n2026-05-29,3,16.3,0.0,0.0,2.1\n2026-05-29,4,16.0,0.0,0.0,2.4\n2026-05-29,5,15.8,0.0,0.0,3.2\n2026-05-29,6,16.2,0.0,0.0,6.3\n2026-05-29,7,16.4,0.0,0.0,6.8\n2026-05-29,8,18.1,0.0,0.0,6.6\n2026-05-29,9,20.0,0.0,0.0,3.4\n2026-05-29,10,21.6,0.0,0.0,3.7\n2026-05-29,11,22.8,0.0,0.0,7.6\n2026-05-29,12,23.3,0.0,0.0,11.0\n2026-05-29,13,23.4,0.1,0.0,13.0\n2026-05-29,14,23.6,0.0,0.0,14.7\n2026-05-29,15,23.9,0.0,0.0,13.3\n2026-05-29,16,23.2,0.0,0.0,13.8\n2026-05-29,17,22.0,0.0,0.0,13.9\n2026-05-29,18,20.8,0.0,0.0,12.9\n2026-05-29,19,19.3,0.0,0.0,11.2\n2026-05-29,20,18.2,0.0,0.0,9.0\n2026-05-29,21,18.0,0.0,0.0,7.9\n2026-05-29,22,17.8,0.0,0.0,7.2\n2026-05-29,23,17.4,0.0,0.0,6.7\n2026-05-30,0,17.0,0.0,0.0,5.0\n2026-05-30,1,16.5,0.0,0.0,5.1\n2026-05-30,2,16.1,0.0,0.0,5.1\n2026-05-30,3,15.5,0.0,0.0,5.6\n2026-05-30,4,15.0,0.0,0.0,6.4\n2026-05-30,5,14.7,0.0,0.0,6.7\n2026-05-30,6,15.0,0.0,0.0,8.4\n2026-05-30,7,16.7,0.0,0.0,7.9\n2026-05-30,8,18.8,0.0,0.0,8.5\n2026-05-30,9,20.6,0.0,0.0,9.8\n2026-05-30,10,21.9,0.0,0.0,11.6\n2026-05-30,11,22.8,0.0,0.0,12.1\n2026-05-30,12,23.4,0.0,0.0,11.9\n2026-05-30,13,23.7,0.0,0.0,12.5\n2026-05-30,14,23.9,0.0,0.0,12.6\n2026-05-30,15,24.7,0.0,0.0,12.4\n2026-05-30,16,23.9,0.0,0.0,12.6\n2026-05-30,17,23.0,0.0,0.0,11.6\n2026-05-30,18,21.9,0.0,0.0,8.4\n2026-05-30,19,20.5,0.0,0.0,5.8\n2026-05-30,20,18.6,0.0,0.0,6.5\n2026-05-30,21,18.4,0.0,0.0,6.7\n2026-05-30,22,17.8,0.0,0.0,7.1\n2026-05-30,23,17.3,0.0,0.0,6.6\n2026-05-31,0,16.9,0.0,0.0,5.9\n2026-05-31,1,16.4,0.0,0.0,3.7\n2026-05-31,2,16.4,0.0,0.0,7.3\n2026-05-31,3,16.3,0.0,0.0,7.1\n2026-05-31,4,16.0,0.0,0.0,7.4\n2026-05-31,5,15.8,0.0,0.0,7.4\n2026-05-31,6,16.2,0.0,0.0,8.5\n2026-05-31,7,18.2,0.0,0.0,6.6\n2026-05-31,8,20.7,0.0,0.0,4.9\n2026-05-31,9,23.0,0.0,0.0,5.0\n2026-05-31,10,25.1,0.0,0.0,4.2\n2026-05-31,11,26.6,0.0,0.0,5.9\n2026-05-31,12,27.3,0.0,0.0,9.0\n2026-05-31,13,27.2,0.0,0.0,9.7\n2026-05-31,14,27.0,0.0,0.0,9.3\n2026-05-31,15,27.3,0.0,0.0,9.7\n2026-05-31,16,26.8,0.0,0.0,9.4\n2026-05-31,17,26.0,0.0,0.0,7.4\n2026-05-31,18,25.1,0.0,0.0,5.4\n2026-05-31,19,23.0,0.0,0.0,3.3\n2026-05-31,20,20.6,0.0,0.0,3.0\n2026-05-31,21,20.1,0.0,0.0,4.2\n2026-05-31,22,19.6,0.0,0.0,1.5\n2026-05-31,23,19.2,0.0,0.0,2.2\n2026-06-01,0,18.6,0.0,0.0,1.1\n2026-06-01,1,17.7,0.0,0.0,2.1\n2026-06-01,2,16.9,0.0,0.0,1.1\n2026-06-01,3,15.8,0.0,0.0,1.1\n2026-06-01,4,15.4,0.0,0.0,1.7\n2026-06-01,5,14.7,0.0,0.0,2.0\n2026-06-01,6,13.9,0.0,0.0,4.2\n2026-06-01,7,16.4,0.0,0.0,3.9\n2026-06-01,8,19.8,0.0,0.0,3.6\n2026-06-01,9,21.2,0.0,0.0,3.1\n2026-06-01,10,24.6,0.0,0.0,5.8\n2026-06-01,11,25.7,0.0,0.0,9.8\n2026-06-01,12,26.1,0.0,0.0,12.1\n2026-06-01,13,26.0,0.0,0.0,9.9\n2026-06-01,14,26.5,0.0,0.0,9.2\n2026-06-01,15,25.4,0.0,0.0,9.3\n2026-06-01,16,24.7,0.0,0.0,9.0\n2026-06-01,17,23.6,0.0,0.0,6.5\n2026-06-01,18,23.1,0.0,0.0,3.0\n2026-06-01,19,22.1,0.0,0.0,4.5\n2026-06-01,20,20.9,0.0,0.0,2.7\n2026-06-01,21,19.3,0.0,0.0,3.8\n2026-06-01,22,18.7,0.0,0.0,0.6\n2026-06-01,23,18.1,0.0,0.0,0.4\n2026-06-02,0,18.0,0.0,0.0,0.5\n2026-06-02,1,17.7,0.0,0.0,2.3\n2026-06-02,2,17.5,0.0,0.0,2.8\n2026-06-02,3,17.1,0.0,0.0,5.6\n2026-06-02,4,17.0,0.0,0.0,7.0\n2026-06-02,5,16.9,0.0,0.0,7.5\n2026-06-02,6,17.5,0.0,0.0,8.9\n2026-06-02,7,19.3,0.0,0.0,7.4\n2026-06-02,8,22.2,0.0,0.0,4.0\n2026-06-02,9,24.8,0.0,0.0,5.7\n2026-06-02,10,26.1,0.0,0.0,6.4\n2026-06-02,11,27.5,0.0,0.0,5.9\n2026-06-02,12,28.5,0.0,0.0,4.0\n2026-06-02,13,29.4,0.0,0.0,3.4\n2026-06-02,14,30.0,0.0,0.0,7.7\n2026-06-02,15,30.5,0.0,0.0,4.4\n2026-06-02,16,30.5,0.0,0.0,7.4\n2026-06-02,17,29.6,0.0,0.0,11.0\n2026-06-02,18,28.3,0.0,0.0,14.0\n2026-06-02,19,26.7,0.0,0.0,12.0\n2026-06-02,20,24.9,0.0,0.0,8.5\n2026-06-02,21,23.6,0.0,0.0,6.2\n2026-06-02,22,23.2,0.0,0.0,6.7\n2026-06-02,23,22.7,0.0,0.0,8.4\n2026-06-03,0,21.8,0.0,0.0,7.2\n2026-06-03,1,20.9,0.0,0.0,4.1\n2026-06-03,2,20.4,0.0,0.0,5.4\n2026-06-03,3,19.6,0.0,0.0,2.7\n2026-06-03,4,19.5,0.0,0.0,6.0\n2026-06-03,5,19.2,0.0,0.0,5.9\n2026-06-03,6,19.3,0.0,0.0,6.4\n2026-06-03,7,20.7,0.0,0.0,4.9\n2026-06-03,8,22.8,0.0,0.0,4.3\n2026-06-03,9,24.7,0.0,0.0,3.4\n2026-06-03,10,26.4,0.0,0.0,4.0\n2026-06-03,11,27.6,0.0,0.0,3.7\n2026-06-03,12,28.3,0.0,0.0,6.1\n2026-06-03,13,28.9,0.0,0.0,9.8\n2026-06-03,14,28.4,0.0,0.0,12.6\n2026-06-03,15,26.7,0.0,0.0,11.7\n2026-06-03,16,26.9,0.0,0.0,10.7\n2026-06-03,17,26.4,0.0,0.0,10.8\n2026-06-03,18,25.3,0.0,0.0,9.0\n2026-06-03,19,24.2,0.0,0.0,8.2\n2026-06-03,20,23.0,0.0,0.0,5.9\n2026-06-03,21,23.0,0.0,0.0,8.3\n2026-06-03,22,22.6,0.0,0.0,7.6\n2026-06-03,23,22.0,0.0,0.0,5.3\n2026-06-04,0,21.5,0.0,0.0,2.9\n2026-06-04,1,21.2,0.0,0.0,1.7\n2026-06-04,2,20.6,0.0,0.0,5.6\n2026-06-04,3,20.0,0.0,0.0,3.9\n2026-06-04,4,19.6,0.0,0.0,2.7\n2026-06-04,5,19.8,0.1,0.0,2.3\n2026-06-04,6,19.5,0.5,0.0,7.6\n2026-06-04,7,20.3,0.6,0.0,4.2\n2026-06-04,8,21.5,0.4,0.0,4.5\n2026-06-04,9,24.8,0.1,0.0,4.2\n2026-06-04,10,24.3,0.6,0.0,7.2\n2026-06-04,11,25.8,0.3,0.0,2.4\n2026-06-04,12,27.1,0.0,0.0,5.9\n2026-06-04,13,25.8,0.3,0.0,12.6\n2026-06-04,14,22.7,3.7,0.0,12.9\n2026-06-04,15,24.9,0.3,0.0,21.5\n2026-06-04,16,23.3,1.4,0.0,21.2\n2026-06-04,17,22.2,0.0,0.0,22.1\n2026-06-04,18,20.8,0.5,0.0,21.9\n2026-06-04,19,20.0,0.4,0.0,14.7\n2026-06-04,20,20.2,0.1,0.0,13.2\n2026-06-04,21,19.7,0.1,0.0,10.4\n2026-06-04,22,19.7,0.1,0.0,13.9\n2026-06-04,23,19.2,0.0,0.0,13.1\n2026-06-05,0,18.7,0.0,0.0,10.6\n2026-06-05,1,18.8,0.0,0.0,12.7\n2026-06-05,2,18.2,0.0,0.0,13.8\n2026-06-05,3,17.9,0.0,0.0,16.4\n2026-06-05,4,17.2,0.0,0.0,12.8\n2026-06-05,5,16.4,0.0,0.0,11.9\n2026-06-05,6,16.6,0.0,0.0,11.9\n2026-06-05,7,17.6,0.0,0.0,13.6\n2026-06-05,8,18.6,0.0,0.0,14.3\n2026-06-05,9,19.8,0.0,0.0,14.3\n2026-06-05,10,20.4,0.0,0.0,15.8\n2026-06-05,11,21.4,0.0,0.0,15.9\n2026-06-05,12,22.6,0.0,0.0,16.4\n2026-06-05,13,23.6,0.1,0.0,15.7\n2026-06-05,14,24.4,0.0,0.0,18.3\n2026-06-05,15,23.2,0.0,0.0,19.6\n2026-06-05,16,22.6,0.0,0.0,19.6\n2026-06-05,17,22.5,0.0,0.0,17.7\n2026-06-05,18,21.9,0.0,0.0,14.8\n2026-06-05,19,21.1,0.0,0.0,12.0\n2026-06-05,20,19.7,0.0,0.0,10.4\n2026-06-05,21,18.8,0.0,0.0,9.9\n2026-06-05,22,18.1,0.0,0.0,10.1\n2026-06-05,23,17.7,0.0,0.0,8.9\n2026-06-06,0,17.1,0.0,0.0,6.1\n2026-06-06,1,16.5,0.0,0.0,2.7\n2026-06-06,2,16.1,0.0,0.0,2.8\n2026-06-06,3,15.4,0.0,0.0,5.5\n2026-06-06,4,14.9,0.0,0.0,3.9\n2026-06-06,5,14.6,0.0,0.0,2.4\n2026-06-06,6,15.2,0.0,0.0,4.0\n2026-06-06,7,15.8,0.0,0.0,3.8\n2026-06-06,8,17.9,0.0,0.0,4.4\n2026-06-06,9,20.3,0.0,0.0,5.5\n2026-06-06,10,23.2,0.0,0.0,5.0\n2026-06-06,11,25.4,0.0,0.0,6.1\n2026-06-06,12,26.9,0.0,0.0,9.0\n2026-06-06,13,28.1,0.0,0.0,7.4\n2026-06-06,14,28.7,0.0,0.0,2.4\n2026-06-06,15,27.8,0.0,0.0,8.8\n2026-06-06,16,25.9,0.0,0.0,14.2\n2026-06-06,17,26.2,0.0,0.0,14.6\n2026-06-06,18,26.2,0.0,0.0,13.9\n2026-06-06,19,25.3,0.0,0.0,17.5\n2026-06-06,20,24.3,0.0,0.0,17.9\n2026-06-06,21,23.7,0.0,0.0,14.2\n2026-06-06,22,23.5,0.0,0.0,10.2\n2026-06-06,23,22.9,0.0,0.0,10.5\n2026-06-07,0,22.2,0.0,0.0,12.6\n2026-06-07,1,22.3,0.0,0.0,15.5\n2026-06-07,2,21.9,0.0,0.0,13.6\n2026-06-07,3,21.8,0.0,0.0,11.6\n2026-06-07,4,20.7,0.0,0.0,11.9\n2026-06-07,5,20.1,0.0,0.0,11.5\n2026-06-07,6,20.1,0.0,0.0,11.4\n2026-06-07,7,20.9,0.0,0.0,9.9\n2026-06-07,8,22.4,0.0,0.0,8.2\n2026-06-07,9,22.8,0.0,0.0,7.5\n2026-06-07,10,23.0,0.0,0.0,8.8\n2026-06-07,11,23.8,0.0,0.0,10.2\n2026-06-07,12,25.1,0.0,0.0,10.5\n2026-06-07,13,24.8,0.0,0.0,12.2\n2026-06-07,14,25.0,0.0,0.0,12.6\n2026-06-07,15,25.4,0.0,0.0,11.5\n2026-06-07,16,24.9,0.0,0.0,14.5\n2026-06-07,17,25.6,0.0,0.0,5.9\n2026-06-07,18,24.6,0.1,0.0,5.0\n2026-06-07,19,21.3,0.6,0.0,11.4\n2026-06-07,20,20.3,0.0,0.0,7.9\n2026-06-07,21,22.7,0.0,0.0,9.5\n2026-06-07,22,21.5,0.0,0.0,8.7\n2026-06-07,23,20.7,0.0,0.0,6.3\n2026-06-08,0,20.2,0.0,0.0,6.5\n2026-06-08,1,19.7,0.0,0.0,6.5\n2026-06-08,2,18.7,0.0,0.0,2.3\n2026-06-08,3,18.2,0.0,0.0,4.6\n2026-06-08,4,18.0,0.0,0.0,7.9\n2026-06-08,5,17.6,0.0,0.0,12.4\n2026-06-08,6,17.4,0.0,0.0,12.4\n2026-06-08,7,17.7,0.0,0.0,10.3\n2026-06-08,8,17.8,0.0,0.0,9.4\n2026-06-08,9,19.9,0.0,0.0,9.2\n2026-06-08,10,21.3,0.0,0.0,8.9\n2026-06-08,11,23.6,0.1,0.0,9.4\n2026-06-08,12,25.1,0.1,0.0,10.1\n2026-06-08,13,25.4,0.0,0.0,13.3\n2026-06-08,14,25.0,0.0,0.0,18.3\n2026-06-08,15,24.8,0.1,0.0,16.9\n2026-06-08,16,23.6,0.3,0.0,18.5\n2026-06-08,17,23.1,0.1,0.0,18.1\n2026-06-08,18,22.8,0.0,0.0,15.7\n2026-06-08,19,22.0,0.0,0.0,13.4\n2026-06-08,20,21.3,0.0,0.0,13.1\n2026-06-08,21,20.5,0.0,0.0,12.3\n2026-06-08,22,20.0,0.0,0.0,10.0\n2026-06-08,23,19.9,0.0,0.0,10.0\n2026-06-09,0,19.1,0.0,0.0,9.6\n2026-06-09,1,18.3,0.0,0.0,9.4\n2026-06-09,2,17.5,0.0,0.0,8.4\n2026-06-09,3,17.3,0.0,0.0,4.5\n2026-06-09,4,16.5,0.0,0.0,5.1\n2026-06-09,5,16.0,0.0,0.0,5.6\n2026-06-09,6,16.2,0.0,0.0,4.1\n2026-06-09,7,17.7,0.0,0.0,2.6\n2026-06-09,8,20.2,0.0,0.0,3.0\n2026-06-09,9,22.5,0.0,0.0,3.2\n2026-06-09,10,24.4,0.0,0.0,4.2\n2026-06-09,11,26.1,0.0,0.0,4.6\n2026-06-09,12,27.1,0.0,0.0,9.5\n2026-06-09,13,26.9,0.0,0.0,14.8\n2026-06-09,14,26.4,0.0,0.0,16.8\n2026-06-09,15,25.9,0.0,0.0,14.4\n2026-06-09,16,25.7,0.0,0.0,12.3\n2026-06-09,17,25.1,0.0,0.0,15.9\n2026-06-09,18,23.4,0.0,0.0,16.7\n2026-06-09,19,21.7,0.0,0.0,12.6\n2026-06-09,20,21.0,0.0,0.0,7.7\n2026-06-09,21,21.3,0.0,0.0,4.4\n2026-06-09,22,20.4,0.0,0.0,5.2\n2026-06-09,23,19.4,0.0,0.0,6.7\n2026-06-10,0,18.9,0.0,0.0,7.3\n2026-06-10,1,18.2,0.0,0.0,7.5\n2026-06-10,2,17.8,0.0,0.0,4.2\n2026-06-10,3,15.7,0.0,0.0,1.4\n2026-06-10,4,15.9,4.4,0.0,1.9\n2026-06-10,5,15.7,1.4,0.0,8.7\n2026-06-10,6,15.7,0.1,0.0,1.2\n2026-06-10,7,16.6,0.0,0.0,5.0\n2026-06-10,8,18.7,0.0,0.0,9.1\n2026-06-10,9,19.9,0.0,0.0,7.5\n2026-06-10,10,21.2,0.3,0.0,9.9\n2026-06-10,11,22.2,0.0,0.0,10.9\n2026-06-10,12,23.0,0.0,0.0,10.6\n2026-06-10,13,23.8,0.0,0.0,13.0\n2026-06-10,14,23.9,0.0,0.0,15.7\n2026-06-10,15,24.4,0.0,0.0,15.1\n2026-06-10,16,23.1,0.0,0.0,16.7\n2026-06-10,17,22.4,0.0,0.0,14.5\n2026-06-10,18,21.2,0.0,0.0,14.4\n2026-06-10,19,19.8,0.0,0.0,12.6\n2026-06-10,20,18.8,0.0,0.0,9.7\n2026-06-10,21,18.4,0.0,0.0,9.5\n2026-06-10,22,17.9,0.0,0.0,7.9\n2026-06-10,23,17.6,0.0,0.0,6.8\n2026-06-11,0,17.3,0.0,0.0,6.9\n2026-06-11,1,16.9,0.0,0.0,5.6\n2026-06-11,2,16.7,0.0,0.0,4.5\n2026-06-11,3,16.6,0.0,0.0,2.5\n2026-06-11,4,16.3,0.0,0.0,1.8\n2026-06-11,5,15.7,0.0,0.0,2.8\n2026-06-11,6,16.1,0.0,0.0,4.8\n2026-06-11,7,17.5,0.0,0.0,2.8\n2026-06-11,8,19.6,0.0,0.0,3.1\n2026-06-11,9,21.9,0.0,0.0,6.0\n2026-06-11,10,23.1,0.0,0.0,8.6\n2026-06-11,11,23.5,0.0,0.0,14.2\n2026-06-11,12,23.4,0.0,0.0,15.7\n2026-06-11,13,23.8,0.0,0.0,15.3\n2026-06-11,14,24.0,0.0,0.0,15.8\n2026-06-11,15,24.2,0.0,0.0,17.8\n2026-06-11,16,22.6,0.0,0.0,17.4\n2026-06-11,17,22.4,0.0,0.0,19.0\n2026-06-11,18,21.8,0.0,0.0,17.6\n2026-06-11,19,20.8,0.0,0.0,14.2\n2026-06-11,20,20.0,0.0,0.0,10.8\n2026-06-11,21,19.7,0.0,0.0,10.6\n2026-06-11,22,19.1,0.0,0.0,7.7\n2026-06-11,23,18.7,0.0,0.0,7.2\n2026-06-12,0,18.4,0.0,0.0,7.7\n2026-06-12,1,18.3,0.0,0.0,8.6\n2026-06-12,2,18.0,0.0,0.0,7.8\n2026-06-12,3,17.4,0.0,0.0,7.2\n2026-06-12,4,17.0,0.0,0.0,7.0\n2026-06-12,5,17.0,0.0,0.0,7.0\n2026-06-12,6,17.7,0.0,0.0,7.0\n2026-06-12,7,19.3,0.0,0.0,7.2\n2026-06-12,8,20.8,0.0,0.0,7.0\n2026-06-12,9,22.2,0.0,0.0,4.3\n2026-06-12,10,24.0,0.0,0.0,6.2\n2026-06-12,11,25.1,0.0,0.0,11.5\n2026-06-12,12,24.7,0.0,0.0,15.7\n2026-06-12,13,24.5,0.0,0.0,15.7\n2026-06-12,14,24.8,0.0,0.0,17.1\n2026-06-12,15,25.5,0.0,0.0,15.5\n2026-06-12,16,25.1,0.0,0.0,14.8\n2026-06-12,17,24.5,0.0,0.0,13.1\n2026-06-12,18,23.3,0.0,0.0,11.1\n2026-06-12,19,21.7,0.0,0.0,8.9\n2026-06-12,20,20.2,0.0,0.0,6.6\n2026-06-12,21,20.0,0.0,0.0,3.1\n2026-06-12,22,19.0,0.0,0.0,5.9\n2026-06-12,23,18.1,0.0,0.0,5.0\n2026-06-13,0,17.5,0.0,0.0,3.9\n2026-06-13,1,17.2,0.0,0.0,2.2\n2026-06-13,2,17.0,0.0,0.0,1.9\n2026-06-13,3,16.4,0.0,0.0,1.5\n2026-06-13,4,16.0,0.0,0.0,2.9\n2026-06-13,5,15.9,0.0,0.0,3.4\n2026-06-13,6,16.0,0.0,0.0,3.1\n2026-06-13,7,18.0,0.0,0.0,2.8\n2026-06-13,8,20.9,0.0,0.0,5.2\n2026-06-13,9,23.3,0.0,0.0,4.5\n2026-06-13,10,25.7,0.0,0.0,4.5\n2026-06-13,11,27.6,0.0,0.0,5.0\n2026-06-13,12,29.0,0.0,0.0,5.5\n2026-06-13,13,30.0,0.0,0.0,6.2\n2026-06-13,14,30.6,0.0,0.0,9.8\n2026-06-13,15,30.9,0.0,0.0,10.9\n2026-06-13,16,30.2,0.0,0.0,11.7\n2026-06-13,17,28.3,0.0,0.0,14.4\n2026-06-13,18,26.2,0.0,0.0,13.7\n2026-06-13,19,23.7,0.0,0.0,13.3\n2026-06-13,20,22.1,0.0,0.0,10.0\n2026-06-13,21,21.9,0.0,0.0,4.9\n2026-06-13,22,21.3,0.0,0.0,3.2\n2026-06-13,23,21.6,0.0,0.0,0.8\n2026-06-14,0,21.4,0.0,0.0,0.9\n2026-06-14,1,22.0,0.0,0.0,3.7\n2026-06-14,2,21.8,0.0,0.0,6.6\n2026-06-14,3,21.4,0.0,0.0,6.1\n2026-06-14,4,21.1,0.0,0.0,8.3\n2026-06-14,5,21.4,0.0,0.0,7.8\n2026-06-14,6,21.7,0.0,0.0,7.6\n2026-06-14,7,22.4,0.0,0.0,9.0\n2026-06-14,8,23.7,0.0,0.0,9.3\n2026-06-14,9,24.5,0.0,0.0,6.7\n2026-06-14,10,24.8,0.0,0.0,5.8\n2026-06-14,11,26.3,0.0,0.0,3.6\n2026-06-14,12,27.6,0.0,0.0,3.4\n2026-06-14,13,28.9,0.0,0.0,2.5\n2026-06-14,14,30.1,0.0,0.0,6.3\n2026-06-14,15,26.5,0.1,0.0,18.7\n2026-06-14,16,25.4,0.4,0.0,10.0\n2026-06-14,17,27.2,0.1,0.0,8.3\n2026-06-14,18,26.8,0.0,0.0,11.5\n2026-06-14,19,26.0,0.0,0.0,10.2\n2026-06-14,20,25.4,0.0,0.0,4.3\n2026-06-14,21,22.3,0.0,0.0,3.1\n2026-06-14,22,21.0,0.4,0.0,6.6\n2026-06-14,23,20.8,0.1,0.0,9.8\n2026-06-15,0,20.9,0.0,0.0,8.2\n2026-06-15,1,20.6,0.0,0.0,6.9\n2026-06-15,2,20.0,0.0,0.0,4.3\n2026-06-15,3,19.8,0.0,0.0,1.9\n2026-06-15,4,19.1,0.0,0.0,1.2\n2026-06-15,5,18.9,0.0,0.0,4.9\n2026-06-15,6,19.3,0.0,0.0,5.6\n2026-06-15,7,20.6,0.0,0.0,5.9\n2026-06-15,8,23.3,0.0,0.0,5.8\n2026-06-15,9,25.8,0.0,0.0,2.9\n2026-06-15,10,27.9,0.0,0.0,4.1\n2026-06-15,11,29.6,0.0,0.0,3.5\n2026-06-15,12,30.8,0.0,0.0,1.4\n2026-06-15,13,30.5,0.1,0.0,8.6\n2026-06-15,14,29.7,0.6,0.0,13.2\n2026-06-15,15,28.5,0.5,0.0,13.5\n2026-06-15,16,28.8,0.2,0.0,11.1\n2026-06-15,17,29.6,0.0,0.0,10.4\n2026-06-15,18,28.8,0.0,0.0,11.5\n2026-06-15,19,27.7,0.0,0.0,11.1\n2026-06-15,20,26.3,0.0,0.0,9.4\n2026-06-15,21,25.4,0.0,0.0,6.9\n2026-06-15,22,24.5,0.0,0.0,6.9\n2026-06-15,23,23.8,0.0,0.0,4.0\n2026-06-16,0,23.7,0.0,0.0,2.2\n2026-06-16,1,23.7,0.0,0.0,0.8\n2026-06-16,2,23.0,0.0,0.0,4.9\n2026-06-16,3,22.2,0.0,0.0,1.2\n2026-06-16,4,21.3,0.0,0.0,4.2\n2026-06-16,5,21.0,0.0,0.0,6.2\n2026-06-16,6,21.0,0.0,0.0,7.4\n2026-06-16,7,22.5,0.0,0.0,7.4\n2026-06-16,8,24.4,0.0,0.0,8.8\n2026-06-16,9,26.9,0.0,0.0,7.9\n2026-06-16,10,28.6,0.0,0.0,6.7\n2026-06-16,11,29.8,0.1,0.0,5.7\n2026-06-16,12,30.6,0.2,0.0,6.1\n2026-06-16,13,31.6,0.2,0.0,6.5\n2026-06-16,14,31.9,0.0,0.0,8.5\n2026-06-16,15,31.7,0.2,0.0,12.1\n2026-06-16,16,31.1,0.2,0.0,13.3\n2026-06-16,17,31.1,0.0,0.0,12.1\n2026-06-16,18,30.4,0.0,0.0,12.6\n2026-06-16,19,29.1,0.0,0.0,11.2\n2026-06-16,20,27.4,0.0,0.0,7.5\n2026-06-16,21,26.8,0.0,0.0,6.4\n2026-06-16,22,25.6,0.0,0.0,5.0\n2026-06-16,23,24.8,0.0,0.0,5.8\n2026-06-17,0,24.3,0.0,0.0,6.1\n2026-06-17,1,23.8,0.0,0.0,1.8\n2026-06-17,2,23.3,0.0,0.0,2.3\n2026-06-17,3,23.1,0.0,0.0,2.4\n2026-06-17,4,21.9,0.0,0.0,6.3\n2026-06-17,5,21.3,0.0,0.0,7.8\n2026-06-17,6,21.7,0.0,0.0,7.4\n2026-06-17,7,23.6,0.0,0.0,6.0\n2026-06-17,8,25.7,0.0,0.0,7.4\n2026-06-17,9,27.1,0.0,0.0,3.2\n2026-06-17,10,28.2,0.0,0.0,1.7\n2026-06-17,11,29.9,0.0,0.0,1.7\n2026-06-17,12,30.7,0.0,0.0,4.1\n2026-06-17,13,32.4,0.0,0.0,8.4\n2026-06-17,14,32.0,0.0,0.0,13.7\n2026-06-17,15,31.2,0.3,0.0,11.7\n2026-06-17,16,30.6,0.1,0.0,9.9\n2026-06-17,17,30.6,0.0,0.0,9.0\n2026-06-17,18,30.0,0.0,0.0,11.0\n2026-06-17,19,28.7,0.0,0.0,10.9\n2026-06-17,20,26.4,0.0,0.0,10.1\n2026-06-17,21,22.9,0.0,0.0,8.8\n2026-06-17,22,23.1,0.0,0.0,6.2\n2026-06-17,23,23.1,0.0,0.0,9.4\n2026-06-18,0,22.8,0.0,0.0,7.1\n2026-06-18,1,22.4,0.1,0.0,4.4\n2026-06-18,2,22.7,0.1,0.0,2.9\n2026-06-18,3,22.8,0.5,0.0,6.2\n2026-06-18,4,22.0,0.0,0.0,6.7\n2026-06-18,5,21.5,0.0,0.0,5.8\n2026-06-18,6,21.8,0.0,0.0,4.6\n2026-06-18,7,23.3,0.0,0.0,4.5\n2026-06-18,8,25.3,0.0,0.0,4.0\n2026-06-18,9,26.7,0.0,0.0,3.1\n2026-06-18,10,28.5,0.0,0.0,5.1\n2026-06-18,11,30.2,0.0,0.0,8.2\n2026-06-18,12,29.5,0.0,0.0,14.4\n2026-06-18,13,30.0,0.0,0.0,15.1\n2026-06-18,14,30.4,0.0,0.0,15.0\n2026-06-18,15,30.7,0.0,0.0,12.5\n2026-06-18,16,30.2,0.0,0.0,12.9\n2026-06-18,17,29.0,0.0,0.0,14.4\n2026-06-18,18,27.9,0.0,0.0,13.3\n2026-06-18,19,27.4,0.0,0.0,10.7\n2026-06-18,20,26.3,0.0,0.0,9.3\n2026-06-18,21,26.1,0.0,0.0,7.1\n2026-06-18,22,24.9,0.0,0.0,8.3\n2026-06-18,23,23.9,0.0,0.0,7.2\n2026-06-19,0,23.4,0.0,0.0,7.0\n2026-06-19,1,23.4,0.0,0.0,4.9\n2026-06-19,2,23.3,0.0,0.0,3.3\n2026-06-19,3,22.9,0.0,0.0,2.2\n2026-06-19,4,22.4,0.0,0.0,2.0\n2026-06-19,5,22.2,0.0,0.0,1.9\n2026-06-19,6,22.8,0.0,0.0,1.3\n2026-06-19,7,24.4,0.0,0.0,1.9\n2026-06-19,8,26.3,0.1,0.0,2.4\n2026-06-19,9,28.2,0.0,0.0,3.7\n2026-06-19,10,30.2,0.1,0.0,4.2\n2026-06-19,11,30.9,0.3,0.0,6.0\n2026-06-19,12,31.5,0.4,0.0,6.4\n2026-06-19,13,31.8,0.6,0.0,9.2\n2026-06-19,14,32.2,0.3,0.0,10.3\n2026-06-19,15,33.5,0.6,0.0,10.7\n2026-06-19,16,31.8,0.3,0.0,11.6\n2026-06-19,17,26.4,1.6,0.0,8.6\n2026-06-19,18,26.5,0.5,0.0,4.2\n2026-06-19,19,25.1,0.9,0.0,4.4\n2026-06-19,20,24.3,1.7,0.0,3.0\n2026-06-19,21,23.8,1.3,0.0,12.1\n2026-06-19,22,23.5,0.6,0.0,14.2\n2026-06-19,23,23.3,0.8,0.0,15.7\n2026-06-20,0,22.9,1.3,0.0,14.3\n2026-06-20,1,22.7,1.4,0.0,15.7\n2026-06-20,2,22.2,5.8,0.0,17.4\n2026-06-20,3,21.9,3.2,0.0,17.7\n2026-06-20,4,21.7,6.3,0.0,18.7\n2026-06-20,5,21.4,6.9,0.0,18.7\n2026-06-20,6,21.3,5.5,0.0,16.4\n2026-06-20,7,21.5,3.6,0.0,15.3\n2026-06-20,8,21.6,3.1,0.0,15.4\n2026-06-20,9,21.8,1.0,0.0,14.7\n2026-06-20,10,21.8,3.7,0.0,17.3\n2026-06-20,11,22.2,3.3,0.0,17.0\n2026-06-20,12,21.7,2.3,0.0,15.0\n2026-06-20,13,23.4,2.2,0.0,14.7\n2026-06-20,14,23.9,0.1,0.0,13.2\n2026-06-20,15,23.3,0.0,0.0,12.6\n2026-06-20,16,23.3,0.0,0.0,11.1\n2026-06-20,17,22.9,0.3,0.0,6.3\n2026-06-20,18,22.1,0.4,0.0,8.9\n2026-06-20,19,21.4,0.2,0.0,12.3\n2026-06-20,20,20.7,0.3,0.0,13.3\n2026-06-20,21,20.8,0.1,0.0,10.9\n2026-06-20,22,20.8,0.0,0.0,12.4\n2026-06-20,23,20.8,0.0,0.0,12.2\n2026-06-21,0,21.1,0.0,0.0,14.8\n2026-06-21,1,20.5,0.0,0.0,14.8\n2026-06-21,2,20.0,0.0,0.0,14.4\n2026-06-21,3,20.0,0.0,0.0,12.0\n2026-06-21,4,19.5,0.0,0.0,10.9\n2026-06-21,5,19.8,0.0,0.0,10.7\n2026-06-21,6,19.8,0.0,0.0,10.0\n2026-06-21,7,20.0,0.0,0.0,8.5\n2026-06-21,8,20.5,0.0,0.0,7.2\n2026-06-21,9,21.8,0.0,0.0,6.2\n2026-06-21,10,24.0,0.0,0.0,4.4\n2026-06-21,11,24.8,0.0,0.0,3.5\n2026-06-21,12,26.0,0.0,0.0,4.8\n2026-06-21,13,26.4,0.0,0.0,4.4\n2026-06-21,14,25.6,0.0,0.0,9.1\n2026-06-21,15,24.5,0.0,0.0,10.6\n2026-06-21,16,24.2,0.0,0.0,10.0\n2026-06-21,17,23.9,0.0,0.0,9.1\n2026-06-21,18,23.1,0.0,0.0,8.6\n2026-06-21,19,22.8,0.0,0.0,6.0\n2026-06-21,20,22.3,0.0,0.0,4.0\n2026-06-21,21,22.4,0.0,0.0,4.9\n2026-06-21,22,21.9,0.0,0.0,5.3\n2026-06-21,23,21.3,0.0,0.0,5.0\n2026-06-22,0,21.3,0.0,0.0,4.4\n2026-06-22,1,21.2,0.0,0.0,2.6\n2026-06-22,2,21.1,0.0,0.0,1.2\n2026-06-22,3,21.4,0.0,0.0,3.4\n2026-06-22,4,21.2,0.0,0.0,3.7\n2026-06-22,5,21.0,0.0,0.0,2.9\n2026-06-22,6,21.3,0.0,0.0,2.0\n2026-06-22,7,21.8,0.0,0.0,1.4\n2026-06-22,8,22.5,0.0,0.0,3.6\n2026-06-22,9,23.4,0.0,0.0,4.2\n2026-06-22,10,24.6,0.2,0.0,3.5\n2026-06-22,11,26.0,0.0,0.0,3.6\n2026-06-22,12,25.8,0.3,0.0,4.2\n2026-06-22,13,26.1,0.0,0.0,5.9\n2026-06-22,14,24.9,0.0,0.0,7.3\n2026-06-22,15,26.2,0.1,0.0,7.4\n2026-06-22,16,25.6,0.4,0.0,7.8\n2026-06-22,17,24.8,0.2,0.0,8.4\n2026-06-22,18,24.5,0.0,0.0,6.5\n2026-06-22,19,23.6,0.0,0.0,5.6\n2026-06-22,20,22.2,0.0,0.0,4.4\n2026-06-22,21,22.3,0.0,0.0,5.0\n2026-06-22,22,21.5,0.1,0.0,3.2\n2026-06-22,23,20.9,0.1,0.0,4.2\n2026-06-23,0,20.5,0.0,0.0,3.6\n2026-06-23,1,20.0,0.0,0.0,4.0\n2026-06-23,2,19.8,0.0,0.0,4.5\n2026-06-23,3,20.4,0.0,0.0,1.5\n2026-06-23,4,20.0,0.0,0.0,0.2\n2026-06-23,5,19.7,0.1,0.0,6.1\n2026-06-23,6,19.7,0.1,0.0,7.0\n2026-06-23,7,21.1,0.0,0.0,6.6\n2026-06-23,8,22.3,0.1,0.0,9.4\n2026-06-23,9,24.2,0.0,0.0,10.2\n2026-06-23,10,25.1,0.0,0.0,12.8\n2026-06-23,11,26.0,0.0,0.0,11.9\n2026-06-23,12,27.0,0.0,0.0,11.5\n2026-06-23,13,27.7,0.0,0.0,11.6\n2026-06-23,14,28.1,0.0,0.0,10.6\n2026-06-23,15,29.2,0.0,0.0,10.2\n2026-06-23,16,29.1,0.0,0.0,11.1\n2026-06-23,17,28.8,0.0,0.0,10.2\n2026-06-23,18,28.2,0.0,0.0,7.2\n2026-06-23,19,26.8,0.0,0.0,2.3\n2026-06-23,20,24.3,0.0,0.0,2.4\n2026-06-23,21,24.0,0.0,0.0,4.8\n2026-06-23,22,22.9,0.0,0.0,7.9\n2026-06-23,23,22.2,0.0,0.0,11.9\n2026-06-24,0,20.9,0.0,0.0,11.5\n2026-06-24,1,20.0,0.0,0.0,10.3\n2026-06-24,2,19.3,0.0,0.0,7.7\n2026-06-24,3,19.2,0.0,0.0,4.4\n2026-06-24,4,18.6,0.0,0.0,5.8\n2026-06-24,5,18.2,0.0,0.0,6.3\n2026-06-24,6,18.2,0.0,0.0,7.8\n2026-06-24,7,19.4,0.0,0.0,7.2\n2026-06-24,8,20.9,0.0,0.0,8.7\n2026-06-24,9,22.5,0.0,0.0,8.1\n2026-06-24,10,23.5,0.0,0.0,9.5\n2026-06-24,11,24.4,0.0,0.0,9.6\n2026-06-24,12,25.8,0.0,0.0,9.8\n2026-06-24,13,27.0,0.0,0.0,9.0\n2026-06-24,14,27.4,0.0,0.0,6.6\n2026-06-24,15,27.4,0.0,0.0,9.0\n2026-06-24,16,27.5,0.0,0.0,9.6\n2026-06-24,17,27.5,0.0,0.0,6.8\n2026-06-24,18,26.8,0.0,0.0,6.0\n2026-06-24,19,25.5,0.0,0.0,3.6\n2026-06-24,20,23.7,0.0,0.0,2.7\n2026-06-24,21,23.2,0.0,0.0,2.2\n2026-06-24,22,22.6,0.0,0.0,4.5\n2026-06-24,23,21.9,0.0,0.0,5.7\n2026-06-25,0,21.4,0.0,0.0,4.2\n2026-06-25,1,21.8,0.0,0.0,4.7\n2026-06-25,2,21.7,0.0,0.0,5.6\n2026-06-25,3,20.8,0.0,0.0,5.3\n2026-06-25,4,20.5,0.0,0.0,6.9\n2026-06-25,5,20.2,0.0,0.0,3.8\n2026-06-25,6,20.6,0.0,0.0,4.3\n2026-06-25,7,21.0,0.0,0.0,5.5\n2026-06-25,8,21.6,0.0,0.0,5.6\n2026-06-25,9,22.4,0.0,0.0,6.7\n2026-06-25,10,23.0,0.0,0.0,5.6\n2026-06-25,11,24.1,0.0,0.0,7.3\n2026-06-25,12,25.6,0.0,0.0,8.4\n2026-06-25,13,25.6,0.0,0.0,9.3\n2026-06-25,14,25.9,0.0,0.0,6.6\n2026-06-25,15,25.2,0.2,0.0,6.4\n2026-06-25,16,24.4,0.0,0.0,4.6\n2026-06-25,17,24.0,0.0,0.0,4.1\n2026-06-25,18,23.2,0.4,0.0,5.0\n2026-06-25,19,21.3,1.0,0.0,9.6\n2026-06-25,20,20.6,0.9,0.0,2.3\n2026-06-25,21,20.7,0.1,0.0,9.5\n2026-06-25,22,20.3,0.0,0.0,6.4\n2026-06-25,23,19.8,0.2,0.0,3.5\n2026-06-26,0,19.7,0.1,0.0,1.1\n2026-06-26,1,19.5,0.0,0.0,0.4\n2026-06-26,2,19.0,0.0,0.0,1.1\n2026-06-26,3,18.9,0.0,0.0,7.4\n2026-06-26,4,18.0,0.0,0.0,8.0\n2026-06-26,5,17.6,0.0,0.0,5.6\n2026-06-26,6,17.7,0.0,0.0,6.8\n2026-06-26,7,18.8,0.0,0.0,5.5\n2026-06-26,8,20.8,0.0,0.0,4.1\n2026-06-26,9,23.2,0.0,0.0,6.4\n2026-06-26,10,24.7,0.0,0.0,6.1\n2026-06-26,11,26.2,0.0,0.0,7.7\n2026-06-26,12,27.5,0.0,0.0,8.8\n2026-06-26,13,27.8,0.0,0.0,10.9\n2026-06-26,14,28.4,0.0,0.0,9.5\n2026-06-26,15,28.7,0.0,0.0,6.1\n2026-06-26,16,28.1,0.0,0.0,7.6\n2026-06-26,17,27.2,0.0,0.0,10.3\n2026-06-26,18,26.2,0.0,0.0,11.9\n2026-06-26,19,25.5,0.0,0.0,9.8\n2026-06-26,20,24.4,0.0,0.0,7.9\n2026-06-26,21,24.0,0.0,0.0,6.1\n2026-06-26,22,23.3,0.0,0.0,4.8\n2026-06-26,23,22.6,0.0,0.0,6.2\n2026-06-27,0,22.1,0.0,0.0,4.1\n2026-06-27,1,21.9,0.0,0.0,4.9\n2026-06-27,2,21.7,0.0,0.0,6.4\n2026-06-27,3,21.3,0.0,0.0,8.2\n2026-06-27,4,20.6,0.0,0.0,7.2\n2026-06-27,5,20.2,0.0,0.0,6.2\n2026-06-27,6,20.1,0.0,0.0,7.0\n2026-06-27,7,21.4,0.0,0.0,4.1\n2026-06-27,8,23.4,0.0,0.0,4.7\n2026-06-27,9,25.8,0.0,0.0,5.1\n2026-06-27,10,28.1,0.0,0.0,6.7\n2026-06-27,11,29.6,0.0,0.0,7.4\n2026-06-27,12,30.8,0.0,0.0,3.5\n2026-06-27,13,31.5,0.0,0.0,7.5\n2026-06-27,14,31.3,0.0,0.0,12.3\n2026-06-27,15,30.7,0.0,0.0,13.6\n2026-06-27,16,29.8,0.0,0.0,16.1\n2026-06-27,17,29.3,0.0,0.0,15.3\n2026-06-27,18,28.4,0.0,0.0,12.8\n2026-06-27,19,27.6,0.0,0.0,9.7\n2026-06-27,20,26.2,0.0,0.0,9.3\n2026-06-27,21,25.7,0.0,0.0,8.7\n2026-06-27,22,24.5,0.0,0.0,7.1\n2026-06-27,23,24.2,0.0,0.0,6.4\n2026-06-28,0,23.9,0.0,0.0,6.5\n2026-06-28,1,23.6,0.0,0.0,4.5\n2026-06-28,2,23.0,0.0,0.0,0.9\n2026-06-28,3,22.4,0.0,0.0,4.6\n2026-06-28,4,21.3,0.0,0.0,6.3\n2026-06-28,5,20.6,0.0,0.0,6.0\n2026-06-28,6,20.5,0.0,0.0,7.4\n2026-06-28,7,21.6,0.0,0.0,5.8\n2026-06-28,8,23.5,0.0,0.0,5.0\n2026-06-28,9,25.7,0.0,0.0,5.9\n2026-06-28,10,27.2,0.0,0.0,6.5\n2026-06-28,11,28.2,0.0,0.0,8.4\n2026-06-28,12,28.4,0.1,0.0,10.7\n2026-06-28,13,28.5,0.0,0.0,11.8\n2026-06-28,14,28.7,0.0,0.0,12.5\n2026-06-28,15,29.1,0.0,0.0,11.7\n2026-06-28,16,28.9,0.0,0.0,11.2\n2026-06-28,17,28.0,0.0,0.0,11.9\n2026-06-28,18,26.3,0.0,0.0,12.6\n2026-06-28,19,24.3,0.0,0.0,10.3\n2026-06-28,20,23.2,0.0,0.0,7.6\n2026-06-28,21,22.5,0.0,0.0,7.2\n2026-06-28,22,22.2,0.0,0.0,6.3\n2026-06-28,23,22.2,0.0,0.0,3.7\n2026-06-29,0,22.1,0.0,0.0,5.1\n2026-06-29,1,21.7,0.0,0.0,6.4\n2026-06-29,2,21.2,0.0,0.0,6.1\n2026-06-29,3,21.0,0.0,0.0,7.3\n2026-06-29,4,20.6,0.0,0.0,5.5\n2026-06-29,5,20.6,0.0,0.0,5.0\n2026-06-29,6,20.5,0.0,0.0,2.5\n2026-06-29,7,21.4,0.0,0.0,4.8\n2026-06-29,8,22.8,0.0,0.0,3.3\n2026-06-29,9,24.5,0.0,0.0,2.8\n2026-06-29,10,26.6,0.0,0.0,4.7\n2026-06-29,11,28.2,0.0,0.0,6.5\n2026-06-29,12,29.4,0.0,0.0,8.1\n2026-06-29,13,30.1,0.0,0.0,9.4\n2026-06-29,14,30.7,0.0,0.0,11.2\n2026-06-29,15,30.3,0.0,0.0,13.5\n2026-06-29,16,29.5,0.0,0.0,15.0\n2026-06-29,17,28.5,0.0,0.0,12.8\n2026-06-29,18,27.4,0.0,0.0,13.2\n2026-06-29,19,26.1,0.0,0.0,11.4\n2026-06-29,20,25.1,0.0,0.0,8.9\n2026-06-29,21,25.5,0.0,0.0,7.1\n2026-06-29,22,24.4,0.0,0.0,7.5\n2026-06-29,23,23.7,0.0,0.0,6.7\n2026-06-30,0,23.3,0.0,0.0,5.9\n2026-06-30,1,23.0,0.0,0.0,4.5\n2026-06-30,2,22.5,0.0,0.0,3.8\n2026-06-30,3,22.7,0.0,0.0,4.6\n2026-06-30,4,22.3,0.0,0.0,1.6\n2026-06-30,5,22.1,0.0,0.0,3.8\n2026-06-30,6,21.9,0.0,0.0,5.9\n2026-06-30,7,23.1,0.0,0.0,5.5\n2026-06-30,8,24.4,0.0,0.0,4.8\n2026-06-30,9,26.1,0.0,0.0,5.6\n2026-06-30,10,27.7,0.0,0.0,4.6\n2026-06-30,11,29.1,0.0,0.0,5.0\n2026-06-30,12,29.9,0.2,0.0,9.4\n2026-06-30,13,29.9,0.1,0.0,12.4\n2026-06-30,14,30.7,0.0,0.0,14.1\n2026-06-30,15,30.7,0.0,0.0,14.3\n2026-06-30,16,29.7,0.0,0.0,14.3\n2026-06-30,17,28.4,0.0,0.0,14.9\n2026-06-30,18,27.7,0.0,0.0,13.6\n2026-06-30,19,26.9,0.0,0.0,11.7\n2026-06-30,20,25.9,0.0,0.0,10.1\n2026-06-30,21,25.7,0.0,0.0,8.8\n2026-06-30,22,25.2,0.0,0.0,7.4\n2026-06-30,23,24.8,0.0,0.0,7.6\n2026-07-01,0,24.1,0.0,0.0,5.6\n2026-07-01,1,23.9,0.0,0.0,5.1\n2026-07-01,2,23.9,0.0,0.0,4.7\n2026-07-01,3,23.8,0.0,0.0,5.7\n2026-07-01,4,23.5,0.0,0.0,4.9\n2026-07-01,5,23.1,0.1,0.0,5.1\n2026-07-01,6,23.2,0.0,0.0,5.0\n2026-07-01,7,24.5,0.1,0.0,6.8\n2026-07-01,8,25.6,0.0,0.0,7.9\n2026-07-01,9,25.0,0.0,0.0,9.7\n2026-07-01,10,27.2,0.1,0.0,9.9\n2026-07-01,11,29.4,0.0,0.0,9.4\n2026-07-01,12,31.4,0.0,0.0,10.2\n2026-07-01,13,31.3,0.0,0.0,9.2\n2026-07-01,14,31.9,0.0,0.0,5.1\n2026-07-01,15,31.3,0.3,0.0,3.9\n2026-07-01,16,28.7,2.0,0.0,9.0\n2026-07-01,17,28.4,1.4,0.0,8.6\n2026-07-01,18,28.2,0.3,0.0,10.7\n2026-07-01,19,27.5,0.1,0.0,9.2\n2026-07-01,20,26.7,0.0,0.0,7.6\n2026-07-01,21,26.7,0.0,0.0,4.5\n2026-07-01,22,25.8,0.0,0.0,5.5\n2026-07-01,23,25.0,0.0,0.0,8.6\n2026-07-02,0,24.0,0.2,0.0,6.7\n2026-07-02,1,23.4,0.4,0.0,2.3\n2026-07-02,2,23.1,0.0,0.0,6.6\n2026-07-02,3,22.0,0.0,0.0,7.2\n2026-07-02,4,21.7,0.4,0.0,9.9\n2026-07-02,5,21.6,0.0,0.0,7.6\n2026-07-02,6,21.9,0.0,0.0,8.0\n2026-07-02,7,22.4,0.0,0.0,8.4\n2026-07-02,8,23.8,0.1,0.0,8.7\n2026-07-02,9,26.2,0.0,0.0,7.1\n2026-07-02,10,26.5,0.0,0.0,7.9\n2026-07-02,11,28.1,0.0,0.0,7.7\n2026-07-02,12,28.8,0.0,0.0,8.5\n2026-07-02,13,29.7,0.0,0.0,11.2\n2026-07-02,14,27.4,1.0,0.0,16.8\n2026-07-02,15,27.8,0.0,0.0,9.8\n2026-07-02,16,28.0,0.1,0.0,8.4\n2026-07-02,17,27.5,0.1,0.0,12.1\n2026-07-02,18,25.8,0.8,0.0,14.1\n2026-07-02,19,25.0,0.2,0.0,11.6\n2026-07-02,20,24.3,0.0,0.0,7.4\n2026-07-02,21,23.8,0.0,0.0,7.9\n2026-07-02,22,23.4,0.0,0.0,7.7\n2026-07-02,23,23.1,0.0,0.0,7.0\n2026-07-03,0,22.9,0.0,0.0,6.1\n2026-07-03,1,22.6,0.0,0.0,4.0\n2026-07-03,2,22.3,0.0,0.0,5.6\n2026-07-03,3,22.4,0.0,0.0,4.9\n2026-07-03,4,21.4,0.8,0.0,4.6\n2026-07-03,5,21.2,0.2,0.0,4.0\n2026-07-03,6,21.3,0.2,0.0,5.8\n2026-07-03,7,21.9,0.1,0.0,6.3\n2026-07-03,8,23.3,0.0,0.0,6.0\n2026-07-03,9,25.2,0.4,0.0,7.4\n2026-07-03,10,26.4,0.5,0.0,7.3\n2026-07-03,11,27.8,0.6,0.0,8.9\n2026-07-03,12,28.4,0.2,0.0,11.0\n2026-07-03,13,28.3,0.1,0.0,15.5\n2026-07-03,14,28.3,0.0,0.0,15.2\n2026-07-03,15,28.7,0.0,0.0,12.1\n2026-07-03,16,28.0,0.0,0.0,11.4\n2026-07-03,17,28.0,0.0,0.0,11.5\n2026-07-03,18,26.6,0.0,0.0,12.8\n2026-07-03,19,24.5,0.0,0.0,12.4\n2026-07-03,20,23.7,0.0,0.0,10.8\n2026-07-03,21,24.0,0.0,0.0,8.6\n2026-07-03,22,23.6,0.0,0.0,10.3\n2026-07-03,23,23.6,0.0,0.0,9.9\n2026-07-04,0,23.5,0.0,0.0,10.6\n2026-07-04,1,23.1,0.1,0.0,9.9\n2026-07-04,2,22.9,0.0,0.0,8.8\n2026-07-04,3,22.9,0.0,0.0,8.5\n2026-07-04,4,22.4,0.1,0.0,6.8\n2026-07-04,5,22.3,0.2,0.0,6.7\n2026-07-04,6,22.5,0.0,0.0,4.7\n2026-07-04,7,23.4,0.1,0.0,8.2\n2026-07-04,8,24.9,0.1,0.0,9.0\n2026-07-04,9,26.8,0.1,0.0,9.7\n2026-07-04,10,27.7,0.1,0.0,8.0\n2026-07-04,11,29.3,0.0,0.0,11.1\n2026-07-04,12,29.4,0.0,0.0,11.9\n2026-07-04,13,29.6,0.0,0.0,13.8\n2026-07-04,14,29.1,0.0,0.0,16.7\n2026-07-04,15,29.8,0.0,0.0,12.5\n2026-07-04,16,29.2,0.0,0.0,12.2\n2026-07-04,17,28.3,0.0,0.0,13.9\n2026-07-04,18,26.7,0.0,0.0,12.8\n2026-07-04,19,25.4,0.0,0.0,12.0\n2026-07-04,20,24.0,0.0,0.0,9.2\n2026-07-04,21,24.4,0.0,0.0,4.3\n2026-07-04,22,24.3,0.0,0.0,1.2\n2026-07-04,23,24.2,0.0,0.0,3.9\n2026-07-05,0,23.8,0.0,0.0,2.9\n2026-07-05,1,23.6,0.0,0.0,2.7\n2026-07-05,2,23.4,0.0,0.0,3.6\n2026-07-05,3,24.1,0.0,0.0,3.6\n2026-07-05,4,23.9,0.0,0.0,5.7\n2026-07-05,5,24.0,0.1,0.0,9.8\n2026-07-05,6,23.5,0.0,0.0,10.6\n2026-07-05,7,23.9,0.0,0.0,11.6\n2026-07-05,8,24.7,0.0,0.0,11.2\n2026-07-05,9,25.6,0.0,0.0,13.3\n2026-07-05,10,28.1,0.0,0.0,11.7\n2026-07-05,11,28.1,0.0,0.0,12.7\n2026-07-05,12,28.3,1.0,0.0,12.8\n2026-07-05,13,29.6,0.7,0.0,14.4\n2026-07-05,14,30.6,0.4,0.0,14.4\n2026-07-05,15,29.4,0.3,0.0,12.5\n2026-07-05,16,26.8,1.5,0.0,10.8\n2026-07-05,17,27.8,0.3,0.0,12.5\n2026-07-05,18,25.2,1.2,0.0,14.6\n2026-07-05,19,24.1,1.6,0.0,15.7\n2026-07-05,20,24.2,0.5,0.0,12.8\n2026-07-05,21,24.9,0.1,0.0,14.8\n2026-07-05,22,23.4,1.3,0.0,14.8\n2026-07-05,23,23.2,4.4,0.0,6.8\n2026-07-06,0,23.2,0.5,0.0,7.4\n2026-07-06,1,23.5,0.5,0.0,10.9\n2026-07-06,2,23.5,0.5,0.0,14.2\n2026-07-06,3,23.8,0.4,0.0,10.6\n2026-07-06,4,23.7,0.1,0.0,11.2\n2026-07-06,5,23.6,0.7,0.0,13.9\n2026-07-06,6,23.3,0.9,0.0,12.0\n2026-07-06,7,23.6,0.0,0.0,10.9\n2026-07-06,8,24.6,0.1,0.0,14.8\n2026-07-06,9,25.7,0.2,0.0,17.0\n2026-07-06,10,26.0,0.5,0.0,16.1\n2026-07-06,11,28.3,0.5,0.0,17.0\n2026-07-06,12,25.7,3.9,0.0,18.7\n2026-07-06,13,27.7,0.7,0.0,16.5\n2026-07-06,14,26.2,1.1,0.0,17.3\n2026-07-06,15,27.5,0.9,0.0,22.7\n2026-07-06,16,27.4,0.2,0.0,20.5\n2026-07-06,17,27.0,0.3,0.0,21.1\n2026-07-06,18,26.7,0.1,0.0,18.7\n2026-07-06,19,26.1,0.0,0.0,14.9\n2026-07-06,20,25.3,0.0,0.0,12.4\n2026-07-06,21,25.4,0.0,0.0,8.4\n2026-07-06,22,25.1,0.0,0.0,10.2\n2026-07-06,23,24.6,0.0,0.0,8.9\n2026-07-07,0,24.1,0.0,0.0,8.2\n2026-07-07,1,23.9,0.0,0.0,8.4\n2026-07-07,2,23.9,0.0,0.0,10.2\n2026-07-07,3,24.2,0.0,0.0,8.1\n2026-07-07,4,24.3,0.0,0.0,4.8\n2026-07-07,5,24.5,0.0,0.0,8.6\n2026-07-07,6,23.2,2.9,0.0,13.4\n2026-07-07,7,23.1,1.6,0.0,8.6\n2026-07-07,8,24.2,0.5,0.0,6.6\n2026-07-07,9,26.1,0.6,0.0,12.1\n2026-07-07,10,26.9,0.1,0.0,12.3\n2026-07-07,11,28.2,0.2,0.0,14.4\n2026-07-07,12,29.1,0.4,0.0,14.0\n2026-07-07,13,28.7,0.1,0.0,12.9\n2026-07-07,14,29.9,0.4,0.0,10.8\n2026-07-07,15,28.1,0.3,0.0,8.1\n2026-07-07,16,27.1,0.8,0.0,7.7\n2026-07-07,17,27.8,0.0,0.0,3.2\n2026-07-07,18,27.6,0.0,0.0,8.3\n2026-07-07,19,26.9,0.0,0.0,7.4\n2026-07-07,20,26.6,0.0,0.0,9.1\n2026-07-07,21,28.1,0.0,0.0,10.6\n2026-07-07,22,28.1,0.0,0.0,13.7\n2026-07-07,23,27.5,0.0,0.0,13.1\n2026-07-08,0,27.2,0.0,0.0,16.5\n2026-07-08,1,27.3,0.0,0.0,20.3\n2026-07-08,2,26.9,0.0,0.0,22.4\n2026-07-08,3,26.8,0.0,0.0,22.8\n2026-07-08,4,24.1,2.4,0.0,25.7\n2026-07-08,5,23.8,1.4,0.0,27.1\n2026-07-08,6,23.5,1.9,0.0,27.0\n2026-07-08,7,23.6,4.1,0.0,23.4\n2026-07-08,8,24.1,4.4,0.0,25.4\n2026-07-08,9,23.7,1.6,0.0,19.0\n2026-07-08,10,24.7,0.4,0.0,18.0\n2026-07-08,11,24.6,0.2,0.0,15.4\n2026-07-08,12,25.6,0.1,0.0,13.8\n2026-07-08,13,26.8,0.0,0.0,14.6\n2026-07-08,14,26.9,0.0,0.0,16.2\n2026-07-08,15,26.7,0.0,0.0,12.4\n2026-07-08,16,26.6,0.1,0.0,10.6\n2026-07-08,17,26.4,0.0,0.0,10.5\n2026-07-08,18,25.3,0.0,0.0,7.3\n2026-07-08,19,25.1,0.0,0.0,6.6\n2026-07-08,20,24.6,0.0,0.0,5.3\n2026-07-08,21,24.4,0.0,0.0,1.4\n2026-07-08,22,24.2,0.0,0.0,1.7\n2026-07-08,23,24.0,0.0,0.0,1.8\n2026-07-09,0,24.4,0.0,0.0,4.1\n2026-07-09,1,24.3,0.3,0.0,6.1\n2026-07-09,2,24.1,0.4,0.0,7.4\n2026-07-09,3,23.9,0.3,0.0,8.8\n2026-07-09,4,24.2,0.1,0.0,8.0\n2026-07-09,5,24.4,0.0,0.0,8.7\n2026-07-09,6,24.4,0.1,0.0,8.3\n2026-07-09,7,24.5,0.4,0.0,8.4\n2026-07-09,8,24.5,0.3,0.0,15.8\n2026-07-09,9,24.7,0.0,0.0,15.8\n2026-07-09,10,25.0,0.7,0.0,11.3\n2026-07-09,11,24.3,2.0,0.0,12.5\n2026-07-09,12,23.4,3.8,0.0,14.1\n2026-07-09,13,24.2,2.0,0.0,15.0\n2026-07-09,14,24.6,0.8,0.0,14.5\n2026-07-09,15,23.4,0.1,0.0,11.1\n2026-07-09,16,24.7,0.1,0.0,10.9\n2026-07-09,17,25.3,0.2,0.0,13.2\n2026-07-09,18,24.2,0.4,0.0,11.2\n2026-07-09,19,23.7,0.6,0.0,10.3\n2026-07-09,20,23.4,0.3,0.0,7.7\n2026-07-09,21,24.1,0.1,0.0,9.1\n2026-07-09,22,24.1,0.0,0.0,10.4\n2026-07-09,23,23.2,0.9,0.0,13.7\n2026-07-10,0,22.5,2.0,0.0,14.7\n2026-07-10,1,22.5,6.5,0.0,13.0\n2026-07-10,2,23.0,1.6,0.0,14.5\n2026-07-10,3,23.9,1.2,0.0,18.9\n2026-07-10,4,22.9,1.6,0.0,16.1\n2026-07-10,5,22.7,8.4,0.0,10.3\n2026-07-10,6,22.4,7.1,0.0,7.9\n2026-07-10,7,23.4,0.9,0.0,9.6\n2026-07-10,8,24.7,0.3,0.0,16.2\n2026-07-10,9,24.6,0.1,0.0,11.2\n2026-07-10,10,24.9,1.4,0.0,12.9\n2026-07-10,11,25.0,0.9,0.0,16.5\n2026-07-10,12,24.8,2.2,0.0,17.1\n2026-07-10,13,25.6,1.2,0.0,15.2\n2026-07-10,14,26.9,1.3,0.0,10.9\n2026-07-10,15,25.0,0.1,0.0,13.6\n2026-07-10,16,25.0,3.1,0.0,3.8\n2026-07-10,17,27.1,0.0,0.0,3.5\n2026-07-10,18,26.1,0.3,0.0,11.5\n2026-07-10,19,25.9,0.4,0.0,10.9\n2026-07-10,20,25.4,0.4,0.0,9.1\n2026-07-10,21,25.0,0.0,0.0,5.2\n2026-07-10,22,25.0,0.1,0.0,4.0\n2026-07-10,23,24.8,0.1,0.0,3.6\n2026-07-11,0,24.6,0.0,0.0,2.5\n2026-07-11,1,24.5,0.0,0.0,5.4\n2026-07-11,2,24.2,0.0,0.0,4.4\n2026-07-11,3,24.5,0.0,0.0,5.1\n2026-07-11,4,24.2,0.0,0.0,4.7\n2026-07-11,5,23.9,0.0,0.0,4.6\n2026-07-11,6,23.9,0.0,0.0,6.0\n2026-07-11,7,25.0,0.0,0.0,5.9\n2026-07-11,8,26.4,0.0,0.0,6.5\n2026-07-11,9,28.4,0.0,0.0,5.7\n2026-07-11,10,29.3,0.0,0.0,7.4\n2026-07-11,11,30.3,0.0,0.0,5.2\n2026-07-11,12,31.1,0.0,0.0,7.9\n2026-07-11,13,31.1,0.0,0.0,10.9\n2026-07-11,14,31.3,0.0,0.0,9.8\n2026-07-11,15,31.6,0.0,0.0,8.7\n2026-07-11,16,31.3,0.0,0.0,8.4\n2026-07-11,17,31.1,0.0,0.0,5.6\n2026-07-11,18,30.8,0.0,0.0,4.3\n2026-07-11,19,29.8,0.0,0.0,5.0\n2026-07-11,20,28.0,0.0,0.0,3.8\n2026-07-11,21,27.7,0.0,0.0,8.0\n2026-07-11,22,27.2,0.0,0.0,8.5\n2026-07-11,23,26.6,0.0,0.0,5.8\n2026-07-12,0,26.0,0.0,0.0,3.6\n2026-07-12,1,25.7,0.0,0.0,2.5\n2026-07-12,2,25.8,0.0,0.0,1.2\n2026-07-12,3,25.4,0.0,0.0,6.7\n2026-07-12,4,25.2,0.0,0.0,5.2\n2026-07-12,5,25.1,0.0,0.0,4.7\n2026-07-12,6,25.4,0.0,0.0,5.1\n2026-07-12,7,27.4,0.0,0.0,5.6\n2026-07-12,8,29.6,0.0,0.0,6.5\n2026-07-12,9,30.6,0.0,0.0,8.3\n2026-07-12,10,31.5,0.0,0.0,11.9\n2026-07-12,11,32.2,0.0,0.0,13.3\n2026-07-12,12,32.8,0.0,0.0,13.9\n2026-07-12,13,33.5,0.0,0.0,16.0\n2026-07-12,14,33.7,0.0,0.0,13.8\n2026-07-12,15,34.0,0.0,0.0,11.3\n2026-07-12,16,33.7,0.0,0.0,11.3\n2026-07-12,17,33.3,0.1,0.0,10.6\n2026-07-12,18,32.2,0.0,0.0,8.6\n2026-07-12,19,30.6,0.0,0.0,7.8\n2026-07-12,20,29.2,0.0,0.0,4.6\n2026-07-12,21,29.1,0.0,0.0,2.0\n2026-07-12,22,28.4,0.0,0.0,3.1\n2026-07-12,23,27.9,0.0,0.0,3.9\n2026-07-13,0,27.5,0.0,0.0,8.1\n2026-07-13,1,27.4,0.0,0.0,11.9\n2026-07-13,2,26.8,0.0,0.0,10.8\n2026-07-13,3,26.5,0.0,0.0,9.0\n2026-07-13,4,26.1,0.0,0.0,8.7\n2026-07-13,5,25.9,0.0,0.0,8.8\n2026-07-13,6,26.0,0.0,0.0,8.9\n2026-07-13,7,27.2,0.0,0.0,9.6\n2026-07-13,8,28.6,0.0,0.0,12.1\n2026-07-13,9,29.7,0.1,0.0,11.3\n2026-07-13,10,30.7,0.1,0.0,12.7\n2026-07-13,11,31.3,0.2,0.0,16.2\n2026-07-13,12,31.4,0.4,0.0,18.4\n2026-07-13,13,31.7,0.4,0.0,18.0\n2026-07-13,14,31.4,0.2,0.0,17.3\n2026-07-13,15,32.3,0.2,0.0,10.2\n2026-07-13,16,32.5,0.2,0.0,11.0\n2026-07-13,17,32.6,0.0,0.0,11.4\n2026-07-13,18,32.3,0.0,0.0,10.4\n2026-07-13,19,31.0,0.2,0.0,7.2\n2026-07-13,20,29.1,0.0,0.0,7.4\n2026-07-13,21,29.0,0.3,0.0,3.1\n2026-07-13,22,27.7,0.3,0.0,5.0\n2026-07-13,23,27.4,0.3,0.0,1.7\n2026-07-14,0,26.2,0.7,0.0,9.4\n2026-07-14,1,26.3,0.2,0.0,12.2\n2026-07-14,2,26.5,0.0,0.0,14.5\n2026-07-14,3,26.8,0.0,0.0,13.2\n2026-07-14,4,26.6,0.0,0.0,14.5\n2026-07-14,5,26.1,0.0,0.0,14.8\n2026-07-14,6,26.2,0.1,0.0,15.5\n2026-07-14,7,26.9,0.0,0.0,17.1\n2026-07-14,8,28.3,0.0,0.0,22.9\n2026-07-14,9,29.7,0.0,0.0,26.1\n2026-07-14,10,30.6,0.0,0.0,26.5\n2026-07-14,11,31.5,0.0,0.0,28.6\n2026-07-14,12,32.3,0.0,0.0,28.8\n2026-07-14,13,32.3,0.1,0.0,30.6\n2026-07-14,14,32.0,0.0,0.0,31.7\n2026-07-14,15,31.6,0.0,0.0,32.0\n2026-07-14,16,30.9,0.4,0.0,29.7\n2026-07-14,17,28.6,1.3,0.0,31.2\n2026-07-14,18,26.6,1.9,0.0,29.7\n2026-07-14,19,26.0,1.2,0.0,34.2\n2026-07-14,20,25.5,4.0,0.0,32.9\n2026-07-14,21,25.8,5.2,0.0,31.2\n2026-07-14,22,25.5,16.0,0.0,34.4\n2026-07-14,23,25.1,10.5,0.0,32.4\n2026-07-15,0,24.8,1.0,0.0,32.1\n2026-07-15,1,24.9,2.0,0.0,29.2\n2026-07-15,2,24.7,1.1,0.0,26.0\n2026-07-15,3,24.8,0.0,0.0,23.1\n2026-07-15,4,24.6,0.4,0.0,20.6\n2026-07-15,5,24.5,0.1,0.0,19.2\n2026-07-15,6,24.6,0.0,0.0,18.6\n2026-07-15,7,24.7,0.0,0.0,17.3\n2026-07-15,8,24.8,0.1,0.0,16.9\n2026-07-15,9,24.6,0.2,0.0,20.5\n2026-07-15,10,24.2,0.2,0.0,19.1\n2026-07-15,11,24.9,0.0,0.0,19.9\n2026-07-15,12,24.9,0.0,0.0,19.6\n2026-07-15,13,26.4,0.0,0.0,22.8\n2026-07-15,14,25.7,0.2,0.0,22.2\n2026-07-15,15,25.9,0.0,0.0,17.7\n2026-07-15,16,26.2,0.0,0.0,17.9\n2026-07-15,17,26.0,0.0,0.0,15.9\n2026-07-15,18,25.4,0.0,0.0,13.9\n2026-07-15,19,24.4,0.0,0.0,11.8\n2026-07-15,20,23.7,0.0,0.0,8.9\n2026-07-15,21,23.8,0.0,0.0,5.2\n2026-07-15,22,23.5,0.0,0.0,6.4\n2026-07-15,23,23.1,0.0,0.0,8.0\n2026-07-16,0,23.0,0.0,0.0,7.7\n2026-07-16,1,22.8,0.0,0.0,6.4\n2026-07-16,2,22.8,0.0,0.0,6.1\n2026-07-16,3,22.7,0.0,0.0,1.7\n2026-07-16,4,22.5,0.0,0.0,1.8\n2026-07-16,5,22.3,0.0,0.0,4.8\n2026-07-16,6,22.5,0.0,0.0,3.6\n2026-07-16,7,23.6,0.0,0.0,4.3\n2026-07-16,8,25.0,0.0,0.0,2.4\n2026-07-16,9,26.0,0.0,0.0,1.9\n2026-07-16,10,27.4,0.0,0.0,2.7\n2026-07-16,11,28.4,0.0,0.0,5.1\n2026-07-16,12,28.0,0.2,0.0,10.2\n2026-07-16,13,28.3,0.1,0.0,13.0\n2026-07-16,14,28.7,0.0,0.0,7.8\n2026-07-16,15,28.4,0.1,0.0,9.5\n2026-07-16,16,28.0,0.0,0.0,10.7\n2026-07-16,17,27.4,0.0,0.0,10.8\n2026-07-16,18,26.8,0.0,0.0,8.5\n2026-07-16,19,26.2,0.0,0.0,5.6\n2026-07-16,20,25.5,0.0,0.0,5.1\n2026-07-16,21,24.9,0.0,0.0,6.8\n2026-07-16,22,24.8,0.0,0.0,7.7\n2026-07-16,23,24.6,0.0,0.0,7.0\n2026-07-17,0,24.0,0.0,0.0,3.5\n2026-07-17,1,23.9,0.0,0.0,0.6\n2026-07-17,2,23.6,0.0,0.0,2.2\n2026-07-17,3,23.3,0.0,0.0,4.2\n2026-07-17,4,22.9,0.0,0.0,1.3\n2026-07-17,5,22.8,0.0,0.0,0.2\n2026-07-17,6,23.1,0.0,0.0,3.2\n2026-07-17,7,24.2,0.0,0.0,4.2\n2026-07-17,8,25.4,0.0,0.0,4.6\n2026-07-17,9,27.1,0.0,0.0,4.9\n2026-07-17,10,28.9,0.0,0.0,4.0\n2026-07-17,11,29.9,0.2,0.0,4.4\n2026-07-17,12,30.3,0.0,0.0,5.7\n2026-07-17,13,29.7,0.0,0.0,5.4\n2026-07-17,14,30.2,0.0,0.0,5.8\n2026-07-17,15,31.1,0.0,0.0,2.7\n2026-07-17,16,30.4,0.0,0.0,1.5\n2026-07-17,17,29.7,0.0,0.0,3.9\n2026-07-17,18,28.4,0.0,0.0,4.9\n2026-07-17,19,27.4,1.0,0.0,5.3\n2026-07-17,20,26.4,1.8,0.0,4.8\n2026-07-17,21,25.7,0.5,0.0,6.4\n2026-07-17,22,25.5,3.7,0.0,6.3\n2026-07-17,23,24.9,2.5,0.0,10.0\n2026-07-18,0,24.7,4.2,0.0,8.9\n2026-07-18,1,24.6,9.1,0.0,9.9\n2026-07-18,2,24.5,5.9,0.0,11.7\n2026-07-18,3,24.6,4.3,0.0,22.2\n2026-07-18,4,24.0,4.0,0.0,22.2\n2026-07-18,5,24.7,7.3,0.0,22.5\n2026-07-18,6,24.5,8.5,0.0,13.9\n2026-07-18,7,24.2,6.8,0.0,14.6\n2026-07-18,8,22.8,5.3,0.0,3.0\n2026-07-18,9,23.1,3.0,0.0,6.4\n2026-07-18,10,24.1,0.6,0.0,7.2\n2026-07-18,11,25.7,0.2,0.0,9.9\n2026-07-18,12,26.3,0.0,0.0,14.2\n2026-07-18,13,27.1,0.0,0.0,10.6\n2026-07-18,14,27.9,0.0,0.0,10.1\n2026-07-18,15,28.1,0.0,0.0,8.6\n2026-07-18,16,27.6,0.0,0.0,10.7\n2026-07-18,17,26.6,0.0,0.0,10.2\n2026-07-18,18,25.7,0.0,0.0,9.4\n2026-07-18,19,24.6,0.0,0.0,8.6\n2026-07-18,20,23.9,0.0,0.0,7.0\n2026-07-18,21,24.0,0.0,0.0,5.1\n2026-07-18,22,23.4,0.0,0.0,5.9\n2026-07-18,23,23.0,0.0,0.0,7.7\n2026-07-19,0,22.8,0.0,0.0,6.7\n2026-07-19,1,22.6,0.0,0.0,6.3\n2026-07-19,2,22.3,0.0,0.0,5.4\n2026-07-19,3,22.2,0.0,0.0,3.3\n2026-07-19,4,22.0,0.0,0.0,3.9\n2026-07-19,5,22.2,0.0,0.0,2.8\n2026-07-19,6,22.5,0.0,0.0,3.0\n2026-07-19,7,22.8,0.0,0.0,3.2\n2026-07-19,8,23.8,0.0,0.0,5.0\n2026-07-19,9,24.8,0.1,0.0,4.3\n2026-07-19,10,26.2,0.0,0.0,6.8\n2026-07-19,11,26.2,0.0,0.0,8.5\n2026-07-19,12,25.9,0.1,0.0,11.2\n2026-07-19,13,26.1,0.1,0.0,9.7\n2026-07-19,14,26.3,0.0,0.0,9.2\n2026-07-19,15,26.6,0.0,0.0,5.7\n2026-07-19,16,26.8,0.0,0.0,5.5\n2026-07-19,17,26.4,0.0,0.0,6.3\n2026-07-19,18,25.4,0.0,0.0,9.5\n2026-07-19,19,24.6,0.0,0.0,7.7\n2026-07-19,20,24.1,0.0,0.0,3.5\n2026-07-19,21,24.0,0.0,0.0,5.0\n2026-07-19,22,23.7,0.0,0.0,3.2\n2026-07-19,23,23.4,0.0,0.0,2.7\n2026-07-20,0,23.6,0.0,0.0,1.0\n2026-07-20,1,23.2,0.2,0.0,2.9\n2026-07-20,2,23.1,0.1,0.0,3.1\n2026-07-20,3,23.3,0.0,0.0,1.9\n2026-07-20,4,23.3,0.0,0.0,0.6\n2026-07-20,5,23.2,0.3,0.0,4.8\n2026-07-20,6,23.1,0.8,0.0,3.6\n2026-07-20,7,23.3,1.4,0.0,3.8\n2026-07-20,8,23.5,1.7,0.0,5.3\n2026-07-20,9,23.5,1.0,0.0,5.3\n2026-07-20,10,23.9,2.7,0.0,2.8\n2026-07-20,11,23.9,7.2,0.0,4.4\n2026-07-20,12,23.7,1.9,0.0,8.3\n2026-07-20,13,25.2,0.0,0.0,8.9\n2026-07-20,14,26.5,0.0,0.0,8.7\n2026-07-20,15,24.7,0.0,0.0,12.1\n2026-07-20,16,26.4,0.1,0.0,11.5\n2026-07-20,17,27.6,0.0,0.0,11.1\n2026-07-20,18,27.5,0.1,0.0,11.4\n2026-07-20,19,26.9,0.3,0.0,13.6\n2026-07-20,20,25.4,0.5,0.0,16.8\n2026-07-20,21,25.0,1.3,0.0,9.2\n2026-07-20,22,25.2,0.9,0.0,14.8\n2026-07-20,23,25.6,0.7,0.0,16.2\n2026-07-21,0,25.6,0.0,0.0,18.1\n2026-07-21,1,25.4,0.3,0.0,19.4\n2026-07-21,2,25.5,0.0,0.0,18.3\n2026-07-21,3,25.4,0.0,0.0,17.6\n2026-07-21,4,25.4,0.0,0.0,18.7\n2026-07-21,5,25.5,0.0,0.0,17.8\n2026-07-21,6,24.7,1.9,0.0,19.0\n2026-07-21,7,24.4,4.4,0.0,17.5\n2026-07-21,8,24.7,3.5,0.0,24.2\n2026-07-21,9,26.3,1.7,0.0,23.6\n2026-07-21,10,26.6,4.3,0.0,27.2\n2026-07-21,11,25.6,5.4,0.0,26.8\n2026-07-21,12,25.0,3.1,0.0,24.5\n2026-07-21,13,24.8,2.8,0.0,20.6\n2026-07-21,14,24.7,1.5,0.0,17.0\n2026-07-21,15,24.8,0.5,0.0,11.3\n2026-07-21,16,25.0,0.6,0.0,12.2\n2026-07-21,17,24.8,0.1,0.0,10.4\n2026-07-21,18,25.0,0.0,0.0,11.1\n2026-07-21,19,24.9,0.0,0.0,10.5\n2026-07-21,20,24.5,0.1,0.0,9.4\n2026-07-21,21,24.5,0.0,0.0,5.0\n2026-07-21,22,24.0,0.0,0.0,5.6\n2026-07-21,23,23.9,0.0,0.0,5.4\n2026-07-22,0,24.1,0.0,0.0,4.7\n2026-07-22,1,24.0,0.0,0.0,3.6\n2026-07-22,2,23.9,0.0,0.0,1.8\n2026-07-22,3,23.3,0.1,0.0,3.7\n2026-07-22,4,23.1,0.0,0.0,4.0\n2026-07-22,5,23.0,0.0,0.0,1.5\n2026-07-22,6,23.4,0.0,0.0,1.5\n2026-07-22,7,24.4,0.0,0.0,3.6\n2026-07-22,8,25.0,0.1,0.0,4.9\n2026-07-22,9,24.3,0.0,0.0,4.9\n2026-07-22,10,25.2,1.0,0.0,3.6\n2026-07-22,11,27.1,0.6,0.0,3.6\n2026-07-22,12,28.5,0.9,0.0,6.7\n2026-07-22,13,28.5,0.2,0.0,8.5\n2026-07-22,14,28.5,0.2,0.0,8.8\n2026-07-22,15,24.9,0.5,0.0,6.5\n2026-07-22,16,24.4,1.2,0.0,4.0\n2026-07-22,17,26.5,0.3,0.0,2.2\n2026-07-22,18,27.3,0.1,0.0,4.9\n2026-07-22,19,26.7,0.2,0.0,6.7\n2026-07-22,20,25.9,0.1,0.0,7.3\n2026-07-22,21,25.5,0.1,0.0,10.0\n2026-07-22,22,25.2,1.5,0.0,9.5\n2026-07-22,23,25.1,0.7,0.0,12.3\n2026-07-23,0,24.2,5.6,0.0,12.7\n2026-07-23,1,23.9,8.3,0.0,11.9\n2026-07-23,2,23.6,2.6,0.0,9.4\n2026-07-23,3,24.2,2.9,0.0,12.1\n2026-07-23,4,24.4,2.2,0.0,9.7\n2026-07-23,5,24.4,1.9,0.0,8.7\n2026-07-23,6,24.8,0.4,0.0,12.0\n2026-07-23,7,24.3,4.3,0.0,11.4\n2026-07-23,8,24.4,8.2,0.0,5.5\n2026-07-23,9,25.5,1.4,0.0,12.5\n2026-07-23,10,25.9,1.4,0.0,15.3\n2026-07-23,11,27.1,0.7,0.0,15.0\n2026-07-23,12,28.5,0.3,0.0,16.6\n2026-07-23,13,25.8,5.7,0.0,18.7\n2026-07-23,14,24.9,6.1,0.0,9.0\n2026-07-23,15,26.9,1.1,0.0,9.4\n2026-07-23,16,27.2,1.8,0.0,9.6\n2026-07-23,17,27.1,0.3,0.0,9.4\n2026-07-23,18,27.2,0.2,0.0,10.5\n2026-07-23,19,26.7,0.1,0.0,11.7\n2026-07-23,20,26.2,0.1,0.0,11.6\n2026-07-23,21,25.9,0.1,0.0,7.9\n2026-07-23,22,25.8,0.3,0.0,10.3\n2026-07-23,23,26.1,0.0,0.0,11.3\n2026-07-24,0,25.6,0.0,0.0,10.9\n2026-07-24,1,25.6,0.0,0.0,11.0\n2026-07-24,2,25.6,0.1,0.0,12.4\n2026-07-24,3,25.3,0.2,0.0,11.5\n2026-07-24,4,25.2,0.1,0.0,11.4\n2026-07-24,5,25.2,0.0,0.0,11.2\n2026-07-24,6,25.0,0.0,0.0,10.4\n2026-07-24,7,25.8,0.2,0.0,13.2\n2026-07-24,8,26.3,0.1,0.0,13.7\n2026-07-24,9,27.2,0.1,0.0,14.0\n2026-07-24,10,28.8,0.2,0.0,16.8\n2026-07-24,11,29.1,0.3,0.0,16.8\n2026-07-24,12,29.6,0.3,0.0,16.7\n2026-07-24,13,29.9,0.1,0.0,15.7\n2026-07-24,14,30.1,0.1,0.0,13.2\n2026-07-24,15,30.6,0.0,0.0,11.4\n2026-07-24,16,30.0,0.1,0.0,9.6\n2026-07-24,17,29.4,0.1,0.0,9.7\n2026-07-24,18,28.7,0.1,0.0,11.1\n2026-07-24,19,27.4,0.0,0.0,8.1\n2026-07-24,20,26.5,0.0,0.0,6.9\n2026-07-24,21,26.7,0.0,0.0,4.1\n2026-07-24,22,26.2,0.0,0.0,6.3\n2026-07-24,23,25.8,0.0,0.0,4.8\n2026-07-25,0,25.6,0.0,0.0,2.8\n2026-07-25,1,25.6,0.0,0.0,6.5\n2026-07-25,2,25.6,0.0,0.0,7.9\n2026-07-25,3,25.3,0.0,0.0,6.3\n2026-07-25,4,25.4,0.1,0.0,6.5\n2026-07-25,5,25.6,0.1,0.0,7.2\n2026-07-25,6,25.6,0.1,0.0,9.4\n2026-07-25,7,25.9,0.1,0.0,10.8\n2026-07-25,8,26.4,0.2,0.0,12.5\n2026-07-25,9,27.9,0.2,0.0,13.1\n2026-07-25,10,28.6,0.4,0.0,14.4\n2026-07-25,11,28.9,0.2,0.0,16.0\n2026-07-25,12,29.5,0.3,0.0,17.0\n2026-07-25,13,29.1,0.1,0.0,16.8\n2026-07-25,14,29.1,0.1,0.0,16.7\n2026-07-25,15,29.8,0.1,0.0,12.6\n2026-07-25,16,29.4,0.2,0.0,12.6\n2026-07-25,17,28.6,0.1,0.0,13.2\n2026-07-25,18,27.7,0.1,0.0,12.5\n2026-07-25,19,27.0,0.0,0.0,9.0\n2026-07-25,20,26.3,0.0,0.0,6.5\n2026-07-25,21,26.6,0.0,0.0,7.7\n2026-07-25,22,26.0,0.2,0.0,6.5\n2026-07-25,23,26.0,0.4,0.0,6.0\n2026-07-26,0,26.0,0.2,0.0,7.5\n2026-07-26,1,25.9,0.1,0.0,7.4\n2026-07-26,2,25.4,0.2,0.0,8.3\n2026-07-26,3,24.9,0.1,0.0,5.2\n2026-07-26,4,25.0,0.1,0.0,6.9\n2026-07-26,5,25.1,0.2,0.0,7.6\n2026-07-26,6,25.2,0.1,0.0,8.3\n2026-07-26,7,25.4,0.2,0.0,5.7\n2026-07-26,8,26.1,0.5,0.0,7.4\n2026-07-26,9,26.4,0.2,0.0,8.8\n2026-07-26,10,27.4,1.1,0.0,5.5\n2026-07-26,11,27.3,1.4,0.0,6.7\n2026-07-26,12,28.9,0.5,0.0,9.2\n2026-07-26,13,28.0,2.5,0.0,11.1\n2026-07-26,14,29.3,0.3,0.0,10.8\n2026-07-26,15,29.9,3.0,0.0,8.3\n2026-07-26,16,28.9,0.4,0.0,9.8\n2026-07-26,17,28.3,0.2,0.0,8.0\n2026-07-26,18,27.6,0.3,0.0,9.9\n2026-07-26,19,27.4,0.0,0.0,5.9\n2026-07-26,20,26.6,0.0,0.0,3.4\n2026-07-26,21,25.6,0.0,0.0,5.8\n2026-07-26,22,25.4,0.1,0.0,6.3\n2026-07-26,23,25.3,0.0,0.0,6.3\n2026-07-27,0,25.2,0.0,0.0,8.7\n2026-07-27,1,25.2,0.0,0.0,9.2\n2026-07-27,2,25.0,0.0,0.0,8.6\n2026-07-27,3,24.9,0.1,0.0,3.5\n2026-07-27,4,24.7,0.8,0.0,10.3\n2026-07-27,5,24.1,0.4,0.0,9.2\n2026-07-27,6,24.0,0.0,0.0,2.7\n2026-07-27,7,24.6,0.0,0.0,4.0\n2026-07-27,8,25.4,0.0,0.0,6.5\n2026-07-27,9,26.5,0.0,0.0,4.5\n2026-07-27,10,25.3,1.2,0.0,5.3\n2026-07-27,11,25.4,1.3,0.0,4.3\n2026-07-27,12,27.8,0.2,0.0,4.4\n2026-07-27,13,28.2,0.9,0.0,11.0\n2026-07-27,14,28.1,0.7,0.0,13.5\n2026-07-27,15,29.6,0.1,0.0,14.7\n2026-07-27,16,28.7,0.5,0.0,13.0\n2026-07-27,17,27.2,2.4,0.0,10.9\n2026-07-27,18,27.1,0.7,0.0,9.4\n2026-07-27,19,26.6,0.5,0.0,12.2\n2026-07-27,20,26.0,0.4,0.0,8.8\n2026-07-27,21,26.1,0.1,0.0,6.7\n2026-07-27,22,25.9,0.0,0.0,6.7\n2026-07-27,23,25.7,0.0,0.0,6.3\n2026-07-28,0,26.0,0.2,0.0,9.5\n2026-07-28,1,26.2,0.1,0.0,12.0\n2026-07-28,2,26.2,0.1,0.0,12.6\n2026-07-28,3,25.9,0.2,0.0,11.3\n2026-07-28,4,25.7,0.1,0.0,9.3\n2026-07-28,5,25.5,0.0,0.0,6.7\n2026-07-28,6,25.5,0.0,0.0,8.4\n2026-07-28,7,25.8,0.1,0.0,8.1\n2026-07-28,8,26.8,0.1,0.0,9.0\n2026-07-28,9,27.7,0.1,0.0,9.0\n2026-07-28,10,29.0,0.0,0.0,8.9\n2026-07-28,11,29.6,0.1,0.0,8.9\n2026-07-28,12,30.0,0.0,0.0,11.2\n2026-07-28,13,29.8,0.0,0.0,13.7\n2026-07-28,14,29.6,0.0,0.0,14.2\n2026-07-28,15,29.1,0.0,0.0,14.4\n2026-07-28,16,28.9,0.1,0.0,14.5\n2026-07-28,17,28.2,0.0,0.0,12.0\n2026-07-28,18,27.6,0.0,0.0,11.6\n2026-07-28,19,27.0,0.0,0.0,11.9\n2026-07-28,20,26.6,0.0,0.0,9.8\n2026-07-28,21,26.9,0.0,0.0,9.3\n2026-07-28,22,26.4,0.0,0.0,9.0\n2026-07-28,23,26.3,0.0,0.0,9.7\n2026-07-29,0,26.0,0.0,0.0,10.4\n2026-07-29,1,26.0,0.0,0.0,9.6\n2026-07-29,2,26.0,0.0,0.0,9.4\n2026-07-29,3,25.9,0.0,0.0,8.8\n2026-07-29,4,25.6,0.0,0.0,8.7\n2026-07-29,5,25.6,0.0,0.0,8.0\n2026-07-29,6,25.5,0.0,0.0,7.5\n2026-07-29,7,26.1,0.1,0.0,8.9\n2026-07-29,8,27.3,0.1,0.0,11.9\n2026-07-29,9,28.2,0.1,0.0,12.9\n2026-07-29,10,28.9,0.1,0.0,13.3\n2026-07-29,11,29.3,0.0,0.0,12.5\n2026-07-29,12,29.8,0.0,0.0,12.6\n2026-07-29,13,29.6,0.0,0.0,13.7\n2026-07-29,14,29.5,0.0,0.0,15.0\n2026-07-29,15,29.4,0.1,0.0,14.7\n2026-07-29,16,28.8,0.0,0.0,14.9\n2026-07-29,17,28.3,0.0,0.0,14.2\n2026-07-29,18,27.9,0.0,0.0,14.9\n2026-07-29,19,27.3,0.0,0.0,14.9\n2026-07-29,20,26.7,0.0,0.0,11.0\n2026-07-29,21,26.5,0.0,0.0,10.0\n2026-07-29,22,26.2,0.0,0.0,10.4\n2026-07-29,23,26.0,0.0,0.0,10.3\n2026-07-30,0,25.7,0.0,0.0,9.0\n2026-07-30,1,25.4,0.0,0.0,7.8\n2026-07-30,2,25.2,0.0,0.0,6.9\n2026-07-30,3,25.2,0.0,0.0,5.5\n2026-07-30,4,24.8,0.0,0.0,5.1\n2026-07-30,5,24.7,0.0,0.0,5.6\n2026-07-30,6,24.7,0.0,0.0,5.9\n2026-07-30,7,25.6,0.1,0.0,8.5\n2026-07-30,8,26.9,0.1,0.0,10.3\n2026-07-30,9,27.9,0.1,0.0,9.0\n2026-07-30,10,28.6,0.0,0.0,10.0\n2026-07-30,11,29.0,0.0,0.0,11.7\n2026-07-30,12,29.3,0.0,0.0,12.6\n2026-07-30,13,29.6,0.0,0.0,12.5\n2026-07-30,14,29.8,0.0,0.0,13.0\n2026-07-30,15,29.9,0.0,0.0,11.4\n2026-07-30,16,29.5,0.0,0.0,11.0\n2026-07-30,17,28.9,0.0,0.0,10.0\n2026-07-30,18,28.1,0.0,0.0,9.9\n2026-07-30,19,27.2,0.0,0.0,9.1\n2026-07-30,20,26.3,0.0,0.0,6.9\n2026-07-30,21,26.2,0.0,0.0,4.2\n2026-07-30,22,25.7,0.0,0.0,2.3\n2026-07-30,23,25.5,0.0,0.0,2.4\n2026-07-31,0,25.3,0.0,0.0,5.2\n2026-07-31,1,25.3,0.0,0.0,6.5\n2026-07-31,2,25.2,0.0,0.0,6.8\n2026-07-31,3,25.1,0.0,0.0,3.8\n2026-07-31,4,24.7,0.0,0.0,2.1\n2026-07-31,5,24.6,0.0,0.0,2.2\n2026-07-31,6,24.9,0.0,0.0,4.1\n2026-07-31,7,26.1,0.0,0.0,6.8\n2026-07-31,8,27.5,0.0,0.0,9.6\n2026-07-31,9,28.9,0.1,0.0,9.7\n2026-07-31,10,29.6,0.1,0.0,11.3\n2026-07-31,11,30.2,0.1,0.0,12.9\n2026-07-31,12,30.5,0.1,0.0,13.3\n2026-07-31,13,30.6,0.2,0.0,14.2\n2026-07-31,14,30.8,0.2,0.0,16.0\n2026-07-31,15,30.9,0.0,0.0,15.0\n2026-07-31,16,30.6,0.2,0.0,14.9\n2026-07-31,17,30.1,0.0,0.0,13.4\n2026-07-31,18,29.5,0.0,0.0,12.3\n2026-07-31,19,28.6,0.0,0.0,10.7\n2026-07-31,20,27.8,0.0,0.0,11.0\n2026-07-31,21,27.9,0.0,0.0,9.7\n2026-07-31,22,27.7,0.0,0.0,10.9\n2026-07-31,23,27.5,0.0,0.0,10.6\n2026-08-01,0,27.2,0.0,0.0,10.5\n2026-08-01,1,26.9,0.0,0.0,10.7\n2026-08-01,2,26.7,0.0,0.0,10.8\n2026-08-01,3,26.8,0.0,0.0,9.2\n2026-08-01,4,26.6,0.0,0.0,7.8\n2026-08-01,5,26.5,0.0,0.0,8.3\n2026-08-01,6,26.4,0.0,0.0,8.4\n2026-08-01,7,27.5,0.0,0.0,9.1\n2026-08-01,8,28.4,0.1,0.0,10.6\n2026-08-01,9,29.0,0.2,0.0,11.3\n2026-08-01,10,29.8,0.2,0.0,13.8\n2026-08-01,11,30.8,0.1,0.0,15.3\n2026-08-01,12,31.5,0.2,0.0,15.6\n2026-08-01,13,31.7,0.1,0.0,15.8\n2026-08-01,14,31.8,0.0,0.0,15.9\n2026-08-01,15,31.7,0.0,0.0,14.5\n2026-08-01,16,31.4,0.0,0.0,14.0\n2026-08-01,17,30.7,0.0,0.0,13.0\n2026-08-01,18,29.8,0.0,0.0,12.1\n2026-08-01,19,28.9,0.0,0.0,10.1\n2026-08-01,20,28.0,0.0,0.0,9.2\n2026-08-01,21,28.1,0.0,0.0,8.3\n2026-08-01,22,27.7,0.0,0.0,8.6\n2026-08-01,23,27.4,0.0,0.0,6.9\n2026-08-02,0,27.2,0.0,0.0,5.9\n2026-08-02,1,27.0,0.0,0.0,6.5\n2026-08-02,2,26.8,0.0,0.0,6.5\n2026-08-02,3,26.4,0.0,0.0,5.6\n2026-08-02,4,26.1,0.0,0.0,5.8\n2026-08-02,5,26.0,0.0,0.0,6.3\n2026-08-02,6,26.0,0.0,0.0,6.0\n2026-08-02,7,27.0,0.0,0.0,6.6\n2026-08-02,8,28.4,0.0,0.0,7.3\n2026-08-02,9,29.6,0.0,0.0,3.5\n2026-08-02,10,30.8,0.0,0.0,2.8\n2026-08-02,11,32.0,0.0,0.0,3.9\n2026-08-02,12,32.7,0.0,0.0,7.4\n2026-08-02,13,32.8,0.0,0.0,9.3\n2026-08-02,14,32.7,0.0,0.0,11.1\n2026-08-02,15,32.9,0.0,0.0,12.2\n2026-08-02,16,32.2,0.0,0.0,12.5\n2026-08-02,17,31.6,0.0,0.0,11.3\n2026-08-02,18,30.8,0.0,0.0,9.9\n2026-08-02,19,29.7,0.0,0.0,7.8\n2026-08-02,20,28.5,0.0,0.0,5.7\n2026-08-02,21,28.3,0.0,0.0,6.0\n2026-08-02,22,27.6,0.0,0.0,4.6\n2026-08-02,23,27.3,0.0,0.0,4.6\n2026-08-03,0,27.0,0.0,0.0,5.2\n2026-08-03,1,26.7,0.0,0.0,4.7\n2026-08-03,2,26.4,0.0,0.0,3.3\n2026-08-03,3,26.1,0.0,0.0,4.5\n2026-08-03,4,25.7,0.0,0.0,3.2\n2026-08-03,5,25.5,0.0,0.0,2.7\n2026-08-03,6,25.7,0.0,0.0,0.7\n2026-08-03,7,26.8,0.0,0.0,0.9\n2026-08-03,8,28.4,0.0,0.0,2.1\n2026-08-03,9,30.1,0.0,0.0,4.6\n2026-08-03,10,31.5,0.0,0.0,5.2\n2026-08-03,11,33.0,0.0,0.0,5.1\n2026-08-03,12,34.0,0.0,0.0,4.1\n2026-08-03,13,35.0,0.0,0.0,3.4\n2026-08-03,14,35.5,0.0,0.0,6.2\n2026-08-03,15,34.6,0.0,0.0,10.4\n2026-08-03,16,34.0,0.2,0.0,12.6\n2026-08-03,17,33.6,0.0,0.0,13.8\n2026-08-03,18,32.8,0.0,0.0,13.0\n2026-08-03,19,31.6,0.0,0.0,11.2\n2026-08-03,20,30.6,0.0,0.0,7.9\n2026-08-03,21,30.5,0.0,0.0,6.9\n2026-08-03,22,29.7,0.0,0.0,6.5\n2026-08-03,23,29.2,0.0,0.0,6.6\n2026-08-04,0,28.7,0.0,0.0,6.2\n2026-08-04,1,28.5,0.0,0.0,6.3\n2026-08-04,2,28.2,0.0,0.0,5.2\n2026-08-04,3,28.3,0.0,0.0,3.0\n2026-08-04,4,28.3,0.0,0.0,0.7\n2026-08-04,5,28.1,0.1,0.0,3.9\n2026-08-04,6,28.3,0.0,0.0,4.9\n2026-08-04,7,29.3,0.0,0.0,5.1\n2026-08-04,8,30.6,0.0,0.0,6.2\n2026-08-04,9,31.9,0.0,0.0,4.6\n2026-08-04,10,33.4,0.1,0.0,2.9\n2026-08-04,11,34.4,0.2,0.0,3.3\n2026-08-04,12,35.0,0.3,0.0,2.4\n2026-08-04,13,35.8,0.6,0.0,2.2\n2026-08-04,14,35.7,0.2,0.0,5.6\n2026-08-04,15,34.5,0.0,0.0,8.0\n2026-08-04,16,35.3,0.1,0.0,7.6\n2026-08-04,17,32.3,0.4,0.0,9.4\n2026-08-04,18,30.5,2.1,0.0,3.4\n2026-08-04,19,31.5,0.0,0.0,4.3\n2026-08-04,20,31.0,0.0,0.0,2.7\n2026-08-04,21,31.7,0.0,0.0,0.7\n2026-08-04,22,30.8,0.0,0.0,3.8\n2026-08-04,23,30.1,0.0,0.0,1.9\n2026-08-05,0,29.5,0.1,0.0,7.3\n2026-08-05,1,28.7,0.0,0.0,7.3\n2026-08-05,2,28.7,0.1,0.0,3.0\n2026-08-05,3,28.0,0.0,0.0,5.3\n2026-08-05,4,28.1,0.0,0.0,3.6\n2026-08-05,5,28.0,0.0,0.0,3.3\n2026-08-05,6,28.0,0.0,0.0,3.3\n2026-08-05,7,28.8,0.0,0.0,5.2\n2026-08-05,8,30.1,0.0,0.0,6.0\n2026-08-05,9,32.2,0.0,0.0,6.8\n2026-08-05,10,33.6,0.0,0.0,5.4\n2026-08-05,11,34.7,0.0,0.0,3.0\n2026-08-05,12,35.9,0.2,0.0,3.7\n2026-08-05,13,35.9,0.2,0.0,4.9\n2026-08-05,14,36.2,0.1,0.0,7.3\n2026-08-05,15,36.0,0.1,0.0,8.7\n2026-08-05,16,35.7,0.4,0.0,9.7\n2026-08-05,17,32.4,1.6,0.0,11.2\n2026-08-05,18,32.2,1.0,0.0,13.0\n2026-08-05,19,32.0,0.0,0.0,9.2\n2026-08-05,20,31.6,0.0,0.0,6.3\n2026-08-05,21,31.8,0.0,0.0,7.0\n2026-08-05,22,31.2,0.0,0.0,5.1\n2026-08-05,23,30.9,0.0,0.0,4.3\n2026-08-06,0,30.8,0.0,0.0,7.0\n2026-08-06,1,30.6,0.0,0.0,5.4\n2026-08-06,2,30.1,0.0,0.0,5.1\n2026-08-06,3,29.6,0.1,0.0,3.5\n2026-08-06,4,29.4,0.0,0.0,3.9\n2026-08-06,5,29.1,0.0,0.0,7.7\n2026-08-06,6,28.5,0.0,0.0,8.8\n2026-08-06,7,29.0,0.0,0.0,8.5\n2026-08-06,8,30.3,0.0,0.0,9.2\n2026-08-06,9,32.2,0.0,0.0,8.8\n2026-08-06,10,33.4,0.2,0.0,6.9\n2026-08-06,11,34.6,0.2,0.0,6.7\n2026-08-06,12,35.8,0.1,0.0,6.3\n2026-08-06,13,36.6,0.1,0.0,5.2\n2026-08-06,14,37.4,0.0,0.0,3.1\n2026-08-06,15,37.3,0.1,0.0,5.9\n2026-08-06,16,34.8,0.0,0.0,14.4\n2026-08-06,17,34.0,0.1,0.0,15.5\n2026-08-06,18,33.3,0.2,0.0,12.4\n2026-08-06,19,32.4,0.0,0.0,10.2\n2026-08-06,20,31.3,0.0,0.0,8.9\n2026-08-06,21,31.3,0.0,0.0,7.7\n2026-08-06,22,30.7,0.0,0.0,4.5\n2026-08-06,23,30.4,0.0,0.0,4.1\n2026-08-07,0,30.1,0.0,0.0,5.2\n2026-08-07,1,29.7,0.0,0.0,3.1\n2026-08-07,2,29.5,0.0,0.0,2.7\n2026-08-07,3,29.2,0.0,0.0,3.0\n2026-08-07,4,28.5,0.0,0.0,3.3\n2026-08-07,5,28.3,0.0,0.0,6.0\n2026-08-07,6,28.0,0.0,0.0,5.9\n2026-08-07,7,29.1,0.0,0.0,6.4\n2026-08-07,8,30.8,0.0,0.0,6.8\n2026-08-07,9,32.9,0.0,0.0,6.5\n2026-08-07,10,34.4,0.0,0.0,5.0\n2026-08-07,11,35.6,0.0,0.0,4.3\n2026-08-07,12,36.9,0.0,0.0,3.1\n2026-08-07,13,38.0,0.0,0.0,1.1\n2026-08-07,14,37.4,0.3,0.0,7.8\n2026-08-07,15,36.8,0.6,0.0,10.7\n2026-08-07,16,36.8,0.0,0.0,9.9\n2026-08-07,17,36.4,0.0,0.0,12.7\n2026-08-07,18,35.5,0.0,0.0,15.5\n2026-08-07,19,34.1,0.0,0.0,16.2\n2026-08-07,20,32.9,0.0,0.0,13.2\n2026-08-07,21,32.9,0.0,0.0,10.4\n2026-08-07,22,32.2,0.0,0.0,8.3\n2026-08-07,23,31.7,0.0,0.0,6.5\n2026-08-08,0,31.2,0.0,0.0,7.1\n2026-08-08,1,30.8,0.0,0.0,7.4\n2026-08-08,2,30.6,0.0,0.0,8.3\n2026-08-08,3,30.8,0.0,0.0,7.8\n2026-08-08,4,30.8,0.0,0.0,13.4\n2026-08-08,5,30.1,0.0,0.0,14.6\n2026-08-08,6,29.3,0.0,0.0,11.5\n2026-08-08,7,29.5,0.0,0.0,12.5\n2026-08-08,8,30.7,0.0,0.0,13.2\n2026-08-08,9,33.5,0.0,0.0,12.8\n2026-08-08,10,34.7,0.0,0.0,10.4\n2026-08-08,11,35.0,0.1,0.0,14.1\n2026-08-08,12,31.5,1.7,0.0,9.4\n2026-08-08,13,33.3,0.4,0.0,11.8\n2026-08-08,14,34.4,0.2,0.0,15.6\n2026-08-08,15,32.3,1.3,0.0,10.1\n2026-08-08,16,31.9,0.3,0.0,18.8\n2026-08-08,17,28.7,3.1,0.0,9.1\n2026-08-08,18,29.0,0.9,0.0,9.5\n2026-08-08,19,28.5,0.3,0.0,15.7\n2026-08-08,20,27.1,0.0,0.0,15.6\n2026-08-08,21,28.4,0.0,0.0,17.4\n2026-08-08,22,27.1,0.0,0.0,17.2\n2026-08-08,23,26.4,0.0,0.0,17.6\n2026-08-09,0,25.5,0.0,0.0,17.6\n2026-08-09,1,24.8,0.0,0.0,15.7\n2026-08-09,2,24.4,0.0,0.0,14.5\n2026-08-09,3,24.4,0.0,0.0,13.7\n2026-08-09,4,24.0,0.0,0.0,12.6\n2026-08-09,5,24.2,0.0,0.0,12.0\n2026-08-09,6,24.4,0.0,0.0,12.2\n2026-08-09,7,24.9,0.0,0.0,13.0\n2026-08-09,8,26.2,0.0,0.0,16.6\n2026-08-09,9,28.1,0.0,0.0,15.9\n2026-08-09,10,29.1,0.0,0.0,18.1\n2026-08-09,11,30.9,0.0,0.0,15.2\n2026-08-09,12,31.8,0.0,0.0,15.7\n2026-08-09,13,32.3,0.0,0.0,17.0\n2026-08-09,14,31.3,0.1,0.0,17.1\n2026-08-09,15,33.1,0.1,0.0,17.2\n2026-08-09,16,31.9,0.3,0.0,17.8\n2026-08-09,17,32.0,0.1,0.0,17.4\n2026-08-09,18,31.5,0.0,0.0,17.8\n2026-08-09,19,30.1,0.0,0.0,17.3\n2026-08-09,20,28.6,0.0,0.0,16.5\n2026-08-09,21,27.8,0.0,0.0,16.9\n2026-08-09,22,26.8,0.0,0.0,16.3\n2026-08-09,23,26.0,0.0,0.0,15.8\n2026-08-10,0,25.5,0.0,0.0,15.2\n2026-08-10,1,25.2,0.0,0.0,13.7\n2026-08-10,2,24.9,0.0,0.0,13.4\n2026-08-10,3,25.1,0.0,0.0,12.7\n2026-08-10,4,24.3,0.2,0.0,14.8\n2026-08-10,5,24.0,0.1,0.0,15.3\n2026-08-10,6,22.9,0.7,0.0,16.3\n2026-08-10,7,22.5,2.4,0.0,16.3\n2026-08-10,8,23.4,1.0,0.0,11.6\n2026-08-10,9,28.6,0.0,0.0,15.1\n2026-08-10,10,30.0,0.0,0.0,13.1\n2026-08-10,11,31.7,0.0,0.0,12.1\n2026-08-10,12,33.0,0.0,0.0,11.5\n2026-08-10,13,34.2,0.0,0.0,11.6\n2026-08-10,14,34.8,0.0,0.0,11.2\n2026-08-10,15,34.2,0.0,0.0,13.2\n2026-08-10,16,33.8,0.0,0.0,13.8\n2026-08-10,17,33.4,0.0,0.0,12.5\n2026-08-10,18,33.1,0.0,0.0,13.2\n2026-08-10,19,32.4,0.0,0.0,13.9\n2026-08-10,20,31.4,0.0,0.0,15.1\n2026-08-10,21,30.1,0.0,0.0,14.3\n2026-08-10,22,29.7,0.0,0.0,12.5\n2026-08-10,23,29.3,0.0,0.0,12.6\n2026-08-11,0,28.4,0.0,0.0,13.5\n2026-08-11,1,27.5,0.0,0.0,12.4\n2026-08-11,2,26.8,0.0,0.0,11.4\n2026-08-11,3,26.4,0.0,0.0,11.4\n2026-08-11,4,25.7,0.0,0.0,12.1\n2026-08-11,5,25.0,0.0,0.0,12.5\n2026-08-11,6,24.4,0.0,0.0,11.8\n2026-08-11,7,24.4,0.0,0.0,10.7\n2026-08-11,8,25.2,0.0,0.0,11.2\n2026-08-11,9,27.1,0.0,0.0,11.4\n2026-08-11,10,28.7,0.0,0.0,11.3\n2026-08-11,11,30.2,0.0,0.0,10.3\n2026-08-11,12,31.7,0.0,0.0,9.2\n2026-08-11,13,33.1,0.0,0.0,9.9\n2026-08-11,14,34.0,0.0,0.0,11.4\n2026-08-11,15,34.0,0.0,0.0,9.1\n2026-08-11,16,33.3,0.0,0.0,4.1\n2026-08-11,17,33.1,0.0,0.0,5.2\n2026-08-11,18,32.5,0.0,0.0,9.3\n2026-08-11,19,30.8,0.0,0.0,19.1\n2026-08-11,20,28.6,0.0,0.0,19.3\n2026-08-11,21,28.1,0.0,0.0,18.4\n2026-08-11,22,26.7,0.0,0.0,18.6\n2026-08-11,23,25.9,0.0,0.0,19.4\n2026-08-12,0,24.8,0.0,0.0,17.4\n2026-08-12,1,23.8,0.0,0.0,14.9\n2026-08-12,2,23.0,0.0,0.0,12.1\n2026-08-12,3,22.8,0.0,0.0,10.9\n2026-08-12,4,22.1,0.0,0.0,11.2\n2026-08-12,5,21.6,0.0,0.0,10.5\n2026-08-12,6,21.4,0.0,0.0,9.5\n2026-08-12,7,22.3,0.0,0.0,9.8\n2026-08-12,8,23.6,0.0,0.0,13.0\n2026-08-12,9,25.6,0.0,0.0,14.0\n2026-08-12,10,26.9,0.0,0.0,13.7\n2026-08-12,11,28.4,0.0,0.0,13.4\n2026-08-12,12,29.5,0.0,0.0,13.6\n2026-08-12,13,30.6,0.0,0.0,11.8\n2026-08-12,14,31.9,0.0,0.0,8.7\n2026-08-12,15,32.9,0.0,0.0,13.5\n2026-08-12,16,31.4,0.0,0.0,15.6\n2026-08-12,17,31.0,0.0,0.0,15.1\n2026-08-12,18,30.6,0.0,0.0,15.5\n2026-08-12,19,29.6,0.0,0.0,15.6\n2026-08-12,20,27.5,0.0,0.0,18.4\n2026-08-12,21,27.4,0.0,0.0,16.3\n2026-08-12,22,26.3,0.0,0.0,15.5\n2026-08-12,23,25.9,0.0,0.0,12.8\n2026-08-13,0,25.5,0.0,0.0,11.0\n2026-08-13,1,25.0,0.0,0.0,8.7\n2026-08-13,2,24.2,0.0,0.0,8.4\n2026-08-13,3,23.5,0.0,0.0,10.1\n2026-08-13,4,23.0,0.0,0.0,10.4\n2026-08-13,5,22.8,0.0,0.0,9.2\n2026-08-13,6,22.8,0.0,0.0,8.7\n2026-08-13,7,23.6,0.0,0.0,9.3\n2026-08-13,8,25.3,0.0,0.0,9.0\n2026-08-13,9,26.8,0.0,0.0,10.5\n2026-08-13,10,28.0,0.0,0.0,11.1\n2026-08-13,11,29.1,0.0,0.0,11.6\n2026-08-13,12,30.0,0.1,0.0,10.1\n2026-08-13,13,31.3,0.2,0.0,10.2\n2026-08-13,14,31.7,0.1,0.0,11.3\n2026-08-13,15,31.0,0.0,0.0,6.3\n2026-08-13,16,31.1,0.0,0.0,4.1\n2026-08-13,17,30.8,0.0,0.0,8.6\n2026-08-13,18,30.0,0.0,0.0,10.9\n2026-08-13,19,29.6,0.0,0.0,9.7\n2026-08-13,20,29.0,0.0,0.0,7.7\n2026-08-13,21,28.9,0.0,0.0,7.7\n2026-08-13,22,27.7,0.0,0.0,4.9\n2026-08-13,23,27.1,0.0,0.0,5.5\n2026-08-14,0,26.4,0.0,0.0,5.5\n2026-08-14,1,26.4,0.0,0.0,4.8\n2026-08-14,2,26.3,0.0,0.0,4.6\n2026-08-14,3,26.1,0.0,0.0,4.1\n2026-08-14,4,25.2,0.0,0.0,4.9\n2026-08-14,5,24.7,0.0,0.0,4.5\n2026-08-14,6,24.4,0.0,0.0,4.3\n2026-08-14,7,25.5,0.0,0.0,4.8\n2026-08-14,8,27.0,0.0,0.0,5.1\n2026-08-14,9,28.7,0.0,0.0,2.9\n2026-08-14,10,29.8,0.1,0.0,2.5\n2026-08-14,11,30.7,0.9,0.0,2.7\n2026-08-14,12,30.7,0.2,0.0,2.3\n2026-08-14,13,28.8,1.0,0.0,8.7\n2026-08-14,14,29.0,0.8,0.0,9.2\n2026-08-14,15,29.3,0.6,0.0,5.2\n2026-08-14,16,29.7,0.1,0.0,6.7\n2026-08-14,17,29.7,0.0,0.0,5.5\n2026-08-14,18,29.1,0.0,0.0,4.3\n2026-08-14,19,28.7,0.0,0.0,2.6\n2026-08-14,20,27.9,0.0,0.0,4.6\n2026-08-14,21,27.7,0.0,0.0,3.8\n2026-08-14,22,27.4,0.0,0.0,5.3\n2026-08-14,23,27.1,0.1,0.0,5.3\n2026-08-15,0,27.0,0.0,0.0,4.7\n2026-08-15,1,26.7,0.0,0.0,3.3\n2026-08-15,2,26.3,0.0,0.0,2.6\n2026-08-15,3,26.2,0.0,0.0,9.4\n2026-08-15,4,25.8,0.0,0.0,6.4\n2026-08-15,5,25.3,0.1,0.0,6.6\n2026-08-15,6,25.4,0.0,0.0,5.4\n2026-08-15,7,24.4,0.4,0.0,7.0\n2026-08-15,8,25.0,0.5,0.0,4.4\n2026-08-15,9,25.0,0.2,0.0,8.7\n2026-08-15,10,24.0,1.7,0.0,9.2\n2026-08-15,11,24.6,1.2,0.0,9.4\n2026-08-15,12,25.8,0.6,0.0,10.7\n2026-08-15,13,26.8,0.3,0.0,6.4\n2026-08-15,14,28.4,0.0,0.0,5.3\n2026-08-15,15,25.7,0.0,0.0,2.8\n2026-08-15,16,26.4,0.5,0.0,8.2\n2026-08-15,17,26.4,0.3,0.0,4.8\n2026-08-15,18,24.6,1.8,0.0,3.2\n2026-08-15,19,24.4,0.2,0.0,4.6\n2026-08-15,20,24.3,0.0,0.0,4.5\n2026-08-15,21,24.9,0.0,0.0,4.6\n2026-08-15,22,24.7,0.2,0.0,6.5\n2026-08-15,23,24.8,0.0,0.0,6.3\n2026-08-16,0,24.9,0.2,0.0,6.4\n2026-08-16,1,24.5,0.2,0.0,8.3\n2026-08-16,2,23.0,2.2,0.0,12.0\n2026-08-16,3,23.2,2.3,0.0,4.4\n2026-08-16,4,23.1,0.7,0.0,4.6\n2026-08-16,5,23.2,1.8,0.0,5.5\n2026-08-16,6,23.4,0.5,0.0,5.7\n2026-08-16,7,24.0,0.0,0.0,9.2\n2026-08-16,8,24.0,0.0,0.0,8.7\n2026-08-16,9,23.8,0.0,0.0,10.1\n2026-08-16,10,24.9,0.9,0.0,10.3\n2026-08-16,11,26.0,0.4,0.0,13.0\n2026-08-16,12,25.5,0.3,0.0,11.3\n2026-08-16,13,26.7,0.0,0.0,13.6\n2026-08-16,14,27.5,0.0,0.0,13.4\n2026-08-16,15,26.0,0.1,0.0,11.4\n2026-08-16,16,27.6,0.1,0.0,13.4\n2026-08-16,17,27.3,0.0,0.0,11.3\n2026-08-16,18,27.3,0.0,0.0,10.5\n2026-08-16,19,26.6,0.0,0.0,11.1\n2026-08-16,20,25.7,0.0,0.0,10.1\n2026-08-16,21,25.1,0.0,0.0,10.1\n2026-08-16,22,24.5,0.5,0.0,8.0\n2026-08-16,23,24.4,0.4,0.0,9.0\n2026-08-17,0,24.0,0.1,0.0,8.6\n2026-08-17,1,23.8,0.0,0.0,8.4\n2026-08-17,2,23.6,0.2,0.0,9.1\n2026-08-17,3,24.0,0.2,0.0,7.0\n2026-08-17,4,23.8,0.1,0.0,6.2\n2026-08-17,5,23.4,0.2,0.0,9.1\n2026-08-17,6,23.4,0.0,0.0,4.9\n2026-08-17,7,23.8,0.1,0.0,6.6\n2026-08-17,8,24.9,0.0,0.0,7.1\n2026-08-17,9,24.8,0.0,0.0,2.9\n2026-08-17,10,25.0,1.2,0.0,2.9\n2026-08-17,11,26.6,0.2,0.0,5.2\n2026-08-17,12,28.0,0.1,0.0,4.0\n2026-08-17,13,27.4,1.7,0.0,9.1\n2026-08-17,14,26.2,4.6,0.0,3.2\n2026-08-17,15,26.4,0.1,0.0,6.2\n2026-08-17,16,27.9,0.1,0.0,2.3\n2026-08-17,17,26.3,0.4,0.0,7.4\n2026-08-17,18,24.6,1.3,0.0,2.4\n2026-08-17,19,24.9,0.1,0.0,1.2\n2026-08-17,20,24.5,0.0,0.0,2.8\n2026-08-17,21,25.8,0.0,0.0,8.2\n2026-08-17,22,25.0,0.0,0.0,6.9\n2026-08-17,23,24.7,0.2,0.0,5.4\n2026-08-18,0,24.7,0.0,0.0,1.8\n2026-08-18,1,24.3,0.0,0.0,1.5\n2026-08-18,2,24.3,0.0,0.0,5.6\n2026-08-18,3,23.9,0.0,0.0,2.8\n2026-08-18,4,23.9,0.0,0.0,1.8\n2026-08-18,5,23.7,0.1,0.0,3.5\n2026-08-18,6,23.6,0.0,0.0,3.5\n2026-08-18,7,24.8,0.0,0.0,4.4\n2026-08-18,8,25.9,0.0,0.0,7.0\n2026-08-18,9,27.9,0.0,0.0,5.6\n2026-08-18,10,28.9,0.0,0.0,6.0\n2026-08-18,11,29.7,0.0,0.0,6.7\n2026-08-18,12,30.1,0.0,0.0,9.6\n2026-08-18,13,30.2,0.0,0.0,12.6\n2026-08-18,14,30.3,0.0,0.0,12.4\n2026-08-18,15,31.3,0.0,0.0,9.7\n2026-08-18,16,30.8,0.0,0.0,11.3\n2026-08-18,17,30.2,0.0,0.0,11.5\n2026-08-18,18,28.8,0.0,0.0,11.7\n2026-08-18,19,27.4,0.0,0.0,10.6\n2026-08-18,20,26.6,0.0,0.0,8.3\n2026-08-18,21,26.8,0.0,0.0,6.7\n2026-08-18,22,26.3,0.0,0.0,5.6\n2026-08-18,23,26.0,0.0,0.0,5.5\n2026-08-19,0,25.7,0.0,0.0,4.4\n2026-08-19,1,25.2,0.0,0.0,5.8\n2026-08-19,2,25.0,0.0,0.0,6.2\n2026-08-19,3,25.0,0.0,0.0,4.0\n2026-08-19,4,24.4,0.4,0.0,6.7\n2026-08-19,5,24.3,0.2,0.0,6.8\n2026-08-19,6,24.0,0.0,0.0,4.9\n2026-08-19,7,24.7,0.0,0.0,5.1\n2026-08-19,8,25.5,0.0,0.0,6.7\n2026-08-19,9,27.0,0.0,0.0,2.8\n2026-08-19,10,28.7,0.2,0.0,4.6\n2026-08-19,11,29.4,0.1,0.0,6.1\n2026-08-19,12,30.8,0.0,0.0,6.5\n2026-08-19,13,30.3,0.0,0.0,7.6\n2026-08-19,14,30.1,0.0,0.0,7.4\n2026-08-19,15,31.0,0.0,0.0,7.3\n2026-08-19,16,31.5,0.0,0.0,6.8\n2026-08-19,17,30.8,0.0,0.0,9.0\n2026-08-19,18,30.0,0.0,0.0,7.4\n2026-08-19,19,28.9,0.0,0.0,8.1\n2026-08-19,20,27.7,0.0,0.0,8.3\n2026-08-19,21,27.5,0.0,0.0,7.8\n2026-08-19,22,26.6,0.0,0.0,6.9\n2026-08-19,23,26.0,0.0,0.0,4.5\n2026-08-20,0,25.9,0.0,0.0,5.3\n2026-08-20,1,25.8,0.0,0.0,2.8\n2026-08-20,2,25.6,0.0,0.0,2.8\n2026-08-20,3,26.1,0.0,0.0,1.8\n2026-08-20,4,25.6,0.0,0.0,5.1\n2026-08-20,5,25.2,0.0,0.0,4.0\n2026-08-20,6,25.1,0.0,0.0,3.5\n2026-08-20,7,25.6,0.0,0.0,1.7\n2026-08-20,8,26.6,0.0,0.0,4.1\n2026-08-20,9,28.9,0.0,0.0,5.3\n2026-08-20,10,29.3,0.0,0.0,6.9\n2026-08-20,11,30.0,0.0,0.0,8.4\n2026-08-20,12,30.2,0.0,0.0,8.4\n2026-08-20,13,27.3,0.8,0.0,8.5\n2026-08-20,14,26.1,1.8,0.0,5.2\n2026-08-20,15,27.9,0.9,0.0,5.6\n2026-08-20,16,28.7,0.0,0.0,3.3\n2026-08-20,17,28.5,0.0,0.0,1.5\n2026-08-20,18,28.9,0.0,0.0,1.3\n2026-08-20,19,27.9,0.2,0.0,0.9\n2026-08-20,20,26.9,0.2,0.0,1.1\n2026-08-20,21,26.0,2.5,0.0,7.7\n2026-08-20,22,25.4,0.0,0.0,6.0\n2026-08-20,23,25.3,0.0,0.0,3.1\n2026-08-21,0,25.2,0.0,0.0,2.9\n2026-08-21,1,25.0,0.0,0.0,4.6\n2026-08-21,2,24.7,0.0,0.0,3.2\n2026-08-21,3,25.0,1.1,0.0,10.8\n2026-08-21,4,23.9,0.1,0.0,12.1\n2026-08-21,5,22.9,0.8,0.0,8.9\n2026-08-21,6,23.3,0.1,0.0,7.7\n2026-08-21,7,23.5,0.8,0.0,5.9\n2026-08-21,8,23.9,1.4,0.0,10.4\n2026-08-21,9,24.9,4.0,0.0,5.8\n2026-08-21,10,23.9,4.4,0.0,4.6\n2026-08-21,11,23.8,3.9,0.0,9.6\n2026-08-21,12,24.1,4.5,0.0,9.6\n2026-08-21,13,24.2,3.0,0.0,10.3\n2026-08-21,14,25.3,1.3,0.0,9.3\n2026-08-21,15,27.2,1.8,0.0,6.2\n2026-08-21,16,27.7,0.5,0.0,6.9\n2026-08-21,17,26.6,0.4,0.0,4.2\n2026-08-21,18,26.4,0.3,0.0,2.0\n2026-08-21,19,25.4,1.0,0.0,4.1\n2026-08-21,20,24.6,0.5,0.0,10.0\n2026-08-21,21,24.6,0.0,0.0,7.1\n2026-08-21,22,24.5,0.0,0.0,6.7\n2026-08-21,23,24.2,0.1,0.0,16.9\n2026-08-22,0,23.2,0.5,0.0,8.5\n2026-08-22,1,23.3,0.1,0.0,2.7\n2026-08-22,2,22.8,4.2,0.0,3.1\n2026-08-22,3,23.4,1.5,0.0,5.7\n2026-08-22,4,23.5,10.6,0.0,3.5\n2026-08-22,5,23.3,2.8,0.0,10.4\n2026-08-22,6,23.3,0.1,0.0,12.0\n2026-08-22,7,23.5,1.9,0.0,8.0\n2026-08-22,8,23.5,1.4,0.0,11.0\n2026-08-22,9,24.7,0.5,0.0,11.2\n2026-08-22,10,25.8,0.6,0.0,11.1\n2026-08-22,11,27.4,0.2,0.0,9.4\n2026-08-22,12,28.7,0.1,0.0,8.4\n2026-08-22,13,29.1,0.4,0.0,3.6\n2026-08-22,14,26.7,3.0,0.0,4.4\n2026-08-22,15,24.2,3.6,0.0,1.6\n2026-08-22,16,23.8,5.5,0.0,3.6\n2026-08-22,17,23.7,4.7,0.0,7.9\n2026-08-22,18,23.7,3.9,0.0,8.7\n2026-08-22,19,23.6,2.1,0.0,7.4\n2026-08-22,20,23.4,0.1,0.0,10.5\n2026-08-22,21,23.4,0.0,0.0,8.9\n2026-08-22,22,23.3,0.0,0.0,3.3\n2026-08-22,23,23.5,0.6,0.0,3.5\n2026-08-23,0,23.1,0.9,0.0,7.2\n2026-08-23,1,22.7,0.0,0.0,7.2\n2026-08-23,2,22.6,0.0,0.0,8.9\n2026-08-23,3,22.8,0.0,0.0,9.9\n2026-08-23,4,22.7,1.7,0.0,10.1\n2026-08-23,5,22.9,2.1,0.0,13.0\n2026-08-23,6,22.8,0.0,0.0,9.5\n2026-08-23,7,23.6,0.0,0.0,8.7\n2026-08-23,8,25.0,0.0,0.0,6.5\n2026-08-23,9,26.9,0.0,0.0,3.9\n2026-08-23,10,28.7,0.0,0.0,4.0\n2026-08-23,11,29.9,0.0,0.0,7.3\n2026-08-23,12,30.5,0.0,0.0,7.6\n2026-08-23,13,30.9,0.0,0.0,7.3\n2026-08-23,14,31.2,0.0,0.0,8.1\n2026-08-23,15,32.1,0.0,0.0,6.3\n2026-08-23,16,31.7,0.0,0.0,8.3\n2026-08-23,17,30.4,0.0,0.0,10.5\n2026-08-23,18,28.8,0.0,0.0,10.3\n2026-08-23,19,27.7,0.0,0.0,6.2\n2026-08-23,20,26.7,0.0,0.0,2.9\n2026-08-23,21,26.9,0.0,0.0,4.1\n2026-08-23,22,26.3,0.0,0.0,3.0\n2026-08-23,23,26.0,0.0,0.0,2.4\n2026-08-24,0,25.8,0.0,0.0,1.8\n2026-08-24,1,25.5,0.0,0.0,1.8\n2026-08-24,2,25.2,0.0,0.0,3.2\n2026-08-24,3,25.2,0.0,0.0,0.8\n2026-08-24,4,24.8,0.0,0.0,2.2\n2026-08-24,5,24.7,0.0,0.0,4.1\n2026-08-24,6,24.4,0.0,0.0,2.0\n2026-08-24,7,25.3,0.0,0.0,3.9\n2026-08-24,8,27.1,0.0,0.0,3.5\n2026-08-24,9,28.5,0.0,0.0,3.2\n2026-08-24,10,29.8,0.0,0.0,3.9\n2026-08-24,11,30.4,0.2,0.0,7.5\n2026-08-24,12,30.5,0.1,0.0,10.3\n2026-08-24,13,30.5,0.0,0.0,11.1\n2026-08-24,14,30.4,0.1,0.0,12.0\n2026-08-24,15,30.4,0.0,0.0,12.4\n2026-08-24,16,29.6,0.0,0.0,13.2\n2026-08-24,17,28.9,0.0,0.0,12.7\n2026-08-24,18,28.2,0.0,0.0,11.8\n2026-08-24,19,27.3,0.0,0.0,8.6\n2026-08-24,20,26.8,0.0,0.0,6.9\n2026-08-24,21,26.5,0.0,0.0,12.8\n2026-08-24,22,25.9,0.2,0.0,7.7\n2026-08-24,23,25.4,0.0,0.0,6.5\n2026-08-25,0,25.3,0.0,0.0,5.9\n2026-08-25,1,25.4,0.0,0.0,5.0\n2026-08-25,2,25.2,0.6,0.0,9.2\n2026-08-25,3,25.5,0.5,0.0,8.6\n2026-08-25,4,25.3,0.1,0.0,7.2\n2026-08-25,5,25.2,0.1,0.0,10.0\n2026-08-25,6,25.1,0.2,0.0,12.3\n2026-08-25,7,25.0,0.9,0.0,8.5\n2026-08-25,8,26.0,0.0,0.0,4.1\n2026-08-25,9,26.9,0.0,0.0,7.2\n2026-08-25,10,28.1,0.3,0.0,8.8\n2026-08-25,11,29.0,0.1,0.0,8.0\n2026-08-25,12,29.5,0.3,0.0,8.7\n2026-08-25,13,30.1,0.0,0.0,9.6\n2026-08-25,14,29.9,0.0,0.0,10.7\n2026-08-25,15,29.8,0.1,0.0,7.3\n2026-08-25,16,29.2,2.2,0.0,9.5\n2026-08-25,17,28.3,1.0,0.0,7.2\n2026-08-25,18,28.3,0.0,0.0,6.7\n2026-08-25,19,27.5,0.0,0.0,6.0\n2026-08-25,20,26.8,0.0,0.0,4.0\n2026-08-25,21,26.8,0.0,0.0,1.5\n2026-08-25,22,26.5,0.0,0.0,3.0\n2026-08-25,23,26.4,0.0,0.0,1.7\n2026-08-26,0,26.2,0.0,0.0,1.8\n2026-08-26,1,25.9,0.0,0.0,1.9\n2026-08-26,2,25.6,0.1,0.0,2.5\n2026-08-26,3,25.6,0.0,0.0,2.6\n2026-08-26,4,25.2,0.0,0.0,3.4\n2026-08-26,5,24.9,0.0,0.0,1.9\n2026-08-26,6,24.7,0.0,0.0,4.2\n2026-08-26,7,25.4,0.0,0.0,2.3\n2026-08-26,8,26.5,0.0,0.0,3.2\n2026-08-26,9,27.4,0.0,0.0,3.3\n2026-08-26,10,28.1,0.0,0.0,5.4\n2026-08-26,11,29.4,0.0,0.0,4.5\n2026-08-26,12,29.3,0.0,0.0,4.9\n2026-08-26,13,30.1,0.0,0.0,7.6\n2026-08-26,14,29.3,0.0,0.0,8.3\n2026-08-26,15,29.9,0.0,0.0,6.7\n2026-08-26,16,29.4,0.0,0.0,6.2\n2026-08-26,17,29.3,0.0,0.0,6.4\n2026-08-26,18,28.9,0.0,0.0,4.4\n2026-08-26,19,27.5,0.0,0.0,4.7\n2026-08-26,20,26.7,0.0,0.0,6.7\n2026-08-26,21,26.7,0.0,0.0,5.7\n2026-08-26,22,26.3,0.0,0.0,6.8\n2026-08-26,23,25.9,0.0,0.0,5.8\n2026-08-27,0,25.5,0.0,0.0,4.4\n2026-08-27,1,25.2,0.0,0.0,4.1\n2026-08-27,2,24.7,0.0,0.0,5.6\n2026-08-27,3,24.3,0.0,0.0,8.1\n2026-08-27,4,24.4,0.0,0.0,6.1\n2026-08-27,5,24.3,0.0,0.0,2.7\n2026-08-27,6,24.3,0.0,0.0,3.9\n2026-08-27,7,24.4,0.0,0.0,5.8\n2026-08-27,8,24.9,0.0,0.0,4.2\n2026-08-27,9,26.9,0.0,0.0,2.9\n2026-08-27,10,28.6,0.0,0.0,5.1\n2026-08-27,11,29.2,0.0,0.0,3.8\n2026-08-27,12,30.3,0.0,0.0,5.4\n2026-08-27,13,30.7,0.0,0.0,5.9\n2026-08-27,14,30.1,0.3,0.0,8.9\n2026-08-27,15,29.6,0.3,0.0,6.7\n2026-08-27,16,29.0,0.8,0.0,7.5\n2026-08-27,17,28.1,0.4,0.0,10.3\n2026-08-27,18,27.9,0.0,0.0,5.3\n2026-08-27,19,27.3,0.0,0.0,4.5\n2026-08-27,20,26.6,0.0,0.0,2.8\n2026-08-27,21,26.9,0.0,0.0,4.5\n2026-08-27,22,26.4,0.0,0.0,4.4\n2026-08-27,23,26.2,0.0,0.0,3.6\n2026-08-28,0,26.2,0.0,0.0,8.4\n2026-08-28,1,25.6,0.0,0.0,5.3\n2026-08-28,2,25.3,0.0,0.0,6.1\n2026-08-28,3,25.6,0.0,0.0,3.9\n2026-08-28,4,25.1,0.0,0.0,8.0\n2026-08-28,5,24.9,0.0,0.0,8.8\n2026-08-28,6,24.8,0.0,0.0,8.1\n2026-08-28,7,25.5,0.0,0.0,8.9\n2026-08-28,8,26.1,0.0,0.0,9.7\n2026-08-28,9,27.7,0.0,0.0,8.0\n2026-08-28,10,29.0,0.0,0.0,8.7\n2026-08-28,11,28.1,0.1,0.0,8.8\n2026-08-28,12,28.7,1.5,0.0,8.8\n2026-08-28,13,28.8,0.2,0.0,5.8\n2026-08-28,14,30.2,0.1,0.0,4.5\n2026-08-28,15,30.0,0.0,0.0,4.1\n2026-08-28,16,28.4,0.4,0.0,6.3\n2026-08-28,17,27.0,0.2,0.0,8.1\n2026-08-28,18,26.3,0.8,0.0,3.0\n2026-08-28,19,25.8,0.4,0.0,6.6\n2026-08-28,20,25.4,0.0,0.0,6.2\n2026-08-28,21,25.9,0.0,0.0,7.2\n2026-08-28,22,25.6,0.0,0.0,4.0\n2026-08-28,23,25.3,0.0,0.0,4.1\n2026-08-29,0,24.9,0.7,0.0,2.2\n2026-08-29,1,24.8,0.1,0.0,3.9\n2026-08-29,2,24.4,0.0,0.0,4.1\n2026-08-29,3,24.7,0.0,0.0,2.2\n2026-08-29,4,25.0,0.0,0.0,0.4\n2026-08-29,5,24.9,0.0,0.0,1.3\n2026-08-29,6,24.5,0.0,0.0,2.3\n2026-08-29,7,25.1,0.0,0.0,4.0\n2026-08-29,8,25.9,0.1,0.0,7.2\n2026-08-29,9,26.8,0.0,0.0,10.0\n2026-08-29,10,27.5,0.0,0.0,8.7\n2026-08-29,11,27.7,0.0,0.0,9.3\n2026-08-29,12,28.5,0.1,0.0,10.4\n2026-08-29,13,28.8,0.0,0.0,11.8\n2026-08-29,14,28.9,0.0,0.0,12.2\n2026-08-29,15,29.9,0.0,0.0,12.2\n2026-08-29,16,29.5,0.0,0.0,11.6\n2026-08-29,17,28.8,0.0,0.0,11.6\n2026-08-29,18,27.8,0.0,0.0,10.0\n2026-08-29,19,26.8,0.0,0.0,6.7\n2026-08-29,20,26.1,0.0,0.0,7.9\n2026-08-29,21,25.3,0.0,0.0,10.4\n2026-08-29,22,24.6,0.1,0.0,6.0\n2026-08-29,23,24.0,0.0,0.0,7.0\n2026-08-30,0,23.6,0.0,0.0,5.9\n2026-08-30,1,23.5,0.0,0.0,4.8\n2026-08-30,2,23.2,0.0,0.0,3.6\n2026-08-30,3,23.3,0.0,0.0,3.4\n2026-08-30,4,22.9,0.0,0.0,2.1\n2026-08-30,5,22.7,0.0,0.0,5.9\n2026-08-30,6,22.1,0.0,0.0,8.1\n2026-08-30,7,22.2,0.0,0.0,8.3\n2026-08-30,8,22.9,0.0,0.0,5.8\n2026-08-30,9,25.1,0.0,0.0,3.2\n2026-08-30,10,25.7,0.0,0.0,3.1\n2026-08-30,11,26.8,0.0,0.0,2.7\n2026-08-30,12,26.0,0.1,0.0,3.7\n2026-08-30,13,25.9,0.3,0.0,3.3\n2026-08-30,14,25.8,0.4,0.0,4.1\n2026-08-30,15,26.7,0.4,0.0,0.9\n2026-08-30,16,25.0,0.8,0.0,5.2\n2026-08-30,17,25.3,0.1,0.0,5.0\n2026-08-30,18,25.4,0.1,0.0,3.9\n2026-08-30,19,24.8,0.2,0.0,5.8\n2026-08-30,20,24.6,0.0,0.0,6.1\n2026-08-30,21,25.7,0.0,0.0,4.2\n2026-08-30,22,25.0,0.4,0.0,6.6\n2026-08-30,23,24.6,0.1,0.0,3.8\n2026-08-31,0,24.4,0.3,0.0,4.0\n2026-08-31,1,24.1,0.7,0.0,3.3\n2026-08-31,2,23.5,0.4,0.0,5.1\n2026-08-31,3,23.6,2.1,0.0,5.1\n2026-08-31,4,23.4,0.8,0.0,2.8\n2026-08-31,5,23.3,1.3,0.0,5.3\n2026-08-31,6,23.2,0.9,0.0,4.4\n2026-08-31,7,23.0,0.9,0.0,9.7\n2026-08-31,8,23.5,1.0,0.0,9.4\n2026-08-31,9,24.5,1.0,0.0,5.6\n2026-08-31,10,24.2,1.4,0.0,7.5\n2026-08-31,11,23.7,3.3,0.0,10.5\n2026-08-31,12,23.9,2.3,0.0,10.2\n2026-08-31,13,23.9,2.8,0.0,10.1\n2026-08-31,14,24.1,1.2,0.0,11.8\n2026-08-31,15,25.3,0.5,0.0,9.3\n2026-08-31,16,24.8,3.9,0.0,11.6\n2026-08-31,17,24.5,2.7,0.0,10.0\n2026-08-31,18,24.3,2.2,0.0,4.5\n2026-08-31,19,23.4,4.1,0.0,6.7\n2026-08-31,20,23.0,2.4,0.0,6.6\n2026-08-31,21,23.9,3.7,0.0,10.3\n2026-08-31,22,23.6,3.2,0.0,4.5\n2026-08-31,23,23.5,3.2,0.0,4.2\n2026-09-01,0,23.5,2.3,0.0,6.1\n2026-09-01,1,23.5,1.4,0.0,6.5\n2026-09-01,2,23.4,0.6,0.0,6.0\n2026-09-01,3,23.2,0.9,0.0,6.4\n2026-09-01,4,23.3,0.9,0.0,2.3\n2026-09-01,5,23.4,4.6,0.0,4.2\n2026-09-01,6,23.3,2.3,0.0,6.4\n2026-09-01,7,23.5,2.1,0.0,7.3\n2026-09-01,8,23.5,4.2,0.0,8.2\n2026-09-01,9,24.0,4.8,0.0,5.5\n2026-09-01,10,23.9,4.8,0.0,8.6\n2026-09-01,11,23.9,2.6,0.0,8.9\n2026-09-01,12,23.5,2.7,0.0,15.2\n2026-09-01,13,24.7,2.2,0.0,11.3\n2026-09-01,14,25.9,0.6,0.0,12.2\n2026-09-01,15,24.3,0.2,0.0,10.0\n2026-09-01,16,24.5,0.3,0.0,6.8\n2026-09-01,17,24.1,0.1,0.0,8.6\n2026-09-01,18,24.0,0.0,0.0,8.9\n2026-09-01,19,23.8,0.0,0.0,7.7\n2026-09-01,20,23.7,0.0,0.0,6.7\n2026-09-01,21,22.9,0.0,0.0,5.4\n2026-09-01,22,22.7,0.0,0.0,7.8\n2026-09-01,23,22.4,0.0,0.0,7.3\n2026-09-02,0,22.2,0.0,0.0,5.6\n2026-09-02,1,22.0,0.0,0.0,4.2\n2026-09-02,2,21.8,0.0,0.0,4.7\n2026-09-02,3,22.1,0.0,0.0,5.5\n2026-09-02,4,22.0,0.0,0.0,5.2\n2026-09-02,5,21.8,0.0,0.0,4.7\n2026-09-02,6,21.6,0.0,0.0,3.1\n2026-09-02,7,21.8,0.0,0.0,3.1\n2026-09-02,8,22.3,0.0,0.0,4.0\n2026-09-02,9,23.6,0.0,0.0,5.1\n2026-09-02,10,24.8,0.0,0.0,3.6\n2026-09-02,11,26.4,0.1,0.0,3.3\n2026-09-02,12,26.5,0.3,0.0,5.5\n2026-09-02,13,27.1,0.1,0.0,6.8\n2026-09-02,14,27.4,0.2,0.0,5.6\n2026-09-02,15,27.5,0.3,0.0,3.2\n2026-09-02,16,26.7,0.3,0.0,5.7\n2026-09-02,17,26.4,0.3,0.0,6.8\n2026-09-02,18,25.7,0.1,0.0,7.3\n2026-09-02,19,24.6,0.0,0.0,6.5\n2026-09-02,20,23.9,0.0,0.0,6.6\n2026-09-02,21,23.8,0.0,0.0,4.6\n2026-09-02,22,23.0,0.0,0.0,3.1\n2026-09-02,23,22.9,0.0,0.0,4.8\n2026-09-03,0,22.5,0.0,0.0,3.5\n2026-09-03,1,22.1,0.0,0.0,3.7\n2026-09-03,2,21.8,0.0,0.0,4.5\n2026-09-03,3,21.8,0.0,0.0,4.0\n2026-09-03,4,21.5,0.0,0.0,4.7\n2026-09-03,5,21.9,0.0,0.0,4.8\n2026-09-03,6,22.1,0.0,0.0,5.7\n2026-09-03,7,22.2,0.0,0.0,7.7\n2026-09-03,8,22.7,0.0,0.0,8.3\n2026-09-03,9,24.3,0.0,0.0,6.0\n2026-09-03,10,26.0,0.0,0.0,5.6\n2026-09-03,11,27.5,0.0,0.0,6.5\n2026-09-03,12,28.3,0.0,0.0,7.8\n2026-09-03,13,28.9,0.0,0.0,8.9\n2026-09-03,14,29.1,0.0,0.0,9.7\n2026-09-03,15,29.8,0.0,0.0,9.1\n2026-09-03,16,29.7,0.0,0.0,7.9\n2026-09-03,17,29.5,0.0,0.0,6.4\n2026-09-03,18,28.3,0.0,0.0,4.3\n2026-09-03,19,26.2,0.0,0.0,5.2\n2026-09-03,20,25.7,0.0,0.0,12.6\n2026-09-03,21,25.4,0.0,0.0,13.8\n2026-09-03,22,24.8,0.0,0.0,15.0\n2026-09-03,23,23.9,0.0,0.0,13.9\n2026-09-04,0,23.2,0.0,0.0,13.8\n2026-09-04,1,22.7,0.0,0.0,13.1\n2026-09-04,2,22.1,0.0,0.0,11.0\n2026-09-04,3,21.8,0.0,0.0,10.4\n2026-09-04,4,21.5,0.0,0.0,9.6\n2026-09-04,5,21.2,0.0,0.0,9.8\n2026-09-04,6,20.8,0.0,0.0,9.3\n2026-09-04,7,21.2,0.0,0.0,9.6\n2026-09-04,8,22.7,0.0,0.0,10.8\n2026-09-04,9,23.9,0.0,0.0,11.1\n2026-09-04,10,25.4,0.0,0.0,13.5\n2026-09-04,11,26.6,0.0,0.0,13.9\n2026-09-04,12,27.5,0.0,0.0,13.5\n2026-09-04,13,28.1,0.0,0.0,13.4\n2026-09-04,14,28.6,0.0,0.0,13.3\n2026-09-04,15,28.9,0.0,0.0,14.5\n2026-09-04,16,28.7,0.0,0.0,15.4\n2026-09-04,17,27.7,0.0,0.0,15.7\n2026-09-04,18,26.3,0.0,0.0,14.9\n2026-09-04,19,24.7,0.0,0.0,14.2\n2026-09-04,20,23.6,0.0,0.0,15.0\n2026-09-04,21,22.8,0.0,0.0,14.0\n2026-09-04,22,22.2,0.0,0.0,14.2\n2026-09-04,23,21.9,0.0,0.0,12.8\n2026-09-05,0,21.0,0.0,0.0,9.9\n2026-09-05,1,20.4,0.0,0.0,10.7\n2026-09-05,2,20.2,0.0,0.0,11.3\n2026-09-05,3,19.7,0.0,0.0,11.9\n2026-09-05,4,19.4,0.0,0.0,12.5\n2026-09-05,5,19.3,0.0,0.0,12.2\n2026-09-05,6,19.0,0.0,0.0,10.1\n2026-09-05,7,19.3,0.0,0.0,10.8\n2026-09-05,8,20.9,0.0,0.0,10.6\n2026-09-05,9,22.8,0.0,0.0,9.9\n2026-09-05,10,24.0,0.0,0.0,12.9\n2026-09-05,11,25.2,0.0,0.0,12.8\n2026-09-05,12,26.2,0.0,0.0,13.2\n2026-09-05,13,27.1,0.0,0.0,12.6\n2026-09-05,14,27.9,0.0,0.0,13.0\n2026-09-05,15,28.1,0.0,0.0,12.1\n2026-09-05,16,27.9,0.0,0.0,12.1\n2026-09-05,17,27.4,0.0,0.0,13.1\n2026-09-05,18,25.4,0.0,0.0,16.9\n2026-09-05,19,24.2,0.0,0.0,15.9\n2026-09-05,20,23.2,0.0,0.0,16.6\n2026-09-05,21,23.0,0.0,0.0,16.6\n2026-09-05,22,22.3,0.0,0.0,17.6\n2026-09-05,23,21.7,0.0,0.0,15.5\n2026-09-06,0,21.0,0.0,0.0,13.8\n2026-09-06,1,20.5,0.0,0.0,12.9\n2026-09-06,2,20.2,0.0,0.0,12.4\n2026-09-06,3,19.9,0.0,0.0,12.6\n2026-09-06,4,19.5,0.0,0.0,11.2\n2026-09-06,5,19.1,0.0,0.0,11.1\n2026-09-06,6,18.9,0.0,0.0,11.0\n2026-09-06,7,19.3,0.0,0.0,11.7\n2026-09-06,8,20.9,0.0,0.0,10.4\n2026-09-06,9,22.7,0.0,0.0,11.0\n2026-09-06,10,24.3,0.0,0.0,13.6\n2026-09-06,11,25.7,0.0,0.0,14.6\n2026-09-06,12,26.6,0.0,0.0,15.4\n2026-09-06,13,27.4,0.0,0.0,15.3\n2026-09-06,14,27.9,0.0,0.0,14.3\n2026-09-06,15,28.1,0.0,0.0,14.5\n2026-09-06,16,28.1,0.0,0.0,16.2\n2026-09-06,17,26.9,0.0,0.0,17.3\n2026-09-06,18,25.6,0.0,0.0,16.3\n2026-09-06,19,24.2,0.0,0.0,16.1\n2026-09-06,20,23.0,0.0,0.0,16.1\n2026-09-06,21,22.4,0.0,0.0,14.7\n2026-09-06,22,21.9,0.0,0.0,14.2\n2026-09-06,23,21.2,0.0,0.0,11.8\n2026-09-07,0,20.5,0.0,0.0,10.8\n2026-09-07,1,20.0,0.0,0.0,10.2\n2026-09-07,2,19.4,0.0,0.0,9.6\n2026-09-07,3,19.2,0.0,0.0,10.6\n2026-09-07,4,18.9,0.0,0.0,10.2\n2026-09-07,5,18.5,0.0,0.0,9.3\n2026-09-07,6,17.9,0.0,0.0,9.6\n2026-09-07,7,18.1,0.0,0.0,9.4\n2026-09-07,8,19.7,0.0,0.0,7.7\n2026-09-07,9,22.2,0.0,0.0,8.7\n2026-09-07,10,23.9,0.0,0.0,10.8\n2026-09-07,11,25.1,0.0,0.0,10.0\n2026-09-07,12,26.3,0.0,0.0,8.4\n2026-09-07,13,27.1,0.0,0.0,7.3\n2026-09-07,14,27.7,0.0,0.0,7.4\n2026-09-07,15,28.3,0.0,0.0,8.1\n2026-09-07,16,28.0,0.0,0.0,9.4\n2026-09-07,17,27.6,0.0,0.0,9.6\n2026-09-07,18,26.6,0.0,0.0,9.0\n2026-09-07,19,24.8,0.0,0.0,10.7\n2026-09-07,20,23.9,0.0,0.0,11.1\n2026-09-07,21,23.2,0.0,0.0,11.0\n2026-09-07,22,22.1,0.0,0.0,8.0\n2026-09-07,23,21.4,0.0,0.0,5.8\n2026-09-08,0,20.7,0.0,0.0,4.5\n2026-09-08,1,20.1,0.0,0.0,4.5\n2026-09-08,2,19.5,0.0,0.0,5.1\n2026-09-08,3,19.2,0.0,0.0,3.6\n2026-09-08,4,18.4,0.0,0.0,4.4\n2026-09-08,5,18.0,0.0,0.0,4.5\n2026-09-08,6,17.7,0.0,0.0,5.2\n2026-09-08,7,18.4,0.0,0.0,5.3\n2026-09-08,8,20.2,0.0,0.0,3.8\n2026-09-08,9,22.5,0.0,0.0,3.9\n2026-09-08,10,24.5,0.0,0.0,4.8\n2026-09-08,11,26.0,0.0,0.0,6.0\n2026-09-08,12,26.9,0.0,0.0,6.4\n2026-09-08,13,27.7,0.0,0.0,6.9\n2026-09-08,14,28.3,0.0,0.0,6.3\n2026-09-08,15,29.7,0.0,0.0,6.1\n2026-09-08,16,29.2,0.0,0.0,8.7\n2026-09-08,17,28.8,0.0,0.0,9.0\n2026-09-08,18,27.4,0.0,0.0,8.6\n2026-09-08,19,25.4,0.0,0.0,3.6\n2026-09-08,20,24.4,0.0,0.0,4.6\n2026-09-08,21,24.3,0.0,0.0,4.7\n2026-09-08,22,23.6,0.0,0.0,5.7\n2026-09-08,23,23.4,0.0,0.0,13.3\n2026-09-09,0,22.5,0.0,0.0,17.1\n2026-09-09,1,21.6,0.0,0.0,19.0\n2026-09-09,2,20.6,0.0,0.0,19.1\n2026-09-09,3,20.7,0.0,0.0,19.3\n2026-09-09,4,20.1,0.0,0.0,19.4\n2026-09-09,5,19.7,0.0,0.0,16.7\n2026-09-09,6,19.6,0.0,0.0,16.2\n2026-09-09,7,20.0,0.0,0.0,16.6\n2026-09-09,8,20.4,0.0,0.0,17.2\n2026-09-09,9,21.3,0.0,0.0,16.6\n2026-09-09,10,21.4,0.0,0.0,16.2\n2026-09-09,11,21.6,0.0,0.0,13.9\n2026-09-09,12,22.9,0.0,0.0,13.0\n2026-09-09,13,24.9,0.0,0.0,14.6\n2026-09-09,14,25.5,0.0,0.0,15.4\n2026-09-09,15,25.2,0.0,0.0,14.0\n2026-09-09,16,25.3,0.0,0.0,14.5\n2026-09-09,17,24.2,0.0,0.0,15.6\n2026-09-09,18,23.1,0.0,0.0,16.6\n2026-09-09,19,21.4,0.0,0.0,17.6\n2026-09-09,20,20.4,0.0,0.0,15.5\n2026-09-09,21,20.4,0.0,0.0,15.2\n2026-09-09,22,19.4,0.0,0.0,12.4\n2026-09-09,23,18.7,0.0,0.0,10.5\n2026-09-10,0,18.0,0.0,0.0,9.6\n2026-09-10,1,17.5,0.0,0.0,8.8\n2026-09-10,2,17.0,0.0,0.0,8.6\n2026-09-10,3,16.6,0.0,0.0,10.4\n2026-09-10,4,16.1,0.0,0.0,9.8\n2026-09-10,5,15.9,0.0,0.0,10.1\n2026-09-10,6,16.2,0.0,0.0,9.7\n2026-09-10,7,16.5,0.0,0.0,10.0\n2026-09-10,8,17.8,0.0,0.0,8.9\n2026-09-10,9,19.2,0.0,0.0,11.1\n2026-09-10,10,19.5,0.0,0.0,10.7\n2026-09-10,11,21.1,0.0,0.0,10.9\n2026-09-10,12,21.8,0.0,0.0,10.4\n2026-09-10,13,21.2,0.0,0.0,8.5\n2026-09-10,14,21.3,0.0,0.0,6.6\n2026-09-10,15,21.5,0.0,0.0,7.3\n2026-09-10,16,21.3,0.0,0.0,7.7\n2026-09-10,17,21.0,0.0,0.0,5.6\n2026-09-10,18,20.7,0.0,0.0,6.6\n2026-09-10,19,19.9,0.0,0.0,7.5\n2026-09-10,20,18.8,0.0,0.0,7.2\n2026-09-10,21,18.0,0.0,0.0,7.7\n2026-09-10,22,17.3,0.0,0.0,8.1\n2026-09-10,23,16.9,0.0,0.0,8.1\n2026-09-11,0,16.5,0.0,0.0,6.5\n2026-09-11,1,16.2,0.0,0.0,6.9\n2026-09-11,2,15.8,0.0,0.0,6.6\n2026-09-11,3,15.1,0.0,0.0,7.5\n2026-09-11,4,14.6,0.0,0.0,7.2\n2026-09-11,5,14.5,0.0,0.0,7.2\n2026-09-11,6,14.2,0.0,0.0,6.5\n2026-09-11,7,14.8,0.0,0.0,6.8\n2026-09-11,8,16.9,0.0,0.0,4.6\n2026-09-11,9,19.3,0.0,0.0,5.6\n2026-09-11,10,21.2,0.0,0.0,6.4\n2026-09-11,11,22.5,0.0,0.0,7.9\n2026-09-11,12,23.4,0.0,0.0,8.9\n2026-09-11,13,24.3,0.0,0.0,8.7\n2026-09-11,14,24.9,0.0,0.0,9.0\n2026-09-11,15,25.8,0.0,0.0,9.9\n2026-09-11,16,25.6,0.0,0.0,10.5\n2026-09-11,17,25.1,0.0,0.0,10.2\n2026-09-11,18,24.0,0.0,0.0,8.7\n2026-09-11,19,22.3,0.0,0.0,5.5\n2026-09-11,20,21.4,0.0,0.0,8.8\n2026-09-11,21,21.1,0.0,0.0,7.5\n2026-09-11,22,20.0,0.0,0.0,6.8\n2026-09-11,23,19.2,0.0,0.0,5.8\n2026-09-12,0,18.4,0.0,0.0,4.1\n2026-09-12,1,17.8,0.0,0.0,4.0\n2026-09-12,2,17.3,0.0,0.0,3.4\n2026-09-12,3,17.2,0.0,0.0,4.1\n2026-09-12,4,16.7,0.0,0.0,4.1\n2026-09-12,5,16.7,0.0,0.0,3.9\n2026-09-12,6,16.9,0.0,0.0,6.1\n2026-09-12,7,17.3,0.0,0.0,7.4\n2026-09-12,8,18.3,0.0,0.0,3.9\n2026-09-12,9,19.4,0.0,0.0,2.5\n2026-09-12,10,20.2,0.0,0.0,0.8\n2026-09-12,11,23.2,0.0,0.0,0.9\n2026-09-12,12,23.0,0.0,0.0,2.7\n2026-09-12,13,23.7,0.0,0.0,3.1\n2026-09-12,14,23.8,0.0,0.0,1.0\n2026-09-12,15,24.0,0.0,0.0,1.2\n2026-09-12,16,23.5,0.0,0.0,2.8\n2026-09-12,17,24.0,0.0,0.0,3.3\n2026-09-12,18,23.3,0.0,0.0,4.2\n2026-09-12,19,22.0,0.1,0.0,7.5\n2026-09-12,20,21.1,0.0,0.0,7.6\n2026-09-12,21,21.4,0.0,0.0,5.3\n2026-09-12,22,20.9,0.0,0.0,3.7\n2026-09-12,23,20.7,0.0,0.0,3.6\n2026-09-13,0,20.2,0.0,0.0,2.8\n2026-09-13,1,20.1,0.0,0.0,3.3\n2026-09-13,2,19.8,0.0,0.0,4.1\n2026-09-13,3,18.6,0.0,0.0,2.9\n2026-09-13,4,18.0,0.0,0.0,1.3\n2026-09-13,5,17.7,0.0,0.0,1.3\n2026-09-13,6,18.4,0.0,0.0,1.5\n2026-09-13,7,18.7,0.0,0.0,0.6\n2026-09-13,8,19.1,0.1,0.0,0.7\n2026-09-13,9,20.0,0.0,0.0,0.8\n2026-09-13,10,20.9,0.0,0.0,1.3\n2026-09-13,11,23.9,0.0,0.0,3.4\n2026-09-13,12,23.3,0.0,0.0,6.5\n2026-09-13,13,23.8,0.0,0.0,6.1\n2026-09-13,14,23.8,0.0,0.0,6.5\n2026-09-13,15,25.0,0.0,0.0,7.6\n2026-09-13,16,24.3,0.0,0.0,7.3\n2026-09-13,17,23.8,0.0,0.0,6.5\n2026-09-13,18,23.2,0.0,0.0,7.6\n2026-09-13,19,22.4,0.0,0.0,5.9\n2026-09-13,20,22.0,0.0,0.0,4.6\n2026-09-13,21,22.0,0.0,0.0,4.9\n2026-09-13,22,21.8,0.0,0.0,5.4\n2026-09-13,23,21.6,0.0,0.0,4.3\n2026-09-14,0,21.5,0.0,0.0,6.0\n2026-09-14,1,21.2,0.0,0.0,6.4\n2026-09-14,2,20.6,0.0,0.0,3.5\n2026-09-14,3,20.2,0.0,0.0,2.3\n2026-09-14,4,19.7,0.0,0.0,2.5\n2026-09-14,5,19.4,0.0,0.0,3.8\n2026-09-14,6,19.2,0.0,0.0,4.0\n2026-09-14,7,19.6,0.0,0.0,3.8\n2026-09-14,8,20.7,0.0,0.0,2.9\n2026-09-14,9,22.4,0.0,0.0,4.8\n2026-09-14,10,23.3,0.1,0.0,7.5\n2026-09-14,11,24.0,0.1,0.0,9.8\n2026-09-14,12,24.9,0.2,0.0,12.7\n2026-09-14,13,25.5,0.0,0.0,13.8\n2026-09-14,14,26.2,0.0,0.0,13.5\n2026-09-14,15,26.8,0.0,0.0,10.6\n2026-09-14,16,26.2,0.0,0.0,11.0\n2026-09-14,17,25.5,0.0,0.0,11.2\n2026-09-14,18,24.3,0.0,0.0,11.4\n2026-09-14,19,22.5,0.0,0.0,9.7\n2026-09-14,20,21.4,0.0,0.0,10.1\n2026-09-14,21,20.9,0.0,0.0,11.8\n2026-09-14,22,19.6,0.0,0.0,11.7\n2026-09-14,23,18.6,0.0,0.0,11.6\n2026-09-15,0,17.7,0.0,0.0,11.4\n2026-09-15,1,17.0,0.0,0.0,10.6\n2026-09-15,2,16.3,0.0,0.0,9.3\n2026-09-15,3,15.6,0.0,0.0,8.9\n2026-09-15,4,14.7,0.0,0.0,8.0\n2026-09-15,5,14.2,0.0,0.0,6.8\n2026-09-15,6,13.7,0.0,0.0,6.7\n2026-09-15,7,14.1,0.0,0.0,7.6\n2026-09-15,8,16.6,0.0,0.0,5.4\n2026-09-15,9,19.3,0.0,0.0,5.6\n2026-09-15,10,21.7,0.0,0.0,6.0\n2026-09-15,11,23.3,0.0,0.0,7.9\n2026-09-15,12,24.3,0.0,0.0,8.4\n2026-09-15,13,25.1,0.0,0.0,8.9\n2026-09-15,14,25.4,0.0,0.0,9.8\n2026-09-15,15,25.9,0.0,0.0,10.2\n2026-09-15,16,25.2,0.0,0.0,11.8\n2026-09-15,17,24.4,0.0,0.0,11.4\n2026-09-15,18,23.3,0.0,0.0,10.4\n2026-09-15,19,21.9,0.0,0.0,8.2\n2026-09-15,20,20.9,0.0,0.0,7.5\n2026-09-15,21,20.3,0.0,0.0,5.7\n2026-09-15,22,19.4,0.0,0.0,6.0\n2026-09-15,23,18.6,0.0,0.0,6.0\n2026-09-16,0,17.8,0.0,0.0,6.6\n2026-09-16,1,17.0,0.0,0.0,5.7\n2026-09-16,2,16.6,0.0,0.0,6.2\n2026-09-16,3,16.1,0.0,0.0,5.3\n2026-09-16,4,15.6,0.0,0.0,5.0\n2026-09-16,5,15.3,0.0,0.0,4.7\n2026-09-16,6,14.8,0.0,0.0,4.9\n2026-09-16,7,15.4,0.0,0.0,6.7\n2026-09-16,8,18.0,0.0,0.0,5.6\n2026-09-16,9,20.8,0.0,0.0,5.2\n2026-09-16,10,23.1,0.0,0.0,6.0\n2026-09-16,11,24.8,0.0,0.0,6.1\n2026-09-16,12,26.1,0.0,0.0,4.5\n2026-09-16,13,27.0,0.0,0.0,2.6\n2026-09-16,14,27.6,0.0,0.0,1.9\n2026-09-16,15,28.0,0.0,0.0,1.3\n2026-09-16,16,28.2,0.0,0.0,1.1\n2026-09-16,17,27.8,0.0,0.0,2.6\n2026-09-16,18,26.5,0.0,0.0,4.2\n2026-09-16,19,24.0,0.0,0.0,2.1\n2026-09-16,20,22.8,0.0,0.0,1.5\n2026-09-16,21,22.2,0.0,0.0,2.2\n2026-09-16,22,21.6,0.0,0.0,2.4\n2026-09-16,23,21.1,0.0,0.0,5.6\n2026-09-17,0,21.0,0.0,0.0,10.8\n2026-09-17,1,19.9,0.0,0.0,10.4\n2026-09-17,2,19.1,0.0,0.0,10.2\n2026-09-17,3,18.9,0.0,0.0,12.0\n2026-09-17,4,18.7,0.0,0.0,12.0\n2026-09-17,5,18.4,0.0,0.0,11.0\n2026-09-17,6,18.2,0.0,0.0,9.9\n2026-09-17,7,18.3,0.0,0.0,8.8\n2026-09-17,8,20.2,0.0,0.0,8.4\n2026-09-17,9,22.6,0.0,0.0,11.8\n2026-09-17,10,24.1,0.0,0.0,11.9\n2026-09-17,11,25.6,0.0,0.0,11.8\n2026-09-17,12,26.8,0.0,0.0,11.6\n2026-09-17,13,28.2,0.0,0.0,15.8\n2026-09-17,14,27.3,0.0,0.0,16.4\n2026-09-17,15,27.4,0.0,0.0,15.9\n2026-09-17,16,27.2,0.0,0.0,15.4\n2026-09-17,17,26.6,0.0,0.0,14.0\n2026-09-17,18,25.7,0.0,0.0,7.7\n2026-09-17,19,24.6,0.0,0.0,7.5\n2026-09-17,20,24.2,0.0,0.0,9.5\n2026-09-17,21,24.0,0.0,0.0,13.0\n2026-09-17,22,23.2,0.0,0.0,12.5\n2026-09-17,23,22.4,0.0,0.0,12.4\n2026-09-18,0,21.8,0.0,0.0,12.4\n2026-09-18,1,20.9,0.0,0.0,12.0\n2026-09-18,2,20.4,0.0,0.0,12.8\n2026-09-18,3,20.2,0.0,0.0,13.7\n2026-09-18,4,19.6,0.0,0.0,13.2\n2026-09-18,5,19.4,0.0,0.0,13.8\n2026-09-18,6,19.7,0.0,0.0,13.2\n2026-09-18,7,19.7,0.0,0.0,13.0\n2026-09-18,8,21.1,0.0,0.0,11.5\n2026-09-18,9,23.4,0.0,0.0,12.6\n2026-09-18,10,25.0,0.0,0.0,16.0\n2026-09-18,11,26.5,0.0,0.0,15.7\n2026-09-18,12,28.1,0.0,0.0,18.0\n2026-09-18,13,28.5,0.0,0.0,16.9\n2026-09-18,14,28.6,0.0,0.0,14.6\n2026-09-18,15,28.2,0.0,0.0,14.4\n2026-09-18,16,27.9,0.0,0.0,13.3\n2026-09-18,17,27.8,0.0,0.0,12.1\n2026-09-18,18,27.5,0.0,0.0,11.3\n2026-09-18,19,27.0,0.0,0.0,9.3\n2026-09-18,20,24.7,0.0,0.0,16.9\n2026-09-18,21,24.5,0.0,0.0,18.1\n2026-09-18,22,23.2,0.0,0.0,17.2\n2026-09-18,23,22.6,0.0,0.0,15.5\n2026-09-19,0,21.9,0.0,0.0,15.1\n2026-09-19,1,21.5,0.0,0.0,13.2\n2026-09-19,2,20.9,0.0,0.0,10.3\n2026-09-19,3,20.5,0.0,0.0,9.7\n2026-09-19,4,20.3,0.0,0.0,11.1\n2026-09-19,5,20.3,0.0,0.0,12.6\n2026-09-19,6,20.2,0.0,0.0,11.3\n2026-09-19,7,20.2,0.0,0.0,9.7\n2026-09-19,8,21.0,0.0,0.0,8.5\n2026-09-19,9,22.8,0.0,0.0,9.0\n2026-09-19,10,24.4,0.0,0.0,7.7\n2026-09-19,11,26.0,0.0,0.0,7.1\n2026-09-19,12,27.6,0.0,0.0,5.9\n2026-09-19,13,28.9,0.0,0.0,6.5\n2026-09-19,14,28.9,0.0,0.0,5.1\n2026-09-19,15,29.1,0.0,0.0,3.8\n2026-09-19,16,28.3,0.0,0.0,9.9\n2026-09-19,17,27.2,0.0,0.0,14.0\n2026-09-19,18,26.0,0.0,0.0,12.8\n2026-09-19,19,24.9,0.0,0.0,9.7\n2026-09-19,20,24.4,0.0,0.0,5.5\n2026-09-19,21,24.5,0.0,0.0,0.6\n2026-09-19,22,24.1,0.0,0.0,7.7\n2026-09-19,23,23.9,0.0,0.0,10.6\n2026-09-20,0,22.8,0.0,0.0,10.2\n2026-09-20,1,22.0,0.0,0.0,10.2\n2026-09-20,2,21.2,0.0,0.0,9.2\n2026-09-20,3,20.7,0.0,0.0,7.8\n2026-09-20,4,19.9,0.0,0.0,6.6\n2026-09-20,5,19.4,0.0,0.0,5.4\n2026-09-20,6,19.0,0.0,0.0,5.1\n2026-09-20,7,19.2,0.0,0.0,5.5\n2026-09-20,8,20.7,0.0,0.0,4.2\n2026-09-20,9,23.1,0.0,0.0,4.7\n2026-09-20,10,25.1,0.0,0.0,4.7\n2026-09-20,11,26.7,0.0,0.0,6.9\n2026-09-20,12,27.8,0.0,0.0,9.2\n2026-09-20,13,28.6,0.0,0.0,11.1\n2026-09-20,14,28.9,0.0,0.0,13.0\n2026-09-20,15,29.2,0.0,0.0,12.5\n2026-09-20,16,28.2,0.0,0.0,14.4\n2026-09-20,17,27.3,0.0,0.0,14.8\n2026-09-20,18,25.5,0.0,0.0,14.4\n2026-09-20,19,23.8,0.0,0.0,13.0\n2026-09-20,20,23.2,0.0,0.0,10.5\n2026-09-20,21,23.4,0.0,0.0,8.4\n2026-09-20,22,22.8,0.0,0.0,9.0\n2026-09-20,23,22.4,0.0,0.0,11.3\n2026-09-21,0,21.7,0.0,0.0,10.4\n2026-09-21,1,21.0,0.0,0.0,9.5\n2026-09-21,2,20.4,0.0,0.0,8.9\n2026-09-21,3,19.7,0.0,0.0,8.3\n2026-09-21,4,19.2,0.0,0.0,7.9\n2026-09-21,5,18.6,0.0,0.0,6.5\n2026-09-21,6,18.3,0.0,0.0,4.6\n2026-09-21,7,18.6,0.0,0.0,4.6\n2026-09-21,8,20.4,0.0,0.0,2.5\n2026-09-21,9,22.6,0.0,0.0,3.1\n2026-09-21,10,24.7,0.0,0.0,1.6\n2026-09-21,11,26.6,0.0,0.0,2.3\n2026-09-21,12,28.0,0.0,0.0,4.0\n2026-09-21,13,29.2,0.0,0.0,3.8\n2026-09-21,14,30.0,0.0,0.0,3.6\n2026-09-21,15,30.1,0.0,0.0,5.4\n2026-09-21,16,28.8,0.0,0.0,9.9\n2026-09-21,17,27.3,0.0,0.0,13.7\n2026-09-21,18,26.1,0.0,0.0,12.8\n2026-09-21,19,25.0,0.0,0.0,9.6\n2026-09-21,20,24.7,0.0,0.0,8.2\n2026-09-21,21,24.0,0.0,0.0,5.8\n2026-09-21,22,22.9,0.0,0.0,3.8\n2026-09-21,23,22.5,0.0,0.0,3.6\n2026-09-22,0,22.3,0.0,0.0,5.2\n2026-09-22,1,22.1,0.0,0.0,8.3\n2026-09-22,2,21.4,0.0,0.0,8.7\n2026-09-22,3,20.6,0.0,0.0,9.4\n2026-09-22,4,20.0,0.0,0.0,8.5\n2026-09-22,5,19.6,0.0,0.0,7.8\n2026-09-22,6,19.5,0.0,0.0,8.3\n2026-09-22,7,19.7,0.0,0.0,9.3\n2026-09-22,8,21.4,0.0,0.0,8.7\n2026-09-22,9,23.7,0.0,0.0,10.8\n2026-09-22,10,25.5,0.0,0.0,10.6\n2026-09-22,11,27.0,0.0,0.0,9.0\n2026-09-22,12,28.1,0.0,0.0,6.5\n2026-09-22,13,29.4,0.0,0.0,3.5\n2026-09-22,14,30.0,0.0,0.0,1.5\n2026-09-22,15,30.0,0.0,0.0,5.4\n2026-09-22,16,28.4,0.0,0.0,13.4\n2026-09-22,17,27.0,0.0,0.0,14.7\n2026-09-22,18,25.7,0.0,0.0,11.1\n2026-09-22,19,24.6,0.0,0.0,7.8\n2026-09-22,20,24.2,0.0,0.0,4.8\n2026-09-22,21,24.0,0.0,0.0,3.4\n2026-09-22,22,23.1,0.0,0.0,4.6\n2026-09-22,23,22.5,0.0,0.0,4.0\n2026-09-23,0,22.2,0.0,0.0,3.0\n2026-09-23,1,21.9,0.0,0.0,2.3\n2026-09-23,2,21.4,0.0,0.0,4.2\n2026-09-23,3,20.8,0.0,0.0,6.5\n2026-09-23,4,20.2,0.0,0.0,7.7\n2026-09-23,5,19.8,0.0,0.0,7.6\n2026-09-23,6,19.5,0.0,0.0,7.5\n2026-09-23,7,19.7,0.0,0.0,8.6\n2026-09-23,8,21.5,0.0,0.0,7.7\n2026-09-23,9,23.5,0.0,0.0,7.5\n2026-09-23,10,25.2,0.0,0.0,7.7\n2026-09-23,11,26.6,0.0,0.0,6.6\n2026-09-23,12,27.4,0.1,0.0,7.3\n2026-09-23,13,28.0,0.0,0.0,8.7\n2026-09-23,14,28.1,0.0,0.0,11.5\n2026-09-23,15,28.2,0.0,0.0,11.7\n2026-09-23,16,27.1,0.0,0.0,14.7\n2026-09-23,17,25.7,0.0,0.0,13.8\n2026-09-23,18,24.5,0.0,0.0,12.2\n2026-09-23,19,23.5,0.0,0.0,9.9\n2026-09-23,20,23.2,0.0,0.0,8.8\n2026-09-23,21,23.2,0.0,0.0,7.6\n2026-09-23,22,22.7,0.0,0.0,5.6\n2026-09-23,23,22.2,0.0,0.0,4.8\n2026-09-24,0,21.7,0.0,0.0,3.2\n2026-09-24,1,21.3,0.0,0.0,3.3\n2026-09-24,2,20.9,0.0,0.0,4.0\n2026-09-24,3,19.9,0.0,0.0,5.1\n2026-09-24,4,19.1,0.0,0.0,4.3\n2026-09-24,5,18.6,0.0,0.0,2.2\n2026-09-24,6,18.3,0.0,0.0,1.5\n2026-09-24,7,18.5,0.0,0.0,4.0\n2026-09-24,8,19.0,0.0,0.0,3.1\n2026-09-24,9,21.0,0.0,0.0,1.8\n2026-09-24,10,23.1,0.0,0.0,4.9\n2026-09-24,11,25.0,0.0,0.0,5.4\n2026-09-24,12,26.0,0.0,0.0,4.9\n2026-09-24,13,27.1,0.0,0.0,6.8\n2026-09-24,14,27.8,0.0,0.0,8.1\n2026-09-24,15,27.6,0.0,0.0,7.0\n2026-09-24,16,26.4,0.0,0.0,8.4\n2026-09-24,17,25.6,0.0,0.0,9.3\n2026-09-24,18,24.8,0.0,0.0,8.5\n2026-09-24,19,24.1,0.0,0.0,6.8\n2026-09-24,20,23.4,0.0,0.0,5.1\n2026-09-24,21,22.7,0.0,0.0,2.8\n2026-09-24,22,22.3,0.0,0.0,4.8\n2026-09-24,23,21.5,0.0,0.0,6.3\n2026-09-25,0,20.7,0.0,0.0,7.0\n2026-09-25,1,20.0,0.0,0.0,5.8\n2026-09-25,2,19.7,0.0,0.0,4.0\n2026-09-25,3,20.1,0.0,0.0,3.2\n2026-09-25,4,19.7,0.0,0.0,5.6\n2026-09-25,5,19.4,0.0,0.0,6.1\n2026-09-25,6,19.5,0.0,0.0,7.1\n2026-09-25,7,20.3,0.0,0.0,8.4\n2026-09-25,8,21.3,0.0,0.0,7.8\n2026-09-25,9,22.3,0.0,0.0,7.9\n2026-09-25,10,24.0,0.0,0.0,10.4\n2026-09-25,11,24.6,0.0,0.0,9.1\n2026-09-25,12,24.9,0.0,0.0,10.3\n2026-09-25,13,23.6,0.8,0.0,9.8\n2026-09-25,14,19.0,4.4,0.0,8.6\n2026-09-25,15,20.4,4.9,0.0,7.4\n2026-09-25,16,19.2,3.6,0.0,10.2\n2026-09-25,17,18.5,1.1,0.0,10.5\n2026-09-25,18,18.2,0.6,0.0,8.7\n2026-09-25,19,17.8,0.6,0.0,12.3\n2026-09-25,20,17.8,0.2,0.0,11.9\n2026-09-25,21,19.2,0.2,0.0,7.1\n2026-09-25,22,19.1,1.0,0.0,6.6\n2026-09-25,23,19.0,0.5,0.0,7.6\n2026-09-26,0,18.8,0.1,0.0,9.4\n2026-09-26,1,18.8,0.0,0.0,8.8\n2026-09-26,2,18.8,0.0,0.0,8.9\n2026-09-26,3,18.5,0.0,0.0,8.1\n2026-09-26,4,18.2,0.0,0.0,8.2\n2026-09-26,5,18.3,0.0,0.0,9.0\n2026-09-26,6,18.4,0.0,0.0,9.1\n2026-09-26,7,18.6,0.0,0.0,8.3\n2026-09-26,8,19.0,0.1,0.0,6.5\n2026-09-26,9,20.0,0.0,0.0,6.5\n2026-09-26,10,21.0,0.0,0.0,6.3\n2026-09-26,11,22.2,0.0,0.0,5.6\n2026-09-26,12,23.3,0.0,0.0,6.2\n2026-09-26,13,24.7,0.0,0.0,7.1\n2026-09-26,14,25.3,0.0,0.0,4.8\n2026-09-26,15,26.1,0.0,0.0,4.8\n2026-09-26,16,25.7,0.0,0.0,5.5\n2026-09-26,17,25.3,0.0,0.0,6.7\n2026-09-26,18,24.0,0.0,0.0,2.5\n2026-09-26,19,23.1,0.0,0.0,3.0\n2026-09-26,20,22.3,0.0,0.0,6.3\n2026-09-26,21,21.0,0.0,0.0,7.3\n2026-09-26,22,20.3,0.0,0.0,8.3\n2026-09-26,23,19.5,0.0,0.0,3.8\n2026-09-27,0,18.9,0.0,0.0,2.8\n2026-09-27,1,18.7,0.0,0.0,3.0\n2026-09-27,2,18.5,0.0,0.0,2.7\n2026-09-27,3,17.9,0.0,0.0,3.2\n2026-09-27,4,17.5,0.0,0.0,2.7\n2026-09-27,5,17.2,0.0,0.0,3.2\n2026-09-27,6,17.1,0.0,0.0,4.1\n2026-09-27,7,17.4,0.0,0.0,5.7\n2026-09-27,8,18.8,0.0,0.0,5.2\n2026-09-27,9,21.2,0.0,0.0,4.0\n2026-09-27,10,23.2,0.0,0.0,2.4\n2026-09-27,11,24.7,0.0,0.0,3.2\n2026-09-27,12,25.6,0.0,0.0,4.0\n2026-09-27,13,26.2,0.0,0.0,6.9\n2026-09-27,14,26.0,0.0,0.0,10.8\n2026-09-27,15,26.1,0.0,0.0,10.5\n2026-09-27,16,25.2,0.0,0.0,11.6\n2026-09-27,17,24.2,0.0,0.0,11.2\n2026-09-27,18,22.9,0.0,0.0,8.6\n2026-09-27,19,21.6,0.0,0.0,8.3\n2026-09-27,20,21.1,0.0,0.0,9.3\n2026-09-27,21,20.9,0.0,0.0,7.9\n2026-09-27,22,20.6,0.0,0.0,8.3\n2026-09-27,23,20.3,0.0,0.0,7.1\n2026-09-28,0,19.8,0.0,0.0,7.3\n2026-09-28,1,19.4,0.0,0.0,9.2\n2026-09-28,2,18.9,0.0,0.0,8.0\n2026-09-28,3,18.1,0.0,0.0,5.8\n2026-09-28,4,18.1,0.0,0.0,4.5\n2026-09-28,5,18.1,0.0,0.0,5.3\n2026-09-28,6,18.0,0.0,0.0,7.4\n2026-09-28,7,17.8,0.0,0.0,8.1\n2026-09-28,8,18.2,0.0,0.0,7.5\n2026-09-28,9,19.7,0.0,0.0,9.8\n2026-09-28,10,21.0,0.0,0.0,8.8\n2026-09-28,11,22.2,0.0,0.0,8.2\n2026-09-28,12,23.2,0.0,0.0,9.0\n2026-09-28,13,23.6,0.0,0.0,11.4\n2026-09-28,14,23.4,0.0,0.0,13.1\n2026-09-28,15,23.4,0.0,0.0,12.8\n2026-09-28,16,22.8,0.0,0.0,13.2\n2026-09-28,17,22.0,0.0,0.0,10.1\n2026-09-28,18,21.2,0.0,0.0,8.2\n2026-09-28,19,20.5,0.0,0.0,7.4\n2026-09-28,20,20.1,0.0,0.0,5.2\n2026-09-28,21,19.9,0.0,0.0,8.8\n2026-09-28,22,18.6,0.0,0.0,10.3\n2026-09-28,23,17.7,0.0,0.0,9.8\n2026-09-29,0,17.0,0.0,0.0,9.5\n2026-09-29,1,16.6,0.0,0.0,7.9\n2026-09-29,2,16.2,0.0,0.0,8.0\n2026-09-29,3,15.6,0.0,0.0,6.9\n2026-09-29,4,14.9,0.0,0.0,5.8\n2026-09-29,5,14.6,0.0,0.0,6.1\n2026-09-29,6,14.2,0.0,0.0,5.0\n2026-09-29,7,14.4,0.0,0.0,4.7\n2026-09-29,8,16.4,0.0,0.0,4.5\n2026-09-29,9,18.7,0.0,0.0,3.1\n2026-09-29,10,20.9,0.0,0.0,3.1\n2026-09-29,11,22.7,0.0,0.0,3.6\n2026-09-29,12,23.8,0.0,0.0,4.6\n2026-09-29,13,24.6,0.0,0.0,3.5\n2026-09-29,14,25.2,0.0,0.0,4.6\n2026-09-29,15,26.0,0.0,0.0,6.0\n2026-09-29,16,25.4,0.0,0.0,11.6\n2026-09-29,17,24.2,0.0,0.0,11.7\n2026-09-29,18,22.7,0.0,0.0,10.3\n2026-09-29,19,21.3,0.0,0.0,7.3\n2026-09-29,20,20.3,0.0,0.0,5.0\n2026-09-29,21,19.9,0.0,0.0,3.5\n2026-09-29,22,19.3,0.0,0.0,3.1\n2026-09-29,23,18.9,0.0,0.0,1.9\n2026-09-30,0,18.4,0.0,0.0,1.1\n2026-09-30,1,18.0,0.0,0.0,0.7\n2026-09-30,2,17.8,0.0,0.0,0.4\n2026-09-30,3,17.7,0.0,0.0,2.5\n2026-09-30,4,16.9,0.0,0.0,4.2\n2026-09-30,5,16.3,0.0,0.0,2.0\n2026-09-30,6,16.1,0.0,0.0,5.0\n2026-09-30,7,16.5,0.0,0.0,7.4\n2026-09-30,8,18.8,0.0,0.0,7.3\n2026-09-30,9,20.4,0.0,0.0,4.4\n2026-09-30,10,22.3,0.0,0.0,5.8\n2026-09-30,11,24.2,0.0,0.0,7.5\n2026-09-30,12,25.1,0.0,0.0,9.0\n2026-09-30,13,25.6,0.0,0.0,10.0\n2026-09-30,14,25.5,0.0,0.0,12.2\n2026-09-30,15,25.4,0.0,0.0,15.5\n2026-09-30,16,24.4,0.0,0.0,16.9\n2026-09-30,17,23.4,0.0,0.0,16.3\n2026-09-30,18,22.7,0.0,0.0,17.3\n2026-09-30,19,22.2,0.2,0.0,16.6\n2026-09-30,20,21.5,0.4,0.0,17.5\n2026-09-30,21,21.8,0.3,0.0,14.4\n2026-09-30,22,20.8,0.0,0.0,14.9\n2026-09-30,23,19.9,0.0,0.0,18.7\n2026-10-01,0,19.0,0.0,0.0,18.0\n2026-10-01,1,18.2,0.0,0.0,18.2\n2026-10-01,2,17.3,0.0,0.0,15.4\n2026-10-01,3,16.7,0.0,0.0,14.4\n2026-10-01,4,16.3,0.0,0.0,15.0\n2026-10-01,5,15.9,0.0,0.0,12.4\n2026-10-01,6,15.5,0.0,0.0,12.1\n2026-10-01,7,14.7,0.0,0.0,11.8\n2026-10-01,8,15.3,0.0,0.0,11.0\n2026-10-01,9,16.7,0.0,0.0,8.7\n2026-10-01,10,18.2,0.0,0.0,11.4\n2026-10-01,11,19.4,0.0,0.0,13.7\n2026-10-01,12,20.2,0.0,0.0,13.0\n2026-10-01,13,20.9,0.0,0.0,13.1\n2026-10-01,14,21.2,0.0,0.0,14.4\n2026-10-01,15,21.4,0.0,0.0,14.0\n2026-10-01,16,20.7,0.0,0.0,14.5\n2026-10-01,17,19.9,0.0,0.0,14.3\n2026-10-01,18,19.1,0.0,0.0,11.2\n2026-10-01,19,18.4,0.0,0.0,7.9\n2026-10-01,20,18.5,0.0,0.0,13.6\n2026-10-01,21,17.3,0.0,0.0,11.6\n2026-10-01,22,16.0,0.0,0.0,12.5\n2026-10-01,23,15.2,0.0,0.0,10.6\n2026-10-02,0,14.2,0.0,0.0,8.4\n2026-10-02,1,13.5,0.0,0.0,6.7\n2026-10-02,2,13.2,0.0,0.0,5.9\n2026-10-02,3,12.7,0.0,0.0,7.3\n2026-10-02,4,11.9,0.0,0.0,8.3\n2026-10-02,5,11.4,0.0,0.0,8.6\n2026-10-02,6,11.1,0.0,0.0,8.1\n2026-10-02,7,11.0,0.0,0.0,7.4\n2026-10-02,8,13.0,0.0,0.0,5.2\n2026-10-02,9,15.6,0.0,0.0,5.6\n2026-10-02,10,17.6,0.0,0.0,8.6\n2026-10-02,11,18.9,0.0,0.0,12.8\n2026-10-02,12,19.8,0.0,0.0,13.2\n2026-10-02,13,20.5,0.0,0.0,13.7\n2026-10-02,14,20.9,0.0,0.0,14.6\n2026-10-02,15,21.4,0.0,0.0,12.8\n2026-10-02,16,21.2,0.0,0.0,12.8\n2026-10-02,17,20.5,0.0,0.0,13.6\n2026-10-02,18,19.3,0.0,0.0,13.0\n2026-10-02,19,17.9,0.0,0.0,11.0\n2026-10-02,20,16.9,0.0,0.0,10.4\n2026-10-02,21,15.9,0.0,0.0,8.6\n2026-10-02,22,14.7,0.0,0.0,7.8\n2026-10-02,23,13.9,0.0,0.0,8.2\n2026-10-03,0,13.1,0.0,0.0,7.6\n2026-10-03,1,12.4,0.0,0.0,7.2\n2026-10-03,2,11.9,0.0,0.0,6.0\n2026-10-03,3,11.5,0.0,0.0,6.4\n2026-10-03,4,11.3,0.0,0.0,7.2\n2026-10-03,5,11.3,0.0,0.0,5.9\n2026-10-03,6,11.1,0.0,0.0,4.4\n2026-10-03,7,11.6,0.0,0.0,4.5\n2026-10-03,8,13.3,0.0,0.0,4.0\n2026-10-03,9,15.5,0.0,0.0,6.2\n2026-10-03,10,17.8,0.0,0.0,7.7\n2026-10-03,11,19.9,0.0,0.0,9.9\n2026-10-03,12,21.3,0.0,0.0,11.1\n2026-10-03,13,22.2,0.0,0.0,9.8\n2026-10-03,14,22.2,0.0,0.0,9.8\n2026-10-03,15,21.9,0.0,0.0,7.8\n2026-10-03,16,21.9,0.0,0.0,6.8\n2026-10-03,17,21.6,0.0,0.0,6.0\n2026-10-03,18,20.3,0.0,0.0,4.2\n2026-10-03,19,19.5,0.0,0.0,5.0\n2026-10-03,20,18.9,0.0,0.0,5.7\n2026-10-03,21,18.2,0.0,0.0,6.4\n2026-10-03,22,17.6,0.0,0.0,6.4\n2026-10-03,23,17.3,0.0,0.0,8.1\n2026-10-04,0,17.3,0.0,0.0,9.9\n2026-10-04,1,16.9,0.0,0.0,10.1\n2026-10-04,2,16.5,0.0,0.0,10.3\n2026-10-04,3,16.3,0.0,0.0,10.1\n2026-10-04,4,15.6,0.0,0.0,11.0\n2026-10-04,5,15.5,0.0,0.0,9.7\n2026-10-04,6,15.2,0.0,0.0,9.0\n2026-10-04,7,15.1,0.0,0.0,10.2\n2026-10-04,8,16.8,0.0,0.0,7.9\n2026-10-04,9,19.3,0.0,0.0,10.5\n2026-10-04,10,21.3,0.0,0.0,12.8\n2026-10-04,11,22.9,0.0,0.0,12.2\n2026-10-04,12,24.0,0.0,0.0,9.7\n2026-10-04,13,23.3,0.0,0.0,12.2\n2026-10-04,14,24.3,0.0,0.0,11.4\n2026-10-04,15,24.0,0.0,0.0,11.2\n2026-10-04,16,23.1,0.1,0.0,14.4\n2026-10-04,17,21.9,0.0,0.0,15.9\n2026-10-04,18,20.7,0.1,0.0,17.1\n2026-10-04,19,19.4,0.4,0.0,14.7\n2026-10-04,20,19.1,0.1,0.0,16.2\n2026-10-04,21,18.9,0.2,0.0,12.4\n2026-10-04,22,18.0,0.6,0.0,11.6\n2026-10-04,23,17.4,0.3,0.0,15.0\n2026-10-05,0,16.5,0.8,0.0,10.0\n2026-10-05,1,16.4,0.2,0.0,11.5\n2026-10-05,2,15.8,0.0,0.0,10.5\n2026-10-05,3,15.1,0.0,0.0,10.3\n2026-10-05,4,14.9,0.1,0.0,8.0\n2026-10-05,5,14.9,0.0,0.0,11.5\n2026-10-05,6,14.6,0.1,0.0,10.8\n2026-10-05,7,14.3,0.0,0.0,8.7\n2026-10-05,8,15.5,0.0,0.0,9.6\n2026-10-05,9,17.3,0.0,0.0,14.1\n2026-10-05,10,18.0,0.1,0.0,15.8\n2026-10-05,11,18.6,0.1,0.0,16.9\n2026-10-05,12,19.4,0.1,0.0,17.3\n2026-10-05,13,19.6,0.1,0.0,18.8\n2026-10-05,14,19.4,0.1,0.0,20.1\n2026-10-05,15,19.0,0.1,0.0,18.1\n2026-10-05,16,19.1,0.1,0.0,17.9\n2026-10-05,17,19.0,0.0,0.0,17.5\n2026-10-05,18,18.0,0.0,0.0,16.1\n2026-10-05,19,16.9,0.0,0.0,16.1\n2026-10-05,20,16.1,0.0,0.0,14.7\n2026-10-05,21,15.3,0.0,0.0,12.7\n2026-10-05,22,14.8,0.0,0.0,11.8\n2026-10-05,23,14.6,0.0,0.0,11.7\n2026-10-06,0,14.5,0.0,0.0,10.9\n2026-10-06,1,14.3,0.0,0.0,10.8\n2026-10-06,2,14.3,0.0,0.0,10.7\n2026-10-06,3,12.5,0.0,0.0,7.2\n2026-10-06,4,11.8,0.0,0.0,6.5\n2026-10-06,5,11.3,0.0,0.0,7.1\n2026-10-06,6,10.8,0.0,0.0,6.1\n2026-10-06,7,10.8,0.0,0.0,4.8\n2026-10-06,8,12.8,0.0,0.0,4.7\n2026-10-06,9,15.0,0.0,0.0,3.6\n2026-10-06,10,17.2,0.0,0.0,4.6\n2026-10-06,11,18.5,0.0,0.0,9.3\n2026-10-06,12,19.6,0.0,0.0,10.1\n2026-10-06,13,20.4,0.0,0.0,11.2\n2026-10-06,14,20.9,0.0,0.0,11.6\n2026-10-06,15,21.3,0.0,0.0,12.5\n2026-10-06,16,20.6,0.0,0.0,14.4\n2026-10-06,17,19.6,0.0,0.0,15.7\n2026-10-06,18,18.6,0.0,0.0,11.9\n2026-10-06,19,18.3,0.0,0.0,12.3\n2026-10-06,20,18.2,0.0,0.0,11.4\n2026-10-06,21,17.8,0.0,0.0,13.1\n2026-10-06,22,16.4,0.0,0.0,8.3\n2026-10-06,23,15.2,0.0,0.0,6.8\n2026-10-07,0,14.6,0.0,0.0,5.0\n2026-10-07,1,14.6,0.0,0.0,5.4\n2026-10-07,2,14.5,0.0,0.0,7.0\n2026-10-07,3,13.0,0.0,0.0,3.9\n2026-10-07,4,12.1,0.0,0.0,4.1\n2026-10-07,5,12.0,0.0,0.0,5.9\n2026-10-07,6,11.7,0.0,0.0,5.2\n2026-10-07,7,11.8,0.0,0.0,4.1\n2026-10-07,8,14.3,0.0,0.0,3.3\n2026-10-07,9,16.5,0.0,0.0,2.0\n2026-10-07,10,18.8,0.0,0.0,0.6\n2026-10-07,11,20.4,0.0,0.0,2.7\n2026-10-07,12,21.6,0.0,0.0,5.2\n2026-10-07,13,22.2,0.0,0.0,7.7\n2026-10-07,14,22.1,0.0,0.0,9.9\n2026-10-07,15,22.2,0.0,0.0,12.5\n2026-10-07,16,21.6,0.0,0.0,12.6\n2026-10-07,17,20.9,0.0,0.0,11.2\n2026-10-07,18,19.3,0.0,0.0,8.0\n2026-10-07,19,17.8,0.0,0.0,5.9\n2026-10-07,20,17.3,0.0,0.0,4.6\n2026-10-07,21,16.8,0.0,0.0,1.4\n2026-10-07,22,16.3,0.0,0.0,1.1\n2026-10-07,23,15.4,0.0,0.0,2.2\n2026-10-08,0,14.6,0.0,0.0,3.0\n2026-10-08,1,13.6,0.0,0.0,4.7\n2026-10-08,2,12.8,0.0,0.0,4.0\n2026-10-08,3,12.1,0.0,0.0,3.8\n2026-10-08,4,12.1,0.0,0.0,4.8\n2026-10-08,5,12.0,0.0,0.0,5.4\n2026-10-08,6,11.6,0.0,0.0,4.4\n2026-10-08,7,11.5,0.0,0.0,3.9\n2026-10-08,8,13.8,0.0,0.0,3.9\n2026-10-08,9,16.2,0.0,0.0,4.0\n2026-10-08,10,18.5,0.0,0.0,4.4\n2026-10-08,11,20.2,0.0,0.0,2.7\n2026-10-08,12,21.4,0.0,0.0,2.4\n2026-10-08,13,22.2,0.0,0.0,2.8\n2026-10-08,14,22.7,0.0,0.0,4.5\n2026-10-08,15,23.5,0.0,0.0,5.1\n2026-10-08,16,23.0,0.0,0.0,8.5\n2026-10-08,17,21.8,0.0,0.0,10.4\n2026-10-08,18,20.1,0.0,0.0,8.0\n2026-10-08,19,18.8,0.0,0.0,6.9\n2026-10-08,20,17.7,0.0,0.0,5.5\n2026-10-08,21,16.9,0.0,0.0,3.9\n2026-10-08,22,16.2,0.0,0.0,4.0\n2026-10-08,23,15.7,0.0,0.0,4.6\n2026-10-09,0,15.0,0.0,0.0,5.7\n2026-10-09,1,14.7,0.0,0.0,6.8\n2026-10-09,2,14.5,0.0,0.0,6.7\n2026-10-09,3,13.8,0.0,0.0,5.0\n2026-10-09,4,13.4,0.0,0.0,4.5\n2026-10-09,5,13.2,0.0,0.0,2.8\n2026-10-09,6,13.0,0.0,0.0,1.4\n2026-10-09,7,12.8,0.0,0.0,1.9\n2026-10-09,8,14.8,0.0,0.0,3.5\n2026-10-09,9,16.9,0.0,0.0,2.4\n2026-10-09,10,19.2,0.0,0.0,1.7\n2026-10-09,11,21.1,0.0,0.0,3.3\n2026-10-09,12,22.5,0.0,0.0,5.0\n2026-10-09,13,23.6,0.0,0.0,6.8\n2026-10-09,14,23.9,0.0,0.0,7.7\n2026-10-09,15,24.5,0.0,0.0,7.1\n2026-10-09,16,23.9,0.0,0.0,8.6\n2026-10-09,17,22.9,0.0,0.0,8.7\n2026-10-09,18,21.1,0.0,0.0,7.2\n2026-10-09,19,19.6,0.0,0.0,7.1\n2026-10-09,20,19.0,0.0,0.0,6.5\n2026-10-09,21,18.4,0.0,0.0,5.6\n2026-10-09,22,18.0,0.0,0.0,4.7\n2026-10-09,23,17.3,0.0,0.0,3.7\n2026-10-10,0,16.3,0.0,0.0,4.5\n2026-10-10,1,15.9,0.0,0.0,5.4\n2026-10-10,2,15.3,0.0,0.0,5.2\n2026-10-10,3,14.4,0.0,0.0,3.8\n2026-10-10,4,14.0,0.0,0.0,4.3\n2026-10-10,5,13.8,0.0,0.0,3.6\n2026-10-10,6,13.4,0.0,0.0,5.2\n2026-10-10,7,12.8,0.0,0.0,4.7\n2026-10-10,8,14.5,0.0,0.0,3.4\n2026-10-10,9,17.7,0.0,0.0,2.7\n2026-10-10,10,20.4,0.0,0.0,2.8\n2026-10-10,11,22.4,0.0,0.0,3.5\n2026-10-10,12,23.8,0.0,0.0,4.7\n2026-10-10,13,24.7,0.0,0.0,4.6\n2026-10-10,14,25.3,0.0,0.0,4.7\n2026-10-10,15,26.0,0.0,0.0,3.2\n2026-10-10,16,25.3,0.0,0.0,7.4\n2026-10-10,17,24.0,0.0,0.0,9.4\n2026-10-10,18,22.0,0.0,0.0,8.9\n2026-10-10,19,20.8,0.0,0.0,8.7\n2026-10-10,20,19.9,0.0,0.0,8.7\n2026-10-10,21,18.8,0.0,0.0,6.8\n2026-10-10,22,18.3,0.0,0.0,8.9\n2026-10-10,23,18.0,0.0,0.0,6.5\n2026-10-11,0,16.9,0.0,0.0,7.3\n2026-10-11,1,16.4,0.0,0.0,6.8\n2026-10-11,2,16.3,0.0,0.0,5.7\n2026-10-11,3,15.5,0.0,0.0,5.9\n2026-10-11,4,15.1,0.0,0.0,6.7\n2026-10-11,5,14.8,0.0,0.0,5.4\n2026-10-11,6,14.5,0.0,0.0,4.1\n2026-10-11,7,14.7,0.0,0.0,3.7\n2026-10-11,8,17.1,0.0,0.0,3.8\n2026-10-11,9,19.3,0.0,0.0,1.8\n2026-10-11,10,21.7,0.0,0.0,3.3\n2026-10-11,11,24.2,0.0,0.0,4.2\n2026-10-11,12,26.1,0.0,0.0,3.4\n2026-10-11,13,27.1,0.0,0.0,2.0\n2026-10-11,14,28.0,0.0,0.0,4.3\n2026-10-11,15,28.2,0.0,0.0,7.1\n2026-10-11,16,27.7,0.0,0.0,9.5\n2026-10-11,17,26.8,0.0,0.0,8.1\n2026-10-11,18,25.1,0.0,0.0,6.4\n2026-10-11,19,23.9,0.0,0.0,5.5\n2026-10-11,20,23.7,0.0,0.0,8.4\n2026-10-11,21,23.5,0.0,0.0,8.7\n2026-10-11,22,22.9,0.0,0.0,6.3\n2026-10-11,23,22.0,0.0,0.0,4.3\n2026-10-12,0,21.6,0.0,0.0,4.9\n2026-10-12,1,21.0,0.0,0.0,5.0\n2026-10-12,2,20.8,0.0,0.0,6.6\n2026-10-12,3,20.6,0.0,0.0,10.4\n2026-10-12,4,20.8,0.0,0.0,11.6\n2026-10-12,5,20.6,0.0,0.0,10.0\n2026-10-12,6,20.4,0.0,0.0,10.8\n2026-10-12,7,19.9,0.0,0.0,8.3\n2026-10-12,8,20.9,0.0,0.0,5.2\n2026-10-12,9,21.9,0.0,0.0,9.1\n2026-10-12,10,24.3,0.0,0.0,9.4\n2026-10-12,11,25.8,0.1,0.0,5.6\n2026-10-12,12,27.1,0.0,0.0,4.2\n2026-10-12,13,28.2,0.1,0.0,7.5\n2026-10-12,14,27.3,0.0,0.0,8.5\n2026-10-12,15,28.6,0.7,0.0,7.8\n2026-10-12,16,25.8,0.4,0.0,14.9\n2026-10-12,17,24.8,0.0,0.0,12.1\n2026-10-12,18,23.4,0.0,0.0,8.0\n2026-10-12,19,22.6,0.1,0.0,5.2\n2026-10-12,20,21.8,0.4,0.0,6.1\n2026-10-12,21,22.4,0.2,0.0,4.0\n2026-10-12,22,21.7,0.0,0.0,9.8\n2026-10-12,23,20.0,0.4,0.0,11.3\n2026-10-13,0,19.0,1.6,0.0,6.4\n2026-10-13,1,19.5,0.9,0.0,0.4\n2026-10-13,2,19.5,1.4,0.0,2.7\n2026-10-13,3,19.5,2.0,0.0,4.2\n2026-10-13,4,19.4,1.6,0.0,5.0\n2026-10-13,5,19.5,0.5,0.0,2.2\n2026-10-13,6,19.1,1.8,0.0,7.4\n2026-10-13,7,18.7,2.5,0.0,12.6\n2026-10-13,8,18.6,2.9,0.0,11.8\n2026-10-13,9,16.8,1.1,0.0,9.2\n2026-10-13,10,17.1,0.6,0.0,6.7\n2026-10-13,11,17.2,0.7,0.0,7.9\n2026-10-13,12,17.8,0.0,0.0,4.0\n2026-10-13,13,20.4,0.0,0.0,8.1\n2026-10-13,14,20.9,0.0,0.0,15.9\n2026-10-13,15,20.2,0.0,0.0,15.8\n2026-10-13,16,20.2,0.0,0.0,16.2\n2026-10-13,17,19.5,0.0,0.0,16.0\n2026-10-13,18,18.6,0.0,0.0,17.3\n2026-10-13,19,18.3,0.0,0.0,18.0\n2026-10-13,20,17.3,0.0,0.0,19.3\n2026-10-13,21,16.4,0.0,0.0,16.5\n2026-10-13,22,16.0,0.0,0.0,18.1\n2026-10-13,23,15.7,0.0,0.0,20.8\n2026-10-14,0,15.1,0.0,0.0,20.1\n2026-10-14,1,14.4,0.0,0.0,21.0\n2026-10-14,2,13.7,0.0,0.0,19.6\n2026-10-14,3,12.8,0.0,0.0,18.3\n2026-10-14,4,12.1,0.0,0.0,17.7\n2026-10-14,5,11.5,0.0,0.0,17.3\n2026-10-14,6,10.9,0.0,0.0,16.2\n2026-10-14,7,10.4,0.0,0.0,14.0\n2026-10-14,8,11.5,0.0,0.0,10.5\n2026-10-14,9,13.8,0.0,0.0,10.2\n2026-10-14,10,15.3,0.0,0.0,11.1\n2026-10-14,11,16.4,0.0,0.0,11.6\n2026-10-14,12,17.3,0.0,0.0,12.2\n2026-10-14,13,17.9,0.0,0.0,13.2\n2026-10-14,14,18.2,0.0,0.0,15.2\n2026-10-14,15,18.2,0.0,0.0,14.5\n2026-10-14,16,18.3,0.0,0.0,12.5\n2026-10-14,17,18.1,0.0,0.0,13.6\n2026-10-14,18,17.2,0.0,0.0,14.9\n2026-10-14,19,17.0,0.0,0.0,16.5\n2026-10-14,20,16.4,0.0,0.0,15.3\n2026-10-14,21,15.9,0.0,0.0,9.5\n2026-10-14,22,15.8,0.0,0.0,9.4\n2026-10-14,23,16.2,0.0,0.0,9.6\n2026-10-15,0,16.1,0.0,0.0,10.4\n2026-10-15,1,16.0,0.0,0.0,8.4\n2026-10-15,2,15.3,0.0,0.0,4.3\n2026-10-15,3,14.3,0.0,0.0,4.3\n2026-10-15,4,13.4,0.0,0.0,6.8\n2026-10-15,5,12.9,0.0,0.0,10.6\n2026-10-15,6,12.6,0.0,0.0,12.1\n2026-10-15,7,12.7,0.0,0.0,12.1\n2026-10-15,8,14.2,0.0,0.0,12.4\n2026-10-15,9,14.3,0.0,0.0,5.6\n2026-10-15,10,14.4,1.0,0.0,12.8\n2026-10-15,11,19.1,0.1,0.0,13.2\n2026-10-15,12,20.6,0.0,0.0,20.2\n2026-10-15,13,21.1,0.0,0.0,21.1\n2026-10-15,14,19.4,0.0,0.0,19.1\n2026-10-15,15,20.0,0.0,0.0,21.3\n2026-10-15,16,20.0,0.0,0.0,24.4\n2026-10-15,17,19.5,0.0,0.0,23.2\n2026-10-15,18,18.9,0.0,0.0,19.6\n2026-10-15,19,18.3,0.0,0.0,17.7\n2026-10-15,20,17.8,0.0,0.0,15.3\n2026-10-15,21,17.4,0.0,0.0,13.9\n2026-10-15,22,16.8,0.0,0.0,12.9\n2026-10-15,23,16.5,0.0,0.0,12.9\n2026-10-16,0,16.6,0.0,0.0,11.4\n2026-10-16,1,16.9,0.1,0.0,11.7\n2026-10-16,2,16.7,0.1,0.0,9.9\n2026-10-16,3,15.8,0.2,0.0,8.9\n2026-10-16,4,15.4,0.3,0.0,8.6\n2026-10-16,5,15.4,0.4,0.0,9.4\n2026-10-16,6,14.9,0.1,0.0,9.7\n2026-10-16,7,14.8,0.0,0.0,8.4\n2026-10-16,8,15.1,0.0,0.0,9.4\n2026-10-16,9,16.6,0.0,0.0,8.8\n2026-10-16,10,18.3,0.2,0.0,7.7\n2026-10-16,11,19.2,0.4,0.0,7.6\n2026-10-16,12,20.4,0.3,0.0,6.1\n2026-10-16,13,21.3,0.0,0.0,5.2\n2026-10-16,14,21.8,0.1,0.0,6.9\n2026-10-16,15,22.4,0.2,0.0,7.0\n2026-10-16,16,22.1,0.0,0.0,6.2\n2026-10-16,17,21.4,0.0,0.0,4.9\n2026-10-16,18,19.5,0.0,0.0,3.6\n2026-10-16,19,18.7,0.0,0.0,4.3\n2026-10-16,20,18.0,0.0,0.0,4.3\n2026-10-16,21,17.6,0.0,0.0,4.0\n2026-10-16,22,17.0,0.0,0.0,5.7\n2026-10-16,23,16.6,0.0,0.0,4.8\n2026-10-17,0,16.0,0.0,0.0,4.3\n2026-10-17,1,15.7,0.0,0.0,4.6\n2026-10-17,2,15.5,0.0,0.0,5.3\n2026-10-17,3,15.4,0.0,0.0,6.5\n2026-10-17,4,15.2,0.0,0.0,6.5\n2026-10-17,5,14.9,0.0,0.0,6.9\n2026-10-17,6,14.7,0.0,0.0,7.8\n2026-10-17,7,14.6,0.0,0.0,7.1\n2026-10-17,8,15.2,0.0,0.0,7.7\n2026-10-17,9,16.8,0.0,0.0,10.0\n2026-10-17,10,20.0,0.0,0.0,11.2\n2026-10-17,11,22.0,0.0,0.0,13.1\n2026-10-17,12,23.3,0.0,0.0,15.4\n2026-10-17,13,24.2,0.0,0.0,14.6\n2026-10-17,14,24.6,0.0,0.0,15.3\n2026-10-17,15,24.9,0.0,0.0,13.1\n2026-10-17,16,24.7,0.0,0.0,12.5\n2026-10-17,17,23.6,0.0,0.0,11.3\n2026-10-17,18,21.6,0.0,0.0,9.6\n2026-10-17,19,20.9,0.0,0.0,12.3\n2026-10-17,20,21.0,0.0,0.0,21.4\n2026-10-17,21,20.5,0.0,0.0,22.3\n2026-10-17,22,19.8,0.0,0.0,22.3\n2026-10-17,23,19.1,0.0,0.0,22.8\n2026-10-18,0,18.5,0.0,0.0,22.5\n2026-10-18,1,18.2,0.0,0.0,22.8\n2026-10-18,2,18.1,0.0,0.0,23.0\n2026-10-18,3,18.2,0.0,0.0,22.8\n2026-10-18,4,18.4,0.0,0.0,21.5\n2026-10-18,5,18.4,0.0,0.0,23.0\n2026-10-18,6,17.3,0.8,0.0,20.5\n2026-10-18,7,17.7,0.0,0.0,20.6\n2026-10-18,8,18.7,0.0,0.0,23.3\n2026-10-18,9,17.2,0.0,0.0,25.1\n2026-10-18,10,20.0,0.1,0.0,28.1\n2026-10-18,11,20.7,0.4,0.0,25.9\n2026-10-18,12,20.4,2.5,0.0,28.5\n2026-10-18,13,21.0,3.9,0.0,32.7\n2026-10-18,14,21.0,6.7,0.0,37.3\n2026-10-18,15,21.0,3.1,0.0,31.6\n2026-10-18,16,19.2,6.3,0.0,21.6\n2026-10-18,17,18.9,2.7,0.0,18.5\n2026-10-18,18,17.8,1.6,0.0,19.4\n2026-10-18,19,17.6,0.1,0.0,15.2\n2026-10-18,20,17.7,0.0,0.0,14.8\n2026-10-18,21,17.1,0.0,0.0,13.0\n2026-10-18,22,16.4,0.0,0.0,12.6\n2026-10-18,23,15.8,0.0,0.0,13.0\n2026-10-19,0,15.5,0.0,0.0,12.3\n2026-10-19,1,14.7,0.0,0.0,11.6\n2026-10-19,2,13.7,0.0,0.0,10.9\n2026-10-19,3,13.4,0.0,0.0,9.6\n2026-10-19,4,12.5,0.0,0.0,9.9\n2026-10-19,5,11.9,0.0,0.0,8.9\n2026-10-19,6,11.2,0.0,0.0,7.1\n2026-10-19,7,10.8,0.0,0.0,6.9\n2026-10-19,8,12.0,0.0,0.0,5.4\n2026-10-19,9,13.2,0.0,0.0,6.4\n2026-10-19,10,14.7,0.0,0.0,4.9\n2026-10-19,11,15.0,0.0,0.0,5.5\n2026-10-19,12,15.8,0.0,0.0,8.0\n2026-10-19,13,16.8,0.0,0.0,7.0\n2026-10-19,14,17.3,0.0,0.0,5.2\n2026-10-19,15,17.0,0.0,0.0,3.3\n2026-10-19,16,16.6,0.0,0.0,4.7\n2026-10-19,17,16.3,0.0,0.0,5.2\n2026-10-19,18,15.5,0.0,0.0,6.2\n2026-10-19,19,14.6,0.0,0.0,6.2\n2026-10-19,20,13.7,0.0,0.0,7.7\n2026-10-19,21,14.2,0.0,0.0,6.3\n2026-10-19,22,13.7,0.0,0.0,7.1\n2026-10-19,23,13.0,0.0,0.0,5.5\n2026-10-20,0,12.8,0.0,0.0,7.4\n2026-10-20,1,12.0,0.0,0.0,5.4\n2026-10-20,2,12.5,0.0,0.0,4.5\n2026-10-20,3,11.6,0.0,0.0,3.7\n2026-10-20,4,12.3,0.1,0.0,6.7\n2026-10-20,5,12.0,0.0,0.0,5.3\n2026-10-20,6,11.0,0.0,0.0,8.6\n2026-10-20,7,10.7,0.0,0.0,7.9\n2026-10-20,8,11.9,0.0,0.0,6.3\n2026-10-20,9,14.3,0.0,0.0,5.0\n2026-10-20,10,16.5,0.0,0.0,2.6\n2026-10-20,11,18.1,0.0,0.0,0.2\n2026-10-20,12,19.4,0.0,0.0,1.2\n2026-10-20,13,20.2,0.0,0.0,4.8\n2026-10-20,14,20.2,0.0,0.0,7.8\n2026-10-20,15,20.7,0.0,0.0,4.0\n2026-10-20,16,20.0,0.0,0.0,8.1\n2026-10-20,17,18.3,0.0,0.0,10.6\n2026-10-20,18,17.1,0.0,0.0,9.1\n2026-10-20,19,16.4,0.1,0.0,8.5\n2026-10-20,20,16.2,0.0,0.0,8.2\n2026-10-20,21,16.1,0.0,0.0,4.5\n2026-10-20,22,16.0,0.1,0.0,6.2\n2026-10-20,23,15.4,0.0,0.0,6.0\n2026-10-21,0,15.5,0.0,0.0,5.9\n2026-10-21,1,15.3,0.0,0.0,7.2\n2026-10-21,2,14.5,0.0,0.0,8.9\n2026-10-21,3,14.1,0.0,0.0,8.6\n2026-10-21,4,14.1,0.0,0.0,9.1\n2026-10-21,5,15.2,0.0,0.0,9.2\n2026-10-21,6,14.4,0.4,0.0,8.7\n2026-10-21,7,14.3,0.4,0.0,13.5\n2026-10-21,8,13.7,0.5,0.0,17.9\n2026-10-21,9,14.0,0.0,0.0,17.2\n2026-10-21,10,14.9,0.0,0.0,22.9\n2026-10-21,11,15.5,0.0,0.0,26.2\n2026-10-21,12,15.6,0.0,0.0,26.3\n2026-10-21,13,15.3,0.0,0.0,27.3\n2026-10-21,14,14.9,0.0,0.0,25.0\n2026-10-21,15,14.4,0.0,0.0,23.8\n2026-10-21,16,13.7,0.0,0.0,23.5\n2026-10-21,17,12.7,0.0,0.0,20.0\n2026-10-21,18,11.1,0.0,0.0,20.6\n2026-10-21,19,9.9,0.0,0.0,15.8\n2026-10-21,20,9.1,0.0,0.0,9.6\n2026-10-21,21,8.6,0.0,0.0,10.3\n2026-10-21,22,7.7,0.0,0.0,9.4\n2026-10-21,23,7.6,0.0,0.0,11.1\n2026-10-22,0,7.4,0.0,0.0,6.8\n2026-10-22,1,7.6,0.0,0.0,9.4\n2026-10-22,2,7.1,0.0,0.0,12.9\n2026-10-22,3,6.1,0.0,0.0,8.9\n2026-10-22,4,5.5,0.0,0.0,7.6\n2026-10-22,5,5.1,0.0,0.0,7.8\n2026-10-22,6,4.6,0.0,0.0,6.7\n2026-10-22,7,4.4,0.0,0.0,7.5\n2026-10-22,8,5.9,0.0,0.0,8.2\n2026-10-22,9,7.3,0.0,0.0,6.6\n2026-10-22,10,9.1,0.0,0.0,7.4\n2026-10-22,11,10.3,0.0,0.0,7.9\n2026-10-22,12,11.2,0.0,0.0,8.1\n2026-10-22,13,12.1,0.0,0.0,8.5\n2026-10-22,14,12.7,0.0,0.0,9.0\n2026-10-22,15,13.8,0.0,0.0,7.3\n2026-10-22,16,13.5,0.0,0.0,6.9\n2026-10-22,17,12.8,0.0,0.0,5.2\n2026-10-22,18,10.8,0.0,0.0,4.9\n2026-10-22,19,10.4,0.0,0.0,4.5\n2026-10-22,20,10.1,0.0,0.0,3.1\n2026-10-22,21,9.2,0.0,0.0,4.4\n2026-10-22,22,8.6,0.0,0.0,3.8\n2026-10-22,23,8.5,0.0,0.0,7.8\n2026-10-23,0,8.1,0.0,0.0,7.3\n2026-10-23,1,8.0,0.0,0.0,7.7\n2026-10-23,2,7.8,0.0,0.0,6.7\n2026-10-23,3,7.2,0.0,0.0,6.7\n2026-10-23,4,6.8,0.0,0.0,6.9\n2026-10-23,5,6.8,0.0,0.0,8.6\n2026-10-23,6,7.0,0.0,0.0,8.3\n2026-10-23,7,6.9,0.0,0.0,7.6\n2026-10-23,8,8.5,0.0,0.0,8.6\n2026-10-23,9,10.6,0.0,0.0,8.2\n2026-10-23,10,13.5,0.0,0.0,5.5\n2026-10-23,11,15.6,0.0,0.0,6.8\n2026-10-23,12,16.9,0.0,0.0,7.0\n2026-10-23,13,18.3,0.0,0.0,5.9\n2026-10-23,14,19.7,0.0,0.0,10.1\n2026-10-23,15,20.5,0.0,0.0,4.9\n2026-10-23,16,19.9,0.0,0.0,7.4\n2026-10-23,17,18.0,0.0,0.0,10.1\n2026-10-23,18,16.0,0.0,0.0,8.7\n2026-10-23,19,14.9,0.0,0.0,3.9\n2026-10-23,20,15.2,0.0,0.0,6.8\n2026-10-23,21,14.9,0.0,0.0,4.4\n2026-10-23,22,14.6,0.0,0.0,3.4\n2026-10-23,23,15.1,0.0,0.0,5.9\n2026-10-24,0,14.8,0.0,0.0,7.3\n2026-10-24,1,14.9,0.0,0.0,8.8\n2026-10-24,2,15.4,0.0,0.0,10.2\n2026-10-24,3,15.1,0.0,0.0,6.2\n2026-10-24,4,14.7,0.0,0.0,5.4\n2026-10-24,5,14.6,0.0,0.0,5.8\n2026-10-24,6,14.4,0.0,0.0,6.1\n2026-10-24,7,14.2,0.0,0.0,5.0\n2026-10-24,8,14.7,0.0,0.0,5.3\n2026-10-24,9,15.1,0.0,0.0,5.9\n2026-10-24,10,15.8,0.1,0.0,5.7\n2026-10-24,11,16.5,0.3,0.0,5.5\n2026-10-24,12,17.1,0.2,0.0,4.2\n2026-10-24,13,17.3,0.2,0.0,4.3\n2026-10-24,14,17.3,0.2,0.0,6.7\n2026-10-24,15,16.3,0.0,0.0,6.1\n2026-10-24,16,16.2,0.2,0.0,7.8\n2026-10-24,17,15.9,0.4,0.0,6.1\n2026-10-24,18,15.6,0.2,0.0,7.4\n2026-10-24,19,15.5,0.2,0.0,6.7\n2026-10-24,20,15.1,0.1,0.0,5.3\n2026-10-24,21,15.5,0.1,0.0,4.8\n2026-10-24,22,15.2,0.0,0.0,6.1\n2026-10-24,23,14.8,0.0,0.0,6.5\n2026-10-25,0,14.5,0.0,0.0,7.7\n2026-10-25,1,14.2,0.0,0.0,7.8\n2026-10-25,2,13.7,0.0,0.0,10.3\n2026-10-25,3,12.8,0.0,0.0,9.9\n2026-10-25,4,12.3,0.0,0.0,8.4\n2026-10-25,5,11.9,0.0,0.0,9.0\n2026-10-25,6,11.9,0.0,0.0,9.4\n2026-10-25,7,11.5,0.0,0.0,7.4\n2026-10-25,8,13.0,0.0,0.0,6.2\n2026-10-25,9,14.4,0.0,0.0,6.4\n2026-10-25,10,17.2,0.0,0.0,6.9\n2026-10-25,11,19.5,0.0,0.0,8.7\n2026-10-25,12,21.2,0.0,0.0,9.8\n2026-10-25,13,22.1,0.0,0.0,8.0\n2026-10-25,14,22.5,0.0,0.0,8.3\n2026-10-25,15,22.7,0.0,0.0,9.3\n2026-10-25,16,22.7,0.0,0.0,8.9\n2026-10-25,17,21.5,0.0,0.0,6.2\n2026-10-25,18,20.2,0.0,0.0,10.4\n2026-10-25,19,18.9,0.0,0.0,14.4\n2026-10-25,20,18.0,0.0,0.0,13.5\n2026-10-25,21,18.2,0.0,0.0,13.2\n2026-10-25,22,17.1,0.1,0.0,13.4\n2026-10-25,23,16.4,0.0,0.0,13.4\n2026-10-26,0,15.6,0.0,0.0,12.8\n2026-10-26,1,15.2,0.0,0.0,12.1\n2026-10-26,2,14.7,0.1,0.0,11.7\n2026-10-26,3,14.8,0.8,0.0,10.4\n2026-10-26,4,13.9,0.8,0.0,12.2\n2026-10-26,5,13.7,0.4,0.0,12.2\n2026-10-26,6,13.5,0.2,0.0,11.8\n2026-10-26,7,13.5,0.0,0.0,10.1\n2026-10-26,8,13.9,0.0,0.0,9.3\n2026-10-26,9,14.8,0.0,0.0,10.7\n2026-10-26,10,16.8,0.0,0.0,9.6\n2026-10-26,11,18.2,0.0,0.0,7.4\n2026-10-26,12,18.6,0.0,0.0,5.8\n2026-10-26,13,18.9,0.0,0.0,7.3\n2026-10-26,14,18.9,0.0,0.0,8.3\n2026-10-26,15,20.6,0.0,0.0,7.3\n2026-10-26,16,20.1,0.0,0.0,8.9\n2026-10-26,17,19.2,0.0,0.0,7.6\n2026-10-26,18,17.3,0.0,0.0,9.0\n2026-10-26,19,16.4,0.0,0.0,8.6\n2026-10-26,20,15.9,0.0,0.0,10.0\n2026-10-26,21,15.9,0.0,0.0,9.8\n2026-10-26,22,16.3,0.0,0.0,11.6\n2026-10-26,23,15.8,0.0,0.0,12.9\n2026-10-27,0,14.8,0.0,0.0,13.8\n2026-10-27,1,13.7,0.0,0.0,14.2\n2026-10-27,2,13.0,0.0,0.0,15.9\n2026-10-27,3,12.2,0.0,0.0,17.0\n2026-10-27,4,11.5,0.0,0.0,17.1\n2026-10-27,5,10.7,0.0,0.0,16.6\n2026-10-27,6,10.0,0.0,0.0,15.5\n2026-10-27,7,9.5,0.0,0.0,15.7\n2026-10-27,8,10.3,0.0,0.0,17.4\n2026-10-27,9,11.7,0.0,0.0,14.1\n2026-10-27,10,13.5,0.0,0.0,14.2\n2026-10-27,11,14.6,0.0,0.0,16.8\n2026-10-27,12,15.5,0.0,0.0,18.6\n2026-10-27,13,15.6,0.0,0.0,22.0\n2026-10-27,14,15.1,0.0,0.0,23.5\n2026-10-27,15,14.9,0.0,0.0,22.1\n2026-10-27,16,13.6,0.0,0.0,21.6\n2026-10-27,17,12.2,0.0,0.0,19.5\n2026-10-27,18,10.5,0.0,0.0,18.9\n2026-10-27,19,9.5,0.0,0.0,19.4\n2026-10-27,20,8.8,0.0,0.0,17.0\n2026-10-27,21,8.8,0.0,0.0,15.2\n2026-10-27,22,8.4,0.0,0.0,14.5\n2026-10-27,23,8.2,0.0,0.0,16.4\n2026-10-28,0,7.3,0.0,0.0,16.4\n2026-10-28,1,6.4,0.0,0.0,15.8\n2026-10-28,2,5.5,0.0,0.0,16.2\n2026-10-28,3,4.6,0.0,0.0,14.2\n2026-10-28,4,3.9,0.0,0.0,13.9\n2026-10-28,5,3.4,0.0,0.0,13.1\n2026-10-28,6,2.9,0.0,0.0,12.0\n2026-10-28,7,2.4,0.0,0.0,10.9\n2026-10-28,8,3.5,0.0,0.0,8.9\n2026-10-28,9,5.2,0.0,0.0,8.3\n2026-10-28,10,7.0,0.0,0.0,7.0\n2026-10-28,11,9.0,0.0,0.0,5.2\n2026-10-28,12,10.5,0.0,0.0,5.3\n2026-10-28,13,11.5,0.0,0.0,6.9\n2026-10-28,14,12.1,0.0,0.0,8.0\n2026-10-28,15,12.6,0.0,0.0,7.3\n2026-10-28,16,11.9,0.0,0.0,8.6\n2026-10-28,17,10.8,0.0,0.0,10.1\n2026-10-28,18,9.1,0.0,0.0,9.3\n2026-10-28,19,8.3,0.0,0.0,7.4\n2026-10-28,20,7.7,0.0,0.0,5.8\n2026-10-28,21,7.6,0.0,0.0,4.2\n2026-10-28,22,7.3,0.0,0.0,4.2\n2026-10-28,23,7.0,0.0,0.0,3.4\n2026-10-29,0,6.3,0.0,0.0,4.5\n2026-10-29,1,5.9,0.0,0.0,6.7\n2026-10-29,2,5.4,0.0,0.0,5.8\n2026-10-29,3,5.0,0.0,0.0,6.5\n2026-10-29,4,4.4,0.0,0.0,6.5\n2026-10-29,5,4.8,0.0,0.0,10.0\n2026-10-29,6,5.0,0.0,0.0,10.7\n2026-10-29,7,5.0,0.0,0.0,13.3\n2026-10-29,8,5.9,0.0,0.0,11.9\n2026-10-29,9,6.5,0.0,0.0,11.6\n2026-10-29,10,5.7,1.1,0.0,16.4\n2026-10-29,11,7.9,0.2,0.0,12.2\n2026-10-29,12,8.7,0.0,0.0,8.5\n2026-10-29,13,11.2,0.0,0.0,9.9\n2026-10-29,14,12.2,0.0,0.0,10.7\n2026-10-29,15,11.6,0.0,0.0,9.5\n2026-10-29,16,11.4,0.0,0.0,10.0\n2026-10-29,17,11.5,0.0,0.0,10.3\n2026-10-29,18,10.6,0.0,0.0,11.0\n2026-10-29,19,11.0,0.0,0.0,11.0\n2026-10-29,20,9.7,0.1,0.0,12.5\n2026-10-29,21,8.9,0.0,0.0,12.1\n2026-10-29,22,8.9,1.0,0.0,9.7\n2026-10-29,23,8.9,0.4,0.0,10.1\n2026-10-30,0,8.6,0.0,0.0,9.9\n2026-10-30,1,8.1,0.0,0.0,7.7\n2026-10-30,2,7.7,0.0,0.0,6.5\n2026-10-30,3,6.8,0.0,0.0,5.8\n2026-10-30,4,6.9,0.0,0.0,6.1\n2026-10-30,5,7.0,0.0,0.0,4.5\n2026-10-30,6,7.2,0.0,0.0,5.9\n2026-10-30,7,7.1,0.0,0.0,6.0\n2026-10-30,8,8.8,0.0,0.0,8.8\n2026-10-30,9,11.3,0.0,0.0,6.7\n2026-10-30,10,13.6,0.0,0.0,6.3\n2026-10-30,11,15.5,0.0,0.0,7.3\n2026-10-30,12,16.4,0.0,0.0,9.9\n2026-10-30,13,17.0,0.0,0.0,13.3\n2026-10-30,14,17.4,0.0,0.0,15.4\n2026-10-30,15,17.5,0.0,0.0,14.7\n2026-10-30,16,17.0,0.0,0.0,14.4\n2026-10-30,17,16.0,0.0,0.0,12.2\n2026-10-30,18,14.6,0.0,0.0,11.1\n2026-10-30,19,14.3,0.0,0.0,10.7\n2026-10-30,20,14.0,0.0,0.0,9.5\n2026-10-30,21,13.6,0.0,0.0,8.0\n2026-10-30,22,13.1,0.0,0.0,9.3\n2026-10-30,23,13.2,0.0,0.0,10.3\n2026-10-31,0,13.3,0.0,0.0,9.2\n2026-10-31,1,14.0,0.3,0.0,11.8\n2026-10-31,2,15.0,0.0,0.0,16.4\n2026-10-31,3,12.1,0.0,0.0,7.0\n2026-10-31,4,12.7,0.0,0.0,6.0\n2026-10-31,5,11.9,0.0,0.0,6.8\n2026-10-31,6,11.7,0.0,0.0,12.9\n2026-10-31,7,10.3,0.0,0.0,11.2\n2026-10-31,8,10.3,0.0,0.0,11.0\n2026-10-31,9,11.8,0.0,0.0,10.3\n2026-10-31,10,13.2,0.0,0.0,13.1\n2026-10-31,11,14.0,0.0,0.0,14.4\n2026-10-31,12,14.5,0.0,0.0,15.5\n2026-10-31,13,14.7,0.0,0.0,14.0\n2026-10-31,14,14.6,0.0,0.0,14.0\n2026-10-31,15,14.6,0.0,0.0,13.1\n2026-10-31,16,13.8,0.0,0.0,13.0\n2026-10-31,17,12.6,0.0,0.0,10.2\n2026-10-31,18,10.8,0.0,0.0,7.2\n2026-10-31,19,10.1,0.0,0.0,5.0\n2026-10-31,20,9.7,0.0,0.0,4.4\n2026-10-31,21,9.3,0.0,0.0,3.6\n2026-10-31,22,8.5,0.0,0.0,6.4\n2026-10-31,23,8.3,0.0,0.0,3.7\n", "ssg_home_schedule.csv": "date,start_hour\n2025-03-22,14\n2025-03-23,14\n2025-03-25,18\n2025-03-26,18\n2025-03-27,18\n2025-04-04,18\n2025-04-05,17\n2025-04-06,14\n2025-04-15,18\n2025-04-16,18\n2025-04-17,18\n2025-04-18,18\n2025-04-19,14\n2025-04-20,14\n2025-04-25,18\n2025-04-26,17\n2025-04-27,14\n2025-04-29,18\n2025-04-30,18\n2025-05-01,18\n2025-05-09,18\n2025-05-10,14\n2025-05-11,14\n2025-05-13,18\n2025-05-14,18\n2025-05-15,18\n2025-05-23,18\n2025-05-24,17\n2025-05-25,14\n2025-05-27,18\n2025-05-28,18\n2025-05-29,18\n2025-06-03,17\n2025-06-04,18\n2025-06-05,18\n2025-06-13,18\n2025-06-14,17\n2025-06-15,17\n2025-06-20,18\n2025-06-21,17\n2025-06-22,17\n2025-06-27,18\n2025-06-28,17\n2025-06-29,17\n2025-07-08,18\n2025-07-09,18\n2025-07-10,18\n2025-07-17,18\n2025-07-18,18\n2025-07-19,18\n2025-07-20,18\n2025-07-29,18\n2025-07-30,18\n2025-07-31,18\n2025-08-05,18\n2025-08-06,18\n2025-08-07,18\n2025-08-12,18\n2025-08-13,18\n2025-08-14,18\n2025-08-15,18\n2025-08-16,18\n2025-08-17,18\n2025-08-26,18\n2025-08-27,18\n2025-08-28,18\n2025-08-29,18\n2025-08-30,18\n2025-08-31,18\n2026-03-28,14\n2026-03-29,14\n2026-03-31,18\n2026-04-01,18\n2026-04-02,18\n2026-04-07,18\n2026-04-08,18\n2026-04-09,18\n2026-04-14,18\n2026-04-15,18\n2026-04-16,18\n2026-04-24,18\n2026-04-25,17\n2026-04-26,14\n2026-05-01,17\n2026-05-02,14\n2026-05-03,14\n2026-05-05,14\n2026-05-06,18\n2026-05-07,18\n2026-05-15,18\n2026-05-16,17\n2026-05-17,14\n2026-05-26,18\n2026-05-27,18\n2026-05-28,18\n2026-06-02,18\n2026-06-03,17\n2026-06-04,18\n2026-06-05,18\n2026-06-06,14\n2026-06-07,17\n2026-06-16,18\n2026-06-17,18\n2026-06-18,18\n2026-06-26,18\n2026-06-27,17\n2026-06-28,17\n2026-07-03,18\n2026-07-04,18\n2026-07-05,18\n2026-07-16,18\n2026-07-17,18\n2026-07-18,18\n2026-07-19,18\n2026-07-24,18\n2026-07-25,18\n2026-07-26,18\n2026-07-28,18\n2026-07-29,18\n2026-07-30,18\n2026-08-04,18\n2026-08-05,18\n2026-08-06,18\n2026-08-11,19\n2026-08-12,19\n2026-08-13,19\n2026-08-21,19\n2026-08-22,19\n2026-08-23,19\n2026-08-25,18\n2026-08-26,18\n2026-08-27,18\n2026-09-04,18\n2026-09-05,17\n2026-09-06,17\n", "ssg_home_actual.csv": "date,start_hour\n2025-03-22,14\n2025-03-23,14\n2025-03-25,18\n2025-03-26,18\n2025-03-27,18\n2025-04-04,18\n2025-04-06,14\n2025-04-15,18\n2025-04-16,18\n2025-04-17,18\n2025-04-18,18\n2025-04-19,14\n2025-04-20,14\n2025-04-25,18\n2025-04-26,17\n2025-04-27,14\n2025-04-29,18\n2025-04-30,18\n2025-05-01,18\n2025-05-10,17\n2025-05-11,14\n2025-05-13,18\n2025-05-14,18\n2025-05-15,18\n2025-05-23,18\n2025-05-24,17\n2025-05-25,14\n2025-05-27,18\n2025-05-28,18\n2025-05-29,18\n2025-06-03,17\n2025-06-04,18\n2025-06-05,18\n2025-06-14,17\n2025-06-15,17\n2025-06-21,17\n2025-06-22,17\n2025-06-27,18\n2025-06-28,17\n2025-06-29,17\n2025-07-08,18\n2025-07-09,18\n2025-07-10,18\n2025-07-19,18\n2025-07-20,18\n2025-07-29,18\n2025-07-30,18\n2025-07-31,18\n2025-08-05,18\n2025-08-06,18\n2025-08-07,18\n2025-08-12,18\n2025-08-14,18\n2025-08-15,18\n2025-08-16,18\n2025-08-17,18\n2025-08-26,18\n2025-08-27,18\n2025-08-28,18\n2025-08-29,18\n2025-08-30,18\n2025-08-31,18\n2025-09-02,18\n2025-09-05,18\n2025-09-20,17\n2025-09-21,14\n2025-09-22,18\n2025-09-23,18\n2025-09-25,18\n2025-09-26,18\n2025-09-29,18\n2025-10-01,18\n2026-03-28,14\n2026-03-29,14\n2026-03-31,18\n2026-04-01,18\n2026-04-02,18\n2026-04-07,18\n2026-04-08,18\n2026-04-14,18\n2026-04-15,18\n2026-04-16,18\n2026-04-24,18\n2026-04-25,17\n2026-04-26,14\n2026-05-01,17\n2026-05-02,14\n2026-05-03,14\n2026-05-05,14\n2026-05-06,18\n2026-05-07,18\n2026-05-15,18\n2026-05-16,17\n2026-05-17,14\n2026-05-27,18\n2026-05-28,18\n2026-06-02,18\n2026-06-03,17\n2026-06-04,18\n2026-06-05,18\n2026-06-06,14\n2026-06-07,17\n2026-06-16,18\n2026-06-17,18\n2026-06-18,18\n2026-06-26,18\n2026-06-27,17\n2026-06-28,17\n2026-07-03,18\n2026-07-04,18\n2026-07-05,18\n2026-07-16,18\n2026-07-17,18\n2026-07-18,18\n2026-07-19,18\n2026-07-24,18\n2026-07-25,18\n2026-07-26,18\n2026-07-28,18\n2026-07-29,18\n2026-07-30,18\n2026-08-04,18\n2026-08-11,19\n2026-08-12,19\n2026-08-13,19\n2026-08-21,19\n2026-08-23,19\n2026-08-25,18\n2026-08-26,18\n2026-08-27,18\n2026-09-04,18\n2026-09-05,17\n2026-09-06,17\n2026-09-10,18\n2026-09-15,18\n2026-09-20,14\n2026-09-22,18\n2026-09-24,17\n2026-09-29,18\n2026-09-30,18\n2026-10-01,18\n2026-10-07,18\n2026-10-09,14\n", "bucheon_home_schedule.csv": "date,start_hour\n2025-02-23,16\n2025-03-08,14\n2025-03-16,16\n2025-04-12,16\n2025-04-26,16\n2025-05-04,14\n2025-05-17,16\n2025-06-14,19\n2025-06-28,19\n2025-07-13,19\n2025-07-20,19\n2025-08-02,19\n2025-08-23,19\n2025-09-07,19\n2025-09-20,16\n2025-10-08,16\n2025-10-19,16\n2025-11-01,16\n2025-11-23,14\n2025-11-30,14\n2026-03-07,16\n2026-03-15,14\n2026-03-18,19\n2026-04-18,16\n2026-04-25,16\n2026-05-05,14\n2026-05-13,19\n2026-05-17,19\n2026-07-19,19\n2026-07-22,19\n2026-08-08,20\n2026-08-16,19\n2026-08-22,19\n2026-08-29,19\n2026-09-05,19\n2026-09-12,16\n2026-09-19,16\n2026-10-24,14\n", "bucheon_home_actual.csv": "date,start_hour\n2025-02-23,16\n2025-03-08,14\n2025-03-16,16\n2025-04-12,16\n2025-04-26,16\n2025-05-04,14\n2025-05-17,16\n2025-06-14,19\n2025-06-28,19\n2025-07-13,19\n2025-07-20,19\n2025-08-02,19\n2025-08-23,19\n2025-09-07,19\n2025-09-20,16\n2025-10-08,16\n2025-10-19,16\n2025-11-01,16\n2025-11-23,14\n2025-11-30,14\n2026-03-07,16\n2026-03-15,14\n2026-03-18,19\n2026-04-18,16\n2026-04-25,16\n2026-05-05,14\n2026-05-13,19\n2026-05-17,19\n2026-07-19,19\n2026-07-22,19\n2026-08-08,20\n2026-08-16,19\n2026-08-22,19\n2026-08-29,19\n2026-09-05,19\n2026-09-12,16\n2026-09-19,16\n2026-10-24,14\n"}
for name, text in AUX.items():
    (WORK / name).write_text(text, encoding="utf-8")
print({k: len(v) for k, v in AUX.items()})

In [ ]:
# 코드 내장: sincon_v2.py (모델), final_post.py (혼합·보정)
CODES = {"sincon_v2.py": "\"\"\"SINCON 2026 v2 파이프라인 (단일 파일, Kaggle/로컬 공용).\n\nCONFIG 는 커널 생성 시 파일 맨 위 CONFIG_OVERRIDE 로 덮어쓴다.\n  mode      : \"cv\" (검증 폴드 점수 + 예측 저장) | \"full\" (6/30 기준 학습, 제출 파일)\n  model     : \"lgb\" (CPU) | \"xgb\" (GPU 있으면 cuda)\n  groups    : 사용할 피처 그룹 목록\n  folds     : cv 에서 돌릴 폴드\n  seeds, rounds, anchor_step\n\"\"\"\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nCONFIG = dict(mode=\"cv\", model=\"lgb\", groups=[\"base\", \"newst\", \"graph\", \"analog\"], folds=[\"A\", \"B\", \"C\"],\n              seeds=[0], rounds=1000, lr=0.05, anchor_step=7, cs=True, tag=\"v2\")\nCONFIG.update(globals().get(\"CONFIG_OVERRIDE\", {}))\n\nON_KAGGLE = bool(os.environ.get(\"KAGGLE_KERNEL_RUN_TYPE\"))\ntry:\n    _ROOT = Path(__file__).resolve().parent.parent\nexcept NameError:  # 노트북/exec 실행\n    _ROOT = Path.cwd()\n# 탐색 순서: SINCON_DIRS(콜론 구분) -> 작업 폴더 -> /kaggle/input -> 로컬 프로젝트 data/geo\n_DIRS = [Path(x) for x in os.environ.get(\"SINCON_DIRS\", \"\").split(\":\") if x] + [Path.cwd()]\nif ON_KAGGLE:\n    _DIRS += [Path(\"/kaggle/working\")]\n_DIRS += [_ROOT / \"data\", _ROOT / \"geo\"]\n\n\ndef _find(name):\n    for d in _DIRS:\n        if (d / name).exists():\n            return d / name\n    if Path(\"/kaggle/input\").exists():\n        hits = sorted(Path(\"/kaggle/input\").glob(f\"**/{name}\"))\n        if hits:\n            return hits[0]\n    return None\n\n\nOUT = Path(\"/kaggle/working\") if ON_KAGGLE else Path(os.environ.get(\"SINCON_OUT\", _ROOT / \"out\"))\nOUT.mkdir(exist_ok=True)\nLOG = open(OUT / f\"log_{CONFIG['tag']}.txt\", \"w\", encoding=\"utf-8\")\n\n\ndef p(*a):\n    s = \" \".join(str(x) for x in a)\n    print(s, flush=True)\n    LOG.write(s + \"\\n\")\n    LOG.flush()\n\n\nHORIZON = 123\nHOLIDAYS = pd.to_datetime([\n    \"2025-01-01\", \"2025-01-27\", \"2025-01-28\", \"2025-01-29\", \"2025-01-30\",\n    \"2025-03-01\", \"2025-03-03\", \"2025-05-05\", \"2025-05-06\", \"2025-06-03\", \"2025-06-06\",\n    \"2025-08-15\", \"2025-10-03\", \"2025-10-05\", \"2025-10-06\", \"2025-10-07\", \"2025-10-08\",\n    \"2025-10-09\", \"2025-12-25\",\n    \"2026-01-01\", \"2026-02-16\", \"2026-02-17\", \"2026-02-18\", \"2026-03-01\", \"2026-03-02\",\n    \"2026-05-01\", \"2026-05-05\", \"2026-05-24\", \"2026-05-25\", \"2026-06-03\", \"2026-06-06\",\n    \"2026-07-17\", \"2026-08-15\", \"2026-08-17\", \"2026-09-24\", \"2026-09-25\", \"2026-09-26\",\n    \"2026-10-03\", \"2026-10-05\", \"2026-10-09\",\n])\nSOFT_HOLIDAYS = pd.to_datetime([\"2025-05-01\"])\nBIG_HOLIDAYS = pd.to_datetime([\"2025-01-29\", \"2025-10-06\", \"2026-02-17\", \"2026-09-25\"])\n# 방학 (연초 공표 학사일정 기준 근사). 대학: 인천대·인하대·연세대 국제캠 / 초중고: 인천 일반 학교\nUNIV_VAC = [(\"2024-12-21\", \"2025-03-02\"), (\"2025-06-21\", \"2025-08-31\"), (\"2025-12-20\", \"2026-03-01\"),\n            (\"2026-06-20\", \"2026-08-31\"), (\"2026-12-19\", \"2027-02-28\")]\nSCHOOL_VAC = [(\"2024-12-28\", \"2025-03-03\"), (\"2025-07-25\", \"2025-08-17\"), (\"2026-01-02\", \"2026-03-02\"),\n              (\"2026-07-24\", \"2026-08-16\"), (\"2027-01-01\", \"2027-03-01\")]\nUNIV_START = pd.to_datetime([\"2024-09-02\", \"2025-03-03\", \"2025-09-01\", \"2026-03-02\", \"2026-09-01\"])\nHSET = set(HOLIDAYS) | set(SOFT_HOLIDAYS)\nINT_COLS = {\"station_id\", \"line\", \"hour\", \"dow\", \"daytype\", \"month\", \"dom\", \"doy\", \"woy\", \"horizon\", \"hol_prev\",\n            \"hol_next\", \"bridge\", \"after_long_off\", \"big_hol_rel\", \"is_new\", \"univ_vac\", \"school_vac\"}\nCAT_COLS = (\"station_id\", \"line\")\nFOLDS = {\"A\": (\"2025-06-30\", \"2025-10-31\"), \"B\": (\"2026-02-28\", \"2026-06-30\"), \"C\": (\"2025-10-31\", \"2026-02-28\")}\n\n\ndef load():\n    train = pd.read_csv(_find(\"train.csv\"), parse_dates=[\"date\"])\n    test = pd.read_csv(_find(\"test.csv\"), parse_dates=[\"date\"])\n    meta = pd.read_csv(_find(\"metaData.csv\"), parse_dates=[\"first_date\"])\n    geo = pd.read_csv(_find(\"station_geo_graph.csv\"), encoding=\"utf-8-sig\")\n    train[\"lb\"] = np.log1p(train[\"boardings\"]).astype(\"float32\")\n    train[\"la\"] = np.log1p(train[\"alightings\"]).astype(\"float32\")\n    for df in (train, test):\n        df[\"dow\"] = df.date.dt.dayofweek\n        df[\"daytype\"] = np.where(df.date.isin(HSET), 2, np.where(df.dow >= 5, 1, 0))\n    return train, test, meta, geo\n\n\ndef calendar_feats(df):\n    d = df.date\n    df[\"month\"] = d.dt.month\n    df[\"dom\"] = d.dt.day\n    df[\"doy\"] = d.dt.dayofyear\n    df[\"woy\"] = d.dt.isocalendar().week.astype(int).to_numpy()\n    prev_, next_ = d - pd.Timedelta(days=1), d + pd.Timedelta(days=1)\n    df[\"hol_prev\"] = prev_.isin(HSET).astype(np.int8)\n    df[\"hol_next\"] = next_.isin(HSET).astype(np.int8)\n    off_prev = prev_.isin(HSET) | (prev_.dt.dayofweek >= 5)\n    off_next = next_.isin(HSET) | (next_.dt.dayofweek >= 5)\n    df[\"bridge\"] = ((df.daytype == 0) & off_prev & off_next).astype(np.int8)\n    df[\"after_long_off\"] = ((df.daytype == 0) & prev_.isin(HSET)).astype(np.int8)\n    rel = np.full(len(df), 99, dtype=np.int16)\n    for b in BIG_HOLIDAYS:\n        r = (d - b).dt.days.to_numpy()\n        m = np.abs(r) <= 10\n        rel[m] = r[m]\n    df[\"big_hol_rel\"] = rel\n    for nm, per in ((\"univ_vac\", UNIV_VAC), (\"school_vac\", SCHOOL_VAC)):\n        m = np.zeros(len(df), dtype=np.int8)\n        for a_, b_ in per:\n            m |= ((d >= a_) & (d <= b_)).to_numpy()\n        df[nm] = m\n    dn = d.to_numpy()\n    last = np.full(len(df), np.nan)\n    for st in UNIV_START:\n        k = (dn - st.to_datetime64()) / np.timedelta64(1, \"D\")\n        last = np.where((k >= 0) & (np.isnan(last) | (k < last)), k, last)\n    df[\"univ_term_day\"] = np.minimum(last, 60)\n    return df\n\n\nclass Builder:\n    \"\"\"[날짜 x 역 x 시간] 텐서 + 날짜 누적합으로 모든 창 통계를 O(1) 로 계산하는 벡터화 빌더.\"\"\"\n\n    def __init__(self, train, meta, geo, groups):\n        self.groups = set(groups)\n        self.d0 = train.date.min()\n        self.dates = pd.date_range(self.d0, train.date.max())\n        self.sids = np.array(sorted(meta.station_id))\n        self.sidx = {s: i for i, s in enumerate(self.sids)}\n        D, S, H = len(self.dates), len(self.sids), 18\n        di = (train.date - self.d0).dt.days.to_numpy()\n        si = train.station_id.map(self.sidx).to_numpy()\n        hi = train.hour.to_numpy() - 6\n        self.LB = np.full((D, S, H), np.nan, np.float64)\n        self.LA = np.full((D, S, H), np.nan, np.float64)\n        self.LB[di, si, hi] = train.lb.to_numpy()\n        self.LA[di, si, hi] = train.la.to_numpy()\n        self.dt = np.where(self.dates.isin(HSET), 2, np.where(self.dates.dayofweek >= 5, 1, 0))\n        self.dow = self.dates.dayofweek.to_numpy()\n        ok = ~np.isnan(self.LB)\n        lb0 = np.where(ok, self.LB, 0.0)\n        la0 = np.where(ok, self.LA, 0.0)\n\n        def cum(x):\n            c = np.zeros((D + 1,) + x.shape[1:])\n            np.cumsum(x, axis=0, out=c[1:])\n            return c\n\n        def sel(m):\n            m = m[:, None, None]\n            return cum(np.where(m, lb0, 0)), cum(np.where(m, ok, 0)), cum(np.where(m, lb0 ** 2, 0))\n        self.C_dt = [sel(self.dt == k) for k in (0, 1, 2)]\n        self.C_dow = [sel((self.dow == j) & (self.dt < 2)) for j in range(7)]\n        self.C_la = [cum(np.where((self.dt == k)[:, None, None], la0, 0)) for k in (0, 1)]\n        self.G = np.nanmean(self.LB.reshape(D, -1), axis=1)\n        self.fd = meta.set_index(\"station_id\").first_date.reindex(self.sids).to_numpy()\n        self.line = meta.set_index(\"station_id\").canonical_line.reindex(self.sids).to_numpy()\n        if \"nbr\" in self.groups:\n            _, A_ = adjacency(meta, geo)  # 노선 + 1.5km 근접, sids 와 같은 정렬 순서\n            self.W = A_ / np.maximum(A_.sum(1, keepdims=True), 1)\n            self.W1 = (A_ + np.eye(len(A_))) / (A_.sum(1, keepdims=True) + 1)  # 자기 포함 평활용\n        self.cats = {\"station_id\": [int(x) for x in self.sids], \"line\": sorted(int(x) for x in set(self.line))}\n        self.g_feats = [c for c in geo.columns if c != \"station_id\" and not c.startswith(\"g2_\")]\n        self.G_arr = geo.set_index(\"station_id\").reindex(self.sids)[self.g_feats].to_numpy(np.float32)\n        if \"event\" in self.groups:\n            # SSG 문학 홈경기: 시즌 전 발표된 원 일정만 (잔여경기·취소 여부는 사후 정보라 제외)\n            sch = pd.read_csv(_find(ev_file(\"ssg\")), parse_dates=[\"date\"])\n            self.ssg_start = sch.set_index(\"date\").start_hour\n            self.munhak_w = np.exp(-geo.set_index(\"station_id\").reindex(self.sids).dist_munhak.to_numpy() / 1000.0)\n            # 부천FC 부천종합운동장 홈경기 (시즌 전 공표분: K리그1 2026 은 스플릿 전 33라운드까지)\n            bfc = pd.read_csv(_find(ev_file(\"bucheon\")), parse_dates=[\"date\"])\n            self.bfc_start = bfc.set_index(\"date\").start_hour\n            self.bfc_w = np.exp(-geo.set_index(\"station_id\").reindex(self.sids).dist_bucheon_stadium.to_numpy() / 1000.0)\n\n    @staticmethod\n    def _win(C, lo, hi):\n        \"\"\"누적합 C 에서 날짜 인덱스 (lo, hi] 구간 합.\"\"\"\n        lo = max(lo, -1)\n        return C[hi + 1] - C[lo + 1]\n\n    def _mean(self, sel, lo, hi):\n        s, n, _ = sel\n        n_ = self._win(n, lo, hi)\n        with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n            return np.where(n_ > 0, self._win(s, lo, hi) / np.maximum(n_, 1), np.nan)\n\n    def _std(self, sel, lo, hi):\n        s, n, q = sel\n        s_, n_, q_ = self._win(s, lo, hi), self._win(n, lo, hi), self._win(q, lo, hi)\n        with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n            v = (q_ - s_ ** 2 / np.maximum(n_, 1)) / np.maximum(n_ - 1, 1)\n            return np.where(n_ > 1, np.sqrt(np.maximum(v, 0)), np.nan)\n\n    def _gather(self, idx, ss, hh, ok):\n        v = np.full(len(idx), np.nan)\n        v[ok] = self.LB[idx[ok], ss[ok], hh[ok]]\n        return v\n\n    def rows(self, anchor, end=None, target=None):\n        anchor = pd.Timestamp(anchor)\n        a = (anchor - self.d0).days\n        if target is None:\n            hi_ = min((pd.Timestamp(end) - self.d0).days, len(self.dates) - 1)\n            if hi_ <= a:\n                return pd.DataFrame()\n            sub = self.LB[a + 1:hi_ + 1]\n            dd, ss, hh = np.nonzero(~np.isnan(sub))\n            di = dd + a + 1\n            out = pd.DataFrame({\"date\": self.dates[di], \"station_id\": self.sids[ss], \"hour\": hh + 6,\n                                \"lb\": sub[dd, ss, hh].astype(np.float32)})\n        else:\n            out = target.reset_index(drop=True)[[\"example_id\", \"date\", \"station_id\", \"hour\"]].copy()\n            di = (out.date - self.d0).dt.days.to_numpy()\n            ss = out.station_id.map(self.sidx).to_numpy()\n            hh = out.hour.to_numpy() - 6\n        dates = out.date\n        dow = dates.dt.dayofweek.to_numpy()\n        dtp = np.where(dates.isin(HSET), 2, np.where(dow >= 5, 1, 0))\n        out[\"dow\"], out[\"daytype\"] = dow, dtp\n        k01 = np.where(dtp < 2, dtp, 0)\n        nanhol = np.where(dtp < 2, 1.0, np.nan)\n        F = {}\n        for w, days in ((\"4w\", 28), (\"8w\", 56), (\"26w\", 182), (\"all\", a + 1)):\n            M = np.stack([self._mean(self.C_dt[k], a - days, a) for k in (0, 1)])\n            F[f\"p_dt_{w}\"] = M[k01, ss, hh] * nanhol\n            if w != \"4w\":\n                Mw = np.stack([self._mean(self.C_dow[j], a - days, a) for j in range(7)])\n                F[f\"p_dow_{w}\"] = Mw[dow, ss, hh]\n        F[\"p_hol_all\"] = self._mean(self.C_dt[2], -1, a)[ss, hh]\n        F[\"p_sun_all\"] = self._mean(self.C_dow[6], -1, a)[ss, hh]\n        Sd = np.stack([self._std(self.C_dow[j], a - 56, a) for j in range(7)])\n        F[\"sd_dow_8w\"] = Sd[dow, ss, hh]\n        La = []\n        for k in (0, 1):\n            n_ = self._win(self.C_dt[k][1], a - 56, a)\n            with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n                La.append(np.where(n_ > 0, self._win(self.C_la[k], a - 56, a) / np.maximum(n_, 1), np.nan))\n        F[\"la_dt_8w\"] = np.stack(La)[k01, ss, hh] * nanhol\n\n        def st_level(days):\n            s_ = sum(self._win(self.C_dt[k][0], a - days, a).sum(1) for k in (0, 1))\n            n_ = sum(self._win(self.C_dt[k][1], a - days, a).sum(1) for k in (0, 1))\n            with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n                return np.where(n_ > 0, s_ / np.maximum(n_, 1), np.nan)\n        F[\"st_level_4w\"] = st_level(28)[ss]\n        F[\"st_level_26w\"] = st_level(182)[ss]\n        F[\"st_level_all\"] = st_level(a + 1)[ss]\n        F[\"trend_4_26\"] = F[\"st_level_4w\"] - F[\"st_level_26w\"]\n        F[\"trend_26_all\"] = F[\"st_level_26w\"] - F[\"st_level_all\"]\n        if \"trend\" in self.groups:\n            # 평일만으로 수준을 잰다 (창마다 주말·공휴일 비중이 달라 생기는 왜곡 제거)\n            def wd_lvl(days, end):\n                if end - days < -1:\n                    return np.full(len(self.sids), np.nan)\n                s_ = self._win(self.C_dt[0][0], end - days, end).sum(1)\n                n_ = self._win(self.C_dt[0][1], end - days, end).sum(1)\n                with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n                    return np.where(n_ >= 18 * 10, s_ / np.maximum(n_, 1), np.nan)\n            l4, l13 = wd_lvl(28, a), wd_lvl(91, a)\n            yo = wd_lvl(56, a) - wd_lvl(56, a - 364)\n            F[\"wd_tr_4_13\"] = (l4 - l13)[ss]\n            F[\"wd_yoy_8w\"] = yo[ss]\n            with np.errstate(invalid=\"ignore\"):\n                F[\"net_tr_4_13\"] = np.full(len(out), np.nanmean(l4 - l13) if np.isfinite(l4 - l13).any() else np.nan)\n                F[\"net_yoy_8w\"] = np.full(len(out), np.nanmean(yo) if np.isfinite(yo).any() else np.nan)\n            if \"nbr\" in self.groups:\n                def nb(v):\n                    m = np.isfinite(v)\n                    with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n                        return (self.W @ np.where(m, v, 0)) / (self.W @ m)\n                F[\"nbr_tr_4_13\"] = nb(l4 - l13)[ss]\n                F[\"nbr_yoy_8w\"] = nb(yo)[ss]\n        F[\"hol_minus_sun\"] = F[\"p_hol_all\"] - F[\"p_sun_all\"]\n\n        yd = di - 364\n        ya = a - 364\n        yd_ok = (yd >= 0) & (yd <= a)\n        ly = self._gather(yd, ss, hh, yd_ok)\n        if ya - 28 >= 0:\n            Bm = np.stack([self._mean(self.C_dt[k], ya - 28, ya) for k in (0, 1)])\n            F[\"ya_seas_st\"] = ly - Bm[k01, ss, hh] * nanhol\n            ydt = np.where(yd_ok, self.dt[np.clip(yd, 0, len(self.dt) - 1)], -1)\n            ball = []\n            for k in (0, 1):\n                s_ = self._win(self.C_dt[k][0], ya - 28, ya).sum()\n                n_ = self._win(self.C_dt[k][1], ya - 28, ya).sum()\n                ball.append(s_ / n_ if n_ > 0 else np.nan)\n            ball = np.array(ball + [np.nan])\n            F[\"ya_daytype\"] = np.where(ydt >= 0, ydt, np.nan)\n            gy = np.where(yd_ok, self.G[np.clip(yd, 0, len(self.G) - 1)], np.nan)\n            F[\"ya_seas_all\"] = np.where(ydt >= 0, gy - ball[np.clip(ydt, 0, 2)], np.nan)\n        else:\n            F[\"ya_seas_st\"] = np.full(len(out), np.nan)\n            F[\"ya_daytype\"] = np.full(len(out), np.nan)\n            F[\"ya_seas_all\"] = np.full(len(out), np.nan)\n\n        F[\"horizon\"] = di - a\n        F[\"line\"] = self.line[ss]\n        fd = pd.to_datetime(self.fd[ss])\n        isnew = np.asarray(fd > pd.Timestamp(\"2025-01-01\"))\n        F[\"is_new\"] = isnew.astype(np.int8)\n        if \"newst\" in self.groups:\n            F[\"days_open_new\"] = np.where(isnew, (dates.to_numpy() - fd.to_numpy()) / np.timedelta64(1, \"D\"), np.nan)\n        if \"analog\" in self.groups:\n            # 절대 수준이 아니라 '그해 직전 4주 대비 변화'만 넣는다 (수준을 넣으면 연간 성장분만큼 과소예측)\n            base_ya = (np.stack([self._mean(self.C_dt[k], ya - 28, ya) for k in (0, 1)])[k01, ss, hh]\n                       if ya - 28 >= 0 else np.full(len(out), np.nan))\n            acc = np.where(np.isnan(ly), 0.0, ly)\n            cnt = (~np.isnan(ly)).astype(float)\n            for k in (-7, 7):\n                y2 = yd + k\n                v = self._gather(y2, ss, hh, (y2 >= 0) & (y2 <= a))\n                g = ~np.isnan(v)\n                acc[g] += v[g]\n                cnt[g] += 1\n            F[\"ly_wk_rel\"] = np.where(cnt > 0, acc / np.maximum(cnt, 1), np.nan) - base_ya\n        if \"nbr\" in self.groups:\n            # 작년 계절 편차를 이웃 역(자기 포함) x 앞뒤 1시간으로 평활 -> 역x시간 단위 잡음 감소\n            v = np.asarray(F[\"ya_seas_st\"], dtype=float)\n            d0i = di - di.min()\n            Y = np.full((d0i.max() + 1, len(self.sids), 18), np.nan)\n            Y[d0i, ss, hh] = v\n            Mk = np.isfinite(Y).astype(float)\n            Y0 = np.where(Mk > 0, Y, 0.0)\n            Ys, Ms = Y0.copy(), Mk.copy()\n            Ys[:, :, 1:] += Y0[:, :, :-1]\n            Ys[:, :, :-1] += Y0[:, :, 1:]\n            Ms[:, :, 1:] += Mk[:, :, :-1]\n            Ms[:, :, :-1] += Mk[:, :, 1:]\n            num = np.einsum(\"ij,djh->dih\", self.W1, Ys)\n            den = np.einsum(\"ij,djh->dih\", self.W1, Ms)\n            with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n                F[\"ya_seas_sm\"] = np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)[d0i, ss, hh]\n        out = pd.concat([out, pd.DataFrame(F)], axis=1)\n        out = calendar_feats(out)\n        if \"analog\" in self.groups:\n            rel = out.big_hol_rel.to_numpy().astype(int)\n            acc = np.zeros(len(out))\n            cnt = np.zeros(len(out))\n            for b in BIG_HOLIDAYS:\n                b0 = (b - self.d0).days\n                if b0 - 11 - 28 < 0 or b0 - 11 > a:  # 기준 구간이 anchor 이전에 관측돼야 함 (누출 방지)\n                    continue\n                bi = b0 + np.clip(rel, -10, 10)\n                v = self._gather(bi, ss, hh, (rel != 99) & (bi >= 0) & (bi <= a))\n                # 해당 명절 직전(-11일 이전) 4주 같은 요일유형 평균 대비\n                base_b = np.stack([self._mean(self.C_dt[k], b0 - 11 - 28, b0 - 11) for k in (0, 1)])[k01, ss, hh]\n                v = v - base_b\n                g = ~np.isnan(v)\n                acc[g] += v[g]\n                cnt[g] += 1\n            out[\"analog_bighol_rel\"] = np.where(cnt > 0, acc / np.maximum(cnt, 1), np.nan)\n        if \"event\" in self.groups:\n            st = out.date.map(self.ssg_start).to_numpy(dtype=float)\n            game = ~np.isnan(st)\n            out[\"ssg_hr_rel\"] = np.where(game, out.hour.to_numpy() - st, np.nan)\n            out[\"ssg_expo\"] = np.where(game, self.munhak_w[ss], 0.0)\n            bst = out.date.map(self.bfc_start).to_numpy(dtype=float)\n            bg = ~np.isnan(bst)\n            out[\"bfc_hr_rel\"] = np.where(bg, out.hour.to_numpy() - bst, np.nan)\n            out[\"bfc_expo\"] = np.where(bg, self.bfc_w[ss], 0.0)\n        if \"graph\" in self.groups:\n            out = pd.concat([out, pd.DataFrame(self.G_arr[ss], columns=self.g_feats)], axis=1)\n        if \"wx\" in self.groups:\n            out = pd.concat([out, weather_feats(out)], axis=1)\n        for c in out.columns:\n            if c in (\"date\", \"example_id\"):\n                continue\n            out[c] = out[c].astype(np.int32 if c in INT_COLS else np.float32)\n        return out\n\n    def feats(self):\n        f = [\"station_id\", \"line\", \"hour\", \"dow\", \"daytype\", \"month\", \"dom\", \"doy\", \"woy\", \"horizon\",\n             \"hol_prev\", \"hol_next\", \"bridge\", \"after_long_off\", \"big_hol_rel\",\n             \"p_dt_4w\", \"p_dt_8w\", \"p_dt_26w\", \"p_dow_8w\", \"p_dow_26w\", \"p_dt_all\", \"p_dow_all\", \"p_hol_all\",\n             \"p_sun_all\", \"sd_dow_8w\", \"la_dt_8w\", \"st_level_4w\", \"st_level_26w\", \"st_level_all\", \"trend_4_26\",\n             \"trend_26_all\", \"hol_minus_sun\", \"ya_seas_st\", \"ya_seas_all\", \"ya_daytype\", \"is_new\"]\n        if \"newst\" in self.groups:\n            f.append(\"days_open_new\")\n        if \"vac\" in self.groups:\n            f += [\"univ_vac\", \"school_vac\", \"univ_term_day\"]\n        if \"trend\" in self.groups:\n            f += [\"wd_tr_4_13\", \"wd_yoy_8w\", \"net_tr_4_13\", \"net_yoy_8w\"]\n            if \"nbr\" in self.groups:\n                f += [\"nbr_tr_4_13\", \"nbr_yoy_8w\"]\n        if \"nbr\" in self.groups:\n            f += [\"ya_seas_sm\"]\n        if \"analog\" in self.groups:\n            f += [\"ly_wk_rel\", \"analog_bighol_rel\"]\n        if \"event\" in self.groups:\n            f += [\"ssg_hr_rel\", \"ssg_expo\"] + ([\"bfc_hr_rel\", \"bfc_expo\"] if CONFIG.get(\"event_bfc\", True) else [])\n        if \"graph\" in self.groups:\n            f += self.g_feats\n        if \"wx\" in self.groups:\n            f += list(WX_COLS)\n        return f\n\n    def train_set(self, last_target):\n        last = pd.Timestamp(last_target)\n        anchors = pd.date_range(\"2025-02-26\", last - pd.Timedelta(days=7), freq=f\"{CONFIG['anchor_step']}D\")\n        if CONFIG.get(\"smoke\"):\n            anchors = anchors[-4:]\n        parts = [self.rows(a, min(a + pd.Timedelta(days=HORIZON), last)) for a in anchors]\n        return pd.concat(parts, ignore_index=True), len(anchors)\n\n\nclass Tick:\n    \"\"\"단계별 경과 시간 기록.\"\"\"\n\n    def __init__(self, name):\n        self.name, self.t0, self.t = name, time.time(), time.time()\n\n    def __call__(self, msg):\n        now = time.time()\n        p(f\"  [{self.name}] {msg}  (+{now - self.t:.1f}s, 누적 {now - self.t0:.0f}s)\")\n        self.t = now\n\n\ndef gpu_available():\n    import shutil\n    import subprocess\n    if not shutil.which(\"nvidia-smi\"):\n        return False\n    return subprocess.run([\"nvidia-smi\", \"-L\"], capture_output=True).returncode == 0\n\n\ndef norm_base(df):\n    \"\"\"정규화 기준: 최근 8주 같은 요일 평균 (없으면 26주 -> 전체 -> 역 수준).\"\"\"\n    b = df[\"p_dow_8w\"].astype(float)\n    for c in (\"p_dow_26w\", \"p_dow_all\", \"st_level_4w\"):\n        b = b.fillna(df[c].astype(float))\n    return b.to_numpy()\n\n\ndef fit_predict(tr, X_list, feats, seeds, rounds, cats):\n    if CONFIG.get(\"norm\"):\n        bt = norm_base(tr)\n        fill = float(np.nanmean(tr.lb.to_numpy() - bt) + np.nanmean(bt))\n        bt = np.where(np.isnan(bt), fill, bt)\n        tr = tr.assign(lb=tr.lb.to_numpy() - bt)\n        bxs = [np.where(np.isnan(b_), fill, b_) for b_ in (norm_base(x) for x in X_list)]\n        outs = _fit_predict(tr, X_list, feats, seeds, rounds, cats, clip=False)\n        return [np.clip(o + b_, 0, None) for o, b_ in zip(outs, bxs)]\n    return _fit_predict(tr, X_list, feats, seeds, rounds, cats)\n\n\ndef _fit_predict(tr, X_list, feats, seeds, rounds, cats, clip=True):\n    \"\"\"X_list 의 각 프레임에 대한 예측(log 공간) 목록. cats: 범주형 열의 고정 범주 목록.\"\"\"\n    outs = [np.zeros(len(x)) for x in X_list]\n    tk = Tick(f\"fit {CONFIG['model']} n={len(tr):,}\")\n    if CONFIG[\"model\"] == \"xgb\":\n        import xgboost as xgb\n        dev = \"cuda\" if gpu_available() else \"cpu\"\n        if ON_KAGGLE and CONFIG.get(\"require_gpu\", True) and dev != \"cuda\":\n            raise RuntimeError(\"GPU 커널인데 GPU 를 찾지 못함\")\n\n        def prep(df):\n            X = df[feats].copy()\n            for c in CAT_COLS:\n                X[c] = pd.Categorical(X[c], categories=cats[c])\n                assert X[c].notna().all(), f\"{c} 에 범주 목록 밖 값\"\n            return X\n        dtr = xgb.QuantileDMatrix(prep(tr), tr.lb, enable_categorical=True, max_bin=256)\n        dxs = [xgb.QuantileDMatrix(prep(x), ref=dtr, enable_categorical=True) for x in X_list]\n        tk(f\"DMatrix 생성 device={dev}\")\n\n        class Prog(xgb.callback.TrainingCallback):\n            def after_iteration(self, model, epoch, evals_log):\n                if (epoch + 1) % 200 == 0:\n                    tk(f\"round {epoch + 1}/{rounds}\")\n                return False\n        for sd in seeds:\n            prm = dict(objective=\"reg:squarederror\", eta=CONFIG[\"lr\"], max_depth=10, min_child_weight=50,\n                       subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method=\"hist\", device=dev,\n                       max_cat_to_onehot=1, max_bin=256, seed=sd)\n            prm.update(CONFIG.get(\"xgb_params\", {}))\n            m = xgb.train(prm, dtr, rounds, callbacks=[Prog()])\n            for i, dx in enumerate(dxs):\n                outs[i] += m.predict(dx) / len(seeds)\n            tk(f\"seed {sd} 학습+예측 완료\")\n        p(f\"    xgb device={dev} rounds={rounds} seeds={len(seeds)}\")\n    else:\n        import lightgbm as lgb\n\n        def prep(df):\n            X = df[feats].copy()\n            for c in CAT_COLS:\n                X[c] = pd.Categorical(X[c], categories=cats[c])\n                assert X[c].notna().all(), f\"{c} 에 범주 목록 밖 값\"\n            return X\n        ds = lgb.Dataset(prep(tr), tr.lb, categorical_feature=list(CAT_COLS), free_raw_data=False)\n        Xs = [prep(x) for x in X_list]\n        for sd in seeds:\n            prm = dict(objective=\"regression\", learning_rate=CONFIG[\"lr\"], num_leaves=127, min_data_in_leaf=200,\n                       feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,\n                       num_threads=0, verbose=-1, seed=sd)\n            prm.update(CONFIG.get(\"lgb_params\", {}))\n            m = lgb.train(prm, ds, rounds, callbacks=[lambda env: (env.iteration + 1) % 200 == 0 and tk(\n                f\"round {env.iteration + 1}/{rounds}\")])\n            for i, x in enumerate(Xs):\n                outs[i] += m.predict(x) / len(seeds)\n            tk(f\"seed {sd} 학습+예측 완료\")\n    return [np.clip(o, 0, None) for o in outs] if clip else outs\n\n\ndef hav(lat1, lon1, lat2, lon2):\n    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])\n    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2\n    return 2 * 6371000 * np.arcsin(np.sqrt(h))\n\n\ndef adjacency(meta, geo):\n    ids = sorted(meta.station_id)\n    idx = {s: i for i, s in enumerate(ids)}\n    A = np.zeros((len(ids), len(ids)))\n    for ln in (1, 2, 7):\n        seq = sorted(meta[meta.canonical_line == ln].station_id)\n        for a, b in zip(seq, seq[1:]):\n            A[idx[a], idx[b]] = A[idx[b], idx[a]] = 1\n    for a, b in [(3124, 3221), (3118, 3761), (3213, 3763)]:\n        A[idx[a], idx[b]] = A[idx[b], idx[a]] = 1\n    g = geo.set_index(\"station_id\").loc[ids]\n    D = hav(g.lat.values[:, None], g.lon.values[:, None], g.lat.values[None], g.lon.values[None])\n    A = np.clip(A + ((D <= 1500) & (D > 0)), 0, 1)\n    return ids, A\n\n\ndef propagate(r, A, b, iters=2):\n    deg = A.sum(1)\n    out = r.copy()\n    for _ in range(iters):\n        nb = np.where(deg > 0, (A @ out) / np.maximum(deg, 1), out)\n        out = (1 - b) * r + b * nb\n    return out\n\n\ndef rmsle(a, b):\n    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))\n\n\ndef recent_resid(B, anchor, feats):\n    \"\"\"anchor-56 까지로 학습해 (anchor-56, anchor] 를 예측한 역별 표본 외 잔차 (C&S 용).\"\"\"\n    a2 = pd.Timestamp(anchor) - pd.Timedelta(days=56)\n    tk = Tick(\"C&S 보조\")\n    tr2, _ = B.train_set(a2)\n    va2 = B.rows(a2, pd.Timestamp(anchor))\n    tk(f\"피처 생성 train {len(tr2):,}\")\n    (p2,) = fit_predict(tr2, [va2], feats, [0], CONFIG[\"rounds\"], B.cats)\n    return va2.assign(e=p2 - va2.lb).groupby(\"station_id\").e.mean()\n\n\ndef run_cv(train, meta, geo):\n    B = Builder(train, meta, geo, CONFIG[\"groups\"])\n    feats = B.feats()\n    ids, A = adjacency(meta, geo)\n    summary = {}\n    for name in CONFIG[\"folds\"]:\n        t0 = time.time()\n        a, e = FOLDS[name]\n        tk = Tick(f\"fold {name}\")\n        tr, n = B.train_set(a)\n        va = B.rows(a, pd.Timestamp(e))\n        tk(f\"피처 생성 anchors {n} train {len(tr):,} feats {len(feats)}\")\n        (pred,) = fit_predict(tr, [va], feats, CONFIG[\"seeds\"], CONFIG[\"rounds\"], B.cats)\n        old = (va.is_new == 0).to_numpy()\n        res = {\"all\": rmsle(pred, va.lb), \"old\": rmsle(pred[old], va.lb[old]),\n               \"new\": rmsle(pred[~old], va.lb[~old]) if (~old).any() else None,\n               \"bias\": float(np.mean(pred - va.lb))}\n        err = pd.Series(pred - va.lb.to_numpy())\n        res[\"by_daytype\"] = {int(k): round(float(np.sqrt((v ** 2).mean())), 4) for k, v in err.groupby(va.daytype.to_numpy())}\n        res[\"by_month\"] = {int(k): round(float(np.sqrt((v ** 2).mean())), 4) for k, v in err.groupby(va.month.to_numpy())}\n        out = va[[\"date\", \"station_id\", \"hour\", \"lb\"]].assign(pred=pred)\n        if CONFIG[\"cs\"]:\n            r = recent_resid(B, a, feats).reindex(ids).fillna(0).to_numpy()\n            best = (res[\"old\"], 0.0, 0.0)\n            grid = {}\n            for bb in (0.0, 0.3, 0.6, 0.9):\n                rs = pd.Series(propagate(r, A, bb), index=ids)\n                for aa in (0.25, 0.5, 0.75, 1.0):\n                    adj = pred - aa * va.station_id.map(rs).to_numpy()\n                    sc = rmsle(adj[old], va.lb[old])\n                    grid[f\"a{aa}_b{bb}\"] = round(sc, 5)\n                    if sc < best[0]:\n                        best = (sc, aa, bb)\n            res[\"cs_grid_old\"] = grid\n            res[\"cs_best_old\"] = best\n            out[\"resid_station\"] = va.station_id.map(pd.Series(r, index=ids)).to_numpy()\n        out.to_csv(OUT / f\"cv_{CONFIG['tag']}_{name}.csv\", index=False)\n        res[\"n_anchor\"], res[\"n_train\"], res[\"sec\"] = n, len(tr), round(time.time() - t0)\n        summary[name] = res\n        p(f\"[{name}] all {res['all']:.5f} old {res['old']:.5f} new {res['new']} bias {res['bias']:+.4f} \"\n          f\"| daytype {res['by_daytype']} | month {res['by_month']} | {res['sec']}s\")\n        if CONFIG[\"cs\"]:\n            p(f\"    C&S best(old) {res['cs_best_old']}\")\n    json.dump(summary, open(OUT / f\"cv_{CONFIG['tag']}.json\", \"w\"), indent=1, ensure_ascii=False)\n\n\ndef run_full(train, test, meta, geo):\n    B = Builder(train, meta, geo, CONFIG[\"groups\"])\n    feats = B.feats()\n    t0 = time.time()\n    last = train.date.max()\n    tk = Tick(\"full\")\n    tr, n = B.train_set(last)\n    te = B.rows(last, target=test)\n    tk(f\"피처 생성 anchors {n} train {len(tr):,} feats {len(feats)}\")\n    (pred,) = fit_predict(tr, [te], feats, CONFIG[\"seeds\"], CONFIG[\"rounds\"], B.cats)\n    if CONFIG[\"cs\"] and CONFIG.get(\"cs_a\", 0) > 0:\n        ids, A = adjacency(meta, geo)\n        r = recent_resid(B, last, feats).reindex(ids).fillna(0).to_numpy()\n        rs = pd.Series(propagate(r, A, CONFIG[\"cs_b\"]), index=ids)\n        pred = pred - CONFIG[\"cs_a\"] * te.station_id.map(rs).to_numpy()\n    sub = test[[\"example_id\"]].merge(pd.DataFrame({\"example_id\": te.example_id, \"boardings\": np.expm1(np.clip(pred, 0, None))}),\n                                     on=\"example_id\", how=\"left\")\n    assert len(sub) == 157194 and sub.boardings.notna().all() and (sub.boardings >= 0).all()\n    sub.to_csv(OUT / f\"sub_{CONFIG['tag']}.csv\", index=False)\n    te[[\"example_id\"]].assign(pred_log=pred).to_csv(OUT / f\"testpred_{CONFIG['tag']}.csv\", index=False)\n    p(f\"full: anchors {n} train {len(tr):,} feats {len(feats)} | {time.time() - t0:.0f}s\")\n    p(sub.boardings.describe().round(1).to_string())\n\n\ndef ev_file(kind):\n    \"\"\"경기 일정 파일: ev_actual 이면 실제 치러진 경기(취소 제외·추가 편성 포함), 아니면 시즌 전 공표 일정.\"\"\"\n    return f\"{kind}_home_{'actual' if CONFIG.get('ev_actual') else 'schedule'}.csv\"\n\n\n_WX = None\n\n\ndef weather_feats(df):\n    \"\"\"인천 시간별 실측 날씨 (규칙 변경 11/11: 7/1 이후 메타데이터 허용, 정답 승차만 금지).\n    wx_prcp: 해당 시간 강수, wx_prcp3: 직전 2시간 포함 3시간 합, wx_prcp_day: 그날 6~23시 합,\n    wx_temp: 기온, wx_snow: 적설, wx_wind: 풍속.\"\"\"\n    global _WX\n    if _WX is None:\n        w = pd.read_csv(_find(\"weather_incheon_hourly.csv\"), parse_dates=[\"date\"]).sort_values([\"date\", \"hour\"])\n        w[\"wx_prcp3\"] = w.precipitation.rolling(3, min_periods=1).sum()\n        day = w[w.hour.between(6, 23)].groupby(\"date\").precipitation.sum().rename(\"wx_prcp_day\")\n        w = w.merge(day, on=\"date\", how=\"left\")\n        _WX = w.rename(columns={\"precipitation\": \"wx_prcp\", \"temperature_2m\": \"wx_temp\", \"snowfall\": \"wx_snow\",\n                                \"wind_speed_10m\": \"wx_wind\"}).set_index([\"date\", \"hour\"])[list(WX_COLS)]\n    idx = pd.MultiIndex.from_arrays([pd.DatetimeIndex(df.date), df.hour.to_numpy()])\n    out = _WX.reindex(idx)\n    assert out.notna().all().all(), \"날씨 자료 누락\"\n    return out.reset_index(drop=True).set_axis(df.index)\n\n\nWX_COLS = (\"wx_prcp\", \"wx_prcp3\", \"wx_prcp_day\", \"wx_temp\", \"wx_snow\", \"wx_wind\")\n\n\ndef chronos_covariates(df, geo):\n    \"\"\"미래에 미리 알 수 있는 공변량 (달력·방학·사전 공표된 경기 일정).\"\"\"\n    c = calendar_feats(df[[\"date\", \"station_id\", \"hour\"]].assign(\n        daytype=np.where(df.date.isin(HSET), 2, np.where(df.date.dt.dayofweek >= 5, 1, 0))))\n    out = pd.DataFrame({\n        \"off\": (c.daytype > 0).astype(float), \"hol\": (c.daytype == 2).astype(float),\n        \"bridge\": c.bridge.astype(float), \"after_off\": c.after_long_off.astype(float),\n        \"bighol\": np.where(c.big_hol_rel == 99, 0.0, 1.0 - np.abs(c.big_hol_rel) / 11.0),\n        \"univ_vac\": c.univ_vac.astype(float), \"school_vac\": c.school_vac.astype(float)}, index=df.index)\n    sch = pd.read_csv(_find(ev_file(\"ssg\")), parse_dates=[\"date\"]).set_index(\"date\").start_hour\n    st = df.date.map(sch).to_numpy(dtype=float)\n    w = np.exp(-df.station_id.map(geo.set_index(\"station_id\").dist_munhak).to_numpy() / 1000.0)\n    rel = df.hour.to_numpy() - st\n    out[\"ssg\"] = np.where(~np.isnan(st) & (rel >= 0) & (rel <= 5), w, 0.0)\n    if CONFIG.get(\"chr_bfc\"):\n        bs = pd.read_csv(_find(ev_file(\"bucheon\")), parse_dates=[\"date\"]).set_index(\"date\").start_hour\n        bst = df.date.map(bs).to_numpy(dtype=float)\n        wb = np.exp(-df.station_id.map(geo.set_index(\"station_id\").dist_bucheon_stadium).to_numpy() / 1000.0)\n        rb = df.hour.to_numpy() - bst\n        out[\"bfc\"] = np.where(~np.isnan(bst) & (rb >= -1) & (rb <= 4), wb, 0.0)\n    if CONFIG.get(\"chr_wx\"):\n        wx = weather_feats(df)\n        out[\"prcp\"] = np.log1p(wx.wx_prcp3.to_numpy())\n        out[\"prcp_day\"] = np.log1p(wx.wx_prcp_day.to_numpy())\n        out[\"cold\"] = np.clip(-wx.wx_temp.to_numpy(), 0, None) / 10.0\n        out[\"hot\"] = np.clip(wx.wx_temp.to_numpy() - 28, 0, None) / 5.0\n        out[\"snow\"] = np.log1p(wx.wx_snow.to_numpy() * 10)\n    if CONFIG.get(\"chr_geo\"):\n        g = geo.set_index(\"station_id\")\n        lat, lon = df.station_id.map(g.lat).to_numpy(), df.station_id.map(g.lon).to_numpy()\n        univ = [(37.4500, 126.6535), (37.3754, 126.6328), (37.3824, 126.6697)]  # 인하대, 인천대 송도, 연세대 국제캠\n        prox = np.max([np.exp(-hav(lat, lon, a_, b_) / 1000.0) for a_, b_ in univ], axis=0)\n        out[\"univ_vac_x\"] = out.univ_vac.to_numpy() * prox\n        out[\"univ_term_x\"] = (1 - out.univ_vac.to_numpy()) * prox\n        # 펜타포트 (사전 공표: 2025-08-01~03, 2026-07-31~08-02), 송도달빛축제공원 거리 감쇠\n        fest = df.date.isin(pd.to_datetime([\"2025-08-01\", \"2025-08-02\", \"2025-08-03\",\n                                            \"2026-07-31\", \"2026-08-01\", \"2026-08-02\"])).to_numpy()\n        wf = np.exp(-df.station_id.map(g.dist_songdo_park).to_numpy() / 1000.0)\n        out[\"pentaport\"] = np.where(fest & (df.hour.to_numpy() >= 12), wf, 0.0)\n    return out\n\n\ndef structural_ratio(train, geo, anchor, dates, K=10):\n    \"\"\"통행 목적 분해 기반 '달력이 설명하는 역·시간별 예상 변화율' (log).\n\n    1) anchor 이전 기존역 [날짜 x (역,시간,승/하)] 인원수 행렬을 K개 목적으로 NMF(KL) 분해: X ≈ F·G\n    2) 목적별 log 활동량 F[:,k] 를 달력 변수로 릿지 회귀 -> 과거·미래 모두 '달력 설명분' 만 사용 (성격 일치)\n    3) 비율 = log((F̂[d]·G)[s,h]) - log((mean F · G)[s,h])   (평소 대비 예상 변화, 신규역은 0)\n    반환: DataFrame(date, station_id, hour, struct)\n    \"\"\"\n    from sklearn.decomposition import NMF\n    from sklearn.linear_model import Ridge\n    anchor = pd.Timestamp(anchor)\n    new_st = train.groupby(\"station_id\").date.min() > pd.Timestamp(\"2025-01-01\")\n    old_ids = sorted(new_st[~new_st].index)\n    hist = train[(train.date <= anchor) & train.station_id.isin(old_ids)]\n    cols_ = pd.MultiIndex.from_product([old_ids, range(6, 24)])\n    B = hist.pivot_table(index=\"date\", columns=[\"station_id\", \"hour\"], values=\"boardings\").reindex(columns=cols_)\n    A = hist.pivot_table(index=\"date\", columns=[\"station_id\", \"hour\"], values=\"alightings\").reindex(columns=cols_)\n    X = np.hstack([B.to_numpy(), A.to_numpy()])\n    ok = ~np.isnan(X).any(1)\n    nmf = NMF(n_components=K, beta_loss=\"kullback-leibler\", solver=\"mu\", init=\"nndsvda\", max_iter=600, random_state=0)\n    Fd = nmf.fit_transform(X[ok])\n    G = nmf.components_[:, :B.shape[1]]\n\n    def design(ds):\n        c = calendar_feats(pd.DataFrame({\"date\": ds, \"daytype\": np.where(pd.DatetimeIndex(ds).isin(HSET), 2,\n                                                                           np.where(pd.DatetimeIndex(ds).dayofweek >= 5, 1, 0))}))\n        D = pd.get_dummies(c.date.dt.dayofweek, prefix=\"dow\").astype(float)\n        for k_ in (\"bridge\", \"after_long_off\", \"univ_vac\", \"school_vac\"):\n            D[k_] = c[k_].astype(float)\n        D[\"hol\"] = (c.daytype == 2).astype(float)\n        D[\"bighol\"] = np.where(c.big_hol_rel == 99, 0.0, 1.0 - np.abs(c.big_hol_rel) / 11.0)\n        D[\"hol_x_vac\"] = D.hol * D.school_vac\n        return D.reindex(columns=[f\"dow_{i}\" for i in range(7)] + [\"bridge\", \"after_long_off\", \"univ_vac\",\n                                                                     \"school_vac\", \"hol\", \"bighol\", \"hol_x_vac\"],\n                         fill_value=0.0).to_numpy()\n    reg = Ridge(alpha=1.0).fit(design(B.index[ok]), np.log(Fd + 1e-3))\n    Fh = np.exp(reg.predict(design(pd.DatetimeIndex(dates))))\n    base = np.log(Fd.mean(0) @ G + 1e-3)\n    R = np.log(Fh @ G + 1e-3) - base  # (len(dates), S*H)\n    out = pd.DataFrame(R, index=pd.DatetimeIndex(dates), columns=cols_).stack([0, 1], future_stack=True)\n    out = out.rename(\"struct\").reset_index()\n    out.columns = [\"date\", \"station_id\", \"hour\", \"struct\"]\n    return out\n\n\ndef chronos_predict(train, geo, anchor, target):\n    \"\"\"(역,시간) 일별 log 승차 시계열 1278개를 Chronos-2 로 예측. target: date/station_id/hour 프레임.\"\"\"\n    if CONFIG.get(\"chr_daily\"):\n        return chronos_daily_predict(train, geo, anchor, target)\n    import torch\n    from chronos import Chronos2Pipeline\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    global _PIPE\n    if \"_PIPE\" not in globals():\n        _PIPE = Chronos2Pipeline.from_pretrained(CONFIG.get(\"chronos_model\", \"amazon/chronos-2\"), device_map=dev)\n    pipe = _PIPE\n    anchor = pd.Timestamp(anchor)\n    ctx = train[train.date <= anchor][[\"date\", \"station_id\", \"hour\", \"lb\"]].copy()\n    fut = target[[\"date\", \"station_id\", \"hour\"]].drop_duplicates().copy()\n    cols = None\n    for df in (ctx, fut):\n        cv = chronos_covariates(df, geo)\n        cols = list(cv.columns)\n        for c_ in cols:\n            df[c_] = cv[c_].to_numpy()\n        # item_id 사전순 = cross-learning 묶음 순서 (station 순: 노선상 이웃 / hour 순: 같은 시간대 전 역)\n        hs = df.hour.map(\"{:02d}\".format)\n        df[\"item_id\"] = (df.station_id.astype(str) + \"_\" + hs if CONFIG.get(\"cl_order\", \"station\") == \"station\"\n                         else hs + \"_\" + df.station_id.astype(str))\n        df.rename(columns={\"date\": \"timestamp\"}, inplace=True)\n    H = int((fut.timestamp.max() - anchor).days)\n    fut = fut[fut.item_id.isin(set(ctx.item_id))]\n    ctx = ctx[ctx.item_id.isin(set(fut.item_id))]  # Chronos 는 과거·미래의 시계열 ID 집합이 같아야 함\n    if CONFIG.get(\"chr_struct\"):\n        t1 = time.time()\n        alld = pd.DatetimeIndex(sorted(set(ctx.timestamp) | set(fut.timestamp)))\n        st_ = structural_ratio(train, geo, anchor, alld, K=CONFIG[\"chr_struct\"].get(\"K\", 10))\n        st_ = st_.rename(columns={\"date\": \"timestamp\"})\n        ctx = ctx.merge(st_, on=[\"timestamp\", \"station_id\", \"hour\"], how=\"left\")\n        fut = fut.merge(st_, on=[\"timestamp\", \"station_id\", \"hour\"], how=\"left\")\n        ctx[\"struct\"] = ctx.struct.fillna(0.0)\n        fut[\"struct\"] = fut.struct.fillna(0.0)\n        cols = cols + [\"struct\"]\n        p(f\"  구조 공변량 K={CONFIG['chr_struct'].get('K', 10)} {time.time() - t1:.0f}s\")\n    ctx = ctx.sort_values([\"item_id\", \"timestamp\"], kind=\"stable\")\n    fut = fut.sort_values([\"item_id\", \"timestamp\"], kind=\"stable\")\n    ft = CONFIG.get(\"chr_ft\")\n    if ft:\n        # anchor 이전 자료만으로 미세조정 (폴드마다 새로) -> 누출 없음\n        from chronos.chronos2.preprocess import from_data_frame\n        t0 = time.time()\n        h_ft = ft.get(\"h\", 63)\n        inputs = from_data_frame(ctx[[\"item_id\", \"timestamp\", \"lb\"] + cols], target_columns=[\"lb\"],\n                                 prediction_length=h_ft, known_covariates_names=cols)\n        pipe = pipe.fit(inputs, prediction_length=h_ft, finetune_mode=ft.get(\"mode\", \"lora\"),\n                        learning_rate=ft.get(\"lr\", 1e-4), num_steps=ft.get(\"steps\", 1000),\n                        batch_size=ft.get(\"batch\", 256), min_past=ft.get(\"min_past\", 63),\n                        output_dir=str(OUT / \"chr_ft_tmp\"), remove_printer_callback=True)\n        p(f\"  chronos 미세조정 {ft} {time.time() - t0:.0f}s\")\n    t0 = time.time()\n    pr = pipe.predict_df(ctx[[\"item_id\", \"timestamp\", \"lb\"] + cols], future_df=fut[[\"item_id\", \"timestamp\"] + cols],\n                         target=\"lb\", prediction_length=H, quantile_levels=[0.5],\n                         cross_learning=CONFIG.get(\"cross_learning\", False), batch_size=CONFIG.get(\"batch_size\", 256),\n                         context_length=CONFIG.get(\"context_length\"))\n    p(f\"  chronos device={dev} series={ctx.item_id.nunique()} H={H} {time.time() - t0:.0f}s\")\n    pr = pr.rename(columns={\"timestamp\": \"date\"})\n    sid_h = pr.item_id.str.split(\"_\", expand=True).astype(int)\n    if CONFIG.get(\"cl_order\", \"station\") == \"station\":\n        pr[\"station_id\"], pr[\"hour\"] = sid_h[0], sid_h[1]\n    else:\n        pr[\"station_id\"], pr[\"hour\"] = sid_h[1], sid_h[0]\n    m = target[[\"date\", \"station_id\", \"hour\"]].merge(pr[[\"date\", \"station_id\", \"hour\", \"predictions\"]],\n                                                     on=[\"date\", \"station_id\", \"hour\"], how=\"left\")\n    return np.clip(m.predictions.to_numpy(), 0, None)\n\n\ndef chronos_daily_predict(train, geo, anchor, target):\n    \"\"\"역별 하루 총승차 log 시계열(71개)을 Chronos-2 로 예측 -> 최근 시간대 분포로 시간별 분배.\n    시간대 분포: 평일·주말은 anchor 직전 8주, 공휴일은 전체 이력 공휴일 (없으면 주말).\"\"\"\n    import torch\n    from chronos import Chronos2Pipeline\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    global _PIPE\n    if \"_PIPE\" not in globals():\n        _PIPE = Chronos2Pipeline.from_pretrained(CONFIG.get(\"chronos_model\", \"amazon/chronos-2\"), device_map=dev)\n    anchor = pd.Timestamp(anchor)\n    hist = train[train.date <= anchor][[\"date\", \"station_id\", \"hour\", \"boardings\"]].copy()\n    fut_h = target[[\"date\", \"station_id\", \"hour\"]].drop_duplicates().copy()\n\n    def daily(df_h, with_y):\n        cv = chronos_covariates(df_h, geo)\n        cv[[\"date\", \"station_id\"]] = df_h[[\"date\", \"station_id\"]].to_numpy()\n        agg = cv.groupby([\"date\", \"station_id\"]).mean(numeric_only=True)\n        if with_y:\n            n = df_h.groupby([\"date\", \"station_id\"]).boardings.agg([\"sum\", \"count\"])\n            agg[\"lb\"] = np.log1p(n[\"sum\"] * 18.0 / n[\"count\"])  # 누락 시간 보정\n        return agg.reset_index()\n    ctx, fut = daily(hist, True), daily(fut_h, False)\n    cols = [c for c in ctx.columns if c not in (\"date\", \"station_id\", \"lb\")]\n    for d in (ctx, fut):\n        d[\"item_id\"] = d.station_id.astype(str)\n        d.rename(columns={\"date\": \"timestamp\"}, inplace=True)\n    fut = fut[fut.item_id.isin(set(ctx.item_id))]\n    ctx = ctx[ctx.item_id.isin(set(fut.item_id))].sort_values([\"item_id\", \"timestamp\"])\n    fut = fut.sort_values([\"item_id\", \"timestamp\"])\n    H = int((fut.timestamp.max() - anchor).days)\n    t0 = time.time()\n    pr = _PIPE.predict_df(ctx[[\"item_id\", \"timestamp\", \"lb\"] + cols], future_df=fut[[\"item_id\", \"timestamp\"] + cols],\n                          target=\"lb\", prediction_length=H, quantile_levels=[0.5], batch_size=CONFIG.get(\"batch_size\", 256))\n    p(f\"  chronos daily device={dev} series={ctx.item_id.nunique()} H={H} {time.time() - t0:.0f}s\")\n    pr = pr.rename(columns={\"timestamp\": \"date\"})\n    pr[\"station_id\"] = pr.item_id.astype(int)\n    # 시간대 분포\n    hist[\"t\"] = np.where(hist.date.isin(HSET), 2, np.where(hist.date.dt.dayofweek >= 5, 1, 0))\n    rec = hist[(hist.date > anchor - pd.Timedelta(days=56)) | (hist.t == 2)]\n    tot = rec.groupby([\"date\", \"station_id\"]).boardings.transform(\"sum\")\n    rec = rec.assign(sh=rec.boardings / tot.clip(lower=1))\n    S = rec.groupby([\"station_id\", \"t\", \"hour\"]).sh.mean().rename(\"sh\").reset_index()\n    S1 = S[S.t == 1].drop(columns=\"t\").rename(columns={\"sh\": \"sh1\"})\n    out = target[[\"date\", \"station_id\", \"hour\"]].copy()\n    out[\"t\"] = np.where(out.date.isin(HSET), 2, np.where(out.date.dt.dayofweek >= 5, 1, 0))\n    out = out.merge(pr[[\"date\", \"station_id\", \"predictions\"]], on=[\"date\", \"station_id\"], how=\"left\")\n    out = out.merge(S, on=[\"station_id\", \"t\", \"hour\"], how=\"left\").merge(S1, on=[\"station_id\", \"hour\"], how=\"left\")\n    sh = out.sh.fillna(out.sh1).to_numpy()\n    return np.log1p(np.expm1(out.predictions.to_numpy()) * sh)\n\n\ndef run_chronos(train, test, meta, geo):\n    new_st = meta.set_index(\"station_id\").first_date > \"2025-01-01\"\n    if CONFIG[\"mode\"] == \"chronos_cv\" and CONFIG.get(\"variants\"):\n        base_cfg = dict(CONFIG)\n        for v in CONFIG[\"variants\"]:\n            CONFIG.clear()\n            CONFIG.update(base_cfg, variants=None, **v)\n            p(f\"=== variant {v}\")\n            run_chronos(train, test, meta, geo)\n        CONFIG.clear()\n        CONFIG.update(base_cfg)\n        return\n    if CONFIG[\"mode\"] == \"chronos_cv\":\n        for name in CONFIG[\"folds\"]:\n            a, e = FOLDS[name]\n            va = train[(train.date > a) & (train.date <= e)].reset_index(drop=True)\n            pred = chronos_predict(train, geo, a, va)\n            old = ~va.station_id.map(new_st).to_numpy()\n            ok = ~np.isnan(pred)\n            p(f\"[{name}] chronos all {rmsle(pred[ok], va.lb[ok]):.5f} old {rmsle(pred[ok & old], va.lb[ok & old]):.5f} \"\n              f\"missing {(~ok).sum()} bias {np.nanmean(pred - va.lb):+.4f}\")\n            va[[\"date\", \"station_id\", \"hour\", \"lb\"]].assign(pred=pred).to_csv(OUT / f\"cv_{CONFIG['tag']}_{name}.csv\", index=False)\n            if CONFIG.get(\"aux_resid\"):\n                # 보정용: anchor-56 기준으로 (anchor-56, anchor] 를 예측한 역별 표본 외 잔차\n                a2 = pd.Timestamp(a) - pd.Timedelta(days=56)\n                v2 = train[(train.date > a2) & (train.date <= a)].reset_index(drop=True)\n                p2 = chronos_predict(train, geo, a2, v2)\n                v2.assign(e=p2 - v2.lb).groupby(\"station_id\").e.mean().rename(\"r\").to_csv(\n                    OUT / f\"resid_{CONFIG['tag']}_{name}.csv\")\n    else:\n        pred = chronos_predict(train, geo, train.date.max(), test)\n        assert not np.isnan(pred).any(), \"chronos test 예측 누락\"\n        if CONFIG.get(\"aux_resid\"):\n            a2 = train.date.max() - pd.Timedelta(days=56)\n            v2 = train[train.date > a2].reset_index(drop=True)\n            p2 = chronos_predict(train, geo, a2, v2)\n            v2.assign(e=p2 - v2.lb).groupby(\"station_id\").e.mean().rename(\"r\").to_csv(OUT / f\"resid_{CONFIG['tag']}.csv\")\n        test[[\"example_id\"]].assign(pred_log=pred).to_csv(OUT / f\"testpred_{CONFIG['tag']}.csv\", index=False)\n        p(f\"chronos full: mean log {pred.mean():.4f}\")\n\n\n# ============================== 구조 모델 (통행 목적 x 왕복 x 평소 수준) ==============================\ndef struct_date_features(dates, geo=None):\n    \"\"\"날짜 단위 입력: 달력·방학·사전 공표 경기/축제 (목적별 활동량 신경망 입력).\"\"\"\n    d = pd.DatetimeIndex(dates)\n    c = calendar_feats(pd.DataFrame({\"date\": d, \"daytype\": np.where(d.isin(HSET), 2, np.where(d.dayofweek >= 5, 1, 0))}))\n    X = pd.get_dummies(c.date.dt.dayofweek, prefix=\"dow\").astype(float).reindex(\n        columns=[f\"dow_{i}\" for i in range(7)], fill_value=0.0)\n    X[\"hol\"] = (c.daytype == 2).astype(float).to_numpy()\n    for k in (\"bridge\", \"after_long_off\", \"hol_prev\", \"hol_next\", \"univ_vac\", \"school_vac\"):\n        X[k] = c[k].astype(float).to_numpy()\n    for r in range(-3, 4):\n        X[f\"bh{r}\"] = (c.big_hol_rel == r).astype(float).to_numpy()\n    X[\"term\"] = np.nan_to_num(c.univ_term_day.to_numpy(dtype=float), nan=60.0) / 60.0\n    sch = pd.read_csv(_find(ev_file(\"ssg\")), parse_dates=[\"date\"]).set_index(\"date\").start_hour\n    st = pd.Series(d).map(sch).to_numpy(dtype=float)\n    X[\"ssg\"] = (~np.isnan(st)).astype(float)\n    for hh in (14, 17, 18):\n        X[f\"ssg{hh}\"] = (st == hh).astype(float)\n    bs = pd.read_csv(_find(ev_file(\"bucheon\")), parse_dates=[\"date\"]).set_index(\"date\").start_hour\n    bst = pd.Series(d).map(bs).to_numpy(dtype=float)\n    X[\"bfc\"] = (~np.isnan(bst)).astype(float)\n    X[\"bfc_eve\"] = (bst >= 19).astype(float)\n    X[\"penta\"] = d.isin(pd.to_datetime([\"2025-08-01\", \"2025-08-02\", \"2025-08-03\",\n                                        \"2026-07-31\", \"2026-08-01\", \"2026-08-02\"])).astype(float)\n    doy = d.dayofyear.to_numpy()\n    for k in (1, 2):\n        X[f\"s{k}\"] = np.sin(2 * np.pi * k * doy / 365.25)\n        X[f\"c{k}\"] = np.cos(2 * np.pi * k * doy / 365.25)\n    return X.to_numpy(dtype=np.float32), c.daytype.to_numpy()\n\n\ndef struct_fit_predict(train, meta, geo, anchor, target_end):\n    \"\"\"anchor 이전 자료로 구조 모델 학습 -> (anchor, target_end] 승차 log1p 예측 [D_t, S, H].\n\n    μ[d,s,h,c] = 평소수준_j[유형(d),s,h,c] x (Σ_k W[d,k]·L[k,s,h,c]) / (Σ_k W̄_j[유형(d),k]·L[k,s,h,c]) x exp(drift_s·horizon)\n      W = softplus(MLP(날짜 입력))  : 목적별 활동량 (달력·방학·경기로 결정)\n      L = softplus(param)           : 역·시간·승하차별 목적 비중 (역의 성격)\n      W̄_j = 평소 수준 창(anchor j 직전) 의 평균 목적 활동량 -> 비율 = '평소 대비 구조적 변화'\n      c ∈ {승차, 하차}: 같은 W 를 공유 -> 왕복 통행 구조로 하차 자료가 목적 식별을 돕는다\n    손실: 포아송 (인원수 공간). anchor 이전 자료만 사용.\n    \"\"\"\n    import torch\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    cfg = CONFIG.get(\"struct\", {})\n    Kp, steps, lr = cfg.get(\"K\", 12), cfg.get(\"steps\", 1500), cfg.get(\"lr\", 0.02)\n    anchor, target_end = pd.Timestamp(anchor), pd.Timestamp(target_end)\n    d0 = train.date.min()\n    dates = pd.date_range(d0, target_end)\n    sids = np.array(sorted(meta.station_id))\n    sidx = {x: i for i, x in enumerate(sids)}\n    Dn, S, H = len(dates), len(sids), 18\n    Y = np.full((Dn, S, H, 2), np.nan, np.float32)\n    train = train[train.date <= anchor]  # anchor 이후 자료는 처음부터 넣지 않음 (누출 방지, 폴드 끝 이후 날짜 포함 방지)\n    di = (train.date - d0).dt.days.to_numpy()\n    si, hi = train.station_id.map(sidx).to_numpy(), train.hour.to_numpy() - 6\n    Y[di, si, hi, 0] = train.boardings.to_numpy()\n    Y[di, si, hi, 1] = train.alightings.to_numpy()\n    Y[np.arange(Dn) > (anchor - d0).days] = np.nan  # anchor 이후 자료 차단 (누출 방지)\n    a = (anchor - d0).days\n    Xd, dt = struct_date_features(dates, geo)\n\n    def baseline(ai):\n        \"\"\"anchor ai 시점 평소 수준 [3, S, H, 2] (평일/주말: 최근 8주, 공휴일: 전체 이력, 없으면 주말).\"\"\"\n        Bm = np.full((3, S, H, 2), np.nan, np.float32)\n        Wm = np.zeros((3, Dn), np.float32)\n        for k in (0, 1, 2):\n            lo = 0 if k == 2 else max(0, ai - 55)\n            idx = np.arange(lo, ai + 1)\n            idx = idx[dt[idx] == k]\n            if len(idx):\n                with np.errstate(all=\"ignore\"):\n                    Bm[k] = np.nanmean(Y[idx], axis=0)\n                Wm[k, idx] = 1.0 / len(idx)\n            all_idx = np.arange(0, ai + 1)\n            all_idx = all_idx[dt[all_idx] == k]\n            if len(all_idx):  # 최근 창에 자료 없는 역(신규역 등): 전체 이력 같은 유형 평균\n                with np.errstate(all=\"ignore\"):\n                    Bm[k] = np.where(np.isnan(Bm[k]), np.nanmean(Y[all_idx], axis=0), Bm[k])\n        Bm[2] = np.where(np.isnan(Bm[2]), Bm[1], Bm[2])\n        if Wm[2].sum() == 0:\n            Wm[2] = Wm[1]\n        return Bm, Wm\n\n    step = CONFIG.get(\"anchor_step\", 7)\n    anchors = list(range((pd.Timestamp(\"2025-02-26\") - d0).days, a - 6, step))\n    if CONFIG.get(\"smoke\"):\n        anchors = anchors[-4:]\n    pairs, bases, wms = [], [], []\n    for j, ai in enumerate(anchors):\n        Bm, Wm = baseline(ai)\n        bases.append(Bm)\n        wms.append(Wm)\n        pairs += [(j, dd) for dd in range(ai + 1, min(ai + HORIZON, a) + 1)]\n    Bt, Wt = baseline(a)\n    bases.append(Bt)\n    wms.append(Wt)\n    jt = len(anchors)\n    te_days = np.arange(a + 1, Dn)\n\n    def T(x, dtype=None):\n        return torch.tensor(np.asarray(x), device=dev, dtype=dtype)\n    Xd_t, dt_t = T(Xd), T(dt, torch.long)\n    Bs = np.stack(bases)\n    BOK = T(np.isfinite(Bs))                       # 평소 수준이 있는 칸 (개통 전 신규역 등은 False)\n    BASE = T(np.where(np.isfinite(Bs), Bs, 1.0))   # 계산 경로에 NaN 이 없도록 채움 (역전파 0·NaN 방지)\n    YOK = T(np.isfinite(Y))\n    WM, Yt = T(np.stack(wms)), T(np.nan_to_num(Y))\n    pj, pdd = T([q[0] for q in pairs], torch.long), T([q[1] for q in pairs], torch.long)\n    hz = (pdd - T([anchors[q[0]] for q in pairs], torch.long)).float() / 365.0\n    torch.manual_seed(cfg.get(\"seed\", 0))\n    net = torch.nn.Sequential(torch.nn.Linear(Xd.shape[1], 64), torch.nn.GELU(), torch.nn.Linear(64, Kp)).to(dev)\n    Lp = torch.nn.Parameter(torch.randn(Kp, S, H, 2, device=dev) * 0.1 - 2.0)\n    drift = torch.nn.Parameter(torch.zeros(S, device=dev))\n    opt = torch.optim.Adam(list(net.parameters()) + [Lp, drift], lr=lr)\n    sp = torch.nn.functional.softplus\n\n    def mu(jj, dd, hzz):\n        W = sp(net(Xd_t))                                   # [Dn, K]\n        L = sp(Lp)                                          # [K, S, H, 2]\n        num = torch.einsum(\"dk,kshc->dshc\", W[dd], L)       # [B, S, H, 2]\n        Wbar = torch.einsum(\"jtd,dk->jtk\", WM, W)            # [J, 3, K]\n        k_ = dt_t[dd]\n        den = torch.einsum(\"bk,kshc->bshc\", Wbar[jj, k_], L)\n        return BASE[jj, k_] * num / den.clamp_min(1e-6) * torch.exp(drift[None, :, None, None] * hzz[:, None, None, None])\n\n    t0 = time.time()\n    nb = cfg.get(\"batch\", 256)\n    wa = torch.tensor([1.0, cfg.get(\"w_alight\", 0.5)], device=dev)\n    gen = torch.Generator().manual_seed(0)\n    for it in range(steps):\n        b = torch.randint(0, len(pairs), (nb,), generator=gen).to(dev)\n        m_ = mu(pj[b], pdd[b], hz[b])\n        y_ = Yt[pdd[b]]\n        ok = YOK[pdd[b]] & BOK[pj[b], dt_t[pdd[b]]]\n        m2 = m_.clamp_min(1e-3)\n        nll = (m2 - y_ * torch.log(m2)) * wa\n        loss = nll[ok].mean() + cfg.get(\"l2_drift\", 1.0) * (drift ** 2).mean()\n        if not torch.isfinite(loss):\n            raise RuntimeError(f\"struct 학습 손실이 유한하지 않음 (step {it})\")\n        opt.zero_grad()\n        loss.backward()\n        opt.step()\n        if (it + 1) % max(1, steps // 4) == 0:\n            with torch.no_grad():\n                e_ = (torch.log1p(m2[..., 0]) - torch.log1p(y_[..., 0]))[ok[..., 0]]\n            p(f\"    struct step {it + 1}/{steps} loss {loss.item():.3f} 학습 rmsle {e_.pow(2).mean().sqrt().item():.4f} \"\n              f\"({time.time() - t0:.0f}s, {dev})\")\n    with torch.no_grad():\n        dd = T(te_days, torch.long)\n        jj = torch.full_like(dd, jt)\n        out = mu(jj, dd, (dd - a).float() / 365.0)[..., 0]\n        out = torch.where(BOK[jj, dt_t[dd]][..., 0], out, torch.full_like(out, float(\"nan\")))\n    return dates[te_days], sids, np.log1p(out.clamp_min(0).cpu().numpy())\n\n\ndef run_struct(train, test, meta, geo):\n    def to_rows(dts, sids, P, target):\n        D = pd.DataFrame(P.reshape(len(dts), -1), index=dts,\n                         columns=pd.MultiIndex.from_product([sids, range(6, 24)])).stack([0, 1], future_stack=True)\n        D = D.rename(\"pred\").reset_index()\n        D.columns = [\"date\", \"station_id\", \"hour\", \"pred\"]\n        return target.merge(D, on=[\"date\", \"station_id\", \"hour\"], how=\"left\")\n    new_st = meta.set_index(\"station_id\").first_date > \"2025-01-01\"\n    if CONFIG[\"mode\"] == \"struct_cv\":\n        for name in CONFIG[\"folds\"]:\n            a, e = FOLDS[name]\n            dts, sids, P = struct_fit_predict(train, meta, geo, a, e)\n            va = to_rows(dts, sids, P, train[(train.date > a) & (train.date <= e)][[\"date\", \"station_id\", \"hour\", \"lb\"]])\n            ok = va.pred.notna().to_numpy()\n            old = ~va.station_id.map(new_st).to_numpy()\n            if (~ok & old).any():\n                raise RuntimeError(f\"struct {name}: 기존역 예측 누락 {(~ok & old).sum()}행\")\n            p(f\"[{name}] struct all {rmsle(va.pred[ok], va.lb[ok]):.5f} old {rmsle(va.pred[ok & old], va.lb[ok & old]):.5f} \"\n              f\"missing {(~ok).sum()} bias {np.nanmean(va.pred - va.lb):+.4f}\")\n            va.to_csv(OUT / f\"cv_{CONFIG['tag']}_{name}.csv\", index=False)\n    else:\n        dts, sids, P = struct_fit_predict(train, meta, geo, train.date.max(), test.date.max())\n        te = to_rows(dts, sids, P, test[[\"example_id\", \"date\", \"station_id\", \"hour\"]])\n        assert te.pred.notna().all(), \"struct test 예측 누락\"\n        te[[\"example_id\"]].assign(pred_log=te.pred.to_numpy()).to_csv(OUT / f\"testpred_{CONFIG['tag']}.csv\", index=False)\n        p(f\"struct full: mean log {te.pred.mean():.4f}\")\n\n\nif __name__ == \"__main__\":\n    p(\"CONFIG\", json.dumps(CONFIG, ensure_ascii=False))\n    _t = time.time()\n    tr_, te_, meta_, geo_ = load()\n    p(f\"load {time.time() - _t:.1f}s | cpu {os.cpu_count()} | gpu {gpu_available()}\")\n    if CONFIG[\"mode\"].startswith(\"chronos\"):\n        run_chronos(tr_, te_, meta_, geo_)\n    elif CONFIG[\"mode\"].startswith(\"struct\"):\n        run_struct(tr_, te_, meta_, geo_)\n    elif CONFIG[\"mode\"] == \"cv\":\n        run_cv(tr_, meta_, geo_)\n    elif CONFIG[\"mode\"] == \"resid\":\n        # 최종 보정용: 6/30 기준 역별 표본 외 잔차만 계산해 저장\n        B_ = Builder(tr_, meta_, geo_, CONFIG[\"groups\"])\n        r_ = recent_resid(B_, tr_.date.max(), B_.feats())\n        r_.rename(\"r\").to_csv(OUT / f\"resid_{CONFIG['tag']}.csv\")\n        p(r_.describe().round(4).to_string())\n    else:\n        run_full(tr_, te_, meta_, geo_)\n", "final_post.py": "\"\"\"Final post-processing: blend model predictions (testpred_*.csv) + correction steps -> submission csv.\n\nModels (log1p space): tree = (1-XW)*LightGBM(lt1, lt2 mean) + XW*XGBoost(v3_x10, v3_x8 mean)   [env FP_XGB_W, v8d 0.25]\n                      Chronos = mean of FP_CHR tags                                            [v8d chr_e3 + wx_chr]\n                      P = 0.5*tree + 0.5*Chronos;  optional FP_EXTRA=\"tag:w\" -> P = (1-w)*P + w*tag\nCorrections (all learned from boardings up to 6/30; after 7/1 only non-boarding metadata is used):\n  1. weather (rain/snow): deviation from usual ~ precipitation (LightGBM, monotone)            x1.0\n  2. games: SSG / Bucheon FC actual game days, stations within 3 km, mean deviation by hour     x0.4\n  3. Chuseok: +-3 days blended 50/50 with the past-holiday pattern target\n  4. concerts/festivals: announced event days, stations within 2 km, past event-day deviation  x0.5~1.0\n  5. new stations: shift new-station tree level half-way toward the seasonal change of old ones\nUsage: python final_post_v9.py <work dir>   (folder containing testpred_*.csv and aux csv files)\n\"\"\"\nimport sys\nfrom pathlib import Path\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\n\nWORK = Path(sys.argv[1]) if len(sys.argv) > 1 else Path.cwd()\nsys.argv = sys.argv[:1]\nsys.path.insert(0, str(WORK))\nimport sincon_v2 as V  # noqa: E402\n\ntr, te, meta, geo = V.load()\nG = geo.set_index(\"station_id\")\nNEW = [3107, 3108, 3109]\nK = [\"date\", \"station_id\", \"hour\"]\nANCHOR = tr.date.max()\n\n\ndef typ(d):\n    return (d.date.isin(V.HSET) | (d.date.dt.dayofweek >= 5)).astype(int)\n\n\ndef rolling_dev(h, win):\n    h = h.sort_values(\"date\").copy()\n    h[\"base\"] = h.groupby([\"station_id\", \"hour\", \"off\"]).lb.transform(\n        lambda s: s.rolling(win, center=True, min_periods=(5 if win == 11 else 8)).median())\n    h[\"dev\"] = h.lb - h.base\n    return h\n\n\ndef load_pred(tag):\n    d = te[[\"example_id\"]].merge(pd.read_csv(WORK / f\"testpred_{tag}.csv\"), on=\"example_id\", how=\"left\")\n    assert d.pred_log.notna().all(), tag\n    return d.pred_log.to_numpy()\n\n\n# ---------------- 1. weather correction ----------------\ndef wx_correction():\n    W = pd.read_csv(V._find(\"weather_incheon_hourly.csv\"), parse_dates=[\"date\"]).sort_values([\"date\", \"hour\"])\n    W[\"p3\"] = W.precipitation.rolling(3, min_periods=1).sum()\n    W = W.merge(W[W.hour.between(6, 23)].groupby(\"date\").precipitation.sum().rename(\"pday\"), on=\"date\")\n    W = W.rename(columns={\"precipitation\": \"p1\", \"snowfall\": \"snow\"})\n    WX = [\"p1\", \"p3\", \"pday\", \"snow\"]\n    FE = WX + [\"hour\", \"off\"]\n\n    def prep(df):\n        d = df.merge(W[[\"date\", \"hour\"] + WX], on=[\"date\", \"hour\"], how=\"left\")\n        d[\"off\"] = typ(d)\n        d[\"month\"] = d.date.dt.month\n        return d\n    h = rolling_dev(prep(tr[[\"date\", \"station_id\", \"hour\", \"lb\"]]), 11)\n    h = h[h.base.notna()]\n    h = h[h.dev.abs() < 2]\n    g = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=7, min_child_samples=300,\n                          monotone_constraints=[-1, -1, -1, -1, 0, 0], verbose=-1).fit(h[FE], h.dev)\n    h[\"g\"] = g.predict(h[FE])\n    ref = h.groupby([\"station_id\", \"hour\", \"off\", \"month\"]).g.mean().rename(\"gref\")\n    ref2 = h.groupby([\"station_id\", \"hour\", \"off\"]).g.mean().rename(\"gref2\")\n    d = prep(te[K].copy())\n    d[\"g\"] = g.predict(d[FE])\n    d = d.merge(ref, on=[\"station_id\", \"hour\", \"off\", \"month\"], how=\"left\").merge(\n        ref2, on=[\"station_id\", \"hour\", \"off\"], how=\"left\")\n    return (d.g - d.gref.fillna(d.gref2).fillna(0.0)).to_numpy()\n\n\n# ---------------- 2. game correction ----------------\ndef game_correction():\n    evs = [(pd.read_csv(V._find(f\"{f}_home_actual.csv\"), parse_dates=[\"date\"]).set_index(\"date\").start_hour, col)\n           for f, col in ((\"ssg\", \"dist_munhak\"), (\"bucheon\", \"dist_bucheon_stadium\"))]\n\n    def tag(d):\n        kind, rel = np.full(len(d), -1), np.full(len(d), 99)\n        for i, (s, col) in enumerate(evs):\n            st = d.date.map(s).to_numpy(float)\n            r = d.hour.to_numpy() - st\n            hit = ~np.isnan(st) & (d.station_id.map(G[col]).to_numpy() < 3000) & (r >= -3) & (r <= 6)\n            kind[hit], rel[hit] = i, r[hit].astype(int)\n        return d.assign(kind=kind, rel=rel, off=typ(d))\n    h = rolling_dev(tag(tr[[\"date\", \"station_id\", \"hour\", \"lb\"]].copy()), 11)\n    h = h[(h.kind >= 0) & h.base.notna()]\n    g = h.groupby([\"station_id\", \"off\", \"kind\", \"rel\"]).dev.agg([\"sum\", \"count\"])\n    T = (g[\"sum\"] / (g[\"count\"] + 3.0)).rename(\"ev\").reset_index()\n    d = tag(te[K].copy()).merge(T, on=[\"station_id\", \"off\", \"kind\", \"rel\"], how=\"left\")\n    return np.where(d.kind.to_numpy() >= 0, d.ev.fillna(0.0).to_numpy(), 0.0)\n\n\n# ---------------- 3. Chuseok correction (target) ----------------\ndef holiday_target():\n    def rel_of(dates):\n        rel = pd.Series(99, index=dates.index)\n        for b in V.BIG_HOLIDAYS:\n            r = (dates - b).dt.days\n            rel = rel.where(~((r.abs() <= 3) & (rel == 99)), r)\n        return rel\n\n    def lvl(h, end):\n        x = h[(h.date > end - pd.Timedelta(days=28)) & (h.date <= end)]\n        return x.groupby([\"station_id\", \"hour\", \"t\"]).lb.median().rename(\"lvl\")\n    h = tr[[\"date\", \"station_id\", \"hour\", \"lb\"]].copy()\n    h[\"t\"] = typ(h)\n    devs = []\n    for b in V.BIG_HOLIDAYS:\n        if b + pd.Timedelta(days=3) > ANCHOR:\n            continue\n        x = h[(h.date - b).dt.days.abs() <= 3].copy()\n        x[\"rel\"] = (x.date - b).dt.days\n        x = x.merge(lvl(h, b - pd.Timedelta(days=11)).reset_index(), on=[\"station_id\", \"hour\", \"t\"], how=\"left\")\n        x[\"dev\"] = x.lb - x.lvl\n        devs.append(x[[\"station_id\", \"hour\", \"rel\", \"dev\"]])\n    D = pd.concat(devs).groupby([\"station_id\", \"hour\", \"rel\"]).dev.mean().rename(\"hdev\").reset_index()\n    d = te[K].copy()\n    d[\"t\"], d[\"rel\"] = typ(d), rel_of(d.date)\n    d = d.merge(D, on=[\"station_id\", \"hour\", \"rel\"], how=\"left\").merge(\n        lvl(h, ANCHOR).reset_index(), on=[\"station_id\", \"hour\", \"t\"], how=\"left\")\n    tg = (d.lvl + d.hdev).to_numpy()\n    return np.where((d.rel != 99).to_numpy() & ~np.isnan(tg), tg, np.nan)\n\n\n# ---------------- 4. concert / festival correction ----------------\nVENUES = {\"munhak\": [3127], \"asiad\": [3209, 3210], \"songdo\": [3139], \"bupyeong\": [3119]}\n# venues whose past event days are set from announced schedules instead of spike detection (Bupyeong festival 2025: 9/26-28)\nHIST_DAYS = {\"bupyeong\": [\"2025-09-26\", \"2025-09-27\", \"2025-09-28\"]}\nEVENTS = {  # announced schedules (web): venue -> (dates, scale); concert 1.0, day festival 1.0, Songdo festival 0.5 (checked on public LB)\n    # Bupyeong festival (10/2-4) dropped: no effect on public LB\n    \"munhak\": ([\"2026-09-12\", \"2026-09-13\"], 1.0),                                        # PLAVE concert\n    \"munhak_fest\": ([\"2026-08-29\", \"2026-08-30\", \"2026-10-24\", \"2026-10-25\"], 1.0),        # Sustainable Wave, GHOSTIVAL\n    \"asiad\": ([\"2026-07-11\"], 1.0),                                                        # PSY summer concert\n    \"songdo\": ([\"2026-07-31\", \"2026-08-01\", \"2026-08-02\", \"2026-08-08\", \"2026-08-15\",      # Pentaport, Songdo beach festival\n                \"2026-08-22\", \"2026-08-23\", \"2026-08-24\", \"2026-08-25\", \"2026-08-26\",      # Songdo beer festival\n                \"2026-08-27\", \"2026-08-28\", \"2026-08-29\", \"2026-08-30\", \"2026-09-12\"], 0.5),  # Isle Live\n}\n\n\ndef venue_correction():\n    ssg = set(pd.read_csv(V._find(\"ssg_home_actual.csv\"), parse_dates=[\"date\"]).date)\n    h = tr[[\"date\", \"station_id\", \"hour\", \"lb\", \"boardings\"]].copy()\n    h[\"off\"] = typ(h)\n    h = rolling_dev(h, 21)\n    T = {}\n    for v, reps in VENUES.items():\n        if v in HIST_DAYS:\n            days = list(pd.to_datetime(HIST_DAYS[v]))\n        else:\n            r = h[h.station_id.isin(reps) & (h.dev > 1.5) & (h.boardings > 300)]\n            if v == \"munhak\":\n                r = r[~r.date.isin(ssg)]\n            days = sorted(set(r.date))\n        lat, lon = G.loc[reps[0], \"lat\"], G.loc[reps[0], \"lon\"]\n        near = [s for s in G.index if V.hav(G.loc[s, \"lat\"], G.loc[s, \"lon\"], lat, lon) < 2000]\n        x = h[h.date.isin(days) & h.station_id.isin(near)]\n        g = x.groupby([\"station_id\", \"hour\"]).dev.agg([\"sum\", \"count\"])\n        T[v] = (g[\"sum\"] / (g[\"count\"] + 2.0)).rename(\"vn\").reset_index()\n        print(f\"  venue {v}: {len(days)} past event days, {len(near)} nearby stations\", flush=True)\n    out = np.zeros(len(te))\n    for k, (ds, sc) in EVENTS.items():\n        hit = te.date.isin(pd.to_datetime(ds)).to_numpy()\n        m = te[[\"station_id\", \"hour\"]].merge(T[k.split(\"_\")[0]], on=[\"station_id\", \"hour\"], how=\"left\").vn.fillna(0.0)\n        out += sc * np.where(hit, m.to_numpy(), 0.0)\n    return out\n\n\n# ---------------- 5. new-station correction ----------------\ndef newst_shift(P, alpha=0.5):\n    h = tr[(tr.date > ANCHOR - pd.Timedelta(days=28))][[\"date\", \"station_id\", \"hour\", \"lb\"]].copy()\n    h[\"t\"] = typ(h)\n    L = h.groupby([\"station_id\", \"t\", \"hour\"]).lb.mean().rename(\"lvl\").reset_index()\n    p = te[K].assign(p=P)\n    p[\"t\"], p[\"m\"] = typ(p), p.date.dt.to_period(\"M\")\n    p = p.merge(L, on=[\"station_id\", \"t\", \"hour\"], how=\"left\")\n    p[\"d\"] = p.p - p.lvl\n    isnew = p.station_id.isin(NEW).to_numpy()\n    D = p[~isnew].groupby([\"m\", \"t\", \"hour\"]).d.mean().rename(\"D\").reset_index()\n    q = p[isnew].merge(D, on=[\"m\", \"t\", \"hour\"], how=\"left\")\n    q[\"gap\"] = q.lvl + q.D - q.p\n    sh = q.groupby([\"station_id\", \"m\", \"t\"]).gap.mean().rename(\"sh\").reset_index()\n    p = p.merge(sh, on=[\"station_id\", \"m\", \"t\"], how=\"left\")\n    return np.where(isnew, P + alpha * p.sh.fillna(0.0).to_numpy(), P)\n\n\nif __name__ == \"__main__\":\n    import os\n    LGB_TAGS = os.environ.get(\"FP_LGB\", \"lt1_full,lt2_full\").split(\",\")\n    OUT_NAME = os.environ.get(\"FP_OUT\", \"submission.csv\")\n    lgbm = np.mean([load_pred(t) for t in LGB_TAGS], axis=0)\n    XW = float(os.environ.get(\"FP_XGB_W\", \"0.25\"))  # v9: XGBoost share inside the tree part (v8d 0.25)\n    CHR = os.environ.get(\"FP_CHR\", \"chr_e3_full,wx_chr_full\").split(\",\")  # v9: Chronos members\n    xgb = (load_pred(\"v3_x10\") + load_pred(\"v3_x8\")) / 2 if XW > 0 else lgbm  # XGB files not needed when XW=0\n    tree = newst_shift((1 - XW) * lgbm + XW * xgb)\n    chron = np.mean([load_pred(t) for t in CHR], axis=0)\n    print(\"FP_XGB_W\", XW, \"FP_CHR\", CHR, flush=True)\n    P = 0.5 * tree + 0.5 * chron\n    EX = os.environ.get(\"FP_EXTRA\", \"\")  # v10: \"tag:w\" -> P = (1-w)*P + w*model (before corrections)\n    if EX:\n        tg, w = EX.split(\":\")\n        P = (1 - float(w)) * P + float(w) * load_pred(tg)\n        print(\"FP_EXTRA\", tg, w, flush=True)\n    print(\"mean log: lgbm %.4f xgb %.4f chronos %.4f\" % (lgbm.mean(), xgb.mean(), chron.mean()), flush=True)\n    cr, ev = wx_correction(), game_correction()\n    print(f\"weather corr mean {cr.mean():+.4f} sd {cr.std():.4f} | game-corrected rows {(ev != 0).sum()}\", flush=True)\n    P = P + cr + 0.4 * ev\n    htg = holiday_target()\n    P = np.where(np.isnan(htg), P, 0.5 * P + 0.5 * np.nan_to_num(htg))\n    print(f\"Chuseok-corrected rows {(~np.isnan(htg)).sum()}\", flush=True)\n    vn = venue_correction()\n    P = np.clip(P + vn, 0, None)\n    print(f\"event-corrected rows {(vn != 0).sum()}, max {vn.max():.2f}\", flush=True)\n    sub = pd.DataFrame({\"example_id\": te.example_id, \"boardings\": np.expm1(P)})\n    assert len(sub) == 157194 and sub.boardings.notna().all() and (sub.boardings >= 0).all()\n    sub.to_csv(WORK / OUT_NAME, index=False)\n    print(\"->\", WORK / OUT_NAME)\n    print(sub.boardings.describe().round(1))\n", "exp3_newmodels.py": "\"\"\"v10 experiment: two model families different from the existing blend, for folds A, B and the test period.\n  lgbL : light LightGBM (organizer daily weather + calendar + 8-week level before the anchor date)\n  mlp  : embedding MLP (train-only standardization, AdamW, LayerNorm, Dropout, cosine schedule,\n         gradient clipping, early stopping on the last 8 weeks before the anchor, then refit on all history)\nTarget = log1p(boardings) - 8-week level (lvl); lvl is added back to the prediction.\nUsage : python exp3_newmodels.py A|B|full [seed...]   (env MODELS=lgbL,mlp selects models)\nOutput: cv_<model>_<fold>.csv (date,station_id,hour,lb,pred) / testpred_<model>_full.csv (example_id,pred_log)\n\"\"\"\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\n\nROOT = Path(__file__).resolve().parent.parent\nOUT = Path(__file__).resolve().parent / \"results_v10\"\nOUT.mkdir(exist_ok=True)\nMODE = sys.argv[1]\nSEEDS = [int(s) for s in sys.argv[2:]] or [0, 1, 2]\nK = [\"date\", \"station_id\", \"hour\"]\ntorch.set_num_threads(16)\n\ntr = pd.read_csv(ROOT / \"sincon-2026/train.csv\", parse_dates=[\"date\"])\nte = pd.read_csv(ROOT / \"sincon-2026/test.csv\", parse_dates=[\"date\"])\ncal = pd.read_csv(ROOT / \"calendar.csv\", encoding=\"utf-8-sig\", parse_dates=[\"date\"]).drop(columns=\"holiday_name\")\nW = pd.concat([pd.read_csv(ROOT / f\"weather_data/{f}_weather.csv\", encoding=\"utf-8-sig\", parse_dates=[\"date\"])\n               for f in (\"train\", \"test\")])\nWX = [\"avg_temp\", \"max_temp\", \"min_temp\", \"rainfall\", \"avg_wind\", \"sunshine\", \"humidity\", \"has_rain\"]\nW = W.groupby(\"date\")[WX].mean().reset_index()\ntr[\"lb\"] = np.log1p(tr.boardings)\nanchor = {\"A\": \"2025-06-30\", \"B\": \"2026-02-28\", \"full\": \"2026-06-30\"}[MODE]\nend = {\"A\": \"2025-10-31\", \"B\": \"2026-06-30\", \"full\": None}[MODE]\nanchor = pd.Timestamp(anchor)\nhist = tr[tr.date <= anchor]\ntgt = te[K + [\"example_id\"]].copy() if MODE == \"full\" else tr[(tr.date > anchor) & (tr.date <= pd.Timestamp(end))][K + [\"lb\"]].copy()\n\n\ndef build(anchor, hist, tgt):\n    df = pd.concat([hist[K + [\"lb\"]].assign(is_tgt=0), tgt.assign(is_tgt=1)], ignore_index=True)\n    df = df.merge(cal, on=\"date\", how=\"left\").merge(W, on=\"date\", how=\"left\")\n    d = df.date\n    df[\"dow\"] = d.dt.day_of_week\n    df[\"daytype\"] = np.where(df.is_holiday == 1, 2, np.where(df.dow >= 5, 1, 0))\n    df[\"off\"] = (df.daytype > 0).astype(int)\n    df[\"month\"], df[\"doy\"], df[\"dom\"] = d.dt.month, d.dt.day_of_year, d.dt.day\n    df[\"t_year\"] = (d - pd.Timestamp(\"2025-01-01\")).dt.days / 365.0\n    rec = hist[hist.date > anchor - pd.Timedelta(days=56)].merge(cal[[\"date\", \"is_offday\"]], on=\"date\")\n    L = rec.groupby([\"station_id\", \"hour\", \"is_offday\"]).lb.mean().rename(\"lvl\").reset_index().rename(columns={\"is_offday\": \"off\"})\n    Ld = rec.groupby([\"station_id\", \"hour\", rec.date.dt.day_of_week.rename(\"dow\")]).lb.mean().rename(\"lvl_dow\").reset_index()\n    La = rec.groupby([\"station_id\", \"hour\"]).lb.mean().rename(\"lvl_all\").reset_index()\n    df = df.merge(L, on=[\"station_id\", \"hour\", \"off\"], how=\"left\").merge(Ld, on=[\"station_id\", \"hour\", \"dow\"], how=\"left\") \\\n           .merge(La, on=[\"station_id\", \"hour\"], how=\"left\")\n    df[\"lvl\"] = df.lvl.fillna(df.lvl_all)\n    df[\"lvl_dow\"] = df.lvl_dow.fillna(df.lvl)\n    df[\"lvl\"] = df.lvl.fillna(df.groupby(\"hour\").lvl.transform(\"median\"))\n    df[\"lvl_dow\"] = df.lvl_dow.fillna(df.lvl)\n    df[\"lvl_dow_d\"] = df.lvl_dow - df.lvl\n    df[\"y\"] = df.lb - df.lvl\n    return df\n\n\ndf = build(anchor, hist, tgt)\nCAL = [\"is_holiday\", \"is_offday\", \"offday_run_len\", \"offday_day_idx\", \"offday_days_left\", \"is_before_long_offday\",\n       \"is_after_long_offday\", \"is_univ_vacation\", \"is_school_vacation\", \"daytype\", \"dom\", \"doy\", \"t_year\"]\nNUM = CAL + WX + [\"lvl\", \"lvl_dow_d\"]\nEMB = {\"station_id\": None, \"hour\": None, \"dow\": 7, \"month\": 13}\nH, T = df[df.is_tgt == 0].copy(), df[df.is_tgt == 1].copy()\nst_codes = {s: i for i, s in enumerate(sorted(df.station_id.unique()))}\nfor x in (H, T):\n    x[\"st_i\"] = x.station_id.map(st_codes)\n    x[\"hr_i\"] = x.hour - 6\n\n\ndef fit_lgb(seed):\n    F = [\"station_id\", \"hour\", \"dow\", \"month\"] + NUM\n    ds = lgb.Dataset(H[F], H.y, categorical_feature=[\"station_id\"])\n    m = lgb.train(dict(objective=\"regression\", learning_rate=0.03, num_leaves=127, min_data_in_leaf=50,\n                       bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8, verbose=-1, num_threads=16,\n                       seed=seed), ds, num_boost_round=1500)\n    return m.predict(T[F])\n\n\nclass Net(nn.Module):\n    def __init__(self, n_num, n_st, hidden=256, drop=0.1):\n        super().__init__()\n        self.e_st, self.e_hr = nn.Embedding(n_st, 16), nn.Embedding(18, 8)\n        self.e_dw, self.e_mo = nn.Embedding(7, 4), nn.Embedding(13, 4)\n        self.e_sh = nn.Embedding(n_st * 18 * 3, 8)  # station x hour x day-type interaction\n        d = n_num + 16 + 8 + 4 + 4 + 8\n        self.body = nn.Sequential(nn.Linear(d, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(drop),\n                                  nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(drop),\n                                  nn.Linear(hidden, 1))\n\n    def forward(self, xn, st, hr, dw, mo, dt):\n        z = torch.cat([xn, self.e_st(st), self.e_hr(hr), self.e_dw(dw), self.e_mo(mo),\n                       self.e_sh((st * 18 + hr) * 3 + dt)], 1)\n        return self.body(z).squeeze(1)\n\n\ndef tens(x, mu, sd):\n    return (torch.tensor(((x[NUM].to_numpy(float) - mu) / sd), dtype=torch.float32),\n            torch.tensor(x.st_i.to_numpy()), torch.tensor(x.hr_i.to_numpy()), torch.tensor(x.dow.to_numpy()),\n            torch.tensor(x.month.to_numpy()), torch.tensor(x.daytype.to_numpy()))\n\n\ndef train_mlp(Htr, seed, epochs, Hva=None, lr=2e-3):\n    torch.manual_seed(seed)\n    mu, sd = Htr[NUM].to_numpy(float).mean(0), Htr[NUM].to_numpy(float).std(0) + 1e-6  # train statistics only\n    Xtr, ytr = tens(Htr, mu, sd), torch.tensor(Htr.y.to_numpy(), dtype=torch.float32)\n    net = Net(len(NUM), len(st_codes))\n    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)\n    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)\n    g = torch.Generator().manual_seed(seed + 1000)\n    n, bs, hist_ = len(ytr), 2048, []\n    best, best_ep = 1e9, epochs\n    for ep in range(epochs):\n        net.train()\n        for idx in torch.randperm(n, generator=g).split(bs):\n            opt.zero_grad(set_to_none=True)\n            loss = ((net(*[a[idx] for a in Xtr]) - ytr[idx]) ** 2).mean()\n            loss.backward()\n            nn.utils.clip_grad_norm_(net.parameters(), 1.0)\n            opt.step()\n        sch.step()\n        if Hva is not None:\n            net.eval()\n            with torch.no_grad():\n                p = net(*tens(Hva, mu, sd)).numpy()\n            v = float(np.sqrt(np.mean((p - Hva.y.to_numpy()) ** 2)))\n            hist_.append(v)\n            if v < best:\n                best, best_ep = v, ep + 1\n    return net, mu, sd, best_ep, hist_\n\n\ndef fit_mlp(seed):\n    # 1) hold out the last 8 weeks before the anchor to pick the epoch count (time split, no leakage)\n    cut = anchor - pd.Timedelta(days=56)\n    _, _, _, best_ep, hv = train_mlp(H[H.date <= cut], seed, 16, H[H.date > cut])\n    # 2) refit on the full history for that many epochs\n    net, mu, sd, _, _ = train_mlp(H, seed, best_ep)\n    net.eval()\n    with torch.no_grad():\n        p = net(*tens(T, mu, sd)).numpy()\n    print(f\"  mlp seed{seed}: inner-val {[round(x, 4) for x in hv]} -> epoch {best_ep}\", flush=True)\n    return p\n\n\nMODELS = os.environ.get(\"MODELS\", \"lgbL,mlp\").split(\",\")\nfor name, fn in [(n, f) for n, f in ((\"lgbL\", fit_lgb), (\"mlp\", fit_mlp)) if n in MODELS]:\n    t0 = time.time()\n    ps = [fn(s) for s in SEEDS]\n    pred = np.mean(ps, axis=0) + T.lvl.to_numpy()\n    if MODE == \"full\":\n        T[[\"example_id\"]].assign(pred_log=pred).to_csv(OUT / f\"testpred_{name}_full.csv\", index=False)\n        print(f\"{name} full saved, mean {pred.mean():.4f} ({time.time() - t0:.0f}s)\", flush=True)\n    else:\n        r = lambda p: np.sqrt(np.mean((p - T.lb.to_numpy()) ** 2))\n        each = [round(r(p + T.lvl.to_numpy()), 4) for p in ps]\n        print(f\"{name} fold {MODE}: per-seed {each} seed-avg {r(pred):.4f} ({time.time() - t0:.0f}s)\", flush=True)\n        T[K + [\"lb\"]].assign(pred=pred).to_csv(OUT / f\"cv_{name}_{MODE}.csv\", index=False)\n"}
for name, text in CODES.items():
    (WORK / name).write_text(text, encoding="utf-8")
print({k: len(v) for k, v in CODES.items()})

In [ ]:
RECIPE = {'steps': [{'mode': 'full', 'model': 'lgb', 'groups': ['base', 'newst', 'graph', 'event', 'analog', 'trend', 'vac', 'wx'], 'seeds': [0], 'rounds': 2500, 'lr': 0.03, 'cs': False, 'tag': 'lt1_full', 'ev_actual': True, 'lgb_params': {'num_leaves': 255, 'min_data_in_leaf': 100, 'feature_fraction': 0.6}}, {'mode': 'full', 'model': 'lgb', 'groups': ['base', 'newst', 'graph', 'event', 'analog', 'trend', 'vac', 'wx'], 'seeds': [1], 'rounds': 2500, 'lr': 0.03, 'cs': False, 'tag': 'lt2_full', 'ev_actual': True, 'lgb_params': {'num_leaves': 63, 'min_data_in_leaf': 300, 'feature_fraction': 0.5, 'lambda_l2': 10.0}}, {'mode': 'chronos_full', 'tag': 'chr_e3_full', 'chr_geo': True}, {'mode': 'chronos_full', 'tag': 'wx_chr_st_full', 'chr_struct': {'K': 10}, 'chr_geo': True, 'chr_bfc': True, 'chr_wx': True, 'ev_actual': True}]}

def run(cmd, env=None, name=""):
    if os.environ.get("SINCON_DRY"):  # 생성기 점검용: 실제 학습 없이 셀 흐름만 실행
        print("DRY", name, cmd[-1:])
        return
    t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "PYTHONIOENCODING": "utf-8", **(env or {})})
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError(f"{name} 실패")
    print(f"== {name} {time.time() - t:.0f}s", flush=True)

for cfg in RECIPE["steps"]:
    path = WORK / f"_run_{cfg['tag']}.py"
    path.write_text(f"CONFIG_OVERRIDE = {cfg!r}" + chr(10) + CODES["sincon_v2.py"], encoding="utf-8")
    run([sys.executable, "-u", str(path)], name=cfg["tag"])

In [ ]:
# aux data (not in competition files): calendar, organizer daily weather, log(h9) - log(v8d)
EMB_CAL = 'date,is_holiday,holiday_name,is_offday,offday_run_len,offday_day_idx,offday_days_left,is_before_offday,is_after_offday,is_before_long_offday,is_after_long_offday,is_univ_vacation,is_school_vacation\n2025-01-01,1,신정,1,1,1,0,0,0,0,0,1,1\n2025-01-02,0,,0,0,0,0,0,1,0,0,1,1\n2025-01-03,0,,0,0,0,0,1,0,0,0,1,1\n2025-01-04,0,,1,2,1,1,0,0,0,0,1,1\n2025-01-05,0,,1,2,2,0,0,0,0,0,1,1\n2025-01-06,0,,0,0,0,0,0,1,0,0,1,1\n2025-01-07,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-08,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-09,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-10,0,,0,0,0,0,1,0,0,0,1,1\n2025-01-11,0,,1,2,1,1,0,0,0,0,1,1\n2025-01-12,0,,1,2,2,0,0,0,0,0,1,1\n2025-01-13,0,,0,0,0,0,0,1,0,0,1,1\n2025-01-14,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-15,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-16,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-17,0,,0,0,0,0,1,0,0,0,1,1\n2025-01-18,0,,1,2,1,1,0,0,0,0,1,1\n2025-01-19,0,,1,2,2,0,0,0,0,0,1,1\n2025-01-20,0,,0,0,0,0,0,1,0,0,1,1\n2025-01-21,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-22,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-23,0,,0,0,0,0,0,0,0,0,1,1\n2025-01-24,0,,0,0,0,0,1,0,1,0,1,1\n2025-01-25,0,,1,6,1,5,0,0,0,0,1,1\n2025-01-26,0,,1,6,2,4,0,0,0,0,1,1\n2025-01-27,1,임시공휴일,1,6,3,3,0,0,0,0,1,1\n2025-01-28,1,설날 전날,1,6,4,2,0,0,0,0,1,1\n2025-01-29,1,설날,1,6,5,1,0,0,0,0,1,1\n2025-01-30,1,설날 다음날,1,6,6,0,0,0,0,0,1,1\n2025-01-31,0,,0,0,0,0,1,1,0,1,1,1\n2025-02-01,0,,1,2,1,1,0,0,0,0,1,1\n2025-02-02,0,,1,2,2,0,0,0,0,0,1,1\n2025-02-03,0,,0,0,0,0,0,1,0,0,1,1\n2025-02-04,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-05,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-06,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-07,0,,0,0,0,0,1,0,0,0,1,1\n2025-02-08,0,,1,2,1,1,0,0,0,0,1,1\n2025-02-09,0,,1,2,2,0,0,0,0,0,1,1\n2025-02-10,0,,0,0,0,0,0,1,0,0,1,1\n2025-02-11,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-12,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-13,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-14,0,,0,0,0,0,1,0,0,0,1,1\n2025-02-15,0,,1,2,1,1,0,0,0,0,1,1\n2025-02-16,0,,1,2,2,0,0,0,0,0,1,1\n2025-02-17,0,,0,0,0,0,0,1,0,0,1,1\n2025-02-18,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-19,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-20,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-21,0,,0,0,0,0,1,0,0,0,1,1\n2025-02-22,0,,1,2,1,1,0,0,0,0,1,1\n2025-02-23,0,,1,2,2,0,0,0,0,0,1,1\n2025-02-24,0,,0,0,0,0,0,1,0,0,1,1\n2025-02-25,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-26,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-27,0,,0,0,0,0,0,0,0,0,1,1\n2025-02-28,0,,0,0,0,0,1,0,1,0,1,1\n2025-03-01,1,삼일절,1,3,1,2,0,0,0,0,0,0\n2025-03-02,0,,1,3,2,1,0,0,0,0,0,0\n2025-03-03,1,삼일절 대체공휴일,1,3,3,0,0,0,0,0,0,0\n2025-03-04,0,,0,0,0,0,0,1,0,1,0,0\n2025-03-05,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-06,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-07,0,,0,0,0,0,1,0,0,0,0,0\n2025-03-08,0,,1,2,1,1,0,0,0,0,0,0\n2025-03-09,0,,1,2,2,0,0,0,0,0,0,0\n2025-03-10,0,,0,0,0,0,0,1,0,0,0,0\n2025-03-11,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-12,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-13,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-14,0,,0,0,0,0,1,0,0,0,0,0\n2025-03-15,0,,1,2,1,1,0,0,0,0,0,0\n2025-03-16,0,,1,2,2,0,0,0,0,0,0,0\n2025-03-17,0,,0,0,0,0,0,1,0,0,0,0\n2025-03-18,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-19,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-20,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-21,0,,0,0,0,0,1,0,0,0,0,0\n2025-03-22,0,,1,2,1,1,0,0,0,0,0,0\n2025-03-23,0,,1,2,2,0,0,0,0,0,0,0\n2025-03-24,0,,0,0,0,0,0,1,0,0,0,0\n2025-03-25,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-26,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-27,0,,0,0,0,0,0,0,0,0,0,0\n2025-03-28,0,,0,0,0,0,1,0,0,0,0,0\n2025-03-29,0,,1,2,1,1,0,0,0,0,0,0\n2025-03-30,0,,1,2,2,0,0,0,0,0,0,0\n2025-03-31,0,,0,0,0,0,0,1,0,0,0,0\n2025-04-01,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-02,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-03,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-04,0,,0,0,0,0,1,0,0,0,0,0\n2025-04-05,0,,1,2,1,1,0,0,0,0,0,0\n2025-04-06,0,,1,2,2,0,0,0,0,0,0,0\n2025-04-07,0,,0,0,0,0,0,1,0,0,0,0\n2025-04-08,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-09,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-10,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-11,0,,0,0,0,0,1,0,0,0,0,0\n2025-04-12,0,,1,2,1,1,0,0,0,0,0,0\n2025-04-13,0,,1,2,2,0,0,0,0,0,0,0\n2025-04-14,0,,0,0,0,0,0,1,0,0,0,0\n2025-04-15,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-16,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-17,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-18,0,,0,0,0,0,1,0,0,0,0,0\n2025-04-19,0,,1,2,1,1,0,0,0,0,0,0\n2025-04-20,0,,1,2,2,0,0,0,0,0,0,0\n2025-04-21,0,,0,0,0,0,0,1,0,0,0,0\n2025-04-22,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-23,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-24,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-25,0,,0,0,0,0,1,0,0,0,0,0\n2025-04-26,0,,1,2,1,1,0,0,0,0,0,0\n2025-04-27,0,,1,2,2,0,0,0,0,0,0,0\n2025-04-28,0,,0,0,0,0,0,1,0,0,0,0\n2025-04-29,0,,0,0,0,0,0,0,0,0,0,0\n2025-04-30,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-01,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-02,0,,0,0,0,0,1,0,1,0,0,0\n2025-05-03,0,,1,4,1,3,0,0,0,0,0,0\n2025-05-04,0,,1,4,2,2,0,0,0,0,0,0\n2025-05-05,1,어린이날·부처님오신날,1,4,3,1,0,0,0,0,0,0\n2025-05-06,1,어린이날·부처님오신날 대체공휴일,1,4,4,0,0,0,0,0,0,0\n2025-05-07,0,,0,0,0,0,0,1,0,1,0,0\n2025-05-08,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-09,0,,0,0,0,0,1,0,0,0,0,0\n2025-05-10,0,,1,2,1,1,0,0,0,0,0,0\n2025-05-11,0,,1,2,2,0,0,0,0,0,0,0\n2025-05-12,0,,0,0,0,0,0,1,0,0,0,0\n2025-05-13,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-14,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-15,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-16,0,,0,0,0,0,1,0,0,0,0,0\n2025-05-17,0,,1,2,1,1,0,0,0,0,0,0\n2025-05-18,0,,1,2,2,0,0,0,0,0,0,0\n2025-05-19,0,,0,0,0,0,0,1,0,0,0,0\n2025-05-20,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-21,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-22,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-23,0,,0,0,0,0,1,0,0,0,0,0\n2025-05-24,0,,1,2,1,1,0,0,0,0,0,0\n2025-05-25,0,,1,2,2,0,0,0,0,0,0,0\n2025-05-26,0,,0,0,0,0,0,1,0,0,0,0\n2025-05-27,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-28,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-29,0,,0,0,0,0,0,0,0,0,0,0\n2025-05-30,0,,0,0,0,0,1,0,0,0,0,0\n2025-05-31,0,,1,2,1,1,0,0,0,0,0,0\n2025-06-01,0,,1,2,2,0,0,0,0,0,0,0\n2025-06-02,0,,0,0,0,0,1,1,0,0,0,0\n2025-06-03,1,대통령 선거일,1,1,1,0,0,0,0,0,0,0\n2025-06-04,0,,0,0,0,0,0,1,0,0,0,0\n2025-06-05,0,,0,0,0,0,1,0,1,0,0,0\n2025-06-06,1,현충일,1,3,1,2,0,0,0,0,0,0\n2025-06-07,0,,1,3,2,1,0,0,0,0,0,0\n2025-06-08,0,,1,3,3,0,0,0,0,0,0,0\n2025-06-09,0,,0,0,0,0,0,1,0,1,0,0\n2025-06-10,0,,0,0,0,0,0,0,0,0,0,0\n2025-06-11,0,,0,0,0,0,0,0,0,0,0,0\n2025-06-12,0,,0,0,0,0,0,0,0,0,0,0\n2025-06-13,0,,0,0,0,0,1,0,0,0,0,0\n2025-06-14,0,,1,2,1,1,0,0,0,0,0,0\n2025-06-15,0,,1,2,2,0,0,0,0,0,0,0\n2025-06-16,0,,0,0,0,0,0,1,0,0,0,0\n2025-06-17,0,,0,0,0,0,0,0,0,0,0,0\n2025-06-18,0,,0,0,0,0,0,0,0,0,0,0\n2025-06-19,0,,0,0,0,0,0,0,0,0,0,0\n2025-06-20,0,,0,0,0,0,1,0,0,0,0,0\n2025-06-21,0,,1,2,1,1,0,0,0,0,1,0\n2025-06-22,0,,1,2,2,0,0,0,0,0,1,0\n2025-06-23,0,,0,0,0,0,0,1,0,0,1,0\n2025-06-24,0,,0,0,0,0,0,0,0,0,1,0\n2025-06-25,0,,0,0,0,0,0,0,0,0,1,0\n2025-06-26,0,,0,0,0,0,0,0,0,0,1,0\n2025-06-27,0,,0,0,0,0,1,0,0,0,1,0\n2025-06-28,0,,1,2,1,1,0,0,0,0,1,0\n2025-06-29,0,,1,2,2,0,0,0,0,0,1,0\n2025-06-30,0,,0,0,0,0,0,1,0,0,1,0\n2025-07-01,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-02,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-03,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-04,0,,0,0,0,0,1,0,0,0,1,0\n2025-07-05,0,,1,2,1,1,0,0,0,0,1,0\n2025-07-06,0,,1,2,2,0,0,0,0,0,1,0\n2025-07-07,0,,0,0,0,0,0,1,0,0,1,0\n2025-07-08,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-09,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-10,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-11,0,,0,0,0,0,1,0,0,0,1,0\n2025-07-12,0,,1,2,1,1,0,0,0,0,1,0\n2025-07-13,0,,1,2,2,0,0,0,0,0,1,0\n2025-07-14,0,,0,0,0,0,0,1,0,0,1,0\n2025-07-15,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-16,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-17,0,,0,0,0,0,0,0,0,0,1,0\n2025-07-18,0,,0,0,0,0,1,0,0,0,1,0\n2025-07-19,0,,1,2,1,1,0,0,0,0,1,0\n2025-07-20,0,,1,2,2,0,0,0,0,0,1,0\n2025-07-21,0,,0,0,0,0,0,1,0,0,1,1\n2025-07-22,0,,0,0,0,0,0,0,0,0,1,1\n2025-07-23,0,,0,0,0,0,0,0,0,0,1,1\n2025-07-24,0,,0,0,0,0,0,0,0,0,1,1\n2025-07-25,0,,0,0,0,0,1,0,0,0,1,1\n2025-07-26,0,,1,2,1,1,0,0,0,0,1,1\n2025-07-27,0,,1,2,2,0,0,0,0,0,1,1\n2025-07-28,0,,0,0,0,0,0,1,0,0,1,1\n2025-07-29,0,,0,0,0,0,0,0,0,0,1,1\n2025-07-30,0,,0,0,0,0,0,0,0,0,1,1\n2025-07-31,0,,0,0,0,0,0,0,0,0,1,1\n2025-08-01,0,,0,0,0,0,1,0,0,0,1,1\n2025-08-02,0,,1,2,1,1,0,0,0,0,1,1\n2025-08-03,0,,1,2,2,0,0,0,0,0,1,1\n2025-08-04,0,,0,0,0,0,0,1,0,0,1,1\n2025-08-05,0,,0,0,0,0,0,0,0,0,1,1\n2025-08-06,0,,0,0,0,0,0,0,0,0,1,1\n2025-08-07,0,,0,0,0,0,0,0,0,0,1,1\n2025-08-08,0,,0,0,0,0,1,0,0,0,1,1\n2025-08-09,0,,1,2,1,1,0,0,0,0,1,1\n2025-08-10,0,,1,2,2,0,0,0,0,0,1,1\n2025-08-11,0,,0,0,0,0,0,1,0,0,1,1\n2025-08-12,0,,0,0,0,0,0,0,0,0,1,1\n2025-08-13,0,,0,0,0,0,0,0,0,0,1,1\n2025-08-14,0,,0,0,0,0,1,0,1,0,1,1\n2025-08-15,1,광복절,1,3,1,2,0,0,0,0,1,1\n2025-08-16,0,,1,3,2,1,0,0,0,0,1,1\n2025-08-17,0,,1,3,3,0,0,0,0,0,1,1\n2025-08-18,0,,0,0,0,0,0,1,0,1,1,0\n2025-08-19,0,,0,0,0,0,0,0,0,0,1,0\n2025-08-20,0,,0,0,0,0,0,0,0,0,1,0\n2025-08-21,0,,0,0,0,0,0,0,0,0,1,0\n2025-08-22,0,,0,0,0,0,1,0,0,0,1,0\n2025-08-23,0,,1,2,1,1,0,0,0,0,1,0\n2025-08-24,0,,1,2,2,0,0,0,0,0,1,0\n2025-08-25,0,,0,0,0,0,0,1,0,0,1,0\n2025-08-26,0,,0,0,0,0,0,0,0,0,1,0\n2025-08-27,0,,0,0,0,0,0,0,0,0,1,0\n2025-08-28,0,,0,0,0,0,0,0,0,0,1,0\n2025-08-29,0,,0,0,0,0,1,0,0,0,1,0\n2025-08-30,0,,1,2,1,1,0,0,0,0,1,0\n2025-08-31,0,,1,2,2,0,0,0,0,0,1,0\n2025-09-01,0,,0,0,0,0,0,1,0,0,0,0\n2025-09-02,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-03,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-04,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-05,0,,0,0,0,0,1,0,0,0,0,0\n2025-09-06,0,,1,2,1,1,0,0,0,0,0,0\n2025-09-07,0,,1,2,2,0,0,0,0,0,0,0\n2025-09-08,0,,0,0,0,0,0,1,0,0,0,0\n2025-09-09,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-10,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-11,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-12,0,,0,0,0,0,1,0,0,0,0,0\n2025-09-13,0,,1,2,1,1,0,0,0,0,0,0\n2025-09-14,0,,1,2,2,0,0,0,0,0,0,0\n2025-09-15,0,,0,0,0,0,0,1,0,0,0,0\n2025-09-16,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-17,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-18,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-19,0,,0,0,0,0,1,0,0,0,0,0\n2025-09-20,0,,1,2,1,1,0,0,0,0,0,0\n2025-09-21,0,,1,2,2,0,0,0,0,0,0,0\n2025-09-22,0,,0,0,0,0,0,1,0,0,0,0\n2025-09-23,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-24,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-25,0,,0,0,0,0,0,0,0,0,0,0\n2025-09-26,0,,0,0,0,0,1,0,0,0,0,0\n2025-09-27,0,,1,2,1,1,0,0,0,0,0,0\n2025-09-28,0,,1,2,2,0,0,0,0,0,0,0\n2025-09-29,0,,0,0,0,0,0,1,0,0,0,0\n2025-09-30,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-01,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-02,0,,0,0,0,0,1,0,1,0,0,0\n2025-10-03,1,개천절,1,7,1,6,0,0,0,0,0,0\n2025-10-04,0,,1,7,2,5,0,0,0,0,0,0\n2025-10-05,1,추석 전날,1,7,3,4,0,0,0,0,0,0\n2025-10-06,1,추석,1,7,4,3,0,0,0,0,0,0\n2025-10-07,1,추석 다음날,1,7,5,2,0,0,0,0,0,0\n2025-10-08,1,추석 대체공휴일,1,7,6,1,0,0,0,0,0,0\n2025-10-09,1,한글날,1,7,7,0,0,0,0,0,0,0\n2025-10-10,0,,0,0,0,0,1,1,0,1,0,0\n2025-10-11,0,,1,2,1,1,0,0,0,0,0,0\n2025-10-12,0,,1,2,2,0,0,0,0,0,0,0\n2025-10-13,0,,0,0,0,0,0,1,0,0,0,0\n2025-10-14,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-15,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-16,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-17,0,,0,0,0,0,1,0,0,0,0,0\n2025-10-18,0,,1,2,1,1,0,0,0,0,0,0\n2025-10-19,0,,1,2,2,0,0,0,0,0,0,0\n2025-10-20,0,,0,0,0,0,0,1,0,0,0,0\n2025-10-21,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-22,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-23,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-24,0,,0,0,0,0,1,0,0,0,0,0\n2025-10-25,0,,1,2,1,1,0,0,0,0,0,0\n2025-10-26,0,,1,2,2,0,0,0,0,0,0,0\n2025-10-27,0,,0,0,0,0,0,1,0,0,0,0\n2025-10-28,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-29,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-30,0,,0,0,0,0,0,0,0,0,0,0\n2025-10-31,0,,0,0,0,0,1,0,0,0,0,0\n2025-11-01,0,,1,2,1,1,0,0,0,0,0,0\n2025-11-02,0,,1,2,2,0,0,0,0,0,0,0\n2025-11-03,0,,0,0,0,0,0,1,0,0,0,0\n2025-11-04,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-05,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-06,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-07,0,,0,0,0,0,1,0,0,0,0,0\n2025-11-08,0,,1,2,1,1,0,0,0,0,0,0\n2025-11-09,0,,1,2,2,0,0,0,0,0,0,0\n2025-11-10,0,,0,0,0,0,0,1,0,0,0,0\n2025-11-11,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-12,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-13,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-14,0,,0,0,0,0,1,0,0,0,0,0\n2025-11-15,0,,1,2,1,1,0,0,0,0,0,0\n2025-11-16,0,,1,2,2,0,0,0,0,0,0,0\n2025-11-17,0,,0,0,0,0,0,1,0,0,0,0\n2025-11-18,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-19,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-20,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-21,0,,0,0,0,0,1,0,0,0,0,0\n2025-11-22,0,,1,2,1,1,0,0,0,0,0,0\n2025-11-23,0,,1,2,2,0,0,0,0,0,0,0\n2025-11-24,0,,0,0,0,0,0,1,0,0,0,0\n2025-11-25,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-26,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-27,0,,0,0,0,0,0,0,0,0,0,0\n2025-11-28,0,,0,0,0,0,1,0,0,0,0,0\n2025-11-29,0,,1,2,1,1,0,0,0,0,0,0\n2025-11-30,0,,1,2,2,0,0,0,0,0,0,0\n2025-12-01,0,,0,0,0,0,0,1,0,0,0,0\n2025-12-02,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-03,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-04,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-05,0,,0,0,0,0,1,0,0,0,0,0\n2025-12-06,0,,1,2,1,1,0,0,0,0,0,0\n2025-12-07,0,,1,2,2,0,0,0,0,0,0,0\n2025-12-08,0,,0,0,0,0,0,1,0,0,0,0\n2025-12-09,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-10,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-11,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-12,0,,0,0,0,0,1,0,0,0,0,0\n2025-12-13,0,,1,2,1,1,0,0,0,0,0,0\n2025-12-14,0,,1,2,2,0,0,0,0,0,0,0\n2025-12-15,0,,0,0,0,0,0,1,0,0,0,0\n2025-12-16,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-17,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-18,0,,0,0,0,0,0,0,0,0,0,0\n2025-12-19,0,,0,0,0,0,1,0,0,0,0,0\n2025-12-20,0,,1,2,1,1,0,0,0,0,0,0\n2025-12-21,0,,1,2,2,0,0,0,0,0,0,0\n2025-12-22,0,,0,0,0,0,0,1,0,0,1,0\n2025-12-23,0,,0,0,0,0,0,0,0,0,1,0\n2025-12-24,0,,0,0,0,0,1,0,0,0,1,0\n2025-12-25,1,성탄절,1,1,1,0,0,0,0,0,1,0\n2025-12-26,0,,0,0,0,0,1,1,0,0,1,1\n2025-12-27,0,,1,2,1,1,0,0,0,0,1,1\n2025-12-28,0,,1,2,2,0,0,0,0,0,1,1\n2025-12-29,0,,0,0,0,0,0,1,0,0,1,1\n2025-12-30,0,,0,0,0,0,0,0,0,0,1,1\n2025-12-31,0,,0,0,0,0,1,0,0,0,1,1\n2026-01-01,1,신정,1,1,1,0,0,0,0,0,1,1\n2026-01-02,0,,0,0,0,0,1,1,0,0,1,1\n2026-01-03,0,,1,2,1,1,0,0,0,0,1,1\n2026-01-04,0,,1,2,2,0,0,0,0,0,1,1\n2026-01-05,0,,0,0,0,0,0,1,0,0,1,1\n2026-01-06,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-07,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-08,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-09,0,,0,0,0,0,1,0,0,0,1,1\n2026-01-10,0,,1,2,1,1,0,0,0,0,1,1\n2026-01-11,0,,1,2,2,0,0,0,0,0,1,1\n2026-01-12,0,,0,0,0,0,0,1,0,0,1,1\n2026-01-13,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-14,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-15,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-16,0,,0,0,0,0,1,0,0,0,1,1\n2026-01-17,0,,1,2,1,1,0,0,0,0,1,1\n2026-01-18,0,,1,2,2,0,0,0,0,0,1,1\n2026-01-19,0,,0,0,0,0,0,1,0,0,1,1\n2026-01-20,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-21,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-22,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-23,0,,0,0,0,0,1,0,0,0,1,1\n2026-01-24,0,,1,2,1,1,0,0,0,0,1,1\n2026-01-25,0,,1,2,2,0,0,0,0,0,1,1\n2026-01-26,0,,0,0,0,0,0,1,0,0,1,1\n2026-01-27,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-28,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-29,0,,0,0,0,0,0,0,0,0,1,1\n2026-01-30,0,,0,0,0,0,1,0,0,0,1,1\n2026-01-31,0,,1,2,1,1,0,0,0,0,1,1\n2026-02-01,0,,1,2,2,0,0,0,0,0,1,1\n2026-02-02,0,,0,0,0,0,0,1,0,0,1,1\n2026-02-03,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-04,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-05,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-06,0,,0,0,0,0,1,0,0,0,1,1\n2026-02-07,0,,1,2,1,1,0,0,0,0,1,1\n2026-02-08,0,,1,2,2,0,0,0,0,0,1,1\n2026-02-09,0,,0,0,0,0,0,1,0,0,1,1\n2026-02-10,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-11,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-12,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-13,0,,0,0,0,0,1,0,1,0,1,1\n2026-02-14,0,,1,5,1,4,0,0,0,0,1,1\n2026-02-15,0,,1,5,2,3,0,0,0,0,1,1\n2026-02-16,1,설날 전날,1,5,3,2,0,0,0,0,1,1\n2026-02-17,1,설날,1,5,4,1,0,0,0,0,1,1\n2026-02-18,1,설날 다음날,1,5,5,0,0,0,0,0,1,1\n2026-02-19,0,,0,0,0,0,0,1,0,1,1,1\n2026-02-20,0,,0,0,0,0,1,0,0,0,1,1\n2026-02-21,0,,1,2,1,1,0,0,0,0,1,1\n2026-02-22,0,,1,2,2,0,0,0,0,0,1,1\n2026-02-23,0,,0,0,0,0,0,1,0,0,1,1\n2026-02-24,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-25,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-26,0,,0,0,0,0,0,0,0,0,1,1\n2026-02-27,0,,0,0,0,0,1,0,1,0,1,1\n2026-02-28,0,,1,3,1,2,0,0,0,0,1,1\n2026-03-01,1,삼일절,1,3,2,1,0,0,0,0,0,0\n2026-03-02,1,삼일절 대체공휴일,1,3,3,0,0,0,0,0,0,0\n2026-03-03,0,,0,0,0,0,0,1,0,1,0,0\n2026-03-04,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-05,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-06,0,,0,0,0,0,1,0,0,0,0,0\n2026-03-07,0,,1,2,1,1,0,0,0,0,0,0\n2026-03-08,0,,1,2,2,0,0,0,0,0,0,0\n2026-03-09,0,,0,0,0,0,0,1,0,0,0,0\n2026-03-10,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-11,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-12,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-13,0,,0,0,0,0,1,0,0,0,0,0\n2026-03-14,0,,1,2,1,1,0,0,0,0,0,0\n2026-03-15,0,,1,2,2,0,0,0,0,0,0,0\n2026-03-16,0,,0,0,0,0,0,1,0,0,0,0\n2026-03-17,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-18,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-19,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-20,0,,0,0,0,0,1,0,0,0,0,0\n2026-03-21,0,,1,2,1,1,0,0,0,0,0,0\n2026-03-22,0,,1,2,2,0,0,0,0,0,0,0\n2026-03-23,0,,0,0,0,0,0,1,0,0,0,0\n2026-03-24,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-25,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-26,0,,0,0,0,0,0,0,0,0,0,0\n2026-03-27,0,,0,0,0,0,1,0,0,0,0,0\n2026-03-28,0,,1,2,1,1,0,0,0,0,0,0\n2026-03-29,0,,1,2,2,0,0,0,0,0,0,0\n2026-03-30,0,,0,0,0,0,0,1,0,0,0,0\n2026-03-31,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-01,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-02,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-03,0,,0,0,0,0,1,0,0,0,0,0\n2026-04-04,0,,1,2,1,1,0,0,0,0,0,0\n2026-04-05,0,,1,2,2,0,0,0,0,0,0,0\n2026-04-06,0,,0,0,0,0,0,1,0,0,0,0\n2026-04-07,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-08,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-09,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-10,0,,0,0,0,0,1,0,0,0,0,0\n2026-04-11,0,,1,2,1,1,0,0,0,0,0,0\n2026-04-12,0,,1,2,2,0,0,0,0,0,0,0\n2026-04-13,0,,0,0,0,0,0,1,0,0,0,0\n2026-04-14,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-15,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-16,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-17,0,,0,0,0,0,1,0,0,0,0,0\n2026-04-18,0,,1,2,1,1,0,0,0,0,0,0\n2026-04-19,0,,1,2,2,0,0,0,0,0,0,0\n2026-04-20,0,,0,0,0,0,0,1,0,0,0,0\n2026-04-21,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-22,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-23,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-24,0,,0,0,0,0,1,0,0,0,0,0\n2026-04-25,0,,1,2,1,1,0,0,0,0,0,0\n2026-04-26,0,,1,2,2,0,0,0,0,0,0,0\n2026-04-27,0,,0,0,0,0,0,1,0,0,0,0\n2026-04-28,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-29,0,,0,0,0,0,0,0,0,0,0,0\n2026-04-30,0,,0,0,0,0,1,0,1,0,0,0\n2026-05-01,1,노동절,1,3,1,2,0,0,0,0,0,0\n2026-05-02,0,,1,3,2,1,0,0,0,0,0,0\n2026-05-03,0,,1,3,3,0,0,0,0,0,0,0\n2026-05-04,0,,0,0,0,0,1,1,0,1,0,0\n2026-05-05,1,어린이날,1,1,1,0,0,0,0,0,0,0\n2026-05-06,0,,0,0,0,0,0,1,0,0,0,0\n2026-05-07,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-08,0,,0,0,0,0,1,0,0,0,0,0\n2026-05-09,0,,1,2,1,1,0,0,0,0,0,0\n2026-05-10,0,,1,2,2,0,0,0,0,0,0,0\n2026-05-11,0,,0,0,0,0,0,1,0,0,0,0\n2026-05-12,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-13,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-14,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-15,0,,0,0,0,0,1,0,0,0,0,0\n2026-05-16,0,,1,2,1,1,0,0,0,0,0,0\n2026-05-17,0,,1,2,2,0,0,0,0,0,0,0\n2026-05-18,0,,0,0,0,0,0,1,0,0,0,0\n2026-05-19,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-20,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-21,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-22,0,,0,0,0,0,1,0,1,0,0,0\n2026-05-23,0,,1,3,1,2,0,0,0,0,0,0\n2026-05-24,1,부처님오신날,1,3,2,1,0,0,0,0,0,0\n2026-05-25,1,부처님오신날 대체공휴일,1,3,3,0,0,0,0,0,0,0\n2026-05-26,0,,0,0,0,0,0,1,0,1,0,0\n2026-05-27,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-28,0,,0,0,0,0,0,0,0,0,0,0\n2026-05-29,0,,0,0,0,0,1,0,0,0,0,0\n2026-05-30,0,,1,2,1,1,0,0,0,0,0,0\n2026-05-31,0,,1,2,2,0,0,0,0,0,0,0\n2026-06-01,0,,0,0,0,0,0,1,0,0,0,0\n2026-06-02,0,,0,0,0,0,1,0,0,0,0,0\n2026-06-03,1,지방선거일,1,1,1,0,0,0,0,0,0,0\n2026-06-04,0,,0,0,0,0,0,1,0,0,0,0\n2026-06-05,0,,0,0,0,0,1,0,0,0,0,0\n2026-06-06,1,현충일,1,2,1,1,0,0,0,0,0,0\n2026-06-07,0,,1,2,2,0,0,0,0,0,0,0\n2026-06-08,0,,0,0,0,0,0,1,0,0,0,0\n2026-06-09,0,,0,0,0,0,0,0,0,0,0,0\n2026-06-10,0,,0,0,0,0,0,0,0,0,0,0\n2026-06-11,0,,0,0,0,0,0,0,0,0,0,0\n2026-06-12,0,,0,0,0,0,1,0,0,0,0,0\n2026-06-13,0,,1,2,1,1,0,0,0,0,0,0\n2026-06-14,0,,1,2,2,0,0,0,0,0,0,0\n2026-06-15,0,,0,0,0,0,0,1,0,0,0,0\n2026-06-16,0,,0,0,0,0,0,0,0,0,0,0\n2026-06-17,0,,0,0,0,0,0,0,0,0,0,0\n2026-06-18,0,,0,0,0,0,0,0,0,0,0,0\n2026-06-19,0,,0,0,0,0,1,0,0,0,0,0\n2026-06-20,0,,1,2,1,1,0,0,0,0,1,0\n2026-06-21,0,,1,2,2,0,0,0,0,0,1,0\n2026-06-22,0,,0,0,0,0,0,1,0,0,1,0\n2026-06-23,0,,0,0,0,0,0,0,0,0,1,0\n2026-06-24,0,,0,0,0,0,0,0,0,0,1,0\n2026-06-25,0,,0,0,0,0,0,0,0,0,1,0\n2026-06-26,0,,0,0,0,0,1,0,0,0,1,0\n2026-06-27,0,,1,2,1,1,0,0,0,0,1,0\n2026-06-28,0,,1,2,2,0,0,0,0,0,1,0\n2026-06-29,0,,0,0,0,0,0,1,0,0,1,0\n2026-06-30,0,,0,0,0,0,0,0,0,0,1,0\n2026-07-01,0,,0,0,0,0,0,0,0,0,1,0\n2026-07-02,0,,0,0,0,0,0,0,0,0,1,0\n2026-07-03,0,,0,0,0,0,1,0,0,0,1,0\n2026-07-04,0,,1,2,1,1,0,0,0,0,1,0\n2026-07-05,0,,1,2,2,0,0,0,0,0,1,0\n2026-07-06,0,,0,0,0,0,0,1,0,0,1,0\n2026-07-07,0,,0,0,0,0,0,0,0,0,1,0\n2026-07-08,0,,0,0,0,0,0,0,0,0,1,0\n2026-07-09,0,,0,0,0,0,0,0,0,0,1,0\n2026-07-10,0,,0,0,0,0,1,0,0,0,1,0\n2026-07-11,0,,1,2,1,1,0,0,0,0,1,0\n2026-07-12,0,,1,2,2,0,0,0,0,0,1,0\n2026-07-13,0,,0,0,0,0,0,1,0,0,1,0\n2026-07-14,0,,0,0,0,0,0,0,0,0,1,0\n2026-07-15,0,,0,0,0,0,0,0,0,0,1,0\n2026-07-16,0,,0,0,0,0,1,0,1,0,1,0\n2026-07-17,1,제헌절,1,3,1,2,0,0,0,0,1,0\n2026-07-18,0,,1,3,2,1,0,0,0,0,1,0\n2026-07-19,0,,1,3,3,0,0,0,0,0,1,0\n2026-07-20,0,,0,0,0,0,0,1,0,1,1,1\n2026-07-21,0,,0,0,0,0,0,0,0,0,1,1\n2026-07-22,0,,0,0,0,0,0,0,0,0,1,1\n2026-07-23,0,,0,0,0,0,0,0,0,0,1,1\n2026-07-24,0,,0,0,0,0,1,0,0,0,1,1\n2026-07-25,0,,1,2,1,1,0,0,0,0,1,1\n2026-07-26,0,,1,2,2,0,0,0,0,0,1,1\n2026-07-27,0,,0,0,0,0,0,1,0,0,1,1\n2026-07-28,0,,0,0,0,0,0,0,0,0,1,1\n2026-07-29,0,,0,0,0,0,0,0,0,0,1,1\n2026-07-30,0,,0,0,0,0,0,0,0,0,1,1\n2026-07-31,0,,0,0,0,0,1,0,0,0,1,1\n2026-08-01,0,,1,2,1,1,0,0,0,0,1,1\n2026-08-02,0,,1,2,2,0,0,0,0,0,1,1\n2026-08-03,0,,0,0,0,0,0,1,0,0,1,1\n2026-08-04,0,,0,0,0,0,0,0,0,0,1,1\n2026-08-05,0,,0,0,0,0,0,0,0,0,1,1\n2026-08-06,0,,0,0,0,0,0,0,0,0,1,1\n2026-08-07,0,,0,0,0,0,1,0,0,0,1,1\n2026-08-08,0,,1,2,1,1,0,0,0,0,1,1\n2026-08-09,0,,1,2,2,0,0,0,0,0,1,1\n2026-08-10,0,,0,0,0,0,0,1,0,0,1,1\n2026-08-11,0,,0,0,0,0,0,0,0,0,1,1\n2026-08-12,0,,0,0,0,0,0,0,0,0,1,1\n2026-08-13,0,,0,0,0,0,0,0,0,0,1,1\n2026-08-14,0,,0,0,0,0,1,0,1,0,1,1\n2026-08-15,1,광복절,1,3,1,2,0,0,0,0,1,1\n2026-08-16,0,,1,3,2,1,0,0,0,0,1,1\n2026-08-17,1,광복절 대체공휴일,1,3,3,0,0,0,0,0,1,0\n2026-08-18,0,,0,0,0,0,0,1,0,1,1,0\n2026-08-19,0,,0,0,0,0,0,0,0,0,1,0\n2026-08-20,0,,0,0,0,0,0,0,0,0,1,0\n2026-08-21,0,,0,0,0,0,1,0,0,0,1,0\n2026-08-22,0,,1,2,1,1,0,0,0,0,1,0\n2026-08-23,0,,1,2,2,0,0,0,0,0,1,0\n2026-08-24,0,,0,0,0,0,0,1,0,0,1,0\n2026-08-25,0,,0,0,0,0,0,0,0,0,1,0\n2026-08-26,0,,0,0,0,0,0,0,0,0,1,0\n2026-08-27,0,,0,0,0,0,0,0,0,0,1,0\n2026-08-28,0,,0,0,0,0,1,0,0,0,1,0\n2026-08-29,0,,1,2,1,1,0,0,0,0,1,0\n2026-08-30,0,,1,2,2,0,0,0,0,0,1,0\n2026-08-31,0,,0,0,0,0,0,1,0,0,1,0\n2026-09-01,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-02,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-03,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-04,0,,0,0,0,0,1,0,0,0,0,0\n2026-09-05,0,,1,2,1,1,0,0,0,0,0,0\n2026-09-06,0,,1,2,2,0,0,0,0,0,0,0\n2026-09-07,0,,0,0,0,0,0,1,0,0,0,0\n2026-09-08,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-09,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-10,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-11,0,,0,0,0,0,1,0,0,0,0,0\n2026-09-12,0,,1,2,1,1,0,0,0,0,0,0\n2026-09-13,0,,1,2,2,0,0,0,0,0,0,0\n2026-09-14,0,,0,0,0,0,0,1,0,0,0,0\n2026-09-15,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-16,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-17,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-18,0,,0,0,0,0,1,0,0,0,0,0\n2026-09-19,0,,1,2,1,1,0,0,0,0,0,0\n2026-09-20,0,,1,2,2,0,0,0,0,0,0,0\n2026-09-21,0,,0,0,0,0,0,1,0,0,0,0\n2026-09-22,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-23,0,,0,0,0,0,1,0,1,0,0,0\n2026-09-24,1,추석 전날,1,4,1,3,0,0,0,0,0,0\n2026-09-25,1,추석,1,4,2,2,0,0,0,0,0,0\n2026-09-26,1,추석 다음날,1,4,3,1,0,0,0,0,0,0\n2026-09-27,0,,1,4,4,0,0,0,0,0,0,0\n2026-09-28,0,,0,0,0,0,0,1,0,1,0,0\n2026-09-29,0,,0,0,0,0,0,0,0,0,0,0\n2026-09-30,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-01,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-02,0,,0,0,0,0,1,0,1,0,0,0\n2026-10-03,1,개천절,1,3,1,2,0,0,0,0,0,0\n2026-10-04,0,,1,3,2,1,0,0,0,0,0,0\n2026-10-05,1,개천절 대체공휴일,1,3,3,0,0,0,0,0,0,0\n2026-10-06,0,,0,0,0,0,0,1,0,1,0,0\n2026-10-07,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-08,0,,0,0,0,0,1,0,1,0,0,0\n2026-10-09,1,한글날,1,3,1,2,0,0,0,0,0,0\n2026-10-10,0,,1,3,2,1,0,0,0,0,0,0\n2026-10-11,0,,1,3,3,0,0,0,0,0,0,0\n2026-10-12,0,,0,0,0,0,0,1,0,1,0,0\n2026-10-13,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-14,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-15,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-16,0,,0,0,0,0,1,0,0,0,0,0\n2026-10-17,0,,1,2,1,1,0,0,0,0,0,0\n2026-10-18,0,,1,2,2,0,0,0,0,0,0,0\n2026-10-19,0,,0,0,0,0,0,1,0,0,0,0\n2026-10-20,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-21,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-22,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-23,0,,0,0,0,0,1,0,0,0,0,0\n2026-10-24,0,,1,2,1,1,0,0,0,0,0,0\n2026-10-25,0,,1,2,2,0,0,0,0,0,0,0\n2026-10-26,0,,0,0,0,0,0,1,0,0,0,0\n2026-10-27,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-28,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-29,0,,0,0,0,0,0,0,0,0,0,0\n2026-10-30,0,,0,0,0,0,1,0,0,0,0,0\n2026-10-31,0,,1,2,1,1,0,0,0,0,0,0\n'
EMB_WX = 'date,avg_temp,max_temp,min_temp,rainfall,avg_wind,sunshine,humidity,has_rain\r\n2025-01-01,3.100000000000001,7.700000000000002,-1.4000000000000004,0.0,3.0,7.0,63.100000000000016,0.0\r\n2025-01-02,0.3,4.7,-2.4,0.0,3.0,8.6,46.399999999999984,0.0\r\n2025-01-03,-1.2,2.1,-4.0,0.0,2.899999999999999,6.5,50.80000000000001,0.0\r\n2025-01-04,-0.3,4.0,-4.3,0.0,2.1999999999999993,7.6,46.399999999999984,0.0\r\n2025-01-05,2.3000000000000007,3.600000000000001,0.5,7.799999999999998,3.3000000000000007,1.6999999999999995,87.5,1.0\r\n2025-01-06,0.6,3.600000000000001,-2.6,2.6,3.5,4.5,76.10000000000002,1.0\r\n2025-01-07,-2.5,0.3999999999999999,-4.7,0.0,3.3000000000000007,6.5,53.5,0.0\r\n2025-01-08,-3.1999999999999993,-0.3999999999999999,-6.9,0.0,3.5,4.399999999999999,57.600000000000016,0.0\r\n2025-01-09,-9.799999999999997,-7.0,-11.099999999999996,0.0,5.899999999999999,7.9,42.5,0.0\r\n2025-01-10,-7.4,-2.6999999999999993,-11.700000000000003,0.0,2.1999999999999993,8.799999999999997,53.300000000000004,0.0\r\n2025-01-11,-5.399999999999999,-1.3,-8.799999999999997,0.0,1.6999999999999995,8.799999999999997,60.899999999999984,0.0\r\n2025-01-12,-3.1999999999999993,0.7000000000000002,-6.700000000000002,0.0,1.4000000000000004,8.0,62.3,0.0\r\n2025-01-13,0.5,3.8,-4.2,0.3,2.6,2.5,72.5,1.0\r\n2025-01-14,1.6999999999999995,4.3,-2.0,0.0,2.6999999999999993,3.7,74.3,0.0\r\n2025-01-15,-3.7,-1.8000000000000005,-5.5,0.0,4.3,8.9,42.5,0.0\r\n2025-01-16,-0.6,3.0,-5.600000000000001,0.0,3.1999999999999993,0.3,63.600000000000016,0.0\r\n2025-01-17,-0.7000000000000002,4.0,-4.3,0.0,2.0,8.9,54.100000000000016,0.0\r\n2025-01-18,1.0,5.399999999999999,-2.899999999999999,0.0,1.4000000000000004,7.9,55.100000000000016,0.0\r\n2025-01-19,4.0,8.200000000000003,0.09999999999999998,0.0,2.4,3.899999999999999,67.60000000000002,0.0\r\n2025-01-20,3.3000000000000007,6.200000000000002,1.5,0.0,1.5,3.100000000000001,81.10000000000002,0.0\r\n2025-01-21,2.1,5.3,-1.5,0.0,1.6999999999999995,4.8,85.3,0.0\r\n2025-01-22,2.1,4.2,0.7000000000000002,0.0,2.1,4.0,84.5,0.0\r\n2025-01-23,0.9000000000000002,5.600000000000001,-2.1999999999999993,0.0,2.1,9.299999999999997,58.100000000000016,0.0\r\n2025-01-24,2.1,7.4,-2.5,0.0,2.6,9.200000000000003,50.399999999999984,0.0\r\n2025-01-25,4.399999999999999,10.0,-0.19999999999999996,0.0,2.899999999999999,6.9,53.600000000000016,0.0\r\n2025-01-26,4.600000000000001,7.0,2.6,0.0,3.0,1.6999999999999995,49.0,0.0\r\n2025-01-27,-0.7999999999999998,3.600000000000001,-2.899999999999999,6.200000000000002,4.899999999999999,1.9,76.89999999999998,1.0\r\n2025-01-28,-4.2,-2.6999999999999993,-5.0,1.0,4.3,0.7999999999999998,80.39999999999998,1.0\r\n2025-01-29,-4.7,-2.6,-6.100000000000001,0.0,4.0,7.6,66.10000000000002,0.0\r\n2025-01-30,-3.899999999999999,-2.1999999999999993,-7.1,0.0,1.8000000000000005,1.5,45.30000000000001,0.0\r\n2025-01-31,-1.3,0.3999999999999999,-3.3000000000000007,1.6999999999999995,2.1,0.0,81.0,1.0\r\n2025-02-01,0.3999999999999999,5.2,-3.1999999999999993,0.0,2.6,7.299999999999998,69.3,0.0\r\n2025-02-02,0.7000000000000002,5.399999999999999,-2.6,0.0,2.4,7.1,69.0,0.0\r\n2025-02-03,-4.7,0.0,-8.1,0.0,4.8,8.9,54.8,0.0\r\n2025-02-04,-9.4,-6.700000000000002,-11.299999999999997,0.0,4.5,9.299999999999997,58.0,0.0\r\n2025-02-05,-8.5,-4.5,-11.099999999999996,0.0,3.899999999999999,6.5,49.399999999999984,0.0\r\n2025-02-06,-5.399999999999999,-1.0,-11.099999999999996,0.6,2.8000000000000007,6.200000000000002,62.0,1.0\r\n2025-02-07,-8.4,-4.3,-10.299999999999997,0.3999999999999999,5.899999999999999,7.700000000000002,61.8,1.0\r\n2025-02-08,-7.9,-4.5,-11.0,0.0,3.600000000000001,9.799999999999997,54.399999999999984,0.0\r\n2025-02-09,-4.7,-0.9000000000000002,-8.299999999999997,0.0,2.0,8.799999999999997,65.3,0.0\r\n2025-02-10,-2.8000000000000007,1.5999999999999996,-7.299999999999998,0.0,1.9,7.4,66.5,0.0\r\n2025-02-11,0.6,5.600000000000001,-4.2,0.0,2.0,9.0,61.8,0.0\r\n2025-02-12,0.3,2.5,-2.899999999999999,2.5,2.6999999999999993,0.09999999999999998,77.89999999999998,1.0\r\n2025-02-13,0.9000000000000002,4.8,-4.3,0.0,3.1999999999999993,9.9,58.600000000000016,0.0\r\n2025-02-14,3.8,8.299999999999997,-0.09999999999999998,0.0,2.6,9.5,80.3,0.0\r\n2025-02-15,3.600000000000001,7.5,0.9000000000000002,0.0,2.1,1.8000000000000005,81.60000000000002,0.0\r\n2025-02-16,3.5,8.0,0.09999999999999998,0.0,3.100000000000001,4.2,78.60000000000002,0.0\r\n2025-02-17,-0.6,1.9,-3.1999999999999993,0.0,4.399999999999999,10.1,45.399999999999984,0.0\r\n2025-02-18,-2.1999999999999993,1.3,-4.399999999999999,0.0,4.399999999999999,10.200000000000003,41.5,0.0\r\n2025-02-19,-3.0,0.09999999999999998,-5.100000000000001,0.0,3.8,10.299999999999997,45.600000000000016,0.0\r\n2025-02-20,-2.899999999999999,0.7000000000000002,-5.3,0.0,3.1999999999999993,10.200000000000003,47.399999999999984,0.0\r\n2025-02-21,-2.6,1.0999999999999996,-5.3,0.0,3.3000000000000007,10.200000000000003,47.600000000000016,0.0\r\n2025-02-22,-2.8000000000000007,0.6,-5.2,0.0,3.3000000000000007,10.200000000000003,51.0,0.0\r\n2025-02-23,-3.7,-0.7000000000000002,-5.399999999999999,0.0,3.899999999999999,10.1,47.80000000000001,0.0\r\n2025-02-24,-1.0,2.899999999999999,-5.100000000000001,0.0,2.6999999999999993,10.299999999999997,54.600000000000016,0.0\r\n2025-02-25,2.1999999999999993,5.399999999999999,-1.9,0.0,3.5,5.0,77.60000000000002,0.0\r\n2025-02-26,2.4,7.0,-0.3,0.0,2.5,10.299999999999997,58.0,0.0\r\n2025-02-27,5.0,11.799999999999997,-1.2,0.0,1.8000000000000005,9.9,37.0,0.0\r\n2025-02-28,6.5,12.299999999999997,2.899999999999999,0.0,1.8000000000000005,5.5,64.89999999999998,0.0\r\n2025-03-01,7.1,11.5,0.7000000000000002,0.5,2.1,1.9,82.10000000000002,1.0\r\n2025-03-02,8.200000000000003,15.2,2.6,10.6,4.2,0.7000000000000002,70.8,1.0\r\n2025-03-03,2.8000000000000007,6.299999999999998,0.7000000000000002,7.799999999999998,6.5,5.600000000000001,64.3,1.0\r\n2025-03-04,2.1,3.100000000000001,0.3999999999999999,1.8000000000000005,4.8,1.0,69.89999999999998,1.0\r\n2025-03-05,3.7,7.0,1.5999999999999996,0.0,3.0,2.1,66.5,0.0\r\n2025-03-06,4.100000000000001,8.6,0.5,0.0,3.1999999999999993,10.5,56.399999999999984,0.0\r\n2025-03-07,4.2,9.0,0.0,0.0,1.8000000000000005,8.4,53.0,0.0\r\n2025-03-08,4.8,9.200000000000003,1.8000000000000005,0.0,2.1,9.0,64.5,0.0\r\n2025-03-09,4.899999999999999,10.0,1.5,0.0,1.6999999999999995,10.1,80.60000000000002,0.0\r\n2025-03-10,6.799999999999998,12.099999999999996,2.4,0.0,1.6999999999999995,10.5,69.39999999999998,0.0\r\n2025-03-11,7.5,13.200000000000003,3.100000000000001,0.0,1.8000000000000005,10.0,65.3,0.0\r\n2025-03-12,6.600000000000001,9.6,4.7,0.19999999999999996,2.1,0.0,81.10000000000002,1.0\r\n2025-03-13,6.399999999999999,12.299999999999997,2.1999999999999993,0.0,1.6999999999999995,8.5,66.3,0.0\r\n2025-03-14,11.200000000000003,17.5,5.3,0.0,3.1999999999999993,9.200000000000003,33.30000000000001,0.0\r\n2025-03-15,11.0,16.099999999999994,7.5,0.0,2.8000000000000007,6.399999999999999,43.899999999999984,0.0\r\n2025-03-16,3.899999999999999,8.299999999999997,1.2,0.0,4.5,5.799999999999998,58.5,0.0\r\n2025-03-17,1.9,4.899999999999999,-0.7999999999999998,2.899999999999999,3.600000000000001,10.0,51.399999999999984,1.0\r\n2025-03-18,1.6999999999999995,4.7,0.3,2.1,4.7,4.3,70.10000000000002,1.0\r\n2025-03-19,2.4,5.5,-1.2,0.0,3.0,11.299999999999997,50.5,0.0\r\n2025-03-20,6.600000000000001,12.0,1.3,0.0,3.8,9.700000000000003,73.39999999999998,0.0\r\n2025-03-21,8.9,13.5,5.899999999999999,0.0,3.7,10.6,75.8,0.0\r\n2025-03-22,10.900000000000004,15.7,7.5,0.0,3.600000000000001,11.0,63.3,0.0\r\n2025-03-23,12.700000000000003,19.7,7.4,0.0,2.8000000000000007,8.5,59.0,0.0\r\n2025-03-24,12.400000000000004,16.0,8.5,0.0,4.100000000000001,7.299999999999998,67.0,0.0\r\n2025-03-25,10.0,17.900000000000006,7.0,0.0,4.2,3.7,75.3,0.0\r\n2025-03-26,14.3,23.300000000000008,5.5,0.0,2.6,9.700000000000003,51.100000000000016,0.0\r\n2025-03-27,11.200000000000003,16.599999999999994,6.100000000000001,0.6,3.7,2.4,72.60000000000002,1.0\r\n2025-03-28,6.0,10.299999999999997,3.3000000000000007,0.0,5.100000000000001,9.700000000000003,37.5,0.0\r\n2025-03-29,3.100000000000001,6.100000000000001,0.7999999999999998,0.0,4.3,8.1,45.80000000000001,0.0\r\n2025-03-30,3.7,6.799999999999998,-0.6,0.0,3.399999999999999,8.9,50.600000000000016,0.0\r\n2025-03-31,6.9,11.099999999999996,2.6,0.0,2.4,10.299999999999997,64.3,0.0\r\n2025-04-01,8.9,14.0,4.8,0.0,2.5,11.0,65.8,0.0\r\n2025-04-02,7.799999999999998,11.200000000000003,4.899999999999999,0.0,2.0,3.399999999999999,68.89999999999998,0.0\r\n2025-04-03,8.4,13.299999999999997,4.600000000000001,0.0,2.4,11.299999999999997,68.0,0.0\r\n2025-04-04,8.799999999999997,13.7,5.399999999999999,0.0,2.5,10.1,68.3,0.0\r\n2025-04-05,7.700000000000002,8.4,6.9,17.7,2.0,0.0,90.89999999999998,1.0\r\n2025-04-06,8.299999999999997,11.5,5.399999999999999,0.0,3.100000000000001,9.1,82.5,0.0\r\n2025-04-07,12.299999999999997,17.5,6.200000000000002,0.0,4.0,10.900000000000004,62.100000000000016,0.0\r\n2025-04-08,10.6,15.0,8.6,0.0,3.100000000000001,7.299999999999998,74.89999999999998,0.0\r\n2025-04-09,10.700000000000003,14.8,7.299999999999998,0.6,2.0,2.1999999999999993,64.39999999999998,1.0\r\n2025-04-10,11.200000000000003,16.5,7.6,0.0,2.3000000000000007,11.299999999999997,72.10000000000002,0.0\r\n2025-04-11,12.700000000000003,19.5,7.200000000000002,0.0,2.1999999999999993,11.700000000000003,66.89999999999998,0.0\r\n2025-04-12,11.5,21.3,1.3,26.199999999999992,3.600000000000001,0.5,68.10000000000002,1.0\r\n2025-04-13,5.399999999999999,7.4,1.2,1.5999999999999996,6.100000000000001,5.399999999999999,73.60000000000002,1.0\r\n2025-04-14,4.8,6.399999999999999,2.5,7.6,3.7,0.3,88.10000000000002,1.0\r\n2025-04-15,7.700000000000002,11.700000000000003,3.7,0.0,3.5,12.099999999999996,77.39999999999998,0.0\r\n2025-04-16,14.099999999999996,20.3,6.299999999999998,0.0,3.600000000000001,10.299999999999997,63.600000000000016,0.0\r\n2025-04-17,17.599999999999994,20.7,15.099999999999996,0.0,3.8,2.4,74.5,0.0\r\n2025-04-18,14.7,18.900000000000006,11.400000000000004,0.0,2.4,6.299999999999998,82.89999999999998,0.0\r\n2025-04-19,13.400000000000004,16.900000000000006,9.9,10.200000000000003,3.5,0.0,96.89999999999998,1.0\r\n2025-04-20,14.2,20.8,7.9,0.0,2.0,10.1,78.60000000000002,0.0\r\n2025-04-21,19.2,24.599999999999994,13.299999999999997,0.0,2.3000000000000007,8.700000000000003,74.5,0.0\r\n2025-04-22,16.5,18.5,14.3,42.30000000000001,2.1999999999999993,0.0,92.10000000000002,1.0\r\n2025-04-23,15.099999999999996,19.900000000000006,11.200000000000003,0.0,2.8000000000000007,11.900000000000004,71.3,0.0\r\n2025-04-24,12.400000000000004,15.2,9.799999999999997,0.0,3.5,11.900000000000004,64.8,0.0\r\n2025-04-25,12.099999999999996,15.900000000000004,8.6,0.0,3.7,12.0,45.0,0.0\r\n2025-04-26,12.400000000000004,16.2,7.700000000000002,0.0,3.600000000000001,12.099999999999996,63.5,0.0\r\n2025-04-27,13.200000000000003,15.7,11.099999999999996,0.0,3.5,4.399999999999999,56.3,0.0\r\n2025-04-28,13.5,18.3,9.4,0.0,3.0,12.400000000000004,60.0,0.0\r\n2025-04-29,13.400000000000004,18.400000000000006,9.1,0.0,2.4,12.299999999999997,47.5,0.0\r\n2025-04-30,17.8,23.5,11.799999999999997,0.0,3.8,12.0,35.9,0.0\r\n2025-05-01,13.299999999999997,16.8,10.299999999999997,20.400000000000006,3.600000000000001,0.0,76.60000000000002,1.0\r\n2025-05-02,12.5,16.7,8.700000000000003,0.0,3.399999999999999,10.200000000000003,62.899999999999984,0.0\r\n2025-05-03,10.4,13.200000000000003,8.799999999999997,9.200000000000003,2.6999999999999993,3.7,84.60000000000002,1.0\r\n2025-05-04,12.799999999999997,17.900000000000006,9.299999999999997,0.0,3.7,12.5,67.5,0.0\r\n2025-05-05,12.200000000000003,14.400000000000004,9.6,0.0,2.3000000000000007,0.09999999999999998,64.39999999999998,0.0\r\n2025-05-06,13.299999999999997,17.3,8.9,0.0,2.4,7.700000000000002,62.399999999999984,0.0\r\n2025-05-07,13.299999999999997,16.7,10.299999999999997,0.0,2.4,12.599999999999996,70.39999999999998,0.0\r\n2025-05-08,16.099999999999994,22.0,10.200000000000003,0.0,1.8000000000000005,10.700000000000003,61.5,0.0\r\n2025-05-09,15.2,16.900000000000006,13.7,20.599999999999994,3.100000000000001,0.0,83.0,1.0\r\n2025-05-10,11.5,14.8,10.700000000000003,10.5,4.8,0.09999999999999998,98.5,1.0\r\n2025-05-11,13.200000000000003,17.599999999999994,10.5,0.0,2.5,7.700000000000002,86.39999999999998,0.0\r\n2025-05-12,16.400000000000006,20.8,10.700000000000003,0.0,2.8000000000000007,12.599999999999996,67.89999999999998,0.0\r\n2025-05-13,20.0,24.099999999999994,16.599999999999994,0.0,3.8,9.700000000000003,47.899999999999984,0.0\r\n2025-05-14,20.400000000000006,24.599999999999994,16.8,0.0,2.899999999999999,11.400000000000004,56.399999999999984,0.0\r\n2025-05-15,17.2,20.7,14.0,0.7000000000000002,2.5,2.8000000000000007,89.79999999999997,1.0\r\n2025-05-16,19.099999999999994,21.699999999999992,17.2,30.1,1.8000000000000005,0.0,95.60000000000002,1.0\r\n2025-05-17,16.400000000000006,21.2,12.900000000000004,3.399999999999999,2.6999999999999993,6.600000000000001,92.0,1.0\r\n2025-05-18,14.0,18.0,10.700000000000003,0.0,2.4,10.5,66.89999999999998,0.0\r\n2025-05-19,15.099999999999996,18.2,11.099999999999996,0.0,3.100000000000001,0.19999999999999996,77.0,0.0\r\n2025-05-20,20.099999999999994,22.800000000000008,15.900000000000004,0.0,2.899999999999999,0.0,94.0,0.0\r\n2025-05-21,23.099999999999994,27.4,19.7,0.0,2.0,9.4,85.5,0.0\r\n2025-05-22,21.699999999999992,26.300000000000008,18.2,0.09999999999999998,2.1,7.6,76.39999999999998,1.0\r\n2025-05-23,19.2,21.599999999999994,16.0,1.2,2.4,0.09999999999999998,58.100000000000016,1.0\r\n2025-05-24,16.099999999999994,19.599999999999994,13.299999999999997,2.0,2.8000000000000007,0.6,65.10000000000002,1.0\r\n2025-05-25,15.8,19.2,12.5,0.0,2.4,13.299999999999997,68.60000000000002,0.0\r\n2025-05-26,17.900000000000006,23.0,13.0,0.0,1.9,12.599999999999996,69.10000000000002,0.0\r\n2025-05-27,18.5,22.800000000000008,14.7,0.0,2.3000000000000007,12.200000000000003,74.0,0.0\r\n2025-05-28,18.599999999999994,23.199999999999992,15.099999999999996,0.0,2.0,8.0,74.3,0.0\r\n2025-05-29,20.099999999999994,25.5,16.2,0.0,1.6999999999999995,12.799999999999997,64.3,0.0\r\n2025-05-30,20.2,25.5,16.8,0.0,1.8000000000000005,11.900000000000004,66.60000000000002,0.0\r\n2025-05-31,18.900000000000006,21.800000000000008,15.900000000000004,0.0,1.9,5.3,64.39999999999998,0.0\r\n2025-06-01,19.599999999999994,24.699999999999992,15.7,0.0,1.8000000000000005,11.200000000000003,71.89999999999998,0.0\r\n2025-06-02,19.0,22.5,17.2,0.0,1.4000000000000004,0.0,70.89999999999998,0.0\r\n2025-06-03,18.900000000000006,21.900000000000006,16.3,0.0,3.1999999999999993,13.200000000000003,70.0,0.0\r\n2025-06-04,19.0,23.0,15.5,0.0,3.5,13.7,64.8,0.0\r\n2025-06-05,19.8,23.5,15.7,0.0,2.5,12.700000000000003,57.899999999999984,0.0\r\n2025-06-06,22.699999999999992,26.400000000000006,18.7,0.0,3.600000000000001,12.0,45.80000000000001,0.0\r\n2025-06-07,21.900000000000006,25.699999999999992,19.599999999999994,0.0,2.8000000000000007,5.2,73.60000000000002,0.0\r\n2025-06-08,23.099999999999994,26.800000000000008,19.599999999999994,0.0,1.8000000000000005,9.700000000000003,82.3,0.0\r\n2025-06-09,21.2,25.400000000000006,18.7,0.0,2.3000000000000007,9.0,83.10000000000002,0.0\r\n2025-06-10,19.3,21.5,18.0,0.0,2.6999999999999993,5.2,86.60000000000002,0.0\r\n2025-06-11,20.2,24.099999999999994,16.5,0.0,2.5,13.5,70.5,0.0\r\n2025-06-12,23.099999999999994,28.9,17.400000000000006,0.0,1.9,12.400000000000004,65.10000000000002,0.0\r\n2025-06-13,21.5,24.300000000000008,19.400000000000006,1.2,1.9,1.8000000000000005,74.39999999999998,1.0\r\n2025-06-14,23.800000000000008,30.1,20.0,0.0,2.1999999999999993,9.200000000000003,73.89999999999998,0.0\r\n2025-06-15,22.900000000000006,30.199999999999992,19.0,8.9,2.8000000000000007,8.1,83.39999999999998,1.0\r\n2025-06-16,21.400000000000006,22.800000000000008,20.400000000000006,13.0,3.1999999999999993,0.09999999999999998,96.79999999999997,1.0\r\n2025-06-17,22.5,26.099999999999998,20.3,0.0,2.6,9.1,87.29999999999997,0.0\r\n2025-06-18,22.400000000000006,26.5,18.900000000000006,0.0,2.0,8.700000000000003,77.39999999999998,0.0\r\n2025-06-19,25.5,29.800000000000008,20.3,0.19999999999999996,2.0,8.700000000000003,71.5,1.0\r\n2025-06-20,22.5,25.699999999999992,20.0,113.5,5.700000000000002,0.0,99.10000000000002,1.0\r\n2025-06-21,19.8,21.099999999999994,18.3,0.09999999999999998,1.9,0.0,98.5,1.0\r\n2025-06-22,20.2,23.0,17.2,0.0,2.5,13.599999999999996,90.29999999999997,0.0\r\n2025-06-23,23.400000000000006,28.5,18.5,0.0,1.9,12.299999999999997,82.10000000000002,0.0\r\n2025-06-24,24.699999999999992,28.0,20.7,2.1,2.1,0.3999999999999999,71.0,1.0\r\n2025-06-25,20.7,22.199999999999992,19.400000000000006,7.799999999999998,2.3000000000000007,0.0,90.5,1.0\r\n2025-06-26,21.3,24.300000000000008,19.7,0.3,2.0,5.3,89.79999999999997,1.0\r\n2025-06-27,23.699999999999992,27.800000000000008,20.400000000000006,0.0,1.5,4.5,83.8,0.0\r\n2025-06-28,24.5,25.80000000000001,23.900000000000006,0.5,1.8000000000000007,0.19999999999999996,93.10000000000002,1.0\r\n2025-06-29,25.400000000000006,28.69999999999999,23.400000000000006,0.3,2.3000000000000007,5.3,88.79999999999998,1.0\r\n2025-06-30,26.5,28.5,24.699999999999996,0.0,2.6999999999999993,4.399999999999999,86.10000000000002,0.0\r\n2025-07-01,26.69999999999999,29.0,24.300000000000004,0.7000000000000002,3.0,5.2,90.89999999999998,1.0\r\n2025-07-02,26.400000000000002,28.69999999999999,25.199999999999996,0.0,3.0,3.6000000000000014,91.39999999999998,0.0\r\n2025-07-03,26.099999999999998,29.0,25.099999999999994,0.0,2.8000000000000007,4.600000000000001,90.10000000000002,0.0\r\n2025-07-04,25.80000000000001,28.19999999999999,24.300000000000004,0.0,3.5,0.7000000000000002,88.5,0.0\r\n2025-07-05,26.599999999999998,29.30000000000001,25.0,0.0,3.1000000000000005,9.5,85.5,0.0\r\n2025-07-06,27.0,30.099999999999998,25.099999999999994,0.0,2.8000000000000007,8.5,86.89999999999998,0.0\r\n2025-07-07,27.19999999999999,30.5,25.099999999999994,0.0,2.1999999999999993,5.0,88.39999999999998,0.0\r\n2025-07-08,30.0,35.6,25.199999999999996,1.6999999999999993,2.1999999999999993,10.700000000000003,79.10000000000002,1.0\r\n2025-07-09,30.80000000000001,34.1,27.5,0.0,1.6999999999999993,8.799999999999997,69.5,0.0\r\n2025-07-10,31.099999999999998,35.19999999999999,26.69999999999999,0.0,2.4,12.700000000000003,48.5,0.0\r\n2025-07-11,30.19999999999999,34.69999999999999,25.300000000000008,0.0,2.8999999999999995,13.400000000000006,46.80000000000001,0.0\r\n2025-07-12,29.19999999999999,33.1,25.099999999999994,0.0,2.1999999999999993,13.0,49.10000000000001,0.0\r\n2025-07-13,25.199999999999996,28.19999999999999,23.699999999999996,0.19999999999999996,1.8000000000000007,1.5,78.0,1.0\r\n2025-07-14,23.5,27.599999999999998,21.400000000000006,15.099999999999994,2.1999999999999993,0.7999999999999998,86.60000000000002,1.0\r\n2025-07-15,22.5,24.400000000000006,20.7,25.0,2.4,0.7999999999999998,91.60000000000002,1.0\r\n2025-07-16,22.300000000000004,23.699999999999996,21.099999999999994,64.0,3.1000000000000005,0.0,96.5,1.0\r\n2025-07-17,22.800000000000004,24.300000000000004,21.900000000000006,55.699999999999996,3.3000000000000003,0.3999999999999999,98.89999999999998,1.0\r\n2025-07-18,23.800000000000004,25.900000000000002,22.099999999999994,14.700000000000001,1.9000000000000001,0.09999999999999998,97.60000000000002,1.0\r\n2025-07-19,23.599999999999994,25.69999999999999,21.5,52.39999999999998,2.6,0.09999999999999998,98.5,1.0\r\n2025-07-20,26.80000000000001,29.80000000000001,24.199999999999996,30.099999999999998,3.3999999999999986,8.5,89.29999999999998,1.0\r\n2025-07-21,27.19999999999999,29.5,25.69999999999999,0.5,2.6999999999999993,8.700000000000003,88.79999999999998,1.0\r\n2025-07-22,27.400000000000002,29.80000000000001,25.80000000000001,0.0,2.5,9.9,88.79999999999998,0.0\r\n2025-07-23,27.80000000000001,31.19999999999999,25.0,0.0,1.9000000000000001,11.5,81.0,0.0\r\n2025-07-24,28.099999999999998,32.1,25.300000000000008,0.0,1.8000000000000007,8.299999999999997,79.89999999999998,0.0\r\n2025-07-25,29.599999999999998,33.9,25.300000000000008,0.0,2.1,13.0,77.10000000000002,0.0\r\n2025-07-26,30.400000000000002,34.80000000000001,26.599999999999998,0.0,2.0,12.599999999999998,74.89999999999998,0.0\r\n2025-07-27,30.900000000000002,34.4,28.0,0.0,2.0,13.200000000000001,70.5,0.0\r\n2025-07-28,29.69999999999999,34.1,26.69999999999999,0.0,1.8000000000000007,13.200000000000001,75.60000000000002,0.0\r\n2025-07-29,30.099999999999998,34.1,26.30000000000001,0.0,1.9000000000000001,12.700000000000003,71.8,0.0\r\n2025-07-30,30.0,33.5,26.099999999999998,0.0,1.8000000000000007,4.5,77.10000000000002,0.0\r\n2025-07-31,29.599999999999998,32.30000000000001,28.099999999999998,0.0,2.3000000000000007,2.4,70.0,0.0\r\n2025-08-01,29.19999999999999,32.1,27.400000000000002,0.0,2.0,5.0,80.8,0.0\r\n2025-08-02,29.19999999999999,32.19999999999999,27.19999999999999,0.0,1.8000000000000007,10.700000000000003,78.10000000000002,0.0\r\n2025-08-03,27.900000000000002,31.099999999999998,23.5,3.1000000000000005,1.9000000000000001,0.6,78.60000000000002,1.0\r\n2025-08-04,27.30000000000001,32.1,23.699999999999996,0.9000000000000004,2.6,8.200000000000003,83.39999999999998,1.0\r\n2025-08-05,27.30000000000001,30.19999999999999,24.699999999999996,0.0,2.5,9.4,81.3,0.0\r\n2025-08-06,26.69999999999999,28.5,25.099999999999994,7.299999999999997,3.3000000000000003,1.4000000000000004,93.29999999999998,1.0\r\n2025-08-07,27.30000000000001,30.80000000000001,25.300000000000008,0.0,2.5,11.400000000000002,80.8,0.0\r\n2025-08-08,26.19999999999999,29.099999999999998,24.0,0.0,2.0,8.799999999999997,80.10000000000002,0.0\r\n2025-08-09,25.69999999999999,27.099999999999998,24.099999999999994,0.0,1.5,0.6,78.89999999999998,0.0\r\n2025-08-10,26.69999999999999,31.099999999999998,22.599999999999994,0.0,2.1999999999999993,12.599999999999998,75.39999999999998,0.0\r\n2025-08-11,26.30000000000001,29.19999999999999,24.800000000000004,0.0,2.1,2.6999999999999993,83.60000000000002,0.0\r\n2025-08-12,27.69999999999999,32.6,22.400000000000006,0.0,2.1,11.700000000000003,68.89999999999998,0.0\r\n2025-08-13,25.199999999999996,26.69999999999999,23.400000000000006,189.90000000000003,3.6999999999999997,0.0,95.89999999999998,1.0\r\n2025-08-14,24.5,25.599999999999994,23.800000000000004,31.0,2.6,0.19999999999999996,98.5,1.0\r\n2025-08-15,25.599999999999994,28.400000000000002,23.699999999999996,60.699999999999996,1.5,4.399999999999999,95.89999999999998,1.0\r\n2025-08-16,27.19999999999999,29.80000000000001,25.400000000000006,0.0,2.4,10.1,90.0,0.0\r\n2025-08-17,27.30000000000001,30.400000000000002,25.5,0.0,2.4,3.3000000000000003,88.60000000000002,0.0\r\n2025-08-18,27.900000000000002,30.400000000000002,26.69999999999999,0.0,3.1999999999999993,5.799999999999999,89.29999999999998,0.0\r\n2025-08-19,28.099999999999998,30.5,26.400000000000002,0.0,3.1000000000000005,8.6,88.5,0.0\r\n2025-08-20,28.30000000000001,30.69999999999999,26.599999999999998,0.0,3.0,8.299999999999997,85.89999999999998,0.0\r\n2025-08-21,28.599999999999998,31.5,26.900000000000002,0.0,2.3000000000000007,9.1,82.10000000000002,0.0\r\n2025-08-22,28.30000000000001,30.80000000000001,26.5,0.0,2.3000000000000007,11.099999999999998,84.60000000000002,0.0\r\n2025-08-23,28.69999999999999,31.5,26.80000000000001,0.0,2.1999999999999993,8.0,85.10000000000002,0.0\r\n2025-08-24,29.19999999999999,32.30000000000001,26.900000000000002,0.0,1.9000000000000001,8.799999999999997,79.5,0.0\r\n2025-08-25,28.5,31.900000000000002,25.900000000000002,4.2,3.6999999999999997,5.700000000000001,87.29999999999998,1.0\r\n2025-08-26,26.900000000000002,30.5,25.199999999999996,10.0,3.0,7.1000000000000005,86.60000000000002,1.0\r\n2025-08-27,26.5,29.900000000000002,23.800000000000004,0.0,1.8000000000000007,7.8999999999999995,77.3,0.0\r\n2025-08-28,27.599999999999998,30.400000000000002,24.599999999999994,0.0,2.8000000000000007,5.0,78.89999999999998,0.0\r\n2025-08-29,28.400000000000002,30.900000000000002,26.599999999999998,2.3000000000000007,3.8000000000000003,4.899999999999999,89.5,1.0\r\n2025-08-30,28.19999999999999,31.19999999999999,26.19999999999999,0.7999999999999998,2.6,7.5,86.39999999999998,1.0\r\n2025-08-31,28.0,30.80000000000001,26.0,0.0,2.6,7.200000000000003,82.89999999999998,0.0\r\n2025-09-01,26.19999999999999,29.400000000000002,24.199999999999996,10.799999999999997,3.3000000000000003,2.3000000000000007,88.79999999999998,1.0\r\n2025-09-02,25.69999999999999,29.5,21.5,0.0,1.8000000000000007,10.700000000000003,80.39999999999998,0.0\r\n2025-09-03,27.69999999999999,30.900000000000002,24.099999999999994,0.0,1.6999999999999993,8.5,72.60000000000002,0.0\r\n2025-09-04,25.69999999999999,28.69999999999999,23.0,35.69999999999999,2.6,1.9000000000000001,85.5,1.0\r\n2025-09-05,27.80000000000001,31.69999999999999,24.199999999999996,0.0,2.4,7.3999999999999995,74.0,0.0\r\n2025-09-06,26.5,30.0,24.699999999999996,17.5,3.6000000000000014,3.0,92.39999999999998,1.0\r\n2025-09-07,25.099999999999994,27.400000000000002,23.199999999999996,0.0,2.0,1.6999999999999993,82.10000000000002,0.0\r\n2025-09-08,24.599999999999994,27.80000000000001,21.900000000000006,0.0,1.5999999999999996,3.1000000000000005,76.8,0.0\r\n2025-09-09,24.300000000000004,26.80000000000001,22.699999999999996,0.0,1.5,1.3,75.10000000000002,0.0\r\n2025-09-10,25.199999999999996,30.5,20.0,0.0,2.3000000000000007,11.099999999999998,65.3,0.0\r\n2025-09-11,26.400000000000002,31.30000000000001,20.8,0.0,2.1,10.900000000000002,55.5,0.0\r\n2025-09-12,23.900000000000006,26.69999999999999,20.599999999999994,5.600000000000001,2.6999999999999993,0.09999999999999998,69.39999999999998,1.0\r\n2025-09-13,22.800000000000004,25.599999999999994,20.2,36.69999999999999,2.5,0.0,86.10000000000002,1.0\r\n2025-09-14,24.300000000000004,27.80000000000001,21.2,0.0,1.6999999999999993,7.299999999999997,78.89999999999998,0.0\r\n2025-09-15,25.400000000000006,29.099999999999998,22.300000000000004,0.0,1.3,6.299999999999999,81.3,0.0\r\n2025-09-16,24.0,27.19999999999999,22.099999999999994,17.0,2.1999999999999993,0.6,92.5,1.0\r\n2025-09-17,22.099999999999994,23.900000000000006,20.400000000000006,51.89999999999998,2.5,1.0,95.39999999999998,1.0\r\n2025-09-18,22.0,26.19999999999999,19.3,0.0,2.4,3.8000000000000003,77.5,0.0\r\n2025-09-19,19.7,21.5,17.8,13.799999999999999,2.5,0.0,82.5,1.0\r\n2025-09-20,19.5,23.0,17.400000000000006,30.5,2.5,2.6999999999999993,89.79999999999998,1.0\r\n2025-09-21,21.400000000000006,25.69999999999999,16.900000000000006,0.0,2.4,9.5,68.8,0.0\r\n2025-09-22,21.699999999999996,24.199999999999996,19.0,0.0,2.1999999999999993,2.6,60.300000000000004,0.0\r\n2025-09-23,23.0,26.599999999999998,20.099999999999994,0.0,2.4,4.5,54.800000000000004,0.0\r\n2025-09-24,21.800000000000004,23.599999999999994,20.599999999999994,25.300000000000008,3.0,0.7000000000000002,85.60000000000002,1.0\r\n2025-09-25,22.099999999999994,24.900000000000006,20.3,0.6,1.5,5.399999999999999,87.10000000000002,1.0\r\n2025-09-26,21.900000000000006,25.599999999999994,18.3,0.0,1.9000000000000001,10.299999999999997,82.5,0.0\r\n2025-09-27,23.199999999999996,27.099999999999998,19.3,0.0,1.9000000000000001,7.3999999999999995,77.5,0.0\r\n2025-09-28,20.3,22.400000000000006,18.2,48.5,2.0,1.5,88.60000000000002,1.0\r\n2025-09-29,19.900000000000006,23.099999999999994,17.3,0.0,1.5999999999999996,7.1000000000000005,77.10000000000002,0.0\r\n2025-09-30,19.900000000000006,22.800000000000004,17.5,0.0,1.3,1.8000000000000007,80.8,0.0\r\n2025-10-01,20.400000000000006,24.699999999999996,17.599999999999994,3.1999999999999993,1.9000000000000001,7.1000000000000005,83.5,1.0\r\n2025-10-02,21.099999999999994,24.199999999999996,17.900000000000006,0.0,1.4000000000000004,4.2,78.5,0.0\r\n2025-10-03,21.099999999999994,22.199999999999996,19.3,2.1999999999999993,2.8999999999999995,0.0,76.89999999999998,1.0\r\n2025-10-04,22.400000000000006,24.599999999999994,20.900000000000006,0.0,2.0,0.7000000000000002,77.89999999999998,0.0\r\n2025-10-05,21.900000000000006,23.099999999999994,21.2,7.700000000000003,1.2,0.0,91.79999999999998,1.0\r\n2025-10-06,18.0,21.400000000000006,15.299999999999999,37.80000000000001,4.0,0.0,95.10000000000002,1.0\r\n2025-10-07,17.8,19.099999999999994,15.400000000000006,7.1000000000000005,3.3999999999999986,0.0,98.29999999999998,1.0\r\n2025-10-08,20.7,25.80000000000001,16.099999999999994,0.0,2.1999999999999993,10.4,77.10000000000002,0.0\r\n2025-10-09,20.3,22.099999999999994,18.0,0.3999999999999999,2.6999999999999993,0.0,71.8,1.0\r\n2025-10-10,17.8,18.599999999999994,16.5,26.099999999999998,2.4,0.0,95.60000000000002,1.0\r\n2025-10-11,20.599999999999994,23.099999999999994,18.099999999999994,21.2,3.5,0.7999999999999998,97.0,1.0\r\n2025-10-12,18.5,21.699999999999996,16.5,0.3,2.1999999999999993,1.9000000000000001,91.0,1.0\r\n2025-10-13,16.099999999999994,18.599999999999994,14.0,20.900000000000006,3.1000000000000005,0.0,95.29999999999998,1.0\r\n2025-10-14,17.7,20.0,14.299999999999999,0.09999999999999998,2.8000000000000007,0.19999999999999996,82.89999999999998,1.0\r\n2025-10-15,20.400000000000006,25.199999999999996,17.3,3.8000000000000003,2.5,5.0,77.60000000000002,1.0\r\n2025-10-16,20.2,24.400000000000006,16.3,1.3,2.8999999999999995,10.0,78.0,1.0\r\n2025-10-17,20.0,24.699999999999996,17.0,14.400000000000006,2.6,4.899999999999999,77.60000000000002,1.0\r\n2025-10-18,14.400000000000006,17.8,12.700000000000003,9.4,3.1000000000000005,0.09999999999999998,89.0,1.0\r\n2025-10-19,13.599999999999994,18.900000000000006,7.8999999999999995,0.0,2.6999999999999993,5.700000000000001,76.10000000000002,0.0\r\n2025-10-20,9.0,14.400000000000006,4.899999999999999,0.0,3.5,7.200000000000003,60.39999999999998,0.0\r\n2025-10-21,9.0,13.299999999999999,4.8,0.0,2.6,6.200000000000001,57.800000000000004,0.0\r\n2025-10-22,12.799999999999997,19.2,6.200000000000001,0.0,2.4,8.0,68.8,0.0\r\n2025-10-23,14.799999999999999,19.7,11.400000000000002,0.0,2.3000000000000007,7.299999999999997,73.39999999999998,0.0\r\n2025-10-24,14.900000000000006,20.400000000000006,10.4,0.0,2.1,7.299999999999997,73.5,0.0\r\n2025-10-25,14.099999999999994,17.599999999999994,11.200000000000003,0.0,2.4,8.5,65.10000000000002,0.0\r\n2025-10-26,12.799999999999997,14.400000000000006,11.099999999999998,1.3,1.9000000000000001,0.0,68.89999999999998,1.0\r\n2025-10-27,8.0,11.099999999999998,4.600000000000001,0.0,3.6999999999999997,9.4,44.10000000000001,0.0\r\n2025-10-28,8.5,13.099999999999994,3.5,0.0,1.5999999999999996,9.200000000000003,47.10000000000001,0.0\r\n2025-10-29,12.200000000000003,17.400000000000006,7.5,0.0,1.8000000000000007,8.9,63.5,0.0\r\n2025-10-30,13.400000000000006,17.8,9.0,0.0,1.5999999999999996,6.200000000000001,67.0,0.0\r\n2025-10-31,14.5,18.5,12.400000000000002,0.5,1.8000000000000007,1.2,78.10000000000002,1.0\r\n2025-11-01,12.900000000000006,16.5,10.200000000000003,1.0999999999999996,2.8000000000000007,7.700000000000003,68.0,1.0\r\n2025-11-02,8.1,11.099999999999998,3.1999999999999993,0.0,4.0,8.200000000000003,54.800000000000004,0.0\r\n2025-11-03,6.6000000000000005,12.400000000000002,0.9000000000000004,0.0,1.8000000000000007,9.299999999999997,55.39999999999998,0.0\r\n2025-11-04,9.700000000000003,14.5,5.399999999999999,0.0,1.5999999999999996,8.799999999999997,65.39999999999998,0.0\r\n2025-11-05,11.900000000000002,17.3,6.799999999999997,0.0,2.0,9.1,66.10000000000002,0.0\r\n2025-11-06,12.599999999999998,17.099999999999994,8.6,0.0,2.0,6.6000000000000005,72.89999999999998,0.0\r\n2025-11-07,14.099999999999994,18.400000000000006,9.700000000000003,0.0,2.1,8.6,69.0,0.0\r\n2025-11-08,14.599999999999994,17.400000000000006,13.099999999999994,0.0,1.9000000000000001,0.0,68.60000000000002,0.0\r\n2025-11-09,13.299999999999999,16.7,11.099999999999998,0.0,3.3000000000000003,7.3999999999999995,69.0,0.0\r\n2025-11-10,8.299999999999997,11.099999999999998,5.2,0.0,3.1999999999999993,9.200000000000003,37.5,0.0\r\n2025-11-11,9.200000000000003,13.5,3.8000000000000003,0.0,1.5,7.5,58.800000000000004,0.0\r\n2025-11-12,11.0,15.599999999999994,7.1000000000000005,0.0,1.5,8.700000000000003,70.10000000000002,0.0\r\n2025-11-13,11.400000000000002,16.900000000000006,8.0,0.0,2.3000000000000007,8.0,66.10000000000002,0.0\r\n2025-11-14,9.700000000000003,13.799999999999999,6.299999999999999,0.0,2.1,9.1,54.300000000000004,0.0\r\n2025-11-15,10.200000000000003,15.5,5.5,0.0,1.5999999999999996,8.4,55.60000000000002,0.0\r\n2025-11-16,12.200000000000003,17.0,7.1000000000000005,0.19999999999999996,2.8000000000000007,7.6000000000000005,63.0,1.0\r\n2025-11-17,3.0,9.4,0.0,0.0,4.8,5.3,44.30000000000001,0.0\r\n2025-11-18,1.9000000000000001,4.7,-1.3,0.0,3.1000000000000005,7.0,50.0,0.0\r\n2025-11-19,3.1000000000000005,7.700000000000003,-0.7999999999999998,0.0,1.8000000000000007,8.9,54.60000000000002,0.0\r\n2025-11-20,7.700000000000003,12.799999999999997,3.0,0.0,3.6999999999999997,5.5,63.39999999999998,0.0\r\n2025-11-21,8.0,11.599999999999998,2.1999999999999993,0.0,3.0,6.5,57.800000000000004,0.0\r\n2025-11-22,10.900000000000002,14.400000000000006,7.6000000000000005,0.0,2.1999999999999993,8.200000000000003,63.10000000000002,0.0\r\n2025-11-23,11.5,14.0,8.5,0.0,3.3000000000000003,1.4000000000000004,77.60000000000002,0.0\r\n2025-11-24,12.099999999999998,15.799999999999999,10.299999999999997,0.0,3.1000000000000005,6.399999999999999,79.0,0.0\r\n2025-11-25,7.1000000000000005,10.5,4.7,4.100000000000001,3.6999999999999997,1.2,80.0,1.0\r\n2025-11-26,6.5,10.0,3.1000000000000005,0.0,1.5999999999999996,3.5,68.3,0.0\r\n2025-11-27,5.600000000000001,9.0,2.8000000000000007,11.400000000000002,4.600000000000001,0.9000000000000004,78.3,1.0\r\n2025-11-28,2.4,6.700000000000003,-1.0,0.0,2.1,9.0,53.89999999999998,0.0\r\n2025-11-29,7.8999999999999995,13.0,1.2,0.0,3.3000000000000003,2.8000000000000007,63.89999999999998,0.0\r\n2025-11-30,12.400000000000002,15.200000000000001,9.9,0.0,3.1000000000000005,4.7,88.5,0.0\r\n2025-12-01,4.899999999999999,11.200000000000003,2.0,0.0,3.1999999999999993,8.799999999999997,56.60000000000002,0.0\r\n2025-12-02,-0.6,2.1999999999999993,-5.3,0.0,4.2,8.200000000000003,50.0,0.0\r\n2025-12-03,-6.299999999999999,-3.8000000000000003,-8.1,0.0,4.2,6.299999999999999,45.5,0.0\r\n2025-12-04,-3.1999999999999993,3.0,-9.1,1.0,3.6000000000000014,7.700000000000003,52.89999999999998,1.0\r\n2025-12-05,-1.5,2.6999999999999993,-5.0,0.0,2.8000000000000007,8.700000000000003,52.300000000000004,0.0\r\n2025-12-06,5.0,9.9,-0.3,0.19999999999999996,3.8999999999999986,0.09999999999999998,71.8,1.0\r\n2025-12-07,9.1,12.799999999999997,6.0,0.0,3.3000000000000003,6.200000000000001,75.39999999999998,0.0\r\n2025-12-08,3.3000000000000003,6.700000000000003,0.7999999999999998,0.0,3.1000000000000005,8.700000000000003,47.80000000000001,0.0\r\n2025-12-09,3.6999999999999997,8.4,-1.0999999999999996,0.0,2.1999999999999993,7.5,59.10000000000002,0.0\r\n2025-12-10,6.399999999999999,10.299999999999997,2.6999999999999993,0.0,2.8000000000000007,1.8000000000000007,79.3,0.0\r\n2025-12-11,3.6000000000000014,7.3999999999999995,-2.8999999999999995,15.5,3.8000000000000003,5.799999999999999,70.60000000000002,1.0\r\n2025-12-12,1.2,5.5,-3.8999999999999986,0.0,3.0,7.200000000000003,56.5,0.0\r\n2025-12-13,2.5,4.600000000000001,-0.7999999999999998,8.299999999999997,2.1999999999999993,0.0,84.39999999999998,1.0\r\n2025-12-14,0.5,2.6999999999999993,-1.2,0.0,3.0,7.6000000000000005,71.10000000000002,0.0\r\n2025-12-15,1.8000000000000007,5.5,-1.6999999999999993,0.0,2.1999999999999993,2.1,70.10000000000002,0.0\r\n2025-12-16,4.899999999999999,5.799999999999999,3.5,0.7000000000000002,2.1,0.0,88.29999999999998,1.0\r\n2025-12-17,3.3999999999999986,6.0,1.0,0.0,2.5,8.5,74.39999999999998,0.0\r\n2025-12-18,2.6,6.700000000000003,-1.5999999999999996,0.0,1.8000000000000007,8.6,62.10000000000002,0.0\r\n2025-12-19,6.399999999999999,11.900000000000002,0.3,0.0,2.1,4.7,65.8,0.0\r\n2025-12-20,7.700000000000003,11.200000000000003,0.9000000000000004,0.3,3.1000000000000005,0.0,78.10000000000002,1.0\r\n2025-12-21,-1.4000000000000004,0.9000000000000004,-3.8999999999999986,0.0,3.6999999999999997,8.4,45.30000000000001,0.0\r\n2025-12-22,-0.7000000000000002,4.2,-5.700000000000001,0.0,1.9000000000000001,8.299999999999997,47.60000000000001,0.0\r\n2025-12-23,2.6,5.600000000000001,0.3999999999999999,10.299999999999997,2.3000000000000007,0.0,73.60000000000002,1.0\r\n2025-12-24,1.5999999999999996,3.8000000000000003,-1.5,0.0,3.3000000000000003,6.799999999999997,83.0,0.0\r\n2025-12-25,-3.8000000000000003,-0.9000000000000004,-9.299999999999997,0.0,5.3,7.3999999999999995,54.39999999999998,0.0\r\n2025-12-26,-8.299999999999997,-5.2,-11.200000000000003,0.0,3.3999999999999986,8.700000000000003,42.5,0.0\r\n2025-12-27,-2.3000000000000007,1.4000000000000004,-7.6000000000000005,0.0,2.1,2.4,65.60000000000002,0.0\r\n2025-12-28,3.1999999999999993,7.299999999999997,-1.0,1.0,2.8000000000000007,1.8000000000000007,85.3,1.0\r\n2025-12-29,4.600000000000001,8.200000000000003,0.0,0.7000000000000002,3.1000000000000005,7.1000000000000005,76.10000000000002,1.0\r\n2025-12-30,-0.5,2.1999999999999993,-2.8000000000000007,0.0,3.3000000000000003,8.6,42.89999999999999,0.0\r\n2025-12-31,-6.100000000000001,-1.4000000000000004,-8.9,0.0,4.399999999999999,8.700000000000003,42.6,0.0\r\n2026-01-01,-7.6000000000000005,-4.0,-9.9,0.0,4.0,8.299999999999997,43.30000000000001,0.0\r\n2026-01-02,-8.4,-5.0,-11.599999999999998,0.0,3.1999999999999993,8.700000000000003,48.89999999999999,0.0\r\n2026-01-03,-4.0,-0.9000000000000004,-8.6,0.0,1.4000000000000004,3.3000000000000003,59.39999999999998,0.0\r\n2026-01-04,0.0,3.8000000000000003,-3.1000000000000005,0.0,1.9000000000000001,7.5,68.60000000000002,0.0\r\n2026-01-05,-2.0,0.3999999999999999,-4.2,0.0,3.6000000000000014,8.799999999999997,47.89999999999999,0.0\r\n2026-01-06,0.09999999999999998,4.100000000000001,-6.200000000000001,0.0,2.3000000000000007,6.299999999999999,56.39999999999998,0.0\r\n2026-01-07,-0.09999999999999998,4.2,-4.600000000000001,0.0,4.600000000000001,8.700000000000003,52.39999999999998,0.0\r\n2026-01-08,-4.899999999999999,-2.1,-7.700000000000003,0.0,3.6000000000000014,8.799999999999997,40.4,0.0\r\n2026-01-09,1.0999999999999996,4.5,-5.0,0.0,3.8000000000000003,2.0,68.10000000000002,0.0\r\n2026-01-10,1.9000000000000001,7.3999999999999995,-4.8,0.19999999999999996,7.299999999999997,4.600000000000001,60.800000000000004,1.0\r\n2026-01-11,-6.700000000000003,-4.3,-9.200000000000003,0.0,4.5,8.6,44.0,0.0\r\n2026-01-12,-2.5,3.1000000000000005,-9.6,1.8000000000000007,3.3000000000000003,1.5999999999999996,73.10000000000002,1.0\r\n2026-01-13,-3.6000000000000014,3.3999999999999986,-7.299999999999997,0.0,4.2,8.9,57.0,0.0\r\n2026-01-14,-2.5,1.5999999999999996,-8.799999999999997,0.0,2.1,6.299999999999999,43.10000000000001,0.0\r\n2026-01-15,4.8,8.9,1.0,0.0,2.1999999999999993,4.8,80.10000000000002,0.0\r\n2026-01-16,2.6,9.799999999999997,0.0,0.0,2.0,5.0,87.10000000000002,0.0\r\n2026-01-17,-1.5,1.6999999999999993,-3.5,0.0,2.8999999999999995,8.700000000000003,67.5,0.0\r\n2026-01-18,-0.7000000000000002,2.0,-4.2,0.0,2.8999999999999995,0.19999999999999996,64.5,0.0\r\n2026-01-19,-4.899999999999999,0.0,-10.0,0.5,4.600000000000001,6.399999999999999,55.60000000000002,1.0\r\n2026-01-20,-9.200000000000003,-5.700000000000001,-12.799999999999997,0.0,3.3000000000000003,8.799999999999997,46.10000000000001,0.0\r\n2026-01-21,-9.6,-6.399999999999999,-11.400000000000002,0.0,4.0,9.299999999999997,52.800000000000004,0.0\r\n2026-01-22,-8.799999999999997,-5.0,-12.5,0.0,3.6999999999999997,9.299999999999997,47.60000000000001,0.0\r\n2026-01-23,-4.8,-1.0,-10.1,1.5,2.0,8.299999999999997,53.60000000000002,1.0\r\n2026-01-24,-5.399999999999999,-1.9000000000000001,-8.4,0.0,3.3000000000000003,9.200000000000003,45.80000000000001,0.0\r\n2026-01-25,-5.899999999999999,-3.3000000000000003,-9.0,0.0,2.8000000000000007,9.200000000000003,44.60000000000001,0.0\r\n2026-01-26,-4.399999999999999,-1.4000000000000004,-7.799999999999997,0.0,1.5,2.1999999999999993,59.800000000000004,0.0\r\n2026-01-27,-4.600000000000001,-1.2,-7.299999999999997,0.0,3.8000000000000003,9.299999999999997,48.89999999999999,0.0\r\n2026-01-28,-5.0,-1.5,-7.799999999999997,0.0,3.8000000000000003,9.200000000000003,41.1,0.0\r\n2026-01-29,-5.3,-1.4000000000000004,-8.4,0.0,2.8000000000000007,9.4,42.6,0.0\r\n2026-01-30,-6.5,-3.5,-8.9,0.0,3.0,9.299999999999997,43.0,0.0\r\n2026-01-31,-4.399999999999999,-0.3,-9.299999999999997,0.0,2.3000000000000007,9.1,50.5,0.0\r\n2026-02-01,-2.4,1.2,-6.399999999999999,0.0,2.3000000000000007,8.799999999999997,53.60000000000002,0.0\r\n2026-02-02,-2.6999999999999993,0.0,-4.600000000000001,1.2,2.8999999999999995,9.299999999999997,66.39999999999998,1.0\r\n2026-02-03,-1.2,3.8999999999999986,-7.1000000000000005,0.0,1.8000000000000007,7.6000000000000005,62.60000000000002,0.0\r\n2026-02-04,3.0,6.5,-0.09999999999999998,0.0,2.4,0.5,77.0,0.0\r\n2026-02-05,2.8000000000000007,7.700000000000003,-4.0,0.0,3.1000000000000005,3.1000000000000005,71.8,0.0\r\n2026-02-06,-7.299999999999997,-4.0,-9.9,0.0,4.399999999999999,8.5,36.0,0.0\r\n2026-02-07,-8.9,-5.3,-11.5,0.0,4.3,9.6,37.80000000000001,0.0\r\n2026-02-08,-8.5,-4.600000000000001,-12.099999999999998,0.0,3.6999999999999997,9.799999999999997,41.9,0.0\r\n2026-02-09,-3.1000000000000005,3.0,-9.5,0.0,1.6999999999999993,9.700000000000003,54.60000000000002,0.0\r\n2026-02-10,1.2,3.8999999999999986,-1.8000000000000007,0.3,2.0,0.0,79.10000000000002,1.0\r\n2026-02-11,1.5999999999999996,5.5,-0.6,0.0,2.0,4.5,88.10000000000002,0.0\r\n2026-02-12,2.6,7.1000000000000005,-2.1999999999999993,0.0,2.3000000000000007,6.299999999999999,76.8,0.0\r\n2026-02-13,5.899999999999999,8.9,3.3999999999999986,0.0,2.4,1.5999999999999996,81.8,0.0\r\n2026-02-14,7.700000000000003,11.5,5.399999999999999,0.0,1.5999999999999996,1.5,80.3,0.0\r\n2026-02-15,4.8,9.200000000000003,0.0,0.0,2.6,8.299999999999997,65.3,0.0\r\n2026-02-16,1.0,5.700000000000001,-2.4,0.0,3.3000000000000003,9.6,47.5,0.0\r\n2026-02-17,0.7999999999999998,5.0,-3.6999999999999997,0.0,2.5,10.299999999999997,63.800000000000004,0.0\r\n2026-02-18,2.4,6.100000000000001,-0.7999999999999998,0.0,3.1999999999999993,10.200000000000003,57.300000000000004,0.0\r\n2026-02-19,1.4000000000000004,5.899999999999999,-3.1000000000000005,0.0,2.0,10.4,63.300000000000004,0.0\r\n2026-02-20,6.0,11.700000000000003,0.3,0.0,3.3000000000000003,9.700000000000003,62.60000000000002,0.0\r\n2026-02-21,10.900000000000002,15.900000000000006,6.399999999999999,0.0,5.3,9.799999999999997,54.39999999999998,0.0\r\n2026-02-22,6.5,11.799999999999997,1.0,0.0,5.100000000000001,5.2,55.89999999999998,0.0\r\n2026-02-23,1.5999999999999996,4.5,-0.3999999999999999,0.0,2.8000000000000007,7.1000000000000005,35.4,0.0\r\n2026-02-24,2.1999999999999993,4.899999999999999,0.3999999999999999,0.0,2.4,1.5,34.0,0.0\r\n2026-02-25,6.0,12.599999999999998,-0.3,0.0,2.1,10.299999999999997,40.5,0.0\r\n2026-02-26,7.5,11.200000000000003,4.3,0.0,2.3000000000000007,5.899999999999999,56.60000000000002,0.0\r\n2026-02-27,8.6,14.099999999999994,4.8,0.0,3.1000000000000005,9.6,57.300000000000004,0.0\r\n2026-02-28,7.6000000000000005,13.799999999999999,2.1,0.0,2.0,10.0,59.39999999999998,0.0\r\n2026-03-01,9.1,11.900000000000002,5.799999999999999,0.0,3.3999999999999986,1.0,54.300000000000004,0.0\r\n2026-03-02,4.7,8.4,2.1,13.200000000000001,5.2,0.0,82.8,1.0\r\n2026-03-03,4.7,10.5,1.2,1.0999999999999996,3.8999999999999986,7.1000000000000005,73.89999999999998,1.0\r\n2026-03-04,4.8,10.1,1.2,0.0,2.3000000000000007,9.9,69.3,0.0\r\n2026-03-05,4.600000000000001,10.900000000000002,0.6,11.0,1.9000000000000001,4.3,69.89999999999998,1.0\r\n2026-03-06,1.5,3.3000000000000003,-0.5,0.0,3.0,0.3,77.39999999999998,0.0\r\n2026-03-07,0.5,4.2,-2.3000000000000007,0.0,3.1999999999999993,10.900000000000002,48.89999999999999,0.0\r\n2026-03-08,1.2,5.399999999999999,-3.1000000000000005,0.0,1.9000000000000001,9.0,54.60000000000002,0.0\r\n2026-03-09,4.2,7.299999999999997,2.1,0.0,3.3000000000000003,8.299999999999997,62.39999999999998,0.0\r\n2026-03-10,3.3000000000000003,8.200000000000003,-0.3999999999999999,0.0,2.1999999999999993,10.200000000000003,56.10000000000002,0.0\r\n2026-03-11,3.6999999999999997,7.299999999999997,1.0999999999999996,0.0,1.5999999999999996,1.0999999999999996,70.60000000000002,0.0\r\n2026-03-12,6.399999999999999,12.099999999999998,1.3,0.0,2.8000000000000007,10.700000000000003,62.300000000000004,0.0\r\n2026-03-13,6.8999999999999995,12.200000000000003,2.6999999999999993,0.0,2.6999999999999993,10.900000000000002,46.80000000000001,0.0\r\n2026-03-14,7.1000000000000005,10.799999999999997,4.899999999999999,0.0,1.9000000000000001,3.3999999999999986,52.60000000000002,0.0\r\n2026-03-15,6.200000000000001,10.0,3.3000000000000003,0.0,2.0,5.600000000000001,74.0,0.0\r\n2026-03-16,5.899999999999999,10.5,2.1,0.0,1.9000000000000001,9.4,62.39999999999998,0.0\r\n2026-03-17,6.799999999999997,12.200000000000003,2.1999999999999993,0.0,2.0,10.4,57.10000000000002,0.0\r\n2026-03-18,5.700000000000001,8.6,4.0,1.9000000000000001,2.8999999999999995,1.6999999999999993,70.89999999999998,1.0\r\n2026-03-19,5.399999999999999,11.200000000000003,1.8000000000000007,0.0,3.1999999999999993,11.400000000000002,51.10000000000002,0.0\r\n2026-03-20,6.100000000000001,10.6,2.6999999999999993,0.0,3.1999999999999993,10.0,74.60000000000002,0.0\r\n2026-03-21,6.200000000000001,11.900000000000002,1.9000000000000001,0.0,2.1,10.799999999999997,65.60000000000002,0.0\r\n2026-03-22,9.200000000000003,16.3,4.600000000000001,0.0,1.8000000000000007,10.299999999999997,46.10000000000001,0.0\r\n2026-03-23,10.200000000000003,16.5,5.3,0.0,2.1,10.799999999999997,45.60000000000001,0.0\r\n2026-03-24,11.099999999999998,14.900000000000006,7.299999999999997,0.0,2.3000000000000007,7.799999999999997,52.5,0.0\r\n2026-03-25,10.4,15.400000000000006,7.299999999999997,0.0,2.6,10.5,74.89999999999998,0.0\r\n2026-03-26,7.6000000000000005,12.700000000000003,5.2,0.0,2.3000000000000007,10.6,85.79999999999998,0.0\r\n2026-03-27,8.5,14.400000000000006,5.399999999999999,0.0,2.0,7.799999999999997,84.3,0.0\r\n2026-03-28,7.6000000000000005,13.200000000000001,4.2,0.0,2.0,10.299999999999997,80.0,0.0\r\n2026-03-29,10.6,16.599999999999994,4.100000000000001,0.0,1.8000000000000007,7.299999999999997,58.10000000000002,0.0\r\n2026-03-30,13.400000000000006,18.599999999999994,9.200000000000003,2.1999999999999993,1.8000000000000007,1.0,54.10000000000002,1.0\r\n2026-03-31,13.099999999999994,18.099999999999994,9.799999999999997,6.700000000000003,2.6,3.5,73.5,1.0\r\n2026-04-01,10.6,14.5,7.6000000000000005,0.0,2.1999999999999993,4.600000000000001,72.8,0.0\r\n2026-04-02,9.0,14.099999999999994,5.700000000000001,0.0,3.0,11.400000000000002,75.39999999999998,0.0\r\n2026-04-03,12.5,19.7,6.700000000000003,0.0,2.3000000000000007,7.799999999999997,68.60000000000002,0.0\r\n2026-04-04,12.099999999999998,17.900000000000006,7.1000000000000005,0.09999999999999998,3.5,9.5,70.5,1.0\r\n2026-04-05,9.0,13.299999999999999,6.299999999999999,4.2,2.8999999999999995,7.799999999999997,72.89999999999998,1.0\r\n2026-04-06,9.5,13.400000000000006,6.399999999999999,0.7999999999999998,2.8999999999999995,3.3000000000000003,65.89999999999998,1.0\r\n2026-04-07,7.6000000000000005,11.599999999999998,4.8,0.0,3.3999999999999986,12.0,52.0,0.0\r\n2026-04-08,8.799999999999997,13.5,3.8000000000000003,0.0,2.0,11.700000000000003,61.89999999999998,0.0\r\n2026-04-09,10.900000000000002,13.099999999999994,8.0,16.2,3.0,0.0,82.0,1.0\r\n2026-04-10,9.1,11.900000000000002,7.799999999999997,4.600000000000001,4.8,0.3,96.39999999999998,1.0\r\n2026-04-11,8.6,12.200000000000003,6.700000000000003,0.0,2.5,1.9000000000000001,91.60000000000002,0.0\r\n2026-04-12,11.900000000000002,19.900000000000006,5.899999999999999,0.0,1.8000000000000007,9.4,64.5,0.0\r\n2026-04-13,16.0,23.199999999999996,9.0,0.0,1.6999999999999993,10.6,51.60000000000002,0.0\r\n2026-04-14,18.0,24.5,13.099999999999994,0.0,2.0,10.799999999999997,49.0,0.0\r\n2026-04-15,16.2,22.5,11.799999999999997,0.0,2.1,11.599999999999998,59.10000000000002,0.0\r\n2026-04-16,18.7,26.400000000000002,13.0,0.0,2.6,11.799999999999997,45.5,0.0\r\n2026-04-17,14.200000000000001,17.8,12.099999999999998,0.0,2.5,3.6999999999999997,71.10000000000002,0.0\r\n2026-04-18,16.099999999999994,22.800000000000004,10.700000000000003,0.0,1.6999999999999993,11.599999999999998,68.60000000000002,0.0\r\n2026-04-19,17.400000000000006,23.800000000000004,12.599999999999998,0.0,2.0,10.799999999999997,66.3,0.0\r\n2026-04-20,11.599999999999998,15.200000000000001,8.5,0.0,3.5,2.8000000000000007,71.89999999999998,0.0\r\n2026-04-21,10.6,14.599999999999994,7.6000000000000005,0.0,3.1000000000000005,5.5,68.0,0.0\r\n2026-04-22,13.200000000000001,18.599999999999994,9.700000000000003,0.0,2.1999999999999993,6.0,50.5,0.0\r\n2026-04-23,16.599999999999994,23.199999999999996,11.099999999999998,0.0,3.0,9.1,41.4,0.0\r\n2026-04-24,16.900000000000006,22.900000000000006,11.299999999999997,0.0,2.0,12.099999999999998,33.9,0.0\r\n2026-04-25,16.2,22.800000000000004,10.799999999999997,0.0,2.1,11.900000000000002,52.60000000000002,0.0\r\n2026-04-26,16.0,22.099999999999994,10.6,0.0,1.8000000000000007,12.0,47.30000000000001,0.0\r\n2026-04-27,13.900000000000006,19.0,11.299999999999997,2.3000000000000007,2.5,4.3,63.39999999999998,1.0\r\n2026-04-28,12.5,14.900000000000006,10.5,0.0,3.1999999999999993,4.399999999999999,74.0,0.0\r\n2026-04-29,13.599999999999994,19.0,9.4,0.0,2.4,11.0,64.10000000000002,0.0\r\n2026-04-30,15.599999999999994,21.400000000000006,10.700000000000003,0.0,2.1,12.099999999999998,54.0,0.0\r\n2026-05-01,15.599999999999994,20.400000000000006,12.599999999999998,0.0,3.3999999999999986,11.400000000000002,55.89999999999998,0.0\r\n2026-05-02,15.299999999999999,19.099999999999994,12.599999999999998,0.6,2.5,0.3,77.10000000000002,1.0\r\n2026-05-03,12.5,15.200000000000001,10.799999999999997,8.700000000000003,2.5,0.0,90.5,1.0\r\n2026-05-04,13.700000000000001,19.099999999999994,9.6,0.0,3.6999999999999997,12.799999999999997,63.300000000000004,0.0\r\n2026-05-05,14.0,19.2,9.5,0.0,2.6,12.799999999999997,60.0,0.0\r\n2026-05-06,16.599999999999994,21.599999999999994,11.400000000000002,0.0,2.8000000000000007,11.700000000000003,56.60000000000002,0.0\r\n2026-05-07,14.599999999999994,19.5,12.5,4.0,3.0,1.2,80.8,1.0\r\n2026-05-08,15.400000000000006,19.7,10.700000000000003,0.0,3.6999999999999997,12.900000000000006,53.0,0.0\r\n2026-05-09,15.5,20.099999999999994,11.0,0.0,2.0,12.099999999999998,50.30000000000001,0.0\r\n2026-05-10,16.3,21.599999999999994,11.5,0.0,2.4,11.700000000000003,49.10000000000001,0.0\r\n2026-05-11,16.3,19.599999999999994,13.799999999999999,6.200000000000001,2.3000000000000007,0.0,82.89999999999998,1.0\r\n2026-05-12,18.3,22.900000000000006,15.400000000000006,1.8000000000000007,1.8000000000000007,7.5,79.89999999999998,1.0\r\n2026-05-13,17.7,21.900000000000006,13.799999999999999,0.0,1.6999999999999993,11.299999999999997,78.3,0.0\r\n2026-05-14,21.099999999999994,27.80000000000001,14.200000000000001,0.0,1.9000000000000001,12.700000000000003,68.5,0.0\r\n2026-05-15,21.599999999999994,27.0,16.099999999999994,0.0,2.0,12.700000000000003,61.10000000000002,0.0\r\n2026-05-16,20.7,26.599999999999998,16.2,0.0,2.0,13.0,63.5,0.0\r\n2026-05-17,20.5,25.69999999999999,15.0,0.0,1.9000000000000001,12.799999999999997,59.5,0.0\r\n2026-05-18,21.800000000000004,27.19999999999999,16.3,0.0,1.8000000000000007,11.0,52.10000000000002,0.0\r\n2026-05-19,21.699999999999996,24.400000000000006,19.900000000000006,0.0,2.1,0.3999999999999999,55.800000000000004,0.0\r\n2026-05-20,17.099999999999994,21.2,15.599999999999994,56.0,3.1999999999999993,0.0,88.10000000000002,1.0\r\n2026-05-21,17.400000000000006,19.900000000000006,15.400000000000006,5.899999999999999,2.8999999999999995,0.19999999999999996,88.60000000000002,1.0\r\n2026-05-22,20.599999999999994,26.599999999999998,14.700000000000001,0.0,2.3000000000000007,12.700000000000003,58.5,0.0\r\n2026-05-23,19.0,22.699999999999996,16.900000000000006,0.09999999999999998,2.3000000000000007,6.799999999999997,65.8,1.0\r\n2026-05-24,21.099999999999994,26.599999999999998,16.099999999999994,0.0,1.8000000000000007,13.0,69.3,0.0\r\n2026-05-25,23.599999999999994,27.900000000000002,18.0,0.0,1.9000000000000001,12.200000000000003,71.8,0.0\r\n2026-05-26,24.199999999999996,28.900000000000002,21.800000000000004,11.400000000000002,1.6999999999999993,0.3,82.10000000000002,1.0\r\n2026-05-27,22.0,25.099999999999994,19.400000000000006,4.3,2.4,0.0,86.89999999999998,1.0\r\n2026-05-28,19.099999999999994,22.400000000000006,16.3,1.9000000000000001,2.4,6.299999999999999,93.29999999999998,1.0\r\n2026-05-29,19.599999999999994,23.900000000000006,16.400000000000006,0.0,2.5,12.0,80.8,0.0\r\n2026-05-30,20.0,24.099999999999994,15.700000000000001,0.0,2.6,13.5,75.39999999999998,0.0\r\n2026-05-31,22.800000000000004,27.5,17.7,0.0,1.8000000000000007,13.5,58.800000000000004,0.0\r\n2026-06-01,21.699999999999996,25.400000000000006,18.3,0.0,1.6999999999999993,6.799999999999997,61.89999999999998,0.0\r\n2026-06-02,24.099999999999994,30.19999999999999,18.099999999999994,0.0,2.1,12.900000000000006,66.5,0.0\r\n2026-06-03,23.800000000000004,28.099999999999998,20.599999999999994,0.0,1.9000000000000001,7.799999999999997,72.3,0.0\r\n2026-06-04,21.699999999999996,26.30000000000001,18.2,0.9000000000000004,2.8000000000000007,2.8000000000000007,84.5,1.0\r\n2026-06-05,19.400000000000006,23.900000000000006,16.3,0.0,3.0,12.0,72.3,0.0\r\n2026-06-06,21.900000000000006,27.099999999999998,16.7,0.0,2.1999999999999993,8.0,66.0,0.0\r\n2026-06-07,22.199999999999996,25.300000000000008,19.400000000000006,0.0,2.3000000000000007,2.0,59.0,0.0\r\n2026-06-08,20.900000000000006,25.099999999999994,17.5,0.0,2.4,9.9,70.0,0.0\r\n2026-06-09,20.900000000000006,24.800000000000004,16.900000000000006,0.0,2.1999999999999993,8.200000000000003,59.800000000000004,0.0\r\n2026-06-10,20.0,23.599999999999994,17.400000000000006,0.0,2.6999999999999993,10.5,82.60000000000002,0.0\r\n2026-06-11,19.400000000000006,23.400000000000006,16.099999999999994,0.0,2.6,12.0,80.5,0.0\r\n2026-06-12,20.900000000000006,24.800000000000004,17.8,0.0,2.5,13.299999999999999,76.89999999999998,0.0\r\n2026-06-13,23.0,28.19999999999999,17.7,0.0,1.8000000000000007,13.200000000000001,66.39999999999998,0.0\r\n2026-06-14,23.900000000000006,27.900000000000002,21.2,0.0,2.1,8.700000000000003,75.5,0.0\r\n2026-06-15,24.400000000000006,28.80000000000001,20.0,0.0,1.9000000000000001,10.299999999999997,67.39999999999998,0.0\r\n2026-06-16,25.199999999999996,30.80000000000001,20.8,0.0,2.0,12.200000000000003,64.89999999999998,0.0\r\n2026-06-17,25.400000000000006,30.69999999999999,21.3,0.0,1.9000000000000001,8.200000000000003,65.3,0.0\r\n2026-06-18,25.300000000000008,30.599999999999998,21.900000000000006,0.0,2.1,12.599999999999998,64.3,0.0\r\n2026-06-19,25.599999999999994,32.6,22.0,6.100000000000001,1.9000000000000001,7.299999999999997,76.89999999999998,1.0\r\n2026-06-20,21.5,24.199999999999996,20.400000000000006,50.39999999999999,3.1000000000000005,0.0,95.5,1.0\r\n2026-06-21,23.400000000000006,27.69999999999999,20.099999999999994,0.0,2.1,7.299999999999997,71.0,0.0\r\n2026-06-22,23.599999999999994,26.5,21.5,0.0,1.6999999999999993,3.6999999999999997,80.8,0.0\r\n2026-06-23,25.400000000000006,30.599999999999998,20.8,0.0,2.1,12.700000000000003,63.39999999999998,0.0\r\n2026-06-24,24.300000000000004,29.099999999999998,19.900000000000006,0.0,2.0,11.400000000000002,55.10000000000002,0.0\r\n2026-06-25,23.099999999999994,26.80000000000001,21.0,0.0,1.9000000000000001,0.3,66.0,0.0\r\n2026-06-26,22.900000000000006,27.30000000000001,18.099999999999994,0.0,2.1999999999999993,13.900000000000006,72.8,0.0\r\n2026-06-27,25.69999999999999,30.400000000000002,21.5,0.0,1.9000000000000001,11.299999999999997,67.0,0.0\r\n2026-06-28,24.199999999999996,27.69999999999999,21.2,0.0,2.3000000000000007,13.200000000000001,76.0,0.0\r\n2026-06-29,24.5,28.69999999999999,21.0,0.0,2.0,12.099999999999998,75.8,0.0\r\n2026-06-30,25.199999999999996,29.5,22.099999999999994,0.0,2.0,9.799999999999997,70.3,0.0\r\n2026-07-01,25.900000000000002,29.599999999999998,23.199999999999996,0.0,2.1,5.100000000000001,73.10000000000002,0.0\r\n2026-07-02,24.5,28.099999999999998,22.5,1.0999999999999996,2.0,5.600000000000001,77.60000000000002,1.0\r\n2026-07-03,24.5,27.80000000000001,22.5,0.0,2.5,8.4,82.8,0.0\r\n2026-07-04,24.900000000000006,28.900000000000002,22.800000000000004,0.09999999999999998,2.4,7.6000000000000005,85.0,1.0\r\n2026-07-05,25.199999999999996,27.900000000000002,23.599999999999994,3.8999999999999986,2.6,1.4000000000000004,89.79999999999998,1.0\r\n2026-07-06,25.0,26.80000000000001,23.800000000000004,4.899999999999999,3.3999999999999986,2.0,93.89999999999998,1.0\r\n2026-07-07,26.80000000000001,30.30000000000001,22.5,23.300000000000004,2.3000000000000007,6.200000000000001,87.29999999999998,1.0\r\n2026-07-08,24.699999999999996,27.900000000000002,23.099999999999994,25.099999999999994,3.3999999999999986,3.1999999999999993,91.29999999999998,1.0\r\n2026-07-09,24.300000000000004,25.80000000000001,22.400000000000006,20.099999999999994,2.4,1.2,96.60000000000002,1.0\r\n2026-07-10,25.5,27.19999999999999,24.199999999999996,4.100000000000001,3.1000000000000005,0.5,94.10000000000002,1.0\r\n2026-07-11,28.30000000000001,31.900000000000002,24.800000000000004,0.0,1.8000000000000007,11.599999999999998,78.89999999999998,0.0\r\n2026-07-12,30.69999999999999,34.6,26.30000000000001,0.0,2.1,13.0,70.3,0.0\r\n2026-07-13,29.599999999999998,32.0,27.599999999999998,0.0,2.5,9.5,74.10000000000002,0.0\r\n2026-07-14,27.5,31.19999999999999,24.400000000000006,43.69999999999999,5.399999999999999,1.5999999999999996,85.3,1.0\r\n2026-07-15,24.300000000000004,26.80000000000001,23.0,8.0,3.8000000000000003,3.8000000000000003,95.29999999999998,1.0\r\n2026-07-16,25.599999999999994,28.30000000000001,23.300000000000004,0.0,1.9000000000000001,5.5,89.39999999999998,0.0\r\n2026-07-17,27.099999999999998,30.599999999999998,23.900000000000006,7.6000000000000005,1.4000000000000004,3.0,88.29999999999998,1.0\r\n2026-07-18,23.699999999999996,26.900000000000002,20.5,102.20000000000005,2.1999999999999993,4.2,95.39999999999998,1.0\r\n2026-07-19,24.5,26.599999999999998,22.800000000000004,0.0,1.9000000000000001,4.5,85.79999999999998,0.0\r\n2026-07-20,24.699999999999996,26.400000000000002,23.699999999999996,19.599999999999994,1.6999999999999993,0.0,95.5,1.0\r\n2026-07-21,24.800000000000004,26.400000000000002,23.599999999999994,33.30000000000001,3.6000000000000014,1.0,98.5,1.0\r\n2026-07-22,25.300000000000008,27.400000000000002,23.099999999999994,8.200000000000003,1.5,0.3,98.10000000000002,1.0\r\n2026-07-23,27.0,28.900000000000002,25.80000000000001,0.0,3.3999999999999986,0.7999999999999998,94.39999999999998,0.0\r\n2026-07-24,28.0,30.5,26.30000000000001,0.0,3.1000000000000005,9.5,87.10000000000002,0.0\r\n2026-07-25,28.30000000000001,30.69999999999999,26.69999999999999,0.0,2.8999999999999995,7.8999999999999995,86.89999999999998,0.0\r\n2026-07-26,28.400000000000002,30.80000000000001,26.69999999999999,0.19999999999999996,2.8000000000000007,9.5,88.39999999999998,1.0\r\n2026-07-27,28.0,30.30000000000001,26.69999999999999,0.0,2.8000000000000007,2.6999999999999993,89.79999999999998,0.0\r\n2026-07-28,27.69999999999999,30.400000000000002,26.30000000000001,0.0,2.6999999999999993,7.8999999999999995,90.29999999999998,0.0\r\n2026-07-29,27.5,29.900000000000002,26.099999999999998,0.0,2.8000000000000007,8.700000000000003,89.29999999999998,0.0\r\n2026-07-30,27.599999999999998,30.19999999999999,25.599999999999994,0.0,2.4,10.299999999999997,87.60000000000002,0.0\r\n2026-07-31,28.30000000000001,31.400000000000002,25.599999999999994,0.0,2.5,10.5,88.79999999999998,0.0\r\n2026-08-01,29.099999999999998,32.6,26.80000000000001,0.0,2.8000000000000007,8.799999999999997,89.39999999999998,0.0\r\n2026-08-02,29.69999999999999,33.4,27.099999999999998,0.0,1.9000000000000001,11.900000000000002,79.39999999999998,0.0\r\n2026-08-03,30.5,35.1,27.0,0.0,1.9000000000000001,12.0,80.89999999999998,0.0\r\n2026-08-04,31.19999999999999,35.30000000000001,28.19999999999999,0.0,1.9000000000000001,9.700000000000003,81.8,0.0\r\n2026-08-05,31.5,35.19999999999999,28.400000000000002,0.0,2.0,11.400000000000002,76.39999999999998,0.0\r\n2026-08-06,31.900000000000002,35.4,29.30000000000001,0.0,1.9000000000000001,10.6,73.3,0.0\r\n2026-08-07,32.30000000000001,36.4,29.0,0.0,2.0,11.5,71.10000000000002,0.0\r\n2026-08-08,31.30000000000001,36.4,27.30000000000001,0.0,3.1000000000000005,8.0,62.39999999999998,0.0\r\n2026-08-09,28.599999999999998,32.4,25.0,0.0,3.5,7.799999999999997,59.0,0.0\r\n2026-08-10,29.400000000000002,33.9,25.80000000000001,0.0,2.8000000000000007,8.5,57.800000000000004,0.0\r\n2026-08-11,28.599999999999998,33.69999999999999,25.199999999999996,0.0,2.8000000000000007,7.799999999999997,57.0,0.0\r\n2026-08-12,27.19999999999999,31.900000000000002,22.800000000000004,0.0,3.3000000000000003,10.900000000000002,51.800000000000004,0.0\r\n2026-08-13,27.5,31.69999999999999,23.099999999999994,0.0,2.4,5.600000000000001,62.89999999999998,0.0\r\n2026-08-14,28.099999999999998,31.5,25.0,0.0,1.6999999999999993,7.5,68.89999999999998,0.0\r\n2026-08-15,26.0,28.30000000000001,24.099999999999994,20.8,1.3,0.0,87.5,1.0\r\n2026-08-16,24.900000000000006,26.30000000000001,23.0,1.8000000000000007,2.3000000000000007,0.0,87.79999999999998,1.0\r\n2026-08-17,26.30000000000001,28.5,24.599999999999994,0.0,1.5,1.4000000000000004,82.60000000000002,0.0\r\n2026-08-18,27.0,30.5,24.699999999999996,0.0,2.4,11.299999999999997,81.8,0.0\r\n2026-08-19,27.099999999999998,29.400000000000002,25.5,0.0,1.9000000000000001,2.4,84.0,0.0\r\n2026-08-20,26.599999999999998,27.80000000000001,25.5,0.7000000000000002,1.2,0.0,85.89999999999998,1.0\r\n2026-08-21,26.5,27.69999999999999,25.80000000000001,0.5,1.4000000000000004,0.0,87.39999999999998,1.0\r\n2026-08-22,24.099999999999994,27.0,22.400000000000006,49.80000000000001,1.6999999999999993,0.3,94.5,1.0\r\n2026-08-23,27.599999999999998,31.5,23.099999999999994,0.0,1.8000000000000007,11.0,83.39999999999998,0.0\r\n2026-08-24,28.400000000000002,31.0,26.30000000000001,0.0,2.4,9.5,83.60000000000002,0.0\r\n2026-08-25,28.19999999999999,31.30000000000001,25.099999999999994,32.69999999999999,2.1999999999999993,10.5,87.60000000000002,1.0\r\n2026-08-26,27.599999999999998,30.19999999999999,26.0,0.0,1.5,3.3000000000000003,84.39999999999998,0.0\r\n2026-08-27,27.69999999999999,31.400000000000002,25.099999999999994,0.0,1.6999999999999993,6.700000000000003,80.5,0.0\r\n2026-08-28,27.900000000000002,30.5,26.5,0.0,1.6999999999999993,1.0999999999999996,83.89999999999998,0.0\r\n2026-08-29,26.900000000000002,30.19999999999999,25.099999999999994,0.0,2.1999999999999993,6.299999999999999,79.8,0.0\r\n2026-08-30,25.80000000000001,27.900000000000002,23.699999999999996,0.0,1.5,0.0,82.8,0.0\r\n2026-08-31,24.699999999999996,25.900000000000002,23.5,16.900000000000006,1.5,0.0,95.89999999999998,1.0\r\n2026-09-01,23.0,24.0,21.699999999999996,38.80000000000001,1.5,0.0,97.89999999999998,1.0\r\n2026-09-02,24.599999999999994,28.30000000000001,22.0,0.0,1.5999999999999996,6.0,89.5,0.0\r\n2026-09-03,25.69999999999999,29.900000000000002,22.300000000000004,0.0,2.1999999999999993,9.299999999999997,76.10000000000002,0.0\r\n2026-09-04,25.80000000000001,29.80000000000001,21.5,0.0,3.3000000000000003,10.700000000000003,54.800000000000004,0.0\r\n2026-09-05,24.800000000000004,29.30000000000001,21.099999999999994,0.0,3.1999999999999993,8.5,52.60000000000002,0.0\r\n2026-09-06,24.5,29.5,20.400000000000006,0.0,3.0,9.5,57.10000000000002,0.0\r\n2026-09-07,24.400000000000006,29.0,19.5,0.0,2.4,10.900000000000002,60.5,0.0\r\n2026-09-08,24.800000000000004,29.0,20.400000000000006,0.0,2.1,10.900000000000002,61.5,0.0\r\n2026-09-09,22.300000000000004,26.099999999999998,18.900000000000006,0.0,3.6999999999999997,4.899999999999999,51.89999999999998,0.0\r\n2026-09-10,19.400000000000006,21.699999999999996,17.599999999999994,0.0,2.4,0.3,56.39999999999998,0.0\r\n2026-09-11,20.599999999999994,25.900000000000002,15.299999999999999,0.0,1.8000000000000007,8.299999999999997,69.3,0.0\r\n2026-09-12,21.5,23.599999999999994,19.0,0.0,1.4000000000000004,0.09999999999999998,73.60000000000002,0.0\r\n2026-09-13,22.0,23.800000000000004,20.099999999999994,0.09999999999999998,1.3,0.19999999999999996,78.0,1.0\r\n2026-09-14,22.5,26.30000000000001,19.099999999999994,0.0,1.9000000000000001,7.8999999999999995,62.39999999999998,0.0\r\n2026-09-15,20.900000000000006,25.80000000000001,16.2,0.0,2.1,11.0,55.39999999999998,0.0\r\n2026-09-16,22.5,28.0,16.400000000000006,0.0,1.8000000000000007,11.0,49.30000000000001,0.0\r\n2026-09-17,23.5,28.400000000000002,19.2,0.0,2.1,6.700000000000003,51.60000000000002,0.0\r\n2026-09-18,24.699999999999996,29.099999999999998,20.3,0.0,3.0,7.1000000000000005,50.5,0.0\r\n2026-09-19,24.400000000000006,29.0,21.0,0.0,2.3000000000000007,7.299999999999997,57.10000000000002,0.0\r\n2026-09-20,23.699999999999996,28.0,20.2,0.0,2.1,10.700000000000003,64.60000000000002,0.0\r\n2026-09-21,23.199999999999996,28.0,18.900000000000006,0.0,1.9000000000000001,10.0,67.0,0.0\r\n2026-09-22,24.099999999999994,28.900000000000002,19.2,0.0,1.9000000000000001,10.299999999999997,60.0,0.0\r\n2026-09-23,23.800000000000004,27.69999999999999,20.2,0.0,2.0,10.1,72.60000000000002,0.0\r\n2026-09-24,22.800000000000004,25.900000000000002,20.2,0.0,1.5999999999999996,4.600000000000001,77.5,0.0\r\n2026-09-25,20.5,23.400000000000006,19.099999999999994,8.4,1.6999999999999993,0.09999999999999998,85.3,1.0\r\n2026-09-26,21.900000000000006,26.400000000000002,19.099999999999994,0.0,1.8000000000000007,5.3,74.5,0.0\r\n2026-09-27,22.5,27.0,18.8,0.0,1.6999999999999993,10.1,76.39999999999998,0.0\r\n2026-09-28,21.099999999999994,24.699999999999996,18.2,0.0,2.1,8.799999999999997,69.3,0.0\r\n2026-09-29,20.7,26.0,16.0,0.0,1.9000000000000001,10.5,64.3,0.0\r\n2026-09-30,21.5,26.099999999999998,18.099999999999994,0.0,2.5,9.799999999999997,67.0,0.0\r\n2026-10-01,17.900000000000006,21.800000000000004,14.200000000000001,0.0,3.1999999999999993,10.200000000000003,37.5,0.0\r\n2026-10-02,17.3,22.300000000000004,12.599999999999998,0.0,2.6,10.700000000000003,41.9,0.0\r\n2026-10-03,17.900000000000006,24.300000000000004,11.700000000000003,0.0,1.6999999999999993,6.100000000000001,52.300000000000004,0.0\r\n2026-10-04,20.2,25.80000000000001,16.0,0.3999999999999999,2.6999999999999993,8.299999999999997,61.800000000000004,1.0\r\n2026-10-05,17.3,20.900000000000006,14.900000000000006,0.0,3.1999999999999993,8.9,57.800000000000004,0.0\r\n2026-10-06,17.400000000000006,21.900000000000006,12.400000000000002,0.0,2.4,10.299999999999997,55.89999999999998,0.0\r\n2026-10-07,18.599999999999994,23.5,14.099999999999994,0.0,1.9000000000000001,10.5,55.800000000000004,0.0\r\n2026-10-08,18.5,24.199999999999996,13.299999999999999,0.0,1.5,10.4,60.0,0.0\r\n2026-10-09,19.3,24.900000000000006,15.099999999999994,0.0,1.4000000000000004,9.799999999999997,68.89999999999998,0.0\r\n2026-10-10,19.900000000000006,25.69999999999999,15.400000000000006,0.0,1.5,9.5,67.39999999999998,0.0\r\n2026-10-11,21.5,27.5,15.799999999999999,0.0,1.9000000000000001,8.200000000000003,61.89999999999998,0.0\r\n2026-10-12,23.199999999999996,28.30000000000001,19.400000000000006,0.0,1.8000000000000007,7.5,67.3,0.0\r\n2026-10-13,17.7,22.0,14.799999999999999,18.3,3.3000000000000003,3.6999999999999997,79.3,1.0\r\n2026-10-14,16.400000000000006,19.599999999999994,12.099999999999998,0.0,3.3999999999999986,10.1,42.80000000000001,0.0\r\n2026-10-15,18.099999999999994,21.699999999999996,14.5,0.3999999999999999,3.5,3.1000000000000005,68.0,1.0\r\n2026-10-16,19.2,23.300000000000004,16.400000000000006,0.19999999999999996,2.1999999999999993,6.200000000000001,82.60000000000002,1.0\r\n2026-10-17,20.900000000000006,26.099999999999998,16.3,0.0,2.5,8.700000000000003,63.10000000000002,0.0\r\n2026-10-18,18.7,21.800000000000004,15.400000000000006,29.900000000000002,5.0,0.0,80.3,1.0\r\n2026-10-19,14.0,16.5,11.0,0.0,2.1,2.8000000000000007,71.3,0.0\r\n2026-10-20,16.400000000000006,19.900000000000006,12.200000000000003,0.0,1.8000000000000007,4.899999999999999,68.8,0.0\r\n2026-10-21,13.299999999999999,16.599999999999994,8.5,0.7000000000000002,3.6000000000000014,8.0,59.0,1.0\r\n2026-10-22,10.6,14.799999999999999,6.6000000000000005,0.0,2.3000000000000007,9.9,42.5,0.0\r\n2026-10-23,15.0,21.3,8.5,0.0,2.0,8.299999999999997,59.0,0.0\r\n2026-10-24,16.2,17.5,15.099999999999994,0.0,1.4000000000000004,0.0,78.3,0.0\r\n2026-10-25,17.5,22.099999999999994,12.200000000000003,0.0,2.3000000000000007,3.6000000000000014,69.39999999999998,0.0\r\n2026-10-26,17.400000000000006,21.5,15.400000000000006,0.0,2.4,3.6000000000000014,68.89999999999998,0.0\r\n2026-10-27,12.0,16.0,8.200000000000003,0.0,4.2,9.4,36.9,0.0\r\n2026-10-28,8.700000000000003,13.0,4.0,0.0,2.6999999999999993,9.700000000000003,34.30000000000001,0.0\r\n2026-10-29,9.700000000000003,12.099999999999998,6.6000000000000005,6.100000000000001,2.4,0.09999999999999998,59.800000000000004,1.0\r\n2026-10-30,14.5,18.5,8.799999999999997,0.0,2.6999999999999993,9.1,71.3,0.0\r\n2026-10-31,12.599999999999998,17.3,9.4,0.0,2.4,9.4,54.800000000000004,0.0\r\n'
EMB_D = '...'

import numpy as np, gzip, base64, io
(WORK / 'sincon-2026').mkdir(exist_ok=True); (WORK / 'weather_data').mkdir(exist_ok=True); (WORK / 'v10').mkdir(exist_ok=True)
for n in ('train.csv', 'test.csv'):
    shutil.copy(WORK / n, WORK / 'sincon-2026' / n)
(WORK / 'calendar.csv').write_text(EMB_CAL, encoding='utf-8-sig')
wx = pd.read_csv(io.StringIO(EMB_WX))
wx.to_csv(WORK / 'weather_data' / 'train_weather.csv', index=False)
wx.iloc[:0].to_csv(WORK / 'weather_data' / 'test_weather.csv', index=False)
(WORK / 'v10' / 'exp3_newmodels.py').write_text(CODES['exp3_newmodels.py'], encoding='utf-8')
run([sys.executable, '-u', str(WORK / 'v10' / 'exp3_newmodels.py'), 'full', '0', '1', '2'], env={'MODELS': 'lgbL'}, name='lgbL')
shutil.copy(WORK / 'v10' / 'results_v10' / 'testpred_lgbL_full.csv', WORK / 'testpred_lgbL_full.csv')

# blend + corrections -> v10
run([sys.executable, '-u', str(WORK / 'final_post.py'), str(WORK)],
    env={'FP_LGB': 'lt1_full,lt2_full', 'FP_XGB_W': '0', 'FP_CHR': 'chr_e3_full,wx_chr_st_full',
         'FP_EXTRA': 'lgbL_full:0.2', 'FP_OUT': 'submission_v10.csv'}, name='v10')

# h9 + (v10 - v8d) = v10 + (h9 - v8d)  (log space)
te = pd.read_csv(WORK / 'test.csv')
v10 = pd.read_csv(WORK / 'submission_v10.csv')
import lzma
D = np.frombuffer(lzma.decompress(base64.b64decode(EMB_D)), dtype=np.int16).astype(np.float64) / 1e3  # log(h9)-log(v8d)
assert len(D) == len(te) and (v10.example_id.values == te.example_id.values).all()
P = np.log1p(v10.boardings.to_numpy()) + D
pd.DataFrame({'example_id': te.example_id, 'boardings': np.expm1(np.clip(P, 0, None))}).to_csv(WORK / 'submission_h9_v10.csv', index=False)
for out_name in ('submission_v10.csv', 'submission_h9_v10.csv'):
    s = pd.read_csv(WORK / out_name)
    print(out_name, 'rows', len(s), 'na', int(s.isna().sum().sum()), 'neg', int((s.boardings < 0).sum()),
          'cols', list(s.columns), 'order_ok', bool((s.example_id.values == te.example_id.values).all()))
shutil.copy(WORK / 'submission_h9_v10.csv', WORK / 'submission.csv')
